# Dark-Matter Trajectory Truncation - v16 with Ground-Plane Event Rate

Edit only the **Main Settings** cell and then choose **Run All**.

This notebook contains the full workflow in one file:

1. trap potential and force construction;
2. Test 0 barrier and percentile scans;
3. Test 0.75 adiabatic/resonant/Rutherford classification;
4. Test 1 trajectory geometry and $R_{\rm far}$ construction;
5. adaptive larger-$R_{\max}$ recovery for unresolved directions;
6. a three-dimensional trap-only bowl/separatrix map constructed from the first radial local maximum outward from the ion equilibrium point, including explicit +z/-z saddle and escape-point verification;
7. focused escape-channel and potential-bowl diagnostics;
8. finite-measure targeted refinement of the observed axial low-$b$ channel;
9. Test 1.5 physical-$b$ support and outward-force filtering;
10. bowl-aware Test 2 with the complete Fourier force pulse from incoming $R_{\rm outer}$ to outgoing $R_{\rm outer}$;
11. a bowl-entry/detectability contingency test;
12. outer-radius convergence auditing;
13. automatic Test 2.5 $b$- and speed-boundary refinement;
14. a post-refinement bowl-necessity report;
15. strict final conditional parameter-space construction; and
16. a separate exploratory report for unresolved or insufficiently sampled rays.

The focused escape-channel samples have zero production weight. They are diagnostic importance samples and do not replace the isotropic angular measure.

The v12 targeted refinement replaces the coarse first low-$b$ annulus inside the selected axial direction cell by finite angular subcells and a dense speed quadrature. The default seed is $b=5\,\mu\mathrm{m}$; Test 2.5 then determines the actual finite $b$ and speed boundaries. The replacement preserves the original Maxwell, solid-angle, and $b^2$ weights.


The v13 bowl is defined from the **trap-only DM potential**.  Along each spatial direction, the first resolved radial local maximum is used as the numerical bowl boundary.  Test 2 records whether the DM crosses this boundary, its residence time inside the bowl, its residence time inside $R_{\rm full}$, and whether any resolved above-threshold trajectory remains outside the bowl.


This revision preserves the original single-point v16 Tests 0-2.5 and adds a ground-structure-aware event-rate stage. The added stage uses the same $(b,R,\psi,\alpha,\theta,v)$ convention, the full v16 DC plus RF pseudopotential force, coupled ion-DM core propagation, and a finite copper prism above the ion.


In [ ]:
# USER SETTINGS - edit only this cell, then click **Run All**.
# Every speed scan and every downstream test is regenerated for this point.

from pathlib import Path
import json
import math
import re

import numpy as np
import pandas as pd
from scipy.stats import maxwell as _maxwell

# -----------------------------------------------------------------------------
# Parameter-space point
# -----------------------------------------------------------------------------
USER_M_DM_KG = 1.3e-24
USER_EPS = 1.0
USER_T_DM_K = 300.0
USER_TARGET_ION_ENERGY_J = 1.0e-27

# -----------------------------------------------------------------------------
# Upper ground plane / new structure and single-point event-rate settings
# -----------------------------------------------------------------------------
# The original Tests 0-2.5 remain unchanged and provide the weighted Test-1.5
# trajectory support.  The final ground-aware stage reruns those weighted
# representatives through a branch-aware full trajectory state machine.
USER_RUN_GROUND_EVENT_RATE = True
USER_DM_NUMBER_DENSITY_M3 = 1.0e9
USER_GROUND_EVENT_RATE_TARGET_MODE = 2
USER_GROUND_EVENT_RATE_EXACT_PHONON_NUMBER = 3

# Branch replicas average reflection/transmission/thermal re-emission outcomes.
# Increase this first when the copper branch model dominates the Monte Carlo
# uncertainty.  None means use every Test-1.5 representative.
USER_GROUND_BRANCH_REPLICAS = 8
USER_GROUND_EVENT_RATE_MAX_ROWS = None
USER_GROUND_EVENT_RATE_RANDOM_SEED = 20260806
USER_GROUND_EVENT_RATE_UNCERTAINTY_Z = 1.96  # half-width multiplier; 1.96 ~ 95% normal CI

# Full incoming and outgoing trajectory radius.  This should enclose the upper
# ground plane and should be included in the existing R_outer convergence audit.
USER_GROUND_EVENT_RATE_OUTER_RADIUS_M = 40.0e-3
USER_GROUND_MAX_INTERACTIONS = 6
USER_GROUND_MAX_CORE_PASSES = 4
USER_GROUND_UNRESOLVED_POLICY = "zero"  # "zero" gives a conservative lower estimate

# Axis-aligned copper rectangular prism above the ion, in v16 ion-centered
# coordinates.  The trap axes are x, y, z and the prism faces are parallel to
# those axes.  No spontaneous source from structures below the ion is included.
USER_UPPER_GROUND_X_BOUNDS_M = (-17.5e-3, 17.5e-3)
USER_UPPER_GROUND_Y_BOUNDS_M = (-17.5e-3, 17.5e-3)
USER_UPPER_GROUND_Z_BOUNDS_M = (5.0e-3, 6.0e-3)
USER_UPPER_GROUND_DOUBLE_LAYER_EV = 3.19
USER_UPPER_GROUND_TEMPERATURE_K = USER_T_DM_K
USER_UPPER_GROUND_ENABLE_DELAYED_REEMISSION = True

# ODE controls for the new ground-aware history calculation.  Outside R_switch
# the DM follows the full trap force and the ion evolves harmonically; inside
# R_switch the coupled ion-DM equations are solved.
USER_GROUND_OUTSIDE_RTOL = 1.0e-8
USER_GROUND_OUTSIDE_ATOL = 1.0e-11
USER_GROUND_OUTSIDE_MAX_STEP_S = 5.0e-7
USER_GROUND_CORE_RTOL = 1.0e-9
USER_GROUND_CORE_ATOL = 1.0e-12
USER_GROUND_CORE_MAX_STEP_S = 1.0e-8
USER_GROUND_COULOMB_SOFTENING_M = 1.0e-12
USER_GROUND_METAL_TRANSPORT_STEPS = 96

# Full coupled trajectories calculate ion energy in two independent ways:
#
#   1. mechanical secular energy from x_ion(t), v_ion(t), and omega_vec;
#   2. Fourier energy from the Coulomb-force quadratures at each secular mode,
#
#          E_j = |integral F_j(t) exp(i omega_j t) dt|^2 / (2 m_ion).
#
# The Fourier result is the primary detection quantity. Both results, their
# individual threshold flags, and their relative discrepancy are saved.
# Available policies: "fourier", "mechanical", "both", or "either".
USER_DETECTION_ENERGY_POLICY = "fourier"

# Set True to require the Fourier and mechanical energies to agree within the
# relative tolerance before a trajectory may be classified as detectable.
# The comparison is always recorded even when this is False.
USER_REQUIRE_FOURIER_MECHANICAL_AGREEMENT = False
USER_FOURIER_MECHANICAL_REL_TOL = 5.0e-2

# Resolve the fastest secular period with at least this many maximum solver
# steps. Increasing this improves Fourier accuracy but increases runtime.
USER_FOURIER_STEPS_PER_SECULAR_PERIOD = 20

# Production Fourier detection uses the full rise-and-fall force pulse.
# Test 2 coherently combines the incoming R_outer -> R_far tail, the existing
# R_far -> R_switch prefix, the fully coupled core, the outgoing escape -> R_far
# tail, and the final R_far -> R_outer tail. The mechanical/Fourier consistency
# diagnostic still compares mechanical energy only with E_FT(R_switch).
USER_FOURIER_DETECTION_START_RADIUS = "R_far"
USER_REQUIRE_R_FAR_PREFIX_COMPLETE = True
USER_INCLUDE_OUTGOING_R_FAR_TAIL = True
USER_REQUIRE_OUTGOING_R_FAR_TAIL_COMPLETE = True
USER_OUTGOING_TAIL_RTOL = 1.0e-6
USER_OUTGOING_TAIL_ATOL = 1.0e-9
USER_OUTGOING_TAIL_MAX_STEP_S = 5.0e-7
USER_OUTGOING_TAIL_TIME_FACTOR = 4.0
USER_OUTGOING_TAIL_MIN_TIME_S = 20.0e-6
USER_OUTGOING_TAIL_QUAD_EPSREL = 1.0e-8
USER_OUTGOING_TAIL_QUAD_EPSABS_DIMENSIONLESS = 1.0e-11
USER_OUTGOING_TAIL_QUAD_LIMIT = 1000

# Extend the weak force history beyond R_far on both sides. R_outer must be
# larger than every trajectory's R_far. The force is not literally zero at any
# finite radius, so convergence is checked by rerunning one trajectory at the
# radii in USER_SANITY_OUTER_RADII_M.
USER_INCLUDE_OUTER_R_TAILS = True
USER_REQUIRE_OUTER_R_TAILS_COMPLETE = True
USER_FOURIER_OUTER_RADIUS_M = 40.0e-3
USER_OUTER_TAIL_RTOL = 1.0e-6
USER_OUTER_TAIL_ATOL = 1.0e-9
USER_OUTER_TAIL_MAX_STEP_S = 5.0e-7
USER_OUTER_TAIL_TIME_FACTOR = 4.0
USER_OUTER_TAIL_MIN_TIME_S = 50.0e-6
USER_OUTER_TAIL_QUAD_EPSREL = 1.0e-8
USER_OUTER_TAIL_QUAD_EPSABS_DIMENSIONLESS = 1.0e-11
USER_OUTER_TAIL_QUAD_LIMIT = 1000
USER_OUTER_TAIL_FORCE_SCALE_SAMPLES = 21

# Accuracy controls for the production R_far -> R_switch Fourier prefix. The
# already-computed dense DM-only trajectory is integrated with oscillatory
# QUADPACK weights, so this adds force evaluations but no extra trap ODE solve.
USER_PREFIX_QUAD_EPSREL = 1.0e-8
USER_PREFIX_QUAD_EPSABS_DIMENSIONLESS = 1.0e-11
USER_PREFIX_QUAD_LIMIT = 1000
USER_PREFIX_FORCE_SCALE_SAMPLES = 17

# Single-trajectory full-pulse outer-radius convergence test.
# The selected trajectory is rerun at 20 mm and 40 mm by default. The relative
# change in E_FT is the direct check that the finite R_outer endpoint is far
# enough into the zero-force tail.
USER_RUN_FOURIER_RADIUS_SANITY_TEST = True
USER_SANITY_TRAJECTORY_ID = None
USER_SANITY_OUTER_RADII_M = (20.0e-3, 40.0e-3, 80.0e-3)

# -----------------------------------------------------------------------------
# Run-All stage controls
# -----------------------------------------------------------------------------
# These switches let one notebook be used for a full clean run or for targeted
# reruns after a later-stage code/settings change.
USER_RUN_ADAPTIVE_RMAX_PREPASS = True
USER_RUN_TARGETED_ESCAPE_REFINEMENT = True
USER_RUN_TEST1P5 = True
USER_RUN_TEST2 = True
USER_RUN_OUTER_RADIUS_SANITY = True
USER_RUN_TEST2P5 = True
USER_RUN_FINAL_BUILDER = True
USER_RUN_EXPLORATORY_REPORT = True

# -----------------------------------------------------------------------------
# Adaptive R_max, escape-channel, and potential-bowl diagnostics
# -----------------------------------------------------------------------------
# Only Test 1 rows unresolved because the finite outer search radius was too
# small are rerun on this geometric radius ladder.
USER_ADAPTIVE_RMAX_LADDER_M = (
    40.0e-3,
    80.0e-3,
    160.0e-3,
    320.0e-3,
    640.0e-3,
)
USER_ADAPTIVE_RMAX_MAX_ROWS = None
USER_ADAPTIVE_RMAX_N_PATH_MIN = 1400
USER_ADAPTIVE_RMAX_N_PATH_MAX = 4200

# Fast replacement for the expensive row-by-row adaptive ladder. The code
# automatically finds the two direction keys containing the most unresolved
# rows, calibrates one large R_max per direction, and caches each U/F path
# across all speed rows sharing the same (theta, alpha, b, psi) geometry.
# Set an explicit tuple such as ((0, 0), (12, 0)) to override auto-detection.
USER_DIRECTIONAL_RMAX_DIRECTION_KEYS = None
USER_DIRECTIONAL_RMAX_MAX_DIRECTIONS = 2
USER_DIRECTIONAL_RMAX_CALIBRATION_LADDER_M = (
    1.28,
    2.56,
    5.12,
    10.24,
)
USER_DIRECTIONAL_RMAX_N_PATH = 900
USER_DIRECTIONAL_RMAX_THREADS = 8
USER_DIRECTIONAL_RMAX_PRINT_EVERY = 20
USER_DIRECTIONAL_RMAX_RUN_PER_ROW_BOWL_AUDIT = False

# Focused angular scan around the physical +z and -z escape directions. The
# code direction convention is u=(cos(theta), sin(theta)cos(alpha),
# sin(theta)sin(alpha)), so physical z is at theta=pi/2 and alpha=pi/2 or 3pi/2.
USER_ESCAPE_SCAN_RMAX_M = 160.0e-3
USER_ESCAPE_SCAN_N_PATH = 2400
USER_ESCAPE_TOP_DIRECTIONS = 24
USER_ESCAPE_CONE_HALF_ANGLES_DEG = (
    0.0,
    0.5,
    1.0,
    2.0,
    5.0,
    10.0,
    15.0,
)
USER_ESCAPE_CONE_AZIMUTH_COUNT = 16

# Materialize a limited set of zero-weight Test 2 diagnostics near the
# minimum-barrier channel. These rows never contribute to production rates.
USER_MATERIALIZE_ESCAPE_FOCUS_ROWS = False
USER_ESCAPE_FOCUS_MAX_ROWS = 96
USER_ESCAPE_FOCUS_MAX_ROWS_PER_SPEED = 16
USER_ESCAPE_FOCUS_RMAX_M = 320.0e-3
USER_ESCAPE_FOCUS_N_PATH = 2800
USER_ESCAPE_FOCUS_B_HALF_WIDTH_FRACTION = 0.02

# Dynamically classify whether a trajectory turns before the bowl lip, enters
# the bowl but misses R_switch, or enters both the bowl and interaction region.
USER_RUN_DYNAMIC_BOWL_AUDIT = False
USER_BOWL_DM_RTOL = 1.0e-7
USER_BOWL_DM_ATOL = 1.0e-10
USER_BOWL_DM_MAX_STEP_S = 2.0e-7

# -----------------------------------------------------------------------------
# Targeted finite-measure refinement of the observed axial low-b branch
# -----------------------------------------------------------------------------
# This is not a zero-weight diagnostic. It replaces the coarse first inner b
# annulus of the selected parent direction cell by finite angular subcells and
# a denser speed quadrature. The total Maxwell, solid-angle, psi, and b^2 weight
# of the replaced parent region is preserved exactly.
USER_ENABLE_TARGETED_ESCAPE_REFINEMENT = True

# The displayed single-trajectory runs start at positive z and move toward the
# origin with v_z < 0, which is the code's physical "minus_z" incoming channel:
# theta=pi/2, alpha=3pi/2. Add "plus_z" to refine both axial channels.
USER_TARGETED_ESCAPE_CHANNELS = ("minus_z",)

# The dense window brackets the observed transition between sub-threshold and
# strongly detectable escaped trajectories. Original Test-0.75 representatives
# outside this window are retained as quadrature nodes so the complete parent
# speed interval remains represented without overlapping probability weight.
USER_TARGETED_ESCAPE_SPEED_WINDOW_M_S = (180.0, 240.0)
USER_TARGETED_ESCAPE_SPEED_ANCHORS_M_S = (
    180.0,
    190.0,
    195.0,
    200.0,
    205.0,
    208.0,
    210.0,
    215.0,
    220.0,
    230.0,
    240.0,
)

# Subdivide the original coarse axial direction cell uniformly in solid angle.
# 3 x 3 gives the exact axial center plus neighboring finite angular cells.
USER_TARGETED_ESCAPE_N_MU_SUBCELLS = 3
USER_TARGETED_ESCAPE_N_ALPHA_SUBCELLS = 3

# Initial impact seed and azimuthal resolution. One positive-weight row is
# created per speed/angular/psi cell. Its physical b support is the complete
# first inner Test-1 annulus; Test 2.5 refines the actual detectable boundary.
USER_TARGETED_ESCAPE_B_SEED_M = 5.0e-6
USER_TARGETED_ESCAPE_N_PSI = 4

# Outer-tail construction for the targeted Test-1 rows.
USER_TARGETED_ESCAPE_RMAX_M = 40.0e-3
USER_TARGETED_ESCAPE_N_PATH = 1000
USER_TARGETED_ESCAPE_THREADS = 8
USER_TARGETED_ESCAPE_MAX_IN_FLIGHT = 32
USER_TARGETED_ESCAPE_PRINT_EVERY = 25

# Keep every targeted seed in the Test-2 pilot instead of allowing Test 1.5 to
# replace the finite subcell by a random representative from a coarser stratum.
USER_TARGETED_ESCAPE_FORCE_PILOT_SELECTION = True

# Test-2.5 uses tighter local boundary tolerances only for targeted rays.
USER_TARGETED_ESCAPE_B_TOLERANCE_M = 1.0e-6
USER_TARGETED_ESCAPE_SPEED_TOLERANCE_M_S = 1.0

# Test 2.5 first maps geometric b reachability, then refines the speed boundary
# using the selected energy threshold. "detectability" means the final speed-b
# support requires above_energy_threshold under USER_DETECTION_ENERGY_POLICY.
# "reachability" restores the previous geometry-only speed refinement.
USER_SPEED_REFINEMENT_TARGET = "detectability"

# Automatic Test 2/Test 2.5 refinement.
#
# With AUTO_RUN_TEST2P5_REFINEMENT=True, one Run All performs the pilot Test 2,
# lets Test 2.5 generate a refinement round, runs Test 2 on that round, reruns
# Test 2.5, and repeats until no new rows remain or the round cap is reached.
AUTO_RUN_TEST2P5_REFINEMENT = True

# Reuse a completed pilot Test 2 calculation only when the exact Test 1.5
# input bytes and the trajectory-refinement configuration fingerprint match.
AUTO_REUSE_COMPLETED_TEST2_PILOT = False
TRAJECTORY_REFINEMENT_CODE_VERSION = "2026-07-27-targeted-escape-resolution-v12"

# False resumes any completed round files already present for this exact
# parameter-point directory and runs only missing rounds. Set True when changes
# to Test 1.5, Test 2, Test 2.5, tolerances, or selection logic require a clean
# refinement from Round 1.
RESTART_TEST2P5_REFINEMENT = True

# Safety cap for automatic Test 2 executions. Test 2.5 also has its own
# MAX_REFINEMENT_ROUNDS setting; the smaller of the two limits is used.
AUTO_TEST2P5_MAX_ROUNDS = 12

# After the b-boundary refinement converges, Test 2.5 also brackets the lowest
# speed at which each retained angular ray reaches R_full. New midpoint-speed
# probes are passed back through Test 2 automatically by the same Run All loop.
AUTO_REFINE_SPEED_REACHABILITY = True
SPEED_REACHABILITY_TOLERANCE_M_S = 5.0
MAX_SPEED_REFINEMENT_ROWS_PER_ROUND = 32
# Refine the speed transition at several confirmed-reaching b values for
# each angular geometry. The final table is conditional in both v and b.
SPEED_REACHABILITY_B_ANCHOR_COUNT = 5

# Operational per-trajectory full-coupling radius.
#
# The energy-threshold radius is intrinsically a function of speed only. The
# actual repulsive Rutherford closest approach depends on both asymptotic speed
# and impact parameter. The staging sphere must not be set only by that closest
# approach, because the trapped DM can turn around much farther out. The default
# encloses both the speed-only threshold radius and r_min(v,b).
#
# Available modes:
#   "rutherford_closest_approach"   -> R_full = r_min(v,b) [diagnostic only]
#   "enclose_threshold_and_closest" -> max(r_threshold(v), r_min(v,b))
#   "legacy_threshold_only"         -> speed-only threshold behavior
USER_R_FULL_VB_MODE = "enclose_threshold_and_closest"
USER_R_SWITCH_FACTOR = 2.0

# Test 0 settings. The single-speed diagnostic is evaluated at these
# Maxwell-Boltzmann quantiles, while the full quantile scan uses the integer
# percentile range below. All speeds are recomputed from USER_M_DM_KG.
USER_TEST0_SINGLE_SPEED_QUANTILES = (0.50,)
USER_TEST0_EXTRA_SPEEDS_M_S = ()
USER_MB_QUANTILE_MIN_PERCENT = 1
USER_MB_QUANTILE_MAX_PERCENT = 99
USER_MB_QUANTILE_STEP_PERCENT = 1
USER_SPEED_UPPER_TAIL_PROBABILITY = 1.0e-10
USER_TEST0_CRITICAL_SCAN_POINTS = 40
USER_TEST0_CRITICAL_MIN_QUANTILE = 1.0e-6

if not np.isfinite(USER_M_DM_KG) or USER_M_DM_KG <= 0.0:
    raise ValueError("USER_M_DM_KG must be finite and positive")
if not np.isfinite(USER_EPS) or USER_EPS <= 0.0:
    raise ValueError("USER_EPS must be finite and positive for the repulsive branch")
if not np.isfinite(USER_T_DM_K) or USER_T_DM_K <= 0.0:
    raise ValueError("USER_T_DM_K must be finite and positive")
if not np.isfinite(USER_TARGET_ION_ENERGY_J) or USER_TARGET_ION_ENERGY_J <= 0.0:
    raise ValueError("USER_TARGET_ION_ENERGY_J must be finite and positive")
if str(USER_DETECTION_ENERGY_POLICY).strip().lower() not in {
    "fourier", "mechanical", "both", "either"
}:
    raise ValueError(
        "USER_DETECTION_ENERGY_POLICY must be 'fourier', 'mechanical', "
        "'both', or 'either'"
    )
if not np.isfinite(USER_FOURIER_MECHANICAL_REL_TOL) or USER_FOURIER_MECHANICAL_REL_TOL < 0.0:
    raise ValueError("USER_FOURIER_MECHANICAL_REL_TOL must be finite and nonnegative")
if int(USER_FOURIER_STEPS_PER_SECULAR_PERIOD) < 8:
    raise ValueError("USER_FOURIER_STEPS_PER_SECULAR_PERIOD must be at least 8")
if str(USER_SPEED_REFINEMENT_TARGET).strip().lower() not in {
    "reachability", "detectability"
}:
    raise ValueError(
        "USER_SPEED_REFINEMENT_TARGET must be 'reachability' or 'detectability'"
    )
if not 0.0 < USER_SPEED_UPPER_TAIL_PROBABILITY < 1.0:
    raise ValueError("USER_SPEED_UPPER_TAIL_PROBABILITY must lie in (0, 1)")

# Canonical names used by the trap and trajectory code.
m_dm = float(USER_M_DM_KG)
eps = float(USER_EPS)
T_dm = float(USER_T_DM_K)
k_B = 1.380649e-23


def rutherford_v_b_radius_policy(
    speed_m_s: float,
    b_m: float,
    *,
    m_dm_kg: float,
    m_ion_kg: float,
    eps_value: float,
    ion_charge_number: float,
    coulomb_constant: float,
    elementary_charge_c: float,
    threshold_j: float,
    mode: str = USER_R_FULL_VB_MODE,
    r_switch_factor: float = USER_R_SWITCH_FACTOR,
) -> dict[str, float | str]:
    """Return threshold and operational radii for one exact ``(v,b)`` row.

    ``r_min_energy_threshold_m`` is the largest repulsive Rutherford closest
    approach that can transfer ``threshold_j`` and therefore depends on speed
    only. ``r_min_rutherford_v_b_m`` is the actual free-space closest approach
    for this trajectory and depends on both speed and impact parameter.
    """
    speed = float(speed_m_s)
    impact = abs(float(b_m))
    dm_mass = float(m_dm_kg)
    ion_mass = float(m_ion_kg)
    threshold = float(threshold_j)
    if not np.isfinite(speed) or speed <= 0.0:
        raise ValueError("speed_m_s must be finite and positive")
    if not np.isfinite(impact) or impact < 0.0:
        raise ValueError("b_m must be finite and nonnegative")
    if dm_mass <= 0.0 or ion_mass <= 0.0 or threshold <= 0.0:
        raise ValueError("masses and threshold_j must be positive")

    mu = dm_mass * ion_mass / (dm_mass + ion_mass)
    coupling = abs(
        float(coulomb_constant)
        * float(ion_charge_number)
        * float(eps_value)
        * float(elementary_charge_c) ** 2
    )
    if coupling <= 0.0:
        raise ValueError("The Coulomb coupling magnitude must be positive")

    a_m = coupling / (mu * speed**2)
    r_closest = a_m + math.sqrt(a_m * a_m + impact * impact)
    head_on_recoil = 2.0 * mu * mu * speed * speed / ion_mass
    recoil = head_on_recoil * a_m * a_m / max(
        a_m * a_m + impact * impact,
        np.finfo(float).tiny,
    )

    if head_on_recoil >= threshold:
        r_threshold = (
            a_m
            + coupling / speed * math.sqrt(2.0 / (ion_mass * threshold))
        )
        b_threshold = a_m * math.sqrt(
            max(head_on_recoil / threshold - 1.0, 0.0)
        )
    else:
        r_threshold = float("nan")
        b_threshold = float("nan")

    normalized_mode = str(mode).strip().lower()
    if normalized_mode == "rutherford_closest_approach":
        r_full = r_closest
    elif normalized_mode == "enclose_threshold_and_closest":
        r_full = (
            max(r_threshold, r_closest)
            if np.isfinite(r_threshold)
            else r_closest
        )
    elif normalized_mode == "legacy_threshold_only":
        if not np.isfinite(r_threshold):
            raise ValueError(
                "legacy_threshold_only is undefined below the head-on "
                "kinematic threshold"
            )
        r_full = r_threshold
    else:
        raise ValueError(
            "USER_R_FULL_VB_MODE must be 'rutherford_closest_approach', "
            "'enclose_threshold_and_closest', or 'legacy_threshold_only'"
        )

    return {
        "rutherford_a_m": float(a_m),
        "r_min_energy_threshold_m": float(r_threshold),
        "r_min_threshold_speed_only_m": float(r_threshold),
        "r_min_rutherford_v_b_m": float(r_closest),
        "rutherford_b_threshold_m": float(b_threshold),
        "rutherford_recoil_v_b_J": float(recoil),
        "rutherford_head_on_recoil_J": float(head_on_recoil),
        "rutherford_above_threshold_v_b": bool(recoil >= threshold),
        "R_full_m": float(r_full),
        "R_switch_m": float(r_switch_factor) * float(r_full),
        "R_switch_factor": float(r_switch_factor),
        "R_full_v_b_mode": normalized_mode,
    }


def maxwell_speed_from_quantile(quantile_fraction: float) -> float:
    q = float(quantile_fraction)
    if not 0.0 < q < 1.0:
        raise ValueError("Maxwell-Boltzmann quantiles must lie in (0, 1)")
    scale = math.sqrt(k_B * T_dm / m_dm)
    return float(_maxwell.ppf(q, scale=scale))


# Every parameter point receives its own directory. This prevents Test 1.5,
# Test 2, Test 2.5, or the final builder from accidentally loading stale CSVs
# created for a different mass or charge.
def _run_value_tag(value: float) -> str:
    return (
        f"{float(value):.10e}"
        .replace("+", "")
        .replace("-", "m")
        .replace(".", "p")
    )


RUN_TAG = (
    f"m_{_run_value_tag(m_dm)}__eps_{_run_value_tag(eps)}__T_{T_dm:.6g}K"
    .replace(".", "p")
)
RESULTS_ROOT = Path("trajectory_parameter_results")
RUN_DIRECTORY = RESULTS_ROOT / RUN_TAG
RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)

TEST0_SINGLE_OUTPUT_PREFIX = str(RUN_DIRECTORY / "test0_reach_aware_v10")
TEST0_QUANTILE_OUTPUT_PREFIX = str(RUN_DIRECTORY / "test0_fast_quantile_sphere_reject_v10")
TEST0P75_OUTPUT_PREFIX = str(RUN_DIRECTORY / "test0p75_vb_reach_aware_v10")
TEST1_OUTPUT_PREFIX = str(RUN_DIRECTORY / "test1_reach_aware_v10")
TEST1P5_OUTPUT_PREFIX = str(RUN_DIRECTORY / "test1p5_exact_annuli_lower_b_v12")
TEST2_OUTPUT_PREFIX = str(RUN_DIRECTORY / "test2_exact_annuli_lower_b_reach_v12")
TEST2P5_OUTPUT_PREFIX = str(RUN_DIRECTORY / "test2p5_exact_annuli_refinement_v12")
FINAL_OUTPUT_PREFIX = str(RUN_DIRECTORY / "final_truncated_parameter_space_v1")

CURRENT_RUN_PARAMETERS = {
    "m_dm_kg": m_dm,
    "eps": eps,
    "temperature_K": T_dm,
    "target_ion_energy_J": float(USER_TARGET_ION_ENERGY_J),
    "detection_energy_policy": str(USER_DETECTION_ENERGY_POLICY),
    "require_fourier_mechanical_agreement": bool(
        USER_REQUIRE_FOURIER_MECHANICAL_AGREEMENT
    ),
    "fourier_mechanical_rel_tolerance": float(
        USER_FOURIER_MECHANICAL_REL_TOL
    ),
    "fourier_steps_per_secular_period": int(
        USER_FOURIER_STEPS_PER_SECULAR_PERIOD
    ),
    "fourier_detection_history": "incoming_R_outer_to_outgoing_R_outer",
    "fourier_outer_radius_m": float(USER_FOURIER_OUTER_RADIUS_M),
    "require_outer_R_tails_complete": bool(USER_REQUIRE_OUTER_R_TAILS_COMPLETE),
    "speed_refinement_target": str(USER_SPEED_REFINEMENT_TARGET),
    "adaptive_Rmax_enabled": bool(USER_RUN_ADAPTIVE_RMAX_PREPASS),
    "adaptive_Rmax_ladder_m": [float(value) for value in USER_ADAPTIVE_RMAX_LADDER_M],
    "escape_channel_scan_Rmax_m": float(USER_ESCAPE_SCAN_RMAX_M),
    "escape_focus_materialized": bool(USER_MATERIALIZE_ESCAPE_FOCUS_ROWS),
    "dynamic_bowl_audit": bool(USER_RUN_DYNAMIC_BOWL_AUDIT),
    "targeted_escape_refinement": bool(USER_ENABLE_TARGETED_ESCAPE_REFINEMENT),
    "targeted_escape_channels": list(USER_TARGETED_ESCAPE_CHANNELS),
    "targeted_escape_speed_window_m_s": [float(v) for v in USER_TARGETED_ESCAPE_SPEED_WINDOW_M_S],
    "targeted_escape_b_seed_m": float(USER_TARGETED_ESCAPE_B_SEED_M),
    "run_tag": RUN_TAG,
}
(RUN_DIRECTORY / "run_parameters.json").write_text(
    json.dumps(CURRENT_RUN_PARAMETERS, indent=2),
    encoding="utf-8",
)


def validate_parameter_dataframe(frame, label: str) -> None:
    """Reject an upstream table that belongs to another parameter point."""
    if frame is None or len(frame) == 0:
        return
    mass_column = "m_dm_kg" if "m_dm_kg" in frame.columns else (
        "m_dm" if "m_dm" in frame.columns else None
    )
    if mass_column is not None:
        masses = np.asarray(frame[mass_column], dtype=float)
        masses = masses[np.isfinite(masses)]
        if masses.size and not np.allclose(masses, m_dm, rtol=1.0e-12, atol=0.0):
            raise ValueError(
                f"{label} contains a mass different from USER_M_DM_KG={m_dm:.12e}"
            )
    if "eps" in frame.columns:
        charges = np.asarray(frame["eps"], dtype=float)
        charges = charges[np.isfinite(charges)]
        if charges.size and not np.allclose(charges, eps, rtol=1.0e-12, atol=0.0):
            raise ValueError(
                f"{label} contains eps different from USER_EPS={eps:.12e}"
            )


def _finite_unique(values, *, rtol: float = 1.0e-9) -> list[float]:
    cleaned = sorted(float(v) for v in values if np.isfinite(v) and float(v) > 0.0)
    unique: list[float] = []
    for value in cleaned:
        if not unique or not math.isclose(value, unique[-1], rel_tol=rtol, abs_tol=0.0):
            unique.append(value)
    return unique


def _test0_bool_array(values) -> np.ndarray:
    """Convert Test 0 decision columns to real booleans, including CSV strings."""
    series = np.asarray(values)
    if np.issubdtype(series.dtype, np.bool_):
        return series.astype(bool)
    true_tokens = {"true", "1", "yes", "y", "t"}
    false_tokens = {"false", "0", "no", "n", "f", "", "nan", "none"}
    output = []
    for value in series:
        if isinstance(value, (bool, np.bool_)):
            output.append(bool(value))
            continue
        text = str(value).strip().lower()
        if text in true_tokens:
            output.append(True)
        elif text in false_tokens:
            output.append(False)
        else:
            raise ValueError(f"Unrecognized Test 0 reject flag: {value!r}")
    return np.asarray(output, dtype=bool)


def resolve_test0_rejected_interval() -> tuple[float, float, str]:
    """Resolve the current pair's rejected speed interval inside the retained tail.

    The old policy required two roots and therefore failed when a rejected band
    reached the numerical Maxwell-tail cutoff. This version reconstructs the
    reject/keep topology from every continuous Test 0 root plus one observed
    decision state. It supports:

      * one bounded rejected interval;
      * a low-speed rejected interval ending at one root;
      * a high-speed rejected interval beginning at one root;
      * no rejected speeds in the retained numerical range.

    The downstream Test 0.75/Test 1 representation still supports at most one
    rejected interval. Multiple disjoint rejected bands raise a clear error.
    """
    critical = globals().get("TEST0_FAST_SPHERE_CRITICAL_DF")
    scan = globals().get("TEST0_QUANTILE_SCAN_DF")
    diagnostic = globals().get("TEST0_FAST_SPHERE_RESULTS_DF")
    transitions = globals().get("TEST0_DECISION_CHANGE_BRACKETS_DF")

    ion_mass = float(globals()["m_ion"])
    mu = m_dm * ion_mass / (m_dm + ion_mass)
    v_kin = math.sqrt(
        float(USER_TARGET_ION_ENERGY_J) * ion_mass / (2.0 * mu**2)
    )
    mb_scale_value = math.sqrt(k_B * T_dm / m_dm)
    v_max = float(
        _maxwell.ppf(
            1.0 - float(USER_SPEED_UPPER_TAIL_PROBABILITY),
            scale=mb_scale_value,
        )
    )
    if not np.isfinite(v_max) or v_max <= v_kin:
        raise RuntimeError(
            "The configured Maxwell-tail cutoff does not exceed the kinematic "
            "minimum speed."
        )

    roots: list[float] = []
    if critical is not None and len(critical) and "speed_m_s" in critical.columns:
        roots = _finite_unique(np.asarray(critical["speed_m_s"], dtype=float))
        roots = [value for value in roots if v_kin < value < v_max]

    # The quantile transition table is a fallback when the continuous critical
    # scan found no usable roots. Do not combine both lists because they describe
    # the same transitions at different resolutions.
    boundary_source = "current Test 0 continuous critical roots"
    boundaries = list(roots)
    if not boundaries and transitions is not None and len(transitions):
        if "midpoint_speed_estimate_m_s" in transitions.columns:
            boundaries = _finite_unique(
                np.asarray(
                    transitions["midpoint_speed_estimate_m_s"],
                    dtype=float,
                )
            )
            boundaries = [value for value in boundaries if v_kin < value < v_max]
            boundary_source = "current Test 0 quantile transition midpoints"

    # Obtain one known decision state. Combined with the ordered roots, parity
    # determines the decision in every speed segment.
    probe_speed = np.nan
    probe_reject = None
    for table in (scan, diagnostic):
        if (
            table is None
            or len(table) == 0
            or not {"speed_m_s", "reject_speed"}.issubset(table.columns)
        ):
            continue
        observed = table[["speed_m_s", "reject_speed"]].copy()
        observed["speed_m_s"] = pd.to_numeric(
            observed["speed_m_s"], errors="coerce"
        )
        observed = observed.loc[
            np.isfinite(observed["speed_m_s"])
            & (observed["speed_m_s"] >= v_kin)
            & (observed["speed_m_s"] <= v_max)
        ].sort_values("speed_m_s")
        if len(observed):
            flags = _test0_bool_array(observed["reject_speed"].to_numpy())
            # Use a central observed point so endpoint numerical noise does not
            # determine the topology.
            index = len(observed) // 2
            probe_speed = float(observed.iloc[index]["speed_m_s"])
            probe_reject = bool(flags[index])
            break

    if probe_reject is None:
        # Two roots with no sampled state retain the legacy bounded-band
        # interpretation. Any other root count is genuinely ambiguous.
        if len(boundaries) == 2:
            return (
                float(boundaries[0]),
                float(boundaries[1]),
                boundary_source + " (bounded-band fallback)",
            )
        raise RuntimeError(
            "Test 0 produced no usable reject/keep decision row, so the speed "
            "topology cannot be inferred."
        )

    boundaries = sorted(float(value) for value in boundaries)
    roots_below_probe = sum(value < probe_speed for value in boundaries)
    state_reject = bool(probe_reject) ^ bool(roots_below_probe % 2)

    edges = [float(v_kin), *boundaries, float(v_max)]
    rejected: list[tuple[float, float]] = []
    for left, right in zip(edges[:-1], edges[1:]):
        if state_reject and right > left:
            rejected.append((float(left), float(right)))
        state_reject = not state_reject

    if len(rejected) == 0:
        # Sentinel: the rejected interval starts at the numerical upper cutoff
        # and extends beyond it, so bounded_allowed_speed_intervals returns the
        # entire retained [v_kin, v_max] range as one low_allowed interval.
        return (
            float(v_max),
            float("inf"),
            "current Test 0 found no rejected speeds inside the retained numerical tail",
        )

    if len(rejected) > 1:
        detail = ", ".join(f"({lo:.9g}, {hi:.9g})" for lo, hi in rejected)
        raise RuntimeError(
            "The current Test 0 scan contains multiple disjoint rejected speed "
            f"bands: {detail}. Test 0.75 must be generalized before continuing."
        )

    lower, upper = rejected[0]
    if not upper > lower > 0.0:
        raise RuntimeError("Test 0 produced an invalid rejected speed interval")

    if math.isclose(lower, v_kin, rel_tol=1.0e-10, abs_tol=0.0):
        topology = "low-speed rejected band ending at one transition"
    elif math.isclose(upper, v_max, rel_tol=1.0e-10, abs_tol=0.0):
        topology = "high-speed rejected band reaching the numerical tail cutoff"
    else:
        topology = "bounded rejected band"

    return float(lower), float(upper), f"{boundary_source}; {topology}"


print("Current trajectory parameter point")
print(f"  m_dm = {m_dm:.12e} kg")
print(f"  eps  = {eps:.12e}")
print(f"  T_dm = {T_dm:.6g} K")
print(f"  output directory = {RUN_DIRECTORY.resolve()}")


# =============================================================================
# CONNECTED EQUIPOTENTIAL BOWL SETTINGS (v15)
# =============================================================================

# The bowl is the origin-connected trap-only sublevel region below the selected
# escape barrier.  The boundary is extracted from one connected equipotential,
# not from unrelated directional local maxima.
USER_RUN_BOWL_GEOMETRY = True
USER_BOWL_CENTER_M = (0.0, 0.0, 0.0)
USER_BOWL_REFERENCE_POTENTIAL_J = 0.0
USER_BOWL_PRINT_ESCAPE_DETAILS = True

# By default the lower of the +z and -z saddle candidates defines the escape
# energy, matching the expected surface-trap escape point.  Set
# "minimum_axes" to compare and select among +/-x, +/-y, and +/-z candidates,
# or "explicit" together with USER_BOWL_ESCAPE_LEVEL_J.
USER_BOWL_ESCAPE_POLICY = "z_axis"
USER_BOWL_ESCAPE_LEVEL_J = None

# One-dimensional candidate scans. x is the weak DC-confined axial direction;
# y and z are predominantly RF-pseudopotential-confined directions.
USER_BOWL_AXIS_SCAN_MIN_M = 0.25e-6
USER_BOWL_AXIS_SCAN_MAX_X_M = 1.5e-3
USER_BOWL_AXIS_SCAN_MAX_Y_M = 0.5e-3
USER_BOWL_AXIS_SCAN_MAX_Z_M = 0.5e-3
USER_BOWL_AXIS_SCAN_POINTS = 1800
USER_BOWL_PEAK_PROMINENCE_FRACTION = 1.0e-6
USER_BOWL_PEAK_PROMINENCE_J = 0.0

# A level slightly below the exact saddle is used for a numerically closed
# near-barrier surface.  Trajectory entry is classified against this connected
# component. Decrease this fraction after grid convergence checks.
# Request the closest-to-saddle level first. The bowl code automatically
# moves farther below the saddle until the origin-connected component is
# closed and does not touch the grid boundary.
USER_BOWL_NEAR_BARRIER_FRACTION = 1.0e-4
USER_BOWL_NEAR_BARRIER_OFFSET_J = 0.0
USER_BOWL_CLOSURE_FRACTION_LADDER = (
    1.0e-4, 5.0e-4, 1.0e-3, 2.0e-3, 5.0e-3,
    1.0e-2, 2.0e-2, 5.0e-2, 1.0e-1,
)
USER_BOWL_REQUIRE_CLOSED_COMPONENT = True
USER_BOWL_COMPONENT_CONNECTIVITY = 1  # six-neighbour; avoids diagonal leaks

# Anisotropic grid: wide along the weak DC axial x direction and finer/shorter
# in the RF-confined y/z directions. The RF pseudopotential is evaluated once
# in y,z and broadcast along x after an explicit x-invariance check.
USER_BOWL_X_REL_BOUNDS_M = (-800e-6, 800e-6)
USER_BOWL_Y_REL_BOUNDS_M = (-220e-6, 220e-6)
USER_BOWL_Z_REL_BOUNDS_M = (-65e-6, 300e-6)
USER_BOWL_NX = 141
USER_BOWL_NY = 91
USER_BOWL_NZ = 111
USER_BOWL_GRID_CHUNK_POINTS = 120000
USER_BOWL_USE_RF_X_INVARIANCE = True
USER_BOWL_RF_X_CHECK_POINTS = 9
USER_BOWL_RF_X_REL_TOL = 1.0e-10
USER_BOWL_RF_X_ABS_TOL_J = 1.0e-35

# Curvature/entry diagnostics and trajectory residence-time resolution.
USER_BOWL_HESSIAN_STEP_M = 0.25e-6
USER_BOWL_ENTRY_MIXED_FRACTION = 0.85
USER_BOWL_INSIDE_TOLERANCE_M = 1.0e-9
USER_BOWL_TIME_SAMPLE_DT_S = 2.0e-8
USER_BOWL_MAX_TIME_SAMPLES = 200000

# Optional analytical escape-point comparison.
USER_ANALYTIC_ESCAPE_POSITION_M = None
USER_ANALYTIC_ESCAPE_POTENTIAL_J = None

USER_BOWL_SHOW_PLOTS = True
USER_BOWL_SAVE_PLOTS = False
USER_BOWL_SURFACE_DOWNSAMPLE = 1

# Test the proposed necessary condition after the pilot and after Test 2.5.
USER_RUN_BOWL_NECESSITY_REPORT = True
USER_BOWL_REPORT_SHOW_PLOTS = True
USER_BOWL_REPORT_SAVE_PLOTS = False
USER_BOWL_REPORT_INPUT_CSV = None



In [ ]:
import constants as c
import numpy as np
import math
import matplotlib.pyplot as plt
import time
import numpy as np
import pandas as pd
import csv
from scipy.stats import maxwell
from scipy.optimize import curve_fit
from scipy.integrate import solve_ivp
from concurrent.futures import ProcessPoolExecutor, as_completed

vk = c.vk
vrf = c.vrf
z_offset = c.z0
z_min_valid = -c.ion_height
z_margin = 1.0e-6   # optional safety margin away from electrode plane
xy1k = c.xy1k
xy2k = c.xy2k
pi = math.pi
um = c.um
cm = c.cm
mm = c.mm
K = c.K
e = c.e
alpha = 1/137.06
d_chain = 3.7066438742e-06  # 3.7 µm separation between ions
delta_ion_2 = 0#-4.9903032463e-14 # small displacement of ion 2
m_ion = c.m 
Z_ion = c.Z
freq_x = 719430.7131391969
freq_y = 3031200.0099101723
freq_z = 3002153.5607483205
omega_vec = np.asarray(2*pi*np.array([freq_x,freq_y,freq_z]), dtype=float) 
hbar = 1.0545718e-34 
mode_y = 2*pi*np.array([3031200.01,2944587.06,2818861.50])
mode_z = 2*pi*np.array([3002153.56,2914677.59,2787603.39])

# Potential/Force Computation
## Compute DC Potential


In [ ]:
## From dc_potential.ipynb
## Functions
# (x,y,z) = coordinate of sample
# (xik,yik,z0) = ith corner coordinates of kth electrode
# vk = voltage applied to kth electrode
def potential_term(x,y,z,xik,yik,z0):
    num = (xik-x)*(yik-y); # numerator
    den = (z-z0)*np.sqrt((z-z0)**2+(xik-x)**2+(yik-y)**2); # denominator
    return num/den

def dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,z0,vk):
    term1 = np.arctan(potential_term(x,y,z,x2k,y2k,z0))
    term2 = np.arctan(potential_term(x,y,z,x1k,y2k,z0))
    term3 = np.arctan(potential_term(x,y,z,x2k,y1k,z0))
    term4 = np.arctan(potential_term(x,y,z,x1k,y1k,z0))
    return (vk/(2*math.pi))*(term1-term2-term3+term4)

def dc_potential_total(x,y,z,xy1k=c.xy1k,xy2k=c.xy2k,z0=c.z0,vk=c.vk):
    # extract individual (xik,yik),vk values from array
    # all arrays must be same size
    # sum potentials from all electrodes
    # Allow x, y, and z to be either scalars or same-shaped arrays.
    x, y, z = np.broadcast_arrays(
        np.asarray(x, dtype=float),
        np.asarray(y, dtype=float),
        np.asarray(z, dtype=float),
    )
    z = z+c.ion_height # account for ion height
    pot = np.zeros_like(x, dtype=float)
    for i in range (0,len(c.vk)):
        x1k = xy1k[i][0]
        y1k = xy1k[i][1]
        x2k = xy2k[i][0]
        y2k = xy2k[i][1]
        v = vk[i]
        if(i==19 or i==39): # make sure Q39-40 are on the M4 layer
            new_pot = dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,0,v); # z0=0 for Q39-40
        else:
            new_pot = dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,z0,v); # z0 = -11.8um for Q1-38
        pot+=new_pot
    return pot

## Compute RF Potential


In [ ]:
## From rf_potential.ipynb
## Functions
## Compute Pseudopotential (J)
"""
# Method 1: numerical RF gradient
Ex,Ey,Ez = np.gradient(-1*phi_rf,res,res,res); # Electric field components
E_squared = Ex**2+Ey**2+Ez**2; # Electric field strength squared
pseudo2 = (c.Z**2*c.e**2)/(4*c.m*c.omega**2)*E_squared
# pseudo_reshaped = pseudo.reshape(int(2*x_max/res+1),-1)
"""
# Method 2: analytical RF gradient
def divatan(up,down):
    #This is d(arctan2(up,down))/ddown up to a minus sign. It's useful for the pseudo-potential
    return up/(up**2+down**2)
def divatanup(up,down):
    #This is d(divatan(up,down))/dup
    return (down**2-up**2)/(up**2+down**2)**2
def divatandown(up,down):
    #This is d(divatan(up,down))/ddown
    return -2*up*down/(up**2+down**2)**2
def pseudopotential(x,y,z,m,Z,q=c.e,omrf=c.omega,VRF=c.vrf,ymin=c.y11,yedge1=c.y21,yedge2=c.y12,ymax=c.y22):
    #this is the pseudo potential, which is Z^2*(Div[PhiRF]/cos(om*t))^2/(4m*omega^2)
    x, y, z = np.broadcast_arrays(
        np.asarray(x, dtype=float),
        np.asarray(y, dtype=float),
        np.asarray(z, dtype=float),
    )
    z = z+c.ion_height # account for ion height
    divypart=divatan(z,yedge2-y)-divatan(z,yedge1-y)+divatan(z,ymin-y)-divatan(z,ymax-y)
    divzpart=divatan(yedge2-y,z)-divatan(yedge1-y,z)+divatan(ymin-y,z)-divatan(ymax-y,z)
    return ((Z**2*q**2)/(4*m*omrf**2)) * (VRF/math.pi)**2 * (divypart**2+divzpart**2)

In [ ]:
## Compute Full Potential Energy
def coulomb_pot(x,y,z,eps,Z=c.Z):
    x, y, z = np.broadcast_arrays(
        np.asarray(x, dtype=float),
        np.asarray(y, dtype=float),
        np.asarray(z, dtype=float),
    )
    return c.K*eps*Z*c.e**2/np.sqrt(x**2+y**2+z**2)
    
def potential_energy(xyz, m_dm, eps, component="total"):
    """
    Compute potential energy for either one point or many points.
    Accepted input shapes
    ---------------------
    xyz.shape == (3,)
        Returns a scalar for a single component.
    xyz.shape == (n, 3)
        Returns an array of shape (n,) for a single component.
    """
    xyz = np.asarray(xyz, dtype=float)
    single_input = False
    if xyz.ndim == 1:
        if xyz.shape != (3,):
            raise ValueError(
                "A single position must have shape (3,)"
            )
        xyz = xyz.reshape(1, 3)
        single_input = True
    elif xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError(
            "xyz must have shape (3,) or (n, 3)"
        )
    x = xyz[:, 0]
    y = xyz[:, 1]
    z = xyz[:, 2]
    dc = eps * c.e * dc_potential_total(x, y, z)
    pseudo_rf = pseudopotential(
        x,
        y,
        z,
        m_dm,
        eps,
    )
    coulomb = coulomb_pot(
        x,
        y,
        z,
        eps,
    )
    if component == "dc":
        out = dc
    elif component == "pseudo_rf":
        out = pseudo_rf
    elif component == "coulomb":
        out = coulomb
    elif component == "trap":
        out = dc + pseudo_rf
    elif component == "total":
        out = dc + pseudo_rf + coulomb
    elif component == "all":
        if single_input:
            return (
                float(dc[0]),
                float(pseudo_rf[0]),
                float(coulomb[0]),
            )
        return dc, pseudo_rf, coulomb
    else:
        raise ValueError(
            f"Unknown potential component: {component}"
        )
    if single_input:
        return float(out[0])
    return np.asarray(out, dtype=float)

## Compute RF/DC Force


In [ ]:
"""
xyz : ndarray, shape (n, 3)
        Particle positions.
"""
# RF pseudo-force
def FRF(xyz,m,Z,q=c.e,omrf=c.omega,VRF=c.vrf,ymin=c.y11,yedge1=c.y21,yedge2=c.y12,ymax=c.y22):
    xyz = np.asarray(xyz, dtype=float)
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("xyz must have shape (n, 3)")
    x = xyz[:, 0]
    y = xyz[:, 1]
    z = xyz[:, 2] + c.ion_height # Take into account ion height
    m = float(np.asarray(m).ravel()[0])
    Z = float(np.asarray(Z).ravel()[0])
    q = float(np.asarray(q).ravel()[0])
    omrf = float(np.asarray(omrf).ravel()[0])
    VRF = float(np.asarray(VRF).ravel()[0])
    #this is the pseudo potential, which is Z^2*(Div[PhiRF]/cos(om*t))^2/(4m*omega^2)
    divypart=divatan(z,yedge2-y)-divatan(z,yedge1-y)+divatan(z,ymin-y)-divatan(z,ymax-y)
    divzpart=divatan(yedge2-y,z)-divatan(yedge1-y,z)+divatan(ymin-y,z)-divatan(ymax-y,z) 
    divyparty=-2*divypart*(divatandown(z,yedge2-y)-divatandown(z,yedge1-y)+divatandown(z,ymin-y)-divatandown(z,ymax-y)) 
    divypartz=2*divypart*(divatanup(z,yedge2-y)-divatanup(z,yedge1-y)+divatanup(z,ymin-y)-divatanup(z,ymax-y)) 
    divzparty=-2*divzpart*(divatanup(yedge2-y,z)-divatanup(yedge1-y,z)+divatanup(ymin-y,z)-divatanup(ymax-y,z)) 
    divzpartz=2*divzpart*(divatandown(yedge2-y,z)-divatandown(yedge1-y,z)+divatandown(ymin-y,z)-divatandown(ymax-y,z)) 
    return -1*((VRF*Z*q)/(2*pi*np.sqrt(m)*omrf))**2*np.column_stack([divypartz*0, divzparty+divyparty, divzpartz+divypartz])

# DC force
def divatan(up,down):
    #This is d(arctan2(up,down))/ddown up to a minus sign. It's useful for the pseudo-potential
    return up/(up**2+down**2)
def divatanup(up,down):
    #This is d(divatan(up,down))/dup
    return (down**2-up**2)/(up**2+down**2)**2
def divatandown(up,down):
    #This is d(divatan(up,down))/ddown
    return -2*up*down/(up**2+down**2)**2

def anatangrad(xi,yi,xyz,v): # gradient term of DC potential
    xyz = np.asarray(xyz, dtype=float)
    x = xyz[:, 0]
    y = xyz[:, 1]
    z = xyz[:, 2]
    dy=y-yi
    dx=x-xi
    r = np.sqrt(dx**2+dy**2+z**2); # added distance
    dry2=z**2+dy**2
    drx2=z**2+dx**2
    divy=z*dx/(r*dry2); # divide by factor r
    divz=-dy*dx*(1/dry2+1/drx2)/r; # divide by factor r
    divx=z*dy/(r*drx2); # divide by factor r
    return (v/(2*np.pi))*np.column_stack([divx, divy, divz])

def FDC_single(x1,y1,x2,y2,xyz,v,Z,q=c.e):
    return -Z*q * (anatangrad(x2,y2,xyz,v)-anatangrad(x2,y1,xyz,v)-anatangrad(x1,y2,xyz,v)+anatangrad(x1,y1,xyz,v))

def FDC(xyz, m, Z, q=c.e, omrf=c.omega,
        ymin=c.y11, yedge1=c.y21, yedge2=c.y12, ymax=c.y22):
    xyz = np.asarray(xyz, dtype=float)
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("xyz must have shape (n, 3)")
    z_original = xyz[:, 2].copy() + c.ion_height # Take into account ion height
    force = np.zeros_like(xyz)
    for k in range(0, 40):
        xyz_k = xyz.copy()
        if (k != 19 and k != 39):
            xyz_k[:, 2] = z_original - z_offset  # keep scalar math
        else:
            xyz_k[:, 2] = z_original
        (x1k, y1k) = xy1k[k]
        (x2k, y2k) = xy2k[k]
        # enforce scalar on vk[k]
        v_k = float(np.asarray(vk[k]).ravel()[0])
        force_single = FDC_single(x1k, y1k, x2k, y2k, xyz_k, v_k, Z, q=q)
        # check shape of force_single 
        force_single = np.asarray(force_single, dtype=float)
        if force_single.shape != xyz.shape:
            raise ValueError(
                f"FDC_single returned unexpected shape {force_single.shape}; "
                f"expected {xyz.shape} at k={k}"
            )
        force += force_single
    return force

def force(xyz, m_dm, eps, component="total"):
    xyz = np.asarray(xyz, dtype=float)

    single_input = False
    if xyz.ndim == 1:
        xyz = xyz.reshape(1, 3)
        single_input = True

    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("xyz must have shape (n, 3) or (3,)")

    dc_force = FDC(xyz, m_dm, eps)
    pseudo_force = FRF(xyz, m_dm, eps)

    r = np.linalg.norm(xyz, axis=1)

    coulomb_force = np.full_like(xyz, np.nan, dtype=float)
    good = r > 0

    coulomb_force[good] = (
        c.K * c.Z * eps * c.e**2
        * xyz[good]
        / r[good, None]**3
    )

    if component == "dc":
        out = dc_force
    elif component == "pseudo_rf":
        out = pseudo_force
    elif component == "coulomb":
        out = coulomb_force
    elif component == "trap":
        out = dc_force + pseudo_force
    elif component == "total":
        out = dc_force + pseudo_force + coulomb_force
    elif component == "all":
        return dc_force, pseudo_force, coulomb_force
    else:
        raise ValueError(f"Unknown force component: {component}")

    if single_input:
        return out[0]

    return out

In [ ]:
def xyz_on_sphere(n, radius, center=(0.0, 0.0, 0.0)):
    """
    Generate positions evenly distributed on the surface of a sphere.

    Parameters
    ----------
    n_dm : int
        Number of coordinate points.
    radius : float
        Sphere radius in meters.
    center : array-like, shape (3,)
        Sphere center in meters.
    
    Returns
    -------
    x0_dms : ndarray, shape (n_dm, 3)
        Positions on the sphere surface.
    """

    center = np.asarray(center, dtype=float)
    if n <= 0:
        raise ValueError("n_dm must be positive")
    indices = np.arange(n)
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))
    z = 1.0 - 2.0 * (indices + 0.5) / n
    r_xy = np.sqrt(1.0 - z**2)
    theta = golden_angle * indices
    x = r_xy * np.cos(theta)
    y = r_xy * np.sin(theta)
    points = np.column_stack([x, y, z])
    x0_dms = center + radius * points
    return x0_dms

def valid_xyz_mask(xyz):
    xyz = np.asarray(xyz, dtype=float)
    return xyz[..., 2] > (z_min_valid + z_margin)

def xyz_on_valid_sphere(n, radius, center=(0.0, 0.0, 0.0)):
    xyz = xyz_on_sphere(n, radius, center=center)
    mask = valid_xyz_mask(xyz)
    return xyz[mask]

def valid_fraction_on_sphere(n, radius):
    xyz = xyz_on_sphere(n, radius)
    return np.mean(valid_xyz_mask(xyz))

def PE_on_sphere(
    n,
    radius,
    m_dm,
    eps,
    component="total",
):
    xyz = xyz_on_valid_sphere(n, radius)
    if xyz.shape[0] == 0:
        return np.array([], dtype=float)
    return np.asarray(
        potential_energy(
            xyz,
            m_dm,
            eps,
            component=component,
        ),
        dtype=float,
    )

def force_on_sphere(
    n,
    radius,
    m_dm,
    eps,
    component="total",
):
    xyz = xyz_on_valid_sphere(n, radius)
    if xyz.shape[0] == 0:
        return np.empty((0, 3), dtype=float)
    return np.asarray(
        force(
            xyz,
            m_dm,
            eps,
            component=component,
        ),
        dtype=float,
    )

def deterministic_MB_speeds(T, m_dm, quantile):
    k_B = 1.380649e-23
    scale = np.sqrt(k_B * T / m_dm)
    speeds = maxwell.ppf(quantile, scale=scale)
    return speeds

# Check Functionality
## Proper Vectorization


In [ ]:
test_xyz = np.array([
    [100.0e-6, 0.0, 100.0e-6],
    [200.0e-6, 0.0, 100.0e-6],
    [300.0e-6, 0.0, 100.0e-6],
])

U_vectorized = potential_energy(
    test_xyz,
    m_dm,
    eps,
    component="trap",
)

U_scalar = np.array([
    potential_energy(
        point,
        m_dm,
        eps,
        component="trap",
    )
    for point in test_xyz
])

print("Vectorized:", U_vectorized)
print("Scalar:    ", U_scalar)
print(
    "Match:",
    np.allclose(
        U_vectorized,
        U_scalar,
        equal_nan=True,
    ),
)

## Consistency of Potential vs Force


In [ ]:
# Uses m_dm and eps from the top USER SETTINGS cell.

def numerical_force_from_potential(xyz, m_dm, eps, component, h=0.05e-6):
    xyz = np.asarray(xyz, dtype=float)

    if xyz.shape != (3,):
        raise ValueError("xyz must have shape (3,)")

    F = np.zeros(3)

    for k in range(3):
        step = np.zeros(3)
        step[k] = h

        xyz_plus = xyz + step
        xyz_minus = xyz - step

        # Avoid finite-difference points outside valid potential domain.
        if xyz_plus[2] <= z_min_valid + z_margin:
            F[k] = np.nan
            continue

        if xyz_minus[2] <= z_min_valid + z_margin:
            F[k] = np.nan
            continue

        U_plus = potential_energy(xyz_plus, m_dm, eps, component)
        U_minus = potential_energy(xyz_minus, m_dm, eps, component)

        F[k] = -(U_plus - U_minus) / (2.0 * h)

    return F


test_points = [
    np.array([0.0, 0.0, 1e-6]),
    np.array([0.0, 0.0, 10e-6]),
    np.array([10e-6, 0.0, 10e-6]),
    np.array([0.0, 10e-6, 10e-6]),
    np.array([0.0, 0.0, 100e-6]),
    np.array([100e-6, 0.0, 300e-6]),
]

for xyz in test_points:
    if xyz[2] <= z_min_valid + z_margin:
        continue

    print()
    print("xyz =", xyz)

    for component in ["dc", "pseudo_rf", "coulomb", "trap", "total"]:
        F_analytic = np.asarray(force(xyz, m_dm, eps, component), dtype=float).reshape(3)
        F_numeric = numerical_force_from_potential(xyz, m_dm, eps, component)

        diff = F_analytic - F_numeric

        denom = max(
            np.linalg.norm(F_analytic),
            np.linalg.norm(F_numeric),
            1e-300,
        )

        rel_err = np.linalg.norm(diff) / denom

        print(component)
        print("  analytic:", F_analytic)
        print("  numeric :", F_numeric)
        print("  diff    :", diff)
        print("  rel_err :", rel_err)

# Connected Equipotential Trap-Potential Bowl

This stage separates the trap-only dark-matter energy into

\[
U_{\rm trap}(x,y,z)=U_{\rm DC}(x,y,z)+U_{\rm pseudo,RF}(y,z).
\]

The RF pseudopotential is checked for numerical invariance along the axial
$ x $ direction, while the DC term is evaluated on the full anisotropic 3-D
grid. Candidate barriers are scanned separately along $\pm x$, $\pm y$, and
$\pm z$. By default, the lower $z$-axis saddle defines the escape energy.

The bowl is the connected sublevel component containing the ion equilibrium.
A level slightly below the exact saddle is used to extract a closed
near-barrier equipotential surface with marching cubes. The model records the
DC/RF energy and force decomposition at every trajectory's first inward bowl
crossing, as well as its entry channel and residence time.


In [ ]:
#!/usr/bin/env python3
"""Closed connected equipotential bowl model with explicit DC/RF decomposition.

Run this cell/script after the trap potential and force definitions.  The model
uses the trap-only dark-matter effective potential

    U_trap = U_DC + U_pseudo_RF,

where the present surface-trap model supplies axial x confinement through the
DC term and transverse y/z confinement primarily through the RF
pseudopotential.

The bowl is no longer constructed by joining unrelated radial local maxima.
Instead the code:

1. evaluates the +z/-z escape saddles and comparison candidates on +/-x,+/-y;
2. chooses an escape energy (z-axis by default);
3. evaluates U_DC(x,y,z) on an anisotropic 3-D grid;
4. evaluates U_pseudo_RF(y,z) once and broadcasts it along x after checking
   that the RF pseudopotential is x independent to numerical precision;
5. labels the connected sublevel component containing the ion equilibrium;
6. extracts a near-barrier connected equipotential surface with marching
   cubes;
7. exposes signed-distance/containment helpers for trajectory classification;
8. records bowl residence time and the entry channel/force decomposition.

The public API remains compatible with the bowl-aware Test 2 code:

    BOWL_MODEL
    bowl_metrics_from_samples
    bowl_metrics_from_dense_solution
    interval_metrics_from_signed_values
    combine_bowl_stage_metrics
    bowl_default_fields

The model is trap-only: the ion-DM Coulomb interaction is intentionally not
used to define the bowl.
"""

from __future__ import annotations

from dataclasses import dataclass, field
import hashlib
import json
import math
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import distance_transform_edt, generate_binary_structure, label
from scipy.optimize import minimize_scalar
from scipy.signal import find_peaks
from skimage.measure import marching_cubes


# =============================================================================
# SETTINGS
# =============================================================================

RUN_BOWL_GEOMETRY = bool(globals().get("USER_RUN_BOWL_GEOMETRY", True))
BOWL_CENTER_M = np.asarray(
    globals().get("USER_BOWL_CENTER_M", (0.0, 0.0, 0.0)), dtype=float
).reshape(3)
BOWL_REFERENCE_POTENTIAL_J = float(
    globals().get("USER_BOWL_REFERENCE_POTENTIAL_J", 0.0)
)
BOWL_INVALID_Z_FLOOR_M = globals().get("USER_BOWL_INVALID_Z_FLOOR_M", None)
BOWL_ESCAPE_POLICY = str(
    globals().get("USER_BOWL_ESCAPE_POLICY", "z_axis")
).strip().lower()
BOWL_EXPLICIT_ESCAPE_LEVEL_J = globals().get("USER_BOWL_ESCAPE_LEVEL_J", None)
BOWL_NEAR_BARRIER_FRACTION = float(
    globals().get("USER_BOWL_NEAR_BARRIER_FRACTION", 1.0e-4)
)
BOWL_NEAR_BARRIER_OFFSET_J = float(
    globals().get("USER_BOWL_NEAR_BARRIER_OFFSET_J", 0.0)
)
# The exact saddle-level sublevel set is open at the escape throat. On a
# finite grid, a level that is too close to the saddle numerically leaks from
# the inner basin into the exterior low-potential sea. Try progressively deeper
# levels and choose the closest one that is closed and does not touch the grid.
BOWL_CLOSURE_FRACTION_LADDER = tuple(
    float(value)
    for value in globals().get(
        "USER_BOWL_CLOSURE_FRACTION_LADDER",
        (1.0e-4, 5.0e-4, 1.0e-3, 2.0e-3, 5.0e-3, 1.0e-2, 2.0e-2, 5.0e-2, 1.0e-1),
    )
)
BOWL_REQUIRE_CLOSED_COMPONENT = bool(
    globals().get("USER_BOWL_REQUIRE_CLOSED_COMPONENT", True)
)
# Six-neighbour connectivity prevents a one-voxel diagonal bridge from joining
# the inner basin to the outside sea near the saddle.
BOWL_COMPONENT_CONNECTIVITY = int(
    globals().get("USER_BOWL_COMPONENT_CONNECTIVITY", 1)
)
if BOWL_COMPONENT_CONNECTIVITY not in (1, 2, 3):
    raise ValueError("USER_BOWL_COMPONENT_CONNECTIVITY must be 1, 2, or 3")

# 1-D saddle scans, relative to BOWL_CENTER_M.
BOWL_AXIS_SCAN_MIN_M = float(globals().get("USER_BOWL_AXIS_SCAN_MIN_M", 0.25e-6))
BOWL_AXIS_SCAN_MAX_X_M = float(globals().get("USER_BOWL_AXIS_SCAN_MAX_X_M", 1.5e-3))
BOWL_AXIS_SCAN_MAX_Y_M = float(globals().get("USER_BOWL_AXIS_SCAN_MAX_Y_M", 0.5e-3))
BOWL_AXIS_SCAN_MAX_Z_M = float(globals().get("USER_BOWL_AXIS_SCAN_MAX_Z_M", 0.5e-3))
BOWL_AXIS_SCAN_POINTS = int(globals().get("USER_BOWL_AXIS_SCAN_POINTS", 1800))
BOWL_PEAK_PROMINENCE_FRACTION = float(
    globals().get("USER_BOWL_PEAK_PROMINENCE_FRACTION", 1.0e-6)
)
BOWL_PEAK_PROMINENCE_J = float(globals().get("USER_BOWL_PEAK_PROMINENCE_J", 0.0))

# Anisotropic grid, in coordinates relative to BOWL_CENTER_M.
BOWL_X_REL_BOUNDS_M = tuple(
    float(v) for v in globals().get("USER_BOWL_X_REL_BOUNDS_M", (-800e-6, 800e-6))
)
BOWL_Y_REL_BOUNDS_M = tuple(
    float(v) for v in globals().get("USER_BOWL_Y_REL_BOUNDS_M", (-220e-6, 220e-6))
)
BOWL_Z_REL_BOUNDS_M = tuple(
    float(v) for v in globals().get("USER_BOWL_Z_REL_BOUNDS_M", (-65e-6, 300e-6))
)
BOWL_NX = int(globals().get("USER_BOWL_NX", 141))
BOWL_NY = int(globals().get("USER_BOWL_NY", 91))
BOWL_NZ = int(globals().get("USER_BOWL_NZ", 111))
BOWL_GRID_CHUNK_POINTS = int(globals().get("USER_BOWL_GRID_CHUNK_POINTS", 120000))
BOWL_USE_RF_X_INVARIANCE = bool(
    globals().get("USER_BOWL_USE_RF_X_INVARIANCE", True)
)
BOWL_RF_X_CHECK_POINTS = int(globals().get("USER_BOWL_RF_X_CHECK_POINTS", 9))
BOWL_RF_X_REL_TOL = float(globals().get("USER_BOWL_RF_X_REL_TOL", 1.0e-10))
BOWL_RF_X_ABS_TOL_J = float(globals().get("USER_BOWL_RF_X_ABS_TOL_J", 1.0e-35))

BOWL_INSIDE_TOLERANCE_M = float(
    globals().get("USER_BOWL_INSIDE_TOLERANCE_M", 1.0e-9)
)
BOWL_TIME_SAMPLE_DT_S = float(globals().get("USER_BOWL_TIME_SAMPLE_DT_S", 2.0e-8))
BOWL_MAX_TIME_SAMPLES = int(globals().get("USER_BOWL_MAX_TIME_SAMPLES", 200000))
BOWL_ENTRY_MIXED_FRACTION = float(
    globals().get("USER_BOWL_ENTRY_MIXED_FRACTION", 0.85)
)
BOWL_HESSIAN_STEP_M = float(globals().get("USER_BOWL_HESSIAN_STEP_M", 0.25e-6))
BOWL_SHOW_PLOTS = bool(globals().get("USER_BOWL_SHOW_PLOTS", True))
BOWL_SAVE_PLOTS = bool(globals().get("USER_BOWL_SAVE_PLOTS", False))
BOWL_SURFACE_DOWNSAMPLE = max(1, int(globals().get("USER_BOWL_SURFACE_DOWNSAMPLE", 1)))
BOWL_PRINT_ESCAPE_DETAILS = bool(globals().get("USER_BOWL_PRINT_ESCAPE_DETAILS", True))
BOWL_ANALYTIC_ESCAPE_POSITION_M = globals().get("USER_ANALYTIC_ESCAPE_POSITION_M", None)
BOWL_ANALYTIC_ESCAPE_POTENTIAL_J = globals().get("USER_ANALYTIC_ESCAPE_POTENTIAL_J", None)

if not np.all(np.isfinite(BOWL_CENTER_M)):
    raise ValueError("USER_BOWL_CENTER_M must be a finite length-3 vector")
if BOWL_AXIS_SCAN_POINTS < 100:
    raise ValueError("USER_BOWL_AXIS_SCAN_POINTS must be at least 100")
if min(BOWL_NX, BOWL_NY, BOWL_NZ) < 16:
    raise ValueError("The 3-D bowl grid is too coarse")
if BOWL_ESCAPE_POLICY not in {"z_axis", "minimum_axes", "explicit"}:
    raise ValueError("USER_BOWL_ESCAPE_POLICY must be z_axis, minimum_axes, or explicit")

RUN_DIRECTORY = Path(globals().get("RUN_DIRECTORY", Path.cwd()))
BOWL_OUTPUT_PREFIX = RUN_DIRECTORY / "bowl_connected_equipotential_dc_rf_v4"
BOWL_SUMMARY_CSV = Path(f"{BOWL_OUTPUT_PREFIX}_summary.csv")
BOWL_AXIS_CANDIDATES_CSV = Path(f"{BOWL_OUTPUT_PREFIX}_axis_candidates.csv")
BOWL_AXIS_PROFILES_CSV = Path(f"{BOWL_OUTPUT_PREFIX}_axis_profiles.csv")
BOWL_SURFACE_VERTICES_CSV = Path(f"{BOWL_OUTPUT_PREFIX}_surface_vertices.csv")
BOWL_MODEL_NPZ = Path(f"{BOWL_OUTPUT_PREFIX}_model.npz")
BOWL_SURFACE_PNG = Path(f"{BOWL_OUTPUT_PREFIX}_surface.png")
BOWL_AXIS_PNG = Path(f"{BOWL_OUTPUT_PREFIX}_axis_profiles.png")
BOWL_COMPONENT_PNG = Path(f"{BOWL_OUTPUT_PREFIX}_components.png")
BOWL_CLOSURE_SEARCH_CSV = Path(f"{BOWL_OUTPUT_PREFIX}_closure_search.csv")


# =============================================================================
# BASIC ADAPTERS
# =============================================================================


def _current_m_dm() -> float:
    for name in ("USER_M_DM_KG", "m_dm"):
        if name in globals():
            value = float(globals()[name])
            if np.isfinite(value) and value > 0.0:
                return value
    raise RuntimeError("No positive USER_M_DM_KG or m_dm is available")


def _current_eps() -> float:
    for name in ("USER_EPS", "eps"):
        if name in globals():
            value = float(globals()[name])
            if np.isfinite(value):
                return value
    raise RuntimeError("No finite USER_EPS or eps is available")


def _resolve_invalid_z_floor() -> float | None:
    if BOWL_INVALID_Z_FLOOR_M is not None:
        return float(BOWL_INVALID_Z_FLOOR_M)
    c_obj = globals().get("c")
    if c_obj is not None and hasattr(c_obj, "ion_height"):
        margin = float(globals().get("USER_INVALID_Z_MARGIN_M", 1.0e-6))
        return -float(c_obj.ion_height) + margin
    return None


def _potential_component(points_m: np.ndarray, component: str) -> np.ndarray:
    points = np.asarray(points_m, dtype=float).reshape(-1, 3)
    fn = globals().get("potential_energy")
    if fn is None:
        raise RuntimeError("potential_energy(...) is required")
    m_dm = _current_m_dm()
    eps = _current_eps()
    aliases = {
        "rf": ("pseudo_rf", "rf", "pseudopotential"),
        "dc": ("dc",),
        "trap": ("trap",),
    }
    errors: list[str] = []
    for alias in aliases.get(component, (component,)):
        try:
            with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
                values = np.asarray(
                    fn(points, m_dm, eps, component=alias), dtype=float
                ).reshape(-1)
            if values.size == len(points):
                return values
        except Exception as exc:
            errors.append(f"{alias}:{type(exc).__name__}")
    output = np.full(len(points), np.nan, dtype=float)
    for index, point in enumerate(points):
        for alias in aliases.get(component, (component,)):
            try:
                with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
                    output[index] = float(np.asarray(
                        fn(point, m_dm, eps, component=alias), dtype=float
                    ).reshape(-1)[0])
                break
            except Exception:
                continue
    if not np.any(np.isfinite(output)):
        raise RuntimeError(
            f"Could not evaluate potential component {component!r}; attempts={errors}"
        )
    return output


def _force_component(points_m: np.ndarray, component: str) -> np.ndarray:
    points = np.asarray(points_m, dtype=float).reshape(-1, 3)
    fn = globals().get("force")
    if fn is not None:
        aliases = {
            "rf": ("pseudo_rf", "rf", "pseudopotential"),
            "dc": ("dc",),
            "trap": ("trap",),
        }
        for alias in aliases.get(component, (component,)):
            try:
                values = np.asarray(
                    fn(points, _current_m_dm(), _current_eps(), component=alias),
                    dtype=float,
                ).reshape(-1, 3)
                if len(values) == len(points):
                    return values
            except Exception:
                pass
    if component == "dc" and "FDC" in globals():
        return np.asarray(
            globals()["FDC"](points, _current_m_dm(), _current_eps()), dtype=float
        ).reshape(-1, 3)
    if component == "rf" and "FRF" in globals():
        return np.asarray(
            globals()["FRF"](points, _current_m_dm(), _current_eps()), dtype=float
        ).reshape(-1, 3)
    if component == "trap":
        return _force_component(points, "dc") + _force_component(points, "rf")
    raise RuntimeError(f"Could not evaluate force component {component!r}")


def _valid_geometry(points_m: np.ndarray, invalid_z_floor_m: float | None) -> np.ndarray:
    points = np.asarray(points_m, dtype=float).reshape(-1, 3)
    valid = np.all(np.isfinite(points), axis=1)
    if invalid_z_floor_m is not None:
        valid &= points[:, 2] > invalid_z_floor_m
    return valid


def _eval_component_chunked(points: np.ndarray, component: str) -> np.ndarray:
    points = np.asarray(points, dtype=float).reshape(-1, 3)
    output = np.full(len(points), np.nan, dtype=float)
    for start in range(0, len(points), BOWL_GRID_CHUNK_POINTS):
        stop = min(start + BOWL_GRID_CHUNK_POINTS, len(points))
        output[start:stop] = _potential_component(points[start:stop], component)
    return output


def _trap_potential(points_m: np.ndarray) -> np.ndarray:
    """Trap-only DM energy, explicitly excluding the ion-DM Coulomb term."""
    points = np.asarray(points_m, dtype=float).reshape(-1, 3)
    return _potential_component(points, "dc") + _potential_component(points, "rf")


def _trap_force(points_m: np.ndarray) -> np.ndarray:
    """Trap-only force, explicitly excluding the ion-DM Coulomb term."""
    points = np.asarray(points_m, dtype=float).reshape(-1, 3)
    return _force_component(points, "dc") + _force_component(points, "rf")


# =============================================================================
# AXIS SADDLES AND CURVATURE
# =============================================================================


AXIS_DIRECTIONS = {
    "+x": np.array([1.0, 0.0, 0.0]),
    "-x": np.array([-1.0, 0.0, 0.0]),
    "+y": np.array([0.0, 1.0, 0.0]),
    "-y": np.array([0.0, -1.0, 0.0]),
    "+z": np.array([0.0, 0.0, 1.0]),
    "-z": np.array([0.0, 0.0, -1.0]),
}


def _axis_max_distance(label_name: str) -> float:
    axis = label_name[-1]
    return {"x": BOWL_AXIS_SCAN_MAX_X_M, "y": BOWL_AXIS_SCAN_MAX_Y_M, "z": BOWL_AXIS_SCAN_MAX_Z_M}[axis]


def _refine_axis_peak(direction: np.ndarray, bracket: tuple[float, float]) -> tuple[float, float]:
    def objective(radius: float) -> float:
        point = BOWL_CENTER_M + float(radius) * direction
        return -float(_trap_potential(point[None, :])[0])
    solution = minimize_scalar(
        objective, bounds=(float(bracket[0]), float(bracket[1])), method="bounded",
        options={"xatol": max(1.0e-12, 1.0e-7 * max(abs(bracket[1]), 1.0e-9))},
    )
    radius = float(solution.x)
    potential = float(-solution.fun)
    return radius, potential


def _scan_axis_candidate(label_name: str, invalid_z_floor_m: float | None) -> tuple[dict[str, Any], pd.DataFrame]:
    direction = AXIS_DIRECTIONS[label_name]
    max_distance = _axis_max_distance(label_name)
    radii = np.geomspace(BOWL_AXIS_SCAN_MIN_M, max_distance, BOWL_AXIS_SCAN_POINTS)
    radii = np.concatenate(([0.0], radii))
    points = BOWL_CENTER_M[None, :] + radii[:, None] * direction[None, :]
    valid = _valid_geometry(points, invalid_z_floor_m)
    u_dc = _potential_component(points, "dc")
    u_rf = _potential_component(points, "rf")
    u_trap = u_dc + u_rf
    u_trap[~valid] = np.nan
    finite = np.isfinite(u_trap)
    finite_u = u_trap[finite]
    prominence = BOWL_PEAK_PROMINENCE_J
    if finite_u.size:
        prominence = max(
            prominence,
            BOWL_PEAK_PROMINENCE_FRACTION * max(float(np.nanmax(finite_u) - np.nanmin(finite_u)), np.finfo(float).tiny),
        )
    indices = np.flatnonzero(finite)
    peak_indices: np.ndarray
    if len(indices) >= 5:
        local_peaks, properties = find_peaks(u_trap[indices], prominence=prominence)
        peak_indices = indices[local_peaks]
    else:
        properties = {}
        peak_indices = np.array([], dtype=int)

    center_u = float(_trap_potential(BOWL_CENTER_M[None, :])[0])
    accepted = [int(i) for i in peak_indices if np.isfinite(u_trap[i]) and u_trap[i] > center_u]
    if accepted:
        index = accepted[0]
        lo = max(index - 1, 0)
        hi = min(index + 1, len(radii) - 1)
        refined_r, refined_u = _refine_axis_peak(direction, (radii[lo], radii[hi]))
        position = BOWL_CENTER_M + refined_r * direction
        components = {
            "dc": float(_potential_component(position[None, :], "dc")[0]),
            "rf": float(_potential_component(position[None, :], "rf")[0]),
        }
        status = "first_local_maximum"
        resolved = True
    else:
        refined_r = np.nan
        refined_u = np.nan
        position = np.full(3, np.nan)
        components = {"dc": np.nan, "rf": np.nan}
        status = "no_local_maximum"
        resolved = False

    candidate = {
        "axis_label": label_name,
        "resolved": resolved,
        "status": status,
        "radius_m": refined_r,
        "x_m": position[0],
        "y_m": position[1],
        "z_m": position[2],
        "potential_total_J": refined_u,
        "potential_dc_J": components["dc"],
        "potential_rf_J": components["rf"],
        "barrier_from_center_J": refined_u - center_u if resolved else np.nan,
    }
    profile = pd.DataFrame({
        "axis_label": label_name,
        "radius_m": radii,
        "radius_um": radii * 1.0e6,
        "x_m": points[:, 0],
        "y_m": points[:, 1],
        "z_m": points[:, 2],
        "U_dc_J": u_dc,
        "U_rf_J": u_rf,
        "U_total_J": u_trap,
        "valid_geometry": valid,
    })
    return candidate, profile


def _finite_difference_hessian(point_m: np.ndarray, component: str) -> np.ndarray:
    """Return the Cartesian Hessian of one trap-potential component.

    All finite-difference samples are evaluated through
    ``_potential_component`` so the DC, RF, and total-trap conventions remain
    identical to those used elsewhere in the bowl calculation.
    """
    point = np.asarray(point_m, dtype=float).reshape(3)
    h = float(BOWL_HESSIAN_STEP_M)
    if not np.isfinite(h) or h <= 0.0:
        raise ValueError("BOWL_HESSIAN_STEP_M must be finite and positive")

    def evaluate(sample_point_m: np.ndarray) -> float:
        sample = np.asarray(sample_point_m, dtype=float).reshape(1, 3)
        value = float(_potential_component(sample, component)[0])
        if not np.isfinite(value):
            raise FloatingPointError(
                f"Non-finite {component!r} potential while evaluating Hessian "
                f"at {sample.reshape(3)}"
            )
        return value

    hessian = np.zeros((3, 3), dtype=float)
    f0 = evaluate(point)
    basis = np.eye(3)

    for i in range(3):
        fp = evaluate(point + h * basis[i])
        fm = evaluate(point - h * basis[i])
        hessian[i, i] = (fp - 2.0 * f0 + fm) / h**2

        for j in range(i + 1, 3):
            fpp = evaluate(point + h * basis[i] + h * basis[j])
            fpm = evaluate(point + h * basis[i] - h * basis[j])
            fmp = evaluate(point - h * basis[i] + h * basis[j])
            fmm = evaluate(point - h * basis[i] - h * basis[j])
            value = (fpp - fpm - fmp + fmm) / (4.0 * h**2)
            hessian[i, j] = hessian[j, i] = value

    return hessian


def _optional_vector(value: Any) -> np.ndarray | None:
    if value is None:
        return None
    try:
        vector = np.asarray(value, dtype=float).reshape(3)
    except Exception:
        return None
    return vector if np.all(np.isfinite(vector)) else None


def _optional_scalar(value: Any) -> float | None:
    if value is None:
        return None
    try:
        scalar = float(value)
    except Exception:
        return None
    return scalar if np.isfinite(scalar) else None


# =============================================================================
# CONNECTED SUBLEVEL AND EQUIPOTENTIAL SURFACE
# =============================================================================


def _axis_grid(bounds: tuple[float, float], count: int, center_coordinate: float) -> np.ndarray:
    if len(bounds) != 2 or not bounds[1] > bounds[0]:
        raise ValueError("Each USER_BOWL_*_REL_BOUNDS_M must be increasing")
    return center_coordinate + np.linspace(bounds[0], bounds[1], count)


def _nearest_index(axis: np.ndarray, value: float) -> int:
    return int(np.argmin(np.abs(np.asarray(axis) - float(value))))


def _component_touches_boundary(mask: np.ndarray) -> bool:
    return bool(
        np.any(mask[0]) or np.any(mask[-1]) or
        np.any(mask[:, 0]) or np.any(mask[:, -1]) or
        np.any(mask[:, :, 0]) or np.any(mask[:, :, -1])
    )


def _build_potential_grid(
    x: np.ndarray, y: np.ndarray, z: np.ndarray, invalid_z_floor_m: float | None
) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict[str, float]]:
    # DC must be evaluated throughout the 3-D domain.
    X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
    points = np.column_stack([X.ravel(), Y.ravel(), Z.ravel()])
    valid = _valid_geometry(points, invalid_z_floor_m).reshape(X.shape)
    u_dc = _eval_component_chunked(points, "dc").reshape(X.shape)

    diagnostics: dict[str, float] = {}
    if BOWL_USE_RF_X_INVARIANCE:
        Y2, Z2 = np.meshgrid(y, z, indexing="ij")
        yz_points = np.column_stack([
            np.full(Y2.size, BOWL_CENTER_M[0]), Y2.ravel(), Z2.ravel()
        ])
        u_rf_yz = _eval_component_chunked(yz_points, "rf").reshape(len(y), len(z))
        u_rf = np.broadcast_to(u_rf_yz[None, :, :], X.shape).copy()

        sample_x = np.linspace(x[0], x[-1], BOWL_RF_X_CHECK_POINTS)
        sample_yz = [
            (BOWL_CENTER_M[1], BOWL_CENTER_M[2]),
            (y[len(y)//4], z[len(z)//2]),
            (y[3*len(y)//4], z[3*len(z)//4]),
        ]
        variations = []
        scales = []
        for y0, z0 in sample_yz:
            check_points = np.column_stack([
                sample_x, np.full_like(sample_x, y0), np.full_like(sample_x, z0)
            ])
            values = _potential_component(check_points, "rf")
            variations.append(float(np.nanmax(values) - np.nanmin(values)))
            scales.append(float(np.nanmax(np.abs(values))))
        max_variation = max(variations) if variations else np.nan
        max_scale = max(scales) if scales else np.nan
        diagnostics["rf_x_max_absolute_variation_J"] = max_variation
        diagnostics["rf_x_max_relative_variation"] = (
            max_variation / max(max_scale, np.finfo(float).tiny)
            if np.isfinite(max_variation) and np.isfinite(max_scale) else np.nan
        )
        diagnostics["rf_x_invariant_within_tolerance"] = bool(
            max_variation <= max(BOWL_RF_X_ABS_TOL_J, BOWL_RF_X_REL_TOL * max(max_scale, 0.0))
        )
    else:
        u_rf = _eval_component_chunked(points, "rf").reshape(X.shape)
        diagnostics["rf_x_max_absolute_variation_J"] = np.nan
        diagnostics["rf_x_max_relative_variation"] = np.nan
        diagnostics["rf_x_invariant_within_tolerance"] = False

    u_dc[~valid] = np.nan
    u_rf[~valid] = np.nan
    u_total = u_dc + u_rf
    return u_dc, u_rf, u_total, diagnostics


@dataclass
class BowlModel:
    center_m: np.ndarray
    center_potential_total_J: float
    center_potential_dc_J: float
    center_potential_rf_J: float
    m_dm_kg: float
    eps: float
    reference_potential_J: float
    invalid_z_floor_m: float | None
    escape_policy: str
    escape_label: str
    escape_position_m: np.ndarray
    escape_potential_total_J: float
    escape_potential_dc_J: float
    escape_potential_rf_J: float
    classification_level_J: float
    near_barrier_offset_J: float
    near_barrier_fraction_selected: float
    component_connectivity: int
    axis_candidates: pd.DataFrame
    x_grid_m: np.ndarray
    y_grid_m: np.ndarray
    z_grid_m: np.ndarray
    component_mask: np.ndarray
    signed_distance_grid_m: np.ndarray
    surface_vertices_m: np.ndarray
    surface_faces: np.ndarray
    surface_vertex_potential_total_J: np.ndarray
    surface_vertex_potential_dc_J: np.ndarray
    surface_vertex_potential_rf_J: np.ndarray
    component_closed: bool
    component_touches_grid_boundary: bool
    component_voxel_count: int
    surface_is_watertight: bool
    rf_x_max_absolute_variation_J: float
    rf_x_max_relative_variation: float
    rf_x_invariant_within_tolerance: bool
    center_hessian_total_J_m2: np.ndarray
    center_hessian_dc_J_m2: np.ndarray
    center_hessian_rf_J_m2: np.ndarray
    escape_hessian_total_J_m2: np.ndarray
    escape_hessian_dc_J_m2: np.ndarray
    escape_hessian_rf_J_m2: np.ndarray
    escape_gradient_norm_N: float
    model_signature: str
    _signed_interpolator: Any = field(init=False, repr=False)

    def __post_init__(self) -> None:
        self._signed_interpolator = RegularGridInterpolator(
            (self.x_grid_m, self.y_grid_m, self.z_grid_m),
            self.signed_distance_grid_m,
            bounds_error=False,
            fill_value=np.nan,
        )

    def signed_distance_m(self, points_m: np.ndarray) -> np.ndarray:
        points = np.asarray(points_m, dtype=float).reshape(-1, 3)
        return np.asarray(self._signed_interpolator(points), dtype=float)

    def contains(self, points_m: np.ndarray, tolerance_m: float = BOWL_INSIDE_TOLERANCE_M) -> np.ndarray:
        signed = self.signed_distance_m(points_m)
        return np.isfinite(signed) & (signed <= float(tolerance_m))

    @property
    def center_potential_J(self) -> float:
        return self.center_potential_total_J

    @property
    def z_axis_min_potential_J(self) -> float:
        return self.escape_potential_total_J

    @property
    def z_axis_escape_potential_direct_J(self) -> float:
        return self.escape_potential_total_J

    @property
    def z_axis_escape_position_m(self) -> np.ndarray:
        return self.escape_position_m

    @property
    def z_axis_escape_label(self) -> str:
        return self.escape_label

    @property
    def minimum_required_kinetic_energy_J(self) -> float:
        return float(max(self.escape_potential_total_J - self.reference_potential_J, 0.0))

    @property
    def barrier_from_center_J(self) -> float:
        return float(self.escape_potential_total_J - self.center_potential_total_J)

    @property
    def minimum_speed_from_reference_m_s(self) -> float:
        return float(math.sqrt(2.0 * self.minimum_required_kinetic_energy_J / self.m_dm_kg))

    @property
    def minimum_speed_from_center_m_s(self) -> float:
        return float(math.sqrt(2.0 * max(self.barrier_from_center_J, 0.0) / self.m_dm_kg))

    @property
    def extents_relative_m(self) -> np.ndarray:
        relative = self.surface_vertices_m - self.center_m[None, :]
        return np.max(np.abs(relative), axis=0) if len(relative) else np.full(3, np.nan)

    def entry_channel(self, point_m: np.ndarray) -> str:
        point = np.asarray(point_m, dtype=float).reshape(3)
        extents = np.maximum(self.extents_relative_m, np.finfo(float).tiny)
        scaled = np.abs(point - self.center_m) / extents
        order = np.argsort(scaled)[::-1]
        labels = np.array(["axial_dc_x", "radial_rf_y", "vertical_rf_z"], dtype=object)
        if scaled[order[1]] >= BOWL_ENTRY_MIXED_FRACTION * scaled[order[0]]:
            return f"mixed_{labels[order[0]]}_{labels[order[1]]}"
        return str(labels[order[0]])


def build_bowl_model() -> BowlModel:
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
    m_dm = _current_m_dm()
    eps = _current_eps()
    invalid_floor = _resolve_invalid_z_floor()

    center_dc = float(_potential_component(BOWL_CENTER_M[None, :], "dc")[0])
    center_rf = float(_potential_component(BOWL_CENTER_M[None, :], "rf")[0])
    center_total = center_dc + center_rf

    candidates = []
    profiles = []
    for axis_label in ("+x", "-x", "+y", "-y", "+z", "-z"):
        candidate, profile = _scan_axis_candidate(axis_label, invalid_floor)
        candidates.append(candidate)
        profiles.append(profile)
        print(f"[bowl axes] {axis_label}: {candidate['status']}", flush=True)
    candidates_df = pd.DataFrame(candidates)
    profiles_df = pd.concat(profiles, ignore_index=True)

    if BOWL_ESCAPE_POLICY == "explicit":
        explicit = _optional_scalar(BOWL_EXPLICIT_ESCAPE_LEVEL_J)
        if explicit is None:
            raise ValueError("USER_BOWL_ESCAPE_LEVEL_J is required for explicit policy")
        z_candidates = candidates_df.loc[
            candidates_df["axis_label"].isin(["+z", "-z"]) & candidates_df["resolved"]
        ]
        if len(z_candidates):
            chosen = z_candidates.iloc[np.argmin(np.abs(z_candidates["potential_total_J"] - explicit))]
            escape_position = chosen[["x_m", "y_m", "z_m"]].to_numpy(float)
            escape_label = f"explicit_level_nearest_{chosen['axis_label']}"
            escape_dc = float(_potential_component(escape_position[None, :], "dc")[0])
            escape_rf = float(_potential_component(escape_position[None, :], "rf")[0])
        else:
            escape_position = BOWL_CENTER_M.copy()
            escape_label = "explicit_level"
            escape_dc = np.nan
            escape_rf = np.nan
        escape_total = explicit
    else:
        if BOWL_ESCAPE_POLICY == "z_axis":
            pool = candidates_df.loc[
                candidates_df["axis_label"].isin(["+z", "-z"]) & candidates_df["resolved"]
            ]
        else:
            pool = candidates_df.loc[candidates_df["resolved"]]
        if len(pool) == 0:
            raise RuntimeError(
                "No escape saddle candidate was found. Increase axis scan ranges or set "
                "USER_BOWL_ESCAPE_POLICY='explicit' and USER_BOWL_ESCAPE_LEVEL_J."
            )
        pool = pool.loc[pool["potential_total_J"] > center_total]
        if len(pool) == 0:
            raise RuntimeError("All resolved saddle candidates are below the bowl-center potential")
        chosen = pool.sort_values("potential_total_J").iloc[0]
        escape_label = str(chosen["axis_label"])
        escape_position = chosen[["x_m", "y_m", "z_m"]].to_numpy(float)
        escape_total = float(chosen["potential_total_J"])
        escape_dc = float(chosen["potential_dc_J"])
        escape_rf = float(chosen["potential_rf_J"])

    barrier = escape_total - center_total
    if not np.isfinite(barrier) or barrier <= 0.0:
        raise RuntimeError("Selected escape energy does not lie above the bowl center")

    x = _axis_grid(BOWL_X_REL_BOUNDS_M, BOWL_NX, BOWL_CENTER_M[0])
    y = _axis_grid(BOWL_Y_REL_BOUNDS_M, BOWL_NY, BOWL_CENTER_M[1])
    z = _axis_grid(BOWL_Z_REL_BOUNDS_M, BOWL_NZ, BOWL_CENTER_M[2])
    if invalid_floor is not None and z[0] <= invalid_floor:
        z = np.linspace(max(invalid_floor + 1.0e-9, z[0]), z[-1], BOWL_NZ)

    print(
        f"[bowl grid] evaluating {len(x)} x {len(y)} x {len(z)} = "
        f"{len(x)*len(y)*len(z):,} points",
        flush=True,
    )
    u_dc, u_rf, u_total, rf_diag = _build_potential_grid(x, y, z, invalid_floor)
    valid = np.isfinite(u_total)
    center_index = (
        _nearest_index(x, BOWL_CENTER_M[0]),
        _nearest_index(y, BOWL_CENTER_M[1]),
        _nearest_index(z, BOWL_CENTER_M[2]),
    )
    if not valid[center_index]:
        raise RuntimeError("The bowl-center grid voxel is invalid")

    connectivity = generate_binary_structure(3, BOWL_COMPONENT_CONNECTIVITY)
    requested_fractions = [BOWL_NEAR_BARRIER_FRACTION, *BOWL_CLOSURE_FRACTION_LADDER]
    fractions = sorted({float(value) for value in requested_fractions if np.isfinite(value) and value > 0.0})
    closure_rows: list[dict[str, Any]] = []
    selected = None

    for fraction in fractions:
        trial_offset = max(BOWL_NEAR_BARRIER_OFFSET_J, fraction * barrier)
        trial_level = escape_total - trial_offset
        trial_sublevel = valid & (u_total <= trial_level)
        trial_labels, trial_n_components = label(trial_sublevel, structure=connectivity)
        trial_center_label = int(trial_labels[center_index])
        if trial_center_label == 0:
            trial_component = np.zeros_like(trial_sublevel, dtype=bool)
            trial_touches = False
            trial_voxels = 0
            trial_closed = False
        else:
            trial_component = trial_labels == trial_center_label
            trial_touches = _component_touches_boundary(trial_component)
            trial_voxels = int(trial_component.sum())
            trial_closed = bool(trial_voxels > 0 and not trial_touches)

        closure_rows.append({
            "near_barrier_fraction": fraction,
            "near_barrier_offset_J": trial_offset,
            "classification_level_J": trial_level,
            "n_sublevel_components": int(trial_n_components),
            "center_component_voxels": trial_voxels,
            "center_component_touches_grid": trial_touches,
            "center_component_closed_on_grid": trial_closed,
        })
        print(
            f"[bowl closure] fraction={fraction:.6g} level={trial_level:.12e} "
            f"voxels={trial_voxels:,} touches_grid={trial_touches}",
            flush=True,
        )
        if trial_closed:
            selected = (fraction, trial_offset, trial_level, trial_component)
            break

    closure_table = pd.DataFrame(closure_rows)
    closure_table.to_csv(BOWL_CLOSURE_SEARCH_CSV, index=False)

    if selected is None:
        # Preserve an inspectable diagnostic model, but never silently call it
        # a valid closed bowl when the origin component reaches the grid edge.
        last = closure_rows[-1]
        fraction = float(last["near_barrier_fraction"])
        near_offset = float(last["near_barrier_offset_J"])
        classification_level = float(last["classification_level_J"])
        sublevel = valid & (u_total <= classification_level)
        labels, _ = label(sublevel, structure=connectivity)
        center_label = int(labels[center_index])
        component = labels == center_label if center_label != 0 else np.zeros_like(sublevel, dtype=bool)
        if BOWL_REQUIRE_CLOSED_COMPONENT:
            raise RuntimeError(
                "No closed origin-connected bowl was resolved by the configured "
                "near-barrier fraction ladder. Inspect the closure-search CSV, "
                "increase the grid resolution, or extend the fraction ladder."
            )
    else:
        fraction, near_offset, classification_level, component = selected

    touches = _component_touches_boundary(component)
    spacing = (float(np.diff(x).mean()), float(np.diff(y).mean()), float(np.diff(z).mean()))
    inside_distance = distance_transform_edt(component, sampling=spacing)
    outside_distance = distance_transform_edt(~component, sampling=spacing)
    signed_grid = outside_distance.copy()
    signed_grid[component] = -inside_distance[component]

    # Isolate the origin component while retaining the true scalar potential
    # on its boundary. With a closed six-neighbour component, disconnected
    # exterior low-potential voxels cannot share a grid edge with this basin.
    masked_scalar = np.asarray(u_total, dtype=float).copy()
    high_finite = float(np.nanmax(u_total[valid]))
    replacement = max(
        classification_level + max(near_offset, 1.0e-30),
        high_finite + max(abs(high_finite), 1.0) * np.finfo(float).eps,
    )
    disconnected_low = valid & (u_total <= classification_level) & ~component
    masked_scalar[disconnected_low] = replacement
    masked_scalar[~valid] = replacement

    vertices_index, faces, normals, values = marching_cubes(
        masked_scalar,
        level=classification_level,
        spacing=spacing,
        allow_degenerate=False,
    )
    vertices = vertices_index + np.array([x[0], y[0], z[0]])
    vertex_dc = _eval_component_chunked(vertices, "dc")
    vertex_rf = _eval_component_chunked(vertices, "rf")
    vertex_total = vertex_dc + vertex_rf

    # Mesh is watertight when every undirected edge belongs to exactly two faces.
    edges = np.sort(np.vstack([faces[:, [0, 1]], faces[:, [1, 2]], faces[:, [2, 0]]]), axis=1)
    _, edge_counts = np.unique(edges, axis=0, return_counts=True)
    watertight = bool(len(edge_counts) > 0 and np.all(edge_counts == 2))
    closed = bool(watertight and not touches)

    center_h_total = _finite_difference_hessian(BOWL_CENTER_M, "trap")
    center_h_dc = _finite_difference_hessian(BOWL_CENTER_M, "dc")
    center_h_rf = _finite_difference_hessian(BOWL_CENTER_M, "rf")
    escape_h_total = _finite_difference_hessian(escape_position, "trap")
    escape_h_dc = _finite_difference_hessian(escape_position, "dc")
    escape_h_rf = _finite_difference_hessian(escape_position, "rf")
    try:
        escape_force = _trap_force(escape_position[None, :])[0]
        escape_grad_norm = float(np.linalg.norm(escape_force))
    except Exception:
        escape_grad_norm = np.nan

    payload = {
        "m_dm": m_dm,
        "eps": eps,
        "center": BOWL_CENTER_M.tolist(),
        "escape_label": escape_label,
        "escape_total": escape_total,
        "classification_level": classification_level,
        "grid_shape": [len(x), len(y), len(z)],
        "component_hash": hashlib.sha256(component.tobytes()).hexdigest(),
    }
    signature = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

    profiles_df.to_csv(BOWL_AXIS_PROFILES_CSV, index=False)
    return BowlModel(
        center_m=BOWL_CENTER_M.copy(),
        center_potential_total_J=center_total,
        center_potential_dc_J=center_dc,
        center_potential_rf_J=center_rf,
        m_dm_kg=m_dm,
        eps=eps,
        reference_potential_J=BOWL_REFERENCE_POTENTIAL_J,
        invalid_z_floor_m=invalid_floor,
        escape_policy=BOWL_ESCAPE_POLICY,
        escape_label=escape_label,
        escape_position_m=escape_position,
        escape_potential_total_J=escape_total,
        escape_potential_dc_J=escape_dc,
        escape_potential_rf_J=escape_rf,
        classification_level_J=classification_level,
        near_barrier_offset_J=near_offset,
        near_barrier_fraction_selected=float(fraction),
        component_connectivity=BOWL_COMPONENT_CONNECTIVITY,
        axis_candidates=candidates_df,
        x_grid_m=x,
        y_grid_m=y,
        z_grid_m=z,
        component_mask=component,
        signed_distance_grid_m=signed_grid,
        surface_vertices_m=vertices,
        surface_faces=faces,
        surface_vertex_potential_total_J=vertex_total,
        surface_vertex_potential_dc_J=vertex_dc,
        surface_vertex_potential_rf_J=vertex_rf,
        component_closed=closed,
        component_touches_grid_boundary=touches,
        component_voxel_count=int(component.sum()),
        surface_is_watertight=watertight,
        rf_x_max_absolute_variation_J=float(rf_diag["rf_x_max_absolute_variation_J"]),
        rf_x_max_relative_variation=float(rf_diag["rf_x_max_relative_variation"]),
        rf_x_invariant_within_tolerance=bool(rf_diag["rf_x_invariant_within_tolerance"]),
        center_hessian_total_J_m2=center_h_total,
        center_hessian_dc_J_m2=center_h_dc,
        center_hessian_rf_J_m2=center_h_rf,
        escape_hessian_total_J_m2=escape_h_total,
        escape_hessian_dc_J_m2=escape_h_dc,
        escape_hessian_rf_J_m2=escape_h_rf,
        escape_gradient_norm_N=escape_grad_norm,
        model_signature=signature,
    )


# =============================================================================
# TRAJECTORY METRICS
# =============================================================================


def interval_metrics_from_signed_values(
    times_s: np.ndarray,
    signed_values: np.ndarray,
    *,
    inside_tolerance: float = 0.0,
) -> dict[str, Any]:
    times = np.asarray(times_s, dtype=float).reshape(-1)
    signed = np.asarray(signed_values, dtype=float).reshape(-1) - float(inside_tolerance)
    if times.size != signed.size or times.size == 0:
        return {
            "entered": False, "crossed_inward": False, "crossed_outward": False,
            "started_inside": False, "ended_inside": False, "time_inside_s": 0.0,
            "entry_time_s": np.nan, "exit_time_s": np.nan, "entry_count": 0,
            "exit_count": 0, "minimum_signed_value": np.nan, "valid_fraction": 0.0,
            "first_entry_segment_index": -1, "first_entry_fraction": np.nan,
        }
    order = np.argsort(times)
    times = times[order]
    signed = signed[order]
    finite = np.isfinite(times) & np.isfinite(signed)
    inside = finite & (signed <= 0.0)
    valid_indices = np.flatnonzero(finite)
    started_inside = bool(inside[valid_indices[0]]) if len(valid_indices) else False
    ended_inside = bool(inside[valid_indices[-1]]) if len(valid_indices) else False
    total_inside = 0.0
    entry_times: list[float] = []
    exit_times: list[float] = []
    first_entry_segment = -1
    first_entry_fraction = np.nan
    for index in range(len(times) - 1):
        if not (finite[index] and finite[index + 1]):
            continue
        t0, t1 = float(times[index]), float(times[index + 1])
        if t1 <= t0:
            continue
        g0, g1 = float(signed[index]), float(signed[index + 1])
        in0, in1 = g0 <= 0.0, g1 <= 0.0
        if in0 and in1:
            total_inside += t1 - t0
        elif in0 == in1:
            continue
        else:
            fraction = float(np.clip(g0 / (g0 - g1) if g0 != g1 else 0.5, 0.0, 1.0))
            crossing = t0 + fraction * (t1 - t0)
            if not in0 and in1:
                if first_entry_segment < 0:
                    first_entry_segment = index
                    first_entry_fraction = fraction
                entry_times.append(crossing)
                total_inside += t1 - crossing
            else:
                exit_times.append(crossing)
                total_inside += crossing - t0
    if started_inside and not entry_times:
        entry_times = [float(times[valid_indices[0]])]
    minimum = float(np.nanmin(signed[finite])) if np.any(finite) else np.nan
    return {
        "entered": bool(np.any(inside)),
        "crossed_inward": bool(first_entry_segment >= 0),
        "crossed_outward": bool(exit_times),
        "started_inside": started_inside,
        "ended_inside": ended_inside,
        "time_inside_s": float(total_inside),
        "entry_time_s": float(entry_times[0]) if entry_times else np.nan,
        "exit_time_s": float(exit_times[-1]) if exit_times else np.nan,
        "entry_count": int(len(entry_times)),
        "exit_count": int(len(exit_times)),
        "minimum_signed_value": minimum,
        "valid_fraction": float(np.mean(finite)) if len(finite) else 0.0,
        "first_entry_segment_index": int(first_entry_segment),
        "first_entry_fraction": float(first_entry_fraction),
    }


def _empty_interval_metrics(prefix: str) -> dict[str, Any]:
    output = {
        f"{prefix}_bowl_model_available": False,
        f"{prefix}_entered_bowl": False,
        f"{prefix}_crossed_bowl_inward": False,
        f"{prefix}_crossed_bowl_outward": False,
        f"{prefix}_started_inside_bowl": False,
        f"{prefix}_ended_inside_bowl": False,
        f"{prefix}_time_inside_bowl_s": 0.0,
        f"{prefix}_time_inside_bowl_us": 0.0,
        f"{prefix}_bowl_entry_time_s": np.nan,
        f"{prefix}_bowl_exit_time_s": np.nan,
        f"{prefix}_bowl_entry_count": 0,
        f"{prefix}_bowl_exit_count": 0,
        f"{prefix}_min_bowl_signed_distance_m": np.nan,
        f"{prefix}_min_bowl_signed_distance_um": np.nan,
        f"{prefix}_bowl_valid_sample_fraction": 0.0,
        f"{prefix}_bowl_entry_channel": "none",
        f"{prefix}_bowl_entry_x_m": np.nan,
        f"{prefix}_bowl_entry_y_m": np.nan,
        f"{prefix}_bowl_entry_z_m": np.nan,
        f"{prefix}_bowl_entry_U_total_J": np.nan,
        f"{prefix}_bowl_entry_U_dc_J": np.nan,
        f"{prefix}_bowl_entry_U_rf_J": np.nan,
        f"{prefix}_bowl_entry_F_dc_toward_center_N": np.nan,
        f"{prefix}_bowl_entry_F_rf_toward_center_N": np.nan,
        f"{prefix}_bowl_entry_F_total_toward_center_N": np.nan,
    }
    return output


def _entry_diagnostics(model: BowlModel, point_m: np.ndarray) -> dict[str, Any]:
    point = np.asarray(point_m, dtype=float).reshape(3)
    u_dc = float(_potential_component(point[None, :], "dc")[0])
    u_rf = float(_potential_component(point[None, :], "rf")[0])
    f_dc = _force_component(point[None, :], "dc")[0]
    f_rf = _force_component(point[None, :], "rf")[0]
    toward = model.center_m - point
    norm = float(np.linalg.norm(toward))
    toward_hat = toward / norm if norm > 0.0 else np.zeros(3)
    return {
        "channel": model.entry_channel(point),
        "point": point,
        "U_dc": u_dc,
        "U_rf": u_rf,
        "U_total": u_dc + u_rf,
        "F_dc_toward": float(np.dot(f_dc, toward_hat)),
        "F_rf_toward": float(np.dot(f_rf, toward_hat)),
        "F_total_toward": float(np.dot(f_dc + f_rf, toward_hat)),
    }


def bowl_metrics_from_samples(
    times_s: np.ndarray,
    dm_positions_m: np.ndarray,
    *,
    prefix: str,
    model: BowlModel | None = None,
) -> dict[str, Any]:
    bowl = model if model is not None else globals().get("BOWL_MODEL")
    if bowl is None:
        return _empty_interval_metrics(prefix)
    positions = np.asarray(dm_positions_m, dtype=float).reshape(-1, 3)
    times = np.asarray(times_s, dtype=float).reshape(-1)
    signed = bowl.signed_distance_m(positions)
    metrics = interval_metrics_from_signed_values(
        times, signed, inside_tolerance=BOWL_INSIDE_TOLERANCE_M
    )
    output = _empty_interval_metrics(prefix)
    output.update({
        f"{prefix}_bowl_model_available": True,
        f"{prefix}_entered_bowl": bool(metrics["entered"]),
        f"{prefix}_crossed_bowl_inward": bool(metrics["crossed_inward"]),
        f"{prefix}_crossed_bowl_outward": bool(metrics["crossed_outward"]),
        f"{prefix}_started_inside_bowl": bool(metrics["started_inside"]),
        f"{prefix}_ended_inside_bowl": bool(metrics["ended_inside"]),
        f"{prefix}_time_inside_bowl_s": float(metrics["time_inside_s"]),
        f"{prefix}_time_inside_bowl_us": float(metrics["time_inside_s"] * 1e6),
        f"{prefix}_bowl_entry_time_s": float(metrics["entry_time_s"]),
        f"{prefix}_bowl_exit_time_s": float(metrics["exit_time_s"]),
        f"{prefix}_bowl_entry_count": int(metrics["entry_count"]),
        f"{prefix}_bowl_exit_count": int(metrics["exit_count"]),
        f"{prefix}_min_bowl_signed_distance_m": float(metrics["minimum_signed_value"]),
        f"{prefix}_min_bowl_signed_distance_um": (
            float(metrics["minimum_signed_value"] * 1e6)
            if np.isfinite(metrics["minimum_signed_value"]) else np.nan
        ),
        f"{prefix}_bowl_valid_sample_fraction": float(metrics["valid_fraction"]),
    })
    index = int(metrics["first_entry_segment_index"])
    fraction = float(metrics["first_entry_fraction"])
    if index >= 0 and index + 1 < len(positions) and np.isfinite(fraction):
        point = positions[index] + fraction * (positions[index + 1] - positions[index])
        diagnostics = _entry_diagnostics(bowl, point)
        output.update({
            f"{prefix}_bowl_entry_channel": diagnostics["channel"],
            f"{prefix}_bowl_entry_x_m": float(point[0]),
            f"{prefix}_bowl_entry_y_m": float(point[1]),
            f"{prefix}_bowl_entry_z_m": float(point[2]),
            f"{prefix}_bowl_entry_U_total_J": diagnostics["U_total"],
            f"{prefix}_bowl_entry_U_dc_J": diagnostics["U_dc"],
            f"{prefix}_bowl_entry_U_rf_J": diagnostics["U_rf"],
            f"{prefix}_bowl_entry_F_dc_toward_center_N": diagnostics["F_dc_toward"],
            f"{prefix}_bowl_entry_F_rf_toward_center_N": diagnostics["F_rf_toward"],
            f"{prefix}_bowl_entry_F_total_toward_center_N": diagnostics["F_total_toward"],
        })
    return output


def bowl_metrics_from_dense_solution(
    dense_solution: Callable[[float], np.ndarray] | None,
    duration_s: float,
    *,
    prefix: str,
    position_slice: slice = slice(0, 3),
    model: BowlModel | None = None,
    sample_dt_s: float | None = None,
) -> dict[str, Any]:
    if dense_solution is None or not np.isfinite(duration_s) or duration_s < 0.0:
        return _empty_interval_metrics(prefix)
    if duration_s == 0.0:
        times = np.array([0.0])
    else:
        dt = float(sample_dt_s if sample_dt_s is not None else BOWL_TIME_SAMPLE_DT_S)
        count = max(2, int(math.ceil(duration_s / max(dt, np.finfo(float).tiny))) + 1)
        times = np.linspace(0.0, duration_s, min(count, BOWL_MAX_TIME_SAMPLES))
    states = np.asarray(dense_solution(times), dtype=float)
    if states.ndim == 1:
        states = states[:, None]
    positions = states[position_slice, :].T
    return bowl_metrics_from_samples(times, positions, prefix=prefix, model=model)


def bowl_default_fields(prefix: str) -> dict[str, Any]:
    return _empty_interval_metrics(prefix)


def _bool_value(value: Any) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def _number(row: dict[str, Any] | pd.Series, key: str, default=np.nan) -> float:
    try:
        return float(row.get(key, default))
    except (TypeError, ValueError):
        return float(default)


def combine_bowl_stage_metrics(row: dict[str, Any]) -> dict[str, Any]:
    output = dict(row)
    stages = ("dm_only", "full", "outgoing_rfar", "outgoing_outer")
    model_available = any(_bool_value(output.get(f"{s}_bowl_model_available", False)) for s in stages)
    entered = any(_bool_value(output.get(f"{s}_entered_bowl", False)) for s in stages)
    crossed_in = any(_bool_value(output.get(f"{s}_crossed_bowl_inward", False)) for s in stages)
    crossed_out = any(_bool_value(output.get(f"{s}_crossed_bowl_outward", False)) for s in stages)
    total_time = sum(
        max(_number(output, f"{s}_time_inside_bowl_s", 0.0), 0.0)
        for s in stages
        if np.isfinite(_number(output, f"{s}_time_inside_bowl_s", 0.0))
    )
    minima = [_number(output, f"{s}_min_bowl_signed_distance_m") for s in stages]
    minima = [v for v in minima if np.isfinite(v)]
    min_signed = min(minima) if minima else np.nan
    entry_stage = "none"
    for stage in stages:
        if _bool_value(output.get(f"{stage}_entered_bowl", False)):
            entry_stage = stage
            break
    entry_channel = str(output.get(f"{entry_stage}_bowl_entry_channel", "none")) if entry_stage != "none" else "none"
    last_stage = next((s for s in reversed(stages) if _bool_value(output.get(f"{s}_bowl_model_available", False))), None)
    ended_inside = _bool_value(output.get(f"{last_stage}_ended_inside_bowl", False)) if last_stage else False
    resolved = _bool_value(output.get("fourier_detection_resolved", False))
    above = _bool_value(output.get("above_energy_threshold", False))
    full_status = str(output.get("full_status", ""))
    dm_status = str(output.get("dm_only_status", ""))
    if not model_available:
        trajectory_class = "bowl_model_unavailable"
    elif not entered:
        if resolved and above:
            trajectory_class = "no_bowl_entry_detectable_counterexample"
        elif resolved:
            trajectory_class = "no_bowl_entry_resolved_subthreshold"
        elif "escaped_without_switch" in dm_status:
            trajectory_class = "no_bowl_entry_resolved_no_switch"
        else:
            trajectory_class = "no_bowl_entry_unresolved"
    elif "timeout" in full_status:
        trajectory_class = "bowl_entered_long_lived_unresolved"
    elif resolved and above:
        trajectory_class = "bowl_entered_detectable_resolved"
    elif resolved:
        trajectory_class = "bowl_entered_subthreshold_resolved"
    else:
        trajectory_class = "bowl_entered_unresolved"
    bowl = globals().get("BOWL_MODEL")
    output.update({
        "bowl_model_available": bool(model_available),
        "entered_bowl": bool(entered),
        "crossed_bowl_inward": bool(crossed_in),
        "crossed_bowl_outward": bool(crossed_out),
        "exited_bowl_after_entry": bool(entered and crossed_out and not ended_inside),
        "ended_inside_bowl": bool(ended_inside),
        "bowl_entry_stage": entry_stage,
        "bowl_entry_channel": entry_channel,
        "interaction_time_inside_bowl_s": float(total_time),
        "interaction_time_inside_bowl_us": float(total_time * 1e6),
        "minimum_bowl_signed_distance_m": float(min_signed),
        "minimum_bowl_signed_distance_um": float(min_signed * 1e6) if np.isfinite(min_signed) else np.nan,
        "bowl_trajectory_class": trajectory_class,
        "bowl_hypothesis_counterexample": bool(resolved and above and not entered),
        "bowl_escape_boundary_potential_J": float(bowl.escape_potential_total_J) if bowl is not None else np.nan,
        "bowl_classification_level_J": float(bowl.classification_level_J) if bowl is not None else np.nan,
        "bowl_escape_potential_dc_J": float(bowl.escape_potential_dc_J) if bowl is not None else np.nan,
        "bowl_escape_potential_rf_J": float(bowl.escape_potential_rf_J) if bowl is not None else np.nan,
        "bowl_minimum_required_kinetic_energy_J": float(bowl.minimum_required_kinetic_energy_J) if bowl is not None else np.nan,
        "bowl_model_signature": str(bowl.model_signature) if bowl is not None else "",
    })
    if entry_stage != "none":
        for suffix in (
            "bowl_entry_x_m", "bowl_entry_y_m", "bowl_entry_z_m",
            "bowl_entry_U_total_J", "bowl_entry_U_dc_J", "bowl_entry_U_rf_J",
            "bowl_entry_F_dc_toward_center_N", "bowl_entry_F_rf_toward_center_N",
            "bowl_entry_F_total_toward_center_N",
        ):
            output[suffix] = output.get(f"{entry_stage}_{suffix}", np.nan)
    return output


# =============================================================================
# OUTPUT, REPORTING, PLOTS
# =============================================================================


def _summary_row(model: BowlModel) -> dict[str, Any]:
    residual = model.surface_vertex_potential_total_J - model.classification_level_J
    extents = model.extents_relative_m
    center_eigs = np.linalg.eigvalsh(model.center_hessian_total_J_m2)
    escape_eigs = np.linalg.eigvalsh(model.escape_hessian_total_J_m2)
    return {
        "m_dm_kg": model.m_dm_kg,
        "eps": model.eps,
        "center_x_m": model.center_m[0], "center_y_m": model.center_m[1], "center_z_m": model.center_m[2],
        "center_U_total_J": model.center_potential_total_J,
        "center_U_dc_J": model.center_potential_dc_J,
        "center_U_rf_J": model.center_potential_rf_J,
        "escape_policy": model.escape_policy,
        "escape_label": model.escape_label,
        "escape_x_m": model.escape_position_m[0], "escape_y_m": model.escape_position_m[1], "escape_z_m": model.escape_position_m[2],
        "escape_U_total_J": model.escape_potential_total_J,
        "escape_U_dc_J": model.escape_potential_dc_J,
        "escape_U_rf_J": model.escape_potential_rf_J,
        "barrier_from_center_J": model.barrier_from_center_J,
        "classification_level_J": model.classification_level_J,
        "near_barrier_offset_J": model.near_barrier_offset_J,
        "near_barrier_fraction_selected": model.near_barrier_fraction_selected,
        "component_connectivity": model.component_connectivity,
        "minimum_speed_from_reference_m_s": model.minimum_speed_from_reference_m_s,
        "minimum_speed_from_center_m_s": model.minimum_speed_from_center_m_s,
        "component_closed": model.component_closed,
        "component_touches_grid_boundary": model.component_touches_grid_boundary,
        "surface_is_watertight": model.surface_is_watertight,
        "component_voxel_count": model.component_voxel_count,
        "surface_vertex_count": len(model.surface_vertices_m),
        "surface_face_count": len(model.surface_faces),
        "surface_U_residual_max_abs_J": float(np.nanmax(np.abs(residual))),
        "surface_U_residual_rms_J": float(np.sqrt(np.nanmean(residual**2))),
        "extent_x_m": extents[0], "extent_y_m": extents[1], "extent_z_m": extents[2],
        "extent_x_over_y": extents[0] / extents[1] if extents[1] > 0 else np.nan,
        "extent_x_over_z": extents[0] / extents[2] if extents[2] > 0 else np.nan,
        "rf_x_max_absolute_variation_J": model.rf_x_max_absolute_variation_J,
        "rf_x_max_relative_variation": model.rf_x_max_relative_variation,
        "rf_x_invariant_within_tolerance": model.rf_x_invariant_within_tolerance,
        "center_Hxx_total_J_m2": model.center_hessian_total_J_m2[0,0],
        "center_Hxx_dc_J_m2": model.center_hessian_dc_J_m2[0,0],
        "center_Hxx_rf_J_m2": model.center_hessian_rf_J_m2[0,0],
        "center_hessian_eig0_J_m2": center_eigs[0],
        "center_hessian_eig1_J_m2": center_eigs[1],
        "center_hessian_eig2_J_m2": center_eigs[2],
        "escape_hessian_eig0_J_m2": escape_eigs[0],
        "escape_hessian_eig1_J_m2": escape_eigs[1],
        "escape_hessian_eig2_J_m2": escape_eigs[2],
        "escape_gradient_norm_N": model.escape_gradient_norm_N,
        "model_signature": model.model_signature,
    }


def save_bowl_outputs(model: BowlModel) -> None:
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
    model.axis_candidates.to_csv(BOWL_AXIS_CANDIDATES_CSV, index=False)
    pd.DataFrame([_summary_row(model)]).to_csv(BOWL_SUMMARY_CSV, index=False)
    pd.DataFrame({
        "vertex_index": np.arange(len(model.surface_vertices_m)),
        "x_m": model.surface_vertices_m[:,0], "y_m": model.surface_vertices_m[:,1], "z_m": model.surface_vertices_m[:,2],
        "x_um": model.surface_vertices_m[:,0]*1e6, "y_um": model.surface_vertices_m[:,1]*1e6, "z_um": model.surface_vertices_m[:,2]*1e6,
        "U_total_J": model.surface_vertex_potential_total_J,
        "U_dc_J": model.surface_vertex_potential_dc_J,
        "U_rf_J": model.surface_vertex_potential_rf_J,
        "U_minus_level_J": model.surface_vertex_potential_total_J - model.classification_level_J,
    }).to_csv(BOWL_SURFACE_VERTICES_CSV, index=False)
    np.savez_compressed(
        BOWL_MODEL_NPZ,
        center_m=model.center_m,
        escape_position_m=model.escape_position_m,
        escape_potential_total_J=model.escape_potential_total_J,
        escape_potential_dc_J=model.escape_potential_dc_J,
        escape_potential_rf_J=model.escape_potential_rf_J,
        classification_level_J=model.classification_level_J,
        x_grid_m=model.x_grid_m, y_grid_m=model.y_grid_m, z_grid_m=model.z_grid_m,
        component_mask=model.component_mask,
        signed_distance_grid_m=model.signed_distance_grid_m,
        surface_vertices_m=model.surface_vertices_m,
        surface_faces=model.surface_faces,
        model_signature=np.array(model.model_signature),
    )


def print_bowl_summary(model: BowlModel) -> None:
    summary = _summary_row(model)
    print("\n-------- Connected Equipotential Bowl Summary --------")
    print("Definition: origin-connected component of U_trap <= U_classification.")
    print(f"Center [m]: {np.array2string(model.center_m, precision=12, separator=', ')}")
    print(
        f"Center potential: total={model.center_potential_total_J:.12e} J, "
        f"DC={model.center_potential_dc_J:.12e} J, RF={model.center_potential_rf_J:.12e} J"
    )
    print("\nAxis saddle candidates:")
    print(model.axis_candidates.to_string(index=False))
    print("\nSelected escape bottleneck:")
    print(f"  policy/label = {model.escape_policy} / {model.escape_label}")
    print(f"  location [m] = {np.array2string(model.escape_position_m, precision=12, separator=', ')}")
    print(f"  location [um]= {np.array2string(model.escape_position_m*1e6, precision=6, separator=', ')}")
    print(f"  U_total      = {model.escape_potential_total_J:.12e} J")
    print(f"  U_DC         = {model.escape_potential_dc_J:.12e} J")
    print(f"  U_RF         = {model.escape_potential_rf_J:.12e} J")
    print(f"  Delta U from center = {model.barrier_from_center_J:.12e} J")
    print(f"  |grad U| at candidate = {model.escape_gradient_norm_N:.12e} N")
    print(f"  total-Hessian eigenvalues = {np.linalg.eigvalsh(model.escape_hessian_total_J_m2)} J/m^2")
    print(f"  classification surface level = {model.classification_level_J:.12e} J")
    print(f"  near-barrier offset = {model.near_barrier_offset_J:.12e} J")
    print(f"  selected near-barrier fraction = {model.near_barrier_fraction_selected:.6g}")
    print(f"  voxel connectivity = {model.component_connectivity}")
    print("\nDC/RF confinement checks:")
    print(f"  RF x variation absolute = {model.rf_x_max_absolute_variation_J:.12e} J")
    print(f"  RF x variation relative = {model.rf_x_max_relative_variation:.12e}")
    print(f"  RF x-invariant within tolerance = {model.rf_x_invariant_within_tolerance}")
    print(f"  center Hxx total/DC/RF = {model.center_hessian_total_J_m2[0,0]:.12e}, "
          f"{model.center_hessian_dc_J_m2[0,0]:.12e}, {model.center_hessian_rf_J_m2[0,0]:.12e} J/m^2")
    print("\nConnected component / surface:")
    print(f"  closed={model.component_closed}, watertight={model.surface_is_watertight}, "
          f"touches grid={model.component_touches_grid_boundary}")
    print(f"  extents relative to center [um] = {model.extents_relative_m*1e6}")
    print(f"  x/y extent ratio={summary['extent_x_over_y']:.6g}, x/z={summary['extent_x_over_z']:.6g}")
    print(f"  surface max |U-level|={summary['surface_U_residual_max_abs_J']:.12e} J")
    print(f"  surface RMS |U-level|={summary['surface_U_residual_rms_J']:.12e} J")

    analytic_position = _optional_vector(BOWL_ANALYTIC_ESCAPE_POSITION_M)
    analytic_potential = _optional_scalar(BOWL_ANALYTIC_ESCAPE_POTENTIAL_J)
    if analytic_position is not None or analytic_potential is not None:
        print("\nAnalytical comparison:")
    if analytic_position is not None:
        delta = model.escape_position_m - analytic_position
        print(f"  analytical location [m] = {analytic_position}")
        print(f"  numerical-analytical [m]= {delta}")
        print(f"  location error norm     = {np.linalg.norm(delta):.12e} m")
    if analytic_potential is not None:
        delta = model.escape_potential_total_J - analytic_potential
        print(f"  analytical potential    = {analytic_potential:.12e} J")
        print(f"  numerical-analytical    = {delta:.12e} J")
        print(f"  relative potential error= {delta/max(abs(analytic_potential),np.finfo(float).tiny):.12e}")
    print(f"\nSaved summary: {BOWL_SUMMARY_CSV}")
    print(f"Saved axis candidates: {BOWL_AXIS_CANDIDATES_CSV}")
    print(f"Saved surface vertices: {BOWL_SURFACE_VERTICES_CSV}")
    print(f"Saved closure search: {BOWL_CLOSURE_SEARCH_CSV}")


def plot_bowl_model(model: BowlModel) -> None:
    vertices = model.surface_vertices_m[::BOWL_SURFACE_DOWNSAMPLE] * 1e6
    figure = plt.figure(figsize=(11, 8))
    axis = figure.add_subplot(111, projection="3d")
    # Use a point cloud for robustness; optionally overlay the triangular mesh
    # at manageable size.
    residual = model.surface_vertex_potential_total_J[::BOWL_SURFACE_DOWNSAMPLE] - model.classification_level_J
    scatter = axis.scatter(vertices[:,0], vertices[:,1], vertices[:,2], c=residual, s=2, alpha=0.65)
    axis.scatter(*((model.center_m*1e6).tolist()), marker="o", s=50, label="ion equilibrium")
    axis.scatter(*((model.escape_position_m*1e6).tolist()), marker="*", s=160, label="selected escape bottleneck")
    axis.set_xlabel("x [um] (DC axial)")
    axis.set_ylabel("y [um] (RF transverse)")
    axis.set_zlabel("z [um] (RF vertical)")
    axis.set_title(
        "Connected near-barrier equipotential bowl\n"
        f"U={model.classification_level_J:.4e} J; closed={model.component_closed}; "
        f"escape={model.escape_label}"
    )
    figure.colorbar(scatter, ax=axis, shrink=0.7, pad=0.08, label="U(vertex)-U(level) [J]")
    axis.legend()
    figure.tight_layout()
    if BOWL_SAVE_PLOTS:
        figure.savefig(BOWL_SURFACE_PNG, dpi=180)
    if BOWL_SHOW_PLOTS:
        plt.show()
    plt.close(figure)

    profiles = pd.read_csv(BOWL_AXIS_PROFILES_CSV)
    figure, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=False)
    for axis_plot, label_name in zip(axes.ravel(), ("+x","-x","+y","-y","+z","-z")):
        group = profiles.loc[profiles["axis_label"].eq(label_name)]
        axis_plot.plot(group["radius_um"], group["U_total_J"], label="total")
        axis_plot.plot(group["radius_um"], group["U_dc_J"], label="DC")
        axis_plot.plot(group["radius_um"], group["U_rf_J"], label="RF pseudo")
        candidate = model.axis_candidates.loc[model.axis_candidates["axis_label"].eq(label_name)]
        if len(candidate) and bool(candidate.iloc[0]["resolved"]):
            axis_plot.scatter(candidate.iloc[0]["radius_m"]*1e6, candidate.iloc[0]["potential_total_J"], marker="x")
        axis_plot.axhline(model.escape_potential_total_J, linestyle="--", linewidth=0.8)
        axis_plot.set_title(label_name)
        axis_plot.set_xlabel("distance from ion [um]")
        axis_plot.set_ylabel("potential [J]")
    axes[0,0].legend()
    figure.suptitle("Axis potential decomposition and saddle candidates")
    figure.tight_layout()
    if BOWL_SAVE_PLOTS:
        figure.savefig(BOWL_AXIS_PNG, dpi=180)
    if BOWL_SHOW_PLOTS:
        plt.show()
    plt.close(figure)

    # Mid-plane component slices make connectivity and grid clipping obvious.
    ix = _nearest_index(model.x_grid_m, model.center_m[0])
    iy = _nearest_index(model.y_grid_m, model.center_m[1])
    iz = _nearest_index(model.z_grid_m, model.center_m[2])
    figure, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    axes[0].imshow(model.component_mask[ix].T, origin="lower", aspect="auto",
                   extent=[model.y_grid_m[0]*1e6,model.y_grid_m[-1]*1e6,model.z_grid_m[0]*1e6,model.z_grid_m[-1]*1e6])
    axes[0].set_title("origin-connected bowl: yz at center x")
    axes[0].set_xlabel("y [um]"); axes[0].set_ylabel("z [um]")
    axes[1].imshow(model.component_mask[:,iy,:].T, origin="lower", aspect="auto",
                   extent=[model.x_grid_m[0]*1e6,model.x_grid_m[-1]*1e6,model.z_grid_m[0]*1e6,model.z_grid_m[-1]*1e6])
    axes[1].set_title("origin-connected bowl: xz at center y")
    axes[1].set_xlabel("x [um]"); axes[1].set_ylabel("z [um]")
    axes[2].imshow(model.component_mask[:,:,iz].T, origin="lower", aspect="auto",
                   extent=[model.x_grid_m[0]*1e6,model.x_grid_m[-1]*1e6,model.y_grid_m[0]*1e6,model.y_grid_m[-1]*1e6])
    axes[2].set_title("origin-connected bowl: xy at center z")
    axes[2].set_xlabel("x [um]"); axes[2].set_ylabel("y [um]")
    figure.tight_layout()
    if BOWL_SAVE_PLOTS:
        figure.savefig(BOWL_COMPONENT_PNG, dpi=180)
    if BOWL_SHOW_PLOTS:
        plt.show()
    plt.close(figure)


if RUN_BOWL_GEOMETRY:
    BOWL_MODEL = build_bowl_model()
    save_bowl_outputs(BOWL_MODEL)
    print_bowl_summary(BOWL_MODEL)
    plot_bowl_model(BOWL_MODEL)
else:
    BOWL_MODEL = globals().get("BOWL_MODEL", None)
    print("[SKIP] Connected equipotential bowl geometry")


# Test 0: v_min_safe Barrier


In [ ]:
"""
Test 0 fast prefilter: minimum trap potential on the analytically matched
Rutherford r_min_threshold sphere.

Run in the same Python/IPython namespace as the trap model, where

    potential_energy(points, mass, charge_fraction, component=...)

and the globals m_ion, Z_ion, K, and e are already defined. Typical use:

    %run -i test0_reach_aware_v10.py

For each configured speed v:

1. Compute the exact free-space Rutherford closest-approach threshold radius
   R(v) for the chosen ion recoil threshold.
2. Minimize the selected trap potential on the sphere |r| = R(v).
3. Compare the incoming absolute energy

       E_in(v) = U_REFERENCE_J + 0.5 * M_DM_KG * v**2

   with a conservative numerical lower estimate of the minimum sphere
   potential.

If E_in is below the minimum potential everywhere on the sphere, the DM cannot
cross that sphere in a static conservative potential and the speed can be
rejected.

This is a fast, one-sided prefilter. Passing the test does not prove the sphere
is reachable because a higher saddle/barrier may exist before the sphere.
The full minimax-to-sphere calculation is stronger but slower.
"""

from __future__ import annotations

import math
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize
from scipy.stats import maxwell


SCRIPT_VERSION = "2026-07-16-fast-sphere-minimum-prefilter-v10-reach-aware-suite"


# ============================================================
# USER SETTINGS
# ============================================================

M_DM_KG = float(USER_M_DM_KG)
EPS = float(USER_EPS)
T_DM_K = float(USER_T_DM_K)
POTENTIAL_COMPONENT = "trap"

# Incoming absolute potential-energy reference. Use zero only when the selected
# trap potential is defined to approach zero in the incoming/asymptotic region.
U_REFERENCE_J = 0.0

RUTHERFORD_ION_ENERGY_THRESHOLD_J = float(USER_TARGET_ION_ENERGY_J)

# Exact analytic R(v) is calculated for each speed; no rounded radius is used.
# These diagnostic speeds are recomputed from the current Maxwell-Boltzmann
# distribution whenever USER_M_DM_KG or USER_T_DM_K changes.
_MB_SCALE_M_S = math.sqrt(k_B * T_DM_K / M_DM_KG)
SPEEDS_M_S = tuple(
    float(maxwell.ppf(float(q), scale=_MB_SCALE_M_S))
    for q in USER_TEST0_SINGLE_SPEED_QUANTILES
) + tuple(float(v) for v in USER_TEST0_EXTRA_SPEEDS_M_S)

# Deterministic angular resolutions. Each level is followed by local continuous
# optimization from the lowest sampled directions.
SPHERE_SAMPLE_COUNTS = (4096, 16384)
LOCAL_OPTIMIZATION_SEEDS = 16
LOCAL_OPTIMIZATION_MAXITER = 300
POTENTIAL_EVAL_CHUNK_SIZE = 100_000

# The sampled/refined minimum is not a mathematically rigorous global lower
# bound. To avoid false rejection, lower it by a convergence margin based on
# the spread between angular resolutions.
SPHERE_MIN_SAFETY_FACTOR = 2.0
RELATIVE_SPHERE_MIN_MARGIN = 0.0
USER_EXTRA_SPHERE_MIN_MARGIN_J = 0.0

# Optional fast critical-speed scan for this simple necessary-condition test.
# It solves
#
#   U_min,sphere_safe(R(v)) = U_REFERENCE_J + 0.5*m*v^2.
#
# Keep False when checking only specified speeds.
FIND_CRITICAL_SPEEDS = True
CRITICAL_SPEED_MIN_M_S = max(
    1.0e-9,
    float(maxwell.ppf(USER_TEST0_CRITICAL_MIN_QUANTILE, scale=_MB_SCALE_M_S)),
)
CRITICAL_SPEED_MAX_M_S = float(
    maxwell.ppf(1.0 - USER_SPEED_UPPER_TAIL_PROBABILITY, scale=_MB_SCALE_M_S)
)
CRITICAL_SPEED_SCAN_POINTS = int(USER_TEST0_CRITICAL_SCAN_POINTS)
CRITICAL_SPEED_ROOT_TOL_M_S = 0.05

OUTPUT_PREFIX = TEST0_SINGLE_OUTPUT_PREFIX
SAVE_RESULTS_CSV = True
SAVE_CRITICAL_SCAN_PLOT = True
SHOW_PLOTS = True


# ============================================================
# ENVIRONMENT AND POTENTIAL EVALUATION
# ============================================================


def progress(message: str = "") -> None:
    print(message, flush=True)


def validate_environment() -> None:
    missing = [
        name
        for name in ("potential_energy", "m_ion", "Z_ion", "K", "e")
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            "Missing required trap-model globals: " + ", ".join(missing)
        )
    if M_DM_KG <= 0.0:
        raise ValueError("M_DM_KG must be positive")
    if EPS <= 0.0:
        raise ValueError("EPS must be positive for the repulsive branch")
    if RUTHERFORD_ION_ENERGY_THRESHOLD_J <= 0.0:
        raise ValueError("RUTHERFORD_ION_ENERGY_THRESHOLD_J must be positive")
    if not SPHERE_SAMPLE_COUNTS:
        raise ValueError("SPHERE_SAMPLE_COUNTS cannot be empty")
    if any(int(n) < 32 for n in SPHERE_SAMPLE_COUNTS):
        raise ValueError("Every sphere sample count must be at least 32")


def evaluate_potential_points(
    xyz: np.ndarray,
    mass: float = M_DM_KG,
    charge_fraction: float = EPS,
    component: str = POTENTIAL_COMPONENT,
) -> np.ndarray:
    """Evaluate potential energy in vectorized chunks with scalar fallback."""
    xyz = np.asarray(xyz, dtype=float).reshape(-1, 3)
    values = np.full(xyz.shape[0], np.nan, dtype=float)

    for start in range(0, xyz.shape[0], POTENTIAL_EVAL_CHUNK_SIZE):
        stop = min(start + POTENTIAL_EVAL_CHUNK_SIZE, xyz.shape[0])
        points = xyz[start:stop]
        try:
            with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
                chunk = np.asarray(
                    potential_energy(
                        points,
                        mass,
                        charge_fraction,
                        component=component,
                    ),
                    dtype=float,
                ).reshape(-1)
            if chunk.size != points.shape[0]:
                raise ValueError("Unexpected vectorized potential shape")
        except Exception:
            scalar_values = []
            for point in points:
                with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
                    value = potential_energy(
                        point,
                        mass,
                        charge_fraction,
                        component=component,
                    )
                scalar_values.append(float(np.asarray(value).reshape(-1)[0]))
            chunk = np.asarray(scalar_values, dtype=float)
        values[start:stop] = chunk

    return values


def evaluate_one_point(point: np.ndarray) -> float:
    return float(evaluate_potential_points(np.asarray(point)[None, :])[0])


# ============================================================
# EXACT RUTHERFORD SPEED-RADIUS RELATION
# ============================================================


def rutherford_threshold_from_speed(speed_m_s: float) -> dict[str, Any]:
    """Exact free-space threshold closest approach for one incoming speed."""
    speed = float(speed_m_s)
    if not np.isfinite(speed) or speed <= 0.0:
        raise ValueError("Speed must be finite and positive")

    ion_mass = float(globals()["m_ion"])
    ion_charge = float(globals()["Z_ion"])
    coulomb_k = (
        float(globals()["K"])
        * ion_charge
        * EPS
        * float(globals()["e"]) ** 2
    )
    if ion_mass <= 0.0:
        raise ValueError("m_ion must be positive")
    if not np.isfinite(coulomb_k) or coulomb_k <= 0.0:
        raise ValueError("A finite positive repulsive Coulomb coupling is required")

    reduced_mass = M_DM_KG * ion_mass / (M_DM_KG + ion_mass)
    threshold = float(RUTHERFORD_ION_ENERGY_THRESHOLD_J)
    maximum_recoil = 2.0 * reduced_mass**2 * speed**2 / ion_mass
    kinematic_min_speed = math.sqrt(
        threshold * ion_mass / (2.0 * reduced_mass**2)
    )

    detectable = bool(
        maximum_recoil >= threshold * (1.0 - 64.0 * np.finfo(float).eps)
    )
    if not detectable:
        return {
            "speed_m_s": speed,
            "r_min_threshold_m": np.nan,
            "r_min_threshold_um": np.nan,
            "impact_parameter_threshold_m": np.nan,
            "impact_parameter_threshold_um": np.nan,
            "maximum_ion_recoil_J": maximum_recoil,
            "kinematic_min_speed_m_s": kinematic_min_speed,
            "reduced_mass_kg": reduced_mass,
            "coulomb_coupling_J_m": coulomb_k,
            "rutherford_detectable": False,
            "pair_status": "below_head_on_recoil_threshold",
        }

    # a is one-half the head-on closest approach.
    a = coulomb_k / (reduced_mass * speed**2)
    radial_term = (
        coulomb_k
        / speed
        * math.sqrt(2.0 / (ion_mass * threshold))
    )
    radius = a + radial_term

    ratio = max(maximum_recoil / threshold - 1.0, 0.0)
    impact_parameter = a * math.sqrt(ratio)

    return {
        "speed_m_s": speed,
        "r_min_threshold_m": radius,
        "r_min_threshold_um": radius * 1e6,
        "impact_parameter_threshold_m": impact_parameter,
        "impact_parameter_threshold_um": impact_parameter * 1e6,
        "maximum_ion_recoil_J": maximum_recoil,
        "kinematic_min_speed_m_s": kinematic_min_speed,
        "reduced_mass_kg": reduced_mass,
        "coulomb_coupling_J_m": coulomb_k,
        "rutherford_detectable": True,
        "pair_status": "analytic_from_speed",
    }


# ============================================================
# SPHERE MINIMUM
# ============================================================


def fibonacci_sphere_directions(n_points: int) -> np.ndarray:
    """Approximately uniform deterministic unit vectors on S^2."""
    n = int(n_points)
    i = np.arange(n, dtype=float) + 0.5
    z = 1.0 - 2.0 * i / n
    golden_angle = math.pi * (3.0 - math.sqrt(5.0))
    phi = golden_angle * i
    radial = np.sqrt(np.maximum(1.0 - z**2, 0.0))
    return np.column_stack(
        [radial * np.cos(phi), radial * np.sin(phi), z]
    )


def direction_to_angles(direction: np.ndarray) -> np.ndarray:
    unit = np.asarray(direction, dtype=float)
    unit = unit / np.linalg.norm(unit)
    theta = math.acos(float(np.clip(unit[2], -1.0, 1.0)))
    phi = math.atan2(float(unit[1]), float(unit[0])) % (2.0 * math.pi)
    return np.array([theta, phi], dtype=float)


def angles_to_direction(theta_phi: np.ndarray) -> np.ndarray:
    theta = float(theta_phi[0])
    phi = float(theta_phi[1]) % (2.0 * math.pi)
    sin_theta = math.sin(theta)
    return np.array(
        [sin_theta * math.cos(phi), sin_theta * math.sin(phi), math.cos(theta)],
        dtype=float,
    )


def minimize_potential_on_sphere_once(
    radius_m: float,
    sample_count: int,
) -> dict[str, Any]:
    """Sample the sphere and refine the best directions continuously."""
    radius = float(radius_m)
    directions = fibonacci_sphere_directions(sample_count)
    points = radius * directions
    sampled_U = evaluate_potential_points(points)
    finite = np.isfinite(sampled_U)
    if not np.any(finite):
        raise RuntimeError(
            f"No finite {POTENTIAL_COMPONENT!r} potential values on sphere "
            f"R={radius * 1e6:.6f} um"
        )

    finite_indices = np.flatnonzero(finite)
    order = finite_indices[np.argsort(sampled_U[finite_indices])]
    n_seeds = min(int(LOCAL_OPTIMIZATION_SEEDS), len(order))
    seed_indices = order[:n_seeds]

    best_U = float(sampled_U[seed_indices[0]])
    best_point = points[seed_indices[0]].copy()
    best_direction = directions[seed_indices[0]].copy()
    best_optimizer_success = False
    best_optimizer_message = "sampled minimum"
    total_nfev = 0

    finite_scale_values = sampled_U[finite]
    energy_scale = max(
        float(np.ptp(finite_scale_values)),
        float(np.max(np.abs(finite_scale_values))),
        1.0e-40,
    )

    def objective(theta_phi: np.ndarray) -> float:
        direction = angles_to_direction(theta_phi)
        value = evaluate_one_point(radius * direction)
        if not np.isfinite(value):
            return 1.0e100
        return value / energy_scale

    for seed_index in seed_indices:
        initial = direction_to_angles(directions[seed_index])
        result = minimize(
            objective,
            x0=initial,
            method="Powell",
            bounds=((0.0, math.pi), (0.0, 2.0 * math.pi)),
            options={
                "maxiter": int(LOCAL_OPTIMIZATION_MAXITER),
                "xtol": 1.0e-11,
                "ftol": 1.0e-12,
            },
        )
        total_nfev += int(getattr(result, "nfev", 0))
        if not np.isfinite(result.fun):
            continue
        direction = angles_to_direction(result.x)
        point = radius * direction
        value = evaluate_one_point(point)
        if np.isfinite(value) and value < best_U:
            best_U = float(value)
            best_point = point
            best_direction = direction
            best_optimizer_success = bool(result.success)
            best_optimizer_message = str(result.message)

    sample_min_index = int(order[0])
    return {
        "sample_count": int(sample_count),
        "sampled_min_U_J": float(sampled_U[sample_min_index]),
        "sampled_min_point_m": points[sample_min_index].copy(),
        "refined_min_U_J": best_U,
        "refined_min_point_m": best_point,
        "refined_min_direction": best_direction,
        "optimizer_success": best_optimizer_success,
        "optimizer_message": best_optimizer_message,
        "optimizer_total_nfev": total_nfev,
        "finite_fraction": float(np.mean(finite)),
        "sample_U_min_J": float(np.min(finite_scale_values)),
        "sample_U_max_J": float(np.max(finite_scale_values)),
        "sample_U_mean_J": float(np.mean(finite_scale_values)),
    }


def conservative_minimum_potential_on_sphere(radius_m: float) -> dict[str, Any]:
    """Multiresolution sphere minimum and conservative numerical lower value."""
    levels = [
        minimize_potential_on_sphere_once(radius_m, int(sample_count))
        for sample_count in SPHERE_SAMPLE_COUNTS
    ]
    values = np.array([level["refined_min_U_J"] for level in levels], dtype=float)
    selected_index = int(np.argmin(values))
    selected = levels[selected_index]

    spread = float(np.max(values) - np.min(values)) if len(values) > 1 else 0.0
    relative_margin = RELATIVE_SPHERE_MIN_MARGIN * max(
        abs(float(np.min(values))), 1.0e-40
    )
    total_margin = (
        SPHERE_MIN_SAFETY_FACTOR * spread
        + relative_margin
        + USER_EXTRA_SPHERE_MIN_MARGIN_J
    )
    conservative_lower = float(np.min(values) - total_margin)

    return {
        "sphere_min_U_estimate_J": float(np.min(values)),
        "sphere_min_U_conservative_lower_J": conservative_lower,
        "sphere_min_resolution_spread_J": spread,
        "sphere_min_convergence_margin_J": SPHERE_MIN_SAFETY_FACTOR * spread,
        "sphere_min_relative_margin_J": relative_margin,
        "sphere_min_user_margin_J": USER_EXTRA_SPHERE_MIN_MARGIN_J,
        "sphere_min_total_margin_J": total_margin,
        "sphere_min_point_m": selected["refined_min_point_m"],
        "sphere_min_direction": selected["refined_min_direction"],
        "sphere_min_selected_sample_count": selected["sample_count"],
        "sphere_min_optimizer_success": selected["optimizer_success"],
        "sphere_min_optimizer_message": selected["optimizer_message"],
        "sphere_min_levels": levels,
    }


# ============================================================
# SPEED TEST AND OPTIONAL ROOT SEARCH
# ============================================================


_EVALUATION_CACHE: dict[float, dict[str, Any]] = {}


def evaluate_speed(speed_m_s: float, use_cache: bool = True) -> dict[str, Any]:
    speed = float(speed_m_s)
    cache_key = float(np.float64(speed))
    if use_cache and cache_key in _EVALUATION_CACHE:
        return dict(_EVALUATION_CACHE[cache_key])

    pair = rutherford_threshold_from_speed(speed)
    kinetic = 0.5 * M_DM_KG * speed**2
    incoming_total = U_REFERENCE_J + kinetic

    if not pair["rutherford_detectable"]:
        row = {
            **pair,
            "kinetic_energy_J": kinetic,
            "incoming_total_energy_J": incoming_total,
            "sphere_min_U_estimate_J": np.nan,
            "sphere_min_U_conservative_lower_J": np.nan,
            "energy_residual_safe_J": np.nan,
            "minimum_speed_for_fixed_sphere_safe_m_s": np.nan,
            "reject_speed": True,
            "reject_reason": "below_head_on_recoil_threshold",
        }
        _EVALUATION_CACHE[cache_key] = dict(row)
        return row

    radius = float(pair["r_min_threshold_m"])
    sphere = conservative_minimum_potential_on_sphere(radius)
    U_min_safe = float(sphere["sphere_min_U_conservative_lower_J"])
    residual = U_min_safe - incoming_total
    tolerance = 128.0 * np.finfo(float).eps * max(
        abs(U_min_safe), abs(incoming_total), 1.0e-40
    )
    reject = bool(residual > tolerance)

    speed_required_fixed_radius = math.sqrt(
        2.0 * max(U_min_safe - U_REFERENCE_J, 0.0) / M_DM_KG
    )

    point = np.asarray(sphere["sphere_min_point_m"], dtype=float)
    direction = np.asarray(sphere["sphere_min_direction"], dtype=float)
    row = {
        **pair,
        "kinetic_energy_J": kinetic,
        "U_reference_J": U_REFERENCE_J,
        "incoming_total_energy_J": incoming_total,
        "sphere_min_U_estimate_J": sphere["sphere_min_U_estimate_J"],
        "sphere_min_U_conservative_lower_J": U_min_safe,
        "sphere_min_resolution_spread_J": sphere[
            "sphere_min_resolution_spread_J"
        ],
        "sphere_min_convergence_margin_J": sphere[
            "sphere_min_convergence_margin_J"
        ],
        "sphere_min_total_margin_J": sphere["sphere_min_total_margin_J"],
        "sphere_min_x_m": point[0],
        "sphere_min_y_m": point[1],
        "sphere_min_z_m": point[2],
        "sphere_min_x_um": point[0] * 1e6,
        "sphere_min_y_um": point[1] * 1e6,
        "sphere_min_z_um": point[2] * 1e6,
        "sphere_min_direction_x": direction[0],
        "sphere_min_direction_y": direction[1],
        "sphere_min_direction_z": direction[2],
        "sphere_min_selected_sample_count": sphere[
            "sphere_min_selected_sample_count"
        ],
        "sphere_min_optimizer_success": sphere["sphere_min_optimizer_success"],
        "energy_residual_safe_J": residual,
        "minimum_speed_for_fixed_sphere_safe_m_s": speed_required_fixed_radius,
        "reject_speed": reject,
        "reject_reason": (
            "incoming_energy_below_minimum_sphere_potential"
            if reject
            else "not_rejected_by_minimum_sphere_potential"
        ),
    }
    _EVALUATION_CACHE[cache_key] = dict(row)
    return row


def critical_residual(speed_m_s: float) -> float:
    row = evaluate_speed(speed_m_s)
    if not row["rutherford_detectable"]:
        # Below the free-space recoil threshold, regard the residual as positive
        # for rejection; root search begins above the kinematic minimum anyway.
        return math.inf
    return float(row["energy_residual_safe_J"])


def find_critical_speeds() -> tuple[pd.DataFrame, pd.DataFrame]:
    ion_mass = float(globals()["m_ion"])
    reduced_mass = M_DM_KG * ion_mass / (M_DM_KG + ion_mass)
    kinematic_min = math.sqrt(
        RUTHERFORD_ION_ENERGY_THRESHOLD_J
        * ion_mass
        / (2.0 * reduced_mass**2)
    )
    lower = max(float(CRITICAL_SPEED_MIN_M_S), kinematic_min * (1.0 + 1.0e-10))
    upper = float(CRITICAL_SPEED_MAX_M_S)
    if upper <= lower:
        raise ValueError("Critical-speed upper bound must exceed lower bound")

    speeds = np.geomspace(lower, upper, int(CRITICAL_SPEED_SCAN_POINTS))
    scan_rows = [evaluate_speed(speed) for speed in speeds]
    scan_df = pd.DataFrame(scan_rows)

    roots: list[dict[str, Any]] = []
    residuals = scan_df["energy_residual_safe_J"].to_numpy(dtype=float)
    for index in range(len(speeds) - 1):
        left = float(speeds[index])
        right = float(speeds[index + 1])
        f_left = float(residuals[index])
        f_right = float(residuals[index + 1])
        if not np.isfinite(f_left) or not np.isfinite(f_right):
            continue
        if f_left == 0.0:
            root = left
        elif f_left * f_right > 0.0:
            continue
        else:
            root = float(
                brentq(
                    critical_residual,
                    left,
                    right,
                    xtol=float(CRITICAL_SPEED_ROOT_TOL_M_S),
                    rtol=4.0 * np.finfo(float).eps,
                    maxiter=100,
                )
            )
        result = evaluate_speed(root)
        roots.append(
            {
                **result,
                "critical_type": "minimum_sphere_potential_energy_equality",
                "bracket_left_m_s": left,
                "bracket_right_m_s": right,
                "bracket_width_m_s": right - left,
            }
        )

    roots_df = pd.DataFrame(roots)
    return scan_df, roots_df


# ============================================================
# MAIN
# ============================================================


def run_fast_sphere_minimum_prefilter() -> tuple[pd.DataFrame, pd.DataFrame]:
    validate_environment()
    progress(f"Test 0 script version: {SCRIPT_VERSION}")
    progress(
        "Fast necessary-condition prefilter: minimum trap potential on the "
        "analytic Rutherford threshold sphere."
    )

    rows = []
    for speed in SPEEDS_M_S:
        progress(f"\nEvaluating v={float(speed):.12g} m/s ...")
        row = evaluate_speed(float(speed), use_cache=False)
        rows.append(row)

        if not row["rutherford_detectable"]:
            progress(
                "  rejected by free-space recoil kinematics: head-on recoil "
                "is below threshold"
            )
            progress(
                f"  kinematic minimum speed = "
                f"{row['kinematic_min_speed_m_s']:.12g} m/s"
            )
            continue

        progress(
            f"  analytic r_min_threshold = "
            f"{row['r_min_threshold_um']:.12g} um"
        )
        progress(
            f"  minimum sampled/refined trap U on sphere = "
            f"{row['sphere_min_U_estimate_J']:.12e} J"
        )
        progress(
            f"  conservative lower sphere minimum = "
            f"{row['sphere_min_U_conservative_lower_J']:.12e} J"
        )
        progress(
            f"  incoming total energy = "
            f"{row['incoming_total_energy_J']:.12e} J"
        )
        progress(
            f"  safe residual U_min_sphere - E_in = "
            f"{row['energy_residual_safe_J']:.12e} J"
        )
        progress(
            f"  minimum-potential point = "
            f"[{row['sphere_min_x_um']:.6f}, "
            f"{row['sphere_min_y_um']:.6f}, "
            f"{row['sphere_min_z_um']:.6f}] um"
        )
        progress(f"  reject speed = {row['reject_speed']}")
        progress(f"  reason = {row['reject_reason']}")

    results_df = pd.DataFrame(rows)
    if SAVE_RESULTS_CSV:
        results_df.to_csv(f"{OUTPUT_PREFIX}_speed_results.csv", index=False)

    critical_df = pd.DataFrame()
    if FIND_CRITICAL_SPEEDS:
        progress("\nScanning for critical speeds ...")
        scan_df, critical_df = find_critical_speeds()
        if SAVE_RESULTS_CSV:
            scan_df.to_csv(f"{OUTPUT_PREFIX}_critical_scan.csv", index=False)
            critical_df.to_csv(
                f"{OUTPUT_PREFIX}_critical_speeds.csv", index=False
            )

        if critical_df.empty:
            progress("  no residual sign changes found in the configured range")
        else:
            for _, root in critical_df.iterrows():
                progress(
                    f"  critical speed = {root['speed_m_s']:.12g} m/s, "
                    f"r_min_threshold={root['r_min_threshold_um']:.12g} um"
                )

        if SAVE_CRITICAL_SCAN_PLOT or SHOW_PLOTS:
            fig, ax = plt.subplots(figsize=(7.5, 5.5))
            ax.plot(
                scan_df["speed_m_s"],
                scan_df["energy_residual_safe_J"],
                marker="o",
            )
            ax.axhline(0.0, linewidth=1.0)
            ax.set_xscale("log")
            ax.set_xlabel("Speed [m/s]")
            ax.set_ylabel(r"$U_{\min,\mathrm{sphere}}^{\mathrm{safe}}-E_{\mathrm{in}}$ [J]")
            ax.set_title("Fast sphere-minimum rejection residual")
            fig.tight_layout()
            if SAVE_CRITICAL_SCAN_PLOT:
                fig.savefig(
                    f"{OUTPUT_PREFIX}_critical_scan.png",
                    dpi=220,
                    bbox_inches="tight",
                )
            if SHOW_PLOTS:
                plt.show()
            else:
                plt.close(fig)

    progress(
        "\nInterpretation: reject=True is a safe one-sided result within the "
        "stated angular-convergence margin. reject=False only means this fast "
        "test cannot reject; a higher barrier along the route may still block "
        "the sphere."
    )
    if SAVE_RESULTS_CSV:
        progress(f"Saved {OUTPUT_PREFIX}_speed_results.csv")

    return results_df, critical_df


if __name__ == "__main__":
    TEST0_FAST_SPHERE_RESULTS_DF, TEST0_FAST_SPHERE_CRITICAL_DF = (
        run_fast_sphere_minimum_prefilter()
    )

In [ ]:
"""
Test 0 fast quantile prefilter: minimum trap potential on analytically
matched Rutherford r_min_threshold spheres.

Run in the same Python/IPython namespace as the trap model, where

    potential_energy(points, mass, charge_fraction, component=...)

and the globals m_ion, Z_ion, K, and e are already defined. Typical use:

    %run -i test0_fast_quantile_sphere_reject_v10.py

For each configured Maxwell-Boltzmann speed quantile v_q:

1. Compute the exact free-space Rutherford closest-approach threshold radius
   R(v) for the chosen ion recoil threshold.
2. Minimize the selected trap potential on the sphere |r| = R(v).
3. Compare the incoming absolute energy

       E_in(v) = U_REFERENCE_J + 0.5 * M_DM_KG * v**2

   with a conservative numerical lower estimate of the minimum sphere
   potential.

If E_in is below the minimum potential everywhere on the sphere, the DM cannot
cross that sphere in a static conservative potential and the speed can be
rejected.

This is a fast, one-sided prefilter. Passing the test does not prove the sphere
is reachable because a higher saddle/barrier may exist before the sphere.
The code scans integer Maxwell-Boltzmann percentiles from 1% through 99%,
plots the binary rejection result, and reports every adjacent percentile pair
where the decision changes. Those adjacent speeds bracket the transition; no
continuous root finder is used.
"""

from __future__ import annotations

import math
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import maxwell


SCRIPT_VERSION = "2026-07-15-fast-quantile-sphere-prefilter-v10"


# ============================================================
# USER SETTINGS
# ============================================================

M_DM_KG = float(USER_M_DM_KG)
EPS = float(USER_EPS)
T_DM_K = float(USER_T_DM_K)
K_B = k_B
POTENTIAL_COMPONENT = "trap"

# Incoming absolute potential-energy reference. Use zero only when the selected
# trap potential is defined to approach zero in the incoming/asymptotic region.
U_REFERENCE_J = 0.0

RUTHERFORD_ION_ENERGY_THRESHOLD_J = float(USER_TARGET_ION_ENERGY_J)

# Scan the integer Maxwell-Boltzmann percentiles 1%, 2%, ..., 99%.
# Speeds are calculated exactly from the 3D Maxwell speed distribution for
# M_DM_KG and T_DM_K. No rounded speed-radius pairs are used.
QUANTILE_PERCENT_MIN = int(USER_MB_QUANTILE_MIN_PERCENT)
QUANTILE_PERCENT_MAX = int(USER_MB_QUANTILE_MAX_PERCENT)
QUANTILE_PERCENT_STEP = int(USER_MB_QUANTILE_STEP_PERCENT)

# Optional extra speeds evaluated in addition to the quantile scan. Leave empty
# for the simplest workflow.
EXTRA_SPEEDS_M_S: tuple[float, ...] = tuple(
    float(value) for value in USER_TEST0_EXTRA_SPEEDS_M_S
)

# Deterministic angular resolutions. Each level is followed by local continuous
# optimization from the lowest sampled directions. These defaults are reduced
# from the single-speed version because 99 spheres are evaluated. Increase to
# (4096, 16384) and 16 seeds for a final high-resolution confirmation near a
# reported transition bracket.
SPHERE_SAMPLE_COUNTS = (2048, 8192)
LOCAL_OPTIMIZATION_SEEDS = 8
LOCAL_OPTIMIZATION_MAXITER = 250
POTENTIAL_EVAL_CHUNK_SIZE = 100_000

# The sampled/refined minimum is not a mathematically rigorous global lower
# bound. To avoid false rejection, lower it by a convergence margin based on
# the spread between angular resolutions.
SPHERE_MIN_SAFETY_FACTOR = 2.0
RELATIVE_SPHERE_MIN_MARGIN = 0.0
USER_EXTRA_SPHERE_MIN_MARGIN_J = 0.0

OUTPUT_PREFIX = TEST0_QUANTILE_OUTPUT_PREFIX
SAVE_RESULTS_CSV = True
SAVE_REJECTION_PLOT = True
SHOW_PLOTS = True


# ============================================================
# ENVIRONMENT AND POTENTIAL EVALUATION
# ============================================================


def progress(message: str = "") -> None:
    print(message, flush=True)


def validate_environment() -> None:
    missing = [
        name
        for name in ("potential_energy", "m_ion", "Z_ion", "K", "e")
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            "Missing required trap-model globals: " + ", ".join(missing)
        )
    if M_DM_KG <= 0.0:
        raise ValueError("M_DM_KG must be positive")
    if T_DM_K <= 0.0:
        raise ValueError("T_DM_K must be positive")
    if EPS <= 0.0:
        raise ValueError("EPS must be positive for the repulsive branch")
    if RUTHERFORD_ION_ENERGY_THRESHOLD_J <= 0.0:
        raise ValueError("RUTHERFORD_ION_ENERGY_THRESHOLD_J must be positive")
    if not SPHERE_SAMPLE_COUNTS:
        raise ValueError("SPHERE_SAMPLE_COUNTS cannot be empty")
    if any(int(n) < 32 for n in SPHERE_SAMPLE_COUNTS):
        raise ValueError("Every sphere sample count must be at least 32")
    if not (0 < QUANTILE_PERCENT_MIN <= QUANTILE_PERCENT_MAX < 100):
        raise ValueError("Quantile percentages must satisfy 0 < min <= max < 100")
    if QUANTILE_PERCENT_STEP <= 0:
        raise ValueError("QUANTILE_PERCENT_STEP must be positive")


def evaluate_potential_points(
    xyz: np.ndarray,
    mass: float = M_DM_KG,
    charge_fraction: float = EPS,
    component: str = POTENTIAL_COMPONENT,
) -> np.ndarray:
    """Evaluate potential energy in vectorized chunks with scalar fallback."""
    xyz = np.asarray(xyz, dtype=float).reshape(-1, 3)
    values = np.full(xyz.shape[0], np.nan, dtype=float)

    for start in range(0, xyz.shape[0], POTENTIAL_EVAL_CHUNK_SIZE):
        stop = min(start + POTENTIAL_EVAL_CHUNK_SIZE, xyz.shape[0])
        points = xyz[start:stop]
        try:
            with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
                chunk = np.asarray(
                    potential_energy(
                        points,
                        mass,
                        charge_fraction,
                        component=component,
                    ),
                    dtype=float,
                ).reshape(-1)
            if chunk.size != points.shape[0]:
                raise ValueError("Unexpected vectorized potential shape")
        except Exception:
            scalar_values = []
            for point in points:
                with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
                    value = potential_energy(
                        point,
                        mass,
                        charge_fraction,
                        component=component,
                    )
                scalar_values.append(float(np.asarray(value).reshape(-1)[0]))
            chunk = np.asarray(scalar_values, dtype=float)
        values[start:stop] = chunk

    return values


def evaluate_one_point(point: np.ndarray) -> float:
    return float(evaluate_potential_points(np.asarray(point)[None, :])[0])


# ============================================================
# EXACT RUTHERFORD SPEED-RADIUS RELATION
# ============================================================


def rutherford_threshold_from_speed(speed_m_s: float) -> dict[str, Any]:
    """Exact free-space threshold closest approach for one incoming speed."""
    speed = float(speed_m_s)
    if not np.isfinite(speed) or speed <= 0.0:
        raise ValueError("Speed must be finite and positive")

    ion_mass = float(globals()["m_ion"])
    ion_charge = float(globals()["Z_ion"])
    coulomb_k = (
        float(globals()["K"])
        * ion_charge
        * EPS
        * float(globals()["e"]) ** 2
    )
    if ion_mass <= 0.0:
        raise ValueError("m_ion must be positive")
    if not np.isfinite(coulomb_k) or coulomb_k <= 0.0:
        raise ValueError("A finite positive repulsive Coulomb coupling is required")

    reduced_mass = M_DM_KG * ion_mass / (M_DM_KG + ion_mass)
    threshold = float(RUTHERFORD_ION_ENERGY_THRESHOLD_J)
    maximum_recoil = 2.0 * reduced_mass**2 * speed**2 / ion_mass
    kinematic_min_speed = math.sqrt(
        threshold * ion_mass / (2.0 * reduced_mass**2)
    )

    detectable = bool(
        maximum_recoil >= threshold * (1.0 - 64.0 * np.finfo(float).eps)
    )
    if not detectable:
        return {
            "speed_m_s": speed,
            "r_min_threshold_m": np.nan,
            "r_min_threshold_um": np.nan,
            "impact_parameter_threshold_m": np.nan,
            "impact_parameter_threshold_um": np.nan,
            "maximum_ion_recoil_J": maximum_recoil,
            "kinematic_min_speed_m_s": kinematic_min_speed,
            "reduced_mass_kg": reduced_mass,
            "coulomb_coupling_J_m": coulomb_k,
            "rutherford_detectable": False,
            "pair_status": "below_head_on_recoil_threshold",
        }

    # a is one-half the head-on closest approach.
    a = coulomb_k / (reduced_mass * speed**2)
    radial_term = (
        coulomb_k
        / speed
        * math.sqrt(2.0 / (ion_mass * threshold))
    )
    radius = a + radial_term

    ratio = max(maximum_recoil / threshold - 1.0, 0.0)
    impact_parameter = a * math.sqrt(ratio)

    return {
        "speed_m_s": speed,
        "r_min_threshold_m": radius,
        "r_min_threshold_um": radius * 1e6,
        "impact_parameter_threshold_m": impact_parameter,
        "impact_parameter_threshold_um": impact_parameter * 1e6,
        "maximum_ion_recoil_J": maximum_recoil,
        "kinematic_min_speed_m_s": kinematic_min_speed,
        "reduced_mass_kg": reduced_mass,
        "coulomb_coupling_J_m": coulomb_k,
        "rutherford_detectable": True,
        "pair_status": "analytic_from_speed",
    }


# ============================================================
# SPHERE MINIMUM
# ============================================================


def fibonacci_sphere_directions(n_points: int) -> np.ndarray:
    """Approximately uniform deterministic unit vectors on S^2."""
    n = int(n_points)
    i = np.arange(n, dtype=float) + 0.5
    z = 1.0 - 2.0 * i / n
    golden_angle = math.pi * (3.0 - math.sqrt(5.0))
    phi = golden_angle * i
    radial = np.sqrt(np.maximum(1.0 - z**2, 0.0))
    return np.column_stack(
        [radial * np.cos(phi), radial * np.sin(phi), z]
    )


def direction_to_angles(direction: np.ndarray) -> np.ndarray:
    unit = np.asarray(direction, dtype=float)
    unit = unit / np.linalg.norm(unit)
    theta = math.acos(float(np.clip(unit[2], -1.0, 1.0)))
    phi = math.atan2(float(unit[1]), float(unit[0])) % (2.0 * math.pi)
    return np.array([theta, phi], dtype=float)


def angles_to_direction(theta_phi: np.ndarray) -> np.ndarray:
    theta = float(theta_phi[0])
    phi = float(theta_phi[1]) % (2.0 * math.pi)
    sin_theta = math.sin(theta)
    return np.array(
        [sin_theta * math.cos(phi), sin_theta * math.sin(phi), math.cos(theta)],
        dtype=float,
    )


def minimize_potential_on_sphere_once(
    radius_m: float,
    sample_count: int,
) -> dict[str, Any]:
    """Sample the sphere and refine the best directions continuously."""
    radius = float(radius_m)
    directions = fibonacci_sphere_directions(sample_count)
    points = radius * directions
    sampled_U = evaluate_potential_points(points)
    finite = np.isfinite(sampled_U)
    if not np.any(finite):
        raise RuntimeError(
            f"No finite {POTENTIAL_COMPONENT!r} potential values on sphere "
            f"R={radius * 1e6:.6f} um"
        )

    finite_indices = np.flatnonzero(finite)
    order = finite_indices[np.argsort(sampled_U[finite_indices])]
    n_seeds = min(int(LOCAL_OPTIMIZATION_SEEDS), len(order))
    seed_indices = order[:n_seeds]

    best_U = float(sampled_U[seed_indices[0]])
    best_point = points[seed_indices[0]].copy()
    best_direction = directions[seed_indices[0]].copy()
    best_optimizer_success = False
    best_optimizer_message = "sampled minimum"
    total_nfev = 0

    finite_scale_values = sampled_U[finite]
    energy_scale = max(
        float(np.ptp(finite_scale_values)),
        float(np.max(np.abs(finite_scale_values))),
        1.0e-40,
    )

    def objective(theta_phi: np.ndarray) -> float:
        direction = angles_to_direction(theta_phi)
        value = evaluate_one_point(radius * direction)
        if not np.isfinite(value):
            return 1.0e100
        return value / energy_scale

    for seed_index in seed_indices:
        initial = direction_to_angles(directions[seed_index])
        result = minimize(
            objective,
            x0=initial,
            method="Powell",
            bounds=((0.0, math.pi), (0.0, 2.0 * math.pi)),
            options={
                "maxiter": int(LOCAL_OPTIMIZATION_MAXITER),
                "xtol": 1.0e-11,
                "ftol": 1.0e-12,
            },
        )
        total_nfev += int(getattr(result, "nfev", 0))
        if not np.isfinite(result.fun):
            continue
        direction = angles_to_direction(result.x)
        point = radius * direction
        value = evaluate_one_point(point)
        if np.isfinite(value) and value < best_U:
            best_U = float(value)
            best_point = point
            best_direction = direction
            best_optimizer_success = bool(result.success)
            best_optimizer_message = str(result.message)

    sample_min_index = int(order[0])
    return {
        "sample_count": int(sample_count),
        "sampled_min_U_J": float(sampled_U[sample_min_index]),
        "sampled_min_point_m": points[sample_min_index].copy(),
        "refined_min_U_J": best_U,
        "refined_min_point_m": best_point,
        "refined_min_direction": best_direction,
        "optimizer_success": best_optimizer_success,
        "optimizer_message": best_optimizer_message,
        "optimizer_total_nfev": total_nfev,
        "finite_fraction": float(np.mean(finite)),
        "sample_U_min_J": float(np.min(finite_scale_values)),
        "sample_U_max_J": float(np.max(finite_scale_values)),
        "sample_U_mean_J": float(np.mean(finite_scale_values)),
    }


def conservative_minimum_potential_on_sphere(radius_m: float) -> dict[str, Any]:
    """Multiresolution sphere minimum and conservative numerical lower value."""
    levels = [
        minimize_potential_on_sphere_once(radius_m, int(sample_count))
        for sample_count in SPHERE_SAMPLE_COUNTS
    ]
    values = np.array([level["refined_min_U_J"] for level in levels], dtype=float)
    selected_index = int(np.argmin(values))
    selected = levels[selected_index]

    spread = float(np.max(values) - np.min(values)) if len(values) > 1 else 0.0
    relative_margin = RELATIVE_SPHERE_MIN_MARGIN * max(
        abs(float(np.min(values))), 1.0e-40
    )
    total_margin = (
        SPHERE_MIN_SAFETY_FACTOR * spread
        + relative_margin
        + USER_EXTRA_SPHERE_MIN_MARGIN_J
    )
    conservative_lower = float(np.min(values) - total_margin)

    return {
        "sphere_min_U_estimate_J": float(np.min(values)),
        "sphere_min_U_conservative_lower_J": conservative_lower,
        "sphere_min_resolution_spread_J": spread,
        "sphere_min_convergence_margin_J": SPHERE_MIN_SAFETY_FACTOR * spread,
        "sphere_min_relative_margin_J": relative_margin,
        "sphere_min_user_margin_J": USER_EXTRA_SPHERE_MIN_MARGIN_J,
        "sphere_min_total_margin_J": total_margin,
        "sphere_min_point_m": selected["refined_min_point_m"],
        "sphere_min_direction": selected["refined_min_direction"],
        "sphere_min_selected_sample_count": selected["sample_count"],
        "sphere_min_optimizer_success": selected["optimizer_success"],
        "sphere_min_optimizer_message": selected["optimizer_message"],
        "sphere_min_levels": levels,
    }


# ============================================================
# SPEED TEST AND QUANTILE SCAN
# ============================================================


_EVALUATION_CACHE: dict[float, dict[str, Any]] = {}


def evaluate_speed(speed_m_s: float, use_cache: bool = True) -> dict[str, Any]:
    speed = float(speed_m_s)
    cache_key = float(np.float64(speed))
    if use_cache and cache_key in _EVALUATION_CACHE:
        return dict(_EVALUATION_CACHE[cache_key])

    pair = rutherford_threshold_from_speed(speed)
    kinetic = 0.5 * M_DM_KG * speed**2
    incoming_total = U_REFERENCE_J + kinetic

    if not pair["rutherford_detectable"]:
        row = {
            **pair,
            "kinetic_energy_J": kinetic,
            "U_reference_J": U_REFERENCE_J,
            "incoming_total_energy_J": incoming_total,
            "sphere_min_U_estimate_J": np.nan,
            "sphere_min_U_conservative_lower_J": np.nan,
            "energy_residual_safe_J": np.nan,
            "minimum_speed_for_fixed_sphere_safe_m_s": np.nan,
            "reject_speed": True,
            "reject_numeric": 1,
            "reject_reason": "below_head_on_recoil_threshold",
        }
        _EVALUATION_CACHE[cache_key] = dict(row)
        return row

    radius = float(pair["r_min_threshold_m"])
    sphere = conservative_minimum_potential_on_sphere(radius)
    U_min_safe = float(sphere["sphere_min_U_conservative_lower_J"])
    residual = U_min_safe - incoming_total
    tolerance = 128.0 * np.finfo(float).eps * max(
        abs(U_min_safe), abs(incoming_total), 1.0e-40
    )
    reject = bool(residual > tolerance)

    speed_required_fixed_radius = math.sqrt(
        2.0 * max(U_min_safe - U_REFERENCE_J, 0.0) / M_DM_KG
    )

    point = np.asarray(sphere["sphere_min_point_m"], dtype=float)
    direction = np.asarray(sphere["sphere_min_direction"], dtype=float)
    row = {
        **pair,
        "kinetic_energy_J": kinetic,
        "U_reference_J": U_REFERENCE_J,
        "incoming_total_energy_J": incoming_total,
        "sphere_min_U_estimate_J": sphere["sphere_min_U_estimate_J"],
        "sphere_min_U_conservative_lower_J": U_min_safe,
        "sphere_min_resolution_spread_J": sphere[
            "sphere_min_resolution_spread_J"
        ],
        "sphere_min_convergence_margin_J": sphere[
            "sphere_min_convergence_margin_J"
        ],
        "sphere_min_total_margin_J": sphere["sphere_min_total_margin_J"],
        "sphere_min_x_m": point[0],
        "sphere_min_y_m": point[1],
        "sphere_min_z_m": point[2],
        "sphere_min_x_um": point[0] * 1e6,
        "sphere_min_y_um": point[1] * 1e6,
        "sphere_min_z_um": point[2] * 1e6,
        "sphere_min_direction_x": direction[0],
        "sphere_min_direction_y": direction[1],
        "sphere_min_direction_z": direction[2],
        "sphere_min_selected_sample_count": sphere[
            "sphere_min_selected_sample_count"
        ],
        "sphere_min_optimizer_success": sphere["sphere_min_optimizer_success"],
        "energy_residual_safe_J": residual,
        "minimum_speed_for_fixed_sphere_safe_m_s": speed_required_fixed_radius,
        "reject_speed": reject,
        "reject_numeric": int(reject),
        "reject_reason": (
            "incoming_energy_below_minimum_sphere_potential"
            if reject
            else "not_rejected_by_minimum_sphere_potential"
        ),
    }
    _EVALUATION_CACHE[cache_key] = dict(row)
    return row


def maxwell_quantile_speed(quantile_fraction: float) -> float:
    q = float(quantile_fraction)
    if not 0.0 < q < 1.0:
        raise ValueError("Quantile fraction must lie strictly between zero and one")
    scale = math.sqrt(K_B * T_DM_K / M_DM_KG)
    return float(maxwell.ppf(q, scale=scale))


def quantile_percentages() -> np.ndarray:
    values = np.arange(
        int(QUANTILE_PERCENT_MIN),
        int(QUANTILE_PERCENT_MAX) + 1,
        int(QUANTILE_PERCENT_STEP),
        dtype=int,
    )
    if values.size == 0:
        raise RuntimeError("No quantiles were generated")
    return values


def scan_maxwell_quantiles() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for percent in quantile_percentages():
        fraction = float(percent) / 100.0
        speed = maxwell_quantile_speed(fraction)
        row = evaluate_speed(speed)
        row.update(
            {
                "quantile_percent": int(percent),
                "quantile_fraction": fraction,
                "scan_source": "maxwell_quantile",
            }
        )
        rows.append(row)
        progress(
            f"  q={percent:02d}%  v={speed:12.6f} m/s  "
            f"r_min={row['r_min_threshold_um']:12.6f} um  "
            f"reject={bool(row['reject_speed'])}"
            if row["rutherford_detectable"]
            else (
                f"  q={percent:02d}%  v={speed:12.6f} m/s  "
                "r_min=unavailable  reject=True (below recoil threshold)"
            )
        )
    return pd.DataFrame(rows)


def find_adjacent_decision_changes(scan_df: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "transition_type",
        "lower_quantile_percent",
        "upper_quantile_percent",
        "lower_speed_m_s",
        "upper_speed_m_s",
        "speed_bracket_width_m_s",
        "midpoint_speed_estimate_m_s",
        "lower_reject_speed",
        "upper_reject_speed",
        "lower_r_min_threshold_um",
        "upper_r_min_threshold_um",
        "lower_energy_residual_safe_J",
        "upper_energy_residual_safe_J",
    ]
    transitions: list[dict[str, Any]] = []
    ordered = scan_df.sort_values("quantile_percent").reset_index(drop=True)
    for index in range(len(ordered) - 1):
        left = ordered.iloc[index]
        right = ordered.iloc[index + 1]
        left_reject = bool(left["reject_speed"])
        right_reject = bool(right["reject_speed"])
        if left_reject == right_reject:
            continue

        left_speed = float(left["speed_m_s"])
        right_speed = float(right["speed_m_s"])
        transitions.append(
            {
                "transition_type": (
                    "reject_to_keep" if left_reject and not right_reject
                    else "keep_to_reject"
                ),
                "lower_quantile_percent": int(left["quantile_percent"]),
                "upper_quantile_percent": int(right["quantile_percent"]),
                "lower_speed_m_s": left_speed,
                "upper_speed_m_s": right_speed,
                "speed_bracket_width_m_s": right_speed - left_speed,
                "midpoint_speed_estimate_m_s": 0.5 * (left_speed + right_speed),
                "lower_reject_speed": left_reject,
                "upper_reject_speed": right_reject,
                "lower_r_min_threshold_um": float(left["r_min_threshold_um"]),
                "upper_r_min_threshold_um": float(right["r_min_threshold_um"]),
                "lower_energy_residual_safe_J": float(
                    left["energy_residual_safe_J"]
                ) if np.isfinite(left["energy_residual_safe_J"]) else np.nan,
                "upper_energy_residual_safe_J": float(
                    right["energy_residual_safe_J"]
                ) if np.isfinite(right["energy_residual_safe_J"]) else np.nan,
            }
        )
    return pd.DataFrame(transitions, columns=columns)


def plot_quantile_rejection_scan(
    scan_df: pd.DataFrame,
    transitions_df: pd.DataFrame,
) -> None:
    if not (SAVE_REJECTION_PLOT or SHOW_PLOTS):
        return

    ordered = scan_df.sort_values("speed_m_s")
    fig, ax = plt.subplots(figsize=(8.0, 5.2))
    ax.step(
        ordered["speed_m_s"],
        ordered["reject_numeric"],
        where="mid",
        linewidth=1.4,
    )
    ax.scatter(
        ordered["speed_m_s"],
        ordered["reject_numeric"],
        s=18,
    )
    for _, transition in transitions_df.iterrows():
        ax.axvspan(
            transition["lower_speed_m_s"],
            transition["upper_speed_m_s"],
            alpha=0.18,
        )

    ax.set_xlabel("Maxwell-Boltzmann quantile speed [m/s]")
    ax.set_ylabel("Rejected (1=yes, 0=no)")
    ax.set_yticks([0, 1], labels=["Keep", "Reject"])
    ax.set_title("Fast sphere-minimum rejection across MB quantiles")
    ax.grid(True, axis="x", alpha=0.25)
    fig.tight_layout()

    if SAVE_REJECTION_PLOT:
        fig.savefig(
            f"{OUTPUT_PREFIX}_quantile_rejection_plot.png",
            dpi=220,
            bbox_inches="tight",
        )
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)


# ============================================================
# MAIN
# ============================================================


def run_fast_quantile_sphere_prefilter() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    validate_environment()
    progress(f"Test 0 script version: {SCRIPT_VERSION}")
    progress(
        "Scanning Maxwell-Boltzmann integer percentiles from "
        f"{QUANTILE_PERCENT_MIN}% through {QUANTILE_PERCENT_MAX}% in steps of "
        f"{QUANTILE_PERCENT_STEP}%."
    )
    progress(
        "Each speed uses the exact analytic Rutherford r_min_threshold and the "
        "conservative minimum trap potential on that sphere."
    )

    scan_df = scan_maxwell_quantiles()
    transitions_df = find_adjacent_decision_changes(scan_df)

    extra_rows: list[dict[str, Any]] = []
    for speed in EXTRA_SPEEDS_M_S:
        row = evaluate_speed(float(speed))
        row.update(
            {
                "quantile_percent": np.nan,
                "quantile_fraction": np.nan,
                "scan_source": "extra_speed",
            }
        )
        extra_rows.append(row)
    extra_df = pd.DataFrame(extra_rows)

    if SAVE_RESULTS_CSV:
        scan_df.to_csv(f"{OUTPUT_PREFIX}_quantile_scan.csv", index=False)
        transitions_df.to_csv(
            f"{OUTPUT_PREFIX}_decision_change_brackets.csv", index=False
        )
        if not extra_df.empty:
            extra_df.to_csv(f"{OUTPUT_PREFIX}_extra_speeds.csv", index=False)

        brackets = [
            (float(row["lower_speed_m_s"]), float(row["upper_speed_m_s"]))
            for _, row in transitions_df.iterrows()
        ]
        midpoints = [
            float(value)
            for value in transitions_df.get(
                "midpoint_speed_estimate_m_s", pd.Series(dtype=float)
            )
        ]
        Path(f"{OUTPUT_PREFIX}_next_speed_brackets.py").write_text(
            "# Generated by test0_fast_quantile_sphere_reject_v10.py\n"
            f"TRANSITION_SPEED_BRACKETS_M_S = {brackets!r}\n"
            f"MIDPOINT_SPEED_ESTIMATES_M_S = {midpoints!r}\n"
        )

    plot_quantile_rejection_scan(scan_df, transitions_df)

    progress("\nAdjacent quantile decision changes:")
    if transitions_df.empty:
        progress(
            "  none found between the scanned quantiles; the rejection decision "
            "did not change from one sampled percentile to the next"
        )
    else:
        for _, transition in transitions_df.iterrows():
            progress(
                f"  {transition['transition_type']}: "
                f"q={int(transition['lower_quantile_percent'])}% "
                f"(v={transition['lower_speed_m_s']:.9g} m/s, "
                f"reject={bool(transition['lower_reject_speed'])}) -> "
                f"q={int(transition['upper_quantile_percent'])}% "
                f"(v={transition['upper_speed_m_s']:.9g} m/s, "
                f"reject={bool(transition['upper_reject_speed'])})"
            )
            progress(
                f"    transition bracket = "
                f"[{transition['lower_speed_m_s']:.9g}, "
                f"{transition['upper_speed_m_s']:.9g}] m/s; "
                f"midpoint estimate = "
                f"{transition['midpoint_speed_estimate_m_s']:.9g} m/s"
            )

    progress(
        "\nInterpretation: a transition bracket is only bounded by two adjacent "
        "Maxwell-Boltzmann percentiles. The code intentionally does not run a "
        "root finder. Narrow the percentile range or add extra speeds around a "
        "bracket for a more precise transition."
    )
    progress(
        "reject=True is a safe one-sided result within the stated angular "
        "convergence margin. reject=False only means this fast test cannot "
        "reject; a higher path barrier may still block the sphere."
    )
    if SAVE_RESULTS_CSV:
        progress(f"Saved {OUTPUT_PREFIX}_quantile_scan.csv")
        progress(f"Saved {OUTPUT_PREFIX}_decision_change_brackets.csv")
        progress(f"Saved {OUTPUT_PREFIX}_next_speed_brackets.py")

    return scan_df, transitions_df, extra_df


if __name__ == "__main__":
    (
        TEST0_QUANTILE_SCAN_DF,
        TEST0_DECISION_CHANGE_BRACKETS_DF,
        TEST0_EXTRA_SPEEDS_DF,
    ) = run_fast_quantile_sphere_prefilter()

# Test 0.5: Adiabatic Regime Sweep


In [ ]:
"""
Test 0.75 v7: adiabatic, resonant, and Rutherford regime classifier.

This notebook-safe script sits between Test 0 v9 and Test 1. It uses the
Test-0-v9 allowed speed branches, the analytic Rutherford recoil, and a
harmonic-oscillator force-spectrum suppression model to locate a candidate
slow-speed adiabatic cutoff and a high-speed Rutherford-substitution cutoff.

Default behavior
----------------
No full ion/DM trajectory ODEs are run. There is no trap propagation and no
numerical Rutherford propagation. Each point is evaluated analytically:

    E_free(v,b)       exact Rutherford recoil energy
    tau_eff(v,b)      effective Coulomb force-pulse duration
    S_j               [x_j K_1(x_j)]^2, x_j = omega_j tau_eff
    E_trap_est        mode-projected suppressed recoil
    E_trap_upper      orientation-independent conservative envelope

A speed is rejected only when the maximum conservative envelope over the
impact-parameter scan is below the detection threshold. Retained speeds are
then classified as resonant/trap-sensitive or Rutherford-like. The Rutherford
classification is evaluated over the full detectable impact-parameter range,
not at one fixed impact parameter. The script writes the same Test-1-ready
policy CSV expected by the revised Test 1 script:

    test0p75_adiabatic_regime_test1_speed_policy.csv

Required notebook globals
-------------------------
    m_ion, Z_ion, K, e, omega_vec

FDC, FRF, and run_simulation_many_dm_single_ion are not required.
"""

from __future__ import annotations

import math
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.special import k1e
from scipy.stats import maxwell

# EMBEDDED TEST-0-v9 SPEED/RADIUS POLICY
# ============================================================
# This is intentionally self-contained for a single Jupyter notebook.
# There is no external speed-policy import.

K_B = 1.380649e-23

# Resolve the rejected interval from the Test 0 calculations executed above.
(
    V9_REJECT_LOWER_M_S,
    V9_REJECT_UPPER_M_S,
    TEST0_REJECT_INTERVAL_SOURCE,
) = resolve_test0_rejected_interval()

# Same ion recoil threshold used in the Rutherford threshold calculation.
ION_ENERGY_THRESHOLD_J = float(USER_TARGET_ION_ENERGY_J)
R_SWITCH_FACTOR = 2.0

# The Maxwell distribution is unbounded.  This is a numerical truncation, not
# a hard physical maximum.  Its omitted probability must remain excluded from
# absolute-rate normalization.
DEFAULT_UPPER_TAIL_PROBABILITY = 1.0e-10

# Representative conditional quantiles inside each *allowed* interval.
DEFAULT_LOW_INTERVAL_QUANTILES = (0.10, 0.50, 0.90, 0.99)
DEFAULT_HIGH_INTERVAL_QUANTILES = (0.25, 0.75)


def mb_scale(temperature_k: float, mass_kg: float) -> float:
    if temperature_k <= 0.0 or mass_kg <= 0.0:
        raise ValueError("temperature and mass must be positive")
    return math.sqrt(K_B * float(temperature_k) / float(mass_kg))


def reduced_mass(m_dm_kg: float, m_ion_kg: float) -> float:
    if m_dm_kg <= 0.0 or m_ion_kg <= 0.0:
        raise ValueError("masses must be positive")
    return float(m_dm_kg * m_ion_kg / (m_dm_kg + m_ion_kg))


def kinematic_min_speed_m_s(
    m_dm_kg: float,
    m_ion_kg: float,
    threshold_j: float = ION_ENERGY_THRESHOLD_J,
) -> float:
    """Minimum speed whose head-on elastic recoil can reach ``threshold_j``."""
    if threshold_j <= 0.0:
        raise ValueError("threshold_j must be positive")
    mu = reduced_mass(m_dm_kg, m_ion_kg)
    return math.sqrt(float(threshold_j) * m_ion_kg / (2.0 * mu**2))


def rutherford_r_min_threshold_m(
    speed_m_s: float,
    *,
    m_dm_kg: float,
    m_ion_kg: float,
    eps: float,
    ion_charge_number: float,
    coulomb_constant: float,
    elementary_charge_c: float,
    threshold_j: float = ION_ENERGY_THRESHOLD_J,
) -> float:
    """Exact repulsive Rutherford closest-approach threshold radius.

    Returns the largest closest approach that still transfers ``threshold_j``
    to the initially stationary ion.  ``nan`` is returned below the head-on
    kinematic threshold.
    """
    speed = float(speed_m_s)
    if not np.isfinite(speed) or speed <= 0.0:
        return float("nan")
    mu = reduced_mass(m_dm_kg, m_ion_kg)
    v_kin = kinematic_min_speed_m_s(m_dm_kg, m_ion_kg, threshold_j)
    if speed < v_kin:
        return float("nan")

    coupling = abs(
        float(coulomb_constant)
        * float(ion_charge_number)
        * float(eps)
        * float(elementary_charge_c) ** 2
    )
    if coupling <= 0.0:
        raise ValueError("The Coulomb coupling magnitude must be positive")

    return (
        coupling / (mu * speed**2)
        + coupling / speed * math.sqrt(2.0 / (m_ion_kg * threshold_j))
    )


def bounded_allowed_speed_intervals(
    *,
    m_dm_kg: float,
    m_ion_kg: float,
    temperature_k: float,
    include_high_speed_branch: bool = True,
    upper_tail_probability: float = DEFAULT_UPPER_TAIL_PROBABILITY,
    reject_lower_m_s: float = V9_REJECT_LOWER_M_S,
    reject_upper_m_s: float = V9_REJECT_UPPER_M_S,
) -> list[dict[str, float | str]]:
    """Return bounded accepted intervals implied by Test 0 v9."""
    if not (0.0 < upper_tail_probability < 1.0):
        raise ValueError("upper_tail_probability must lie in (0, 1)")
    if not reject_upper_m_s > reject_lower_m_s > 0.0:
        raise ValueError("invalid rejected interval")

    scale = mb_scale(temperature_k, m_dm_kg)
    v_kin = kinematic_min_speed_m_s(m_dm_kg, m_ion_kg)
    v_max = float(maxwell.ppf(1.0 - upper_tail_probability, scale=scale))

    intervals: list[dict[str, float | str]] = []
    low_upper = min(float(reject_lower_m_s), v_max)
    if low_upper > v_kin:
        intervals.append(
            {
                "label": "low_allowed",
                "lower_m_s": float(v_kin),
                "upper_m_s": float(low_upper),
            }
        )

    if include_high_speed_branch and v_max > reject_upper_m_s:
        intervals.append(
            {
                "label": "high_allowed",
                "lower_m_s": float(reject_upper_m_s),
                "upper_m_s": float(v_max),
            }
        )
    return intervals


def make_v9_representative_speed_cases(
    *,
    m_dm_kg: float,
    m_ion_kg: float,
    temperature_k: float,
    eps: float,
    ion_charge_number: float,
    coulomb_constant: float,
    elementary_charge_c: float,
    include_high_speed_branch: bool = True,
    upper_tail_probability: float = DEFAULT_UPPER_TAIL_PROBABILITY,
    low_interval_quantiles: Iterable[float] = DEFAULT_LOW_INTERVAL_QUANTILES,
    high_interval_quantiles: Iterable[float] = DEFAULT_HIGH_INTERVAL_QUANTILES,
    r_switch_factor: float = R_SWITCH_FACTOR,
) -> list[dict[str, float | str]]:
    """Create physically weighted representative speeds in each allowed interval."""
    intervals = bounded_allowed_speed_intervals(
        m_dm_kg=m_dm_kg,
        m_ion_kg=m_ion_kg,
        temperature_k=temperature_k,
        include_high_speed_branch=include_high_speed_branch,
        upper_tail_probability=upper_tail_probability,
    )
    if not intervals:
        raise RuntimeError("Test-0-v9 policy produced no allowed speed interval")

    scale = mb_scale(temperature_k, m_dm_kg)
    interval_quantiles = {
        "low_allowed": tuple(float(q) for q in low_interval_quantiles),
        "high_allowed": tuple(float(q) for q in high_interval_quantiles),
    }

    interval_probabilities: dict[str, float] = {}
    for interval in intervals:
        lo = float(interval["lower_m_s"])
        hi = float(interval["upper_m_s"])
        interval_probabilities[str(interval["label"])] = float(
            maxwell.cdf(hi, scale=scale) - maxwell.cdf(lo, scale=scale)
        )
    total_allowed_probability = float(sum(interval_probabilities.values()))
    if total_allowed_probability <= 0.0:
        raise RuntimeError("Allowed MB probability is zero")

    cases: list[dict[str, float | str]] = []
    for interval in intervals:
        label = str(interval["label"])
        lo = float(interval["lower_m_s"])
        hi = float(interval["upper_m_s"])
        quantiles = interval_quantiles[label]
        if not quantiles:
            continue
        if any(not (0.0 < q < 1.0) for q in quantiles):
            raise ValueError(f"{label} quantiles must lie strictly in (0, 1)")

        cdf_lo = float(maxwell.cdf(lo, scale=scale))
        cdf_hi = float(maxwell.cdf(hi, scale=scale))
        probability = interval_probabilities[label]
        n_cases = len(quantiles)
        for q in quantiles:
            unconditional_q = cdf_lo + q * (cdf_hi - cdf_lo)
            unconditional_q = min(
                max(unconditional_q, np.nextafter(0.0, 1.0)),
                np.nextafter(1.0, 0.0),
            )
            speed = float(maxwell.ppf(unconditional_q, scale=scale))
            radius = rutherford_r_min_threshold_m(
                speed,
                m_dm_kg=m_dm_kg,
                m_ion_kg=m_ion_kg,
                eps=eps,
                ion_charge_number=ion_charge_number,
                coulomb_constant=coulomb_constant,
                elementary_charge_c=elementary_charge_c,
            )
            key = f"{label}_q{q:.3f}"
            cases.append(
                {
                    "key": key,
                    "quantile": float(q),
                    "conditional_quantile": float(q),
                    "unconditional_quantile": float(unconditional_q),
                    "speed_m_s": speed,
                    "v_inf_m_s": speed,
                    "speed_interval_label": label,
                    "speed_interval_lower_m_s": lo,
                    "speed_interval_upper_m_s": hi,
                    "speed_interval_probability": probability,
                    "total_allowed_speed_probability": total_allowed_probability,
                    "conditional_speed_weight": (
                        probability / total_allowed_probability / n_cases
                    ),
                    "unconditional_speed_weight": probability / n_cases,
                    "tail_probability": total_allowed_probability,
                    "minimum_speed_m_s": float(intervals[0]["lower_m_s"]),
                    "maximum_speed_m_s": float(intervals[-1]["upper_m_s"]),
                    "r_min_threshold_m": radius,
                    "r_min_threshold_um": radius * 1.0e6,
                    "R_full_m": radius,
                    "R_full_um": radius * 1.0e6,
                    "R_switch_factor": float(r_switch_factor),
                    "R_switch_m": float(r_switch_factor) * radius,
                    "R_switch_um": float(r_switch_factor) * radius * 1.0e6,
                }
            )

    cases.sort(key=lambda row: float(row["speed_m_s"]))
    return cases


def speed_is_allowed_v9(
    speed_m_s: float,
    *,
    lower_bound_m_s: float,
    upper_bound_m_s: float,
    reject_lower_m_s: float = V9_REJECT_LOWER_M_S,
    reject_upper_m_s: float = V9_REJECT_UPPER_M_S,
) -> bool:
    speed = float(speed_m_s)
    return bool(
        np.isfinite(speed)
        and lower_bound_m_s <= speed <= upper_bound_m_s
        and not (reject_lower_m_s < speed < reject_upper_m_s)
    )


def ensure_speed_dependent_radius_columns(
    df: pd.DataFrame,
    *,
    m_ion_kg: float,
    ion_charge_number: float,
    coulomb_constant: float,
    elementary_charge_c: float,
    speed_column: str = "v_inf_m_s",
    r_switch_factor: float = R_SWITCH_FACTOR,
    consistency_rtol: float = 1.0e-9,
) -> pd.DataFrame:
    """Calculate and validate per-row Rutherford ``R_full`` and ``R_switch``."""
    output = df.copy()
    if speed_column not in output.columns:
        for fallback in ("speed_m_s", "v_far_m_s"):
            if fallback in output.columns:
                speed_column = fallback
                break
        else:
            raise ValueError("No asymptotic speed column was found")
    mass_column = "m_dm_kg" if "m_dm_kg" in output.columns else "m_dm"
    if mass_column not in output.columns or "eps" not in output.columns:
        raise ValueError("Input must contain m_dm_kg/m_dm and eps")

    speeds = pd.to_numeric(output[speed_column], errors="coerce").to_numpy(float)
    masses = pd.to_numeric(output[mass_column], errors="coerce").to_numpy(float)
    charges = pd.to_numeric(output["eps"], errors="coerce").to_numpy(float)
    calculated = np.array(
        [
            rutherford_r_min_threshold_m(
                speed,
                m_dm_kg=mass,
                m_ion_kg=m_ion_kg,
                eps=charge,
                ion_charge_number=ion_charge_number,
                coulomb_constant=coulomb_constant,
                elementary_charge_c=elementary_charge_c,
            )
            for speed, mass, charge in zip(speeds, masses, charges)
        ],
        dtype=float,
    )
    if not np.all(np.isfinite(calculated) & (calculated > 0.0)):
        raise ValueError(
            "At least one row lies below the Rutherford kinematic threshold "
            "or has invalid speed/mass/charge data"
        )

    for existing_column in ("r_min_threshold_m", "R_full_m"):
        if existing_column in output.columns:
            existing = pd.to_numeric(
                output[existing_column], errors="coerce"
            ).to_numpy(float)
            finite = np.isfinite(existing)
            if np.any(
                finite
                & ~np.isclose(
                    existing,
                    calculated,
                    rtol=consistency_rtol,
                    atol=1.0e-15,
                )
            ):
                raise ValueError(
                    f"{existing_column} is inconsistent with the analytic "
                    "speed-dependent Rutherford radius"
                )

    output["r_min_threshold_m"] = calculated
    output["r_min_threshold_um"] = calculated * 1.0e6
    output["R_full_m"] = calculated
    output["R_full_um"] = calculated * 1.0e6
    output["R_switch_factor"] = float(r_switch_factor)
    output["R_switch_m"] = float(r_switch_factor) * calculated
    output["R_switch_um"] = float(r_switch_factor) * calculated * 1.0e6
    return output

SCRIPT_VERSION = "2026-07-16-test0p75-vb-reach-aware-policy-v10"

# ============================================================
# USER SETTINGS
# ============================================================

M_DM_KG = float(USER_M_DM_KG)
EPS = float(USER_EPS)
T_DM_K = float(USER_T_DM_K)
ENERGY_THRESHOLD_J = ION_ENERGY_THRESHOLD_J

INCLUDE_HIGH_SPEED_BRANCH = True
UPPER_TAIL_PROBABILITY = float(USER_SPEED_UPPER_TAIL_PROBABILITY)

# Analytic speed scan. This is inexpensive, so the default is denser than v5.
N_LOW_INTERVAL_QUANTILES = 40
N_HIGH_INTERVAL_QUANTILES = 12
ADD_ANALYTIC_CROSSOVER_SPEEDS = True
CROSSOVER_SPEED_MULTIPLIERS = (0.35, 0.50, 0.75, 1.0, 1.5, 2.0, 3.0)

# Output rows use this compact b grid. Rejection/refinement uses a denser grid.
B_OVER_B_THRESHOLD = (
    0.0,
    0.0025,
    0.005,
    0.01,
    0.02,
    0.03,
    0.05,
    0.10,
    0.20,
    0.30,
    0.50,
    0.75,
    1.00,
    1.25,
    1.50,
    2.00,
)
REFINEMENT_B_GRID_POINTS = 320
REFINEMENT_B_MAX_OVER_THRESHOLD = 2.0

# Directions are used for the mode-projected estimate. The rejection decision
# uses the orientation-independent upper envelope, so it does not depend on
# whether this direction list is exhaustive.
INCOMING_DIRECTIONS = (
    ("+x", (1.0, 0.0, 0.0)),
    ("+y", (0.0, 1.0, 0.0)),
    ("+z", (0.0, 0.0, 1.0)),
    ("xyz", (1.0, 1.0, 1.0)),
)
IMPACT_PSI_FRACTIONS = (0.0, 0.25)

# Effective pulse duration:
#     tau_eff = INTERACTION_TIME_SCALE * sqrt(a^2 + b^2) / v
# with a = k/(mu v^2). This remains finite for b=0 and has the expected v^-3
# head-on scaling in the slow regime.
INTERACTION_TIME_SCALE = 1.0

# The force-spectrum model is approximate for strong Rutherford deflection and
# does not include direct RF energy injection. Inflate the predicted maximum
# before rejecting a speed. Increase this for a more conservative cutoff.
MODEL_UPPER_SAFETY_FACTOR = 3.0
MODEL_UPPER_ABSOLUTE_MARGIN_J = 0.0

# Explicit regime classification.
#
# The classification is fundamentally a function of both speed and impact
# parameter.  The script therefore classifies every sampled (v, b, direction)
# geometry, then creates a conservative speed-only classification by scanning
# the complete detectable range 0 <= b <= b_threshold(v).
RUTHERFORD_RELATIVE_TOLERANCE = 0.05
RUTHERFORD_CLASSIFICATION_MODE = "strict"  # "strict" or "energy_weighted"
RUTHERFORD_B_MAX_OVER_THRESHOLD = 1.0
RUTHERFORD_B_GRID_POINTS = 401

# The downstream Test 1 scan currently uses b <= 4 mm.  Test 0.75 therefore
# builds a trajectory policy over the same physical impact-parameter range.
# Increase this whenever Test 1 increases its b_max.
TRAJECTORY_POLICY_B_MAX_M = 4.0e-3
TRAJECTORY_POLICY_B_GRID_POINTS = 801
TRAJECTORY_MAP_INNER_POINTS = 121
TRAJECTORY_MAP_OUTER_POINTS = 120

# Per-geometry labels.  The intermediate class is called "resonant" here in
# the timescale sense: trap-sensitive transition between adiabatic response
# and a sudden Rutherford impulse.  This analytic model does not prove a true
# energy-enhancement resonance; full trap propagation is still recommended
# near the boundaries.
ADIABATIC_GEOMETRY_RATIO_MAX = 0.10
RESONANT_CHI_LOW = 0.30
RESONANT_CHI_HIGH = 3.0

REFINE_RUTHERFORD_TRANSITION = True
MAX_RUTHERFORD_REFINEMENT_STEPS = 30
RUTHERFORD_TRANSITION_SPEED_TOL_M_S = 0.05

# Test-1 representatives are generated separately within each retained regime
# interval so that both trap-sensitive and Rutherford-like ranges are sampled.
TEST1_RESONANT_INTERVAL_QUANTILES = (0.10, 0.50, 0.90)
TEST1_RUTHERFORD_INTERVAL_QUANTILES = (0.25, 0.75)

# Transition refinement is purely analytic and therefore fast.
REFINE_LOW_SPEED_TRANSITION = True
MAX_TRANSITION_REFINEMENT_STEPS = 30
TRANSITION_SPEED_TOL_M_S = 0.05

TEST1_LOW_INTERVAL_QUANTILES = DEFAULT_LOW_INTERVAL_QUANTILES
TEST1_HIGH_INTERVAL_QUANTILES = DEFAULT_HIGH_INTERVAL_QUANTILES

OUTPUT_PREFIX = TEST0P75_OUTPUT_PREFIX
SAVE_RESULTS = True
SAVE_PLOTS = True
SHOW_PLOTS = True
PRINT_EACH_SPEED = True


# ============================================================
# BASIC HELPERS
# ============================================================


def progress(message: str = "") -> None:
    print(message, flush=True)


def integrate_trapezoid(values: np.ndarray, x: np.ndarray) -> float:
    """NumPy-version-compatible trapezoidal integration.

    NumPy 2.x provides ``np.trapezoid`` and may not expose ``np.trapz``.
    The fallback is evaluated only on older NumPy versions.
    """
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(values, x))
    return float(np.trapz(values, x))


def require_environment() -> None:
    required = ("m_ion", "Z_ion", "K", "e", "omega_vec")
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            "Define the trap-model globals before Test 0.75: "
            + ", ".join(missing)
        )
    omega = np.asarray(globals()["omega_vec"], dtype=float).reshape(-1)
    if omega.size != 3 or not np.all(np.isfinite(omega) & (omega > 0.0)):
        raise ValueError("omega_vec must contain three positive angular frequencies")


def unit_vector(vector: Iterable[float]) -> np.ndarray:
    value = np.asarray(tuple(vector), dtype=float)
    norm = float(np.linalg.norm(value))
    if not np.isfinite(norm) or norm <= 0.0:
        raise ValueError("direction must be finite and nonzero")
    return value / norm


def impact_basis(incoming_hat: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    incoming_hat = unit_vector(incoming_hat)
    reference = np.array([0.0, 0.0, 1.0])
    if abs(float(np.dot(incoming_hat, reference))) > 0.85:
        reference = np.array([0.0, 1.0, 0.0])
    e1 = unit_vector(np.cross(incoming_hat, reference))
    e2 = unit_vector(np.cross(incoming_hat, e1))
    return e1, e2


def rutherford_parameters(speed_m_s: float) -> dict[str, float]:
    speed = float(speed_m_s)
    ion_mass = float(globals()["m_ion"])
    mu = reduced_mass(M_DM_KG, ion_mass)
    coupling = abs(
        float(globals()["K"])
        * float(globals()["Z_ion"])
        * EPS
        * float(globals()["e"]) ** 2
    )
    a_m = coupling / (mu * speed**2)
    e_head_on = 2.0 * mu**2 * speed**2 / ion_mass
    if e_head_on < ENERGY_THRESHOLD_J:
        b_threshold_m = float("nan")
    else:
        b_threshold_m = a_m * math.sqrt(
            max(e_head_on / ENERGY_THRESHOLD_J - 1.0, 0.0)
        )
    r_full_m = rutherford_r_min_threshold_m(
        speed,
        m_dm_kg=M_DM_KG,
        m_ion_kg=ion_mass,
        eps=EPS,
        ion_charge_number=float(globals()["Z_ion"]),
        coulomb_constant=float(globals()["K"]),
        elementary_charge_c=float(globals()["e"]),
        threshold_j=ENERGY_THRESHOLD_J,
    )
    return {
        "mu_kg": mu,
        "coupling_J_m": coupling,
        "a_m": a_m,
        "head_on_recoil_J": e_head_on,
        "b_threshold_m": b_threshold_m,
        "R_full_m": r_full_m,
        "R_switch_m": R_SWITCH_FACTOR * r_full_m,
    }


def analytic_rutherford_recoil_j(speed_m_s: float, b_m: float) -> float:
    params = rutherford_parameters(speed_m_s)
    a_m = float(params["a_m"])
    return float(
        params["head_on_recoil_J"]
        * a_m**2
        / (a_m**2 + float(b_m) ** 2)
    )


def x_k1(x: float | np.ndarray) -> np.ndarray:
    """Return x*K1(x) stably, with the correct x->0 limit of one."""
    values = np.asarray(x, dtype=float)
    output = np.empty_like(values)
    small = values <= 1.0e-6
    large = values >= 745.0
    middle = ~(small | large)
    output[small] = 1.0
    output[large] = 0.0
    if np.any(middle):
        xm = values[middle]
        output[middle] = xm * np.exp(-xm) * k1e(xm)
    return np.clip(output, 0.0, 1.0)


def mode_suppression_factors(
    speed_m_s: float,
    b_m: float,
    a_m: float,
) -> tuple[float, np.ndarray, np.ndarray]:
    tau_eff_s = (
        INTERACTION_TIME_SCALE
        * math.sqrt(float(a_m) ** 2 + float(b_m) ** 2)
        / float(speed_m_s)
    )
    omega = np.asarray(globals()["omega_vec"], dtype=float).reshape(3)
    x = omega * tau_eff_s
    suppression = x_k1(x) ** 2
    return tau_eff_s, x, suppression


def momentum_transfer_direction(
    incoming_hat: np.ndarray,
    b_hat: np.ndarray,
    a_m: float,
    b_m: float,
) -> tuple[float, np.ndarray]:
    """Return Rutherford deflection angle and the ion momentum-transfer direction."""
    theta = 2.0 * math.atan2(float(a_m), max(float(b_m), 0.0))
    q_vector = (
        (1.0 - math.cos(theta)) * incoming_hat
        - math.sin(theta) * b_hat
    )
    norm = float(np.linalg.norm(q_vector))
    if norm <= 1.0e-15:
        q_hat = incoming_hat.copy()
    else:
        q_hat = q_vector / norm
    return theta, q_hat


def classify_geometry_regime(
    *,
    trap_to_free_ratio: float,
    chi_modes: np.ndarray,
) -> tuple[str, bool, float]:
    """Classify one fixed-(v,b,direction) geometry.

    Returns ``(regime, resonance_timescale_overlap, chi_effective)``.
    The intermediate label ``resonant`` means trap-sensitive/transition-like;
    it does not by itself prove energy enhancement above Rutherford.
    """
    ratio = float(trap_to_free_ratio)
    chi = np.asarray(chi_modes, dtype=float).reshape(3)
    finite = chi[np.isfinite(chi)]
    chi_effective = float(np.sqrt(np.mean(finite**2))) if finite.size else np.nan
    overlap = bool(
        finite.size
        and np.any((finite >= RESONANT_CHI_LOW) & (finite <= RESONANT_CHI_HIGH))
    )
    if np.isfinite(ratio) and ratio >= 1.0 - RUTHERFORD_RELATIVE_TOLERANCE:
        return "rutherford", overlap, chi_effective
    if np.isfinite(ratio) and ratio <= ADIABATIC_GEOMETRY_RATIO_MAX:
        return "adiabatic", overlap, chi_effective
    return "resonant", overlap, chi_effective


def evaluate_geometry(
    *,
    speed_m_s: float,
    b_over_b_threshold: float,
    direction_label: str,
    incoming_direction: Iterable[float],
    psi_fraction: float,
) -> dict[str, Any]:
    params = rutherford_parameters(speed_m_s)
    b_threshold_m = float(params["b_threshold_m"])
    if not np.isfinite(b_threshold_m) or b_threshold_m < 0.0:
        raise ValueError("speed is below the Rutherford kinematic threshold")
    b_m = float(b_over_b_threshold) * b_threshold_m
    incoming_hat = unit_vector(incoming_direction)
    e1, e2 = impact_basis(incoming_hat)
    psi = 2.0 * math.pi * float(psi_fraction)
    b_hat = math.cos(psi) * e1 + math.sin(psi) * e2

    free_energy_j = analytic_rutherford_recoil_j(speed_m_s, b_m)
    tau_eff_s, x_modes, suppression = mode_suppression_factors(
        speed_m_s,
        b_m,
        float(params["a_m"]),
    )
    theta, q_hat = momentum_transfer_direction(
        incoming_hat,
        b_hat,
        float(params["a_m"]),
        b_m,
    )
    energy_fractions = q_hat**2
    trap_estimate_j = float(free_energy_j * np.dot(energy_fractions, suppression))
    orientation_free_model_j = float(free_energy_j * np.max(suppression))
    conservative_upper_j = float(
        MODEL_UPPER_SAFETY_FACTOR * orientation_free_model_j
        + MODEL_UPPER_ABSOLUTE_MARGIN_J
    )
    trap_to_free_ratio = (
        trap_estimate_j / free_energy_j if free_energy_j > 0.0 else np.nan
    )
    geometry_regime, resonance_overlap, chi_effective = classify_geometry_regime(
        trap_to_free_ratio=trap_to_free_ratio,
        chi_modes=x_modes,
    )

    return {
        "speed_m_s": float(speed_m_s),
        "b_over_b_threshold": float(b_over_b_threshold),
        "b_m": b_m,
        "b_um": b_m * 1.0e6,
        "b_threshold_m": b_threshold_m,
        "b_threshold_um": b_threshold_m * 1.0e6,
        "r_min_threshold_m": float(params["R_full_m"]),
        "r_min_threshold_um": float(params["R_full_m"]) * 1.0e6,
        "R_full_m": float(params["R_full_m"]),
        "R_full_um": float(params["R_full_m"]) * 1.0e6,
        "R_switch_m": float(params["R_switch_m"]),
        "R_switch_um": float(params["R_switch_m"]) * 1.0e6,
        "direction_label": str(direction_label),
        "incoming_x": float(incoming_hat[0]),
        "incoming_y": float(incoming_hat[1]),
        "incoming_z": float(incoming_hat[2]),
        "impact_psi_fraction": float(psi_fraction),
        "impact_psi_deg": float(360.0 * psi_fraction),
        "rutherford_scattering_angle_rad": theta,
        "rutherford_scattering_angle_deg": math.degrees(theta),
        "q_hat_x": float(q_hat[0]),
        "q_hat_y": float(q_hat[1]),
        "q_hat_z": float(q_hat[2]),
        "free_space_rutherford_energy_J": free_energy_j,
        "tau_effective_s": tau_eff_s,
        "chi_sec_x": float(x_modes[0]),
        "chi_sec_y": float(x_modes[1]),
        "chi_sec_z": float(x_modes[2]),
        "suppression_x": float(suppression[0]),
        "suppression_y": float(suppression[1]),
        "suppression_z": float(suppression[2]),
        "trap_energy_force_spectrum_J": trap_estimate_j,
        "orientation_free_model_energy_J": orientation_free_model_j,
        "conservative_trap_energy_upper_J": conservative_upper_j,
        "trap_to_free_force_spectrum_ratio": trap_to_free_ratio,
        "geometry_regime": geometry_regime,
        "resonance_timescale_overlap": resonance_overlap,
        "chi_effective_rms": chi_effective,
        "model_upper_to_free_ratio": (
            conservative_upper_j / free_energy_j if free_energy_j > 0.0 else np.nan
        ),
    }


def test0_allowed_intervals() -> list[dict[str, float | str]]:
    return bounded_allowed_speed_intervals(
        m_dm_kg=M_DM_KG,
        m_ion_kg=float(globals()["m_ion"]),
        temperature_k=T_DM_K,
        include_high_speed_branch=INCLUDE_HIGH_SPEED_BRANCH,
        upper_tail_probability=UPPER_TAIL_PROBABILITY,
        reject_lower_m_s=V9_REJECT_LOWER_M_S,
        reject_upper_m_s=V9_REJECT_UPPER_M_S,
    )


def conditional_quantile_speeds(
    lower_m_s: float,
    upper_m_s: float,
    n: int,
) -> np.ndarray:
    scale = mb_scale(T_DM_K, M_DM_KG)
    cdf_lo = float(maxwell.cdf(lower_m_s, scale=scale))
    cdf_hi = float(maxwell.cdf(upper_m_s, scale=scale))
    if n <= 0 or not cdf_hi > cdf_lo:
        return np.empty(0)
    q = np.linspace(0.01, 0.99, n)
    return np.asarray(
        maxwell.ppf(cdf_lo + q * (cdf_hi - cdf_lo), scale=scale),
        dtype=float,
    )


def build_scan_speeds() -> pd.DataFrame:
    intervals = test0_allowed_intervals()
    omega = np.asarray(globals()["omega_vec"], dtype=float).reshape(3)
    mu = reduced_mass(M_DM_KG, float(globals()["m_ion"]))
    coupling = abs(
        float(globals()["K"])
        * float(globals()["Z_ion"])
        * EPS
        * float(globals()["e"]) ** 2
    )
    crossover = (omega * coupling / mu) ** (1.0 / 3.0)

    rows: list[dict[str, Any]] = []
    crossover_candidates = np.concatenate(
        [crossover * multiplier for multiplier in CROSSOVER_SPEED_MULTIPLIERS]
    )
    for interval in intervals:
        label = str(interval["label"])
        lo = float(interval["lower_m_s"])
        hi = float(interval["upper_m_s"])
        n = (
            N_LOW_INTERVAL_QUANTILES
            if label == "low_allowed"
            else N_HIGH_INTERVAL_QUANTILES
        )
        candidates = list(conditional_quantile_speeds(lo, hi, n))
        candidates.append(hi)
        if ADD_ANALYTIC_CROSSOVER_SPEEDS:
            candidates.extend(
                float(value)
                for value in crossover_candidates
                if lo < float(value) < hi
            )
        unique = sorted(
            {
                round(float(value), 10): float(value)
                for value in candidates
                if np.isfinite(value) and lo < float(value) <= hi
            }.values()
        )
        for speed in unique:
            rows.append(
                {
                    "speed_interval_label": label,
                    "speed_interval_lower_m_s": lo,
                    "speed_interval_upper_m_s": hi,
                    "speed_m_s": speed,
                    "added_near_crossover": bool(
                        np.any(
                            np.isclose(
                                speed,
                                crossover_candidates,
                                rtol=1.0e-10,
                                atol=1.0e-10,
                            )
                        )
                    ),
                }
            )
    frame = pd.DataFrame(rows).sort_values("speed_m_s").drop_duplicates("speed_m_s")
    return frame.reset_index(drop=True)


def dense_refinement_b_fractions() -> np.ndarray:
    positive = np.geomspace(
        1.0e-6,
        REFINEMENT_B_MAX_OVER_THRESHOLD,
        REFINEMENT_B_GRID_POINTS,
    )
    return np.concatenate(([0.0], positive))


def rutherford_b_fractions() -> np.ndarray:
    """Area-uniform grid over the detectable impact disk."""
    if RUTHERFORD_B_GRID_POINTS < 3:
        raise ValueError("RUTHERFORD_B_GRID_POINTS must be at least 3")
    u = np.linspace(
        0.0,
        float(RUTHERFORD_B_MAX_OVER_THRESHOLD) ** 2,
        int(RUTHERFORD_B_GRID_POINTS),
    )
    return np.sqrt(u)


def rutherford_speed_metrics(speed_m_s: float) -> dict[str, float | bool | str]:
    """Quantify whether Rutherford energy is valid over detectable b.

    ``strict_min_ratio`` is the minimum force-spectrum/Rutherford energy ratio
    over 0 <= b <= b_threshold and over every possible momentum-transfer
    orientation.  It is therefore the conservative eventwise criterion.

    ``energy_weighted_ratio`` uses an isotropic mode average and weights by
    Rutherford deposited energy and impact area.  It is useful for rate-level
    diagnostics but is not the default substitution rule.
    """
    params = rutherford_parameters(speed_m_s)
    b_threshold_m = float(params["b_threshold_m"])
    if not np.isfinite(b_threshold_m) or b_threshold_m <= 0.0:
        return {
            "rutherford_strict_min_ratio": 0.0,
            "rutherford_area_weighted_ratio": 0.0,
            "rutherford_energy_weighted_ratio": 0.0,
            "rutherford_worst_b_over_b_threshold": np.nan,
            "rutherford_worst_chi_max": np.nan,
            "rutherford_b_over_b_threshold_max_strict": 0.0,
            "rutherford_b_max_m_strict": 0.0,
            "rutherford_detectable_area_fraction_strict": 0.0,
            "rutherford_criterion_value": 0.0,
            "rutherford_approx_valid": False,
            "rutherford_classification_mode": RUTHERFORD_CLASSIFICATION_MODE,
        }

    fractions = rutherford_b_fractions()
    strict_ratios = []
    isotropic_ratios = []
    free_energies = []
    chi_max_values = []
    for fraction in fractions:
        b_m = float(fraction) * b_threshold_m
        free_energy = analytic_rutherford_recoil_j(speed_m_s, b_m)
        _, chi_modes, suppression = mode_suppression_factors(
            speed_m_s,
            b_m,
            float(params["a_m"]),
        )
        strict_ratios.append(float(np.min(suppression)))
        isotropic_ratios.append(float(np.mean(suppression)))
        free_energies.append(float(free_energy))
        chi_max_values.append(float(np.max(chi_modes)))

    strict_ratios = np.asarray(strict_ratios, dtype=float)
    isotropic_ratios = np.asarray(isotropic_ratios, dtype=float)
    free_energies = np.asarray(free_energies, dtype=float)
    chi_max_values = np.asarray(chi_max_values, dtype=float)
    u = fractions**2

    area_denominator = integrate_trapezoid(np.ones_like(u), u)
    area_weighted = float(integrate_trapezoid(isotropic_ratios, u) / area_denominator)
    energy_denominator = integrate_trapezoid(free_energies, u)
    energy_weighted = float(
        integrate_trapezoid(free_energies * isotropic_ratios, u) / energy_denominator
    ) if energy_denominator > 0.0 else np.nan

    worst_index = int(np.argmin(strict_ratios))
    strict_min = float(strict_ratios[worst_index])
    target = 1.0 - float(RUTHERFORD_RELATIVE_TOLERANCE)

    # Largest contiguous central impact disk for which every orientation is
    # within the Rutherford tolerance.  This provides a useful b-dependent
    # substitution rule even when no speed is Rutherford-valid for the entire
    # detectable disk.
    valid_prefix = strict_ratios >= target
    first_invalid = np.flatnonzero(~valid_prefix)
    if first_invalid.size == 0:
        b_fraction_max_strict = float(fractions[-1])
    elif first_invalid[0] == 0:
        b_fraction_max_strict = 0.0
    else:
        b_fraction_max_strict = float(fractions[first_invalid[0] - 1])
    detectable_area_fraction_strict = float(
        (b_fraction_max_strict / RUTHERFORD_B_MAX_OVER_THRESHOLD) ** 2
    ) if RUTHERFORD_B_MAX_OVER_THRESHOLD > 0.0 else 0.0

    mode = str(RUTHERFORD_CLASSIFICATION_MODE).strip().lower()
    if mode == "strict":
        criterion = strict_min
    elif mode == "energy_weighted":
        criterion = energy_weighted
    else:
        raise ValueError(
            "RUTHERFORD_CLASSIFICATION_MODE must be 'strict' or 'energy_weighted'"
        )
    return {
        "rutherford_strict_min_ratio": strict_min,
        "rutherford_area_weighted_ratio": area_weighted,
        "rutherford_energy_weighted_ratio": energy_weighted,
        "rutherford_worst_b_over_b_threshold": float(fractions[worst_index]),
        "rutherford_worst_chi_max": float(chi_max_values[worst_index]),
        "rutherford_b_over_b_threshold_max_strict": b_fraction_max_strict,
        "rutherford_b_max_m_strict": b_fraction_max_strict * b_threshold_m,
        "rutherford_detectable_area_fraction_strict": detectable_area_fraction_strict,
        "rutherford_criterion_value": float(criterion),
        "rutherford_criterion_target": target,
        "rutherford_approx_valid": bool(np.isfinite(criterion) and criterion >= target),
        "rutherford_classification_mode": mode,
    }



def trajectory_regime_metrics_at_b(
    speed_m_s: float,
    b_m: float,
) -> dict[str, float | bool | str]:
    """Classify one trajectory conservatively from its exact ``(v, b)``.

    The Rutherford decision uses the minimum suppression over the three
    secular modes, so it is valid for every momentum-transfer orientation.
    The adiabatic rejection uses the orientation-independent energy upper
    envelope.  Everything between those conditions is retained as the
    trap-sensitive/resonant regime.
    """
    params = rutherford_parameters(speed_m_s)
    b_value = max(float(b_m), 0.0)
    free_energy = analytic_rutherford_recoil_j(speed_m_s, b_value)
    tau_eff, chi_modes, suppression = mode_suppression_factors(
        speed_m_s,
        b_value,
        float(params["a_m"]),
    )
    strict_ratio = float(np.min(suppression))
    isotropic_ratio = float(np.mean(suppression))
    upper_energy = float(
        MODEL_UPPER_SAFETY_FACTOR * free_energy * np.max(suppression)
        + MODEL_UPPER_ABSOLUTE_MARGIN_J
    )
    adiabatic_reject = bool(upper_energy < ENERGY_THRESHOLD_J)
    rutherford_valid = bool(
        strict_ratio >= 1.0 - RUTHERFORD_RELATIVE_TOLERANCE
    )
    if adiabatic_reject:
        regime = "adiabatic"
        energy_model = "reject_below_threshold"
    elif rutherford_valid:
        regime = "rutherford"
        energy_model = "rutherford_after_reach"
    else:
        regime = "resonant"
        energy_model = "full_coupled_after_reach"
    return {
        "trajectory_regime": regime,
        "collision_energy_regime": regime,
        "ion_energy_model": energy_model,
        "requires_dm_only_reach_screen": bool(regime != "adiabatic"),
        "use_rutherford_after_reach": bool(regime == "rutherford"),
        "requires_full_coupled_after_reach": bool(regime == "resonant"),
        # Compatibility names retained for downstream notebooks.
        "use_rutherford_final_energy": bool(regime == "rutherford"),
        "requires_full_trap_propagation": bool(regime == "resonant"),
        "test0p75_adiabatic_reject": bool(regime == "adiabatic"),
        "free_space_rutherford_energy_J": float(free_energy),
        "conservative_trap_energy_upper_J": upper_energy,
        "rutherford_strict_ratio": strict_ratio,
        "rutherford_isotropic_ratio": isotropic_ratio,
        "tau_effective_s": float(tau_eff),
        "chi_sec_x": float(chi_modes[0]),
        "chi_sec_y": float(chi_modes[1]),
        "chi_sec_z": float(chi_modes[2]),
        "suppression_x": float(suppression[0]),
        "suppression_y": float(suppression[1]),
        "suppression_z": float(suppression[2]),
    }


def trajectory_regime_boundaries(speed_m_s: float) -> dict[str, float | bool]:
    """Return the contiguous small-b Rutherford and large-b adiabatic bounds."""
    params = rutherford_parameters(speed_m_s)
    b_threshold = float(params["b_threshold_m"])
    if not np.isfinite(b_threshold) or b_threshold <= 0.0:
        return {
            "b_threshold_m": np.nan,
            "b_rutherford_max_m": 0.0,
            "b_rutherford_max_um": 0.0,
            "b_rutherford_max_over_b_threshold": 0.0,
            "b_adiabatic_min_m": 0.0,
            "b_adiabatic_min_um": 0.0,
            "b_adiabatic_min_over_b_threshold": 0.0,
            "rutherford_detectable_area_fraction": 0.0,
            "policy_b_max_m": TRAJECTORY_POLICY_B_MAX_M,
            "all_policy_b_adiabatic": True,
            "all_detectable_b_rutherford": False,
        }

    target = 1.0 - RUTHERFORD_RELATIVE_TOLERANCE
    rutherford_cap = min(float(b_threshold), float(TRAJECTORY_POLICY_B_MAX_M))

    def rutherford_residual(b_value: float) -> float:
        return float(
            trajectory_regime_metrics_at_b(speed_m_s, b_value)[
                "rutherford_strict_ratio"
            ]
            - target
        )

    r0 = rutherford_residual(0.0)
    rcap = rutherford_residual(rutherford_cap)
    if r0 < 0.0:
        b_rutherford_max = 0.0
    elif rcap >= 0.0:
        b_rutherford_max = rutherford_cap
    else:
        b_rutherford_max = float(
            brentq(
                rutherford_residual,
                0.0,
                rutherford_cap,
                xtol=1.0e-15,
                rtol=1.0e-12,
                maxiter=200,
            )
        )

    def adiabatic_residual(b_value: float) -> float:
        return float(
            trajectory_regime_metrics_at_b(speed_m_s, b_value)[
                "conservative_trap_energy_upper_J"
            ]
            - ENERGY_THRESHOLD_J
        )

    a0 = adiabatic_residual(0.0)
    amax = adiabatic_residual(float(TRAJECTORY_POLICY_B_MAX_M))
    if a0 < 0.0:
        b_adiabatic_min = 0.0
    elif amax >= 0.0:
        b_adiabatic_min = float("inf")
    else:
        b_adiabatic_min = float(
            brentq(
                adiabatic_residual,
                0.0,
                float(TRAJECTORY_POLICY_B_MAX_M),
                xtol=1.0e-15,
                rtol=1.0e-12,
                maxiter=200,
            )
        )

    b_ruth_fraction = b_rutherford_max / b_threshold
    b_adia_fraction = (
        b_adiabatic_min / b_threshold
        if np.isfinite(b_adiabatic_min)
        else float("inf")
    )
    return {
        "b_threshold_m": b_threshold,
        "b_threshold_um": b_threshold * 1.0e6,
        "b_rutherford_max_m": b_rutherford_max,
        "b_rutherford_max_um": b_rutherford_max * 1.0e6,
        "b_rutherford_max_over_b_threshold": b_ruth_fraction,
        "b_adiabatic_min_m": b_adiabatic_min,
        "b_adiabatic_min_um": (
            b_adiabatic_min * 1.0e6
            if np.isfinite(b_adiabatic_min)
            else float("inf")
        ),
        "b_adiabatic_min_over_b_threshold": b_adia_fraction,
        "rutherford_detectable_area_fraction": float(
            min(max(b_ruth_fraction, 0.0), 1.0) ** 2
        ),
        "policy_b_max_m": float(TRAJECTORY_POLICY_B_MAX_M),
        "all_policy_b_adiabatic": bool(b_adiabatic_min <= 0.0),
        "all_detectable_b_rutherford": bool(
            b_rutherford_max >= b_threshold * (1.0 - 1.0e-10)
        ),
    }


def build_vb_regime_map(speed_scan: pd.DataFrame) -> pd.DataFrame:
    """Create the explicit two-dimensional ``(v, b)`` trajectory policy."""
    rows: list[dict[str, Any]] = []
    for speed_row in speed_scan.itertuples(index=False):
        speed = float(speed_row.speed_m_s)
        params = rutherford_parameters(speed)
        b_threshold = float(params["b_threshold_m"])
        inner = np.sqrt(np.linspace(0.0, 1.0, TRAJECTORY_MAP_INNER_POINTS)) * b_threshold
        if TRAJECTORY_POLICY_B_MAX_M > b_threshold:
            outer = np.geomspace(
                max(b_threshold, 1.0e-15),
                TRAJECTORY_POLICY_B_MAX_M,
                TRAJECTORY_MAP_OUTER_POINTS,
            )
            b_values = np.unique(np.concatenate([inner, outer]))
        else:
            b_values = np.unique(inner[inner <= TRAJECTORY_POLICY_B_MAX_M])
        for b_value in b_values:
            metrics = trajectory_regime_metrics_at_b(speed, float(b_value))
            rows.append(
                {
                    "speed_interval_label": str(speed_row.speed_interval_label),
                    "speed_m_s": speed,
                    "b_m": float(b_value),
                    "b_um": float(b_value) * 1.0e6,
                    "b_over_b_threshold": float(b_value / b_threshold),
                    "b_threshold_m": b_threshold,
                    **metrics,
                }
            )
    return pd.DataFrame(rows)

def conservative_speed_envelope(speed_m_s: float) -> dict[str, float | bool]:
    params = rutherford_parameters(speed_m_s)
    b_threshold_m = float(params["b_threshold_m"])
    if not np.isfinite(b_threshold_m) or b_threshold_m < 0.0:
        return {
            "speed_m_s": float(speed_m_s),
            "max_conservative_trap_energy_upper_J": 0.0,
            "max_orientation_free_model_energy_J": 0.0,
            "max_free_space_rutherford_energy_J": 0.0,
            "maximizing_b_over_b_threshold": np.nan,
            "reject_speed_adiabatic": True,
        }

    best_upper = -np.inf
    best_model = np.nan
    best_free = np.nan
    best_fraction = np.nan
    best_tau = np.nan
    best_chi = np.full(3, np.nan)
    best_suppression = np.full(3, np.nan)
    for fraction in dense_refinement_b_fractions():
        b_m = float(fraction) * b_threshold_m
        free_energy = analytic_rutherford_recoil_j(speed_m_s, b_m)
        tau_eff, x_modes, suppression = mode_suppression_factors(
            speed_m_s,
            b_m,
            float(params["a_m"]),
        )
        model_energy = float(free_energy * np.max(suppression))
        upper_energy = float(
            MODEL_UPPER_SAFETY_FACTOR * model_energy
            + MODEL_UPPER_ABSOLUTE_MARGIN_J
        )
        if upper_energy > best_upper:
            best_upper = upper_energy
            best_model = model_energy
            best_free = free_energy
            best_fraction = float(fraction)
            best_tau = tau_eff
            best_chi = x_modes.copy()
            best_suppression = suppression.copy()

    return {
        "speed_m_s": float(speed_m_s),
        "max_conservative_trap_energy_upper_J": float(best_upper),
        "max_orientation_free_model_energy_J": float(best_model),
        "max_free_space_rutherford_energy_J": float(best_free),
        "maximizing_b_over_b_threshold": float(best_fraction),
        "maximizing_tau_effective_s": float(best_tau),
        "maximizing_chi_sec_x": float(best_chi[0]),
        "maximizing_chi_sec_y": float(best_chi[1]),
        "maximizing_chi_sec_z": float(best_chi[2]),
        "maximizing_suppression_x": float(best_suppression[0]),
        "maximizing_suppression_y": float(best_suppression[1]),
        "maximizing_suppression_z": float(best_suppression[2]),
        "reject_speed_adiabatic": bool(best_upper < ENERGY_THRESHOLD_J),
    }


def run_speed(speed_row: pd.Series | dict[str, Any]) -> tuple[pd.DataFrame, dict[str, Any]]:
    speed = float(speed_row["speed_m_s"])
    interval_label = str(speed_row["speed_interval_label"])
    rows: list[dict[str, Any]] = []
    for b_fraction in B_OVER_B_THRESHOLD:
        for direction_label, direction in INCOMING_DIRECTIONS:
            for psi_fraction in IMPACT_PSI_FRACTIONS:
                row = evaluate_geometry(
                    speed_m_s=speed,
                    b_over_b_threshold=float(b_fraction),
                    direction_label=str(direction_label),
                    incoming_direction=direction,
                    psi_fraction=float(psi_fraction),
                )
                strict_policy = trajectory_regime_metrics_at_b(
                    speed,
                    float(row["b_m"]),
                )
                row.update(
                    {
                        "speed_interval_label": interval_label,
                        "speed_interval_lower_m_s": float(
                            speed_row["speed_interval_lower_m_s"]
                        ),
                        "speed_interval_upper_m_s": float(
                            speed_row["speed_interval_upper_m_s"]
                        ),
                        "trajectory_regime_strict_v_b": strict_policy[
                            "trajectory_regime"
                        ],
                        "trajectory_ion_energy_model_strict_v_b": strict_policy[
                            "ion_energy_model"
                        ],
                    }
                )
                rows.append(row)

    frame = pd.DataFrame(rows)
    envelope = conservative_speed_envelope(speed)
    rutherford_metrics = rutherford_speed_metrics(speed)
    boundaries = trajectory_regime_boundaries(speed)
    speed_regime = (
        "adiabatic" if bool(envelope["reject_speed_adiabatic"]) else "resonant"
    )
    maximizing_sample = frame.loc[
        frame["trap_energy_force_spectrum_J"].astype(float).idxmax()
    ]
    params = rutherford_parameters(speed)
    summary = {
        "speed_m_s": speed,
        "speed_interval_label": interval_label,
        "speed_interval_lower_m_s": float(speed_row["speed_interval_lower_m_s"]),
        "speed_interval_upper_m_s": float(speed_row["speed_interval_upper_m_s"]),
        "n_analytic_geometries": len(frame),
        "max_sampled_trap_energy_force_spectrum_J": float(
            frame["trap_energy_force_spectrum_J"].max()
        ),
        "max_sampled_free_space_rutherford_energy_J": float(
            frame["free_space_rutherford_energy_J"].max()
        ),
        "min_sampled_trap_to_free_ratio": float(
            frame["trap_to_free_force_spectrum_ratio"].min()
        ),
        "max_sampled_trap_to_free_ratio": float(
            frame["trap_to_free_force_spectrum_ratio"].max()
        ),
        "sampled_maximizing_b_over_b_threshold": float(
            maximizing_sample["b_over_b_threshold"]
        ),
        "sampled_maximizing_direction_label": str(
            maximizing_sample["direction_label"]
        ),
        "sampled_maximizing_impact_psi_deg": float(
            maximizing_sample["impact_psi_deg"]
        ),
        "r_min_threshold_m": float(params["R_full_m"]),
        "r_min_threshold_um": float(params["R_full_m"]) * 1.0e6,
        "R_full_m": float(params["R_full_m"]),
        "R_full_um": float(params["R_full_m"]) * 1.0e6,
        "R_switch_m": float(params["R_switch_m"]),
        "R_switch_um": float(params["R_switch_m"]) * 1.0e6,
        "b_threshold_m": float(params["b_threshold_m"]),
        "b_threshold_um": float(params["b_threshold_m"]) * 1.0e6,
        "fraction_sampled_geometries_adiabatic": float(
            np.mean(frame["trajectory_regime_strict_v_b"].eq("adiabatic"))
        ),
        "fraction_sampled_geometries_resonant": float(
            np.mean(frame["trajectory_regime_strict_v_b"].eq("resonant"))
        ),
        "fraction_sampled_geometries_rutherford": float(
            np.mean(frame["trajectory_regime_strict_v_b"].eq("rutherford"))
        ),
        "speed_regime": speed_regime,
        "ion_energy_model": "trajectory_v_b_policy",
        "retain_for_test1": not bool(envelope["reject_speed_adiabatic"]),
        **envelope,
        **rutherford_metrics,
        **boundaries,
    }
    return frame, summary


# ============================================================
# TRANSITION AND TEST-1 POLICY
# ============================================================


def identify_low_edge_transition(
    summary_df: pd.DataFrame,
    interval_label: str,
) -> dict[str, Any]:
    subset = summary_df.loc[
        summary_df["speed_interval_label"] == interval_label
    ].sort_values("speed_m_s")
    if subset.empty:
        return {"interval_label": interval_label, "status": "no_scan_rows"}

    rows = list(subset.itertuples(index=False))
    rejected_prefix = []
    for row in rows:
        if bool(row.reject_speed_adiabatic):
            rejected_prefix.append(row)
        else:
            break

    interval_lower = float(subset["speed_interval_lower_m_s"].iloc[0])
    interval_upper = float(subset["speed_interval_upper_m_s"].iloc[0])
    if not rejected_prefix:
        return {
            "interval_label": interval_label,
            "status": "no_low_edge_rejection",
            "conservative_retained_lower_m_s": interval_lower,
        }
    if len(rejected_prefix) == len(rows):
        return {
            "interval_label": interval_label,
            "status": "all_scanned_speeds_rejected",
            "last_rejected_speed_m_s": float(rejected_prefix[-1].speed_m_s),
            "conservative_retained_lower_m_s": interval_upper,
        }

    left = rejected_prefix[-1]
    right = rows[len(rejected_prefix)]
    return {
        "interval_label": interval_label,
        "status": "reject_to_keep_bracket",
        "adiabatic_monotonic_extrapolation_to_interval_lower": True,
        "last_rejected_speed_m_s": float(left.speed_m_s),
        "first_retained_speed_m_s": float(right.speed_m_s),
        "bracket_lower_m_s": float(left.speed_m_s),
        "bracket_upper_m_s": float(right.speed_m_s),
        "conservative_retained_lower_m_s": float(left.speed_m_s),
    }


def refine_transition(transition: dict[str, Any]) -> dict[str, Any]:
    if (
        transition.get("status") != "reject_to_keep_bracket"
        or not REFINE_LOW_SPEED_TRANSITION
    ):
        return transition
    lo = float(transition["bracket_lower_m_s"])
    hi = float(transition["bracket_upper_m_s"])
    history = []
    for step in range(MAX_TRANSITION_REFINEMENT_STEPS):
        if hi - lo <= TRANSITION_SPEED_TOL_M_S:
            break
        midpoint = 0.5 * (lo + hi)
        envelope = conservative_speed_envelope(midpoint)
        rejected = bool(envelope["reject_speed_adiabatic"])
        history.append(
            {
                "step": step + 1,
                "speed_m_s": midpoint,
                "reject_speed_adiabatic": rejected,
                "max_conservative_trap_energy_upper_J": envelope[
                    "max_conservative_trap_energy_upper_J"
                ],
            }
        )
        if rejected:
            lo = midpoint
        else:
            hi = midpoint
    updated = dict(transition)
    updated.update(
        {
            "refined": True,
            "bracket_lower_m_s": lo,
            "bracket_upper_m_s": hi,
            "bracket_width_m_s": hi - lo,
            "last_rejected_speed_m_s": lo,
            "first_retained_speed_m_s": hi,
            "conservative_retained_lower_m_s": lo,
            "refinement_history": history,
        }
    )
    return updated


def identify_rutherford_high_edge_transition(
    summary_df: pd.DataFrame,
    interval_label: str,
) -> dict[str, Any]:
    """Find the low edge of a Rutherford-valid suffix in one allowed branch."""
    subset = summary_df.loc[
        summary_df["speed_interval_label"] == interval_label
    ].sort_values("speed_m_s")
    if subset.empty:
        return {"interval_label": interval_label, "status": "no_scan_rows"}

    rows = list(subset.itertuples(index=False))
    valid_suffix = []
    for row in reversed(rows):
        if bool(row.rutherford_approx_valid):
            valid_suffix.append(row)
        else:
            break
    interval_lower = float(subset["speed_interval_lower_m_s"].iloc[0])
    interval_upper = float(subset["speed_interval_upper_m_s"].iloc[0])
    if not valid_suffix:
        return {
            "interval_label": interval_label,
            "status": "no_rutherford_suffix",
            "rutherford_lower_m_s": np.nan,
        }
    valid_suffix.reverse()
    first_valid = valid_suffix[0]
    if len(valid_suffix) == len(rows):
        return {
            "interval_label": interval_label,
            "status": "all_scanned_speeds_rutherford",
            "rutherford_lower_m_s": interval_lower,
            "bracket_lower_m_s": interval_lower,
            "bracket_upper_m_s": float(first_valid.speed_m_s),
        }

    first_index = rows.index(first_valid)
    last_invalid = rows[first_index - 1]
    return {
        "interval_label": interval_label,
        "status": "resonant_to_rutherford_bracket",
        "last_non_rutherford_speed_m_s": float(last_invalid.speed_m_s),
        "first_rutherford_speed_m_s": float(first_valid.speed_m_s),
        "bracket_lower_m_s": float(last_invalid.speed_m_s),
        "bracket_upper_m_s": float(first_valid.speed_m_s),
        "rutherford_lower_m_s": float(first_valid.speed_m_s),
        "interval_upper_m_s": interval_upper,
    }


def refine_rutherford_transition(transition: dict[str, Any]) -> dict[str, Any]:
    if (
        transition.get("status") != "resonant_to_rutherford_bracket"
        or not REFINE_RUTHERFORD_TRANSITION
    ):
        return transition
    lo = float(transition["bracket_lower_m_s"])
    hi = float(transition["bracket_upper_m_s"])
    history = []
    for step in range(MAX_RUTHERFORD_REFINEMENT_STEPS):
        if hi - lo <= RUTHERFORD_TRANSITION_SPEED_TOL_M_S:
            break
        midpoint = 0.5 * (lo + hi)
        metrics = rutherford_speed_metrics(midpoint)
        valid = bool(metrics["rutherford_approx_valid"])
        history.append(
            {
                "step": step + 1,
                "speed_m_s": midpoint,
                "rutherford_approx_valid": valid,
                "rutherford_criterion_value": metrics["rutherford_criterion_value"],
                "rutherford_criterion_target": metrics["rutherford_criterion_target"],
            }
        )
        if valid:
            hi = midpoint
        else:
            lo = midpoint
    updated = dict(transition)
    updated.update(
        {
            "refined": True,
            "bracket_lower_m_s": lo,
            "bracket_upper_m_s": hi,
            "bracket_width_m_s": hi - lo,
            "last_non_rutherford_speed_m_s": lo,
            "first_rutherford_speed_m_s": hi,
            "rutherford_lower_m_s": hi,
            "refinement_history": history,
        }
    )
    return updated


def build_regime_intervals(
    retained_interval_df: pd.DataFrame,
    rutherford_transitions: list[dict[str, Any]],
) -> pd.DataFrame:
    """Split each Test-0.75-retained branch into resonant/Rutherford ranges."""
    transition_by_label = {
        str(row["interval_label"]): row for row in rutherford_transitions
    }
    rows: list[dict[str, Any]] = []
    for interval in retained_interval_df.itertuples(index=False):
        label = str(interval.label)
        lower = float(interval.lower_m_s)
        upper = float(interval.upper_m_s)
        transition = transition_by_label.get(label, {})
        status = str(transition.get("status", "no_rutherford_suffix"))
        rutherford_lower = transition.get("rutherford_lower_m_s", np.nan)
        if np.isfinite(rutherford_lower):
            rutherford_lower = min(max(float(rutherford_lower), lower), upper)
        if not np.isfinite(rutherford_lower) or rutherford_lower >= upper:
            rows.append(
                {
                    "speed_interval_label": label,
                    "regime": "resonant",
                    "ion_energy_model": "full_trap_required",
                    "lower_m_s": lower,
                    "upper_m_s": upper,
                    "rutherford_transition_status": status,
                }
            )
            continue
        if rutherford_lower > lower:
            rows.append(
                {
                    "speed_interval_label": label,
                    "regime": "resonant",
                    "ion_energy_model": "full_trap_required",
                    "lower_m_s": lower,
                    "upper_m_s": rutherford_lower,
                    "rutherford_transition_status": status,
                }
            )
        rows.append(
            {
                "speed_interval_label": label,
                "regime": "rutherford",
                "ion_energy_model": "rutherford_analytic",
                "lower_m_s": rutherford_lower,
                "upper_m_s": upper,
                "rutherford_transition_status": status,
            }
        )
    return pd.DataFrame(rows)


def classify_speed_from_regime_intervals(
    speed_m_s: float,
    regime_interval_df: pd.DataFrame,
) -> tuple[str, str]:
    speed = float(speed_m_s)
    for row in regime_interval_df.itertuples(index=False):
        if float(row.lower_m_s) <= speed <= float(row.upper_m_s):
            return str(row.regime), str(row.ion_energy_model)
    raise ValueError(f"speed {speed} m/s is outside the retained regime intervals")


def build_final_allowed_intervals(
    base_intervals: list[dict[str, float | str]],
    transitions: list[dict[str, Any]],
) -> pd.DataFrame:
    transition_by_label = {str(row["interval_label"]): row for row in transitions}
    rows = []
    for interval in base_intervals:
        label = str(interval["label"])
        lower = float(interval["lower_m_s"])
        upper = float(interval["upper_m_s"])
        transition = transition_by_label.get(label, {})
        new_lower = float(transition.get("conservative_retained_lower_m_s", lower))
        new_lower = min(max(new_lower, lower), upper)
        if upper <= new_lower:
            continue
        rows.append(
            {
                "label": label,
                "lower_m_s": new_lower,
                "upper_m_s": upper,
                "test0_lower_m_s": lower,
                "test0_upper_m_s": upper,
                "lower_removed_by_test0p75_m_s": new_lower - lower,
                "transition_status": transition.get("status", "not_scanned"),
                "adiabatic_monotonic_extrapolation_used": bool(
                    transition.get(
                        "adiabatic_monotonic_extrapolation_to_interval_lower",
                        False,
                    )
                ),
                "transition_bracket_lower_m_s": transition.get(
                    "bracket_lower_m_s", np.nan
                ),
                "transition_bracket_upper_m_s": transition.get(
                    "bracket_upper_m_s", np.nan
                ),
            }
        )
    return pd.DataFrame(rows)


def make_test1_speed_policy(
    interval_df: pd.DataFrame,
    regime_interval_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Build representative speeds and attach the full trajectory ``b`` policy."""
    if interval_df.empty:
        raise RuntimeError("Test 0.75 produced no retained speed interval")
    scale = mb_scale(T_DM_K, M_DM_KG)
    quantile_map = {
        "low_allowed": tuple(float(q) for q in TEST1_LOW_INTERVAL_QUANTILES),
        "high_allowed": tuple(float(q) for q in TEST1_HIGH_INTERVAL_QUANTILES),
    }
    probabilities: dict[str, float] = {}
    for row in interval_df.itertuples(index=False):
        probabilities[str(row.label)] = float(
            maxwell.cdf(float(row.upper_m_s), scale=scale)
            - maxwell.cdf(float(row.lower_m_s), scale=scale)
        )
    total_probability = float(sum(probabilities.values()))
    if total_probability <= 0.0:
        raise RuntimeError("Final retained Maxwell probability is zero")

    rows: list[dict[str, Any]] = []
    for interval_index, row in enumerate(interval_df.itertuples(index=False)):
        branch = str(row.label)
        lo = float(row.lower_m_s)
        hi = float(row.upper_m_s)
        probability = probabilities[branch]
        quantiles = quantile_map[branch]
        cdf_lo = float(maxwell.cdf(lo, scale=scale))
        cdf_hi = float(maxwell.cdf(hi, scale=scale))
        for quantile in quantiles:
            uq = cdf_lo + float(quantile) * (cdf_hi - cdf_lo)
            uq = min(max(uq, np.nextafter(0.0, 1.0)), np.nextafter(1.0, 0.0))
            speed = float(maxwell.ppf(uq, scale=scale))
            params = rutherford_parameters(speed)
            boundaries = trajectory_regime_boundaries(speed)
            rows.append(
                {
                    "key": f"{branch}_vb_q{quantile:.3f}",
                    "quantile": float(quantile),
                    "conditional_quantile": float(quantile),
                    "unconditional_quantile": uq,
                    "speed_m_s": speed,
                    "v_inf_m_s": speed,
                    "speed_interval_label": branch,
                    "regime_interval_index": interval_index,
                    "speed_regime": "mixed_v_b",
                    "ion_energy_model": "trajectory_v_b_policy",
                    "speed_interval_lower_m_s": lo,
                    "speed_interval_upper_m_s": hi,
                    "speed_interval_probability": probability,
                    "total_allowed_speed_probability": total_probability,
                    "conditional_speed_weight": probability / total_probability / len(quantiles),
                    "unconditional_speed_weight": probability / len(quantiles),
                    "tail_probability": total_probability,
                    "minimum_speed_m_s": float(interval_df["lower_m_s"].min()),
                    "maximum_speed_m_s": float(interval_df["upper_m_s"].max()),
                    "r_min_threshold_m": params["R_full_m"],
                    "r_min_threshold_um": params["R_full_m"] * 1.0e6,
                    "R_full_m": params["R_full_m"],
                    "R_full_um": params["R_full_m"] * 1.0e6,
                    "R_switch_factor": R_SWITCH_FACTOR,
                    "R_switch_m": params["R_switch_m"],
                    "R_switch_um": params["R_switch_m"] * 1.0e6,
                    **boundaries,
                    "rutherford_relative_tolerance": RUTHERFORD_RELATIVE_TOLERANCE,
                    "test0p75_model_safety_factor": MODEL_UPPER_SAFETY_FACTOR,
                    "test0p75_interaction_time_scale": INTERACTION_TIME_SCALE,
                    "trajectory_policy_b_max_m": TRAJECTORY_POLICY_B_MAX_M,
                    "speed_policy_source": SCRIPT_VERSION,
                    # Test 0.75 classifies the collision-energy model only.
                    # Every non-adiabatic trajectory must still pass a trap-only
                    # reach screen before Rutherford or full-coupled treatment.
                    "policy_requires_reach_screen": True,
                    "rutherford_requires_reach_screen": True,
                    "resonant_requires_reach_screen": True,
                    "adiabatic_is_analytic_reject": True,
                    "test0_v9_reject_lower_m_s": V9_REJECT_LOWER_M_S,
                    "test0_v9_reject_upper_m_s": V9_REJECT_UPPER_M_S,
                }
            )
    return pd.DataFrame(rows).sort_values("speed_m_s").reset_index(drop=True)


def make_validation_points(transitions_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if transitions_df.empty:
        return pd.DataFrame(rows)
    for row in transitions_df.itertuples(index=False):
        lo = getattr(row, "bracket_lower_m_s", np.nan)
        hi = getattr(row, "bracket_upper_m_s", np.nan)
        if not (np.isfinite(lo) and np.isfinite(hi)):
            continue
        for label, speed in (
            ("below", float(lo)),
            ("midpoint", 0.5 * (float(lo) + float(hi))),
            ("above", float(hi)),
        ):
            envelope = conservative_speed_envelope(speed)
            params = rutherford_parameters(speed)
            rows.append(
                {
                    "speed_interval_label": str(row.interval_label),
                    "validation_position": label,
                    "speed_m_s": speed,
                    "b_over_b_threshold": envelope[
                        "maximizing_b_over_b_threshold"
                    ],
                    "b_m": float(envelope["maximizing_b_over_b_threshold"])
                    * float(params["b_threshold_m"]),
                    "R_full_m": float(params["R_full_m"]),
                    "R_switch_m": float(params["R_switch_m"]),
                    "predicted_upper_energy_J": envelope[
                        "max_conservative_trap_energy_upper_J"
                    ],
                    "reject_speed_adiabatic": envelope[
                        "reject_speed_adiabatic"
                    ],
                }
            )
    return pd.DataFrame(rows)


# ============================================================
# OUTPUTS AND PLOTS
# ============================================================


def plot_speed_summary(summary_df: pd.DataFrame) -> None:
    if summary_df.empty:
        return
    data = summary_df.sort_values("speed_m_s")

    fig, ax = plt.subplots(figsize=(8.8, 5.6))
    for column, label, marker in (
        (
            "max_sampled_free_space_rutherford_energy_J",
            "Max analytic Rutherford recoil",
            ".",
        ),
        (
            "max_sampled_trap_energy_force_spectrum_J",
            "Max sampled force-spectrum estimate",
            "o",
        ),
        (
            "max_conservative_trap_energy_upper_J",
            "Conservative force-spectrum upper envelope",
            "s",
        ),
    ):
        positive = data[column].to_numpy(float) > 0.0
        ax.loglog(
            data.loc[positive, "speed_m_s"],
            data.loc[positive, column],
            marker=marker,
            label=label,
        )
    ax.axhline(ENERGY_THRESHOLD_J, linestyle="--", label="Detection threshold")
    ax.axvspan(
        V9_REJECT_LOWER_M_S,
        V9_REJECT_UPPER_M_S,
        alpha=0.12,
        label="Test 0 v9 rejected",
    )
    ax.set_xlabel("DM speed [m/s]")
    ax.set_ylabel("Ion energy [J]")
    ax.set_title("Test 0.75: analytic adiabatic force-spectrum screen")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(f"{OUTPUT_PREFIX}_energy_vs_speed.png", dpi=220, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(8.8, 4.8))
    ax.semilogx(
        data["speed_m_s"],
        data["reject_speed_adiabatic"].astype(int),
        marker="o",
    )
    ax.set_yticks([0, 1], ["Retain", "Reject"])
    ax.set_xlabel("DM speed [m/s]")
    ax.set_title("Test 0.75 analytic prefilter decision")
    ax.grid(True, which="both", alpha=0.3)
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(f"{OUTPUT_PREFIX}_decision_vs_speed.png", dpi=220, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(8.8, 5.0))
    ax.loglog(
        data["speed_m_s"],
        data["maximizing_chi_sec_x"],
        marker="o",
        label=r"$\omega_x\tau$",
    )
    ax.loglog(
        data["speed_m_s"],
        data["maximizing_chi_sec_y"],
        marker=".",
        label=r"$\omega_y\tau$",
    )
    ax.loglog(
        data["speed_m_s"],
        data["maximizing_chi_sec_z"],
        marker="s",
        label=r"$\omega_z\tau$",
    )
    ax.axhline(1.0, linestyle="--", label=r"$\omega\tau=1$")
    ax.set_xlabel("DM speed [m/s]")
    ax.set_ylabel(r"Dimensionless interaction time $\omega\tau$")
    ax.set_title("Interaction timescale at the maximizing impact parameter")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(f"{OUTPUT_PREFIX}_chi_vs_speed.png", dpi=220, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)

    regime_codes = {"adiabatic": 0, "resonant": 1, "rutherford": 2}
    fig, ax = plt.subplots(figsize=(8.8, 4.8))
    ax.semilogx(
        data["speed_m_s"],
        data["speed_regime"].map(regime_codes),
        marker="o",
    )
    ax.set_yticks([0, 1, 2], ["Adiabatic", "Resonant", "Rutherford"])
    ax.axvspan(
        V9_REJECT_LOWER_M_S,
        V9_REJECT_UPPER_M_S,
        alpha=0.12,
        label="Test 0 v9 rejected",
    )
    ax.set_xlabel("DM speed [m/s]")
    ax.set_title("Test 0.75 speed-regime classification")
    ax.grid(True, which="both", alpha=0.3)
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(f"{OUTPUT_PREFIX}_regime_vs_speed.png", dpi=220, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(8.8, 5.0))
    ax.semilogx(
        data["speed_m_s"],
        data["rutherford_strict_min_ratio"],
        marker="o",
        label="Strict minimum trap/Rutherford ratio",
    )
    ax.semilogx(
        data["speed_m_s"],
        data["rutherford_energy_weighted_ratio"],
        marker=".",
        label="Energy-weighted ratio",
    )
    ax.axhline(
        1.0 - RUTHERFORD_RELATIVE_TOLERANCE,
        linestyle="--",
        label="Rutherford tolerance threshold",
    )
    ax.set_xlabel("DM speed [m/s]")
    ax.set_ylabel("Predicted trap / Rutherford final energy")
    ax.set_ylim(0.0, 1.02)
    ax.set_title("Rutherford-approximation validity over detectable impact parameters")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(f"{OUTPUT_PREFIX}_rutherford_ratio_vs_speed.png", dpi=220, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)

    if not data.empty:
        regime_codes = {"adiabatic": 0, "resonant": 1, "rutherford": 2}
        raw_name = f"{OUTPUT_PREFIX}_analytic_sweep.csv"
        # Use the in-memory raw sweep when called from run_test0p75; the CSV
        # fallback makes the plot reusable after a prior run.
        raw_for_map = globals().get("_TEST0P75_RAW_FOR_PLOTTING")
        if raw_for_map is None and Path(raw_name).exists():
            raw_for_map = pd.read_csv(raw_name)
        if isinstance(raw_for_map, pd.DataFrame) and not raw_for_map.empty:
            map_data = raw_for_map.loc[
                raw_for_map["b_over_b_threshold"].astype(float)
                <= RUTHERFORD_B_MAX_OVER_THRESHOLD
            ].copy()
            fig, ax = plt.subplots(figsize=(9.0, 5.4))
            scatter = ax.scatter(
                map_data["speed_m_s"],
                map_data["b_over_b_threshold"],
                c=map_data["geometry_regime"].map(regime_codes),
                s=10,
                vmin=0,
                vmax=2,
            )
            colorbar = fig.colorbar(scatter, ax=ax, ticks=[0, 1, 2])
            colorbar.ax.set_yticklabels(["Adiabatic", "Resonant", "Rutherford"])
            ax.set_xscale("log")
            ax.set_xlabel("DM speed [m/s]")
            ax.set_ylabel(r"Impact parameter $b/b_{threshold}(v)$")
            ax.set_title("Geometry-level regime map")
            ax.grid(True, which="both", alpha=0.25)
            fig.tight_layout()
            if SAVE_PLOTS:
                fig.savefig(
                    f"{OUTPUT_PREFIX}_regime_map_speed_b.png",
                    dpi=220,
                    bbox_inches="tight",
                )
            if SHOW_PLOTS:
                plt.show()
            else:
                plt.close(fig)


def write_config(
    interval_df: pd.DataFrame,
    regime_interval_df: pd.DataFrame,
    policy_df: pd.DataFrame,
    transitions_df: pd.DataFrame,
    rutherford_transitions_df: pd.DataFrame,
) -> None:
    intervals = [
        (str(row.label), float(row.lower_m_s), float(row.upper_m_s))
        for row in interval_df.itertuples(index=False)
    ]
    regime_intervals = [
        (
            str(row.speed_interval_label),
            str(row.regime),
            float(row.lower_m_s),
            float(row.upper_m_s),
            str(row.ion_energy_model),
        )
        for row in regime_interval_df.itertuples(index=False)
    ]
    adiabatic_brackets = []
    if not transitions_df.empty:
        for row in transitions_df.itertuples(index=False):
            lo = getattr(row, "bracket_lower_m_s", np.nan)
            hi = getattr(row, "bracket_upper_m_s", np.nan)
            if np.isfinite(lo) and np.isfinite(hi):
                adiabatic_brackets.append((str(row.interval_label), float(lo), float(hi)))
    rutherford_brackets = []
    if not rutherford_transitions_df.empty:
        for row in rutherford_transitions_df.itertuples(index=False):
            lo = getattr(row, "bracket_lower_m_s", np.nan)
            hi = getattr(row, "bracket_upper_m_s", np.nan)
            if np.isfinite(lo) and np.isfinite(hi):
                rutherford_brackets.append((str(row.interval_label), float(lo), float(hi)))
    Path(f"{OUTPUT_PREFIX}_test1_config.py").write_text(
        "# Generated by test0p75_regime_classification_notebook_v7.py\n"
        f"TEST0P75_SCRIPT_VERSION = {SCRIPT_VERSION!r}\n"
        f"TEST1_SPEED_POLICY_CSV = {str(Path(f'{OUTPUT_PREFIX}_test1_speed_policy.csv'))!r}\n"
        f"TEST1_ALLOWED_SPEED_INTERVALS_M_S = {intervals!r}\n"
        f"TEST1_REGIME_INTERVALS_M_S = {regime_intervals!r}\n"
        f"ADIABATIC_TRANSITION_BRACKETS_M_S = {adiabatic_brackets!r}\n"
        f"RUTHERFORD_TRANSITION_BRACKETS_M_S = {rutherford_brackets!r}\n"
        f"RUTHERFORD_RELATIVE_TOLERANCE = {RUTHERFORD_RELATIVE_TOLERANCE:.17g}\n"
        f"RUTHERFORD_CLASSIFICATION_MODE = {RUTHERFORD_CLASSIFICATION_MODE!r}\n"
        f"TEST1_REPRESENTATIVE_SPEEDS_M_S = {policy_df['speed_m_s'].astype(float).tolist()!r}\n"
        f"TOTAL_RETAINED_MB_PROBABILITY = {float(policy_df['total_allowed_speed_probability'].iloc[0]):.17g}\n"
        f"MODEL_UPPER_SAFETY_FACTOR = {MODEL_UPPER_SAFETY_FACTOR:.17g}\n"
        f"INTERACTION_TIME_SCALE = {INTERACTION_TIME_SCALE:.17g}\n",
        encoding="utf-8",
    )


def plot_vb_policy(boundary_df: pd.DataFrame, vb_map_df: pd.DataFrame) -> None:
    if boundary_df.empty:
        return
    data = boundary_df.sort_values("speed_m_s")
    fig, ax = plt.subplots(figsize=(8.8, 5.3))
    positive = data["b_rutherford_max_m"].to_numpy(float) > 0.0
    ax.loglog(
        data.loc[positive, "speed_m_s"],
        data.loc[positive, "b_rutherford_max_m"] * 1.0e6,
        marker="o",
        label="Rutherford upper b boundary",
    )
    finite_adia = np.isfinite(data["b_adiabatic_min_m"].to_numpy(float)) & (
        data["b_adiabatic_min_m"].to_numpy(float) > 0.0
    )
    ax.loglog(
        data.loc[finite_adia, "speed_m_s"],
        data.loc[finite_adia, "b_adiabatic_min_m"] * 1.0e6,
        marker="o",
        label="Adiabatic lower b boundary",
    )
    if np.isfinite(V9_REJECT_UPPER_M_S):
        ax.axvspan(V9_REJECT_LOWER_M_S, V9_REJECT_UPPER_M_S, alpha=0.12)
    ax.set_xlabel("DM speed [m/s]")
    ax.set_ylabel("Impact parameter [um]")
    ax.set_title("Test 0.75 trajectory-regime boundaries")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(f"{OUTPUT_PREFIX}_vb_regime_boundaries.png", dpi=220, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)

    if vb_map_df.empty:
        return
    codes = {"adiabatic": 0, "resonant": 1, "rutherford": 2}
    plot_data = vb_map_df.copy()
    plot_data["regime_code"] = plot_data["trajectory_regime"].map(codes)
    fig, ax = plt.subplots(figsize=(9.0, 5.5))
    scatter = ax.scatter(
        plot_data["speed_m_s"],
        plot_data["b_over_b_threshold"],
        c=plot_data["regime_code"],
        s=5,
        alpha=0.75,
    )
    ax.set_xscale("log")
    ax.set_yscale("symlog", linthresh=1.0e-4)
    ax.set_xlabel("DM speed [m/s]")
    ax.set_ylabel("b / b_threshold(v)")
    ax.set_title("Test 0.75 trajectory classification in the (v, b) plane")
    ax.grid(True, which="both", alpha=0.25)
    fig.colorbar(scatter, ax=ax, ticks=[0, 1, 2], label="0 adiabatic, 1 resonant, 2 Rutherford")
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(f"{OUTPUT_PREFIX}_vb_regime_map.png", dpi=220, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)


# ============================================================
# MAIN
# ============================================================


def run_test0p75() -> dict[str, Any]:
    require_environment()
    progress(f"Test 0.75 script version: {SCRIPT_VERSION}")
    progress("Mode: analytic trajectory-regime classification in the (v, b) plane")
    progress("No full trap simulation and no DM trajectory ODE propagation are run.")
    progress(
        "Input from current Test 0: rejected interval "
        f"({V9_REJECT_LOWER_M_S:.12g}, {V9_REJECT_UPPER_M_S:.12g}) m/s"
    )
    progress(f"Rejected-interval source: {TEST0_REJECT_INTERVAL_SOURCE}")
    progress(
        f"Rutherford tolerance={RUTHERFORD_RELATIVE_TOLERANCE:.3g}; "
        f"trajectory policy b_max={TRAJECTORY_POLICY_B_MAX_M * 1.0e3:.3f} mm"
    )

    base_intervals = test0_allowed_intervals()
    speed_scan = build_scan_speeds()
    progress(f"Analytic speed scan contains {len(speed_scan)} speeds.")

    raw_frames: list[pd.DataFrame] = []
    summaries: list[dict[str, Any]] = []
    for index, speed_row in speed_scan.iterrows():
        frame, summary = run_speed(speed_row)
        raw_frames.append(frame)
        summaries.append(summary)
        if PRINT_EACH_SPEED:
            b_adia_um = summary["b_adiabatic_min_um"]
            b_adia_text = f"{b_adia_um:.3f}" if np.isfinite(b_adia_um) else "none"
            progress(
                f"[{index + 1:3d}/{len(speed_scan):3d}] "
                f"v={summary['speed_m_s']:11.6f} m/s, "
                f"b_Ruth,max={summary['b_rutherford_max_um']:10.3f} um, "
                f"b_adia,min={b_adia_text:>10s} um, "
                f"speed_reject={summary['reject_speed_adiabatic']}"
            )

    raw_df = pd.concat(raw_frames, ignore_index=True) if raw_frames else pd.DataFrame()
    summary_df = pd.DataFrame(summaries).sort_values("speed_m_s").reset_index(drop=True)
    transitions = [
        identify_low_edge_transition(summary_df, str(interval["label"]))
        for interval in base_intervals
    ]
    transitions = [refine_transition(transition) for transition in transitions]

    serializable_transitions = []
    refinement_rows = []
    for transition in transitions:
        clean = dict(transition)
        history = clean.pop("refinement_history", [])
        serializable_transitions.append(clean)
        refinement_rows.extend(
            {"interval_label": transition["interval_label"], **row}
            for row in history
        )
    transitions_df = pd.DataFrame(serializable_transitions)
    refinement_df = pd.DataFrame(refinement_rows)
    interval_df = build_final_allowed_intervals(base_intervals, transitions)
    policy_df = make_test1_speed_policy(interval_df)
    vb_map_df = build_vb_regime_map(speed_scan)
    boundary_columns = [
        "speed_interval_label", "speed_m_s", "b_threshold_m", "b_threshold_um",
        "b_rutherford_max_m", "b_rutherford_max_um",
        "b_rutherford_max_over_b_threshold",
        "b_adiabatic_min_m", "b_adiabatic_min_um",
        "b_adiabatic_min_over_b_threshold",
        "rutherford_detectable_area_fraction", "policy_b_max_m",
        "reject_speed_adiabatic",
    ]
    boundary_df = summary_df[[c for c in boundary_columns if c in summary_df.columns]].copy()
    regime_interval_df = interval_df.rename(columns={"label": "speed_interval_label"}).copy()
    regime_interval_df["regime"] = "trajectory_v_b_mixed"
    regime_interval_df["ion_energy_model"] = "trajectory_v_b_policy"
    validation_df = make_validation_points(transitions_df)
    plot_vb_policy(boundary_df, vb_map_df)

    if SAVE_RESULTS:
        raw_df.to_csv(f"{OUTPUT_PREFIX}_analytic_directional_sweep.csv", index=False)
        vb_map_df.to_csv(f"{OUTPUT_PREFIX}_trajectory_regime_map.csv", index=False)
        boundary_df.to_csv(f"{OUTPUT_PREFIX}_trajectory_regime_boundaries.csv", index=False)
        summary_df.to_csv(f"{OUTPUT_PREFIX}_speed_summary.csv", index=False)
        transitions_df.to_csv(f"{OUTPUT_PREFIX}_transition_brackets.csv", index=False)
        refinement_df.to_csv(f"{OUTPUT_PREFIX}_transition_refinement.csv", index=False)
        interval_df.to_csv(f"{OUTPUT_PREFIX}_test1_allowed_intervals.csv", index=False)
        regime_interval_df.to_csv(f"{OUTPUT_PREFIX}_regime_intervals.csv", index=False)
        policy_df.to_csv(f"{OUTPUT_PREFIX}_test1_speed_policy.csv", index=False)
        validation_df.to_csv(f"{OUTPUT_PREFIX}_recommended_validation_points.csv", index=False)
        # Compatibility names for older notebooks plus the reach-aware v10 suite.
        policy_df.to_csv(
            f"{OUTPUT_PREFIX}_compat_adiabatic_regime_test1_speed_policy.csv",
            index=False,
        )
        policy_df.to_csv(
            f"{OUTPUT_PREFIX}_compat_vb_reach_aware_test1_speed_policy.csv",
            index=False,
        )

    progress("\nTest 0.75 trajectory policy:")
    for row in interval_df.itertuples(index=False):
        progress(f"  retained speed interval {row.label}: [{row.lower_m_s:.9g}, {row.upper_m_s:.9g}] m/s")
    progress(
        "  At each retained speed: b <= b_Ruth,max uses analytic Rutherford; "
        "b >= b_adia,min is rejected; the interval between them requires full trap propagation."
    )
    progress(f"  Test 1 policy: {OUTPUT_PREFIX}_test1_speed_policy.csv")
    progress(f"  Full (v,b) map: {OUTPUT_PREFIX}_trajectory_regime_map.csv")

    return {
        "raw_trajectories": raw_df,
        "raw_sweep": raw_df,
        "vb_regime_map": vb_map_df,
        "trajectory_boundaries": boundary_df,
        "speed_summary": summary_df,
        "transitions": transitions_df,
        "transition_refinement": refinement_df,
        "allowed_intervals": interval_df,
        "regime_intervals": regime_interval_df,
        "test1_speed_policy": policy_df,
        "recommended_validation_points": validation_df,
    }


if __name__ == "__main__":
    TEST0P75_RESULTS = run_test0p75()
    TEST0P75_TRAJECTORY_DF = TEST0P75_RESULTS["raw_trajectories"]
    TEST0P75_ANALYTIC_SWEEP_DF = TEST0P75_RESULTS["raw_sweep"]
    TEST0P75_VB_REGIME_MAP_DF = TEST0P75_RESULTS["vb_regime_map"]
    TEST0P75_TRAJECTORY_BOUNDARY_DF = TEST0P75_RESULTS["trajectory_boundaries"]
    TEST0P75_SPEED_SUMMARY_DF = TEST0P75_RESULTS["speed_summary"]
    TEST0P75_TRANSITION_DF = TEST0P75_RESULTS["transitions"]
    TEST0P75_TEST1_INTERVAL_DF = TEST0P75_RESULTS["allowed_intervals"]
    TEST0P75_TEST1_SPEED_POLICY_DF = TEST0P75_RESULTS["test1_speed_policy"]

# Test 1: Trajectory Classification/Determining r_far


In [ ]:
"""Test 1 using the Test-0.75 retained speed policy and exact per-speed radii.

The representative speeds are read from the Test 0.75 policy CSV when available.
For every representative speed, this script uses the analytic Rutherford
``r_min_threshold`` and uses it as ``R_full``. It then uses
``R_switch = 2 R_full`` and constructs a separate equal-area impact grid whose
annulus edge contains that speed's exact ``R_full``.
"""

import math
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from scipy.stats import maxwell
from matplotlib.colors import LogNorm, ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ============================================================
# EMBEDDED TEST-0-v9 SPEED/RADIUS POLICY
# ============================================================
# This file is fully standalone. No external speed-policy module is imported.

(
    V9_REJECT_LOWER_M_S,
    V9_REJECT_UPPER_M_S,
    TEST0_REJECT_INTERVAL_SOURCE,
) = resolve_test0_rejected_interval()
ION_ENERGY_THRESHOLD_J = float(USER_TARGET_ION_ENERGY_J)
R_SWITCH_FACTOR = 2.0
DEFAULT_UPPER_TAIL_PROBABILITY = 1.0e-10
DEFAULT_LOW_INTERVAL_QUANTILES = (0.10, 0.50, 0.90, 0.99)
DEFAULT_HIGH_INTERVAL_QUANTILES = (0.25, 0.75)


def _policy_mb_scale(temperature_k: float, mass_kg: float) -> float:
    if temperature_k <= 0.0 or mass_kg <= 0.0:
        raise ValueError("temperature and mass must be positive")
    return math.sqrt(k_B * float(temperature_k) / float(mass_kg))


def _policy_reduced_mass(m_dm_kg: float, m_ion_kg: float) -> float:
    if m_dm_kg <= 0.0 or m_ion_kg <= 0.0:
        raise ValueError("masses must be positive")
    return float(m_dm_kg * m_ion_kg / (m_dm_kg + m_ion_kg))


def _policy_kinematic_min_speed_m_s(
    m_dm_kg: float,
    m_ion_kg: float,
    threshold_j: float = ION_ENERGY_THRESHOLD_J,
) -> float:
    if threshold_j <= 0.0:
        raise ValueError("threshold_j must be positive")
    mu = _policy_reduced_mass(m_dm_kg, m_ion_kg)
    return math.sqrt(float(threshold_j) * m_ion_kg / (2.0 * mu**2))


def _policy_rutherford_r_min_threshold_m(
    speed_m_s: float,
    *,
    m_dm_kg: float,
    m_ion_kg: float,
    eps: float,
    ion_charge_number: float,
    coulomb_constant: float,
    elementary_charge_c: float,
    threshold_j: float = ION_ENERGY_THRESHOLD_J,
) -> float:
    speed = float(speed_m_s)
    if not np.isfinite(speed) or speed <= 0.0:
        return float("nan")
    mu = _policy_reduced_mass(m_dm_kg, m_ion_kg)
    v_kin = _policy_kinematic_min_speed_m_s(
        m_dm_kg, m_ion_kg, threshold_j
    )
    if speed < v_kin:
        return float("nan")
    coupling = abs(
        float(coulomb_constant)
        * float(ion_charge_number)
        * float(eps)
        * float(elementary_charge_c) ** 2
    )
    if coupling <= 0.0:
        raise ValueError("The Coulomb coupling magnitude must be positive")
    return (
        coupling / (mu * speed**2)
        + coupling / speed * math.sqrt(2.0 / (m_ion_kg * threshold_j))
    )


def _policy_bounded_allowed_speed_intervals(
    *,
    m_dm_kg: float,
    m_ion_kg: float,
    temperature_k: float,
    include_high_speed_branch: bool = True,
    upper_tail_probability: float = DEFAULT_UPPER_TAIL_PROBABILITY,
    reject_lower_m_s: float = V9_REJECT_LOWER_M_S,
    reject_upper_m_s: float = V9_REJECT_UPPER_M_S,
) -> list[dict[str, float | str]]:
    if not (0.0 < upper_tail_probability < 1.0):
        raise ValueError("upper_tail_probability must lie in (0, 1)")
    if not reject_upper_m_s > reject_lower_m_s > 0.0:
        raise ValueError("invalid rejected interval")
    scale = _policy_mb_scale(temperature_k, m_dm_kg)
    v_kin = _policy_kinematic_min_speed_m_s(m_dm_kg, m_ion_kg)
    v_max = float(maxwell.ppf(1.0 - upper_tail_probability, scale=scale))
    intervals: list[dict[str, float | str]] = []
    low_upper = min(float(reject_lower_m_s), v_max)
    if low_upper > v_kin:
        intervals.append(
            {
                "label": "low_allowed",
                "lower_m_s": float(v_kin),
                "upper_m_s": float(low_upper),
            }
        )
    if include_high_speed_branch and v_max > reject_upper_m_s:
        intervals.append(
            {
                "label": "high_allowed",
                "lower_m_s": float(reject_upper_m_s),
                "upper_m_s": float(v_max),
            }
        )
    return intervals


def make_v9_representative_speed_cases(
    *,
    m_dm_kg: float,
    m_ion_kg: float,
    temperature_k: float,
    eps: float,
    ion_charge_number: float,
    coulomb_constant: float,
    elementary_charge_c: float,
    include_high_speed_branch: bool = True,
    upper_tail_probability: float = DEFAULT_UPPER_TAIL_PROBABILITY,
    low_interval_quantiles: Iterable[float] = DEFAULT_LOW_INTERVAL_QUANTILES,
    high_interval_quantiles: Iterable[float] = DEFAULT_HIGH_INTERVAL_QUANTILES,
    r_switch_factor: float = R_SWITCH_FACTOR,
) -> list[dict[str, float | str]]:
    intervals = _policy_bounded_allowed_speed_intervals(
        m_dm_kg=m_dm_kg,
        m_ion_kg=m_ion_kg,
        temperature_k=temperature_k,
        include_high_speed_branch=include_high_speed_branch,
        upper_tail_probability=upper_tail_probability,
    )
    if not intervals:
        raise RuntimeError("Test-0-v9 policy produced no allowed speed interval")

    scale = _policy_mb_scale(temperature_k, m_dm_kg)
    interval_quantiles = {
        "low_allowed": tuple(float(q) for q in low_interval_quantiles),
        "high_allowed": tuple(float(q) for q in high_interval_quantiles),
    }
    interval_probabilities: dict[str, float] = {}
    for interval in intervals:
        lo = float(interval["lower_m_s"])
        hi = float(interval["upper_m_s"])
        interval_probabilities[str(interval["label"])] = float(
            maxwell.cdf(hi, scale=scale) - maxwell.cdf(lo, scale=scale)
        )
    total_allowed_probability = float(sum(interval_probabilities.values()))
    if total_allowed_probability <= 0.0:
        raise RuntimeError("Allowed MB probability is zero")

    cases: list[dict[str, float | str]] = []
    for interval in intervals:
        label = str(interval["label"])
        lo = float(interval["lower_m_s"])
        hi = float(interval["upper_m_s"])
        quantiles = interval_quantiles[label]
        if not quantiles:
            continue
        if any(not (0.0 < q < 1.0) for q in quantiles):
            raise ValueError(f"{label} quantiles must lie strictly in (0, 1)")
        cdf_lo = float(maxwell.cdf(lo, scale=scale))
        cdf_hi = float(maxwell.cdf(hi, scale=scale))
        probability = interval_probabilities[label]
        n_cases = len(quantiles)
        for q in quantiles:
            unconditional_q = cdf_lo + q * (cdf_hi - cdf_lo)
            unconditional_q = min(
                max(unconditional_q, np.nextafter(0.0, 1.0)),
                np.nextafter(1.0, 0.0),
            )
            speed = float(maxwell.ppf(unconditional_q, scale=scale))
            radius = _policy_rutherford_r_min_threshold_m(
                speed,
                m_dm_kg=m_dm_kg,
                m_ion_kg=m_ion_kg,
                eps=eps,
                ion_charge_number=ion_charge_number,
                coulomb_constant=coulomb_constant,
                elementary_charge_c=elementary_charge_c,
            )
            key = f"{label}_q{q:.3f}"
            cases.append(
                {
                    "key": key,
                    "quantile": float(q),
                    "conditional_quantile": float(q),
                    "unconditional_quantile": float(unconditional_q),
                    "speed_m_s": speed,
                    "v_inf_m_s": speed,
                    "speed_interval_label": label,
                    "speed_interval_lower_m_s": lo,
                    "speed_interval_upper_m_s": hi,
                    "speed_interval_probability": probability,
                    "total_allowed_speed_probability": total_allowed_probability,
                    "conditional_speed_weight": (
                        probability / total_allowed_probability / n_cases
                    ),
                    "unconditional_speed_weight": probability / n_cases,
                    "tail_probability": total_allowed_probability,
                    "minimum_speed_m_s": float(intervals[0]["lower_m_s"]),
                    "maximum_speed_m_s": float(intervals[-1]["upper_m_s"]),
                    "r_min_threshold_m": radius,
                    "r_min_threshold_um": radius * 1.0e6,
                    "R_full_m": radius,
                    "R_full_um": radius * 1.0e6,
                    "R_switch_factor": float(r_switch_factor),
                    "R_switch_m": float(r_switch_factor) * radius,
                    "R_switch_um": float(r_switch_factor) * radius * 1.0e6,
                }
            )
    cases.sort(key=lambda row: float(row["speed_m_s"]))
    return cases

# ============================================================
# USER SETTINGS
# ============================================================

# Fixed parameter-space point from the top USER SETTINGS cell.
m_dm = float(USER_M_DM_KG)
eps = float(USER_EPS)

T_dm = float(USER_T_DM_K)
k_B = 1.380649e-23

# Test-0-v9 rejects the open interval
#     V9_REJECT_LOWER_M_S < v < V9_REJECT_UPPER_M_S.
# Test 1 samples both retained branches by default. The high-speed branch is
# bounded by a Maxwell upper-tail tolerance because the distribution has no
# finite physical maximum.
INCLUDE_HIGH_SPEED_BRANCH = True
UPPER_TAIL_PROBABILITY = float(USER_SPEED_UPPER_TAIL_PROBABILITY)
LOW_INTERVAL_REPRESENTATIVE_QUANTILES = DEFAULT_LOW_INTERVAL_QUANTILES
HIGH_INTERVAL_REPRESENTATIVE_QUANTILES = DEFAULT_HIGH_INTERVAL_QUANTILES
TARGET_ION_ENERGY_J = float(USER_TARGET_ION_ENERGY_J)

# Optional Test 0.75 output. When enabled, Test 1 uses the final retained
# intervals and representative speeds written by
# test0p75_adiabatic_regime_sweep.py instead of regenerating the original
# Test-0-v9-only policy. The CSV already contains exact speed-dependent
# R_full and R_switch values and physical Maxwell weights.
USE_TEST0P75_SPEED_POLICY = True
TEST0P75_SPEED_POLICY_CSV = Path(
    f"{TEST0P75_OUTPUT_PREFIX}_test1_speed_policy.csv"
)
REQUIRE_TEST0P75_SPEED_POLICY = True


# R_full and R_switch are calculated separately for every representative speed:
#     R_full(v) = analytic Rutherford r_min_threshold(v)
#     R_switch(v) = R_SWITCH_FACTOR * R_full(v).

# Test 1 determines the outer radius R_far beyond which the incoming path
# can be neglected. Test 2 will propagate the DM numerically from an outer
# start radius, switch to the full ion-DM solver at R_switch > R_full, and
# accept the trajectory only if the coupled path reaches R_full.

# Largest physical radius tested for the omitted outer straight-line tail.
# Keep 20 mm because it resolves all non-pole directions in the previous scan.
# The two x-axis pole directions may remain lower bounds and are reported
# separately rather than forcing a larger R_max for the entire calculation.
R_max = 20.0e-3

# ------------------------------------------------------------
# Impact-geometry sampling
# ------------------------------------------------------------
# Revised b scan:
#   1. R_full is an exact annulus boundary.
#   2. The central region 0 <= b < R_full and the nominal-miss region
#      R_full <= b <= b_max are each sampled with equal-area annuli.
#   3. Each annulus receives its physical disk-area weight, so the total
#      positive weight inside R_full is exactly (R_full / b_max)^2.
#
# The 2 mm scan was still controlled by its outermost weighted b sample for
# roughly half of the resolved directions. Use 4 mm for the next convergence
# pass. To run the intermediate 3 mm check instead, change only this line to
# b_max = 3.0e-3.
#
# n_b_outer is increased from 12 to 24 so the radial width of the outermost
# equal-area annuli remains close to that of the previous 2 mm scan.
b_max = 4.0e-3
n_b_inner = 6
n_b_outer = 24
n_psi = 8
INCLUDE_B_ZERO_DIAGNOSTIC = True
INCLUDE_R_FULL_DIAGNOSTIC = True
INCLUDE_B_MAX_DIAGNOSTIC = True

# Reach diagnostics are zero-population-weight rows used only to establish
# whether a fixed (v, theta, alpha, psi) ray can enter R_switch. They prevent
# the weighted pilot from missing a narrow low-b reach region.
INCLUDE_REACH_DIAGNOSTICS = True
REACH_DIAGNOSTIC_FRACTIONS = (0.0, 0.10, 0.25, 0.50, 1.0)
INCLUDE_RUTHERFORD_BOUNDARY_DIAGNOSTIC = True

# Test 1 must find a valid OUTER tail, not require the nominal straight line
# to remain valid all the way toward closest approach. The old analytical
# geometry prefilter was therefore too aggressive for producing Test 2 input.
# Leave it disabled for production Test 1 runs. analyze_R_far still requires
# every point from the selected R_far outward to R_max to be valid.
PREFILTER_INVALID_GEOMETRIES = False

# Exploit the exact x -> -x and y -> -y reflection symmetries of the trap.
# There is no z-reflection symmetry, so the reduced alpha domain retains both
# signs of z: alpha in [0, pi/2] union [3*pi/2, 2*pi).
EXPLOIT_XY_REFLECTION_SYMMETRY = True
RECONSTRUCT_FULL_DIRECTION_OUTPUTS = True

# Remove duplicated alpha values at theta=0 and theta=pi. At either pole,
# alpha does not define a new physical incoming direction.
REMOVE_DUPLICATED_POLES = True

# Do not remove a direction because its INNER nominal straight line enters
# the invalid half-space. Test 2 will propagate the actual curved trajectory
# from R_far and decide whether it reaches the switching/full radius or enters
# an invalid region. Test 1 only needs a valid contiguous outer tail.
PREFILTER_INVALID_DIRECTIONS = False
PREFILTER_POSITIVE_WEIGHT_ONLY = True

# The production maximum is taken only over positive-weight impact samples.
# A second maximum including b=0 and b=b_max diagnostics is also reported.
PRIMARY_MAX_POSITIVE_WEIGHT_ONLY = True

# Save the complete Test 1 audit plus a dedicated, production-ready Test 2
# input table containing every resolved finite R_far state.
SAVE_RESULTS = True
SAVE_TEST2_INPUT = True
TEST2_INCLUDE_ZERO_WEIGHT_DIAGNOSTICS = True
OUTPUT_PREFIX = TEST1_OUTPUT_PREFIX
TEST1_SCRIPT_VERSION = "2026-07-16-v10.1-symmetry-diagnostic-fix"

# ------------------------------------------------------------
# Scan resolution
# ------------------------------------------------------------
# The b-psi extension is much more expensive than the old b=0 scan.
# Start coarse, then set USE_COARSE_FIRST_PASS=False for the full map.
USE_COARSE_FIRST_PASS = True

if USE_COARSE_FIRST_PASS:
    n_theta = 13
    n_alpha = 24
else:
    n_theta = 25
    n_alpha = 48

# Speed representatives are controlled separately above for each accepted
# interval through LOW_INTERVAL_REPRESENTATIVE_QUANTILES and
# HIGH_INTERVAL_REPRESENTATIVE_QUANTILES.

# Number of worker threads used for the outer-tail scan.
N_THREADS = 8

# Limit queued futures so the full-resolution scan does not create tens of
# thousands of Future objects at once. Each task handles one
# (theta, alpha, b, psi) geometry and evaluates all representative speeds.
MAX_IN_FLIGHT_TASKS = 4 * N_THREADS

# Number of samples along each nominal incoming line.
n_path = 800

# Potential-model validity margin above the electrode plane.
z_margin = 1.0e-6

# Outer-tail safety tolerances.
energy_tol = 1.0e-2
angle_tol = 1.0e-3
displacement_tol = 1.0e-6

# Potential at infinity.
U_infinity = 0.0

# ------------------------------------------------------------
# Minimum-speed cutoff at the full-interaction sphere
# ------------------------------------------------------------
# Compute the easiest valid point on the R_full sphere using the total
# potential energy.  Representative speeds are then sampled from the
# Maxwell-Boltzmann distribution conditional on v >= v_cut.
USE_HANDOFF_BARRIER_SPEED_CUTOFF = False
HANDOFF_BARRIER_COMPONENT = "total"
HANDOFF_BARRIER_SPHERE_SAMPLES = 10000

# Reference value from the prior barrier calculation.  This is printed as a
# consistency check, but the dynamically computed value is used.
REFERENCE_MINIMUM_SPEED_M_S = 273.176458
BARRIER_SPEED_REFERENCE_REL_TOL = 5.0e-3

# ------------------------------------------------------------
# Outward-force diagnostic for nominal lines that miss R_full
# ------------------------------------------------------------
# Keep computing the old straight-line outward-force condition as an audit
# flag, but do not use it to remove Test 2 initial conditions. The straight
# path is only trusted in the omitted outer tail; Test 2 must determine actual
# reachability by numerical DM-only propagation from R_far.
EVALUATE_OUTWARD_FORCE_DIAGNOSTIC = True
HARD_REJECT_OUTWARD_FORCE_MISSES = False
OUTWARD_FORCE_DOT_TOL_J = 0.0
OUTWARD_FORCE_REQUIRE_COMPLETE_VALID_PATH = True

# ------------------------------------------------------------
# Output controls
# ------------------------------------------------------------
PRINT_EACH_TRAJECTORY = False
PRINT_AGGREGATED_TABLE_AFTER_EACH_SPEED = False
PRINT_RAW_TABLE_AFTER_EACH_SPEED = False
PRINT_COMBINED_TABLE_AT_END = False
PLOT_EACH_SPEED = True
PLOT_PERCENTILE_MAPS = False
PLOT_WORST_IMPACT_GEOMETRY = True
PLOT_DIAGNOSTIC_MAX_MAP = True
PLOT_VALID_WEIGHT_FRACTION = True
PLOT_WEIGHTED_STATUS_FRACTIONS = False

# Number of largest-R_far theta-alpha directions printed per speed.
N_WORST_DIRECTIONS_TO_PRINT = 20

# ============================================================
# CLASSIFICATION AND PLOT STYLES
# ============================================================

STATUS_ORDER = [
    "straight_safe_to_full_sphere",
    "straight_then_dm_only",
    "straight_path_blocked_needs_dm_only",
    "nominal_path_crosses_invalid_domain_needs_dm_only",
    "nominal_line_misses_full_sphere_needs_dm_only",
    "nominal_line_misses_full_sphere_outward_force_reject",
    "needs_larger_R_max",
    "invalid_far_start_below_electrode",
    "path_crosses_invalid_domain",
    "no_valid_points",
    "invalid_speed",
    "nonpositive_K_at_R_far",
]

STATUS_LABELS = {
    "straight_safe_to_full_sphere": "Straight to full ion-DM sphere",
    "straight_then_dm_only": (
        "Straight outside R_far; DM-only from R_far to R_full"
    ),
    "straight_path_blocked_needs_dm_only": (
        "Nominal straight path blocked; DM-only required"
    ),
    "nominal_path_crosses_invalid_domain_needs_dm_only": (
        "Inner nominal line crosses invalid domain; Test 2 required"
    ),
    "nominal_line_misses_full_sphere_needs_dm_only": (
        "Nominal line misses R_full sphere; DM-only required"
    ),
    "nominal_line_misses_full_sphere_outward_force_reject": (
        "Nominal line misses R_full and force is everywhere outward; reject"
    ),
    "needs_larger_R_max": "R_max too small",
    "invalid_far_start_below_electrode": (
        "Invalid far start below electrode"
    ),
    "path_crosses_invalid_domain": (
        "Nominal path crosses invalid domain; DM-only required"
    ),
    "no_valid_points": "No valid path points",
    "invalid_speed": "Invalid speed",
    "nonpositive_K_at_R_far": "Nonpositive kinetic energy at R_far",
}

STATUS_COLORS = {
    "straight_safe_to_full_sphere": "limegreen",
    "straight_then_dm_only": "royalblue",
    "straight_path_blocked_needs_dm_only": "orange",
    "nominal_path_crosses_invalid_domain_needs_dm_only": "mediumorchid",
    "nominal_line_misses_full_sphere_needs_dm_only": "darkcyan",
    "nominal_line_misses_full_sphere_outward_force_reject": "crimson",
    "needs_larger_R_max": "red",
    "invalid_far_start_below_electrode": "black",
    "path_crosses_invalid_domain": "purple",
    "no_valid_points": "gray",
    "invalid_speed": "saddlebrown",
    "nonpositive_K_at_R_far": "magenta",
}

STATUS_MARKERS = {
    "straight_safe_to_full_sphere": "o",
    "straight_then_dm_only": "s",
    "straight_path_blocked_needs_dm_only": "^",
    "nominal_path_crosses_invalid_domain_needs_dm_only": "D",
    "nominal_line_misses_full_sphere_needs_dm_only": "h",
    "nominal_line_misses_full_sphere_outward_force_reject": ">",
    "needs_larger_R_max": "x",
    "invalid_far_start_below_electrode": "v",
    "path_crosses_invalid_domain": "D",
    "no_valid_points": "P",
    "invalid_speed": "*",
    "nonpositive_K_at_R_far": "X",
}

LIMITING_ORDER = [
    "energy",
    "angle",
    "displacement",
    "R_max_energy",
    "R_max_nonpositive_K",
    "not_applicable",
]

LIMITING_LABELS = {
    "energy": "Energy-tail tolerance",
    "angle": "Angular-bending tolerance",
    "displacement": "Transverse-displacement tolerance",
    "R_max_energy": "Energy tail still too large at R_max",
    "R_max_nonpositive_K": "K <= 0 at R_max",
    "not_applicable": "Not applicable",
}

LIMITING_COLORS = {
    "energy": "gold",
    "angle": "deepskyblue",
    "displacement": "limegreen",
    "R_max_energy": "red",
    "R_max_nonpositive_K": "darkred",
    "not_applicable": "lightgray",
}

RESOLVED_R_FAR_STATUSES = {
    "straight_safe_to_full_sphere",
    "straight_then_dm_only",
    "straight_path_blocked_needs_dm_only",
    "nominal_path_crosses_invalid_domain_needs_dm_only",
    "nominal_line_misses_full_sphere_needs_dm_only",
}

INVALID_TRAJECTORY_STATUSES = {
    "invalid_far_start_below_electrode",
    "path_crosses_invalid_domain",
    "no_valid_points",
    "invalid_speed",
    "nonpositive_K_at_R_far",
}

SCREENED_REJECT_STATUSES = {
    "nominal_line_misses_full_sphere_outward_force_reject",
}

EXCLUDED_TRAJECTORY_STATUSES = (
    INVALID_TRAJECTORY_STATUSES | SCREENED_REJECT_STATUSES
)

WEIGHTED_STATUS_FRACTION_STATUSES = [
    "straight_safe_to_full_sphere",
    "straight_then_dm_only",
    "straight_path_blocked_needs_dm_only",
    "nominal_path_crosses_invalid_domain_needs_dm_only",
    "nominal_line_misses_full_sphere_needs_dm_only",
    "needs_larger_R_max",
]

DM_ONLY_REQUIRED_STATUSES = {
    "straight_then_dm_only",
    "straight_path_blocked_needs_dm_only",
    "nominal_path_crosses_invalid_domain_needs_dm_only",
    "nominal_line_misses_full_sphere_needs_dm_only",
}

TEST2_RESOLVED_OUTER_STATUSES = {
    "straight_safe_to_full_sphere",
    "straight_then_dm_only",
}

# ============================================================
# BASIC GEOMETRY
# ============================================================

def mb_scale(T, mass):
    return np.sqrt(k_B * T / mass)


def fibonacci_sphere_points(n, radius):
    """Return approximately uniform Cartesian points on a sphere."""
    if n < 1:
        raise ValueError("n must be at least one")
    if radius <= 0.0:
        raise ValueError("radius must be positive")

    indices = np.arange(n, dtype=float)
    z = 1.0 - 2.0 * (indices + 0.5) / n
    phi = np.pi * (3.0 - np.sqrt(5.0)) * indices
    radial = np.sqrt(np.maximum(1.0 - z**2, 0.0))

    return radius * np.column_stack([
        radial * np.cos(phi),
        radial * np.sin(phi),
        z,
    ])


def evaluate_potential_points(xyz, mass, charge_fraction, component):
    """Evaluate potential energy at many points with a scalar fallback."""
    xyz = np.asarray(xyz, dtype=float)
    try:
        values = np.asarray(
            potential_energy(
                xyz,
                mass,
                charge_fraction,
                component=component,
            ),
            dtype=float,
        ).reshape(-1)
        if values.shape[0] != xyz.shape[0]:
            raise ValueError("Unexpected vectorized potential shape")
        return values
    except Exception:
        return np.array([
            float(np.asarray(
                potential_energy(
                    point,
                    mass,
                    charge_fraction,
                    component=component,
                )
            ).reshape(-1)[0])
            for point in xyz
        ], dtype=float)


def compute_handoff_barrier_speed(
    mass,
    charge_fraction,
    T,
    radius,
    n_sphere,
    component="total",
    U_reference=0.0,
):
    """Compute the easiest valid sphere point and its MB tail probability.

    This reproduces the prior sphere-barrier calculation: the minimum positive
    potential-energy rise on the valid part of the R_full sphere determines
    the global speed cutoff.  Direction-dependent path barriers are still
    evaluated later for each trajectory.
    """
    xyz_all = fibonacci_sphere_points(n_sphere, radius)
    geometry_valid = valid_xyz_mask_local(xyz_all)
    xyz_valid = xyz_all[geometry_valid]

    if xyz_valid.shape[0] == 0:
        raise RuntimeError("No valid points on the R_full sphere")

    U = evaluate_potential_points(
        xyz_valid,
        mass,
        charge_fraction,
        component,
    )
    finite = np.isfinite(U)
    if not np.any(finite):
        raise RuntimeError("No finite potential values on the R_full sphere")

    xyz_valid = xyz_valid[finite]
    U = U[finite]
    delta_U = np.maximum(U - U_reference, 0.0)
    index = int(np.argmin(delta_U))
    barrier_J = float(delta_U[index])
    v_min = float(np.sqrt(2.0 * barrier_J / mass))
    scale = mb_scale(T, mass)
    tail_probability = float(maxwell.sf(v_min, scale=scale))

    return {
        "minimum_speed_m_s": v_min,
        "barrier_J": barrier_J,
        "tail_probability": tail_probability,
        "easiest_xyz_m": xyz_valid[index].copy(),
        "easiest_U_J": float(U[index]),
        "valid_sphere_fraction": float(np.mean(geometry_valid)),
        "n_valid_finite": int(len(U)),
        "component": component,
    }


def make_truncated_mb_speed_cases(conditional_quantiles, scale, v_cut):
    """Sample midpoint/representative quantiles of MB conditioned on v>=v_cut."""
    conditional_quantiles = np.asarray(conditional_quantiles, dtype=float)
    if np.any((conditional_quantiles <= 0.0) | (conditional_quantiles >= 1.0)):
        raise ValueError("Conditional speed quantiles must lie strictly in (0, 1)")

    cutoff_cdf = float(maxwell.cdf(v_cut, scale=scale))
    tail_probability = float(maxwell.sf(v_cut, scale=scale))
    if not np.isfinite(tail_probability) or tail_probability <= 0.0:
        raise ValueError("The MB probability above the speed cutoff is zero")

    speed_cases = []
    n_cases = len(conditional_quantiles)
    for conditional_q in conditional_quantiles:
        unconditional_q = cutoff_cdf + conditional_q * tail_probability
        unconditional_q = min(unconditional_q, np.nextafter(1.0, 0.0))
        speed = float(maxwell.ppf(unconditional_q, scale=scale))
        if not speed > v_cut:
            speed = float(np.nextafter(v_cut, np.inf))

        speed_cases.append({
            "key": f"tailq_{conditional_q:.3f}",
            "quantile": float(conditional_q),
            "conditional_quantile": float(conditional_q),
            "unconditional_quantile": float(unconditional_q),
            "speed_m_s": speed,
            "tail_probability": tail_probability,
            "conditional_speed_weight": 1.0 / n_cases,
            "unconditional_speed_weight": tail_probability / n_cases,
            "minimum_speed_m_s": float(v_cut),
        })

    return speed_cases


def outward_force_screen(
    F,
    valid,
    b_vec,
    dot_tolerance_J=0.0,
    require_complete_valid_path=True,
):
    """Test whether the force always pushes outward along a nominal miss.

    The requested diagnostic is dot(F, b_vec) > 0.  Since b_vec is fixed and
    perpendicular to the incoming direction, a positive value is an outward
    force component in the impact-parameter direction.
    """
    F = np.asarray(F, dtype=float)
    valid = np.asarray(valid, dtype=bool)
    b_vec = np.asarray(b_vec, dtype=float).reshape(3)
    b_norm = float(np.linalg.norm(b_vec))

    finite_force = np.all(np.isfinite(F), axis=1)
    test_mask = valid & finite_force

    if b_norm <= 0.0 or not np.any(test_mask):
        return {
            "reject": False,
            "n_tested": int(np.sum(test_mask)),
            "complete_valid_path": False,
            "min_F_dot_b_vec_J": np.nan,
            "max_F_dot_b_vec_J": np.nan,
            "min_F_dot_bhat_N": np.nan,
            "max_F_dot_bhat_N": np.nan,
        }

    complete_valid_path = bool(np.all(test_mask))
    projections_J = F[test_mask] @ b_vec
    projections_N = projections_J / b_norm

    reject = bool(np.all(projections_J > dot_tolerance_J))
    if require_complete_valid_path and not complete_valid_path:
        reject = False

    return {
        "reject": reject,
        "n_tested": int(np.sum(test_mask)),
        "complete_valid_path": complete_valid_path,
        "min_F_dot_b_vec_J": float(np.min(projections_J)),
        "max_F_dot_b_vec_J": float(np.max(projections_J)),
        "min_F_dot_bhat_N": float(np.min(projections_N)),
        "max_F_dot_bhat_N": float(np.max(projections_N)),
    }


def direction_basis(theta, alpha, psi):
    """
    Incoming direction convention:

        u = [
            cos(theta),
            sin(theta) cos(alpha),
            sin(theta) sin(alpha)
        ]

    The nominal path is

        r(s) = b_vec - s u

    and the incoming velocity points along +u.
    """
    u = np.array([
        np.cos(theta),
        np.sin(theta) * np.cos(alpha),
        np.sin(theta) * np.sin(alpha),
    ], dtype=float)

    e_theta = np.array([
        -np.sin(theta),
        np.cos(theta) * np.cos(alpha),
        np.cos(theta) * np.sin(alpha),
    ], dtype=float)

    e_alpha = np.array([
        0.0,
        -np.sin(alpha),
        np.cos(alpha),
    ], dtype=float)

    b_hat = np.cos(psi) * e_theta + np.sin(psi) * e_alpha

    u_norm = np.linalg.norm(u)
    if u_norm == 0.0:
        raise ValueError("Incoming direction vector has zero norm")
    u /= u_norm

    b_hat_norm = np.linalg.norm(b_hat)
    if b_hat_norm > 0.0:
        b_hat /= b_hat_norm

    return u, b_hat


def make_impact_geometry_samples(
    b_max,
    R_full,
    n_b_inner,
    n_b_outer,
    n_psi,
    include_b_zero=True,
    include_R_full=True,
    include_b_max=True,
):
    """Build a piecewise equal-area b-psi quadrature.

    R_full is inserted as an exact annulus edge. The inner and outer regions
    are each divided uniformly in b^2, which is the natural area coordinate.
    Unlike the old six-bin scan, the central fraction is represented exactly:

        sum(weight for b < R_full) = (R_full / b_max)^2.

    The representative radius of annulus [b_lo, b_hi] is its area midpoint,

        b_mid = sqrt((b_lo^2 + b_hi^2) / 2).

    Each psi point receives one n_psi-th of the annulus area weight.
    """
    if not (0.0 < R_full < b_max):
        raise ValueError("Require 0 < R_full < b_max")
    if n_b_inner < 1 or n_b_outer < 1:
        raise ValueError("n_b_inner and n_b_outer must be at least 1")
    if n_psi < 1:
        raise ValueError("n_psi must be at least 1")

    psi_values = np.linspace(0.0, 2.0 * np.pi, n_psi, endpoint=False)
    cases = []
    case_id = 0
    weighted_index = 0

    if include_b_zero:
        cases.append({
            "impact_case_id": case_id,
            "b_m": 0.0,
            "psi_rad": 0.0,
            "impact_weight": 0.0,
            "impact_sample_type": "b_zero_diagnostic",
            "b_region": "diagnostic",
            "b_lower_m": 0.0,
            "b_upper_m": 0.0,
            "b_index": -1,
            "psi_index": 0,
        })
        case_id += 1

    inner_edges_sq = np.linspace(0.0, R_full**2, n_b_inner + 1)
    outer_edges_sq = np.linspace(R_full**2, b_max**2, n_b_outer + 1)

    def append_region(edges_sq, region_name):
        nonlocal case_id, weighted_index
        for local_index in range(len(edges_sq) - 1):
            b_lo_sq = float(edges_sq[local_index])
            b_hi_sq = float(edges_sq[local_index + 1])
            b_lo = np.sqrt(b_lo_sq)
            b_hi = np.sqrt(b_hi_sq)
            b_mid = np.sqrt(0.5 * (b_lo_sq + b_hi_sq))
            annulus_weight = (b_hi_sq - b_lo_sq) / b_max**2

            for psi_index, psi_value in enumerate(psi_values):
                cases.append({
                    "impact_case_id": case_id,
                    "b_m": float(b_mid),
                    "psi_rad": float(psi_value),
                    "impact_weight": float(annulus_weight / n_psi),
                    "impact_sample_type": "equal_area_weighted",
                    "b_region": region_name,
                    "b_lower_m": float(b_lo),
                    "b_upper_m": float(b_hi),
                    "b_index": int(weighted_index),
                    "b_region_index": int(local_index),
                    "psi_index": int(psi_index),
                })
                case_id += 1
            weighted_index += 1

    append_region(inner_edges_sq, "intersects_R_full")
    append_region(outer_edges_sq, "nominal_miss")

    if include_R_full:
        for psi_index, psi_value in enumerate(psi_values):
            cases.append({
                "impact_case_id": case_id,
                "b_m": float(R_full),
                "psi_rad": float(psi_value),
                "impact_weight": 0.0,
                "impact_sample_type": "R_full_boundary_diagnostic",
                "b_region": "diagnostic",
                "b_lower_m": float(R_full),
                "b_upper_m": float(R_full),
                "b_index": int(weighted_index),
                "psi_index": int(psi_index),
            })
            case_id += 1

    if include_b_max:
        for psi_index, psi_value in enumerate(psi_values):
            cases.append({
                "impact_case_id": case_id,
                "b_m": float(b_max),
                "psi_rad": float(psi_value),
                "impact_weight": 0.0,
                "impact_sample_type": "b_max_diagnostic",
                "b_region": "diagnostic",
                "b_lower_m": float(b_max),
                "b_upper_m": float(b_max),
                "b_index": int(weighted_index + 1),
                "psi_index": int(psi_index),
            })
            case_id += 1

    positive = [case for case in cases if case["impact_weight"] > 0.0]
    total_positive_weight = sum(case["impact_weight"] for case in positive)
    inner_positive_weight = sum(
        case["impact_weight"]
        for case in positive
        if case["b_region"] == "intersects_R_full"
    )
    expected_inner_weight = (R_full / b_max)**2

    if not np.isclose(total_positive_weight, 1.0, rtol=0.0, atol=1.0e-12):
        raise RuntimeError(
            "Positive impact weights do not sum to one: "
            f"{total_positive_weight}"
        )
    if not np.isclose(
        inner_positive_weight,
        expected_inner_weight,
        rtol=0.0,
        atol=1.0e-12,
    ):
        raise RuntimeError(
            "Inner impact weight does not match the physical disk fraction: "
            f"computed={inner_positive_weight}, expected={expected_inner_weight}"
        )

    return cases


def append_reach_diagnostic_impact_cases(
    impact_cases,
    *,
    speed_case,
    R_full,
    b_max,
    n_psi,
):
    """Append zero-weight low-b diagnostics for every psi ray.

    Test 0.75 identifies where Rutherford collision energetics are accurate,
    but it does not prove that the trap allows the trajectory to reach the
    collision region. These rows give Test 2/Test 2.5 explicit near-axis
    probes, including b=0 for every psi-labelled ray.
    """
    if not INCLUDE_REACH_DIAGNOSTICS:
        return impact_cases

    cases = [dict(case) for case in impact_cases]
    psi_values = np.linspace(0.0, 2.0 * np.pi, n_psi, endpoint=False)
    b_ruth = float(speed_case.get("b_rutherford_max_m", 0.0))
    b_adia = float(speed_case.get("b_adiabatic_min_m", np.inf))

    # Use R_full as the default diagnostic scale. If the conservative
    # adiabatic ceiling lies below R_full, do not create diagnostics beyond it.
    scale = min(float(R_full), float(b_max))
    if np.isfinite(b_adia) and b_adia > 0.0:
        scale = min(scale, b_adia)
    scale = max(scale, 0.0)

    targets = [float(fraction) * scale for fraction in REACH_DIAGNOSTIC_FRACTIONS]
    if INCLUDE_RUTHERFORD_BOUNDARY_DIAGNOSTIC and np.isfinite(b_ruth) and b_ruth > 0.0:
        targets.append(min(b_ruth, float(b_max)))
    targets = sorted({float(np.clip(value, 0.0, b_max)) for value in targets})

    # Diagnostic rows must be unique by diagnostic role as well as by
    # physical (b, psi). A weighted or generic diagnostic row at the same
    # geometry cannot replace a per-ray reach diagnostic because the symmetry
    # reconstruction maps impact_case_id using impact_sample_type too.
    existing = {
        (
            str(case.get("impact_sample_type", "")),
            round(float(case["b_m"]), 15),
            round(float(np.mod(case["psi_rad"], 2.0 * np.pi)), 12),
        )
        for case in cases
    }
    next_case_id = 1 + max(int(case["impact_case_id"]) for case in cases)
    for target_index, b_value in enumerate(targets):
        for psi_index, psi_value in enumerate(psi_values):
            sample_type = (
                "b_zero_ray_diagnostic"
                if np.isclose(b_value, 0.0, rtol=0.0, atol=1.0e-18)
                else "reach_boundary_diagnostic"
            )
            if np.isfinite(b_ruth) and b_ruth > 0.0 and np.isclose(
                b_value, b_ruth, rtol=1.0e-12, atol=1.0e-15
            ):
                sample_type = "rutherford_reach_boundary_diagnostic"

            key = _rounded_geometry_key(sample_type, b_value, psi_value)
            if key in existing:
                continue

            cases.append(
                {
                    "impact_case_id": next_case_id,
                    "b_m": float(b_value),
                    "psi_rad": float(psi_value),
                    "impact_weight": 0.0,
                    "impact_sample_type": sample_type,
                    "b_region": "reach_diagnostic",
                    "b_lower_m": float(b_value),
                    "b_upper_m": float(b_value),
                    "b_index": int(-1000 - target_index),
                    "b_region_index": int(target_index),
                    "psi_index": int(psi_index),
                }
            )
            existing.add(key)
            next_case_id += 1
    return cases



def make_direction_cases(n_theta, n_alpha, remove_duplicated_poles=True):
    """Build unique theta-alpha direction cases.

    The rectangular plotting arrays are retained, but only alpha=0 is sampled
    at theta=0 and theta=pi because every alpha at a pole represents the same
    incoming direction.
    """
    theta_values = np.linspace(0.0, np.pi, n_theta)
    alpha_values = np.linspace(
        0.0,
        2.0 * np.pi,
        n_alpha,
        endpoint=False,
    )

    cases = []
    for i, theta in enumerate(theta_values):
        at_pole = np.isclose(theta, 0.0) or np.isclose(theta, np.pi)
        alpha_indices = [0] if (remove_duplicated_poles and at_pole) else range(n_alpha)

        for j in alpha_indices:
            cases.append({
                "theta_index": int(i),
                "alpha_index": int(j),
                "theta_rad": float(theta),
                "alpha_rad": float(alpha_values[j]),
                "theta_deg": float(np.degrees(theta)),
                "alpha_deg": float(np.degrees(alpha_values[j])),
                "is_pole": bool(at_pole),
            })

    return theta_values, alpha_values, cases



def make_xy_symmetry_direction_cases(
    n_theta,
    n_alpha,
    remove_duplicated_poles=True,
    exploit_symmetry=True,
):
    """Build full directions, symmetry representatives, and reconstruction map.

    Direction convention:

        u = [cos(theta), sin(theta) cos(alpha), sin(theta) sin(alpha)].

    Exact trap symmetries used here:

        Sx: (x, y, z) -> (-x, y, z)
        Sy: (x, y, z) -> (x, -y, z)

    No z reflection is assumed. The canonical domain is therefore

        0 <= theta <= pi/2,
        cos(alpha) >= 0,

    which is alpha in [0, pi/2] union [3*pi/2, 2*pi). Both signs of z are
    retained. For the coarse 13 x 24 grid this reduces 266 unique directions
    to 79 field-evaluated representatives.
    """
    theta_values, alpha_values, full_cases = make_direction_cases(
        n_theta=n_theta,
        n_alpha=n_alpha,
        remove_duplicated_poles=remove_duplicated_poles,
    )

    if not exploit_symmetry:
        mapping_rows = []
        for case in full_cases:
            mapping_rows.append({
                **case,
                "canonical_theta_index": case["theta_index"],
                "canonical_alpha_index": case["alpha_index"],
                "reflect_x": False,
                "reflect_y": False,
            })
        return (
            theta_values,
            alpha_values,
            full_cases,
            full_cases,
            pd.DataFrame(mapping_rows),
        )

    if n_alpha % 2 != 0:
        raise ValueError(
            "n_alpha must be even when exploiting y-reflection symmetry"
        )

    middle_theta_index = (n_theta - 1) // 2
    if not np.isclose(theta_values[middle_theta_index], 0.5 * np.pi):
        raise ValueError(
            "theta grid must contain pi/2 when exploiting x symmetry"
        )

    full_lookup = {
        (case["theta_index"], case["alpha_index"]): case
        for case in full_cases
    }
    scan_keys = set()
    mapping_rows = []

    for case in full_cases:
        i = int(case["theta_index"])
        j = int(case["alpha_index"])
        theta = float(case["theta_rad"])
        alpha = float(case["alpha_rad"])

        reflect_x = bool(theta > 0.5 * np.pi + 1.0e-12)
        canonical_i = n_theta - 1 - i if reflect_x else i

        at_pole = np.isclose(theta, 0.0) or np.isclose(theta, np.pi)
        if at_pole:
            reflect_y = False
            canonical_j = 0
        else:
            reflect_y = bool(np.cos(alpha) < -1.0e-12)
            canonical_j = (
                (n_alpha // 2 - j) % n_alpha
                if reflect_y else j
            )

        canonical_key = (int(canonical_i), int(canonical_j))
        if canonical_key not in full_lookup:
            raise RuntimeError(
                "Could not locate canonical direction for "
                f"theta_index={i}, alpha_index={j}: {canonical_key}"
            )

        scan_keys.add(canonical_key)
        mapping_rows.append({
            **case,
            "canonical_theta_index": canonical_key[0],
            "canonical_alpha_index": canonical_key[1],
            "reflect_x": reflect_x,
            "reflect_y": reflect_y,
        })

    scan_cases = [full_lookup[key] for key in sorted(scan_keys)]
    symmetry_map_df = pd.DataFrame(mapping_rows).sort_values(
        ["theta_index", "alpha_index"],
        kind="mergesort",
    ).reset_index(drop=True)

    return (
        theta_values,
        alpha_values,
        full_cases,
        scan_cases,
        symmetry_map_df,
    )


def transform_psi_by_xy_reflections(psi, reflect_x, reflect_y):
    """Transform the impact-plane angle under x/y reflections."""
    psi = float(psi)
    if reflect_x and reflect_y:
        transformed = psi + np.pi
    elif reflect_x:
        transformed = np.pi - psi
    elif reflect_y:
        transformed = -psi
    else:
        transformed = psi
    return float(np.mod(transformed, 2.0 * np.pi))


def _rounded_geometry_key(sample_type, b_m, psi_rad):
    return (
        str(sample_type),
        round(float(b_m), 15),
        round(float(np.mod(psi_rad, 2.0 * np.pi)), 12),
    )


def build_impact_case_symmetry_lookup(impact_cases):
    """Map transformed (sample type, b, psi) values to impact cases."""
    lookup = {}
    for case in impact_cases:
        preserve_zero_psi = (
            str(case.get("impact_sample_type", "")) == "b_zero_ray_diagnostic"
        )
        psi = (
            case["psi_rad"]
            if preserve_zero_psi or not np.isclose(case["b_m"], 0.0)
            else 0.0
        )
        key = _rounded_geometry_key(
            case["impact_sample_type"],
            case["b_m"],
            psi,
        )
        lookup[key] = case
    return lookup


def expand_rows_by_xy_symmetry(
    rows_df,
    symmetry_map_df,
    impact_cases,
):
    """Reconstruct full-direction rows from symmetry-reduced calculations.

    Scalar diagnostics are copied. x/y components of positions and velocities
    are reflected. psi and impact_case_id are mapped to the corresponding
    sampled impact geometry. This also works for analytical prefilter rows.
    """
    if len(rows_df) == 0:
        return rows_df.copy()

    targets_by_canonical = {
        key: group.to_dict("records")
        for key, group in symmetry_map_df.groupby(
            ["canonical_theta_index", "canonical_alpha_index"],
            sort=False,
        )
    }
    impact_lookup = build_impact_case_symmetry_lookup(impact_cases)
    expanded = []

    for source in rows_df.to_dict("records"):
        source_key = (
            int(source["theta_index"]),
            int(source["alpha_index"]),
        )
        targets = targets_by_canonical.get(source_key)
        if targets is None:
            raise RuntimeError(
                f"Missing symmetry targets for canonical direction {source_key}"
            )

        for target in targets:
            reflect_x = bool(target["reflect_x"])
            reflect_y = bool(target["reflect_y"])
            row = dict(source)

            row["symmetry_source_theta_index"] = source_key[0]
            row["symmetry_source_alpha_index"] = source_key[1]
            row["symmetry_reflect_x"] = reflect_x
            row["symmetry_reflect_y"] = reflect_y
            row["symmetry_reconstructed"] = bool(reflect_x or reflect_y)

            row["theta_index"] = int(target["theta_index"])
            row["alpha_index"] = int(target["alpha_index"])
            row["theta_rad"] = float(target["theta_rad"])
            row["alpha_rad"] = float(target["alpha_rad"])
            row["theta_deg"] = float(target["theta_deg"])
            row["alpha_deg"] = float(target["alpha_deg"])

            b_m = float(row.get("b_m", 0.0))
            preserve_zero_psi = (
                str(row.get("impact_sample_type", ""))
                == "b_zero_ray_diagnostic"
            )
            if np.isclose(b_m, 0.0) and not preserve_zero_psi:
                psi_target = 0.0
            else:
                psi_target = transform_psi_by_xy_reflections(
                    row.get("psi_rad", 0.0),
                    reflect_x,
                    reflect_y,
                )

            if "psi_rad" in row:
                row["psi_rad"] = psi_target
            if "psi_deg" in row:
                row["psi_deg"] = float(np.degrees(psi_target))

            if "impact_sample_type" in row and "impact_case_id" in row:
                key = _rounded_geometry_key(
                    row["impact_sample_type"],
                    b_m,
                    psi_target,
                )
                target_impact = impact_lookup.get(key)
                if target_impact is None and np.isclose(b_m, 0.0):
                    key = _rounded_geometry_key(
                        row["impact_sample_type"],
                        b_m,
                        0.0,
                    )
                    target_impact = impact_lookup.get(key)
                if target_impact is None:
                    raise RuntimeError(
                        "Could not map reflected impact geometry: "
                        f"type={row['impact_sample_type']}, b={b_m}, "
                        f"psi={psi_target}"
                    )
                row["impact_case_id"] = int(target_impact["impact_case_id"])
                if "psi_index" in target_impact:
                    row["psi_index"] = int(target_impact["psi_index"])
                if "b_index" in target_impact:
                    row["b_index"] = int(target_impact["b_index"])

            if reflect_x:
                for column in (
                    "x_far_m",
                    "vx_far_m_s",
                    "u_x",
                    "bvec_x_m",
                ):
                    if column in row and np.isfinite(row[column]):
                        row[column] = -row[column]
            if reflect_y:
                for column in (
                    "y_far_m",
                    "vy_far_m_s",
                    "u_y",
                    "bvec_y_m",
                ):
                    if column in row and np.isfinite(row[column]):
                        row[column] = -row[column]

            expanded.append(row)

    return pd.DataFrame(expanded)


def expand_direction_audit_by_xy_symmetry(
    direction_df,
    symmetry_map_df,
):
    """Expand a direction-level audit table to the complete angular grid."""
    if len(direction_df) == 0:
        return direction_df.copy()

    targets_by_canonical = {
        key: group.to_dict("records")
        for key, group in symmetry_map_df.groupby(
            ["canonical_theta_index", "canonical_alpha_index"],
            sort=False,
        )
    }
    expanded = []
    for source in direction_df.to_dict("records"):
        key = (int(source["theta_index"]), int(source["alpha_index"]))
        for target in targets_by_canonical.get(key, []):
            row = dict(source)
            row.update({
                "theta_index": int(target["theta_index"]),
                "alpha_index": int(target["alpha_index"]),
                "theta_rad": float(target["theta_rad"]),
                "alpha_rad": float(target["alpha_rad"]),
                "theta_deg": float(target["theta_deg"]),
                "alpha_deg": float(target["alpha_deg"]),
                "symmetry_source_theta_index": key[0],
                "symmetry_source_alpha_index": key[1],
                "symmetry_reflect_x": bool(target["reflect_x"]),
                "symmetry_reflect_y": bool(target["reflect_y"]),
                "symmetry_reconstructed": bool(
                    target["reflect_x"] or target["reflect_y"]
                ),
            })
            expanded.append(row)
    return pd.DataFrame(expanded)


def straight_path_geometry_prefilter(
    theta,
    alpha,
    b,
    psi,
    R_inner,
    R_outer,
):
    """Classify electrode-plane validity without constructing the path grid.

    Along r(s) = b_vec - s*u, z(s) is linear in s. Its minimum over the
    sampled segment is therefore attained at one endpoint. This gives an
    exact and inexpensive prefilter for invalid_far_start_below_electrode and
    path_crosses_invalid_domain.
    """
    if b >= R_outer:
        return {
            "complete_valid_path": False,
            "far_valid": False,
            "prefilter_reason": "b_not_smaller_than_R_outer",
            "z_inner_m": np.nan,
            "z_outer_m": np.nan,
            "minimum_z_m": np.nan,
        }

    u, b_hat = direction_basis(theta, alpha, psi)
    b_vec = b * b_hat
    s_outer = np.sqrt(max(R_outer**2 - b**2, 0.0))

    if b < R_inner:
        s_inner = np.sqrt(max(R_inner**2 - b**2, 0.0))
    else:
        s_inner = max(1.0e-9, 0.02 * R_inner)

    if not (s_outer > s_inner > 0.0):
        return {
            "complete_valid_path": False,
            "far_valid": False,
            "prefilter_reason": "invalid_path_coordinate_range",
            "z_inner_m": np.nan,
            "z_outer_m": np.nan,
            "minimum_z_m": np.nan,
        }

    z_inner = float(b_vec[2] - s_inner * u[2])
    z_outer = float(b_vec[2] - s_outer * u[2])
    minimum_z = min(z_inner, z_outer)
    z_floor = -c.ion_height + z_margin

    far_valid = z_outer > z_floor
    complete_valid = minimum_z > z_floor

    if complete_valid:
        reason = "valid"
    elif not far_valid:
        reason = "invalid_far_start_below_electrode"
    else:
        reason = "path_crosses_invalid_domain"

    return {
        "complete_valid_path": bool(complete_valid),
        "far_valid": bool(far_valid),
        "prefilter_reason": reason,
        "z_inner_m": z_inner,
        "z_outer_m": z_outer,
        "minimum_z_m": minimum_z,
    }


def prefilter_geometry_tasks(
    direction_cases,
    impact_cases,
    R_full,
    R_max,
):
    """Remove invalid individual direction-impact geometries before U/F calls."""
    kept_tasks = []
    excluded_rows = []

    for direction in direction_cases:
        for impact_case in impact_cases:
            info = straight_path_geometry_prefilter(
                theta=direction["theta_rad"],
                alpha=direction["alpha_rad"],
                b=impact_case["b_m"],
                psi=impact_case["psi_rad"],
                R_inner=R_full,
                R_outer=R_max,
            )

            if info["complete_valid_path"]:
                kept_tasks.append((direction, impact_case))
            else:
                excluded_rows.append({
                    **direction,
                    **impact_case,
                    **info,
                    "b_um": impact_case["b_m"] * 1.0e6,
                    "psi_deg": np.degrees(impact_case["psi_rad"]),
                })

    return kept_tasks, pd.DataFrame(excluded_rows)


def prefilter_direction_cases(
    direction_cases,
    impact_cases,
    R_full,
    R_max,
    n_path,
    positive_weight_only=True,
):
    """Remove directions whose sampled impact paths are geometrically invalid.

    This prefilter uses only the electrode-plane validity mask; it does not
    call potential_energy or force.  A direction is retained when at least one
    tested impact geometry has a complete nominal path in the valid half-space.
    Individual invalid b-psi trajectories within a retained direction are
    still removed after the full field evaluation.
    """
    if positive_weight_only:
        impact_subset = [
            case for case in impact_cases
            if case["impact_weight"] > 0.0
        ]
    else:
        impact_subset = list(impact_cases)

    kept = []
    excluded_rows = []

    for direction in direction_cases:
        n_tested = 0
        n_far_valid = 0
        n_complete_valid = 0

        for impact_case in impact_subset:
            n_tested += 1
            info = straight_path_geometry_prefilter(
                theta=direction["theta_rad"],
                alpha=direction["alpha_rad"],
                b=impact_case["b_m"],
                psi=impact_case["psi_rad"],
                R_inner=R_full,
                R_outer=R_max,
            )
            if info["far_valid"]:
                n_far_valid += 1
            if info["complete_valid_path"]:
                n_complete_valid += 1

        if n_complete_valid > 0:
            kept.append(direction)
        else:
            reason = (
                "invalid_far_start_below_electrode"
                if n_far_valid == 0
                else "path_crosses_invalid_domain"
            )
            excluded_rows.append({
                **direction,
                "prefilter_reason": reason,
                "n_impact_paths_tested": n_tested,
                "n_far_valid_paths": n_far_valid,
                "n_complete_valid_paths": n_complete_valid,
            })

    return kept, pd.DataFrame(excluded_rows)

def line_intersects_sphere(b, radius):
    return b < radius


def path_coordinate_at_sphere(b, radius):
    if not line_intersects_sphere(b, radius):
        return np.nan
    return np.sqrt(max(radius**2 - b**2, 0.0))


def build_straight_path(
    theta,
    alpha,
    b,
    psi,
    R_inner,
    R_outer,
    n_path,
):
    """
    Construct a nominal line for the outer-tail diagnostic.

    R_inner and R_outer are physical radii from the ion. For b<R_inner,
    the path starts at the line's intersection with the R_inner sphere.
    For b>=R_inner, the nominal line misses that sphere, so the diagnostic
    is extended near closest approach. That case is explicitly classified
    as requiring DM-only propagation.
    """
    if b >= R_outer:
        raise ValueError("b must be smaller than R_outer")

    u, b_hat = direction_basis(theta, alpha, psi)
    b_vec = b * b_hat

    s_outer = np.sqrt(max(R_outer**2 - b**2, 0.0))

    if b < R_inner:
        s_inner = np.sqrt(max(R_inner**2 - b**2, 0.0))
    else:
        # Include the region near nominal closest approach. A small positive
        # floor is required for geomspace.
        s_inner = max(1.0e-9, 0.02 * R_inner)

    if not (s_outer > s_inner > 0.0):
        raise ValueError(
            f"Invalid path-coordinate range: s_inner={s_inner}, "
            f"s_outer={s_outer}, b={b}"
        )

    s_grid = np.geomspace(s_inner, s_outer, n_path)
    xyz = b_vec[None, :] - s_grid[:, None] * u[None, :]

    return s_grid, xyz, u, b_vec

# ============================================================
# POTENTIAL DOMAIN AND EVALUATION
# ============================================================

def valid_xyz_mask_local(xyz):
    xyz = np.asarray(xyz, dtype=float)
    return xyz[..., 2] > (-c.ion_height + z_margin)


def evaluate_trap_path(xyz, mass, charge_fraction):
    """
    Evaluate trap-only potential energy and trap-only force.
    The ion-DM Coulomb interaction is excluded here.
    """
    xyz = np.asarray(xyz, dtype=float)
    n = xyz.shape[0]

    U = np.full(n, np.nan, dtype=float)
    F = np.full((n, 3), np.nan, dtype=float)

    geometry_valid = valid_xyz_mask_local(xyz)
    valid_indices = np.flatnonzero(geometry_valid)

    if valid_indices.size == 0:
        return U, F, geometry_valid & False

    xyz_valid = xyz[valid_indices]

    # Prefer vectorized potential evaluation when the existing implementation
    # supports it; otherwise fall back to point-by-point evaluation.
    try:
        U_valid = np.asarray(
            potential_energy(
                xyz_valid,
                mass,
                charge_fraction,
                component="trap",
            ),
            dtype=float,
        ).reshape(-1)

        if U_valid.shape[0] != xyz_valid.shape[0]:
            raise ValueError("Unexpected vectorized potential shape")

    except Exception:
        U_valid = np.array([
            float(
                np.asarray(
                    potential_energy(
                        point,
                        mass,
                        charge_fraction,
                        component="trap",
                    )
                ).reshape(-1)[0]
            )
            for point in xyz_valid
        ], dtype=float)

    U[valid_indices] = U_valid

    try:
        F_valid = np.asarray(
            force(
                xyz_valid,
                mass,
                charge_fraction,
                component="trap",
            ),
            dtype=float,
        ).reshape(-1, 3)

        if F_valid.shape[0] != xyz_valid.shape[0]:
            raise ValueError("Unexpected vectorized force shape")

    except Exception:
        F_valid = np.vstack([
            np.asarray(
                force(
                    point,
                    mass,
                    charge_fraction,
                    component="trap",
                ),
                dtype=float,
            ).reshape(3)
            for point in xyz_valid
        ])

    F[valid_indices] = F_valid

    valid = (
        geometry_valid
        & np.isfinite(U)
        & np.all(np.isfinite(F), axis=1)
    )

    return U, F, valid


# ============================================================
# STRAIGHT-PATH BARRIER INFORMATION
# ============================================================

def straight_path_geometry_status(xyz, valid):
    """
    Classify whether the OUTER end of the nominal path is usable.

    Invalid inner samples do not prevent an outer-tail R_far from being
    found; analyze_R_far already requires the entire omitted tail from a
    candidate point out to R_max to be valid.
    """
    if xyz.shape[0] == 0:
        return "no_valid_points"

    if not np.any(valid):
        return "no_valid_points"

    if not valid[-1]:
        return "invalid_far_start_below_electrode"

    return "valid"


def direction_barrier_info(
    s_grid,
    U,
    valid,
    mass,
    T,
    U_reference=0.0,
):
    U = np.asarray(U, dtype=float)
    valid = np.asarray(valid, dtype=bool)

    if not np.any(valid):
        return {
            "barrier_J": np.nan,
            "v_min_m_s": np.nan,
            "P_access": 0.0,
            "barrier_radius_m": np.nan,
        }

    delta_U = U[valid] - U_reference
    local_index = int(np.argmax(delta_U))
    valid_indices = np.flatnonzero(valid)
    barrier_index = valid_indices[local_index]

    barrier = max(0.0, float(delta_U[local_index]))
    v_min = np.sqrt(2.0 * barrier / mass)

    scale = mb_scale(T, mass)
    P_access = maxwell.sf(v_min, scale=scale)

    return {
        "barrier_J": barrier,
        "v_min_m_s": v_min,
        "P_access": P_access,
        "barrier_radius_m": s_grid[barrier_index],
    }


# ============================================================
# OUTER-TAIL CUMULATIVE INTEGRALS
# ============================================================

def reverse_cumulative_sum(segment_values):
    """
    Convert values on intervals [i, i+1] into cumulative values from
    point i outward to R_max.
    """
    segment_values = np.asarray(segment_values)

    if segment_values.ndim == 1:
        out = np.zeros(segment_values.shape[0] + 1)
        out[:-1] = np.cumsum(segment_values[::-1])[::-1]
        return out

    out = np.zeros(
        (segment_values.shape[0] + 1,)
        + segment_values.shape[1:]
    )

    out[:-1] = np.cumsum(
        segment_values[::-1],
        axis=0,
    )[::-1]

    return out


def analyze_R_far(
    s_grid,
    xyz,
    u,
    U,
    F,
    valid,
    mass,
    v_inf,
    R_full,
    R_max,
    energy_tol,
    angle_tol,
    displacement_tol,
    U_reference=0.0,
):
    """
    Find the smallest radius for which the omitted outer tail from
    R_max down to that radius is negligible.

    Test 2 uses this result to begin numerical propagation. If its chosen
    switching radius is outside R_far, the nominal geometry saved by Test 1
    can reconstruct a start state at that larger radius.
    """
    s_grid = np.asarray(s_grid, dtype=float)
    xyz = np.asarray(xyz, dtype=float)
    u = np.asarray(u, dtype=float)
    U = np.asarray(U, dtype=float)
    F = np.asarray(F, dtype=float)
    valid = np.asarray(valid, dtype=bool)

    K_inf = 0.5 * mass * v_inf**2

    if not np.isfinite(K_inf) or K_inf <= 0.0:
        return {
            "outer_status": "invalid_speed",
            "R_far_m": np.nan,
            "R_far_actual_m": np.nan,
            "R_far_is_lower_bound": False,
            "v_far_m_s": np.nan,
            "limiting_criterion": "not_applicable",
        }

    geometry_status = straight_path_geometry_status(xyz, valid)

    if geometry_status != "valid":
        return {
            "outer_status": geometry_status,
            "R_far_m": np.nan,
            "R_far_actual_m": np.nan,
            "R_far_is_lower_bound": False,
            "v_far_m_s": np.nan,
            "limiting_criterion": "not_applicable",
            "xyz_far": xyz[-1].copy(),
        }

    K_local = K_inf + U_reference - U

    point_ok = (
        valid
        & np.isfinite(K_local)
        & (K_local > 0.0)
    )

    v_local_sq = np.full(len(s_grid), np.nan)
    v_local_sq[point_ok] = 2.0 * K_local[point_ok] / mass

    F_parallel_scalar = F @ u
    F_parallel = F_parallel_scalar[:, None] * u[None, :]
    F_perp = F - F_parallel

    # q_vec = d(theta_vec)/ds.
    q_vec = np.zeros_like(F_perp)
    q_vec[point_ok] = (
        F_perp[point_ok]
        / (mass * v_local_sq[point_ok, None])
    )

    q_abs = np.linalg.norm(q_vec, axis=1)

    ds = np.diff(s_grid)
    s_mid = 0.5 * (s_grid[:-1] + s_grid[1:])

    segment_ok = point_ok[:-1] & point_ok[1:]

    q_mid_vec = 0.5 * (q_vec[:-1] + q_vec[1:])
    q_mid_abs = 0.5 * (q_abs[:-1] + q_abs[1:])

    q_mid_vec[~segment_ok] = 0.0
    q_mid_abs[~segment_ok] = 0.0

    # Signed angular change.
    dtheta_vec_segments = q_mid_vec * ds[:, None]
    theta_vec_tail = reverse_cumulative_sum(dtheta_vec_segments)
    theta_signed_tail = np.linalg.norm(theta_vec_tail, axis=1)

    # Absolute angular-change bound without cancellations.
    dtheta_abs_segments = q_mid_abs * ds
    theta_abs_tail = reverse_cumulative_sum(dtheta_abs_segments)

    # Signed transverse-displacement estimate.
    moment_vec_segments = (
        s_mid[:, None] * q_mid_vec * ds[:, None]
    )
    moment_vec_tail = reverse_cumulative_sum(moment_vec_segments)

    displacement_vec_tail = (
        moment_vec_tail
        - s_grid[:, None] * theta_vec_tail
    )
    displacement_signed_tail = np.linalg.norm(
        displacement_vec_tail,
        axis=1,
    )

    # Absolute transverse-displacement bound without cancellations.
    moment_abs_segments = s_mid * q_mid_abs * ds
    moment_abs_tail = reverse_cumulative_sum(moment_abs_segments)

    displacement_abs_tail = (
        moment_abs_tail
        - s_grid * theta_abs_tail
    )
    displacement_abs_tail = np.maximum(
        displacement_abs_tail,
        0.0,
    )

    # Entire omitted tail must stay valid.
    valid_tail = np.logical_and.accumulate(valid[::-1])[::-1]

    # Entire omitted tail must have positive local kinetic energy.
    K_for_tail = np.where(valid, K_local, -np.inf)
    K_tail_min = np.minimum.accumulate(K_for_tail[::-1])[::-1]

    energy_ratio_point = np.where(
        valid,
        np.abs(U - U_reference) / K_inf,
        np.inf,
    )
    energy_ratio_tail_max = np.maximum.accumulate(
        energy_ratio_point[::-1]
    )[::-1]

    safe = (
        valid_tail
        & (K_tail_min > 0.0)
        & (energy_ratio_tail_max <= energy_tol)
        & (theta_abs_tail <= angle_tol)
        & (displacement_abs_tail <= displacement_tol)
    )

    safe_indices = np.flatnonzero(safe)

    # Approximate straight-path turning radius, if one exists.
    nonpositive_indices = np.flatnonzero(
        valid & np.isfinite(K_local) & (K_local <= 0.0)
    )

    if nonpositive_indices.size > 0:
        turning_radius_m = np.max(s_grid[nonpositive_indices])
    else:
        turning_radius_m = np.nan

    if safe_indices.size == 0:
        # At R_max, angle and displacement tails are zero. The remaining
        # reasons are usually energy or nonpositive kinetic energy.
        if not point_ok[-1]:
            limiting = "R_max_nonpositive_K"
        else:
            limiting = "R_max_energy"

        return {
            "outer_status": "needs_larger_R_max",
            "R_far_m": R_max,
            "R_far_actual_m": np.linalg.norm(xyz[-1]),
            "R_far_is_lower_bound": True,
            "v_far_m_s": (
                np.sqrt(2.0 * K_local[-1] / mass)
                if point_ok[-1]
                else np.nan
            ),
            "U_far_J": U[-1],
            "K_inf_J": K_inf,
            "energy_tail_max": (
                energy_ratio_tail_max[-1]
                if np.isfinite(energy_ratio_tail_max[-1])
                else np.nan
            ),
            "theta_signed_tail": 0.0,
            "theta_abs_tail": 0.0,
            "displacement_signed_tail_m": 0.0,
            "displacement_abs_tail_m": 0.0,
            "limiting_criterion": limiting,
            "turning_radius_m": turning_radius_m,
            "xyz_far": xyz[-1].copy(),
            "velocity_far": (
                v_inf * u
                if point_ok[-1]
                else np.full(3, np.nan)
            ),
        }

    # s_grid increases outward, so the first safe point is the deepest
    # inward radius whose omitted outer tail is safe.
    idx = safe_indices[0]

    R_far_value = s_grid[idx]
    xyz_far = xyz[idx]
    K_far = K_local[idx]

    if K_far <= 0.0:
        return {
            "outer_status": "nonpositive_K_at_R_far",
            "R_far_m": R_far_value,
            "R_far_actual_m": np.linalg.norm(xyz_far),
            "R_far_is_lower_bound": False,
            "v_far_m_s": np.nan,
            "limiting_criterion": "not_applicable",
            "turning_radius_m": turning_radius_m,
        }

    v_far = np.sqrt(2.0 * K_far / mass)

    normalized_metrics = {
        "energy": energy_ratio_tail_max[idx] / energy_tol,
        "angle": theta_abs_tail[idx] / angle_tol,
        "displacement": (
            displacement_abs_tail[idx] / displacement_tol
        ),
    }

    limiting_criterion = max(
        normalized_metrics,
        key=normalized_metrics.get,
    )

    if idx == 0:
        outer_status = "straight_safe_to_full_sphere"
    else:
        outer_status = "straight_then_dm_only"

    return {
        "outer_status": outer_status,
        "R_far_m": R_far_value,
        "R_far_actual_m": np.linalg.norm(xyz_far),
        "R_far_is_lower_bound": False,
        "v_far_m_s": v_far,
        "U_far_J": U[idx],
        "K_inf_J": K_inf,
        "energy_tail_max": energy_ratio_tail_max[idx],
        "theta_signed_tail": theta_signed_tail[idx],
        "theta_abs_tail": theta_abs_tail[idx],
        "displacement_signed_tail_m": (
            displacement_signed_tail[idx]
        ),
        "displacement_abs_tail_m": (
            displacement_abs_tail[idx]
        ),
        "limiting_criterion": limiting_criterion,
        "turning_radius_m": turning_radius_m,
        "xyz_far": xyz_far.copy(),
        "velocity_far": v_far * u,
    }


# ============================================================
# FINAL TRAJECTORY CLASSIFICATION
# ============================================================

def classify_trajectory(
    outer_result,
    v_inf,
    v_min,
    nominal_intersects_full_sphere,
    nominal_path_to_full_is_valid,
):
    outer_status = outer_result["outer_status"]

    if outer_status not in {
        "straight_safe_to_full_sphere",
        "straight_then_dm_only",
    }:
        return outer_status

    if not nominal_intersects_full_sphere:
        return "nominal_line_misses_full_sphere_needs_dm_only"

    if not nominal_path_to_full_is_valid:
        return "nominal_path_crosses_invalid_domain_needs_dm_only"

    straight_accessible = np.isfinite(v_min) and v_inf >= v_min

    if not straight_accessible:
        return "straight_path_blocked_needs_dm_only"

    return outer_status


def coulomb_tail_energy_at_radius(radius, charge_fraction):
    """
    Ion-DM Coulomb potential energy magnitude at radius.
    Assumes c.Z is the dimensionless ion charge number.
    """
    return abs(c.K * c.Z * charge_fraction * c.e**2 / radius)

# ============================================================
# SCAN INITIALIZATION AND AGGREGATION
# ============================================================

def weighted_quantile(values, quantiles, weights):
    values = np.asarray(values, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = (
        np.isfinite(values)
        & np.isfinite(weights)
        & (weights > 0.0)
    )

    if not np.any(mask):
        return np.full(quantiles.shape, np.nan)

    values = values[mask]
    weights = weights[mask]

    order = np.argsort(values)
    values = values[order]
    weights = weights[order]

    cumulative = np.cumsum(weights)
    cumulative /= cumulative[-1]

    return np.interp(quantiles, cumulative, values)


def initialize_aggregate_maps(theta_values, alpha_values, speed_cases):
    shape = (len(theta_values), len(alpha_values))
    maps_by_speed = {}

    for case in speed_cases:
        maps = {
            "theta_values": theta_values,
            "alpha_values": alpha_values,
            "speed_quantile": case["quantile"],
            "speed_m_s_scalar": case["speed_m_s"],
            # Primary production maximum: positive-weight samples only.
            "R_far_um": np.full(shape, np.nan),
            "R_far_max_um": np.full(shape, np.nan),
            # Separate audit maximum including zero-weight diagnostics.
            "R_far_max_with_diagnostics_um": np.full(shape, np.nan),
            "R_far_p95_um": np.full(shape, np.nan),
            "R_far_p99_um": np.full(shape, np.nan),
            "b_at_max_um": np.full(shape, np.nan),
            "psi_at_max_deg": np.full(shape, np.nan),
            "b_at_diagnostic_max_um": np.full(shape, np.nan),
            "psi_at_diagnostic_max_deg": np.full(shape, np.nan),
            "v_min_m_s": np.full(shape, np.nan),
            "P_access": np.full(shape, np.nan),
            "unresolved_weight_fraction": np.full(shape, np.nan),
            "valid_weight_fraction": np.full(shape, np.nan),
            "outward_reject_weight_fraction": np.full(shape, np.nan),
            "max_retained_miss_b_um": np.full(shape, np.nan),
            "min_outward_rejected_b_um": np.full(shape, np.nan),
            "status": np.full(shape, "", dtype=object),
            "diagnostic_status": np.full(shape, "", dtype=object),
            "limiting_criterion": np.full(
                shape,
                "not_applicable",
                dtype=object,
            ),
            "diagnostic_limiting_criterion": np.full(
                shape,
                "not_applicable",
                dtype=object,
            ),
        }

        for status in WEIGHTED_STATUS_FRACTION_STATUSES:
            maps[f"status_fraction_{status}"] = np.full(shape, np.nan)
            maps[f"conditional_status_fraction_{status}"] = np.full(
                shape,
                np.nan,
            )

        maps_by_speed[case["key"]] = maps

    return maps_by_speed


def _select_maximum_representative(pool, R_max):
    """Select an unresolved lower bound or the largest resolved radius."""
    if len(pool) == 0:
        return None

    unresolved = pool["status"] == "needs_larger_R_max"
    if unresolved.any():
        representative = pool.loc[unresolved].iloc[0]
        return {
            "representative": representative,
            "R_far_max_um": R_max * 1.0e6,
            "R_far_is_lower_bound": True,
            "status": "needs_larger_R_max",
            "limiting_criterion": representative["limiting_criterion"],
        }

    resolved = (
        pool["status"].isin(RESOLVED_R_FAR_STATUSES)
        & np.isfinite(pool["R_far_actual_um"])
    )
    resolved_rows = pool.loc[resolved]
    if len(resolved_rows) == 0:
        return None

    idx_max = resolved_rows["R_far_actual_um"].idxmax()
    representative = pool.loc[idx_max]
    return {
        "representative": representative,
        "R_far_max_um": representative["R_far_actual_um"],
        "R_far_is_lower_bound": bool(
            representative["R_far_is_lower_bound"]
        ),
        "status": representative["status"],
        "limiting_criterion": representative["limiting_criterion"],
    }


def aggregate_one_bpsi_group(group, R_max):
    """Aggregate one speed-theta-alpha group after removing invalid rows."""
    group = group.copy()

    positive_weight = group["impact_weight"] > 0.0
    total_weight = group.loc[positive_weight, "impact_weight"].sum()
    invalid = group["status"].isin(INVALID_TRAJECTORY_STATUSES)
    screened_reject = group["status"].isin(SCREENED_REJECT_STATUSES)
    excluded = group["status"].isin(EXCLUDED_TRAJECTORY_STATUSES)
    included = ~excluded

    valid_positive = positive_weight & included
    valid_weight = group.loc[valid_positive, "impact_weight"].sum()

    # If every positive-weight impact trajectory is invalid, remove the
    # theta-alpha direction from the production aggregate entirely.
    if valid_weight <= 0.0:
        first = group.iloc[0]
        return None, {
            "speed_quantile": first["speed_quantile"],
            "speed_label": first["speed_label"],
            "v_inf_m_s": first["v_inf_m_s"],
            "theta_index": int(first["theta_index"]),
            "alpha_index": int(first["alpha_index"]),
            "theta_deg": first["theta_deg"],
            "alpha_deg": first["alpha_deg"],
            "reason": (
                "all_positive_weight_impact_trajectories_"
                "invalid_or_outward_rejected"
            ),
            "n_impact_samples": len(group),
            "n_invalid_samples": int(invalid.sum()),
            "n_screened_reject_samples": int(screened_reject.sum()),
        }

    unresolved = group["status"] == "needs_larger_R_max"
    unresolved_weight = group.loc[
        valid_positive & unresolved,
        "impact_weight",
    ].sum()
    outward_reject_weight = group.loc[
        positive_weight & screened_reject,
        "impact_weight",
    ].sum()

    unresolved_fraction = (
        unresolved_weight / total_weight
        if total_weight > 0.0
        else np.nan
    )
    valid_fraction = (
        valid_weight / total_weight
        if total_weight > 0.0
        else np.nan
    )
    outward_reject_fraction = (
        outward_reject_weight / total_weight
        if total_weight > 0.0
        else np.nan
    )

    primary_pool = group.loc[valid_positive]
    diagnostic_pool = group.loc[included]

    primary = _select_maximum_representative(primary_pool, R_max)
    diagnostic = _select_maximum_representative(diagnostic_pool, R_max)

    # primary cannot be None because valid_weight > 0 and all retained
    # positive-weight statuses are either resolved or unresolved.
    representative = primary["representative"]

    resolved_positive = primary_pool.loc[
        primary_pool["status"].isin(RESOLVED_R_FAR_STATUSES)
        & np.isfinite(primary_pool["R_far_actual_um"])
    ]

    if len(resolved_positive) > 0:
        p95, p99 = weighted_quantile(
            resolved_positive["R_far_actual_um"].to_numpy(),
            [0.95, 0.99],
            resolved_positive["impact_weight"].to_numpy(),
        )
    else:
        p95 = np.nan
        p99 = np.nan

    finite_vmin = primary_pool.loc[
        np.isfinite(primary_pool["v_min_m_s"]),
        "v_min_m_s",
    ]
    finite_paccess = primary_pool.loc[
        np.isfinite(primary_pool["P_access"]),
        "P_access",
    ]

    diagnostic_rep = (
        diagnostic["representative"]
        if diagnostic is not None
        else representative
    )

    retained_miss_rows = group.loc[
        valid_positive
        & (
            group["status"]
            == "nominal_line_misses_full_sphere_needs_dm_only"
        )
    ]
    outward_rejected_rows = group.loc[
        positive_weight & screened_reject
    ]

    max_retained_miss_b_um = (
        retained_miss_rows["b_um"].max()
        if len(retained_miss_rows) > 0 else np.nan
    )
    min_outward_rejected_b_um = (
        outward_rejected_rows["b_um"].min()
        if len(outward_rejected_rows) > 0 else np.nan
    )

    row = {
        "speed_quantile": representative["speed_quantile"],
        "speed_label": representative["speed_label"],
        "v_inf_m_s": representative["v_inf_m_s"],
        "theta_index": int(representative["theta_index"]),
        "alpha_index": int(representative["alpha_index"]),
        "theta_rad": representative["theta_rad"],
        "alpha_rad": representative["alpha_rad"],
        "theta_deg": representative["theta_deg"],
        "alpha_deg": representative["alpha_deg"],
        "R_far_max_um": primary["R_far_max_um"],
        "R_far_max_with_diagnostics_um": (
            diagnostic["R_far_max_um"] if diagnostic is not None else np.nan
        ),
        "R_far_p95_um": p95,
        "R_far_p99_um": p99,
        "R_far_is_lower_bound": primary["R_far_is_lower_bound"],
        "diagnostic_R_far_is_lower_bound": (
            diagnostic["R_far_is_lower_bound"]
            if diagnostic is not None else False
        ),
        "b_at_max_um": representative["b_um"],
        "psi_at_max_deg": representative["psi_deg"],
        "b_at_diagnostic_max_um": diagnostic_rep["b_um"],
        "psi_at_diagnostic_max_deg": diagnostic_rep["psi_deg"],
        "status": primary["status"],
        "diagnostic_status": (
            diagnostic["status"] if diagnostic is not None else ""
        ),
        "limiting_criterion": primary["limiting_criterion"],
        "diagnostic_limiting_criterion": (
            diagnostic["limiting_criterion"]
            if diagnostic is not None else "not_applicable"
        ),
        "v_min_max_m_s": (
            finite_vmin.max() if len(finite_vmin) > 0 else np.nan
        ),
        "P_access_min": (
            finite_paccess.min() if len(finite_paccess) > 0 else np.nan
        ),
        "n_impact_samples": len(group),
        "n_valid_positive_weight_samples": int(valid_positive.sum()),
        "n_invalid_samples": int(invalid.sum()),
        "n_screened_reject_samples": int(screened_reject.sum()),
        "n_resolved_samples": int(
            (
                valid_positive
                & group["status"].isin(RESOLVED_R_FAR_STATUSES)
            ).sum()
        ),
        "n_unresolved_samples": int(
            (valid_positive & unresolved).sum()
        ),
        "unresolved_weight_fraction": unresolved_fraction,
        "valid_weight_fraction": valid_fraction,
        "outward_reject_weight_fraction": outward_reject_fraction,
        "max_retained_miss_b_um": max_retained_miss_b_um,
        "min_outward_rejected_b_um": min_outward_rejected_b_um,
    }

    for status in WEIGHTED_STATUS_FRACTION_STATUSES:
        status_weight = group.loc[
            valid_positive & (group["status"] == status),
            "impact_weight",
        ].sum()
        row[f"status_fraction_{status}"] = (
            status_weight / total_weight
            if total_weight > 0.0
            else np.nan
        )
        row[f"conditional_status_fraction_{status}"] = (
            status_weight / valid_weight
            if valid_weight > 0.0
            else np.nan
        )

    return row, None


def aggregate_bpsi_results(results_df, R_max):
    aggregate_rows = []
    excluded_rows = []

    group_columns = [
        "speed_label",
        "theta_index",
        "alpha_index",
    ]

    for _, group in results_df.groupby(group_columns, sort=False):
        row, excluded = aggregate_one_bpsi_group(group, R_max)
        if row is not None:
            aggregate_rows.append(row)
        if excluded is not None:
            excluded_rows.append(excluded)

    return pd.DataFrame(aggregate_rows), pd.DataFrame(excluded_rows)


def populate_aggregate_maps(
    aggregate_df,
    theta_values,
    alpha_values,
    speed_cases,
):
    maps_by_speed = initialize_aggregate_maps(
        theta_values,
        alpha_values,
        speed_cases,
    )

    for row in aggregate_df.itertuples(index=False):
        maps = maps_by_speed[row.speed_label]
        i = row.theta_index
        j = row.alpha_index

        maps["R_far_um"][i, j] = row.R_far_max_um
        maps["R_far_max_um"][i, j] = row.R_far_max_um
        maps["R_far_max_with_diagnostics_um"][i, j] = (
            row.R_far_max_with_diagnostics_um
        )
        maps["R_far_p95_um"][i, j] = row.R_far_p95_um
        maps["R_far_p99_um"][i, j] = row.R_far_p99_um
        maps["b_at_max_um"][i, j] = row.b_at_max_um
        maps["psi_at_max_deg"][i, j] = row.psi_at_max_deg
        maps["b_at_diagnostic_max_um"][i, j] = (
            row.b_at_diagnostic_max_um
        )
        maps["psi_at_diagnostic_max_deg"][i, j] = (
            row.psi_at_diagnostic_max_deg
        )
        maps["v_min_m_s"][i, j] = row.v_min_max_m_s
        maps["P_access"][i, j] = row.P_access_min
        maps["unresolved_weight_fraction"][i, j] = (
            row.unresolved_weight_fraction
        )
        maps["valid_weight_fraction"][i, j] = row.valid_weight_fraction
        maps["outward_reject_weight_fraction"][i, j] = (
            row.outward_reject_weight_fraction
        )
        maps["max_retained_miss_b_um"][i, j] = (
            row.max_retained_miss_b_um
        )
        maps["min_outward_rejected_b_um"][i, j] = (
            row.min_outward_rejected_b_um
        )
        maps["status"][i, j] = row.status
        maps["diagnostic_status"][i, j] = row.diagnostic_status
        maps["limiting_criterion"][i, j] = row.limiting_criterion
        maps["diagnostic_limiting_criterion"][i, j] = (
            row.diagnostic_limiting_criterion
        )

        for status in WEIGHTED_STATUS_FRACTION_STATUSES:
            maps[f"status_fraction_{status}"][i, j] = getattr(
                row,
                f"status_fraction_{status}",
            )
            maps[f"conditional_status_fraction_{status}"][i, j] = getattr(
                row,
                f"conditional_status_fraction_{status}",
            )

    return maps_by_speed


def select_dm_only_candidates(results_df):
    """Choose representative valid states for the later DM-only integrator."""
    candidates = []
    usable = results_df.loc[
        (results_df["impact_weight"] > 0.0)
        & results_df["status"].isin(DM_ONLY_REQUIRED_STATUSES)
        & np.isfinite(results_df["R_far_actual_um"])
    ].copy()

    for (speed_label, status), group in usable.groupby(
        ["speed_label", "status"],
        sort=False,
    ):
        group = group.sort_values("R_far_actual_um")
        positions = {
            "minimum": 0,
            "median": len(group) // 2,
            "maximum": len(group) - 1,
        }
        for selection_label, position in positions.items():
            row = group.iloc[position].to_dict()
            row["candidate_selection"] = selection_label
            candidates.append(row)

    return pd.DataFrame(candidates)

def assign_stable_trajectory_ids(results_df):
    """Assign IDs after full x/y symmetry reconstruction.

    The target theta/alpha indices and transformed impact_case_id are used,
    so reconstructed rows receive unique physical IDs rather than inheriting
    the canonical representative's ID.
    """
    df = results_df.copy()
    df["trajectory_id"] = [
        (
            f"{speed_label}_th{int(theta_index):03d}"
            f"_al{int(alpha_index):03d}_impact{int(impact_case_id):04d}"
        )
        for speed_label, theta_index, alpha_index, impact_case_id in zip(
            df["speed_label"],
            df["theta_index"],
            df["alpha_index"],
            df["impact_case_id"],
        )
    ]
    if not df["trajectory_id"].is_unique:
        duplicates = df.loc[
            df["trajectory_id"].duplicated(keep=False),
            "trajectory_id",
        ].head(10).tolist()
        raise RuntimeError(
            "Trajectory IDs are not unique after symmetry reconstruction: "
            f"{duplicates}"
        )
    return df


def build_test2_input_table(
    all_results_df,
    include_zero_weight_diagnostics=False,
):
    """Create the complete Test 2 initial-condition table.

    Readiness is determined only by the resolved outer-tail solution. The
    nominal inner straight-line classification and outward-force diagnostic
    do not remove a trajectory; Test 2 will propagate the actual DM path.
    """
    df = all_results_df.copy()
    state_columns = [
        "x_far_m",
        "y_far_m",
        "z_far_m",
        "vx_far_m_s",
        "vy_far_m_s",
        "vz_far_m_s",
    ]

    finite_state = np.all(
        np.isfinite(df[state_columns].to_numpy(dtype=float)),
        axis=1,
    )
    resolved_outer = df["outer_status"].isin(
        TEST2_RESOLVED_OUTER_STATUSES
    )
    not_lower_bound = ~df["R_far_is_lower_bound"].fillna(True).astype(bool)
    sampled_for_production = (
        np.ones(len(df), dtype=bool)
        if include_zero_weight_diagnostics
        else (df["impact_weight"] > 0.0).to_numpy()
    )

    ready = (
        resolved_outer.to_numpy()
        & not_lower_bound.to_numpy()
        & finite_state
        & sampled_for_production
    )

    reasons = np.full(len(df), "ready", dtype=object)
    reasons[~sampled_for_production] = "zero_weight_diagnostic"
    reasons[sampled_for_production & ~resolved_outer.to_numpy()] = (
        "outer_tail_not_resolved"
    )
    reasons[
        sampled_for_production
        & resolved_outer.to_numpy()
        & ~not_lower_bound.to_numpy()
    ] = "R_far_is_lower_bound"
    reasons[
        sampled_for_production
        & resolved_outer.to_numpy()
        & not_lower_bound.to_numpy()
        & ~finite_state
    ] = "nonfinite_R_far_state"

    df["test2_ready"] = ready
    df["test2_readiness_reason"] = reasons
    df["test2_state_model"] = (
        "trap-only R_far state; energy-corrected speed; nominal direction"
    )
    df["s_far_m"] = df["R_far_um"] * 1.0e-6
    df["r_far_radial_m"] = df["R_far_actual_um"] * 1.0e-6
    df["test2_outer_geometry_available"] = np.all(
        np.isfinite(
            df[[
                "u_x", "u_y", "u_z",
                "bvec_x_m", "bvec_y_m", "bvec_z_m",
            ]].to_numpy(dtype=float)
        ),
        axis=1,
    )

    ready_df = df.loc[ready].copy().reset_index(drop=True)
    not_ready_df = df.loc[~ready].copy().reset_index(drop=True)

    required_unique = ["trajectory_id"]
    if ready_df.duplicated(required_unique).any():
        raise RuntimeError("Duplicate trajectory_id values in Test 2 input")

    return ready_df, not_ready_df


# ============================================================
# INDIVIDUAL TRAJECTORY PRINTING
# ============================================================

def format_float(value, fmt=".3e", nan_text="nan"):
    if value is None or not np.isfinite(value):
        return nan_text
    return format(value, fmt)


def print_trajectory_result(row, current_index, total_count):
    R_far_mm = (
        row["R_far_actual_um"] / 1000.0
        if np.isfinite(row["R_far_actual_um"])
        else np.nan
    )

    print(
        f"[{current_index:7d}/{total_count:7d}] "
        f"tail_q={row['speed_quantile']:.3f}, "
        f"v={row['v_inf_m_s']:.3f} m/s, "
        f"theta={row['theta_deg']:.1f} deg, "
        f"alpha={row['alpha_deg']:.1f} deg, "
        f"b={row['b_um']:.3f} um, "
        f"psi={row['psi_deg']:.1f} deg, "
        f"w={row['impact_weight']:.3e}, "
        f"status={row['status']}, "
        f"R_far_radial={format_float(R_far_mm, '.3f')} mm, "
        f"v_min={format_float(row['v_min_m_s'], '.3f')} m/s, "
        f"turn={format_float(row['turning_radius_um'], '.3f')} um, "
        f"limit={row['limiting_criterion']}",
        flush=True,
    )


# ============================================================
# MAIN THETA-ALPHA, B-PSI, AND SPEED SCAN
# ============================================================

def evaluate_theta_alpha_impact_task(
    i,
    j,
    theta,
    alpha,
    impact_case,
    mass,
    charge_fraction,
    T,
    R_full,
    R_max,
    n_path,
    energy_tol,
    angle_tol,
    displacement_tol,
    speed_cases,
):
    """
    Evaluate one (theta, alpha, b, psi) geometry for every speed.

    Using one task per impact geometry preserves the expensive U/F path
    evaluation across all representative speeds while allowing independent
    geometries to run concurrently.
    """
    b = impact_case["b_m"]
    psi = impact_case["psi_rad"]

    # Test 1 is called once per representative speed. Recalculate the actual
    # Rutherford closest approach for this exact (v,b) trajectory and use it as
    # the operational R_full. The speed-only threshold sphere remains available
    # as a separate diagnostic and as the boundary used to construct the b grid.
    if len(speed_cases) != 1:
        raise ValueError(
            "The per-trajectory R_full(v,b) implementation requires one "
            "representative speed per Test 1 geometry task"
        )
    speed_case_only = speed_cases[0]
    R_full_speed_only = float(R_full)
    radii_v_b = rutherford_v_b_radius_policy(
        float(speed_case_only["speed_m_s"]),
        float(b),
        m_dm_kg=float(mass),
        m_ion_kg=float(m_ion),
        eps_value=float(charge_fraction),
        ion_charge_number=float(Z_ion),
        coulomb_constant=float(K),
        elementary_charge_c=float(e),
        threshold_j=float(TARGET_ION_ENERGY_J),
    )
    R_full = float(radii_v_b["R_full_m"])
    R_switch_v_b = float(radii_v_b["R_switch_m"])

    try:
        s_grid, xyz, u, b_vec = build_straight_path(
            theta=theta,
            alpha=alpha,
            b=b,
            psi=psi,
            R_inner=R_full,
            R_outer=R_max,
            n_path=n_path,
        )

        U, F, valid = evaluate_trap_path(
            xyz=xyz,
            mass=mass,
            charge_fraction=charge_fraction,
        )
    except Exception as exc:
        raise RuntimeError(
            "Failed while evaluating path for "
            f"theta={np.degrees(theta):.6f} deg, "
            f"alpha={np.degrees(alpha):.6f} deg, "
            f"b={b * 1.0e6:.6f} um, "
            f"psi={np.degrees(psi):.6f} deg"
        ) from exc

    geometry_status = straight_path_geometry_status(xyz, valid)
    nominal_intersects = line_intersects_sphere(b, R_full)
    nominal_path_to_full_is_valid = (
        nominal_intersects
        and geometry_status == "valid"
        and np.all(valid)
    )

    if (
        EVALUATE_OUTWARD_FORCE_DIAGNOSTIC
        and not nominal_intersects
        and geometry_status == "valid"
    ):
        outward_screen = outward_force_screen(
            F=F,
            valid=valid,
            b_vec=b_vec,
            dot_tolerance_J=OUTWARD_FORCE_DOT_TOL_J,
            require_complete_valid_path=(
                OUTWARD_FORCE_REQUIRE_COMPLETE_VALID_PATH
            ),
        )
    else:
        outward_screen = {
            "reject": False,
            "n_tested": 0,
            "complete_valid_path": bool(np.all(valid)),
            "min_F_dot_b_vec_J": np.nan,
            "max_F_dot_b_vec_J": np.nan,
            "min_F_dot_bhat_N": np.nan,
            "max_F_dot_bhat_N": np.nan,
        }

    if nominal_path_to_full_is_valid:
        barrier_info = direction_barrier_info(
            s_grid=s_grid,
            U=U,
            valid=valid,
            mass=mass,
            T=T,
            U_reference=U_infinity,
        )
    else:
        barrier_info = {
            "barrier_J": np.nan,
            "v_min_m_s": np.nan,
            "P_access": np.nan,
            "barrier_radius_m": np.nan,
        }

    task_rows = []

    for speed_case in speed_cases:
        quantile = speed_case["quantile"]
        v_inf = speed_case["speed_m_s"]

        hard_outward_reject = bool(
            HARD_REJECT_OUTWARD_FORCE_MISSES
            and outward_screen["reject"]
        )

        if hard_outward_reject:
            outer_result = {
                "outer_status": (
                    "nominal_line_misses_full_sphere_"
                    "outward_force_reject"
                ),
                "R_far_m": np.nan,
                "R_far_actual_m": np.nan,
                "R_far_is_lower_bound": False,
                "v_far_m_s": np.nan,
                "U_far_J": np.nan,
                "K_inf_J": 0.5 * mass * v_inf**2,
                "energy_tail_max": np.nan,
                "theta_signed_tail": np.nan,
                "theta_abs_tail": np.nan,
                "displacement_signed_tail_m": np.nan,
                "displacement_abs_tail_m": np.nan,
                "limiting_criterion": "not_applicable",
                "turning_radius_m": np.nan,
            }
        elif geometry_status != "valid":
            outer_result = {
                "outer_status": geometry_status,
                "R_far_m": np.nan,
                "R_far_actual_m": np.nan,
                "R_far_is_lower_bound": False,
                "v_far_m_s": np.nan,
                "U_far_J": np.nan,
                "K_inf_J": 0.5 * mass * v_inf**2,
                "energy_tail_max": np.nan,
                "theta_signed_tail": np.nan,
                "theta_abs_tail": np.nan,
                "displacement_signed_tail_m": np.nan,
                "displacement_abs_tail_m": np.nan,
                "limiting_criterion": "not_applicable",
                "turning_radius_m": np.nan,
            }
        else:
            outer_result = analyze_R_far(
                s_grid=s_grid,
                xyz=xyz,
                u=u,
                U=U,
                F=F,
                valid=valid,
                mass=mass,
                v_inf=v_inf,
                R_full=R_full,
                R_max=R_max,
                energy_tol=energy_tol,
                angle_tol=angle_tol,
                displacement_tol=displacement_tol,
                U_reference=U_infinity,
            )

        if hard_outward_reject:
            final_status = (
                "nominal_line_misses_full_sphere_"
                "outward_force_reject"
            )
        else:
            final_status = classify_trajectory(
                outer_result=outer_result,
                v_inf=v_inf,
                v_min=barrier_info["v_min_m_s"],
                nominal_intersects_full_sphere=nominal_intersects,
                nominal_path_to_full_is_valid=(
                    nominal_path_to_full_is_valid
                ),
            )

        R_far_m = outer_result.get("R_far_m", np.nan)
        R_far_actual_m = outer_result.get("R_far_actual_m", np.nan)
        turning_radius_m = outer_result.get("turning_radius_m", np.nan)
        xyz_far = np.asarray(
            outer_result.get("xyz_far", np.full(3, np.nan)),
            dtype=float,
        ).reshape(3)
        velocity_far = np.asarray(
            outer_result.get("velocity_far", np.full(3, np.nan)),
            dtype=float,
        ).reshape(3)

        row = {
            "m_dm_kg": mass,
            "eps": charge_fraction,
            "T_dm_K": T,
            "R_full_m": R_full,
            "R_max_m": R_max,
            "b_scan_max_m": b_max,
            "n_path": n_path,
            "energy_tol": energy_tol,
            "angle_tol": angle_tol,
            "displacement_tol_m": displacement_tol,
            "speed_quantile": quantile,
            "speed_conditional_quantile": speed_case.get(
                "conditional_quantile", quantile
            ),
            "speed_unconditional_quantile": speed_case.get(
                "unconditional_quantile", quantile
            ),
            "speed_tail_probability": speed_case.get(
                "tail_probability", 1.0
            ),
            "conditional_speed_weight": speed_case.get(
                "conditional_speed_weight", np.nan
            ),
            "unconditional_speed_weight": speed_case.get(
                "unconditional_speed_weight", np.nan
            ),
            "minimum_speed_cutoff_m_s": speed_case.get(
                "minimum_speed_m_s", np.nan
            ),
            "speed_label": speed_case["key"],
            "v_inf_m_s": v_inf,
            "u_x": u[0],
            "u_y": u[1],
            "u_z": u[2],
            "bvec_x_m": b_vec[0],
            "bvec_y_m": b_vec[1],
            "bvec_z_m": b_vec[2],
            "theta_index": i,
            "alpha_index": j,
            "theta_rad": theta,
            "alpha_rad": alpha,
            "theta_deg": np.degrees(theta),
            "alpha_deg": np.degrees(alpha),
            "impact_case_id": impact_case["impact_case_id"],
            "impact_sample_type": impact_case["impact_sample_type"],
            "impact_weight": impact_case["impact_weight"],
            "b_region": impact_case.get("b_region", "unknown"),
            "b_lower_m": impact_case.get("b_lower_m", np.nan),
            "b_upper_m": impact_case.get("b_upper_m", np.nan),
            "b_m": b,
            "b_um": b * 1.0e6,
            "psi_rad": psi,
            "psi_deg": np.degrees(psi),
            "geometry_status": geometry_status,
            "nominal_intersects_R_full": nominal_intersects,
            "nominal_path_to_R_full_is_valid": (
                nominal_path_to_full_is_valid
            ),
            "v_min_m_s": barrier_info["v_min_m_s"],
            "P_access": barrier_info["P_access"],
            "barrier_J": barrier_info["barrier_J"],
            "barrier_radius_um": (
                barrier_info["barrier_radius_m"] * 1.0e6
                if np.isfinite(barrier_info["barrier_radius_m"])
                else np.nan
            ),
            "straight_path_accessible": (
                nominal_path_to_full_is_valid
                and np.isfinite(barrier_info["v_min_m_s"])
                and v_inf >= barrier_info["v_min_m_s"]
            ),
            "outward_force_diagnostic_flag": outward_screen["reject"],
            "outward_force_hard_reject": hard_outward_reject,
            "outward_force_reject": hard_outward_reject,
            "outward_force_n_tested": outward_screen["n_tested"],
            "outward_force_complete_valid_path": (
                outward_screen["complete_valid_path"]
            ),
            "min_F_dot_b_vec_J": outward_screen[
                "min_F_dot_b_vec_J"
            ],
            "max_F_dot_b_vec_J": outward_screen[
                "max_F_dot_b_vec_J"
            ],
            "min_F_dot_bhat_N": outward_screen[
                "min_F_dot_bhat_N"
            ],
            "max_F_dot_bhat_N": outward_screen[
                "max_F_dot_bhat_N"
            ],
            # Per-trajectory radius diagnostics. ``r_min_threshold_m`` is
            # retained as a compatibility alias for the active v,b operational
            # radius; the strictly energy-threshold radius is stored separately.
            "r_min_threshold_m": R_full,
            "r_min_threshold_um": R_full * 1.0e6,
            "r_min_energy_threshold_m": radii_v_b["r_min_energy_threshold_m"],
            "r_min_energy_threshold_um": (
                radii_v_b["r_min_energy_threshold_m"] * 1.0e6
                if np.isfinite(radii_v_b["r_min_energy_threshold_m"])
                else np.nan
            ),
            "r_min_threshold_speed_only_m": radii_v_b["r_min_threshold_speed_only_m"],
            "r_min_rutherford_v_b_m": radii_v_b["r_min_rutherford_v_b_m"],
            "r_min_rutherford_v_b_um": radii_v_b["r_min_rutherford_v_b_m"] * 1.0e6,
            "rutherford_recoil_v_b_J": radii_v_b["rutherford_recoil_v_b_J"],
            "rutherford_above_threshold_v_b": radii_v_b["rutherford_above_threshold_v_b"],
            "R_full_speed_only_m": R_full_speed_only,
            "R_full_speed_only_um": R_full_speed_only * 1.0e6,
            "R_full_m": R_full,
            "R_full_um": R_full * 1.0e6,
            "R_switch_factor": radii_v_b["R_switch_factor"],
            "R_switch_m": R_switch_v_b,
            "R_switch_um": R_switch_v_b * 1.0e6,
            "R_full_v_b_mode": radii_v_b["R_full_v_b_mode"],
            "R_far_um": (
                R_far_m * 1.0e6
                if np.isfinite(R_far_m)
                else np.nan
            ),
            "R_far_actual_um": (
                R_far_actual_m * 1.0e6
                if np.isfinite(R_far_actual_m)
                else np.nan
            ),
            "R_far_is_lower_bound": outer_result.get(
                "R_far_is_lower_bound",
                False,
            ),
            "v_far_m_s": outer_result.get("v_far_m_s", np.nan),
            "x_far_m": xyz_far[0],
            "y_far_m": xyz_far[1],
            "z_far_m": xyz_far[2],
            "vx_far_m_s": velocity_far[0],
            "vy_far_m_s": velocity_far[1],
            "vz_far_m_s": velocity_far[2],
            "U_far_J": outer_result.get("U_far_J", np.nan),
            "K_inf_J": outer_result.get(
                "K_inf_J",
                0.5 * mass * v_inf**2,
            ),
            "energy_tail_max": outer_result.get(
                "energy_tail_max",
                np.nan,
            ),
            "theta_signed_tail": outer_result.get(
                "theta_signed_tail",
                np.nan,
            ),
            "theta_abs_tail": outer_result.get(
                "theta_abs_tail",
                np.nan,
            ),
            "displacement_signed_tail_um": (
                outer_result.get(
                    "displacement_signed_tail_m",
                    np.nan,
                ) * 1.0e6
            ),
            "displacement_abs_tail_um": (
                outer_result.get(
                    "displacement_abs_tail_m",
                    np.nan,
                ) * 1.0e6
            ),
            "turning_radius_um": (
                turning_radius_m * 1.0e6
                if np.isfinite(turning_radius_m)
                else np.nan
            ),
            "limiting_criterion": outer_result.get(
                "limiting_criterion",
                "not_applicable",
            ),
            "outer_status": outer_result["outer_status"],
            "status": final_status,
        }

        task_rows.append(row)

    return task_rows


def make_prefiltered_geometry_rows(
    prefiltered_geometries_df,
    speed_cases,
    mass,
    R_full,
):
    """Create zero-cost invalid rows so physical impact weights are preserved.

    These rows are never sent to potential_energy or force. They are included
    only so aggregate valid/invalid fractions remain normalized to the full
    sampled impact disk after the analytical geometry prefilter.
    """
    rows = []
    if len(prefiltered_geometries_df) == 0:
        return rows

    for geometry in prefiltered_geometries_df.to_dict("records"):
        for speed_case in speed_cases:
            status = geometry.get(
                "prefilter_reason",
                "path_crosses_invalid_domain",
            )
            rows.append({
                "m_dm_kg": mass,
                "eps": eps,
                "T_dm_K": T_dm,
                "R_full_m": R_full,
                "R_max_m": R_max,
                "b_scan_max_m": b_max,
                "n_path": n_path,
                "energy_tol": energy_tol,
                "angle_tol": angle_tol,
                "displacement_tol_m": displacement_tol,
                "speed_quantile": speed_case["quantile"],
                "speed_conditional_quantile": speed_case.get(
                    "conditional_quantile", speed_case["quantile"]
                ),
                "speed_unconditional_quantile": speed_case.get(
                    "unconditional_quantile", speed_case["quantile"]
                ),
                "speed_tail_probability": speed_case.get(
                    "tail_probability", 1.0
                ),
                "conditional_speed_weight": speed_case.get(
                    "conditional_speed_weight", np.nan
                ),
                "unconditional_speed_weight": speed_case.get(
                    "unconditional_speed_weight", np.nan
                ),
                "minimum_speed_cutoff_m_s": speed_case.get(
                    "minimum_speed_m_s", np.nan
                ),
                "speed_label": speed_case["key"],
                "v_inf_m_s": speed_case["speed_m_s"],
                "u_x": np.nan,
                "u_y": np.nan,
                "u_z": np.nan,
                "bvec_x_m": np.nan,
                "bvec_y_m": np.nan,
                "bvec_z_m": np.nan,
                "theta_index": geometry["theta_index"],
                "alpha_index": geometry["alpha_index"],
                "theta_rad": geometry["theta_rad"],
                "alpha_rad": geometry["alpha_rad"],
                "theta_deg": geometry["theta_deg"],
                "alpha_deg": geometry["alpha_deg"],
                "impact_case_id": geometry["impact_case_id"],
                "impact_sample_type": geometry["impact_sample_type"],
                "impact_weight": geometry["impact_weight"],
                "b_region": geometry.get("b_region", "unknown"),
                "b_lower_m": geometry.get("b_lower_m", np.nan),
                "b_upper_m": geometry.get("b_upper_m", np.nan),
                "b_m": geometry["b_m"],
                "b_um": geometry["b_um"],
                "psi_rad": geometry["psi_rad"],
                "psi_deg": geometry["psi_deg"],
                "geometry_status": status,
                "nominal_intersects_R_full": bool(
                    geometry["b_m"] < R_full
                ),
                "nominal_path_to_R_full_is_valid": False,
                "v_min_m_s": np.nan,
                "P_access": np.nan,
                "barrier_J": np.nan,
                "barrier_radius_um": np.nan,
                "straight_path_accessible": False,
                "outward_force_diagnostic_flag": False,
                "outward_force_hard_reject": False,
                "outward_force_reject": False,
                "outward_force_n_tested": 0,
                "outward_force_complete_valid_path": False,
                "min_F_dot_b_vec_J": np.nan,
                "max_F_dot_b_vec_J": np.nan,
                "min_F_dot_bhat_N": np.nan,
                "max_F_dot_bhat_N": np.nan,
                "R_full_um": R_full * 1.0e6,
                "R_far_um": np.nan,
                "R_far_actual_um": np.nan,
                "R_far_is_lower_bound": False,
                "v_far_m_s": np.nan,
                "x_far_m": np.nan,
                "y_far_m": np.nan,
                "z_far_m": np.nan,
                "vx_far_m_s": np.nan,
                "vy_far_m_s": np.nan,
                "vz_far_m_s": np.nan,
                "U_far_J": np.nan,
                "K_inf_J": 0.5 * mass * speed_case["speed_m_s"]**2,
                "energy_tail_max": np.nan,
                "theta_signed_tail": np.nan,
                "theta_abs_tail": np.nan,
                "displacement_signed_tail_um": np.nan,
                "displacement_abs_tail_um": np.nan,
                "turning_radius_um": np.nan,
                "limiting_criterion": "not_applicable",
                "outer_status": status,
                "status": status,
                "analytically_prefiltered": True,
            })
    return rows


def run_theta_alpha_bpsi_speed_scan(
    mass,
    charge_fraction,
    T,
    impact_cases,
    R_full,
    R_max,
    n_theta,
    n_alpha,
    n_path,
    energy_tol,
    angle_tol,
    displacement_tol,
    representative_quantiles,
    minimum_speed_m_s,
    n_threads=8,
    max_in_flight_tasks=None,
    speed_cases_override=None,
):
    """Run the scan on the x/y symmetry domain and reconstruct full outputs."""
    if n_threads < 1:
        raise ValueError("n_threads must be at least 1")

    if max_in_flight_tasks is None:
        max_in_flight_tasks = 4 * n_threads

    if max_in_flight_tasks < n_threads:
        raise ValueError(
            "max_in_flight_tasks must be at least n_threads"
        )

    if speed_cases_override is None:
        scale = mb_scale(T, mass)
        speed_cases = make_truncated_mb_speed_cases(
            conditional_quantiles=representative_quantiles,
            scale=scale,
            v_cut=minimum_speed_m_s,
        )
    else:
        speed_cases = [dict(case) for case in speed_cases_override]
        if not speed_cases:
            raise ValueError("speed_cases_override cannot be empty")

    (
        theta_values,
        alpha_values,
        full_direction_cases,
        scan_direction_cases,
        symmetry_direction_map_df,
    ) = make_xy_symmetry_direction_cases(
        n_theta=n_theta,
        n_alpha=n_alpha,
        remove_duplicated_poles=REMOVE_DUPLICATED_POLES,
        exploit_symmetry=EXPLOIT_XY_REFLECTION_SYMMETRY,
    )

    use_fixed_radius_prefilters = (
        str(USER_R_FULL_VB_MODE).strip().lower() == "legacy_threshold_only"
    )
    if PREFILTER_INVALID_DIRECTIONS and use_fixed_radius_prefilters:
        kept_direction_cases, prefiltered_directions_scan_df = (
            prefilter_direction_cases(
                direction_cases=scan_direction_cases,
                impact_cases=impact_cases,
                R_full=R_full,
                R_max=R_max,
                n_path=n_path,
                positive_weight_only=PREFILTER_POSITIVE_WEIGHT_ONLY,
            )
        )
    else:
        kept_direction_cases = scan_direction_cases
        prefiltered_directions_scan_df = pd.DataFrame()

    print(
        f"Unique full direction cases: {len(full_direction_cases)}",
        flush=True,
    )
    print(
        "Direction representatives in x/y symmetry domain: "
        f"{len(scan_direction_cases)}",
        flush=True,
    )
    print(
        "Symmetry-domain directions retained after geometry prefilter: "
        f"{len(kept_direction_cases)}",
        flush=True,
    )
    print(
        "Symmetry-domain directions excluded before field evaluation: "
        f"{len(prefiltered_directions_scan_df)}",
        flush=True,
    )

    if PREFILTER_INVALID_GEOMETRIES and use_fixed_radius_prefilters:
        geometry_tasks, prefiltered_geometries_scan_df = (
            prefilter_geometry_tasks(
                direction_cases=kept_direction_cases,
                impact_cases=impact_cases,
                R_full=R_full,
                R_max=R_max,
            )
        )
    else:
        geometry_tasks = [
            (direction, impact_case)
            for direction in kept_direction_cases
            for impact_case in impact_cases
        ]
        prefiltered_geometries_scan_df = pd.DataFrame()

    print(
        "Symmetry-domain direction-impact geometries excluded before field "
        f"evaluation: {len(prefiltered_geometries_scan_df)}",
        flush=True,
    )

    def task_iterator():
        yield from geometry_tasks

    total_geometry_tasks = len(geometry_tasks)
    total_count = total_geometry_tasks * len(speed_cases)
    completed = 0
    completed_geometry_tasks = 0
    rows = make_prefiltered_geometry_rows(
        prefiltered_geometries_scan_df,
        speed_cases,
        mass=mass,
        R_full=R_full,
    )

    full_equivalent_geometry_tasks = (
        len(full_direction_cases) * len(impact_cases)
    )
    print(
        f"Starting parallel scan with {n_threads} worker threads, "
        f"{total_geometry_tasks} symmetry-domain geometry tasks "
        f"instead of {full_equivalent_geometry_tasks} full-grid tasks, and "
        f"at most {max_in_flight_tasks} queued/running tasks.",
        flush=True,
    )

    tasks = iter(task_iterator())

    with ThreadPoolExecutor(max_workers=n_threads) as executor:
        in_flight = {}

        def submit_one_task():
            try:
                direction, impact_case = next(tasks)
            except StopIteration:
                return False

            future = executor.submit(
                evaluate_theta_alpha_impact_task,
                direction["theta_index"],
                direction["alpha_index"],
                direction["theta_rad"],
                direction["alpha_rad"],
                impact_case,
                mass,
                charge_fraction,
                T,
                R_full,
                R_max,
                n_path,
                energy_tol,
                angle_tol,
                displacement_tol,
                speed_cases,
            )
            in_flight[future] = {
                "theta_index": direction["theta_index"],
                "alpha_index": direction["alpha_index"],
                "impact_case_id": impact_case["impact_case_id"],
            }
            return True

        for _ in range(min(max_in_flight_tasks, total_geometry_tasks)):
            if not submit_one_task():
                break

        while in_flight:
            done, _ = wait(
                in_flight,
                return_when=FIRST_COMPLETED,
            )

            for future in done:
                metadata = in_flight.pop(future)

                try:
                    task_rows = future.result()
                except Exception as exc:
                    for pending in in_flight:
                        pending.cancel()
                    raise RuntimeError(
                        "Parallel trajectory task failed for "
                        f"theta_index={metadata['theta_index']}, "
                        f"alpha_index={metadata['alpha_index']}, "
                        f"impact_case_id={metadata['impact_case_id']}"
                    ) from exc

                rows.extend(task_rows)
                completed_geometry_tasks += 1

                for row in task_rows:
                    completed += 1
                    if PRINT_EACH_TRAJECTORY:
                        print_trajectory_result(
                            row,
                            current_index=completed,
                            total_count=total_count,
                        )

                if (
                    completed_geometry_tasks % 100 == 0
                    or completed_geometry_tasks == total_geometry_tasks
                ):
                    print(
                        "Completed "
                        f"{completed_geometry_tasks}/{total_geometry_tasks} "
                        "symmetry-domain geometry tasks "
                        f"({completed}/{total_count} speed trajectories).",
                        flush=True,
                    )

                submit_one_task()

    symmetry_results_df = pd.DataFrame(rows)

    if (
        EXPLOIT_XY_REFLECTION_SYMMETRY
        and RECONSTRUCT_FULL_DIRECTION_OUTPUTS
    ):
        all_results_df = expand_rows_by_xy_symmetry(
            symmetry_results_df,
            symmetry_direction_map_df,
            impact_cases,
        )
        prefiltered_directions_df = expand_direction_audit_by_xy_symmetry(
            prefiltered_directions_scan_df,
            symmetry_direction_map_df,
        )
        prefiltered_geometries_df = expand_rows_by_xy_symmetry(
            prefiltered_geometries_scan_df,
            symmetry_direction_map_df,
            impact_cases,
        )
    else:
        all_results_df = symmetry_results_df.copy()
        prefiltered_directions_df = prefiltered_directions_scan_df.copy()
        prefiltered_geometries_df = prefiltered_geometries_scan_df.copy()

    all_results_df = all_results_df.sort_values(
        [
            "speed_quantile",
            "theta_index",
            "alpha_index",
            "impact_case_id",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
    all_results_df = assign_stable_trajectory_ids(all_results_df)

    test2_input_df, test2_not_ready_df = build_test2_input_table(
        all_results_df,
        include_zero_weight_diagnostics=(
            TEST2_INCLUDE_ZERO_WEIGHT_DIAGNOSTICS
        ),
    )

    aggregate_df, postscan_excluded_directions_df = aggregate_bpsi_results(
        all_results_df,
        R_max,
    )
    aggregate_df = aggregate_df.sort_values(
        ["speed_quantile", "theta_index", "alpha_index"],
        kind="mergesort",
    ).reset_index(drop=True)

    invalid_results_df = all_results_df.loc[
        all_results_df["status"].isin(INVALID_TRAJECTORY_STATUSES)
    ].copy()
    outward_rejected_results_df = all_results_df.loc[
        all_results_df["status"].isin(SCREENED_REJECT_STATUSES)
    ].copy()
    outward_diagnostic_results_df = all_results_df.loc[
        all_results_df["outward_force_diagnostic_flag"].fillna(False)
    ].copy()
    results_df = all_results_df.loc[
        ~all_results_df["status"].isin(EXCLUDED_TRAJECTORY_STATUSES)
    ].copy().reset_index(drop=True)

    maps_by_speed = populate_aggregate_maps(
        aggregate_df,
        theta_values,
        alpha_values,
        speed_cases,
    )

    dm_only_candidates_df = select_dm_only_candidates(results_df)

    return (
        all_results_df,
        results_df,
        invalid_results_df,
        outward_rejected_results_df,
        outward_diagnostic_results_df,
        test2_input_df,
        test2_not_ready_df,
        aggregate_df,
        maps_by_speed,
        speed_cases,
        theta_values,
        alpha_values,
        prefiltered_directions_df,
        prefiltered_geometries_df,
        postscan_excluded_directions_df,
        dm_only_candidates_df,
        symmetry_direction_map_df,
    )

def build_exclusion_pattern_tables(
    results_df,
    prefiltered_geometries_df,
    invalid_results_df,
    outward_rejected_results_df,
    R_full,
):
    """Build speed-independent exclusion masks and pattern summaries.

    Invalid and outward-force outcomes repeat once per speed. The exact masks
    below retain one row per physical (theta, alpha, b, psi) geometry.
    """
    geometry_columns = [
        "theta_index",
        "alpha_index",
        "theta_rad",
        "alpha_rad",
        "theta_deg",
        "alpha_deg",
        "impact_case_id",
        "b_m",
        "b_um",
        "psi_rad",
        "psi_deg",
    ]

    prefiltered_mask = prefiltered_geometries_df.copy()

    if len(invalid_results_df) > 0:
        numerical_invalid_mask = (
            invalid_results_df
            .sort_values(geometry_columns)
            .drop_duplicates(geometry_columns)
            .copy()
        )
    else:
        numerical_invalid_mask = pd.DataFrame(columns=geometry_columns)

    if len(outward_rejected_results_df) > 0:
        outward_mask = (
            outward_rejected_results_df
            .sort_values(geometry_columns)
            .drop_duplicates(geometry_columns)
            .copy()
        )
    else:
        outward_mask = pd.DataFrame(columns=geometry_columns)

    def summarize(mask_df, outcome_name):
        if len(mask_df) == 0:
            return pd.DataFrame(), pd.DataFrame()

        by_direction = (
            mask_df.groupby(
                ["theta_deg", "alpha_deg"],
                dropna=False,
            )
            .agg(
                n_geometries=("impact_case_id", "nunique"),
                min_b_um=("b_um", "min"),
                max_b_um=("b_um", "max"),
                n_unique_psi=("psi_deg", "nunique"),
            )
            .reset_index()
            .sort_values("n_geometries", ascending=False)
        )
        by_direction.insert(0, "outcome", outcome_name)

        by_bpsi = (
            mask_df.groupby(
                ["b_um", "psi_deg"],
                dropna=False,
            )
            .agg(
                n_directions=("theta_index", "size"),
                n_unique_theta=("theta_deg", "nunique"),
                n_unique_alpha=("alpha_deg", "nunique"),
            )
            .reset_index()
            .sort_values("n_directions", ascending=False)
        )
        by_bpsi.insert(0, "outcome", outcome_name)
        return by_direction, by_bpsi

    pre_dir, pre_bpsi = summarize(
        prefiltered_mask,
        "analytic_geometry_prefilter",
    )
    num_dir, num_bpsi = summarize(
        numerical_invalid_mask,
        "post_field_invalid",
    )
    out_dir, out_bpsi = summarize(
        outward_mask,
        "outward_force_reject",
    )

    direction_summary = pd.concat(
        [pre_dir, num_dir, out_dir],
        ignore_index=True,
    )
    bpsi_summary = pd.concat(
        [pre_bpsi, num_bpsi, out_bpsi],
        ignore_index=True,
    )

    # Build a speed-independent b-cutoff audit for every theta-alpha-psi ray.
    # A cutoff is considered bracketed only when rejection is monotone in the
    # sampled b values: once the force screen rejects, all larger sampled b
    # values are also rejected.
    combined_parts = [
        df for df in (
            results_df,
            invalid_results_df,
            outward_rejected_results_df,
        )
        if len(df) > 0
    ]
    combined = (
        pd.concat(combined_parts, ignore_index=True, sort=False)
        if combined_parts
        else pd.DataFrame()
    )
    if len(combined) > 0:
        geometry = (
            combined
            .sort_values(geometry_columns)
            .drop_duplicates(geometry_columns)
        )
        geometry = geometry.loc[
            (geometry["impact_weight"] > 0.0)
            & (geometry["b_m"] >= R_full)
            & ~geometry["status"].isin(INVALID_TRAJECTORY_STATUSES)
        ].copy()
    else:
        geometry = pd.DataFrame()

    cutoff_rows = []
    if len(geometry) > 0:
        for keys, group in geometry.groupby(
            [
                "theta_index",
                "alpha_index",
                "theta_deg",
                "alpha_deg",
                "psi_rad",
                "psi_deg",
            ],
            sort=False,
        ):
            group = group.sort_values("b_m")
            rejected = group["status"].isin(SCREENED_REJECT_STATUSES).to_numpy()
            b_values = group["b_m"].to_numpy(dtype=float)

            if np.any(rejected):
                first_reject = int(np.flatnonzero(rejected)[0])
                monotone = bool(np.all(rejected[first_reject:]))
                min_rejected_b = float(b_values[first_reject])
                retained_below = b_values[:first_reject][~rejected[:first_reject]]
                max_retained_b = (
                    float(np.max(retained_below))
                    if retained_below.size > 0
                    else float(R_full)
                )
            else:
                first_reject = -1
                monotone = False
                min_rejected_b = np.nan
                retained = b_values[~rejected]
                max_retained_b = (
                    float(np.max(retained))
                    if retained.size > 0 else np.nan
                )

            cutoff_rows.append({
                "theta_index": keys[0],
                "alpha_index": keys[1],
                "theta_deg": keys[2],
                "alpha_deg": keys[3],
                "psi_rad": keys[4],
                "psi_deg": keys[5],
                "n_sampled_b": len(group),
                "n_rejected_b": int(np.sum(rejected)),
                "monotone_outward_reject_in_b": monotone,
                "max_retained_b_um": max_retained_b * 1.0e6,
                "min_rejected_b_um": min_rejected_b * 1.0e6,
                "cutoff_bracket_width_um": (
                    (min_rejected_b - max_retained_b) * 1.0e6
                    if np.isfinite(min_rejected_b)
                    and np.isfinite(max_retained_b)
                    else np.nan
                ),
                "safe_to_reuse_as_empirical_cutoff": bool(monotone),
            })

    b_cutoff_brackets = pd.DataFrame(cutoff_rows)

    return {
        "prefiltered_geometry_mask": prefiltered_mask,
        "post_field_invalid_geometry_mask": numerical_invalid_mask,
        "outward_reject_geometry_mask": outward_mask,
        "exclusion_patterns_by_direction": direction_summary,
        "exclusion_patterns_by_bpsi": bpsi_summary,
        "outward_reject_b_cutoff_brackets": b_cutoff_brackets,
    }


# ============================================================
# PLOT HELPERS
# ============================================================

def status_legend_handles():
    handles = []

    for status in STATUS_ORDER:
        marker = STATUS_MARKERS[status]
        color = STATUS_COLORS[status]

        handles.append(
            Line2D(
                [0],
                [0],
                marker=marker,
                linestyle="None",
                markerfacecolor=(
                    "none" if marker not in {"x", "+", "*"}
                    else color
                ),
                markeredgecolor=color,
                color=color,
                markersize=7,
                label=STATUS_LABELS[status],
            )
        )

    return handles


def overlay_status_markers(ax, maps):
    theta_deg = np.degrees(maps["theta_values"])
    alpha_deg = np.degrees(maps["alpha_values"])
    alpha_mesh, theta_mesh = np.meshgrid(alpha_deg, theta_deg)
    status_map = maps["status"]

    for status in STATUS_ORDER:
        mask = status_map == status
        if not np.any(mask):
            continue

        marker = STATUS_MARKERS[status]
        color = STATUS_COLORS[status]

        if marker in {"x", "+", "*"}:
            ax.scatter(
                alpha_mesh[mask],
                theta_mesh[mask],
                marker=marker,
                s=18,
                c=color,
                linewidths=0.8,
            )
        else:
            ax.scatter(
                alpha_mesh[mask],
                theta_mesh[mask],
                marker=marker,
                s=12,
                facecolors="none",
                edgecolors=color,
                linewidths=0.55,
            )

    ax.legend(
        handles=status_legend_handles(),
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        fontsize=8,
    )


def plot_R_far_heatmap(
    maps,
    R_full,
    R_max,
    field="R_far_max_um",
    title_suffix="",
):
    theta_deg = np.degrees(maps["theta_values"])
    alpha_deg = np.degrees(maps["alpha_values"])
    values = np.asarray(maps[field], dtype=float)

    finite_positive = np.isfinite(values) & (values > 0.0)
    if not np.any(finite_positive):
        print(f"No finite positive values available for {field}.")
        return

    plot_data = np.ma.masked_where(~finite_positive, values)

    fig, ax = plt.subplots(figsize=(12, 6))
    pcm = ax.pcolormesh(
        alpha_deg,
        theta_deg,
        plot_data,
        shading="auto",
        norm=LogNorm(
            vmin=R_full * 1.0e6,
            vmax=R_max * 1.0e6,
        ),
    )

    labels = {
        "R_far_max_um": (
            r"Maximum $R_{\rm far}$ over positive-weight $(b,\psi)$"
        ),
        "R_far_max_with_diagnostics_um": (
            r"Maximum $R_{\rm far}$ including diagnostic edge points"
        ),
        "R_far_p95_um": r"Weighted 95th percentile of $R_{\rm far}$",
        "R_far_p99_um": r"Weighted 99th percentile of $R_{\rm far}$",
    }

    ax.set_xlabel(r"$\alpha$ [degrees]")
    ax.set_ylabel(r"$\theta$ [degrees]")
    ax.set_title(labels.get(field, field) + title_suffix)

    cbar = fig.colorbar(pcm, ax=ax)
    cbar.set_label(r"$R_{\rm far}$ [$\mu$m]")

    overlay_status_markers(ax, maps)
    plt.tight_layout()
    plt.show()


def plot_continuous_heatmap(
    maps,
    field,
    title,
    colorbar_label,
    title_suffix="",
    overlay_status=True,
):
    theta_deg = np.degrees(maps["theta_values"])
    alpha_deg = np.degrees(maps["alpha_values"])
    values = np.asarray(maps[field], dtype=float)
    plot_data = np.ma.masked_invalid(values)

    if plot_data.count() == 0:
        return

    fig, ax = plt.subplots(figsize=(12, 6))
    pcm = ax.pcolormesh(
        alpha_deg,
        theta_deg,
        plot_data,
        shading="auto",
    )

    ax.set_xlabel(r"$\alpha$ [degrees]")
    ax.set_ylabel(r"$\theta$ [degrees]")
    ax.set_title(title + title_suffix)

    cbar = fig.colorbar(pcm, ax=ax)
    cbar.set_label(colorbar_label)

    if overlay_status:
        overlay_status_markers(ax, maps)

    plt.tight_layout()
    plt.show()


def plot_vmin_heatmap(maps, title_suffix=""):
    values = np.asarray(maps["v_min_m_s"], dtype=float)
    finite_positive = np.isfinite(values) & (values > 0.0)

    if not np.any(finite_positive):
        return

    theta_deg = np.degrees(maps["theta_values"])
    alpha_deg = np.degrees(maps["alpha_values"])
    plot_data = np.ma.masked_where(~finite_positive, values)

    fig, ax = plt.subplots(figsize=(12, 6))
    pcm = ax.pcolormesh(
        alpha_deg,
        theta_deg,
        plot_data,
        shading="auto",
        norm=LogNorm(
            vmin=np.nanmin(values[finite_positive]),
            vmax=np.nanmax(values[finite_positive]),
        ),
    )

    ax.set_xlabel(r"$\alpha$ [degrees]")
    ax.set_ylabel(r"$\theta$ [degrees]")
    ax.set_title(
        r"Worst straight-path barrier speed over sampled $(b,\psi)$"
        + title_suffix
    )

    cbar = fig.colorbar(pcm, ax=ax)
    cbar.set_label(r"$\max_{b,\psi}v_{\min}$ [m/s]")

    overlay_status_markers(ax, maps)
    plt.tight_layout()
    plt.show()


def plot_access_probability_heatmap(maps, title_suffix=""):
    values = np.asarray(maps["P_access"], dtype=float)
    safe_values = np.maximum(values, 1.0e-300)
    log_values = np.log10(safe_values)
    log_values[~np.isfinite(values)] = np.nan
    plot_data = np.ma.masked_invalid(log_values)

    if plot_data.count() == 0:
        return

    theta_deg = np.degrees(maps["theta_values"])
    alpha_deg = np.degrees(maps["alpha_values"])

    fig, ax = plt.subplots(figsize=(12, 6))
    pcm = ax.pcolormesh(
        alpha_deg,
        theta_deg,
        plot_data,
        shading="auto",
    )

    ax.set_xlabel(r"$\alpha$ [degrees]")
    ax.set_ylabel(r"$\theta$ [degrees]")
    ax.set_title(
        r"Worst straight-path accessibility over sampled $(b,\psi)$"
        + title_suffix
    )

    cbar = fig.colorbar(pcm, ax=ax)
    cbar.set_label(r"$\log_{10}\min_{b,\psi}P_{\rm access}$")

    overlay_status_markers(ax, maps)
    plt.tight_layout()
    plt.show()


def plot_classification_heatmap(maps, title_suffix=""):
    theta_deg = np.degrees(maps["theta_values"])
    alpha_deg = np.degrees(maps["alpha_values"])
    status_map = maps["status"]

    status_to_code = {
        status: idx for idx, status in enumerate(STATUS_ORDER)
    }
    code_map = np.full(status_map.shape, -1, dtype=int)

    for status, code in status_to_code.items():
        code_map[status_map == status] = code

    cmap = ListedColormap([
        STATUS_COLORS[status] for status in STATUS_ORDER
    ])
    norm = BoundaryNorm(
        np.arange(-0.5, len(STATUS_ORDER) + 0.5, 1.0),
        cmap.N,
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.pcolormesh(
        alpha_deg,
        theta_deg,
        code_map,
        shading="auto",
        cmap=cmap,
        norm=norm,
    )

    ax.set_xlabel(r"$\alpha$ [degrees]")
    ax.set_ylabel(r"$\theta$ [degrees]")
    ax.set_title(
        "Worst-case / maximum-R_far classification over sampled b, psi"
        + title_suffix
    )

    handles = [
        Patch(
            facecolor=STATUS_COLORS[status],
            edgecolor="black",
            label=STATUS_LABELS[status],
        )
        for status in STATUS_ORDER
    ]
    ax.legend(
        handles=handles,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        fontsize=8,
    )

    plt.tight_layout()
    plt.show()


def plot_limiting_criterion_heatmap(maps, title_suffix=""):
    theta_deg = np.degrees(maps["theta_values"])
    alpha_deg = np.degrees(maps["alpha_values"])
    criterion_map = maps["limiting_criterion"]

    criterion_to_code = {
        criterion: idx
        for idx, criterion in enumerate(LIMITING_ORDER)
    }
    code_map = np.full(criterion_map.shape, -1, dtype=int)

    for criterion, code in criterion_to_code.items():
        code_map[criterion_map == criterion] = code

    cmap = ListedColormap([
        LIMITING_COLORS[criterion]
        for criterion in LIMITING_ORDER
    ])
    norm = BoundaryNorm(
        np.arange(-0.5, len(LIMITING_ORDER) + 0.5, 1.0),
        cmap.N,
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.pcolormesh(
        alpha_deg,
        theta_deg,
        code_map,
        shading="auto",
        cmap=cmap,
        norm=norm,
    )

    ax.set_xlabel(r"$\alpha$ [degrees]")
    ax.set_ylabel(r"$\theta$ [degrees]")
    ax.set_title(
        "Criterion setting maximum R_far over sampled b, psi"
        + title_suffix
    )

    handles = [
        Patch(
            facecolor=LIMITING_COLORS[criterion],
            edgecolor="black",
            label=LIMITING_LABELS[criterion],
        )
        for criterion in LIMITING_ORDER
    ]
    ax.legend(
        handles=handles,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        fontsize=8,
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# SUMMARY PRINTING
# ============================================================

def print_speed_summary(raw_speed_df, aggregate_speed_df, speed_case):
    quantile = speed_case["quantile"]
    speed = speed_case["speed_m_s"]

    print()
    print("=" * 110)
    print(
        f"Conditional tail quantile {quantile:.3f}: "
        f"unconditional q={speed_case.get('unconditional_quantile', np.nan):.6f}, "
        f"v_inf = {speed:.6f} m/s"
    )
    print("=" * 110)

    print()
    print("Retained valid/unresolved b-psi trajectory status counts:")
    print(
        raw_speed_df["status"]
        .value_counts(dropna=False)
        .reindex(STATUS_ORDER, fill_value=0)
        .to_string()
    )

    print()
    print("Aggregated theta-alpha status counts:")
    print(
        aggregate_speed_df["status"]
        .value_counts(dropna=False)
        .reindex(STATUS_ORDER, fill_value=0)
        .to_string()
    )

    finite_R = aggregate_speed_df.loc[
        np.isfinite(aggregate_speed_df["R_far_max_um"]),
        "R_far_max_um",
    ]

    print()
    print("Maximum R_far over positive-weight b, psi samples [um]:")
    if len(finite_R) > 0:
        print(finite_R.describe().to_string())
    else:
        print("No finite R_far maxima.")

    print()
    print("Limiting criterion at maximum R_far:")
    print(
        aggregate_speed_df["limiting_criterion"]
        .value_counts(dropna=False)
        .reindex(LIMITING_ORDER, fill_value=0)
        .to_string()
    )

    worst_columns = [
        "theta_deg",
        "alpha_deg",
        "R_far_max_um",
        "R_far_max_with_diagnostics_um",
        "R_far_p95_um",
        "R_far_p99_um",
        "b_at_max_um",
        "psi_at_max_deg",
        "status",
        "limiting_criterion",
        "n_unresolved_samples",
        "unresolved_weight_fraction",
    ]

    print()
    print(
        f"Top {N_WORST_DIRECTIONS_TO_PRINT} theta-alpha directions "
        "by maximum R_far:"
    )
    print(
        aggregate_speed_df
        .sort_values("R_far_max_um", ascending=False)
        .head(N_WORST_DIRECTIONS_TO_PRINT)[worst_columns]
        .to_string(index=False)
    )

    weighted_b_max = raw_speed_df.loc[
        raw_speed_df["impact_weight"] > 0.0,
        "b_um",
    ].max()
    n_at_weighted_edge = int(np.isclose(
        aggregate_speed_df["b_at_max_um"],
        weighted_b_max,
    ).sum())
    print()
    print(
        "Directions whose positive-weight maximum is at the outermost "
        f"weighted b sample ({weighted_b_max:.3f} um): "
        f"{n_at_weighted_edge}/{len(aggregate_speed_df)}"
    )
    if n_at_weighted_edge > 0:
        print(
            "NOTE: R_far_actual includes the impact parameter geometrically, "
            "so an outer b sample may maximize it even when the outer-tail "
            "cutoff is converged. Test 2 reachability, not this maximum, will "
            "determine the physical b cutoff."
        )
    n_Rmax_lower_bounds = int(
        aggregate_speed_df["R_far_is_lower_bound"].fillna(False).sum()
    )
    print(
        "Directions still unresolved at the selected R_max: "
        f"{n_Rmax_lower_bounds}/{len(aggregate_speed_df)}"
    )
    print(
        "Mean retained impact weight fraction: "
        f"{aggregate_speed_df['valid_weight_fraction'].mean():.6f}"
    )
    print(
        "Mean outward-force rejected impact weight fraction: "
        f"{aggregate_speed_df['outward_reject_weight_fraction'].mean():.6f}"
    )
    finite_upper = aggregate_speed_df["max_retained_miss_b_um"].dropna()
    if len(finite_upper) > 0:
        print(
            "Median sampled upper bound for retained nominal misses: "
            f"{finite_upper.median():.3f} um"
        )

    if PRINT_AGGREGATED_TABLE_AFTER_EACH_SPEED:
        print()
        print("Complete theta-alpha table after maximizing over b, psi:")
        print(aggregate_speed_df.to_string(index=False))

    if PRINT_RAW_TABLE_AFTER_EACH_SPEED:
        print()
        print("Complete raw b-psi trajectory table for this speed:")
        print(raw_speed_df.to_string(index=False))



# ============================================================
# TEST-0-v9 SPEED-DEPENDENT MAIN WORKFLOW
# ============================================================


def _concat_frames(frames):
    nonempty = [frame for frame in frames if frame is not None and len(frame)]
    return pd.concat(nonempty, ignore_index=True, sort=False) if nonempty else pd.DataFrame()


def _classify_trajectory_from_speed_policy(b_m, speed_case):
    """Return the conservative Test-0.75 regime for this exact ``(v, b)``."""
    b_value = float(b_m)
    b_rutherford = float(speed_case.get("b_rutherford_max_m", 0.0))
    b_adiabatic = float(speed_case.get("b_adiabatic_min_m", np.inf))
    # Rejection takes precedence in the unlikely event of overlapping model
    # bounds near the kinematic endpoint.
    if np.isfinite(b_adiabatic) and b_value >= b_adiabatic:
        return "adiabatic", "reject_below_threshold"
    if np.isfinite(b_rutherford) and b_rutherford > 0.0 and b_value <= b_rutherford:
        return "rutherford", "rutherford_analytic"
    return "resonant", "full_trap_required"


def _attach_speed_policy_columns(frame, speed_case):
    if frame is None or len(frame) == 0:
        return frame
    output = frame.copy()
    # The speed policy contains the old speed-only threshold radii. Copy
    # those under explicit aliases so the exact per-row R_full(v,b) calculated
    # above is not overwritten.
    speed_only_radius_aliases = {
        "r_min_threshold_m": "r_min_threshold_speed_only_m",
        "r_min_threshold_um": "r_min_threshold_speed_only_um",
        "R_full_m": "R_full_speed_only_m",
        "R_full_um": "R_full_speed_only_um",
        "R_switch_m": "R_switch_speed_only_m",
        "R_switch_um": "R_switch_speed_only_um",
    }
    for source, destination in speed_only_radius_aliases.items():
        if source in speed_case:
            output[destination] = speed_case[source]

    metadata_names = (
        "speed_interval_label",
        "speed_interval_lower_m_s",
        "speed_interval_upper_m_s",
        "speed_interval_probability",
        "total_allowed_speed_probability",
        "R_switch_factor",
        "b_threshold_m",
        "b_threshold_um",
        "b_rutherford_max_m",
        "b_rutherford_max_um",
        "b_rutherford_max_over_b_threshold",
        "b_adiabatic_min_m",
        "b_adiabatic_min_um",
        "b_adiabatic_min_over_b_threshold",
        "rutherford_detectable_area_fraction",
        "policy_b_max_m",
        "trajectory_policy_b_max_m",
        "rutherford_relative_tolerance",
        "test0p75_model_safety_factor",
        "test0p75_interaction_time_scale",
        "speed_policy_source",
    )
    for column in metadata_names:
        if column in speed_case:
            output[column] = speed_case[column]
    output["test0_v9_reject_lower_m_s"] = V9_REJECT_LOWER_M_S
    output["test0_v9_reject_upper_m_s"] = V9_REJECT_UPPER_M_S

    if "b_m" in output.columns:
        classified = [
            _classify_trajectory_from_speed_policy(b_value, speed_case)
            for b_value in output["b_m"].to_numpy(float)
        ]
        output["trajectory_regime"] = [value[0] for value in classified]
        output["collision_energy_regime"] = output["trajectory_regime"]
        output["ion_energy_model"] = [value[1] for value in classified]
        output["test0p75_adiabatic_reject"] = output["trajectory_regime"].eq(
            "adiabatic"
        )
        output["requires_dm_only_reach_screen"] = ~output[
            "test0p75_adiabatic_reject"
        ]
        output["use_rutherford_after_reach"] = output["trajectory_regime"].eq(
            "rutherford"
        )
        output["requires_full_coupled_after_reach"] = output[
            "trajectory_regime"
        ].eq("resonant")
        # Compatibility columns. Rutherford is no longer evaluated before the
        # reach screen; this flag now means "use after a confirmed entry".
        output["use_rutherford_final_energy"] = output[
            "use_rutherford_after_reach"
        ]
        output["requires_full_trap_propagation"] = output[
            "requires_full_coupled_after_reach"
        ]
    return output


def load_test0p75_speed_policy(path: Path) -> list[dict]:
    """Load and validate the Test-1-ready policy produced by Test 0.75."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Test 0.75 speed policy was not found: {path.resolve()}"
        )
    frame = pd.read_csv(path)
    validate_parameter_dataframe(frame, "Test 0.75 speed policy")
    required = {
        "key",
        "speed_m_s",
        "v_inf_m_s",
        "speed_interval_label",
        "speed_interval_lower_m_s",
        "speed_interval_upper_m_s",
        "speed_interval_probability",
        "total_allowed_speed_probability",
        "conditional_speed_weight",
        "unconditional_speed_weight",
        "r_min_threshold_m",
        "r_min_threshold_um",
        "R_full_m",
        "R_full_um",
        "R_switch_factor",
        "R_switch_m",
        "R_switch_um",
        "b_threshold_m",
        "b_rutherford_max_m",
        "b_adiabatic_min_m",
    }
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(
            "Test 0.75 policy is missing required columns: " + ", ".join(missing)
        )
    numeric = sorted(required - {"key", "speed_interval_label"})
    for column in numeric:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
    frame = frame.sort_values("speed_m_s").reset_index(drop=True)
    if len(frame) == 0:
        raise ValueError("Test 0.75 policy contains no representative speeds")
    if not np.all(np.isfinite(frame["speed_m_s"]) & (frame["speed_m_s"] > 0.0)):
        raise ValueError("Test 0.75 policy contains invalid speeds")
    if not np.all(np.isfinite(frame["R_full_m"]) & (frame["R_full_m"] > 0.0)):
        raise ValueError("Test 0.75 policy contains invalid R_full values")
    if not np.allclose(
        frame["R_switch_m"],
        frame["R_switch_factor"] * frame["R_full_m"],
        rtol=1.0e-10,
        atol=1.0e-15,
    ):
        raise ValueError("Test 0.75 R_switch values are inconsistent with R_full")
    return frame.to_dict(orient="records")


def run_test1_v9_speed_dependent():
    print(f"Test 1 script version: {TEST1_SCRIPT_VERSION}", flush=True)
    required = ("m_ion", "Z_ion", "K", "e")
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            "Define the ion/Rutherford globals before Test 1: "
            + ", ".join(missing)
        )

    if USE_TEST0P75_SPEED_POLICY:
        if TEST0P75_SPEED_POLICY_CSV.exists():
            speed_cases_policy = load_test0p75_speed_policy(
                TEST0P75_SPEED_POLICY_CSV
            )
            speed_policy_source = (
                f"Test 0.75 CSV: {TEST0P75_SPEED_POLICY_CSV}"
            )
        elif REQUIRE_TEST0P75_SPEED_POLICY:
            raise FileNotFoundError(
                "Run test0p75_adiabatic_regime_sweep.py first. Missing: "
                + str(TEST0P75_SPEED_POLICY_CSV.resolve())
            )
        else:
            speed_cases_policy = make_v9_representative_speed_cases(
                m_dm_kg=m_dm,
                m_ion_kg=float(m_ion),
                temperature_k=T_dm,
                eps=eps,
                ion_charge_number=float(Z_ion),
                coulomb_constant=float(K),
                elementary_charge_c=float(e),
                include_high_speed_branch=INCLUDE_HIGH_SPEED_BRANCH,
                upper_tail_probability=UPPER_TAIL_PROBABILITY,
                low_interval_quantiles=LOW_INTERVAL_REPRESENTATIVE_QUANTILES,
                high_interval_quantiles=HIGH_INTERVAL_REPRESENTATIVE_QUANTILES,
                r_switch_factor=R_SWITCH_FACTOR,
            )
            speed_policy_source = "fallback Test 0 v9 policy"
    else:
        speed_cases_policy = make_v9_representative_speed_cases(
            m_dm_kg=m_dm,
            m_ion_kg=float(m_ion),
            temperature_k=T_dm,
            eps=eps,
            ion_charge_number=float(Z_ion),
            coulomb_constant=float(K),
            elementary_charge_c=float(e),
            include_high_speed_branch=INCLUDE_HIGH_SPEED_BRANCH,
            upper_tail_probability=UPPER_TAIL_PROBABILITY,
            low_interval_quantiles=LOW_INTERVAL_REPRESENTATIVE_QUANTILES,
            high_interval_quantiles=HIGH_INTERVAL_REPRESENTATIVE_QUANTILES,
            r_switch_factor=R_SWITCH_FACTOR,
        )
        speed_policy_source = "Test 0 v9 policy"

    print(f"Test 1 (v,b)-regime policy from {speed_policy_source}:", flush=True)
    print(
        f"  rejected interval = ({V9_REJECT_LOWER_M_S:.12g}, "
        f"{V9_REJECT_UPPER_M_S:.12g}) m/s",
        flush=True,
    )
    print(
        f"  Maxwell upper-tail truncation probability = "
        f"{UPPER_TAIL_PROBABILITY:.3e}",
        flush=True,
    )
    for case in speed_cases_policy:
        print(
            f"  {case['key']}: v={case['speed_m_s']:.12g} m/s, "
            f"R_full={case['R_full_um']:.12g} um, "
            f"R_switch={case['R_switch_um']:.12g} um, "
            f"b_Ruth,max={float(case.get('b_rutherford_max_m', 0.0))*1e6:.3f} um, "
            f"b_adia,min={float(case.get('b_adiabatic_min_m', np.inf))*1e6:.3f} um, "
            f"w_uncond={case['unconditional_speed_weight']:.6e}",
            flush=True,
        )

    tuple_results = []
    impact_tables = []
    for speed_index, speed_case in enumerate(speed_cases_policy, start=1):
        r_full_case = float(speed_case["R_full_m"])
        if not (0.0 < r_full_case < b_max):
            raise ValueError(
                f"Speed {speed_case['speed_m_s']:.12g} m/s has "
                f"R_full={r_full_case*1e6:.6f} um, outside (0, b_max). "
                "Increase b_max or revise the sampled speed interval."
            )

        impact_cases = make_impact_geometry_samples(
            b_max=b_max,
            R_full=r_full_case,
            n_b_inner=n_b_inner,
            n_b_outer=n_b_outer,
            n_psi=n_psi,
            include_b_zero=INCLUDE_B_ZERO_DIAGNOSTIC,
            include_R_full=INCLUDE_R_FULL_DIAGNOSTIC,
            include_b_max=INCLUDE_B_MAX_DIAGNOSTIC,
        )
        impact_cases = append_reach_diagnostic_impact_cases(
            impact_cases,
            speed_case=speed_case,
            R_full=r_full_case,
            b_max=b_max,
            n_psi=n_psi,
        )
        impact_table = pd.DataFrame(impact_cases)
        impact_table["speed_label"] = speed_case["key"]
        impact_table["v_inf_m_s"] = speed_case["speed_m_s"]
        impact_table["R_full_m"] = r_full_case
        impact_table["R_switch_m"] = speed_case["R_switch_m"]
        impact_tables.append(impact_table)

        print(
            f"\n[{speed_index}/{len(speed_cases_policy)}] "
            f"Running {speed_case['key']} at v={speed_case['speed_m_s']:.6f} m/s "
            f"with R_full={r_full_case*1e6:.6f} um and "
            f"R_switch={speed_case['R_switch_um']:.6f} um",
            flush=True,
        )

        result_tuple = run_theta_alpha_bpsi_speed_scan(
            mass=m_dm,
            charge_fraction=eps,
            T=T_dm,
            impact_cases=impact_cases,
            R_full=r_full_case,
            R_max=R_max,
            n_theta=n_theta,
            n_alpha=n_alpha,
            n_path=n_path,
            energy_tol=energy_tol,
            angle_tol=angle_tol,
            displacement_tol=displacement_tol,
            representative_quantiles=np.array([0.5]),
            minimum_speed_m_s=0.0,
            n_threads=N_THREADS,
            max_in_flight_tasks=MAX_IN_FLIGHT_TASKS,
            speed_cases_override=[speed_case],
        )
        tuple_results.append(
            tuple(
                _attach_speed_policy_columns(value, speed_case)
                if isinstance(value, pd.DataFrame)
                else value
                for value in result_tuple
            )
        )

    frame_indices = [0, 1, 2, 3, 4, 5, 6, 7, 12, 13, 14, 15]
    combined = {}
    for index in frame_indices:
        combined[index] = _concat_frames([result[index] for result in tuple_results])

    all_results_df = combined[0]
    results_df = combined[1]
    invalid_results_df = combined[2]
    outward_rejected_results_df = combined[3]
    outward_diagnostic_results_df = combined[4]
    test2_input_df = combined[5]
    test2_not_ready_df = combined[6]
    aggregate_df = combined[7]
    prefiltered_directions_df = combined[12]
    prefiltered_geometries_df = combined[13]
    postscan_excluded_directions_df = combined[14]
    dm_only_candidates_df = combined[15]

    maps_by_speed = {}
    for result in tuple_results:
        maps_by_speed.update(result[8])
    speed_cases = [case for result in tuple_results for case in result[9]]
    theta_values = tuple_results[0][10]
    alpha_values = tuple_results[0][11]
    symmetry_direction_map_df = tuple_results[0][16]

    sort_columns = [
        column
        for column in (
            "v_inf_m_s",
            "theta_index",
            "alpha_index",
            "impact_case_id",
        )
        if column in all_results_df.columns
    ]
    if sort_columns:
        all_results_df = all_results_df.sort_values(sort_columns, kind="mergesort").reset_index(drop=True)

    speed_policy_df = pd.DataFrame(speed_cases_policy)
    speed_policy_df.to_csv(f"{OUTPUT_PREFIX}_speed_policy.csv", index=False)
    _concat_frames(impact_tables).to_csv(
        f"{OUTPUT_PREFIX}_speed_dependent_impact_samples.csv", index=False
    )

    if SAVE_RESULTS:
        all_results_df.to_csv(f"{OUTPUT_PREFIX}_raw_all.csv", index=False)
        results_df.to_csv(f"{OUTPUT_PREFIX}_raw_retained.csv", index=False)
        invalid_results_df.to_csv(f"{OUTPUT_PREFIX}_raw_excluded_invalid.csv", index=False)
        outward_rejected_results_df.to_csv(
            f"{OUTPUT_PREFIX}_raw_outward_force_rejected.csv", index=False
        )
        outward_diagnostic_results_df.to_csv(
            f"{OUTPUT_PREFIX}_raw_outward_force_diagnostic_flags.csv", index=False
        )
        if SAVE_TEST2_INPUT:
            test2_input_df.to_csv(f"{OUTPUT_PREFIX}_test2_input.csv", index=False)
            test2_not_ready_df.to_csv(
                f"{OUTPUT_PREFIX}_test2_not_ready_audit.csv", index=False
            )
        aggregate_df.to_csv(f"{OUTPUT_PREFIX}_aggregate.csv", index=False)
        prefiltered_directions_df.to_csv(
            f"{OUTPUT_PREFIX}_prefiltered_directions.csv", index=False
        )
        prefiltered_geometries_df.to_csv(
            f"{OUTPUT_PREFIX}_prefiltered_geometries.csv", index=False
        )
        postscan_excluded_directions_df.to_csv(
            f"{OUTPUT_PREFIX}_postscan_excluded_directions.csv", index=False
        )
        dm_only_candidates_df.to_csv(
            f"{OUTPUT_PREFIX}_dm_only_candidates.csv", index=False
        )
        symmetry_direction_map_df.to_csv(
            f"{OUTPUT_PREFIX}_xy_symmetry_direction_map.csv", index=False
        )

    print("\nTest 1 v9 speed-dependent summary:")
    print(f"  complete raw rows = {len(all_results_df):,}")
    print(f"  Test 2 ready rows = {len(test2_input_df):,}")
    print(f"  Test 2 not-ready rows = {len(test2_not_ready_df):,}")

    for speed_case in speed_cases:
        key = speed_case["key"]
        raw_speed_df = results_df.loc[results_df["speed_label"] == key].copy()
        aggregate_speed_df = aggregate_df.loc[aggregate_df["speed_label"] == key].copy()
        print_speed_summary(raw_speed_df, aggregate_speed_df, speed_case)

        if PLOT_EACH_SPEED and key in maps_by_speed:
            maps = maps_by_speed[key]
            r_full_case = float(speed_case["R_full_m"])
            speed_title = (
                f"\n{speed_case['speed_interval_label']}, q={speed_case['quantile']:.3f}, "
                f"v={speed_case['speed_m_s']:.1f} m/s, "
                f"R_full={r_full_case*1e6:.1f} um"
            )
            plot_R_far_heatmap(
                maps,
                R_full=r_full_case,
                R_max=R_max,
                field="R_far_max_um",
                title_suffix=speed_title,
            )
            plot_classification_heatmap(maps, title_suffix=speed_title)
            plot_limiting_criterion_heatmap(maps, title_suffix=speed_title)
            if PLOT_VALID_WEIGHT_FRACTION:
                plot_continuous_heatmap(
                    maps,
                    field="valid_weight_fraction",
                    title="Retained positive-weight impact fraction",
                    colorbar_label="Valid impact-weight fraction",
                    title_suffix=speed_title,
                    overlay_status=False,
                )
            plot_vmin_heatmap(
                maps,
                title_suffix=(
                    f"\nR_full(v)={r_full_case*1e6:.1f} um, "
                    f"v={speed_case['speed_m_s']:.1f} m/s"
                ),
            )
            plot_access_probability_heatmap(
                maps,
                title_suffix=(
                    f"\nR_full(v)={r_full_case*1e6:.1f} um, "
                    f"v={speed_case['speed_m_s']:.1f} m/s"
                ),
            )

    return {
        "all_results": all_results_df,
        "results": results_df,
        "test2_input": test2_input_df,
        "test2_not_ready": test2_not_ready_df,
        "aggregate": aggregate_df,
        "speed_cases": speed_policy_df,
        "maps_by_speed": maps_by_speed,
        "theta_values": theta_values,
        "alpha_values": alpha_values,
    }


if __name__ == "__main__":
    TEST1_V9_OUTPUTS = run_test1_v9_speed_dependent()

# Fast Two-Direction $R_{\max}$ Recovery

This stage replaces the expensive row-by-row geometric ladder. It automatically identifies the two angular direction keys containing the most `needs_larger_R_max` rows (or uses `USER_DIRECTIONAL_RMAX_DIRECTION_KEYS`), calibrates one conservative large radius per direction using endpoint-only potential evaluations, and then evaluates each unique $(\theta,\alpha,b,\psi)$ trap path only once. The path is reused for all speed rows sharing that geometry and the geometry groups run concurrently.

The broad escape-cone scan and per-row bowl ODE are intentionally skipped during this production recovery pass. They can be run later as separate diagnostics without blocking Tests 1.5 onward.


In [ ]:
#!/usr/bin/env python3
"""Fast directional R_max recovery for the small set of unresolved directions.

Run after Test 1 and before Test 1.5. This replaces the expensive row-by-row
adaptive ladder. It is designed for the common case where hundreds of unresolved
rows belong to only one or two angular directions.

Speedups
--------
1. Select only the most common unresolved direction keys (two by default).
2. Calibrate one conservative R_max per direction with cheap endpoint-only
   potential evaluations.
3. Evaluate the full U/F path once per unique (theta, alpha, b, psi) geometry,
   then reuse that path for every speed row in the group.
4. Run geometry groups concurrently.
5. Do not run a separate bowl ODE for every row.

The script writes the same merged Test 1 input filename expected by the existing
Test 1.5 code:

    escape_channel_adaptive_rmax_bowl_v2_merged_test1_test2_input.csv

No existing Test 1 output is overwritten.
"""

from __future__ import annotations

import json
import math
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd


# =============================================================================
# USER SETTINGS
# =============================================================================

RUN_DIR = Path(globals().get("RUN_DIRECTORY", "."))
TEST1_PREFIX = globals().get("TEST1_OUTPUT_PREFIX")
if TEST1_PREFIX is None:
    raise RuntimeError("TEST1_OUTPUT_PREFIX is not defined. Run Test 1 first.")

TEST1_RAW_ALL_CSV = Path(f"{TEST1_PREFIX}_raw_all.csv")
TEST1_NOT_READY_CSV = Path(f"{TEST1_PREFIX}_test2_not_ready_audit.csv")
TEST1_READY_INPUT_CSV = Path(f"{TEST1_PREFIX}_test2_input.csv")

# Keep the old prefix so Test 1.5 discovers this file automatically.
OUTPUT_PREFIX = RUN_DIR / "escape_channel_adaptive_rmax_bowl_v2"
ATTEMPTS_CSV = Path(f"{OUTPUT_PREFIX}_adaptive_rmax_attempts.csv")
FINAL_RESCAN_CSV = Path(f"{OUTPUT_PREFIX}_adaptive_rmax_final_rows.csv")
RESOLVED_TEST2_INPUT_CSV = Path(f"{OUTPUT_PREFIX}_newly_resolved_test2_input.csv")
MERGED_TEST1_INPUT_CSV = Path(f"{OUTPUT_PREFIX}_merged_test1_test2_input.csv")
STILL_UNRESOLVED_CSV = Path(f"{OUTPUT_PREFIX}_still_unresolved.csv")
ESCAPE_SCAN_CSV = Path(f"{OUTPUT_PREFIX}_escape_channel_barrier_scan.csv")
ESCAPE_FOCUS_PLAN_CSV = Path(f"{OUTPUT_PREFIX}_escape_channel_focus_plan.csv")
ESCAPE_FOCUS_TEST2_INPUT_CSV = Path(
    f"{OUTPUT_PREFIX}_escape_channel_focus_test2_input.csv"
)
ESCAPE_FOCUS_UNRESOLVED_CSV = Path(
    f"{OUTPUT_PREFIX}_escape_channel_focus_unresolved.csv"
)
BOWL_AUDIT_CSV = Path(f"{OUTPUT_PREFIX}_bowl_entry_audit.csv")
SUMMARY_JSON = Path(f"{OUTPUT_PREFIX}_summary.json")

# Usually all unresolved rows belong to exactly two directions. With no explicit
# list, choose the two direction keys containing the most unresolved rows.
# Explicit format example: ((0, 0), (12, 0)).
USER_DIRECTIONAL_RMAX_DIRECTION_KEYS = globals().get(
    "USER_DIRECTIONAL_RMAX_DIRECTION_KEYS", None
)
USER_DIRECTIONAL_RMAX_MAX_DIRECTIONS = int(
    globals().get("USER_DIRECTIONAL_RMAX_MAX_DIRECTIONS", 2)
)

# Cheap endpoint calibration ladder. The first value is already much larger
# than the previous 640 mm cap. A single radius is selected for each direction
# and then used for every row in that direction.
USER_DIRECTIONAL_RMAX_CALIBRATION_LADDER_M = tuple(
    float(value)
    for value in globals().get(
        "USER_DIRECTIONAL_RMAX_CALIBRATION_LADDER_M",
        (1.28, 2.56, 5.12, 10.24),
    )
)

# Full-path resolution after the radius has been calibrated. Because the path
# is logarithmic and reused across speeds, 900 is normally ample. Increase to
# 1400 for a confirmation run on a few rows.
USER_DIRECTIONAL_RMAX_N_PATH = int(
    globals().get("USER_DIRECTIONAL_RMAX_N_PATH", 900)
)
USER_DIRECTIONAL_RMAX_THREADS = int(
    globals().get("USER_DIRECTIONAL_RMAX_THREADS", 8)
)
USER_DIRECTIONAL_RMAX_PRINT_EVERY = int(
    globals().get("USER_DIRECTIONAL_RMAX_PRINT_EVERY", 20)
)

# Set True only for a later diagnostic pass. The old adaptive stage performed a
# separate path/ODE bowl audit for every one of hundreds of rows, which erased
# most of the intended speedup.
USER_DIRECTIONAL_RMAX_RUN_PER_ROW_BOWL_AUDIT = bool(
    globals().get("USER_DIRECTIONAL_RMAX_RUN_PER_ROW_BOWL_AUDIT", False)
)

SAVE_OUTPUTS = True


# =============================================================================
# HELPERS
# =============================================================================

REQUIRED_GLOBALS = (
    "build_straight_path",
    "evaluate_trap_path",
    "analyze_R_far",
    "classify_trajectory",
    "direction_basis",
    "potential_energy",
    "USER_M_DM_KG",
    "USER_EPS",
    "USER_T_DM_K",
)


def progress(message: str = "") -> None:
    print(message, flush=True)


def validate_environment() -> None:
    missing = [name for name in REQUIRED_GLOBALS if name not in globals()]
    if missing:
        raise RuntimeError(
            "Run the trap model and Test 1 cells first. Missing globals: "
            + ", ".join(missing)
        )


def resolve_existing_path(path: Path) -> Path:
    path = Path(path)
    candidates = [path] if path.is_absolute() else [path, RUN_DIR / path]
    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.is_file() and candidate.stat().st_size > 0:
            return candidate
    raise FileNotFoundError(
        "Could not find input file. Tried: "
        + ", ".join(str(candidate) for candidate in candidates)
    )


def read_csv_optional(path: Path) -> pd.DataFrame:
    try:
        resolved = resolve_existing_path(path)
    except FileNotFoundError:
        return pd.DataFrame()
    try:
        return pd.read_csv(resolved, low_memory=False)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def numeric(row: dict[str, Any] | pd.Series, name: str, default=np.nan) -> float:
    try:
        value = float(row.get(name, default))
    except (TypeError, ValueError):
        return float(default)
    return value if np.isfinite(value) else float(default)


def as_bool(value: Any, default: bool = False) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None:
        return default
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n", "", "none", "nan"}:
        return False
    return default


def direction_key(row: dict[str, Any] | pd.Series) -> tuple[int, int]:
    return (
        int(numeric(row, "theta_index", -1)),
        int(numeric(row, "alpha_index", -1)),
    )


def geometry_key(row: dict[str, Any] | pd.Series) -> tuple[Any, ...]:
    return (
        direction_key(row),
        round(numeric(row, "theta_rad"), 14),
        round(numeric(row, "alpha_rad"), 14),
        round(numeric(row, "b_m"), 15),
        round(numeric(row, "psi_rad", 0.0) % (2.0 * math.pi), 12),
        round(numeric(row, "m_dm_kg", float(USER_M_DM_KG)), 36),
        round(numeric(row, "eps", float(USER_EPS)), 16),
    )


def evaluate_potential_points(
    points: np.ndarray,
    mass: float,
    eps_value: float,
) -> np.ndarray:
    points = np.asarray(points, dtype=float).reshape(-1, 3)
    try:
        values = np.asarray(
            potential_energy(points, mass, eps_value, component="trap"),
            dtype=float,
        ).reshape(-1)
        if values.size != len(points):
            raise ValueError("unexpected vectorized potential shape")
        return values
    except Exception:
        return np.asarray(
            [
                float(
                    np.asarray(
                        potential_energy(
                            point,
                            mass,
                            eps_value,
                            component="trap",
                        )
                    ).reshape(-1)[0]
                )
                for point in points
            ],
            dtype=float,
        )


def endpoint_geometry(
    row: dict[str, Any] | pd.Series,
    radius_m: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    theta = numeric(row, "theta_rad")
    alpha = numeric(row, "alpha_rad")
    psi = numeric(row, "psi_rad", 0.0)
    b = numeric(row, "b_m", 0.0)
    if not radius_m > b:
        raise ValueError(f"R_max={radius_m} must exceed b={b}")
    u, b_hat = direction_basis(theta, alpha, psi)
    u = np.asarray(u, dtype=float).reshape(3)
    b_vec = b * np.asarray(b_hat, dtype=float).reshape(3)
    s = math.sqrt(max(radius_m * radius_m - b * b, 0.0))
    xyz = b_vec - s * u
    return xyz, u, b_vec, s


def endpoint_safe_mask(rows: pd.DataFrame, radius_m: float) -> tuple[np.ndarray, pd.DataFrame]:
    if len(rows) == 0:
        return np.empty(0, dtype=bool), pd.DataFrame()

    details: list[dict[str, Any]] = []
    safe = np.zeros(len(rows), dtype=bool)

    # All rows normally share one mass/eps parameter point, but group safely.
    grouped_indices: dict[tuple[float, float], list[int]] = {}
    for local_index, (_, row) in enumerate(rows.iterrows()):
        key = (
            numeric(row, "m_dm_kg", float(USER_M_DM_KG)),
            numeric(row, "eps", float(USER_EPS)),
        )
        grouped_indices.setdefault(key, []).append(local_index)

    row_list = [row for _, row in rows.iterrows()]
    for (mass, eps_value), local_indices in grouped_indices.items():
        points = []
        geometry = []
        for local_index in local_indices:
            xyz, u, b_vec, s = endpoint_geometry(row_list[local_index], radius_m)
            points.append(xyz)
            geometry.append((xyz, u, b_vec, s))
        potentials = evaluate_potential_points(np.vstack(points), mass, eps_value)

        for group_position, local_index in enumerate(local_indices):
            row = row_list[local_index]
            xyz, u, b_vec, s = geometry[group_position]
            U = float(potentials[group_position])
            v_inf = numeric(row, "v_inf_m_s")
            energy_tol = numeric(
                row,
                "energy_tol",
                float(globals().get("energy_tol", 1.0e-2)),
            )
            U_reference = float(globals().get("U_infinity", 0.0))
            K_inf = 0.5 * mass * v_inf * v_inf
            K_local = K_inf + U_reference - U
            ratio = abs(U - U_reference) / K_inf if K_inf > 0.0 else np.inf
            valid_z = True
            c_obj = globals().get("c")
            if c_obj is not None and hasattr(c_obj, "ion_height"):
                z_margin_value = float(globals().get("z_margin", 1.0e-6))
                valid_z = bool(xyz[2] > -float(c_obj.ion_height) + z_margin_value)
            is_safe = bool(
                valid_z
                and np.isfinite(U)
                and np.isfinite(K_local)
                and K_local > 0.0
                and ratio <= energy_tol
            )
            safe[local_index] = is_safe
            details.append(
                {
                    "row_local_index": local_index,
                    "trajectory_id": str(row.get("trajectory_id", "")),
                    "theta_index": direction_key(row)[0],
                    "alpha_index": direction_key(row)[1],
                    "calibration_R_max_m": float(radius_m),
                    "endpoint_U_J": U,
                    "endpoint_K_inf_J": K_inf,
                    "endpoint_K_local_J": K_local,
                    "endpoint_energy_ratio": ratio,
                    "endpoint_energy_tol": energy_tol,
                    "endpoint_valid_z": valid_z,
                    "endpoint_safe": is_safe,
                    "endpoint_x_m": float(xyz[0]),
                    "endpoint_y_m": float(xyz[1]),
                    "endpoint_z_m": float(xyz[2]),
                    "endpoint_s_m": float(s),
                }
            )

    return safe, pd.DataFrame(details)


# =============================================================================
# TARGET SELECTION AND DIRECTION CALIBRATION
# =============================================================================


def select_unresolved_rows(
    raw_all: pd.DataFrame,
    not_ready: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[tuple[int, int]]]:
    source = not_ready if len(not_ready) else raw_all
    if len(source) == 0:
        return source.copy(), source.copy(), []

    outer_status = source.get(
        "outer_status", pd.Series("", index=source.index)
    ).astype(str)
    lower_bound = source.get(
        "R_far_is_lower_bound", pd.Series(False, index=source.index)
    ).map(as_bool)
    reason = source.get(
        "test2_readiness_reason", pd.Series("", index=source.index)
    ).astype(str)
    limiting = source.get(
        "limiting_criterion", pd.Series("", index=source.index)
    ).astype(str)

    mask = (
        outer_status.eq("needs_larger_R_max")
        | lower_bound
        | reason.eq("R_far_is_lower_bound")
    )
    unresolved = source.loc[mask].copy()
    if "trajectory_id" in unresolved.columns:
        unresolved = unresolved.drop_duplicates("trajectory_id", keep="last")

    unresolved["_direction_key"] = [direction_key(row) for _, row in unresolved.iterrows()]
    counts = unresolved["_direction_key"].value_counts()

    explicit = USER_DIRECTIONAL_RMAX_DIRECTION_KEYS
    if explicit is not None:
        selected_keys = [tuple(map(int, key)) for key in explicit]
    else:
        selected_keys = [
            tuple(key)
            for key in counts.head(max(USER_DIRECTIONAL_RMAX_MAX_DIRECTIONS, 0)).index
        ]

    selected = unresolved.loc[
        unresolved["_direction_key"].isin(selected_keys)
    ].drop(columns="_direction_key")
    skipped = unresolved.loc[
        ~unresolved["_direction_key"].isin(selected_keys)
    ].drop(columns="_direction_key")

    return selected.reset_index(drop=True), skipped.reset_index(drop=True), selected_keys


def calibrate_direction_rmax(
    rows: pd.DataFrame,
    key: tuple[int, int],
) -> tuple[float, pd.DataFrame, bool]:
    attempts: list[pd.DataFrame] = []
    current_max = pd.to_numeric(rows.get("R_max_m"), errors="coerce").max()
    current_max = float(current_max) if np.isfinite(current_max) else 0.0

    ladder = [
        value
        for value in USER_DIRECTIONAL_RMAX_CALIBRATION_LADDER_M
        if value > current_max * (1.0 + 1.0e-12)
    ]
    if not ladder:
        ladder = [max(2.0 * current_max, 1.28)]

    chosen = float(ladder[-1])
    all_safe = False
    for attempt_index, radius in enumerate(ladder, start=1):
        safe, details = endpoint_safe_mask(rows, float(radius))
        details["direction_theta_index"] = int(key[0])
        details["direction_alpha_index"] = int(key[1])
        details["calibration_attempt"] = int(attempt_index)
        details["n_rows_in_direction"] = int(len(rows))
        details["n_rows_safe"] = int(np.sum(safe))
        details["all_rows_safe"] = bool(np.all(safe))
        attempts.append(details)
        progress(
            f"[Rmax calibration] direction={key} attempt={attempt_index} "
            f"Rmax={radius:.3f} m safe={int(np.sum(safe))}/{len(rows)}"
        )
        chosen = float(radius)
        if np.all(safe):
            all_safe = True
            break

    return chosen, pd.concat(attempts, ignore_index=True), all_safe


# =============================================================================
# CACHED FULL-PATH RECOVERY
# =============================================================================


def update_row_from_outer_result(
    row: pd.Series,
    outer: dict[str, Any],
    *,
    rmax_m: float,
    n_path: int,
) -> dict[str, Any]:
    output = row.to_dict()
    original_id = str(output.get("trajectory_id", ""))
    output["original_trajectory_id"] = original_id
    output["trajectory_id"] = (
        f"{original_id}|directionalRmax"
        if original_id
        else "directionalRmax"
    )
    output["adaptive_attempt"] = 1
    output["adaptive_method"] = "fixed_direction_cached_full_path"
    output["adaptive_R_max_used_m"] = float(rmax_m)
    output["adaptive_n_path"] = int(n_path)
    output["R_max_m"] = float(rmax_m)
    output.update(outer)

    R_far_m = float(outer.get("R_far_m", np.nan))
    R_far_actual_m = float(outer.get("R_far_actual_m", np.nan))
    output["R_far_um"] = R_far_m * 1.0e6 if np.isfinite(R_far_m) else np.nan
    output["R_far_actual_um"] = (
        R_far_actual_m * 1.0e6 if np.isfinite(R_far_actual_m) else np.nan
    )
    output["turning_radius_um"] = (
        float(outer.get("turning_radius_m", np.nan)) * 1.0e6
        if np.isfinite(float(outer.get("turning_radius_m", np.nan)))
        else np.nan
    )

    xyz_far = np.asarray(
        outer.get("xyz_far", np.full(3, np.nan)), dtype=float
    ).reshape(3)
    velocity_far = np.asarray(
        outer.get("velocity_far", np.full(3, np.nan)), dtype=float
    ).reshape(3)
    output["x_far_m"], output["y_far_m"], output["z_far_m"] = xyz_far
    output["vx_far_m_s"], output["vy_far_m_s"], output["vz_far_m_s"] = velocity_far

    if outer.get("outer_status") in {
        "straight_safe_to_full_sphere",
        "straight_then_dm_only",
    }:
        output["status"] = classify_trajectory(
            outer_result=outer,
            v_inf=numeric(row, "v_inf_m_s"),
            v_min=numeric(row, "v_min_m_s"),
            nominal_intersects_full_sphere=as_bool(
                row.get("nominal_intersects_R_full", True), True
            ),
            nominal_path_to_full_is_valid=as_bool(
                row.get("nominal_path_to_R_full_is_valid", True), True
            ),
        )
    else:
        output["status"] = str(outer.get("outer_status", "needs_larger_R_max"))

    return output


def recover_geometry_group(
    group_records: list[dict[str, Any]],
    rmax_m: float,
) -> list[dict[str, Any]]:
    group = pd.DataFrame(group_records)
    representative = group.iloc[0]
    b = numeric(representative, "b_m", 0.0)
    min_r_full = pd.to_numeric(group.get("R_full_m"), errors="coerce").min()
    if not np.isfinite(min_r_full) or min_r_full <= 0.0:
        min_r_full = numeric(representative, "R_full_speed_only_m", 1.0e-6)

    s_grid, xyz, u, b_vec = build_straight_path(
        theta=numeric(representative, "theta_rad"),
        alpha=numeric(representative, "alpha_rad"),
        b=b,
        psi=numeric(representative, "psi_rad", 0.0),
        R_inner=float(min_r_full),
        R_outer=float(rmax_m),
        n_path=int(USER_DIRECTIONAL_RMAX_N_PATH),
    )
    mass = numeric(representative, "m_dm_kg", float(USER_M_DM_KG))
    eps_value = numeric(representative, "eps", float(USER_EPS))
    U, F, valid = evaluate_trap_path(
        xyz=xyz,
        mass=mass,
        charge_fraction=eps_value,
    )
    radial = np.linalg.norm(np.asarray(xyz, dtype=float), axis=1)

    outputs: list[dict[str, Any]] = []
    for _, row in group.iterrows():
        r_full = numeric(row, "R_full_m", min_r_full)
        eligible = np.flatnonzero(radial >= r_full * (1.0 - 1.0e-12))
        start = int(eligible[0]) if eligible.size else 0
        outer = analyze_R_far(
            s_grid=np.asarray(s_grid[start:], dtype=float),
            xyz=np.asarray(xyz[start:], dtype=float),
            u=np.asarray(u, dtype=float),
            U=np.asarray(U[start:], dtype=float),
            F=np.asarray(F[start:], dtype=float),
            valid=np.asarray(valid[start:], dtype=bool),
            mass=mass,
            v_inf=numeric(row, "v_inf_m_s"),
            R_full=r_full,
            R_max=float(rmax_m),
            energy_tol=numeric(
                row,
                "energy_tol",
                float(globals().get("energy_tol", 1.0e-2)),
            ),
            angle_tol=numeric(
                row,
                "angle_tol",
                float(globals().get("angle_tol", 1.0e-3)),
            ),
            displacement_tol=numeric(
                row,
                "displacement_tol_m",
                float(globals().get("displacement_tol", 1.0e-6)),
            ),
            U_reference=float(globals().get("U_infinity", 0.0)),
        )
        outputs.append(
            update_row_from_outer_result(
                row,
                outer,
                rmax_m=rmax_m,
                n_path=USER_DIRECTIONAL_RMAX_N_PATH,
            )
        )
    return outputs


def run_cached_recovery(
    selected: pd.DataFrame,
    direction_rmax: dict[tuple[int, int], float],
) -> pd.DataFrame:
    if len(selected) == 0:
        return pd.DataFrame()

    groups: dict[tuple[Any, ...], list[dict[str, Any]]] = {}
    for _, row in selected.iterrows():
        groups.setdefault(geometry_key(row), []).append(row.to_dict())

    progress(
        f"Cached recovery geometries: {len(groups):,} for {len(selected):,} rows"
    )
    completed: list[dict[str, Any]] = []
    total = len(groups)

    with ThreadPoolExecutor(max_workers=max(USER_DIRECTIONAL_RMAX_THREADS, 1)) as executor:
        future_map = {}
        for key, records in groups.items():
            dkey = tuple(key[0])
            rmax = direction_rmax[dkey]
            future = executor.submit(recover_geometry_group, records, rmax)
            future_map[future] = key

        for index, future in enumerate(as_completed(future_map), start=1):
            key = future_map[future]
            try:
                completed.extend(future.result())
            except Exception as exc:
                for record in groups[key]:
                    output = dict(record)
                    output["original_trajectory_id"] = str(
                        output.get("trajectory_id", "")
                    )
                    output["adaptive_method"] = "fixed_direction_cached_full_path"
                    output["adaptive_exception"] = repr(exc)
                    output["outer_status"] = "adaptive_cached_path_exception"
                    output["status"] = "adaptive_cached_path_exception"
                    output["R_far_is_lower_bound"] = True
                    completed.append(output)
            if (
                index == 1
                or index == total
                or index % max(USER_DIRECTIONAL_RMAX_PRINT_EVERY, 1) == 0
            ):
                progress(f"[cached Rmax] geometry {index:,}/{total:,} complete")

    return pd.DataFrame(completed)


# =============================================================================
# MAIN
# =============================================================================


def main() -> None:
    validate_environment()
    raw_all = read_csv_optional(TEST1_RAW_ALL_CSV)
    not_ready = read_csv_optional(TEST1_NOT_READY_CSV)
    original_ready = read_csv_optional(TEST1_READY_INPUT_CSV)
    if len(raw_all) == 0 and len(not_ready) == 0:
        raise RuntimeError("No Test 1 raw/not-ready table is available.")
    if len(raw_all) == 0:
        raw_all = not_ready.copy()

    selected, skipped_unresolved, selected_keys = select_unresolved_rows(
        raw_all, not_ready
    )
    progress(f"Unresolved R_max rows found: {len(selected) + len(skipped_unresolved):,}")
    progress(f"Selected direction keys: {selected_keys}")
    progress(f"Rows selected for fast recovery: {len(selected):,}")
    if len(skipped_unresolved):
        progress(
            f"Unresolved rows outside selected directions: {len(skipped_unresolved):,}"
        )

    calibration_tables: list[pd.DataFrame] = []
    direction_rmax: dict[tuple[int, int], float] = {}
    direction_all_safe: dict[tuple[int, int], bool] = {}

    for key in selected_keys:
        group = selected.loc[
            [direction_key(row) == key for _, row in selected.iterrows()]
        ].copy()
        chosen, attempts, all_safe = calibrate_direction_rmax(group, key)
        direction_rmax[key] = chosen
        direction_all_safe[key] = all_safe
        calibration_tables.append(attempts)
        progress(
            f"Direction {key}: assigned R_max={chosen:.3f} m; "
            f"endpoint criterion all-safe={all_safe}"
        )

    final_df = run_cached_recovery(selected, direction_rmax)
    attempts_df = (
        pd.concat(calibration_tables, ignore_index=True)
        if calibration_tables
        else pd.DataFrame()
    )

    if len(final_df):
        resolved_mask = (
            final_df.get(
                "outer_status", pd.Series("", index=final_df.index)
            ).astype(str).isin(
                {"straight_safe_to_full_sphere", "straight_then_dm_only"}
            )
            & ~final_df.get(
                "R_far_is_lower_bound", pd.Series(True, index=final_df.index)
            ).map(as_bool)
        )
        state_columns = [
            "x_far_m",
            "y_far_m",
            "z_far_m",
            "vx_far_m_s",
            "vy_far_m_s",
            "vz_far_m_s",
        ]
        finite_state = np.ones(len(final_df), dtype=bool)
        for column in state_columns:
            finite_state &= np.isfinite(
                pd.to_numeric(final_df.get(column), errors="coerce")
            )
        resolved_mask &= finite_state
        resolved_df = final_df.loc[resolved_mask].copy()
        unresolved_df = final_df.loc[~resolved_mask].copy()
        resolved_df["test2_ready"] = True
        resolved_df["test2_readiness_reason"] = "directional_R_max_resolved"
        resolved_df["test2_state_model"] = (
            "trap-only cached large-Rmax path; energy-corrected speed; nominal direction"
        )
    else:
        resolved_df = pd.DataFrame()
        unresolved_df = pd.DataFrame()

    if len(skipped_unresolved):
        skipped_unresolved = skipped_unresolved.copy()
        skipped_unresolved["adaptive_skip_reason"] = (
            "outside_selected_direction_keys"
        )
        unresolved_df = pd.concat(
            [unresolved_df, skipped_unresolved], ignore_index=True, sort=False
        )

    if SAVE_OUTPUTS:
        OUTPUT_PREFIX.parent.mkdir(parents=True, exist_ok=True)
        attempts_df.to_csv(ATTEMPTS_CSV, index=False)
        final_df.to_csv(FINAL_RESCAN_CSV, index=False)
        resolved_df.to_csv(RESOLVED_TEST2_INPUT_CSV, index=False)

        merged_ready = pd.concat(
            [original_ready, resolved_df], ignore_index=True, sort=False
        )
        if "trajectory_id" in merged_ready.columns:
            merged_ready["trajectory_id"] = merged_ready["trajectory_id"].astype(str)
            merged_ready = merged_ready.drop_duplicates(
                "trajectory_id", keep="last"
            )
        merged_ready.to_csv(MERGED_TEST1_INPUT_CSV, index=False)
        unresolved_df.to_csv(STILL_UNRESOLVED_CSV, index=False)

        # Header-safe empty diagnostics. The focused escape/bowl scan can be run
        # separately later without blocking the production pipeline.
        pd.DataFrame().to_csv(ESCAPE_SCAN_CSV, index=False)
        pd.DataFrame().to_csv(ESCAPE_FOCUS_PLAN_CSV, index=False)
        pd.DataFrame().to_csv(ESCAPE_FOCUS_TEST2_INPUT_CSV, index=False)
        pd.DataFrame().to_csv(ESCAPE_FOCUS_UNRESOLVED_CSV, index=False)
        pd.DataFrame().to_csv(BOWL_AUDIT_CSV, index=False)

        summary = {
            "mode": "fast_two_direction_cached_Rmax_recovery",
            "selected_direction_keys": [list(key) for key in selected_keys],
            "direction_Rmax_m": {
                f"theta{key[0]}_alpha{key[1]}": value
                for key, value in direction_rmax.items()
            },
            "direction_endpoint_all_safe": {
                f"theta{key[0]}_alpha{key[1]}": value
                for key, value in direction_all_safe.items()
            },
            "n_selected_rows": int(len(selected)),
            "n_unique_cached_geometries": int(
                len({geometry_key(row) for _, row in selected.iterrows()})
            ),
            "n_resolved_rows": int(len(resolved_df)),
            "n_still_unresolved_rows": int(len(unresolved_df)),
            "n_original_ready_rows": int(len(original_ready)),
            "n_merged_ready_rows": int(len(merged_ready)),
            "n_path": int(USER_DIRECTIONAL_RMAX_N_PATH),
            "threads": int(USER_DIRECTIONAL_RMAX_THREADS),
            "note": (
                "The expensive ladder and per-row bowl audit were skipped. "
                "One large Rmax was calibrated per selected direction, and "
                "each U/F path was cached across speed rows."
            ),
        }
        SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    progress("\nFast directional R_max recovery complete")
    progress(f"  selected rows            : {len(selected):,}")
    progress(f"  cached geometries        : {len({geometry_key(row) for _, row in selected.iterrows()}):,}")
    progress(f"  newly resolved rows      : {len(resolved_df):,}")
    progress(f"  still unresolved rows    : {len(unresolved_df):,}")
    progress(f"  merged Test 1.5 input    : {MERGED_TEST1_INPUT_CSV}")


if __name__ == "__main__":
    main()


# Targeted Axial Escape-Channel Resolution

This stage replaces the coarse first positive-weight impact annulus in the selected axial parent direction cell by finite solid-angle subcells, a dense speed quadrature, and a $b=5\,\mu\mathrm{m}$ seed. The original integrated probability weight is preserved; Test 2.5 determines the actual finite detectable support.


In [ ]:
#!/usr/bin/env python3
"""Finite-measure targeted refinement of the observed axial low-b branch.

The original Test 1 uses one equal-area midpoint for its first positive-weight
inner annulus. For R_full around 90--130 um that midpoint is roughly 25--38 um,
so it can miss the observed b=5 um focusing branch. This stage replaces, rather
than overlaps, that parent region for the selected axial direction cell.

For each selected channel it:
  1. removes the original positive-weight first-annulus rows in the parent
     direction cell and relevant Test-0.75 speed interval;
  2. subdivides the parent direction cell uniformly in mu=cos(theta) and alpha;
  3. constructs an exact Maxwell-probability quadrature with dense nodes in the
     configured 180--240 m/s window;
  4. creates one positive-weight b=5 um seed per speed/angular/psi subcell, with
     the full first-annulus bounds retained as physical support for Test 2.5;
  5. reruns the Test-1 R_far calculation for those rows; and
  6. writes an augmented Test-1.5 input without double counting the replaced
     parent measure.
"""

from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from pathlib import Path
import math
import numpy as np
import pandas as pd
from scipy.stats import maxwell as _target_maxwell

TARGETED_ESCAPE_OUTPUT_PREFIX = str(
    Path(RUN_DIRECTORY) / "targeted_escape_resolution_v12"
)
TARGETED_ESCAPE_RAW_CSV = Path(
    f"{TARGETED_ESCAPE_OUTPUT_PREFIX}_raw_rows.csv"
)
TARGETED_ESCAPE_READY_CSV = Path(
    f"{TARGETED_ESCAPE_OUTPUT_PREFIX}_test1_ready_rows.csv"
)
TARGETED_ESCAPE_REPLACED_BASE_CSV = Path(
    f"{TARGETED_ESCAPE_OUTPUT_PREFIX}_replaced_base_rows.csv"
)
TARGETED_ESCAPE_AUGMENTED_INPUT_CSV = Path(
    f"{TARGETED_ESCAPE_OUTPUT_PREFIX}_augmented_test1_test2_input.csv"
)
TARGETED_ESCAPE_SUMMARY_CSV = Path(
    f"{TARGETED_ESCAPE_OUTPUT_PREFIX}_summary.csv"
)


def _target_progress(message: str = "") -> None:
    print(message, flush=True)


def _target_input_csv() -> Path:
    candidates = []
    adaptive = globals().get("MERGED_TEST1_INPUT_CSV")
    if adaptive is not None:
        candidates.append(Path(adaptive))
    candidates.extend(
        [
            Path(RUN_DIRECTORY)
            / "escape_channel_adaptive_rmax_bowl_v2_merged_test1_test2_input.csv",
            Path(f"{TEST1_OUTPUT_PREFIX}_test2_input.csv"),
        ]
    )
    for path in candidates:
        if path.exists() and path.stat().st_size > 0:
            return path
    raise FileNotFoundError(
        "No Test-1-ready input was found for targeted escape refinement"
    )


def _target_bool(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def _target_speed_tag(value: float) -> str:
    return f"{float(value):.6f}".replace("-", "m").replace(".", "p")


def _target_maxwell_probability(lower: float, upper: float) -> float:
    scale = math.sqrt(float(k_B) * float(T_dm) / float(m_dm))
    return float(
        _target_maxwell.cdf(float(upper), scale=scale)
        - _target_maxwell.cdf(float(lower), scale=scale)
    )


def _target_speed_cells(policy: list[dict]) -> list[dict]:
    window_lo, window_hi = map(float, USER_TARGETED_ESCAPE_SPEED_WINDOW_M_S)
    target_nodes = [
        float(value)
        for value in USER_TARGETED_ESCAPE_SPEED_ANCHORS_M_S
        if np.isfinite(value)
    ]
    policy_frame = pd.DataFrame(policy).copy()
    cells: list[dict] = []

    for interval_label, interval_group in policy_frame.groupby(
        "speed_interval_label", sort=False
    ):
        interval_lo = float(interval_group["speed_interval_lower_m_s"].iloc[0])
        interval_hi = float(interval_group["speed_interval_upper_m_s"].iloc[0])
        if min(interval_hi, window_hi) <= max(interval_lo, window_lo):
            continue

        original_nodes = pd.to_numeric(
            interval_group["speed_m_s"], errors="coerce"
        ).to_numpy(float)
        anchors = [
            value
            for value in np.concatenate(
                [original_nodes, np.asarray(target_nodes, dtype=float)]
            )
            if np.isfinite(value) and interval_lo < value < interval_hi
        ]
        if interval_lo < window_lo and not any(value < window_lo for value in anchors):
            anchors.append(0.5 * (interval_lo + window_lo))
        if interval_hi > window_hi and not any(value > window_hi for value in anchors):
            anchors.append(0.5 * (window_hi + interval_hi))
        anchors = sorted(set(round(float(value), 10) for value in anchors))
        if not anchors:
            anchors = [0.5 * (interval_lo + interval_hi)]

        edges = np.empty(len(anchors) + 1, dtype=float)
        edges[0] = interval_lo
        edges[-1] = interval_hi
        if len(anchors) > 1:
            edges[1:-1] = 0.5 * (
                np.asarray(anchors[:-1], dtype=float)
                + np.asarray(anchors[1:], dtype=float)
            )

        parent_probability = _target_maxwell_probability(interval_lo, interval_hi)
        total_allowed_probability = float(
            interval_group["total_allowed_speed_probability"].iloc[0]
        )
        nearest_rows = interval_group.iloc[
            np.argmin(
                np.abs(
                    original_nodes[:, None]
                    - np.asarray(anchors, dtype=float)[None, :]
                ),
                axis=0,
            )
        ].reset_index(drop=True)

        for index, (speed, lower, upper) in enumerate(
            zip(anchors, edges[:-1], edges[1:])
        ):
            probability = _target_maxwell_probability(lower, upper)
            template = nearest_rows.iloc[index].to_dict()
            radii = rutherford_v_b_radius_policy(
                speed,
                0.0,
                m_dm_kg=float(m_dm),
                m_ion_kg=float(m_ion),
                eps_value=float(eps),
                ion_charge_number=float(Z_ion),
                coulomb_constant=float(K),
                elementary_charge_c=float(e),
                threshold_j=float(USER_TARGET_ION_ENERGY_J),
            )
            speed_case = dict(template)
            speed_case.update(
                {
                    "key": (
                        f"escape_refine_{interval_label}_v"
                        f"{_target_speed_tag(speed)}"
                    ),
                    "speed_m_s": float(speed),
                    "v_inf_m_s": float(speed),
                    "quantile": float(
                        _target_maxwell.cdf(
                            speed,
                            scale=math.sqrt(k_B * T_dm / m_dm),
                        )
                    ),
                    "speed_interval_label": str(interval_label),
                    "speed_interval_lower_m_s": float(lower),
                    "speed_interval_upper_m_s": float(upper),
                    "speed_cell_lower_m_s": float(lower),
                    "speed_cell_upper_m_s": float(upper),
                    "speed_interval_probability": float(parent_probability),
                    "conditional_speed_weight": (
                        probability / total_allowed_probability
                        if total_allowed_probability > 0.0
                        else 0.0
                    ),
                    "unconditional_speed_weight": float(probability),
                    "speed_cell_probability": float(probability),
                    "total_allowed_speed_probability": total_allowed_probability,
                    "R_full_m": float(radii["r_min_energy_threshold_m"]),
                    "R_full_um": float(radii["r_min_energy_threshold_m"]) * 1.0e6,
                    "R_switch_factor": float(USER_R_SWITCH_FACTOR),
                    "R_switch_m": (
                        float(USER_R_SWITCH_FACTOR)
                        * float(radii["r_min_energy_threshold_m"])
                    ),
                    "R_switch_um": (
                        float(USER_R_SWITCH_FACTOR)
                        * float(radii["r_min_energy_threshold_m"])
                        * 1.0e6
                    ),
                    "targeted_escape_speed_cell": True,
                }
            )
            cells.append(speed_case)

    return cells


def _target_parent_direction(channel: str) -> dict:
    channel = str(channel).strip().lower()
    if channel == "minus_z":
        theta_center = 0.5 * math.pi
        alpha_center = 1.5 * math.pi
    elif channel == "plus_z":
        theta_center = 0.5 * math.pi
        alpha_center = 0.5 * math.pi
    else:
        raise ValueError(
            "USER_TARGETED_ESCAPE_CHANNELS entries must be 'minus_z' or 'plus_z'"
        )

    theta_grid = np.linspace(0.0, math.pi, int(n_theta))
    alpha_grid = np.linspace(0.0, 2.0 * math.pi, int(n_alpha), endpoint=False)
    theta_index = int(np.argmin(np.abs(theta_grid - theta_center)))
    alpha_delta = 2.0 * math.pi / int(n_alpha)
    alpha_distance = np.abs(
        (alpha_grid - alpha_center + math.pi) % (2.0 * math.pi) - math.pi
    )
    alpha_index = int(np.argmin(alpha_distance))

    theta_low = (
        0.0
        if theta_index == 0
        else 0.5 * (theta_grid[theta_index - 1] + theta_grid[theta_index])
    )
    theta_high = (
        math.pi
        if theta_index == len(theta_grid) - 1
        else 0.5 * (theta_grid[theta_index] + theta_grid[theta_index + 1])
    )
    alpha_low = alpha_grid[alpha_index] - 0.5 * alpha_delta
    alpha_high = alpha_grid[alpha_index] + 0.5 * alpha_delta
    return {
        "channel": channel,
        "theta_index": theta_index,
        "alpha_index": alpha_index,
        "theta_center": float(theta_grid[theta_index]),
        "alpha_center": float(alpha_grid[alpha_index]),
        "theta_low": float(theta_low),
        "theta_high": float(theta_high),
        "alpha_low": float(alpha_low),
        "alpha_high": float(alpha_high),
    }


def _target_direction_subcells(parent: dict, channel_number: int) -> list[dict]:
    n_mu = int(USER_TARGETED_ESCAPE_N_MU_SUBCELLS)
    n_alpha_local = int(USER_TARGETED_ESCAPE_N_ALPHA_SUBCELLS)
    if n_mu < 1 or n_alpha_local < 1:
        raise ValueError("Targeted angular subdivision counts must be positive")

    mu_low = math.cos(float(parent["theta_high"]))
    mu_high = math.cos(float(parent["theta_low"]))
    mu_edges = np.linspace(mu_low, mu_high, n_mu + 1)
    alpha_edges = np.linspace(
        float(parent["alpha_low"]),
        float(parent["alpha_high"]),
        n_alpha_local + 1,
    )
    cells: list[dict] = []
    for mu_index in range(n_mu):
        for alpha_index_local in range(n_alpha_local):
            mu_a = float(mu_edges[mu_index])
            mu_b = float(mu_edges[mu_index + 1])
            alpha_a = float(alpha_edges[alpha_index_local])
            alpha_b = float(alpha_edges[alpha_index_local + 1])
            mu_mid = 0.5 * (mu_a + mu_b)
            alpha_mid = 0.5 * (alpha_a + alpha_b)
            weight = (mu_b - mu_a) * (alpha_b - alpha_a) / (4.0 * math.pi)
            local_id = mu_index * n_alpha_local + alpha_index_local
            cells.append(
                {
                    "channel": str(parent["channel"]),
                    "theta_index": 10000 + channel_number * 100 + mu_index,
                    "alpha_index": 10000 + channel_number * 100 + alpha_index_local,
                    "theta_rad": float(math.acos(np.clip(mu_mid, -1.0, 1.0))),
                    "alpha_rad": float(alpha_mid % (2.0 * math.pi)),
                    "theta_cell_mu_lower": mu_a,
                    "theta_cell_mu_upper": mu_b,
                    "alpha_cell_lower_rad": alpha_a % (2.0 * math.pi),
                    "alpha_cell_upper_rad": alpha_b % (2.0 * math.pi),
                    "direction_solid_angle_weight_full_sphere": float(weight),
                    "direction_solid_angle_weight": float(weight),
                    "targeted_direction_cell_id": (
                        f"{parent['channel']}_mu{mu_index:02d}_al{alpha_index_local:02d}"
                    ),
                    "targeted_direction_local_id": int(local_id),
                }
            )
    return cells


def _target_replacement_mask(base: pd.DataFrame, parents: list[dict], interval_labels: set[str]) -> pd.Series:
    mask = pd.Series(False, index=base.index)
    positive = pd.to_numeric(base.get("impact_weight", 0.0), errors="coerce").fillna(0.0) > 0.0
    lower = pd.to_numeric(base.get("b_lower_m", np.nan), errors="coerce")
    first_annulus = np.isfinite(lower) & np.isclose(lower, 0.0, rtol=0.0, atol=1.0e-15)
    interval_match = base.get("speed_interval_label", pd.Series("", index=base.index)).astype(str).isin(interval_labels)
    for parent in parents:
        direction_match = (
            pd.to_numeric(base["theta_index"], errors="coerce").eq(int(parent["theta_index"]))
            & pd.to_numeric(base["alpha_index"], errors="coerce").eq(int(parent["alpha_index"]))
        )
        mask |= positive & first_annulus & interval_match & direction_match
    return mask


def _target_evaluate_one(payload: tuple) -> dict:
    (
        speed_case,
        direction,
        psi_index,
        psi_value,
        impact_case_id,
    ) = payload
    r_full_speed_only = float(speed_case["R_full_m"])
    parent_b_upper = r_full_speed_only / math.sqrt(float(n_b_inner))
    b_seed = float(USER_TARGETED_ESCAPE_B_SEED_M)
    if not 0.0 <= b_seed < parent_b_upper:
        raise ValueError(
            f"Target b={b_seed*1e6:.6f} um is outside the first annulus "
            f"[0, {parent_b_upper*1e6:.6f}] um at v={speed_case['speed_m_s']:.6f} m/s"
        )
    impact_case = {
        "impact_case_id": int(impact_case_id),
        "b_m": b_seed,
        "psi_rad": float(psi_value),
        "impact_weight": (
            parent_b_upper * parent_b_upper / (float(b_max) ** 2)
            / int(USER_TARGETED_ESCAPE_N_PSI)
        ),
        "impact_sample_type": "targeted_escape_first_annulus_seed",
        "b_region": "intersects_R_full",
        "b_lower_m": 0.0,
        "b_upper_m": float(parent_b_upper),
        "b_index": 0,
        "b_region_index": 0,
        "psi_index": int(psi_index),
    }
    rows = evaluate_theta_alpha_impact_task(
        int(direction["theta_index"]),
        int(direction["alpha_index"]),
        float(direction["theta_rad"]),
        float(direction["alpha_rad"]),
        impact_case,
        float(m_dm),
        float(eps),
        float(T_dm),
        r_full_speed_only,
        float(USER_TARGETED_ESCAPE_RMAX_M),
        int(USER_TARGETED_ESCAPE_N_PATH),
        float(energy_tol),
        float(angle_tol),
        float(displacement_tol),
        [speed_case],
    )
    row = _attach_speed_policy_columns(pd.DataFrame(rows), speed_case).iloc[0].to_dict()
    row.update(direction)
    row.update(
        {
            "targeted_escape_refinement": True,
            "targeted_escape_channel": str(direction["channel"]),
            "targeted_escape_parent_theta_index": int(direction.get("parent_theta_index", -1)),
            "targeted_escape_parent_alpha_index": int(direction.get("parent_alpha_index", -1)),
            "targeted_escape_b_seed_m": b_seed,
            "targeted_escape_b_seed_um": b_seed * 1.0e6,
            "targeted_escape_parent_b_upper_m": float(parent_b_upper),
            "targeted_escape_parent_b_upper_um": float(parent_b_upper) * 1.0e6,
            "force_test2_selection": bool(USER_TARGETED_ESCAPE_FORCE_PILOT_SELECTION),
            "trajectory_regime": "resonant",
            "collision_energy_regime": "resonant",
            "ion_energy_model": "full_trap_required_targeted_escape_refinement",
            "test0p75_adiabatic_reject": False,
            "requires_dm_only_reach_screen": True,
            "use_rutherford_after_reach": False,
            "requires_full_coupled_after_reach": True,
            "use_rutherford_final_energy": False,
            "requires_full_trap_propagation": True,
        }
    )
    row["trajectory_id"] = (
        f"{speed_case['key']}|{direction['targeted_direction_cell_id']}|"
        f"psi{psi_index:02d}|bseed{_target_speed_tag(b_seed*1e6)}um"
    )
    return row


def run_targeted_escape_resolution() -> pd.DataFrame:
    base_path = _target_input_csv()
    base = pd.read_csv(base_path, low_memory=False)
    validate_parameter_dataframe(base, "Targeted escape base Test-1 input")
    base["trajectory_id"] = base["trajectory_id"].astype(str)

    policy = load_test0p75_speed_policy(Path(TEST0P75_SPEED_POLICY_CSV))
    speed_cells = _target_speed_cells(policy)
    if not speed_cells:
        _target_progress(
            "Targeted escape refinement skipped: configured speed window does not overlap a retained Test-0.75 interval."
        )
        base.to_csv(TARGETED_ESCAPE_AUGMENTED_INPUT_CSV, index=False)
        globals()["USER_TEST1P5_INPUT_CSV"] = TARGETED_ESCAPE_AUGMENTED_INPUT_CSV
        return base

    interval_labels = {str(row["speed_interval_label"]) for row in speed_cells}
    parents = [
        _target_parent_direction(channel)
        for channel in USER_TARGETED_ESCAPE_CHANNELS
    ]
    replacement_mask = _target_replacement_mask(base, parents, interval_labels)
    replaced = base.loc[replacement_mask].copy()
    retained = base.loc[~replacement_mask].copy()
    replaced.to_csv(TARGETED_ESCAPE_REPLACED_BASE_CSV, index=False)

    direction_cells: list[dict] = []
    for channel_number, parent in enumerate(parents):
        cells = _target_direction_subcells(parent, channel_number)
        for cell in cells:
            cell["parent_theta_index"] = int(parent["theta_index"])
            cell["parent_alpha_index"] = int(parent["alpha_index"])
        direction_cells.extend(cells)

    psi_values = np.linspace(
        0.0,
        2.0 * math.pi,
        int(USER_TARGETED_ESCAPE_N_PSI),
        endpoint=False,
    )
    payloads = []
    impact_case_id = 900000
    for speed_case in speed_cells:
        for direction in direction_cells:
            for psi_index, psi_value in enumerate(psi_values):
                payloads.append(
                    (
                        speed_case,
                        direction,
                        int(psi_index),
                        float(psi_value),
                        int(impact_case_id),
                    )
                )
                impact_case_id += 1

    _target_progress("\nTargeted axial escape-channel refinement")
    _target_progress(f"  base input: {base_path}")
    _target_progress(f"  replaced coarse positive rows: {len(replaced):,}")
    _target_progress(f"  speed cells: {len(speed_cells):,}")
    _target_progress(f"  angular subcells: {len(direction_cells):,}")
    _target_progress(f"  psi values: {len(psi_values):,}")
    _target_progress(f"  targeted Test-1 rows: {len(payloads):,}")

    rows: list[dict] = []
    completed = 0
    payload_iterator = iter(payloads)
    with ThreadPoolExecutor(
        max_workers=int(USER_TARGETED_ESCAPE_THREADS)
    ) as executor:
        in_flight = {}

        def submit_one() -> bool:
            try:
                payload = next(payload_iterator)
            except StopIteration:
                return False
            future = executor.submit(_target_evaluate_one, payload)
            in_flight[future] = payload
            return True

        for _ in range(
            min(int(USER_TARGETED_ESCAPE_MAX_IN_FLIGHT), len(payloads))
        ):
            if not submit_one():
                break

        while in_flight:
            done, _ = wait(in_flight, return_when=FIRST_COMPLETED)
            for future in done:
                payload = in_flight.pop(future)
                rows.append(future.result())
                completed += 1
                if (
                    completed == 1
                    or completed % int(USER_TARGETED_ESCAPE_PRINT_EVERY) == 0
                    or completed == len(payloads)
                ):
                    _target_progress(
                        f"  targeted Test 1: {completed:,}/{len(payloads):,} complete"
                    )
                submit_one()

    raw = pd.DataFrame(rows)
    raw.to_csv(TARGETED_ESCAPE_RAW_CSV, index=False)
    ready, not_ready = build_test2_input_table(
        raw,
        include_zero_weight_diagnostics=True,
    )
    ready.to_csv(TARGETED_ESCAPE_READY_CSV, index=False)

    augmented = pd.concat([retained, ready], ignore_index=True, sort=False)
    augmented["trajectory_id"] = augmented["trajectory_id"].astype(str)
    augmented = augmented.drop_duplicates("trajectory_id", keep="last")
    augmented.to_csv(TARGETED_ESCAPE_AUGMENTED_INPUT_CSV, index=False)

    # Test 1.5 checks this explicit path before adaptive and standard inputs.
    globals()["USER_TEST1P5_INPUT_CSV"] = TARGETED_ESCAPE_AUGMENTED_INPUT_CSV

    summary = pd.DataFrame(
        [
            {
                "base_input_csv": str(base_path),
                "augmented_input_csv": str(TARGETED_ESCAPE_AUGMENTED_INPUT_CSV),
                "n_base_rows": int(len(base)),
                "n_replaced_base_rows": int(len(replaced)),
                "n_targeted_raw_rows": int(len(raw)),
                "n_targeted_ready_rows": int(len(ready)),
                "n_targeted_not_ready_rows": int(len(not_ready)),
                "n_augmented_rows": int(len(augmented)),
                "target_b_seed_um": float(USER_TARGETED_ESCAPE_B_SEED_M) * 1.0e6,
                "target_speed_window_lower_m_s": float(USER_TARGETED_ESCAPE_SPEED_WINDOW_M_S[0]),
                "target_speed_window_upper_m_s": float(USER_TARGETED_ESCAPE_SPEED_WINDOW_M_S[1]),
                "target_channels": ";".join(map(str, USER_TARGETED_ESCAPE_CHANNELS)),
            }
        ]
    )
    summary.to_csv(TARGETED_ESCAPE_SUMMARY_CSV, index=False)

    _target_progress(f"  targeted ready rows: {len(ready):,}")
    _target_progress(f"  targeted not-ready rows: {len(not_ready):,}")
    _target_progress(f"  augmented Test 1.5 input: {TARGETED_ESCAPE_AUGMENTED_INPUT_CSV}")
    return augmented


if bool(globals().get("USER_RUN_TARGETED_ESCAPE_REFINEMENT", True)) and bool(
    globals().get("USER_ENABLE_TARGETED_ESCAPE_REFINEMENT", True)
):
    TARGETED_ESCAPE_AUGMENTED_DF = run_targeted_escape_resolution()
else:
    TARGETED_ESCAPE_AUGMENTED_DF = pd.DataFrame()


# Test 1.5 — Escape-Aware Physical-$b$ Pilot

The input precedence is: an explicitly configured input, then the adaptive merged Test 1 table, then the ordinary Test 1 ready table. The outward-force certificate remains conservative; empty-support results are handled as valid header-only outputs.


In [ ]:
#!/usr/bin/env python3
"""Standalone Test 1.5 v13: physical-b pilot and trajectory-class plan.

Run directly in Jupyter:

    %run -i 01_test1p5_escape_aware_v13.py

This file contains the complete Test 1.5 logic. It does not import, patch,
execute, or read another Python script. It reads only the upstream Test 1 CSV.

Production representatives are sampled from the collision-energy impact-
parameter support exported by Test 1 through ``b_lower_m`` and ``b_upper_m``.
Sampling is stratified in annular area, i.e. in ``b^2``. Before sampling,
this file runs the conservative outward-force/stopping-distance certificate.
Only rows proven to turn before ``R_full`` are rejected. Test 2 then performs a
separate DM-only reach search inside every surviving trajectory class before any
full collision propagation. The switch radius controls the solver handoff only;
it does not replace the collision-regime impact-parameter support.
"""

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd


# =============================================================================
# USER SETTINGS
# =============================================================================

# Input precedence:
# 1. USER_TEST1P5_INPUT_CSV when explicitly supplied;
# 2. the adaptive-R_max merged Test 1 input when it exists;
# 3. the ordinary Test 1 Test-2-ready input.
_USER_TEST1P5_INPUT = globals().get("USER_TEST1P5_INPUT_CSV", None)
TEST1_INPUT_CSV: Path | None = (
    Path(_USER_TEST1P5_INPUT) if _USER_TEST1P5_INPUT not in (None, "") else None
)
TARGETED_ESCAPE_TEST1_INPUT_CSV = Path(RUN_DIRECTORY) / (
    "targeted_escape_resolution_v12_augmented_test1_test2_input.csv"
)
ADAPTIVE_MERGED_TEST1_INPUT_CSV = Path(RUN_DIRECTORY) / (
    "escape_channel_adaptive_rmax_bowl_v2_merged_test1_test2_input.csv"
)
STANDARD_TEST1_INPUT_CSV = Path(f"{TEST1_OUTPUT_PREFIX}_test2_input.csv")
TEST1_INPUT_PATTERNS = (
    "escape_channel_adaptive_rmax_bowl_v2_merged_test1_test2_input.csv",
    "test1_reach_aware_test2_input.csv",
    "test1_reach_aware_v11_test2_input.csv",
    "test1*_test2_input.csv",
)
OUTPUT_PREFIX = TEST1P5_OUTPUT_PREFIX

# Pilot stratification. For the default six speeds, this gives at most
# 6 x 4 x 4 x 2 x 2 x 2 trajectory-regime groups before empty strata/deduping.
N_MU_BINS = 4
N_ALPHA_BINS = 4
N_PSI_BINS = 2

# Nonuniform cumulative-area bins over the exact union of Test 0.75 annuli.
# The first four bins occupy only the lowest 15% of physical b^2 support,
# deliberately allocating much more simulation resolution near the lower bound.
PHYSICAL_B2_BIN_EDGES = (
    0.0,
    0.005,
    0.020,
    0.060,
    0.150,
    0.400,
    1.0,
)
N_B2_BINS = len(PHYSICAL_B2_BIN_EDGES) - 1
SAMPLE_RANDOM_SEED = 12345
MAX_SELECTED_TRAJECTORIES: int | None = None

# Test 1.5 does not run the trap solver. It exports the exact physical support
# for every selected trajectory class. Test 2 constructs and runs the class-
# level reach probes itself.
TRAJECTORY_CLASS_PSI_DECIMALS = 12

# Preserve upstream exact analytical rejection flags and run the strengthened
# outward-force/stopping-distance certificate in this standalone Test 1.5.
USE_EXISTING_ANALYTIC_REJECTION_FLAGS = True

# Conservative outward-force test. A positive force projection alone is only
# diagnostic; hard rejection requires a lower force certificate plus a stopping-
# distance bound proving that the incoming normal motion turns before R_full.
ENABLE_OUTWARD_STOPPING_DISTANCE_TEST = True
HARD_REJECT_CERTIFIED_OUTWARD_MISSES = True
FORCE_COMPONENT = "trap"
POTENTIAL_COMPONENT = "trap"
B_ABOVE_TARGET_MARGIN_M = 1.0e-6
POSITION_SAFETY_MARGIN_M = 2.0e-6
FORCE_REFINEMENT_SAFETY_FACTOR = 2.0
FORCE_ABSOLUTE_MARGIN_N = 0.0
ENERGY_ACCESS_MARGIN_J = 0.0
MIN_ACCESSIBLE_POINTS_COARSE = 20
MIN_ACCESSIBLE_POINTS_FINE = 50
COARSE_N_DIRECTIONS = 900
FINE_N_DIRECTIONS = 2400
COARSE_N_RADII_NEAR = 10
COARSE_N_RADII_OUTER = 18
FINE_N_RADII_NEAR = 18
FINE_N_RADII_OUTER = 30
NEAR_REGION_OUTER_M = 2.0e-3
OUTER_RADIUS_FACTOR = 1.02
OUTER_RADIUS_EXTRA_M = 0.25e-3
FORCE_EVAL_CHUNK_SIZE = 100_000
POTENTIAL_EVAL_CHUNK_SIZE = 100_000
USE_XY_NORMAL_CANONICALIZATION = True
NORMAL_KEY_DECIMALS = 12

# Output files.
PILOT_OUTPUT_CSV = Path(f"{OUTPUT_PREFIX}_test2_input.csv")
REJECTION_STUB_CSV = Path(f"{OUTPUT_PREFIX}_analytical_rejection_stubs.csv")
STRATUM_SUMMARY_CSV = Path(f"{OUTPUT_PREFIX}_stratum_summary.csv")
WEIGHT_SUMMARY_CSV = Path(f"{OUTPUT_PREFIX}_weight_summary.csv")
PHYSICAL_B_SUPPORT_SUMMARY_CSV = Path(
    f"{OUTPUT_PREFIX}_physical_b_support_summary.csv"
)
TRAJECTORY_CLASS_PLAN_CSV = Path(
    f"{OUTPUT_PREFIX}_trajectory_class_plan.csv"
)
OUTWARD_FORCE_CERTIFICATE_CSV = Path(
    f"{OUTPUT_PREFIX}_outward_force_certificates.csv"
)
OUTWARD_FORCE_ROW_AUDIT_CSV = Path(
    f"{OUTPUT_PREFIX}_outward_force_row_audit.csv"
)


# =============================================================================
# BASIC HELPERS
# =============================================================================


def progress(message: str) -> None:
    print(message, flush=True)


def _candidate_path_variants(path: Path) -> list[Path]:
    path = Path(path)
    if path.is_absolute():
        return [path]
    base = Path(RUN_DIRECTORY)
    variants = [path, base / path]
    output: list[Path] = []
    seen: set[str] = set()
    for candidate in variants:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            output.append(candidate)
    return output


def resolve_test1_input_csv() -> Path:
    preferred: list[Path] = []
    if TEST1_INPUT_CSV is not None:
        preferred.append(Path(TEST1_INPUT_CSV))
    preferred.extend([
        TARGETED_ESCAPE_TEST1_INPUT_CSV,
        ADAPTIVE_MERGED_TEST1_INPUT_CSV,
        STANDARD_TEST1_INPUT_CSV,
    ])

    attempted: list[Path] = []
    for configured in preferred:
        for candidate in _candidate_path_variants(configured):
            attempted.append(candidate)
            if candidate.is_file() and candidate.stat().st_size > 0:
                return candidate

    search_roots = [Path(RUN_DIRECTORY), Path(".")]
    candidates: list[Path] = []
    for root in search_roots:
        for pattern in TEST1_INPUT_PATTERNS:
            candidates.extend(root.glob(pattern))
    candidates = sorted(
        {
            path.resolve()
            for path in candidates
            if path.is_file()
            and path.stat().st_size > 0
            and not path.name.startswith("test1p5_")
        },
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        "No Test 1 input CSV was found. Tried: "
        + ", ".join(str(path) for path in attempted)
        + "; patterns: "
        + ", ".join(TEST1_INPUT_PATTERNS)
    )


def _numeric(df: pd.DataFrame, column: str, default: float = np.nan) -> pd.Series:
    if column not in df.columns:
        return pd.Series(default, index=df.index, dtype=float)
    return pd.to_numeric(df[column], errors="coerce")


def _bool(df: pd.DataFrame, column: str, default: bool = False) -> pd.Series:
    if column not in df.columns:
        return pd.Series(default, index=df.index, dtype=bool)
    value = df[column]
    if pd.api.types.is_bool_dtype(value):
        return value.fillna(default).astype(bool)
    return value.astype(str).str.strip().str.lower().isin(
        {"true", "1", "yes", "y"}
    )


def validate_input(df: pd.DataFrame) -> None:
    required = {
        "trajectory_id",
        "speed_label",
        "v_inf_m_s",
        "theta_index",
        "alpha_index",
        "theta_rad",
        "alpha_rad",
        "psi_rad",
        "b_m",
        "b_lower_m",
        "b_upper_m",
        "impact_weight",
        "x_far_m",
        "y_far_m",
        "z_far_m",
        "vx_far_m_s",
        "vy_far_m_s",
        "vz_far_m_s",
        "eps",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError("Test 1 input is missing columns: " + ", ".join(missing))
    if "m_dm_kg" not in df.columns and "m_dm" not in df.columns:
        raise ValueError("Test 1 input must contain m_dm_kg or m_dm")



def add_physical_b_support(df: pd.DataFrame) -> pd.DataFrame:
    """Validate annuli and attach exact support for each trajectory class."""
    output = df.copy()
    output["trajectory_regime"] = infer_regime(output)
    output["trajectory_class_id"] = trajectory_class_frame(output)

    b = _numeric(output, "b_m")
    lower = _numeric(output, "b_lower_m")
    upper = _numeric(output, "b_upper_m")
    impact = _numeric(output, "impact_weight", 0.0).fillna(0.0)
    positive = impact > 0.0

    invalid_positive = positive & ~(
        np.isfinite(b)
        & np.isfinite(lower)
        & np.isfinite(upper)
        & (lower >= 0.0)
        & (upper > lower)
        & (b >= lower - 1.0e-15)
        & (b <= upper + 1.0e-15)
    )
    if invalid_positive.any():
        examples = output.loc[
            invalid_positive,
            ["trajectory_id", "trajectory_class_id", "b_m", "b_lower_m", "b_upper_m"],
        ].head(10)
        raise ValueError(
            "Positive-weight Test 1 rows have invalid collision-regime b bounds:\n"
            + examples.to_string(index=False)
        )

    output["physical_b_row_lower_m"] = lower
    output["physical_b_row_upper_m"] = upper
    output["physical_b_row_lower_um"] = lower * 1.0e6
    output["physical_b_row_upper_um"] = upper * 1.0e6
    output["physical_b_row_is_weighted_annulus"] = positive

    support_source = output.loc[positive].copy()
    support_source["_support_lower"] = _numeric(support_source, "b_lower_m")
    support_source["_support_upper"] = _numeric(support_source, "b_upper_m")
    support = (
        support_source.groupby("trajectory_class_id", dropna=False)
        .agg(
            physical_b_support_lower_m=("_support_lower", "min"),
            physical_b_support_upper_m=("_support_upper", "max"),
            physical_b_support_annuli=("trajectory_id", "size"),
        )
        .reset_index()
    )
    output = output.merge(
        support,
        on="trajectory_class_id",
        how="left",
        validate="many_to_one",
    )
    output["physical_b_support_lower_um"] = _numeric(
        output, "physical_b_support_lower_m"
    ) * 1.0e6
    output["physical_b_support_upper_um"] = _numeric(
        output, "physical_b_support_upper_m"
    ) * 1.0e6
    return output


def ensure_radius_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Require the per-speed R_full/R_switch policy exported by Test 1."""
    output = df.copy()
    if "R_full_m" not in output.columns:
        if "r_min_threshold_m" in output.columns:
            output["R_full_m"] = _numeric(output, "r_min_threshold_m")
        else:
            raise ValueError("Test 1 input must contain R_full_m")
    if "R_switch_m" not in output.columns:
        factor = _numeric(output, "R_switch_factor", 2.0).fillna(2.0)
        output["R_switch_m"] = factor * _numeric(output, "R_full_m")
    output["R_full_um"] = _numeric(output, "R_full_m") * 1.0e6
    output["R_switch_um"] = _numeric(output, "R_switch_m") * 1.0e6

    bad = ~(
        np.isfinite(_numeric(output, "R_full_m"))
        & (_numeric(output, "R_full_m") > 0.0)
        & np.isfinite(_numeric(output, "R_switch_m"))
        & (_numeric(output, "R_switch_m") > 0.0)
    )
    if bad.any():
        raise ValueError(f"Invalid R_full/R_switch values in {int(bad.sum())} rows")
    return output


def trajectory_class_frame(df: pd.DataFrame) -> pd.Series:
    psi = np.round(_numeric(df, "psi_rad"), TRAJECTORY_CLASS_PSI_DECIMALS)
    regime = infer_regime(df).astype(str)
    return pd.Series(
        [
            (
                f"{speed}|{reg}|th{int(theta):03d}|al{int(alpha):03d}|"
                f"psi{float(psi_value):.{TRAJECTORY_CLASS_PSI_DECIMALS}f}"
            )
            for speed, reg, theta, alpha, psi_value in zip(
                df["speed_label"].astype(str),
                regime,
                _numeric(df, "theta_index").fillna(-1).astype(int),
                _numeric(df, "alpha_index").fillna(-1).astype(int),
                psi,
            )
        ],
        index=df.index,
        dtype=str,
    )


def ray_identity_frame(df: pd.DataFrame) -> pd.Series:
    return trajectory_class_frame(df)


def _full_grid_shape(df: pd.DataFrame) -> tuple[int, int]:
    inferred_theta = int(_numeric(df, "theta_index").max()) + 1
    inferred_alpha = int(_numeric(df, "alpha_index").max()) + 1
    global_theta = globals().get("n_theta")
    global_alpha = globals().get("n_alpha")
    n_theta_value = int(global_theta) if global_theta is not None else inferred_theta
    n_alpha_value = int(global_alpha) if global_alpha is not None else inferred_alpha
    if n_theta_value < 2 or n_alpha_value < 1:
        raise ValueError("Could not infer a valid angular grid")
    return n_theta_value, n_alpha_value


def add_direction_weights(df: pd.DataFrame) -> pd.DataFrame:
    """Attach direction weights while honoring targeted finite subcells.

    Ordinary Test-1 rows are weighted from the global theta/alpha grid. Targeted
    escape rows already carry exact finite-subcell solid-angle weights and may
    use synthetic indices outside that grid; those explicit values take
    precedence and are not renormalized per speed label.
    """
    output = df.copy()
    explicit_full = _numeric(
        output, "direction_solid_angle_weight_full_sphere"
    )
    explicit_conditional = _numeric(
        output, "direction_solid_angle_weight"
    )

    n_theta_value, n_alpha_value = _full_grid_shape(output)
    theta_grid = np.linspace(0.0, math.pi, n_theta_value)
    theta_low = np.empty(n_theta_value)
    theta_high = np.empty(n_theta_value)
    theta_low[0] = 0.0
    theta_high[-1] = math.pi
    theta_low[1:] = 0.5 * (theta_grid[:-1] + theta_grid[1:])
    theta_high[:-1] = 0.5 * (theta_grid[:-1] + theta_grid[1:])
    cap = np.cos(theta_low) - np.cos(theta_high)
    delta_alpha = 2.0 * math.pi / n_alpha_value

    unique = output[["theta_index", "alpha_index"]].drop_duplicates()
    rows: list[dict[str, Any]] = []
    for record in unique.itertuples(index=False):
        theta_index = int(record.theta_index)
        alpha_index = int(record.alpha_index)
        if not (0 <= theta_index < n_theta_value and 0 <= alpha_index < n_alpha_value):
            continue
        alpha_width = (
            2.0 * math.pi
            if theta_index in (0, n_theta_value - 1)
            else delta_alpha
        )
        rows.append(
            {
                "theta_index": theta_index,
                "alpha_index": alpha_index,
                "_computed_direction_weight_full": (
                    float(cap[theta_index]) * alpha_width / (4.0 * math.pi)
                ),
            }
        )
    weights = pd.DataFrame(
        rows,
        columns=["theta_index", "alpha_index", "_computed_direction_weight_full"],
    )
    output = output.drop(
        columns=[
            "direction_solid_angle_weight_full_sphere",
            "direction_solid_angle_weight",
        ],
        errors="ignore",
    )
    if len(weights):
        output = output.merge(
            weights,
            on=["theta_index", "alpha_index"],
            how="left",
            validate="many_to_one",
        )
    else:
        output["_computed_direction_weight_full"] = np.nan

    output["direction_solid_angle_weight_full_sphere"] = explicit_full.where(
        np.isfinite(explicit_full),
        _numeric(output, "_computed_direction_weight_full"),
    )

    represented = (
        output[
            [
                "speed_label",
                "theta_index",
                "alpha_index",
                "direction_solid_angle_weight_full_sphere",
            ]
        ]
        .drop_duplicates(["speed_label", "theta_index", "alpha_index"])
        .groupby("speed_label", dropna=False)[
            "direction_solid_angle_weight_full_sphere"
        ]
        .sum()
        .rename("represented_direction_solid_angle_fraction")
        .reset_index()
    )
    output = output.merge(
        represented,
        on="speed_label",
        how="left",
        validate="many_to_one",
    )
    denominator = _numeric(output, "represented_direction_solid_angle_fraction")
    computed_conditional = np.divide(
        _numeric(output, "direction_solid_angle_weight_full_sphere").to_numpy(float),
        denominator.to_numpy(float),
        out=np.zeros(len(output), dtype=float),
        where=denominator.to_numpy(float) > 0.0,
    )
    output["direction_solid_angle_weight"] = explicit_conditional.where(
        np.isfinite(explicit_conditional),
        pd.Series(computed_conditional, index=output.index),
    )
    return output.drop(columns=["_computed_direction_weight_full"], errors="ignore")

def add_population_weights(df: pd.DataFrame) -> pd.DataFrame:
    output = add_direction_weights(df)
    impact = _numeric(output, "impact_weight", 0.0).fillna(0.0).clip(lower=0.0)

    if "conditional_speed_weight" in output.columns:
        conditional_speed = _numeric(output, "conditional_speed_weight", 0.0).fillna(0.0)
    elif "speed_weight_conditional" in output.columns:
        conditional_speed = _numeric(output, "speed_weight_conditional", 0.0).fillna(0.0)
    else:
        labels = output["speed_label"].astype(str).drop_duplicates()
        mapping = {label: 1.0 / max(len(labels), 1) for label in labels}
        conditional_speed = output["speed_label"].astype(str).map(mapping).astype(float)

    if "unconditional_speed_weight" in output.columns:
        unconditional_speed = _numeric(output, "unconditional_speed_weight", 0.0).fillna(0.0)
    elif "speed_weight_unconditional" in output.columns:
        unconditional_speed = _numeric(output, "speed_weight_unconditional", 0.0).fillna(0.0)
    else:
        tail = _numeric(output, "speed_tail_probability", 1.0).fillna(1.0)
        unconditional_speed = conditional_speed * tail

    output["population_weight_conditional"] = (
        impact
        * _numeric(output, "direction_solid_angle_weight", 0.0).fillna(0.0)
        * conditional_speed
    )
    output["population_weight_unconditional"] = (
        impact
        * _numeric(
            output,
            "direction_solid_angle_weight_full_sphere",
            0.0,
        ).fillna(0.0)
        * unconditional_speed
    )
    return output



# =============================================================================
# CONSERVATIVE OUTWARD-FORCE / STOPPING-DISTANCE TEST
# =============================================================================


def resolve_invalid_z_floor() -> float | None:
    c_obj = globals().get("c")
    if c_obj is not None and hasattr(c_obj, "ion_height"):
        return float(-c_obj.ion_height + 1.0e-6)
    return None


def validate_environment_for_outward_force() -> None:
    if not ENABLE_OUTWARD_STOPPING_DISTANCE_TEST:
        return
    has_force = callable(globals().get("force"))
    has_components = callable(globals().get("FDC")) and callable(globals().get("FRF"))
    if not (has_force or has_components):
        raise RuntimeError(
            "The outward-force test requires either force(..., component='trap') "
            "or both FDC and FRF in the notebook namespace. Set "
            "ENABLE_OUTWARD_STOPPING_DISTANCE_TEST=False only when intentionally "
            "running without this test."
        )


def infer_mass_column(df: pd.DataFrame) -> str:
    return "m_dm_kg" if "m_dm_kg" in df.columns else "m_dm"


def normalize(vector: np.ndarray) -> np.ndarray:
    vector = np.asarray(vector, dtype=float).reshape(3)
    norm = float(np.linalg.norm(vector))
    if not norm > 0.0:
        raise ValueError("Cannot normalize a zero vector")
    return vector / norm


def direction_basis(theta: float, alpha: float, psi: float) -> tuple[np.ndarray, np.ndarray]:
    u_hat = np.array(
        [
            np.cos(theta),
            np.sin(theta) * np.cos(alpha),
            np.sin(theta) * np.sin(alpha),
        ],
        dtype=float,
    )
    e_theta = np.array(
        [
            -np.sin(theta),
            np.cos(theta) * np.cos(alpha),
            np.cos(theta) * np.sin(alpha),
        ],
        dtype=float,
    )
    e_alpha = np.array([0.0, -np.sin(alpha), np.cos(alpha)], dtype=float)
    b_hat = np.cos(psi) * e_theta + np.sin(psi) * e_alpha
    return normalize(u_hat), normalize(b_hat)


def normal_from_row(row: pd.Series | dict[str, Any]) -> np.ndarray:
    columns = ("bvec_x_m", "bvec_y_m", "bvec_z_m")
    if all(column in row for column in columns):
        vector = np.array([row[column] for column in columns], dtype=float)
        if np.all(np.isfinite(vector)) and np.linalg.norm(vector) > 0.0:
            return normalize(vector)
    _, b_hat = direction_basis(
        float(row["theta_rad"]),
        float(row["alpha_rad"]),
        float(row["psi_rad"]),
    )
    return b_hat


def canonical_normal(normal: np.ndarray) -> np.ndarray:
    normal = normalize(normal)
    if USE_XY_NORMAL_CANONICALIZATION:
        normal = normalize(np.array([abs(normal[0]), abs(normal[1]), normal[2]]))
    return normal


def add_normal_and_state_columns(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    normals = np.vstack([normal_from_row(row) for _, row in output.iterrows()])
    canonical = np.vstack([canonical_normal(value) for value in normals])
    output["plane_bhat_x"] = normals[:, 0]
    output["plane_bhat_y"] = normals[:, 1]
    output["plane_bhat_z"] = normals[:, 2]
    output["plane_normal_key"] = [
        "|".join(f"{component:.{NORMAL_KEY_DECIMALS}f}" for component in row)
        for row in canonical
    ]
    position = output[["x_far_m", "y_far_m", "z_far_m"]].to_numpy(float)
    velocity = output[["vx_far_m_s", "vy_far_m_s", "vz_far_m_s"]].to_numpy(float)
    output["plane_state_s0_m"] = np.einsum("ij,ij->i", normals, position)
    output["plane_state_vb0_m_s"] = np.einsum("ij,ij->i", normals, velocity)
    output["r_far_actual_from_state_m"] = np.linalg.norm(position, axis=1)
    output["v_far_actual_from_state_m_s"] = np.linalg.norm(velocity, axis=1)
    return output


def evaluate_force_points(xyz: np.ndarray, mass: float, charge_fraction: float) -> np.ndarray:
    xyz = np.asarray(xyz, dtype=float).reshape(-1, 3)
    output = np.full_like(xyz, np.nan)
    use_force = callable(globals().get("force"))
    for start in range(0, len(xyz), FORCE_EVAL_CHUNK_SIZE):
        stop = min(start + FORCE_EVAL_CHUNK_SIZE, len(xyz))
        points = xyz[start:stop]
        try:
            if use_force:
                values = np.asarray(
                    force(points, mass, charge_fraction, component=FORCE_COMPONENT),
                    dtype=float,
                ).reshape(-1, 3)
            else:
                values = (
                    np.asarray(FDC(points, mass, charge_fraction), dtype=float)
                    + np.asarray(FRF(points, mass, charge_fraction), dtype=float)
                ).reshape(-1, 3)
            if values.shape != points.shape:
                raise ValueError("Unexpected vectorized force shape")
        except Exception:
            rows = []
            for point in points:
                if use_force:
                    value = force(point, mass, charge_fraction, component=FORCE_COMPONENT)
                else:
                    value = (
                        np.asarray(FDC(point[None, :], mass, charge_fraction), dtype=float)[0]
                        + np.asarray(FRF(point[None, :], mass, charge_fraction), dtype=float)[0]
                    )
                rows.append(np.asarray(value, dtype=float).reshape(3))
            values = np.vstack(rows)
        output[start:stop] = values
    return output


def evaluate_potential_points(xyz: np.ndarray, mass: float, charge_fraction: float) -> np.ndarray:
    xyz = np.asarray(xyz, dtype=float).reshape(-1, 3)
    potential_function = globals().get("potential_energy")
    if not callable(potential_function):
        return np.full(len(xyz), np.nan, dtype=float)
    output = np.full(len(xyz), np.nan, dtype=float)
    for start in range(0, len(xyz), POTENTIAL_EVAL_CHUNK_SIZE):
        stop = min(start + POTENTIAL_EVAL_CHUNK_SIZE, len(xyz))
        points = xyz[start:stop]
        try:
            values = np.asarray(
                potential_function(
                    points,
                    mass,
                    charge_fraction,
                    component=POTENTIAL_COMPONENT,
                ),
                dtype=float,
            ).reshape(-1)
            if values.size != len(points):
                raise ValueError("Unexpected vectorized potential shape")
        except Exception:
            values = np.array(
                [
                    float(
                        np.asarray(
                            potential_function(
                                point,
                                mass,
                                charge_fraction,
                                component=POTENTIAL_COMPONENT,
                            )
                        ).reshape(-1)[0]
                    )
                    for point in points
                ],
                dtype=float,
            )
        output[start:stop] = values
    return output


def fibonacci_unit_vectors(n: int, phase: float = 0.0) -> np.ndarray:
    if n < 1:
        raise ValueError("n must be positive")
    index = np.arange(n, dtype=float)
    z = 1.0 - 2.0 * (index + 0.5) / n
    golden_angle = math.pi * (3.0 - math.sqrt(5.0))
    phi = golden_angle * index + phase
    radius = np.sqrt(np.maximum(1.0 - z**2, 0.0))
    return np.column_stack([radius * np.cos(phi), radius * np.sin(phi), z])


def build_radial_grid(r_inner: float, r_outer: float, n_near: int, n_outer: int) -> np.ndarray:
    if not r_outer > r_inner:
        raise ValueError("r_outer must exceed r_inner")
    split = min(max(NEAR_REGION_OUTER_M, 1.05 * r_inner), 0.95 * r_outer)
    near = np.linspace(r_inner, split, n_near, endpoint=False)
    outer = np.geomspace(split, r_outer, n_outer)
    return np.unique(np.concatenate([near, outer, [r_inner, r_outer]]))


def build_shell_cloud(
    r_inner: float,
    r_outer: float,
    n_directions: int,
    n_near: int,
    n_outer: int,
    phase: float,
) -> np.ndarray:
    directions = fibonacci_unit_vectors(n_directions, phase=phase)
    radii = build_radial_grid(r_inner, r_outer, n_near, n_outer)
    points = (radii[:, None, None] * directions[None, :, :]).reshape(-1, 3)
    z_floor = resolve_invalid_z_floor()
    if z_floor is not None:
        points = points[points[:, 2] > z_floor]
    return points


def _cloud_force_certificate(
    points: np.ndarray,
    forces: np.ndarray,
    potential: np.ndarray,
    normal: np.ndarray,
    energy_cap_j: float,
    r_target_m: float,
) -> tuple[float, int, bool]:
    s = points @ normal
    g = forces @ normal
    mask = (
        np.isfinite(s)
        & np.isfinite(g)
        & (s >= r_target_m + POSITION_SAFETY_MARGIN_M)
    )
    energy_restricted = np.any(np.isfinite(potential)) and np.isfinite(energy_cap_j)
    if energy_restricted:
        mask &= np.isfinite(potential) & (
            potential <= energy_cap_j + ENERGY_ACCESS_MARGIN_J
        )
    count = int(np.sum(mask))
    minimum = float(np.min(g[mask])) if count else np.nan
    return minimum, count, bool(energy_restricted)


def build_outward_force_certificates(df: pd.DataFrame) -> pd.DataFrame:
    if not ENABLE_OUTWARD_STOPPING_DISTANCE_TEST:
        return pd.DataFrame()
    mass_column = infer_mass_column(df)
    parameter_sets = df[[mass_column, "eps"]].drop_duplicates()
    if len(parameter_sets) != 1:
        raise NotImplementedError(
            "The outward-force certificate currently requires one (m_dm, eps) "
            "parameter set per Test 1 input file."
        )
    mass = float(parameter_sets.iloc[0][mass_column])
    charge = float(parameter_sets.iloc[0]["eps"])
    # R_full now varies with b inside a speed label. Use the minimum target
    # radius for each speed when constructing the outward-force certificate.
    # This evaluates the force over the largest radial domain and is therefore
    # conservative for rows whose individual R_full(v,b) is larger.
    radius_by_speed = (
        df[["speed_label", "R_full_m"]]
        .assign(speed_label=lambda value: value["speed_label"].astype(str))
        .groupby("speed_label", sort=False)["R_full_m"]
        .min()
        .astype(float)
        .to_dict()
    )
    if not radius_by_speed or any(
        not np.isfinite(value) or value <= 0.0 for value in radius_by_speed.values()
    ):
        raise ValueError("Every speed label must have one finite positive R_full_m")

    far_position = df[["x_far_m", "y_far_m", "z_far_m"]].to_numpy(float)
    far_velocity = df[["vx_far_m_s", "vy_far_m_s", "vz_far_m_s"]].to_numpy(float)
    far_radius = np.linalg.norm(far_position, axis=1)
    far_speed = np.linalg.norm(far_velocity, axis=1)
    max_r_far = float(np.nanmax(far_radius))
    min_target = float(min(radius_by_speed.values()))
    max_target = float(max(radius_by_speed.values()))
    r_outer = max(
        OUTER_RADIUS_FACTOR * max_r_far,
        max_r_far + OUTER_RADIUS_EXTRA_M,
        1.25 * max_target,
    )
    r_inner = min_target + POSITION_SAFETY_MARGIN_M

    progress(
        "Outward-force cloud: "
        f"R=[{r_inner * 1e6:.3f} um, {r_outer * 1e3:.3f} mm]"
    )
    coarse_points = build_shell_cloud(
        r_inner,
        r_outer,
        COARSE_N_DIRECTIONS,
        COARSE_N_RADII_NEAR,
        COARSE_N_RADII_OUTER,
        phase=0.0,
    )
    fine_points = build_shell_cloud(
        r_inner,
        r_outer,
        FINE_N_DIRECTIONS,
        FINE_N_RADII_NEAR,
        FINE_N_RADII_OUTER,
        phase=0.5 * math.pi * (3.0 - math.sqrt(5.0)),
    )
    progress(f"Outward-force coarse points: {len(coarse_points):,}")
    progress(f"Outward-force fine points: {len(fine_points):,}")

    coarse_forces = evaluate_force_points(coarse_points, mass, charge)
    fine_forces = evaluate_force_points(fine_points, mass, charge)
    coarse_potential = evaluate_potential_points(coarse_points, mass, charge)
    fine_potential = evaluate_potential_points(fine_points, mass, charge)
    coarse_valid = np.all(np.isfinite(coarse_forces), axis=1)
    fine_valid = np.all(np.isfinite(fine_forces), axis=1)
    coarse_points, coarse_forces, coarse_potential = (
        coarse_points[coarse_valid],
        coarse_forces[coarse_valid],
        coarse_potential[coarse_valid],
    )
    fine_points, fine_forces, fine_potential = (
        fine_points[fine_valid],
        fine_forces[fine_valid],
        fine_potential[fine_valid],
    )

    u_far = evaluate_potential_points(far_position, mass, charge)
    e_total = 0.5 * mass * far_speed**2 + u_far
    energy_caps = (
        pd.DataFrame(
            {"speed_label": df["speed_label"].astype(str), "E_total_J": e_total}
        )
        .groupby("speed_label", sort=False)["E_total_J"]
        .max()
        .to_dict()
    )

    representative_normals = df[
        ["plane_normal_key", "plane_bhat_x", "plane_bhat_y", "plane_bhat_z"]
    ].drop_duplicates("plane_normal_key")
    rows: list[dict[str, Any]] = []
    total_profiles = len(representative_normals) * len(energy_caps)
    completed = 0
    for normal_row in representative_normals.itertuples(index=False):
        normal = canonical_normal(
            np.array(
                [
                    normal_row.plane_bhat_x,
                    normal_row.plane_bhat_y,
                    normal_row.plane_bhat_z,
                ],
                dtype=float,
            )
        )
        for speed_label, energy_cap in energy_caps.items():
            r_target_m = float(radius_by_speed[str(speed_label)])
            g_coarse, n_coarse, coarse_restricted = _cloud_force_certificate(
                coarse_points,
                coarse_forces,
                coarse_potential,
                normal,
                float(energy_cap),
                r_target_m,
            )
            g_fine, n_fine, fine_restricted = _cloud_force_certificate(
                fine_points,
                fine_forces,
                fine_potential,
                normal,
                float(energy_cap),
                r_target_m,
            )
            enough = (
                n_coarse >= MIN_ACCESSIBLE_POINTS_COARSE
                and n_fine >= MIN_ACCESSIBLE_POINTS_FINE
                and np.isfinite(g_coarse)
                and np.isfinite(g_fine)
            )
            if enough:
                disagreement = abs(g_coarse - g_fine)
                g_lower = (
                    min(g_coarse, g_fine)
                    - FORCE_REFINEMENT_SAFETY_FACTOR * disagreement
                    - FORCE_ABSOLUTE_MARGIN_N
                )
            else:
                disagreement = np.nan
                g_lower = np.nan
            rows.append(
                {
                    "plane_normal_key": normal_row.plane_normal_key,
                    "speed_label": str(speed_label),
                    "R_full_m": r_target_m,
                    "energy_cap_J": float(energy_cap),
                    "g_min_coarse_N": g_coarse,
                    "g_min_fine_N": g_fine,
                    "g_refinement_disagreement_N": disagreement,
                    "g_lower_certificate_N": g_lower,
                    "n_accessible_coarse": n_coarse,
                    "n_accessible_fine": n_fine,
                    "energy_restriction_used": bool(coarse_restricted and fine_restricted),
                    "force_certificate_positive": bool(
                        enough and np.isfinite(g_lower) and g_lower > 0.0
                    ),
                }
            )
            completed += 1
            if completed % 100 == 0 or completed == total_profiles:
                progress(f"Outward-force certificates: {completed:,}/{total_profiles:,}")
    return pd.DataFrame(rows)


def apply_outward_stopping_distance_test(
    df: pd.DataFrame,
    certificate_df: pd.DataFrame,
) -> pd.DataFrame:
    output = df.copy()
    if not ENABLE_OUTWARD_STOPPING_DISTANCE_TEST:
        output["force_certificate_positive"] = False
        output["outward_acceleration_lower_m_s2"] = np.nan
        output["normal_inward_stopping_distance_m"] = np.nan
        output["normal_coordinate_min_lower_bound_m"] = np.nan
        output["stopping_distance_filter_eligible"] = False
        output["stopping_distance_filter_reject"] = False
        output["posttest1_filter_status"] = "retained_outward_test_disabled"
        return output

    certificate_merge = certificate_df.drop(columns=["R_full_m"], errors="ignore")
    output["speed_label"] = output["speed_label"].astype(str)
    output = output.merge(
        certificate_merge,
        on=["plane_normal_key", "speed_label"],
        how="left",
        validate="many_to_one",
    )
    mass = _numeric(output, infer_mass_column(output)).to_numpy(float)
    g_lower = _numeric(output, "g_lower_certificate_N").to_numpy(float)
    acceleration = np.divide(
        g_lower,
        mass,
        out=np.full(len(output), np.nan),
        where=np.isfinite(g_lower) & np.isfinite(mass) & (mass > 0.0),
    )
    vb0 = _numeric(output, "plane_state_vb0_m_s").to_numpy(float)
    inward_speed = np.maximum(-vb0, 0.0)
    stopping_distance = np.divide(
        inward_speed**2,
        2.0 * acceleration,
        out=np.full(len(output), np.inf),
        where=np.isfinite(acceleration) & (acceleration > 0.0),
    )
    s0 = _numeric(output, "plane_state_s0_m").to_numpy(float)
    s_min_bound = s0 - stopping_distance
    b = _numeric(output, "b_m").to_numpy(float)
    r_target = _numeric(output, "R_full_m").to_numpy(float)
    impact = _numeric(output, "impact_weight", 0.0).fillna(0.0).to_numpy(float)
    eligible = (
        (impact > 0.0)
        & (b > r_target + B_ABOVE_TARGET_MARGIN_M)
        & np.isfinite(s0)
        & np.isfinite(r_target)
        & (s0 > r_target + POSITION_SAFETY_MARGIN_M)
    )
    positive_certificate = _bool(output, "force_certificate_positive").to_numpy(bool)
    certified_miss = (
        eligible
        & positive_certificate
        & np.isfinite(s_min_bound)
        & (s_min_bound > r_target + POSITION_SAFETY_MARGIN_M)
    )
    reject = certified_miss & bool(HARD_REJECT_CERTIFIED_OUTWARD_MISSES)

    output["outward_acceleration_lower_m_s2"] = acceleration
    output["normal_inward_stopping_distance_m"] = stopping_distance
    output["normal_coordinate_min_lower_bound_m"] = s_min_bound
    output["stopping_distance_filter_eligible"] = eligible
    output["outward_force_certified_miss"] = certified_miss
    output["stopping_distance_filter_reject"] = reject
    output["posttest1_filter_status"] = np.where(
        reject,
        "stopping_distance_outward_reject",
        np.where(
            certified_miss,
            "certified_outward_miss_diagnostic_only",
            np.where(
                ~eligible,
                "retained_not_eligible_b_or_state",
                np.where(
                    ~positive_certificate,
                    "retained_no_positive_force_certificate",
                    "retained_stopping_bound_reaches_target_side",
                ),
            ),
        ),
    )
    return output



def _merge_physical_annuli(
    intervals: Iterable[tuple[float, float]],
) -> list[tuple[float, float]]:
    """Return the exact union of valid annuli, preserving real gaps."""
    clean = sorted(
        (float(lower), float(upper))
        for lower, upper in intervals
        if np.isfinite(lower)
        and np.isfinite(upper)
        and 0.0 <= lower < upper
    )
    if not clean:
        return []
    merged: list[list[float]] = [[clean[0][0], clean[0][1]]]
    for lower, upper in clean[1:]:
        previous = merged[-1]
        tolerance = max(1.0e-15, 1.0e-12 * max(previous[1], upper, 1.0e-12))
        if lower <= previous[1] + tolerance:
            previous[1] = max(previous[1], upper)
        else:
            merged.append([lower, upper])
    return [(lower, upper) for lower, upper in merged]


def _annuli_json(annuli: Iterable[tuple[float, float]]) -> str:
    return json.dumps(
        [[float(lower), float(upper)] for lower, upper in annuli],
        separators=(",", ":"),
    )


def _fraction_in_annulus_union(
    b2_value: float,
    annuli: list[tuple[float, float]],
) -> float:
    widths = [upper * upper - lower * lower for lower, upper in annuli]
    total = float(sum(widths))
    if total <= 0.0 or not np.isfinite(b2_value):
        return 0.0
    cumulative = 0.0
    for (lower, upper), width in zip(annuli, widths):
        lower2 = lower * lower
        upper2 = upper * upper
        if b2_value <= upper2 + max(1.0e-30, 1.0e-12 * upper2):
            local = float(np.clip(b2_value - lower2, 0.0, width))
            return float(np.clip((cumulative + local) / total, 0.0, 1.0))
        cumulative += width
    return 1.0


def refresh_surviving_b_support(df: pd.DataFrame) -> pd.DataFrame:
    """Attach the exact surviving Test 0.75 annulus union to every class row.

    The previous version returned immediately when no positive-impact rows
    survived the analytical filters. That left downstream summary columns
    undefined and caused ``groupby(...).agg(...)`` to raise a KeyError.
    Always create the full output schema, even for an empty surviving set.
    """
    output = df.copy()

    # Preserve the pre-filter collision support for auditing. These assignments
    # are valid for both populated and header-only/empty DataFrames.
    output["collision_b_support_lower_m"] = _numeric(
        output, "physical_b_support_lower_m"
    )
    output["collision_b_support_upper_m"] = _numeric(
        output, "physical_b_support_upper_m"
    )

    # Header-safe defaults required by support summaries and Test 2.
    if "surviving_b_support_lower_m" not in output.columns:
        output["surviving_b_support_lower_m"] = np.nan
    if "surviving_b_support_upper_m" not in output.columns:
        output["surviving_b_support_upper_m"] = np.nan
    if "surviving_b_support_annuli" not in output.columns:
        output["surviving_b_support_annuli"] = 0
    if "physical_b_annuli_json" not in output.columns:
        output["physical_b_annuli_json"] = "[]"
    if "physical_b_union_area2_m2" not in output.columns:
        output["physical_b_union_area2_m2"] = 0.0
    if "physical_b_union_fraction_midpoint" not in output.columns:
        output["physical_b_union_fraction_midpoint"] = np.nan

    positive = _numeric(output, "impact_weight", 0.0).fillna(0.0) > 0.0
    source = output.loc[positive].copy()
    if len(source) == 0:
        # No positive-area annulus survived. Return a schema-complete table so
        # downstream code can write empty audit products instead of crashing.
        output["physical_b_support_lower_um"] = _numeric(
            output, "physical_b_support_lower_m"
        ) * 1.0e6
        output["physical_b_support_upper_um"] = _numeric(
            output, "physical_b_support_upper_m"
        ) * 1.0e6
        return output

    class_annuli: dict[str, list[tuple[float, float]]] = {}
    metadata: list[dict[str, Any]] = []
    for class_id, group in source.groupby("trajectory_class_id", sort=False):
        intervals = list(
            zip(
                _numeric(group, "b_lower_m").to_numpy(float),
                _numeric(group, "b_upper_m").to_numpy(float),
            )
        )
        annuli = _merge_physical_annuli(intervals)
        if not annuli:
            raise ValueError(f"No valid surviving b annuli for class {class_id}")
        class_annuli[str(class_id)] = annuli
        total_area2 = float(
            sum(upper * upper - lower * lower for lower, upper in annuli)
        )
        metadata.append(
            {
                "trajectory_class_id": str(class_id),
                "surviving_b_support_lower_m": annuli[0][0],
                "surviving_b_support_upper_m": annuli[-1][1],
                "surviving_b_support_annuli": len(annuli),
                "physical_b_annuli_json": _annuli_json(annuli),
                "physical_b_union_area2_m2": total_area2,
            }
        )

    support = pd.DataFrame(metadata)
    output = output.drop(
        columns=[
            "surviving_b_support_lower_m",
            "surviving_b_support_upper_m",
            "surviving_b_support_annuli",
            "physical_b_annuli_json",
            "physical_b_union_area2_m2",
        ],
        errors="ignore",
    ).merge(
        support,
        on="trajectory_class_id",
        how="left",
        validate="many_to_one",
    )

    output["collision_b_support_lower_m"] = _numeric(
        output, "physical_b_support_lower_m"
    )
    output["collision_b_support_upper_m"] = _numeric(
        output, "physical_b_support_upper_m"
    )
    output["physical_b_support_lower_m"] = _numeric(
        output, "surviving_b_support_lower_m"
    ).fillna(_numeric(output, "physical_b_support_lower_m"))
    output["physical_b_support_upper_m"] = _numeric(
        output, "surviving_b_support_upper_m"
    ).fillna(_numeric(output, "physical_b_support_upper_m"))
    output["physical_b_support_lower_um"] = _numeric(
        output, "physical_b_support_lower_m"
    ) * 1.0e6
    output["physical_b_support_upper_um"] = _numeric(
        output, "physical_b_support_upper_m"
    ) * 1.0e6

    union_fraction = np.full(len(output), np.nan, dtype=float)
    row_lower = _numeric(output, "b_lower_m").to_numpy(float)
    row_upper = _numeric(output, "b_upper_m").to_numpy(float)
    class_ids = output["trajectory_class_id"].astype(str).to_numpy()
    for position, (class_id, lower, upper) in enumerate(
        zip(class_ids, row_lower, row_upper)
    ):
        annuli = class_annuli.get(class_id)
        if annuli and np.isfinite(lower) and np.isfinite(upper):
            midpoint2 = 0.5 * (lower * lower + upper * upper)
            union_fraction[position] = _fraction_in_annulus_union(midpoint2, annuli)
    output["physical_b_union_fraction_midpoint"] = union_fraction
    return output

# =============================================================================
# REGIME AND REJECTION ROUTING
# =============================================================================


def infer_regime(df: pd.DataFrame) -> pd.Series:
    if "trajectory_regime" in df.columns:
        regime = df["trajectory_regime"].astype(str).str.lower()
    elif "collision_regime" in df.columns:
        regime = df["collision_regime"].astype(str).str.lower()
    else:
        regime = pd.Series("resonant", index=df.index, dtype=object)

    adiabatic = (
        _bool(df, "test0p75_adiabatic_reject")
        | _bool(df, "adiabatic_analytic_reject")
        | regime.str.contains("adiabatic", na=False)
    )
    rutherford = (
        _bool(df, "use_rutherford_after_reach")
        | regime.str.contains("rutherford", na=False)
    ) & ~adiabatic
    result = pd.Series("resonant", index=df.index, dtype=object)
    result.loc[rutherford] = "rutherford"
    result.loc[adiabatic] = "adiabatic"
    return result


def existing_analytic_reject_mask(df: pd.DataFrame) -> pd.Series:
    mask = infer_regime(df).eq("adiabatic")
    if USE_EXISTING_ANALYTIC_REJECTION_FLAGS:
        for column in (
            "test1p5_analytic_reject",
            "stopping_distance_filter_reject",
            "analytical_reject",
        ):
            mask |= _bool(df, column)
    return mask


def build_rejection_stubs(rejected: pd.DataFrame) -> pd.DataFrame:
    output = rejected.copy()
    if len(output) == 0:
        return output
    adiabatic = infer_regime(output).eq("adiabatic")
    outward = _bool(output, "stopping_distance_filter_reject")
    reason = np.select(
        [adiabatic, outward],
        ["test0p75_adiabatic_reject", "stopping_distance_outward_reject"],
        default="existing_exact_analytic_reject",
    )
    output["test1p5_analytic_reject"] = True
    output["test1p5_rejection_reason"] = reason
    output["test2_analysis_weight_conditional"] = _numeric(
        output, "population_weight_conditional", 0.0
    ).fillna(0.0)
    output["test2_analysis_weight_unconditional"] = _numeric(
        output, "population_weight_unconditional", 0.0
    ).fillna(0.0)
    output["test2_sample_role"] = "analytic_rejection_stub"
    output["prefilter_skip_test2"] = True
    output["dm_only_success"] = True
    output["dm_only_status"] = reason
    output["entered_switch"] = False
    output["full_success"] = True
    output["full_status"] = reason
    output["full_simulation_run"] = False
    output["reached_R_full"] = False
    output["above_energy_threshold"] = False
    output["delta_E_ion_J"] = 0.0
    output["energy_model_used"] = np.where(
        outward,
        "outward_force_stopping_distance_certificate",
        "analytic_prefilter",
    )
    return output


# =============================================================================
# WEIGHTED STRATIFIED PILOT
# =============================================================================


def _safe_bin(values: pd.Series, n_bins: int, *, circular: bool = False) -> pd.Series:
    array = pd.to_numeric(values, errors="coerce").to_numpy(float)
    if circular:
        array = np.mod(array, 2.0 * math.pi)
        scaled = array / (2.0 * math.pi)
    else:
        finite = array[np.isfinite(array)]
        if finite.size == 0 or np.isclose(finite.min(), finite.max()):
            return pd.Series(0, index=values.index, dtype=int)
        scaled = (array - finite.min()) / (finite.max() - finite.min())
    result = np.floor(np.clip(scaled, 0.0, 1.0 - np.finfo(float).eps) * n_bins)
    result[~np.isfinite(result)] = 0
    return pd.Series(result.astype(int), index=values.index)



def add_strata(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    output["trajectory_regime"] = infer_regime(output)
    mu = np.cos(_numeric(output, "theta_rad"))
    output["test1p5_mu_bin"] = _safe_bin(mu, N_MU_BINS)
    output["test1p5_alpha_bin"] = _safe_bin(
        _numeric(output, "alpha_rad"), N_ALPHA_BINS, circular=True
    )
    output["test1p5_psi_bin"] = _safe_bin(
        _numeric(output, "psi_rad"), N_PSI_BINS, circular=True
    )

    # The coordinate is cumulative annular area across the exact union of
    # surviving Test 0.75 intervals. Real gaps contribute no probability mass.
    if "physical_b_union_fraction_midpoint" in output.columns:
        fraction = _numeric(output, "physical_b_union_fraction_midpoint")
    else:
        row_lower = _numeric(output, "b_lower_m")
        row_upper = _numeric(output, "b_upper_m")
        support_lower = _numeric(output, "physical_b_support_lower_m")
        support_upper = _numeric(output, "physical_b_support_upper_m")
        row_area_coordinate = 0.5 * (row_lower**2 + row_upper**2)
        support_lower2 = support_lower**2
        support_width2 = support_upper**2 - support_lower2
        fallback = np.divide(
            (row_area_coordinate - support_lower2).to_numpy(float),
            support_width2.to_numpy(float),
            out=np.zeros(len(output), dtype=float),
            where=support_width2.to_numpy(float) > 0.0,
        )
        fraction = pd.Series(fallback, index=output.index)

    fraction_array = np.clip(fraction.to_numpy(float), 0.0, 1.0)
    edges = np.asarray(PHYSICAL_B2_BIN_EDGES, dtype=float)
    if not (
        edges.ndim == 1
        and len(edges) >= 2
        and np.isclose(edges[0], 0.0)
        and np.isclose(edges[-1], 1.0)
        and np.all(np.diff(edges) > 0.0)
    ):
        raise ValueError("PHYSICAL_B2_BIN_EDGES must increase strictly from 0 to 1")
    bin_index = np.searchsorted(edges, fraction_array, side="right") - 1
    bin_index = np.clip(bin_index, 0, len(edges) - 2).astype(int)
    output["test1p5_physical_b2_fraction"] = fraction_array
    output["test1p5_b2_bin"] = bin_index
    output["test1p5_b2_bin_lower_fraction"] = edges[bin_index]
    output["test1p5_b2_bin_upper_fraction"] = edges[bin_index + 1]
    return output

def sample_weighted_strata(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    data = add_strata(df)
    positive = _numeric(data, "population_weight_conditional", 0.0).fillna(0.0) > 0.0
    data = data.loc[positive].copy()
    if len(data) == 0:
        return data, pd.DataFrame()

    forced_mask = _bool(data, "force_test2_selection", False)
    forced = data.loc[forced_mask].copy()
    regular = data.loc[~forced_mask].copy()

    selected_rows: list[pd.Series] = []
    summaries: list[dict[str, Any]] = []

    # Every targeted finite subcell is a distinct quadrature cell. Keep it in
    # the pilot with its exact population weight rather than randomly replacing
    # it by another row from a coarse stratum.
    for _, chosen in forced.iterrows():
        chosen = chosen.copy()
        chosen["test2_analysis_weight_conditional"] = float(
            chosen.get("population_weight_conditional", 0.0)
        )
        chosen["test2_analysis_weight_unconditional"] = float(
            chosen.get("population_weight_unconditional", 0.0)
        )
        chosen["test2_sampling_probability_within_stratum"] = 1.0
        chosen["test2_population_rows_in_stratum"] = 1
        chosen["test2_sample_seed"] = int(SAMPLE_RANDOM_SEED)
        chosen["test2_sample_preset"] = "targeted_escape_resolution_v12"
        chosen["test2_sample_role"] = "targeted_escape_finite_subcell_seed"
        chosen["test1p5_stratum_key"] = (
            "targeted|" + str(chosen.get("trajectory_id", ""))
        )
        selected_rows.append(chosen)
        summaries.append(
            {
                "speed_label": str(chosen.get("speed_label", "")),
                "trajectory_regime": str(chosen.get("trajectory_regime", "")),
                "test1p5_mu_bin": chosen.get("test1p5_mu_bin", np.nan),
                "test1p5_alpha_bin": chosen.get("test1p5_alpha_bin", np.nan),
                "test1p5_psi_bin": chosen.get("test1p5_psi_bin", np.nan),
                "test1p5_b2_bin": chosen.get("test1p5_b2_bin", np.nan),
                "n_population_rows": 1,
                "stratum_weight_conditional": float(
                    chosen.get("population_weight_conditional", 0.0)
                ),
                "stratum_weight_unconditional": float(
                    chosen.get("population_weight_unconditional", 0.0)
                ),
                "selected_trajectory_id": str(chosen.get("trajectory_id", "")),
                "selection_probability": 1.0,
                "forced_targeted_selection": True,
            }
        )

    rng = np.random.default_rng(SAMPLE_RANDOM_SEED)
    group_columns = [
        "speed_label",
        "trajectory_regime",
        "test1p5_mu_bin",
        "test1p5_alpha_bin",
        "test1p5_psi_bin",
        "test1p5_b2_bin",
    ]
    for keys, group in regular.groupby(group_columns, sort=True, dropna=False):
        conditional = _numeric(group, "population_weight_conditional", 0.0).fillna(0.0)
        unconditional = _numeric(group, "population_weight_unconditional", 0.0).fillna(0.0)
        total_conditional = float(conditional.sum())
        total_unconditional = float(unconditional.sum())
        probabilities = conditional.to_numpy(dtype=float, copy=True)
        if total_conditional > 0.0:
            probabilities /= total_conditional
        else:
            probabilities = np.full(len(group), 1.0 / len(group))
        local_index = int(rng.choice(len(group), p=probabilities))
        chosen = group.iloc[local_index].copy()
        chosen["test2_analysis_weight_conditional"] = total_conditional
        chosen["test2_analysis_weight_unconditional"] = total_unconditional
        chosen["test2_sampling_probability_within_stratum"] = float(
            probabilities[local_index]
        )
        chosen["test2_population_rows_in_stratum"] = int(len(group))
        chosen["test2_sample_seed"] = int(SAMPLE_RANDOM_SEED)
        chosen["test2_sample_preset"] = "physical_b_class_reach_v11"
        chosen["test2_sample_role"] = "weighted_stratum_representative"
        chosen["test1p5_stratum_key"] = "|".join(map(str, keys))
        selected_rows.append(chosen)

        row = {column: value for column, value in zip(group_columns, keys)}
        row.update(
            {
                "n_population_rows": int(len(group)),
                "stratum_weight_conditional": total_conditional,
                "stratum_weight_unconditional": total_unconditional,
                "selected_trajectory_id": str(chosen["trajectory_id"]),
                "selection_probability": float(probabilities[local_index]),
                "forced_targeted_selection": False,
            }
        )
        summaries.append(row)

    selected = pd.DataFrame(selected_rows)
    summary = pd.DataFrame(summaries)
    if MAX_SELECTED_TRAJECTORIES is not None and len(selected) > MAX_SELECTED_TRAJECTORIES:
        raise RuntimeError(
            f"Pilot selected {len(selected)} rows, exceeding "
            f"MAX_SELECTED_TRAJECTORIES={MAX_SELECTED_TRAJECTORIES}"
        )
    return selected.reset_index(drop=True), summary


# =============================================================================
# TRAJECTORY-CLASS REACH-SCREEN PLAN
# =============================================================================



def build_trajectory_class_plan(selected: pd.DataFrame) -> pd.DataFrame:
    if len(selected) == 0:
        return pd.DataFrame()
    rows: list[dict[str, Any]] = []
    for class_id, group in selected.groupby("trajectory_class_id", sort=True):
        representative = group.iloc[0]
        lower = float(_numeric(group, "physical_b_support_lower_m").dropna().iloc[0])
        upper = float(_numeric(group, "physical_b_support_upper_m").dropna().iloc[0])
        rows.append(
            {
                "trajectory_class_id": str(class_id),
                "speed_label": str(representative["speed_label"]),
                "trajectory_regime": str(representative["trajectory_regime"]),
                "theta_index": int(representative["theta_index"]),
                "alpha_index": int(representative["alpha_index"]),
                "theta_rad": float(representative["theta_rad"]),
                "alpha_rad": float(representative["alpha_rad"]),
                "psi_rad": float(representative["psi_rad"]),
                "R_switch_m": float(representative["R_switch_m"]),
                "physical_b_support_lower_m": lower,
                "physical_b_support_upper_m": upper,
                "physical_b_support_lower_um": lower * 1.0e6,
                "physical_b_support_upper_um": upper * 1.0e6,
                "physical_b_annuli_json": str(
                    representative.get("physical_b_annuli_json", "[]")
                ),
                "physical_b_support_annuli": int(
                    numeric_value_for_plan(
                        representative.get("surviving_b_support_annuli", 0)
                    )
                ),
                "physical_b_union_area2_m2": float(
                    representative.get("physical_b_union_area2_m2", np.nan)
                ),
                "n_weighted_representatives": int(len(group)),
                "class_requires_dm_only_reach_screen": True,
            }
        )
    return pd.DataFrame(rows)


def numeric_value_for_plan(value: Any, default: float = 0.0) -> float:
    try:
        result = float(value)
    except (TypeError, ValueError):
        return default
    return result if np.isfinite(result) else default

# =============================================================================
# MAIN
# =============================================================================


def main() -> None:
    input_csv = resolve_test1_input_csv()
    population = pd.read_csv(input_csv)
    validate_input(population)
    validate_parameter_dataframe(population, "Test 1 input")
    population["trajectory_id"] = population["trajectory_id"].astype(str)
    population = ensure_radius_columns(population)
    population["trajectory_regime"] = infer_regime(population)
    population = add_physical_b_support(population)
    population = add_population_weights(population)
    population = add_normal_and_state_columns(population)

    validate_environment_for_outward_force()
    if ENABLE_OUTWARD_STOPPING_DISTANCE_TEST:
        certificates = build_outward_force_certificates(population)
    else:
        certificates = pd.DataFrame()
    certificates.to_csv(OUTWARD_FORCE_CERTIFICATE_CSV, index=False)
    population = apply_outward_stopping_distance_test(population, certificates)
    population.to_csv(OUTWARD_FORCE_ROW_AUDIT_CSV, index=False)

    reject_mask = existing_analytic_reject_mask(population)
    rejected = population.loc[reject_mask].copy()
    dynamic = population.loc[~reject_mask].copy()
    dynamic = refresh_surviving_b_support(dynamic)

    rejection_stubs = build_rejection_stubs(rejected)
    rejection_stubs.to_csv(REJECTION_STUB_CSV, index=False)

    support_source = dynamic.loc[
        _numeric(dynamic, "impact_weight", 0.0).fillna(0.0) > 0.0
    ].copy()
    if len(support_source):
        support_summary = (
            support_source.groupby(
                ["trajectory_class_id", "speed_label", "trajectory_regime"],
                dropna=False,
            )
            .agg(
                b_support_lower_m=("physical_b_support_lower_m", "first"),
                b_support_upper_m=("physical_b_support_upper_m", "first"),
                physical_b_annuli_json=("physical_b_annuli_json", "first"),
                physical_b_union_area2_m2=("physical_b_union_area2_m2", "first"),
                n_weighted_annuli=("trajectory_id", "size"),
            )
            .reset_index()
        )
    else:
        # A fully rejected population is a valid result. Save a header-only
        # summary rather than attempting named aggregation on missing columns.
        support_summary = pd.DataFrame(
            columns=[
                "trajectory_class_id",
                "speed_label",
                "trajectory_regime",
                "b_support_lower_m",
                "b_support_upper_m",
                "physical_b_annuli_json",
                "physical_b_union_area2_m2",
                "n_weighted_annuli",
            ]
        )
    support_summary["b_support_lower_um"] = pd.to_numeric(
        support_summary["b_support_lower_m"], errors="coerce"
    ) * 1.0e6
    support_summary["b_support_upper_um"] = pd.to_numeric(
        support_summary["b_support_upper_m"], errors="coerce"
    ) * 1.0e6
    support_summary.to_csv(PHYSICAL_B_SUPPORT_SUMMARY_CSV, index=False)

    weighted_selected, stratum_summary = sample_weighted_strata(dynamic)
    stratum_summary.to_csv(STRATUM_SUMMARY_CSV, index=False)
    weighted_selected = weighted_selected.sort_values(
        ["speed_label", "theta_index", "alpha_index", "psi_rad", "b_m"],
        kind="mergesort",
    ).reset_index(drop=True)
    weighted_selected.to_csv(PILOT_OUTPUT_CSV, index=False)

    class_plan = build_trajectory_class_plan(weighted_selected)
    class_plan.to_csv(TRAJECTORY_CLASS_PLAN_CSV, index=False)

    n_selected_classes = int(
        weighted_selected["trajectory_class_id"].nunique()
    ) if "trajectory_class_id" in weighted_selected.columns else 0

    weight_summary = pd.DataFrame(
        [{
            "test1_input_csv": str(input_csv),
            "n_test1_rows": len(population),
            "n_analytic_rejection_rows": len(rejected),
            "n_outward_force_certified_miss_rows": int(
                _bool(population, "outward_force_certified_miss").sum()
            ),
            "n_outward_force_hard_rejection_rows": int(
                _bool(population, "stopping_distance_filter_reject").sum()
            ),
            "n_dynamic_population_rows": len(dynamic),
            "n_weighted_pilot_rows": len(weighted_selected),
            "n_selected_trajectory_classes": n_selected_classes,
            "n_total_test2_rows": len(weighted_selected),
            "dynamic_population_weight_conditional": float(
                _numeric(dynamic, "population_weight_conditional", 0.0).sum()
            ),
            "selected_analysis_weight_conditional": float(
                _numeric(weighted_selected, "test2_analysis_weight_conditional", 0.0).sum()
            ),
            "analytic_rejection_weight_conditional": float(
                _numeric(rejection_stubs, "test2_analysis_weight_conditional", 0.0).sum()
            ),
        }]
    )
    weight_summary.to_csv(WEIGHT_SUMMARY_CSV, index=False)

    progress(f"Loaded Test 1 rows: {len(population):,} from {input_csv}")
    progress(f"Weighted pilot rows: {len(weighted_selected):,}")
    progress(
        f"Selected trajectory classes: {n_selected_classes:,}"
    )
    progress(f"Analytical rejection stubs: {len(rejection_stubs):,}")
    progress(
        "Certified outward misses: "
        f"{int(_bool(population, 'outward_force_certified_miss').sum()):,}; "
        "hard rejected: "
        f"{int(_bool(population, 'stopping_distance_filter_reject').sum()):,}"
    )
    progress(f"Saved outward-force certificates: {OUTWARD_FORCE_CERTIFICATE_CSV}")
    progress(f"Saved outward-force row audit: {OUTWARD_FORCE_ROW_AUDIT_CSV}")
    progress(f"Saved Test 2 input: {PILOT_OUTPUT_CSV}")
    progress(f"Saved class reach-screen plan: {TRAJECTORY_CLASS_PLAN_CSV}")


if __name__ == "__main__":
    if bool(globals().get("USER_RUN_TEST1P5", True)):
        main()
    else:
        progress("[SKIP] Test 1.5")


# Test 2 — Connected-Bowl-Aware Complete Fourier Force Pulse

In addition to the complete incoming-$R_{\rm outer}$ to outgoing-$R_{\rm outer}$ Fourier energy, Test 2 records:

- entry into the origin-connected near-barrier equipotential bowl;
- the entry channel (`axial_dc_x`, `radial_rf_y`, `vertical_rf_z`, or mixed);
- DC and RF potential contributions at entry;
- DC and RF force components directed toward the ion at entry;
- residence time in the bowl and inside $R_{\rm full}$;
- whether a resolved above-threshold trajectory provides a no-bowl counterexample.


In [ ]:
#!/usr/bin/env python3
"""Standalone Test 2 v18 connected-equipotential bowl-aware: full rise-and-fall Fourier pulse through R_outer.

Run after the trap-model notebook cells and Test 1.5:

    %run -i test2_exact_annuli_lower_b_reach_v12.py

Required notebook globals
-------------------------
FDC, FRF, m_ion, Z_ion, omega_vec, K, e

Optional notebook global
------------------------
c.ion_height, used to define the lower valid-z boundary.

This file contains the complete Test 2 logic. It does not import, patch,
execute, or read another Python script. It performs:

1. a class-level DM-only reach search over b values inside each physical
   collision-energy support interval;
2. rejection of a trajectory class only when every planned b probe is a
   resolved miss of R_switch;
3. ordinary DM-only screening of production rows for accepted or unresolved
   classes;
4. local Rutherford or full ion-DM propagation after confirmed switch entry;
5. a fast perturbative outgoing tail from the coupled escape surface to R_far.

The class screen starts at the physical lower b limit and proceeds upward in
b^2. It stops after the first switch entry. Thus every accepted class contains
an explicit zero-weight anchor trajectory that reaches R_switch.
"""

from __future__ import annotations

import json
import hashlib
import math
import time
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, as_completed, wait
from pathlib import Path
from typing import Any, Callable, Iterable

import numpy as np
import pandas as pd
from scipy.integrate import quad, solve_ivp


# =============================================================================
# USER SETTINGS
# =============================================================================

TEST1_INPUT_CSV = Path(
    globals().get(
        "USER_TEST2_INPUT_CSV",
        f"{TEST1P5_OUTPUT_PREFIX}_test2_input.csv",
    )
)
PREFILTER_REJECTION_STUB_CSV: Path | None = Path(
    f"{TEST1P5_OUTPUT_PREFIX}_analytical_rejection_stubs.csv"
)
EXTRA_DIAGNOSTIC_INPUT_CSV: Path | None = Path(
    globals().get(
        "USER_TEST2_EXTRA_DIAGNOSTIC_INPUT_CSV",
        Path(RUN_DIRECTORY)
        / "escape_channel_adaptive_rmax_bowl_v2_escape_channel_focus_test2_input.csv",
    )
)
OUTPUT_PREFIX = TEST2_OUTPUT_PREFIX

TARGET_ION_ENERGY_J = float(USER_TARGET_ION_ENERGY_J)
R_SWITCH_FACTOR = 2.0
FULL_ESCAPE_FACTOR = 1.5
DM_ONLY_ESCAPE_FACTOR = 1.05

DM_ONLY_RTOL = 1.0e-6
DM_ONLY_ATOL = 1.0e-9
DM_ONLY_MAX_STEP_S = 5.0e-7
DM_ONLY_TIME_FACTOR = 3.0
DM_ONLY_MIN_AFTER_ARRIVAL_S = 20.0e-6

FULL_RTOL = 1.0e-6
FULL_ATOL = 1.0e-9

# The coupled ODE carries cosine and sine Coulomb-force quadratures for each
# secular mode. Limit the maximum step by the fastest secular period so the
# Fourier integrals are resolved rather than inferred from sparse output.
DETECTION_ENERGY_POLICY = str(
    globals().get("USER_DETECTION_ENERGY_POLICY", "fourier")
).strip().lower()
REQUIRE_FOURIER_MECHANICAL_AGREEMENT = bool(
    globals().get("USER_REQUIRE_FOURIER_MECHANICAL_AGREEMENT", False)
)
FOURIER_MECHANICAL_REL_TOL = float(
    globals().get("USER_FOURIER_MECHANICAL_REL_TOL", 5.0e-2)
)
FOURIER_STEPS_PER_SECULAR_PERIOD = int(
    globals().get("USER_FOURIER_STEPS_PER_SECULAR_PERIOD", 20)
)
_omega_for_resolution = np.abs(np.asarray(omega_vec, dtype=float).reshape(3))
_positive_omega_for_resolution = _omega_for_resolution[
    np.isfinite(_omega_for_resolution) & (_omega_for_resolution > 0.0)
]
if _positive_omega_for_resolution.size == 0:
    raise ValueError("omega_vec must contain at least one positive secular angular frequency")
FASTEST_SECULAR_PERIOD_S = float(
    2.0 * np.pi / np.max(_positive_omega_for_resolution)
)
FOURIER_RESOLUTION_STEP_S = (
    FASTEST_SECULAR_PERIOD_S / FOURIER_STEPS_PER_SECULAR_PERIOD
)
FULL_SAMPLE_DT_S = min(5.0e-9, FOURIER_RESOLUTION_STEP_S)
FULL_MAX_STEP_S = min(5.0e-9, FOURIER_RESOLUTION_STEP_S)
FOURIER_DETECTION_START_RADIUS = str(
    globals().get("USER_FOURIER_DETECTION_START_RADIUS", "R_far")
).strip()
if FOURIER_DETECTION_START_RADIUS.lower() != "r_far":
    raise ValueError(
        "This Test 2 version requires USER_FOURIER_DETECTION_START_RADIUS='R_far'"
    )
REQUIRE_R_FAR_PREFIX_COMPLETE = bool(
    globals().get("USER_REQUIRE_R_FAR_PREFIX_COMPLETE", True)
)
PREFIX_QUAD_EPSREL = float(globals().get("USER_PREFIX_QUAD_EPSREL", 1.0e-8))
PREFIX_QUAD_EPSABS_DIMENSIONLESS = float(
    globals().get("USER_PREFIX_QUAD_EPSABS_DIMENSIONLESS", 1.0e-11)
)
PREFIX_QUAD_LIMIT = int(globals().get("USER_PREFIX_QUAD_LIMIT", 1000))
PREFIX_FORCE_SCALE_SAMPLES = int(
    globals().get("USER_PREFIX_FORCE_SCALE_SAMPLES", 17)
)
if PREFIX_FORCE_SCALE_SAMPLES < 3:
    raise ValueError("USER_PREFIX_FORCE_SCALE_SAMPLES must be at least 3")

# After the expensive coupled stage stops at FULL_ESCAPE_FACTOR * R_switch,
# continue the outgoing DM path to R_far without Coulomb backreaction. The ion
# follows its exact free harmonic evolution from the coupled escape state. The
# omitted Coulomb force is then integrated perturbatively on the same global
# Fourier timeline. This completes the force history from incoming R_far to
# outgoing R_far at low additional cost.
INCLUDE_OUTGOING_R_FAR_TAIL = bool(
    globals().get("USER_INCLUDE_OUTGOING_R_FAR_TAIL", True)
)
REQUIRE_OUTGOING_R_FAR_TAIL_COMPLETE = bool(
    globals().get("USER_REQUIRE_OUTGOING_R_FAR_TAIL_COMPLETE", True)
)
OUTGOING_TAIL_RTOL = float(
    globals().get("USER_OUTGOING_TAIL_RTOL", DM_ONLY_RTOL)
)
OUTGOING_TAIL_ATOL = float(
    globals().get("USER_OUTGOING_TAIL_ATOL", DM_ONLY_ATOL)
)
OUTGOING_TAIL_MAX_STEP_S = float(
    globals().get("USER_OUTGOING_TAIL_MAX_STEP_S", DM_ONLY_MAX_STEP_S)
)
OUTGOING_TAIL_TIME_FACTOR = float(
    globals().get("USER_OUTGOING_TAIL_TIME_FACTOR", 4.0)
)
OUTGOING_TAIL_MIN_TIME_S = float(
    globals().get("USER_OUTGOING_TAIL_MIN_TIME_S", 20.0e-6)
)
OUTGOING_TAIL_QUAD_EPSREL = float(
    globals().get("USER_OUTGOING_TAIL_QUAD_EPSREL", PREFIX_QUAD_EPSREL)
)
OUTGOING_TAIL_QUAD_EPSABS_DIMENSIONLESS = float(
    globals().get(
        "USER_OUTGOING_TAIL_QUAD_EPSABS_DIMENSIONLESS",
        PREFIX_QUAD_EPSABS_DIMENSIONLESS,
    )
)
OUTGOING_TAIL_QUAD_LIMIT = int(
    globals().get("USER_OUTGOING_TAIL_QUAD_LIMIT", PREFIX_QUAD_LIMIT)
)
OUTGOING_TAIL_FORCE_SCALE_SAMPLES = int(
    globals().get(
        "USER_OUTGOING_TAIL_FORCE_SCALE_SAMPLES", PREFIX_FORCE_SCALE_SAMPLES
    )
)
if OUTGOING_TAIL_FORCE_SCALE_SAMPLES < 3:
    raise ValueError("USER_OUTGOING_TAIL_FORCE_SCALE_SAMPLES must be at least 3")
FULL_TIME_FACTOR = 8.0
MIN_FULL_TIME_S = 20.0e-6
ENERGY_AVERAGE_WINDOW_S = 2.0e-6

USE_HARMONIC_ION = True
RUN_RUTHERFORD_FREE_SPACE = False
COULOMB_SOFTENING_M = 0.0

INVALID_Z_MARGIN_M = 1.0e-6
INVALID_Z_FLOOR_M: float | None = None

N_THREADS_DM_ONLY = 8
N_THREADS_FULL = 8
MAX_IN_FLIGHT_DM_ONLY = N_THREADS_DM_ONLY
MAX_IN_FLIGHT_FULL = N_THREADS_FULL
CHECKPOINT_EVERY = 8
RESUME_EXISTING_OUTPUT = False
SORT_FASTEST_FIRST = True
PRINT_EACH_TRAJECTORY_SUMMARY = True
PRINT_EACH_CLASS_REACH_PROBE = True
PRINT_CLASS_REACH_FINAL_SUMMARY = True
BATCH_HEARTBEAT_S: float | None = 30.0

ROW_QUERY: str | None = None
MAX_TRAJECTORIES: int | None = None
KEEP_SINGLE_TRAJECTORY_ARRAYS = False

# Per-class reach search. Fractions are in physical annular area:
# b(f) = sqrt(b_min^2 + f * (b_max^2 - b_min^2)). Low-b points are dense
# because they are most likely to enter R_switch. A class is rejected only if
# every planned probe is a resolved no-switch miss.
ENABLE_TRAJECTORY_CLASS_REACH_SCREEN = True
# Fractions are local to each exact Test 0.75 annulus. The lowest annulus is
# sampled very densely near its lower boundary; later disjoint annuli receive
# a lighter scan. No probe is ever placed in a gap between allowed annuli.
LOWEST_ANNULUS_LOCAL_B2_FRACTIONS = (
    0.0,
    1.0e-6,
    1.0e-5,
    1.0e-4,
    5.0e-4,
    1.0e-3,
    2.5e-3,
    5.0e-3,
    1.0e-2,
    2.0e-2,
    4.0e-2,
    8.0e-2,
    1.6e-1,
    3.2e-1,
    6.4e-1,
    1.0,
)
OTHER_ANNULUS_LOCAL_B2_FRACTIONS = (
    0.0, 0.01, 0.04, 0.16, 0.36, 0.64, 1.0,
)
CLASS_REACH_INCLUDE_PRODUCTION_B_VALUES = True
IMPACT_PARAMETER_RECONSTRUCTION_ATOL_M = 1.0e-12
IMPACT_PARAMETER_RECONSTRUCTION_RTOL = 1.0e-9
CLASS_REACH_STOP_AFTER_FIRST_ENTRY = True
N_THREADS_CLASS_REACH = 8
TRAJECTORY_CLASS_PSI_DECIMALS = 12

CLASS_REACH_PROBES_CSV = Path(f"{OUTPUT_PREFIX}_class_reach_probes.csv")
CLASS_REACH_SUMMARY_CSV = Path(f"{OUTPUT_PREFIX}_class_reach_summary.csv")
DM_ONLY_SCREEN_CSV = Path(f"{OUTPUT_PREFIX}_dm_only_screen.csv")
FULL_STAGE_RESULTS_CSV = Path(f"{OUTPUT_PREFIX}_full_stage_results.csv")
COMBINED_RESULTS_CSV = Path(f"{OUTPUT_PREFIX}_trajectory_results.csv")
WEIGHTED_ANALYSIS_CSV = Path(f"{OUTPUT_PREFIX}_weighted_analysis_results.csv")
WEIGHTED_GLOBAL_SUMMARY_CSV = Path(f"{OUTPUT_PREFIX}_weighted_global_summary.csv")
WEIGHTED_CLASSIFICATION_SUMMARY_CSV = Path(
    f"{OUTPUT_PREFIX}_weighted_classification_summary.csv"
)


# =============================================================================
# ENVIRONMENT AND DATA HELPERS
# =============================================================================

REQUIRED_TRAP_GLOBALS = ("FDC", "FRF", "m_ion", "Z_ion", "omega_vec", "K", "e")


def progress(message: str) -> None:
    print(message, flush=True)


def validate_trap_environment() -> None:
    missing = [name for name in REQUIRED_TRAP_GLOBALS if name not in globals()]
    if missing:
        raise RuntimeError(
            "Run the trap-model notebook cells first. Missing globals: "
            + ", ".join(missing)
        )


def resolve_invalid_z_floor() -> float | None:
    if INVALID_Z_FLOOR_M is not None:
        return float(INVALID_Z_FLOOR_M)
    c_obj = globals().get("c")
    if c_obj is not None and hasattr(c_obj, "ion_height"):
        return float(-c_obj.ion_height + INVALID_Z_MARGIN_M)
    return None


def numeric_value(row: dict[str, Any] | pd.Series, name: str, default=np.nan) -> float:
    try:
        value = float(row.get(name, default))
    except (TypeError, ValueError):
        return float(default)
    return value


def bool_value(row: dict[str, Any] | pd.Series, name: str, default=False) -> bool:
    value = row.get(name, default)
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}



def _test2_bowl_metrics_from_samples(
    times_s: np.ndarray,
    positions_m: np.ndarray,
    *,
    prefix: str,
) -> dict[str, Any]:
    helper = globals().get("bowl_metrics_from_samples")
    if helper is None:
        return {}
    try:
        return helper(times_s, positions_m, prefix=prefix)
    except Exception as exc:
        return {
            f"{prefix}_bowl_model_available": False,
            f"{prefix}_bowl_metric_error": f"{type(exc).__name__}:{exc}",
        }


def _test2_bowl_metrics_from_dense_solution(
    dense_solution: Callable[[float], np.ndarray] | None,
    duration_s: float,
    *,
    prefix: str,
    position_slice: slice = slice(0, 3),
) -> dict[str, Any]:
    helper = globals().get("bowl_metrics_from_dense_solution")
    if helper is None:
        return {}
    try:
        return helper(
            dense_solution,
            duration_s,
            prefix=prefix,
            position_slice=position_slice,
        )
    except Exception as exc:
        return {
            f"{prefix}_bowl_model_available": False,
            f"{prefix}_bowl_metric_error": f"{type(exc).__name__}:{exc}",
        }


def _test2_interval_metrics(
    times_s: np.ndarray,
    signed_values: np.ndarray,
) -> dict[str, Any]:
    helper = globals().get("interval_metrics_from_signed_values")
    if helper is not None:
        return helper(times_s, signed_values)
    # Fallback used only when the bowl geometry cell was skipped.
    times = np.asarray(times_s, dtype=float)
    signed = np.asarray(signed_values, dtype=float)
    inside = np.isfinite(signed) & (signed <= 0.0)
    if len(times) < 2:
        duration = 0.0
    else:
        duration = float(np.sum(np.diff(times) * (inside[:-1] | inside[1:])))
    return {
        "entered": bool(np.any(inside)),
        "crossed_inward": False,
        "crossed_outward": False,
        "started_inside": bool(inside[0]) if len(inside) else False,
        "ended_inside": bool(inside[-1]) if len(inside) else False,
        "time_inside_s": duration,
        "entry_time_s": np.nan,
        "exit_time_s": np.nan,
        "entry_count": 0,
        "exit_count": 0,
        "minimum_signed_value": float(np.nanmin(signed)) if np.any(np.isfinite(signed)) else np.nan,
        "valid_fraction": float(np.mean(np.isfinite(signed))) if len(signed) else 0.0,
    }


def _test2_combine_bowl_fields(row: dict[str, Any]) -> dict[str, Any]:
    helper = globals().get("combine_bowl_stage_metrics")
    if helper is None:
        return row
    try:
        return helper(row)
    except Exception as exc:
        output = dict(row)
        output["bowl_trajectory_class"] = f"bowl_classification_error:{type(exc).__name__}"
        output["bowl_classification_error"] = str(exc)
        return output


def bool_series(df: pd.DataFrame, column: str, default=False) -> pd.Series:
    if column not in df.columns:
        return pd.Series(default, index=df.index, dtype=bool)
    if pd.api.types.is_bool_dtype(df[column]):
        return df[column].fillna(default).astype(bool)
    return df[column].astype(str).str.strip().str.lower().isin(
        {"true", "1", "yes", "y"}
    )


def numeric_series(df: pd.DataFrame, column: str, default=np.nan) -> pd.Series:
    if column not in df.columns:
        return pd.Series(default, index=df.index, dtype=float)
    return pd.to_numeric(df[column], errors="coerce")



def resolve_input_path(path: Path, *, required: bool, label: str) -> Path | None:
    """Resolve paths that may already contain RUN_DIRECTORY."""
    path = Path(path)
    base = Path(RUN_DIRECTORY)
    candidates = [path] if path.is_absolute() else [path, base / path]
    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.is_file() and candidate.stat().st_size > 0:
            return candidate
    if required:
        raise FileNotFoundError(
            f"{label} not found or empty. Tried: "
            + ", ".join(str(candidate) for candidate in candidates)
        )
    return None


def read_csv_optional(path: Path | None) -> pd.DataFrame:
    if path is None:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, low_memory=False)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def load_test2_inputs() -> tuple[pd.DataFrame, Path, Path | None]:
    production_path = resolve_input_path(
        TEST1_INPUT_CSV, required=True, label="Test 1.5 input"
    )
    assert production_path is not None
    production = read_csv_optional(production_path)

    extra_path = None
    diagnostics = pd.DataFrame()
    if EXTRA_DIAGNOSTIC_INPUT_CSV is not None:
        extra_path = resolve_input_path(
            EXTRA_DIAGNOSTIC_INPUT_CSV,
            required=False,
            label="escape-channel diagnostic input",
        )
        diagnostics = read_csv_optional(extra_path)

    if len(diagnostics):
        for column in (
            "impact_weight",
            "population_weight_conditional",
            "population_weight_unconditional",
            "test2_analysis_weight_conditional",
            "test2_analysis_weight_unconditional",
        ):
            diagnostics[column] = 0.0
        diagnostics["test2_diagnostic_only"] = True
        diagnostics["probability_interpretation"] = (
            "diagnostic_only_requires_solid_angle_and_b_area_reweighting"
        )
        selected = pd.concat(
            [production, diagnostics], ignore_index=True, sort=False
        )
    else:
        selected = production.copy()

    if "trajectory_id" in selected.columns:
        selected["trajectory_id"] = selected["trajectory_id"].astype(str)
        selected = selected.drop_duplicates("trajectory_id", keep="last")
    return selected, production_path, extra_path


def infer_mass_column(df: pd.DataFrame) -> str:
    if "m_dm_kg" in df.columns:
        return "m_dm_kg"
    if "m_dm" in df.columns:
        return "m_dm"
    raise ValueError("Input must contain m_dm_kg or m_dm")


def recalculate_row_v_b_radii(row: dict[str, Any] | pd.Series) -> dict[str, Any]:
    """Return a row with R_full/R_switch recalculated from its exact v and b."""
    output = dict(row)
    mass_name = "m_dm_kg" if "m_dm_kg" in output else "m_dm"
    radii = rutherford_v_b_radius_policy(
        numeric_value(output, "v_inf_m_s"),
        numeric_value(output, "b_m"),
        m_dm_kg=numeric_value(output, mass_name),
        m_ion_kg=float(m_ion),
        eps_value=numeric_value(output, "eps"),
        ion_charge_number=float(Z_ion),
        coulomb_constant=float(K),
        elementary_charge_c=float(e),
        threshold_j=float(TARGET_ION_ENERGY_J),
    )
    output["r_min_energy_threshold_m"] = radii["r_min_energy_threshold_m"]
    output["r_min_threshold_speed_only_m"] = radii["r_min_threshold_speed_only_m"]
    output["r_min_rutherford_v_b_m"] = radii["r_min_rutherford_v_b_m"]
    output["r_min_threshold_m"] = radii["R_full_m"]
    output["R_full_m"] = radii["R_full_m"]
    output["R_switch_m"] = radii["R_switch_m"]
    output["R_switch_factor"] = radii["R_switch_factor"]
    output["R_full_v_b_mode"] = radii["R_full_v_b_mode"]
    output["rutherford_recoil_v_b_J"] = radii["rutherford_recoil_v_b_J"]
    output["rutherford_above_threshold_v_b"] = radii["rutherford_above_threshold_v_b"]
    output["R_full_um"] = float(radii["R_full_m"]) * 1.0e6
    output["R_switch_um"] = float(radii["R_switch_m"]) * 1.0e6
    return output


def infer_regime(row: dict[str, Any] | pd.Series) -> str:
    text = str(
        row.get("trajectory_regime", row.get("collision_regime", "resonant"))
    ).lower()
    if (
        "adiabatic" in text
        or bool_value(row, "test0p75_adiabatic_reject")
        or bool_value(row, "adiabatic_analytic_reject")
        or bool_value(row, "test1p5_analytic_reject")
    ):
        return "adiabatic"
    if "rutherford" in text or bool_value(row, "use_rutherford_after_reach"):
        return "rutherford"
    return "resonant"


def validate_input(df: pd.DataFrame) -> None:
    required = {
        "trajectory_id",
        "speed_label",
        "v_inf_m_s",
        "R_full_m",
        "R_switch_m",
        "b_m",
        "b_lower_m",
        "b_upper_m",
        "x_far_m",
        "y_far_m",
        "z_far_m",
        "vx_far_m_s",
        "vy_far_m_s",
        "vz_far_m_s",
        "eps",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError("Test 2 input is missing columns: " + ", ".join(missing))
    infer_mass_column(df)



def annotate_and_validate_b_support(df: pd.DataFrame) -> pd.DataFrame:
    """Validate exact Test 1 annulus bounds without generating new b values."""
    output = df.copy()
    b = numeric_series(output, "b_m")
    lower = numeric_series(output, "b_lower_m")
    upper = numeric_series(output, "b_upper_m")
    impact = numeric_series(output, "impact_weight", 0.0).fillna(0.0)
    weighted = impact > 0.0

    output["b_within_exported_row_bounds"] = (
        np.isfinite(b)
        & np.isfinite(lower)
        & np.isfinite(upper)
        & (lower >= 0.0)
        & (upper >= lower)
        & (b >= lower - 1.0e-15)
        & (b <= upper + 1.0e-15)
    )
    invalid_weighted = weighted & ~output["b_within_exported_row_bounds"]
    invalid_width = weighted & ~(upper > lower)
    invalid = invalid_weighted | invalid_width
    if invalid.any():
        examples = output.loc[
            invalid,
            ["trajectory_id", "b_m", "b_lower_m", "b_upper_m", "impact_weight"],
        ].head(10)
        raise ValueError(
            "Test 2 received weighted rows outside their Test 1 collision-regime "
            "impact-parameter annuli:\n" + examples.to_string(index=False)
        )
    output["b_lower_um"] = lower * 1.0e6
    output["b_upper_um"] = upper * 1.0e6
    output["b_sampling_source"] = "test1_collision_regime_annulus"
    return output




def _merge_annuli(intervals: Iterable[tuple[float, float]]) -> list[tuple[float, float]]:
    clean = sorted(
        (float(lower), float(upper))
        for lower, upper in intervals
        if np.isfinite(lower) and np.isfinite(upper) and 0.0 <= lower < upper
    )
    if not clean:
        return []
    merged: list[list[float]] = [[clean[0][0], clean[0][1]]]
    for lower, upper in clean[1:]:
        previous = merged[-1]
        tolerance = max(1.0e-15, 1.0e-12 * max(previous[1], upper, 1.0e-12))
        if lower <= previous[1] + tolerance:
            previous[1] = max(previous[1], upper)
        else:
            merged.append([lower, upper])
    return [(lower, upper) for lower, upper in merged]


def _parse_annuli_json(value: Any) -> list[tuple[float, float]]:
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return []
    try:
        raw = json.loads(str(value))
    except (TypeError, ValueError, json.JSONDecodeError):
        return []
    intervals: list[tuple[float, float]] = []
    if isinstance(raw, list):
        for item in raw:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                try:
                    intervals.append((float(item[0]), float(item[1])))
                except (TypeError, ValueError):
                    pass
    return _merge_annuli(intervals)


def _annuli_json_value(annuli: Iterable[tuple[float, float]]) -> str:
    return json.dumps(
        [[float(lower), float(upper)] for lower, upper in annuli],
        separators=(",", ":"),
    )


def _class_annuli_from_group(group: pd.DataFrame) -> list[tuple[float, float]]:
    for column in ("class_b_annuli_json", "physical_b_annuli_json"):
        if column in group.columns:
            for value in group[column].dropna():
                annuli = _parse_annuli_json(value)
                if annuli:
                    return annuli
    positive = numeric_series(group, "impact_weight", 0.0).fillna(0.0) > 0.0
    source = group.loc[positive] if positive.any() else group
    intervals = list(
        zip(
            numeric_series(source, "b_lower_m").to_numpy(float),
            numeric_series(source, "b_upper_m").to_numpy(float),
        )
    )
    return _merge_annuli(intervals)


def _containing_annulus(
    b_value: float,
    annuli: list[tuple[float, float]],
) -> tuple[int, float, float] | None:
    for index, (lower, upper) in enumerate(annuli):
        tolerance = max(1.0e-15, 1.0e-12 * max(upper, 1.0e-12))
        if lower - tolerance <= b_value <= upper + tolerance:
            return index, lower, upper
    return None

def trajectory_class_id_from_row(row: dict[str, Any] | pd.Series) -> str:
    existing = str(row.get("trajectory_class_id", "")).strip()
    if existing and existing.lower() != "nan":
        return existing
    speed = str(row.get("speed_label", "unknown"))
    regime = infer_regime(row)
    theta = int(numeric_value(row, "theta_index", -1))
    alpha = int(numeric_value(row, "alpha_index", -1))
    psi = round(numeric_value(row, "psi_rad", 0.0), TRAJECTORY_CLASS_PSI_DECIMALS)
    return (
        f"{speed}|{regime}|th{theta:03d}|al{alpha:03d}|"
        f"psi{psi:.{TRAJECTORY_CLASS_PSI_DECIMALS}f}"
    )



def add_trajectory_class_support(df: pd.DataFrame) -> pd.DataFrame:
    """Attach exact class-level b support, including for an empty input.

    Test 1.5 can legitimately export a header-only Test 2 input when every
    dynamic trajectory class was filtered out.  In that case ``class_bounds``
    has no records, so constructing it without an explicit schema produces a
    DataFrame with no ``trajectory_class_id`` column and the merge fails.
    """
    output = df.copy()
    output["trajectory_class_id"] = pd.Series(
        [trajectory_class_id_from_row(row) for _, row in output.iterrows()],
        index=output.index,
        dtype=str,
    )

    bound_columns = [
        "trajectory_class_id",
        "class_b_support_lower_m",
        "class_b_support_upper_m",
        "class_b_annuli_json",
        "class_b_annulus_count",
        "class_b_union_area2_m2",
    ]
    class_bounds: list[dict[str, Any]] = []
    for class_id, group in output.groupby("trajectory_class_id", sort=False):
        annuli = _class_annuli_from_group(group)
        if not annuli:
            raise ValueError(f"No valid Test 0.75 b annuli for class {class_id}")
        class_bounds.append(
            {
                "trajectory_class_id": class_id,
                "class_b_support_lower_m": annuli[0][0],
                "class_b_support_upper_m": annuli[-1][1],
                "class_b_annuli_json": _annuli_json_value(annuli),
                "class_b_annulus_count": len(annuli),
                "class_b_union_area2_m2": float(
                    sum(upper * upper - lower * lower for lower, upper in annuli)
                ),
            }
        )

    # Supplying columns is essential when class_bounds is empty.
    bounds = pd.DataFrame(class_bounds, columns=bound_columns)
    output = output.drop(
        columns=bound_columns[1:],
        errors="ignore",
    ).merge(
        bounds,
        on="trajectory_class_id",
        how="left",
        validate="many_to_one",
    )
    output["class_b_support_lower_um"] = numeric_series(
        output, "class_b_support_lower_m"
    ) * 1.0e6
    output["class_b_support_upper_um"] = numeric_series(
        output, "class_b_support_upper_m"
    ) * 1.0e6
    return output

def _fallback_impact_direction(velocity_hat: np.ndarray, psi_rad: float) -> np.ndarray:
    reference = np.array([0.0, 0.0, 1.0])
    if abs(float(np.dot(reference, velocity_hat))) > 0.9:
        reference = np.array([0.0, 1.0, 0.0])
    basis_1 = np.cross(velocity_hat, reference)
    basis_1 /= np.linalg.norm(basis_1)
    basis_2 = np.cross(velocity_hat, basis_1)
    direction = math.cos(psi_rad) * basis_1 + math.sin(psi_rad) * basis_2
    return direction / np.linalg.norm(direction)



def rebuild_row_at_b(
    template: dict[str, Any] | pd.Series,
    b_target_m: float,
    *,
    role: str,
    suffix: str,
) -> dict[str, Any]:
    row = dict(template)
    x_far = vector_from_row(row, ("x_far_m", "y_far_m", "z_far_m"))
    v_far = vector_from_row(row, ("vx_far_m_s", "vy_far_m_s", "vz_far_m_s"))
    speed = float(np.linalg.norm(v_far))
    radius = float(np.linalg.norm(x_far))
    if speed <= 0.0 or radius <= 0.0:
        raise ValueError("Cannot rebuild a probe from a zero far-state vector")
    if not (0.0 <= b_target_m < radius):
        raise ValueError(
            f"Requested b={b_target_m:.6e} m is outside far sphere radius "
            f"{radius:.6e} m"
        )

    velocity_hat = v_far / speed
    longitudinal = float(np.dot(x_far, velocity_hat))
    impact_vector = x_far - longitudinal * velocity_hat
    impact_norm = float(np.linalg.norm(impact_vector))
    if impact_norm > max(1.0e-15, 1.0e-12 * radius):
        impact_hat = impact_vector / impact_norm
    else:
        impact_hat = _fallback_impact_direction(
            velocity_hat,
            numeric_value(row, "psi_rad", 0.0),
        )

    incoming_longitudinal = -math.sqrt(
        max(radius * radius - b_target_m * b_target_m, 0.0)
    )
    rebuilt_position = incoming_longitudinal * velocity_hat + b_target_m * impact_hat
    reconstructed_b = float(np.linalg.norm(np.cross(rebuilt_position, velocity_hat)))
    reconstruction_error = reconstructed_b - float(b_target_m)
    if not np.isclose(
        reconstructed_b,
        b_target_m,
        rtol=IMPACT_PARAMETER_RECONSTRUCTION_RTOL,
        atol=IMPACT_PARAMETER_RECONSTRUCTION_ATOL_M,
    ):
        raise ValueError(
            "Impact-parameter reconstruction failed: "
            f"requested={b_target_m:.12e} m, reconstructed={reconstructed_b:.12e} m"
        )

    class_id = trajectory_class_id_from_row(row)
    row.update(
        {
            "x_far_m": rebuilt_position[0],
            "y_far_m": rebuilt_position[1],
            "z_far_m": rebuilt_position[2],
            "vx_far_m_s": v_far[0],
            "vy_far_m_s": v_far[1],
            "vz_far_m_s": v_far[2],
            "b_m": float(b_target_m),
            "b_requested_m": float(b_target_m),
            "b_requested_um": float(b_target_m) * 1.0e6,
            "b_reconstructed_m": reconstructed_b,
            "b_reconstructed_um": reconstructed_b * 1.0e6,
            "b_reconstruction_error_m": reconstruction_error,
            "trajectory_id": f"{class_id}|{suffix}",
            "trajectory_class_id": class_id,
            "test2_sample_role": role,
            "trajectory_class_probe": role == "trajectory_class_reach_probe",
        }
    )
    row = recalculate_row_v_b_radii(row)
    for column in (
        "impact_weight",
        "population_weight_conditional",
        "population_weight_unconditional",
        "test2_analysis_weight_conditional",
        "test2_analysis_weight_unconditional",
    ):
        if column in row:
            row[f"probe_original_{column}"] = row[column]
        row[column] = 0.0
    return row


def build_class_probe_rows(class_group: pd.DataFrame) -> list[dict[str, Any]]:
    """Build zero-weight probes only inside exact surviving Test 0.75 annuli."""
    class_id = str(class_group["trajectory_class_id"].iloc[0])
    annuli = _class_annuli_from_group(class_group)
    if not annuli:
        raise ValueError(f"No valid physical b annuli for {class_id}")

    template = class_group.sort_values("b_m", kind="mergesort").iloc[0]
    widths2 = [upper * upper - lower * lower for lower, upper in annuli]
    total_width2 = float(sum(widths2))
    candidates: list[dict[str, Any]] = []
    cumulative2 = 0.0

    for annulus_index, ((lower, upper), width2) in enumerate(zip(annuli, widths2)):
        fractions = (
            LOWEST_ANNULUS_LOCAL_B2_FRACTIONS
            if annulus_index == 0
            else OTHER_ANNULUS_LOCAL_B2_FRACTIONS
        )
        for local_fraction in fractions:
            local_fraction = float(np.clip(local_fraction, 0.0, 1.0))
            b_target = math.sqrt(lower * lower + local_fraction * width2)
            union_fraction = (
                (cumulative2 + local_fraction * width2) / total_width2
                if total_width2 > 0.0
                else 0.0
            )
            candidates.append(
                {
                    "b_m": b_target,
                    "source": "lowest_annulus_dense" if annulus_index == 0 else "later_annulus_scan",
                    "annulus_index": annulus_index,
                    "annulus_lower_m": lower,
                    "annulus_upper_m": upper,
                    "local_b2_fraction": local_fraction,
                    "union_b2_fraction": union_fraction,
                }
            )
        cumulative2 += width2

    if CLASS_REACH_INCLUDE_PRODUCTION_B_VALUES:
        production_b = pd.to_numeric(class_group["b_m"], errors="coerce").to_numpy(float)
        cumulative_before = np.cumsum([0.0] + widths2[:-1])
        for b_target in production_b[np.isfinite(production_b)]:
            containing = _containing_annulus(float(b_target), annuli)
            if containing is None:
                continue
            annulus_index, lower, upper = containing
            width2 = widths2[annulus_index]
            local_fraction = (
                (b_target * b_target - lower * lower) / width2 if width2 > 0.0 else 0.0
            )
            union_fraction = (
                (cumulative_before[annulus_index] + local_fraction * width2) / total_width2
                if total_width2 > 0.0
                else 0.0
            )
            candidates.append(
                {
                    "b_m": float(np.clip(b_target, lower, upper)),
                    "source": "production_representative",
                    "annulus_index": annulus_index,
                    "annulus_lower_m": lower,
                    "annulus_upper_m": upper,
                    "local_b2_fraction": float(np.clip(local_fraction, 0.0, 1.0)),
                    "union_b2_fraction": float(np.clip(union_fraction, 0.0, 1.0)),
                }
            )

    candidates.sort(key=lambda item: float(item["b_m"]))
    unique: list[dict[str, Any]] = []
    for candidate in candidates:
        b_target = float(candidate["b_m"])
        if unique and np.isclose(
            b_target,
            float(unique[-1]["b_m"]),
            rtol=0.0,
            atol=max(1.0e-15, 1.0e-12 * max(float(candidate["annulus_upper_m"]), 1.0e-12)),
        ):
            if candidate["source"] == "production_representative":
                unique[-1]["source"] += "+production"
            continue
        unique.append(candidate)

    rows: list[dict[str, Any]] = []
    annuli_json = _annuli_json_value(annuli)
    for index, candidate in enumerate(unique):
        b_target = float(candidate["b_m"])
        probe = rebuild_row_at_b(
            template,
            b_target,
            role="trajectory_class_reach_probe",
            suffix=f"reachprobe{index:03d}",
        )
        probe["b_lower_m"] = float(candidate["annulus_lower_m"])
        probe["b_upper_m"] = float(candidate["annulus_upper_m"])
        probe["physical_b_support_lower_m"] = annuli[0][0]
        probe["physical_b_support_upper_m"] = annuli[-1][1]
        probe["physical_b_annuli_json"] = annuli_json
        probe["class_b_annuli_json"] = annuli_json
        probe["class_probe_annulus_index"] = int(candidate["annulus_index"])
        probe["class_probe_local_b2_fraction"] = float(candidate["local_b2_fraction"])
        probe["class_probe_union_b2_fraction"] = float(candidate["union_b2_fraction"])
        probe["class_probe_b2_fraction"] = float(candidate["union_b2_fraction"])
        probe["class_probe_source"] = str(candidate["source"])
        probe["class_probe_index"] = index
        probe["class_probe_support_lower_m"] = float(candidate["annulus_lower_m"])
        probe["class_probe_support_upper_m"] = float(candidate["annulus_upper_m"])
        probe["class_probe_support_lower_um"] = float(candidate["annulus_lower_m"]) * 1.0e6
        probe["class_probe_support_upper_um"] = float(candidate["annulus_upper_m"]) * 1.0e6
        probe["class_probe_plan_size"] = len(unique)
        probe["trajectory_class_id"] = class_id
        rows.append(probe)
    return rows


def class_probe_summary(
    class_id: str,
    probe_number: int,
    probe_total: int,
    row: dict[str, Any],
) -> str:
    """Return one flushed result line for an exact-annulus reach probe."""
    status = str(row.get("dm_only_status", ""))
    entered = bool_value(row, "entered_switch")
    b_um = numeric_value(row, "b_m") * 1.0e6
    b_reconstructed_um = numeric_value(row, "b_reconstructed_um")
    r_switch_um = numeric_value(row, "R_switch_m") * 1.0e6
    r_min_um = numeric_value(row, "dm_only_min_radius_um")
    runtime_s = numeric_value(row, "dm_only_runtime_s")
    local_fraction = numeric_value(row, "class_probe_local_b2_fraction")
    annulus_index = int(numeric_value(row, "class_probe_annulus_index", -1))
    source = str(row.get("class_probe_source", ""))
    return (
        f"[CLASS-PROBE {probe_number:3d}/{probe_total:3d}] "
        f"class={class_id} annulus={annulus_index} "
        f"b={b_um:.6f}um brecon={b_reconstructed_um:.6f}um "
        f"local_b2frac={local_fraction:.6g} "
        f"Rswitch={r_switch_um:.3f}um "
        f"switch={'Y' if entered else 'N'} "
        f"rmin={r_min_um:.3f}um "
        f"status={status} source={source} runtime={runtime_s:.3f}s"
    )

def class_reach_worker(
    payload: tuple[str, list[dict[str, Any]]],
    invalid_z_floor_m: float | None,
) -> dict[str, Any]:
    class_id, class_rows = payload
    group = pd.DataFrame(class_rows)
    try:
        probes = build_class_probe_rows(group)
        results: list[dict[str, Any]] = []
        probe_total = len(probes)
        for probe_number, probe in enumerate(probes, start=1):
            result = dm_only_worker(probe, invalid_z_floor_m)
            results.append(result)
            if PRINT_EACH_CLASS_REACH_PROBE:
                progress(
                    class_probe_summary(
                        class_id, probe_number, probe_total, result
                    )
                )
            if CLASS_REACH_STOP_AFTER_FIRST_ENTRY and bool_value(result, "entered_switch"):
                break
    except Exception as exc:
        probes = []
        results = []
        return {
            "summary": {
                "trajectory_class_id": class_id,
                "trajectory_class_reach_status": "unresolved_class_reach_screen",
                "class_reach_accepted": False,
                "class_reach_rejected": False,
                "class_reach_unresolved": True,
                "class_reach_n_planned": 0,
                "class_reach_n_run": 0,
                "class_reach_n_entered": 0,
                "class_reach_n_resolved_miss": 0,
                "class_reach_message": repr(exc),
                "class_reach_support_lower_m": float(group["class_b_support_lower_m"].iloc[0]),
                "class_reach_support_upper_m": float(group["class_b_support_upper_m"].iloc[0]),
            },
            "probes": [],
            "anchor": None,
        }

    entered = [row for row in results if bool_value(row, "entered_switch")]
    resolved_miss = [
        row for row in results
        if str(row.get("dm_only_status", "")) == "dm_only_escaped_without_switch"
    ]
    if entered:
        status = "accepted_has_switch_entry"
    elif len(results) == len(probes) and len(resolved_miss) == len(results):
        status = "rejected_no_in_range_sample_reaches_switch"
    else:
        status = "unresolved_class_reach_screen"

    minimum_radii = np.array(
        [numeric_value(row, "dm_only_min_radius_m") for row in results],
        dtype=float,
    )
    finite_minimum = minimum_radii[np.isfinite(minimum_radii)]
    representative = group.iloc[0]
    summary = {
        "trajectory_class_id": class_id,
        "speed_label": str(representative.get("speed_label", "")),
        "trajectory_regime": infer_regime(representative),
        "theta_index": int(numeric_value(representative, "theta_index", -1)),
        "alpha_index": int(numeric_value(representative, "alpha_index", -1)),
        "theta_rad": numeric_value(representative, "theta_rad"),
        "alpha_rad": numeric_value(representative, "alpha_rad"),
        "psi_rad": numeric_value(representative, "psi_rad"),
        "R_switch_m": numeric_value(representative, "R_switch_m"),
        "trajectory_class_reach_status": status,
        "class_reach_accepted": status == "accepted_has_switch_entry",
        "class_reach_rejected": status == "rejected_no_in_range_sample_reaches_switch",
        "class_reach_unresolved": status == "unresolved_class_reach_screen",
        "class_reach_n_planned": len(probes),
        "class_reach_n_run": len(results),
        "class_reach_n_entered": len(entered),
        "class_reach_n_resolved_miss": len(resolved_miss),
        "class_reach_sampled_b_um": ";".join(
            f"{numeric_value(row, 'b_m') * 1.0e6:.6g}" for row in results
        ),
        "class_reach_first_entering_b_m": numeric_value(entered[0], "b_m") if entered else np.nan,
        "class_reach_first_entering_b_um": (
            numeric_value(entered[0], "b_m") * 1.0e6 if entered else np.nan
        ),
        "class_reach_min_sampled_radius_m": (
            float(np.min(finite_minimum)) if finite_minimum.size else np.nan
        ),
        "class_reach_min_sampled_radius_um": (
            float(np.min(finite_minimum)) * 1.0e6 if finite_minimum.size else np.nan
        ),
        "class_reach_support_lower_m": float(group["class_b_support_lower_m"].iloc[0]),
        "class_reach_support_upper_m": float(group["class_b_support_upper_m"].iloc[0]),
        "class_reach_annuli_json": str(group["class_b_annuli_json"].iloc[0]),
        "class_reach_annulus_count": int(group["class_b_annulus_count"].iloc[0]),
        "class_reach_rejection_basis": (
            "all_configured_exact_annulus_probes_resolved_miss"
            if status == "rejected_no_in_range_sample_reaches_switch"
            else ""
        ),
        "class_reach_message": "",
    }
    return {"summary": summary, "probes": results, "anchor": entered[0] if entered else None}


def run_class_reach_screen(
    selected: pd.DataFrame,
    invalid_z_floor_m: float | None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not ENABLE_TRAJECTORY_CLASS_REACH_SCREEN:
        summaries = []
        for class_id, group in selected.groupby("trajectory_class_id", sort=False):
            summaries.append(
                {
                    "trajectory_class_id": class_id,
                    "trajectory_class_reach_status": "screen_disabled",
                    "class_reach_accepted": True,
                    "class_reach_rejected": False,
                    "class_reach_unresolved": False,
                    "class_reach_n_planned": 0,
                    "class_reach_n_run": 0,
                    "class_reach_n_entered": 0,
                    "class_reach_n_resolved_miss": 0,
                    "class_reach_support_lower_m": float(group["class_b_support_lower_m"].iloc[0]),
                    "class_reach_support_upper_m": float(group["class_b_support_upper_m"].iloc[0]),
                }
            )
        return pd.DataFrame(), pd.DataFrame(summaries), pd.DataFrame()

    payloads = [
        (str(class_id), group.to_dict("records"))
        for class_id, group in selected.groupby("trajectory_class_id", sort=True)
    ]
    all_probes: list[dict[str, Any]] = []
    summaries: list[dict[str, Any]] = []
    anchors: list[dict[str, Any]] = []
    total = len(payloads)

    with ThreadPoolExecutor(max_workers=N_THREADS_CLASS_REACH) as executor:
        future_map = {
            executor.submit(class_reach_worker, payload, invalid_z_floor_m): payload[0]
            for payload in payloads
        }
        for completed, future in enumerate(as_completed(future_map), start=1):
            result = future.result()
            summaries.append(result["summary"])
            all_probes.extend(result["probes"])
            if result["anchor"] is not None:
                anchors.append(result["anchor"])
            summary = result["summary"]
            if PRINT_CLASS_REACH_FINAL_SUMMARY:
                progress(
                    f"[CLASS-REACH {completed:4d}/{total:4d}] "
                    f"{summary['trajectory_class_reach_status']} "
                    f"id={summary['trajectory_class_id']} "
                    f"run={summary['class_reach_n_run']}/{summary['class_reach_n_planned']} "
                    f"entered={summary['class_reach_n_entered']} "
                    f"resolved_miss={summary['class_reach_n_resolved_miss']} "
                    f"first_b={summary['class_reach_first_entering_b_um']:.3f}um"
                )
            if completed % CHECKPOINT_EVERY == 0 or completed == total:
                pd.DataFrame(all_probes).to_csv(CLASS_REACH_PROBES_CSV, index=False)
                pd.DataFrame(summaries).to_csv(CLASS_REACH_SUMMARY_CSV, index=False)

    return pd.DataFrame(all_probes), pd.DataFrame(summaries), pd.DataFrame(anchors)


def make_class_rejection_stubs(rows: pd.DataFrame) -> pd.DataFrame:
    stubs: list[dict[str, Any]] = []
    for record in rows.to_dict("records"):
        output = dict(record)
        output.update(DM_ONLY_DEFAULTS)
        output.update(FULL_DEFAULTS)
        output.update(
            {
                "dm_only_success": True,
                "dm_only_status": "trajectory_class_rejected_no_in_range_sample_reaches_switch",
                "dm_only_message": (
                    "All planned class-level b probes inside the physical support "
                    "were resolved no-switch misses"
                ),
                "entered_switch": False,
                "full_success": True,
                "full_status": "trajectory_class_rejected_no_in_range_sample_reaches_switch",
                "full_message": "Full propagation skipped after class reach rejection",
                "reached_R_full": False,
                "above_energy_threshold": False,
            }
        )
        stubs.append(output)
    return pd.DataFrame(stubs)


def vector_from_row(row: dict[str, Any], names: Iterable[str]) -> np.ndarray:
    vector = np.array([numeric_value(row, name) for name in names], dtype=float)
    if vector.shape != (3,) or not np.all(np.isfinite(vector)):
        raise ValueError(f"Invalid vector columns: {tuple(names)}")
    return vector


def trap_force(points: np.ndarray, mass: float, charge: float) -> np.ndarray:
    points = np.asarray(points, dtype=float).reshape(-1, 3)
    if RUN_RUTHERFORD_FREE_SPACE:
        return np.zeros_like(points)
    return (
        np.asarray(FDC(points, mass, charge), dtype=float).reshape(-1, 3)
        + np.asarray(FRF(points, mass, charge), dtype=float).reshape(-1, 3)
    )


def append_event_state(sol: Any, event_index: int) -> tuple[np.ndarray, np.ndarray]:
    times = np.asarray(sol.t, dtype=float)
    states = np.asarray(sol.y, dtype=float)
    if (
        sol.t_events is None
        or len(sol.t_events) <= event_index
        or len(sol.t_events[event_index]) == 0
    ):
        return times, states
    event_time = float(sol.t_events[event_index][0])
    event_state = np.asarray(sol.y_events[event_index][0], dtype=float)
    if times.size == 0 or not np.isclose(times[-1], event_time):
        times = np.append(times, event_time)
        states = np.column_stack([states, event_state])
    return times, states


def sampled_min_radius(states: np.ndarray, position_slice: slice) -> float:
    try:
        position = np.asarray(states[position_slice], dtype=float)
        radius = np.linalg.norm(position, axis=0)
        finite = radius[np.isfinite(radius)]
        return float(np.min(finite)) if finite.size else np.nan
    except Exception:
        return np.nan


def radial_velocity(position: np.ndarray, velocity: np.ndarray) -> float:
    radius = float(np.linalg.norm(position))
    if radius <= 0.0:
        return 0.0
    return float(np.dot(position, velocity) / radius)


# =============================================================================
# DM-ONLY REACH SCREEN
# =============================================================================


def dm_only_rhs(m_dm_value: float, eps_value: float) -> Callable[[float, np.ndarray], np.ndarray]:
    def rhs(_time: float, state: np.ndarray) -> np.ndarray:
        position = state[:3]
        velocity = state[3:6]
        acceleration = trap_force(position[None, :], m_dm_value, eps_value)[0] / m_dm_value
        return np.concatenate([velocity, acceleration])

    return rhs


def fixed_origin_coulomb_force_on_ion(
    dm_position_m: np.ndarray,
    eps_value: float,
) -> np.ndarray:
    """Coulomb force on an ion fixed at the unperturbed trap origin."""
    dm_position = np.asarray(dm_position_m, dtype=float).reshape(3)
    delta = -dm_position
    radius2 = max(
        float(np.dot(delta, delta)) + float(COULOMB_SOFTENING_M) ** 2,
        np.finfo(float).tiny,
    )
    return (
        float(K)
        * float(Z_ion)
        * float(eps_value)
        * float(e) ** 2
        * delta
        * radius2 ** (-1.5)
    )


def empty_fourier_prefix_fields(
    *,
    status: str,
    complete: bool,
    duration_s: float = 0.0,
) -> dict[str, Any]:
    return {
        "fourier_prefix_complete": bool(complete),
        "fourier_prefix_status": str(status),
        "fourier_prefix_duration_s": float(duration_s),
        "fourier_prefix_time_origin": "incoming_R_switch_crossing",
        "fourier_prefix_cos_x_N_s": 0.0,
        "fourier_prefix_cos_y_N_s": 0.0,
        "fourier_prefix_cos_z_N_s": 0.0,
        "fourier_prefix_sin_x_N_s": 0.0,
        "fourier_prefix_sin_y_N_s": 0.0,
        "fourier_prefix_sin_z_N_s": 0.0,
        "fourier_prefix_quad_abs_error_x_N_s": np.nan,
        "fourier_prefix_quad_abs_error_y_N_s": np.nan,
        "fourier_prefix_quad_abs_error_z_N_s": np.nan,
        "E_fourier_prefix_standalone_J": 0.0,
        "E_fourier_prefix_x_J": 0.0,
        "E_fourier_prefix_y_J": 0.0,
        "E_fourier_prefix_z_J": 0.0,
    }


def integrate_fourier_prefix_from_dense_dm_path(
    dense_solution: Callable[[float], np.ndarray],
    duration_s: float,
    eps_value: float,
) -> dict[str, Any]:
    """Integrate R_far -> R_switch force on a switch-centered timeline.

    Local DM-only time is 0 <= u <= duration. The coupled stage starts at
    t=0, so the global Fourier time for the prefix is t=u-duration.
    """
    duration = float(duration_s)
    if not np.isfinite(duration) or duration < 0.0:
        return empty_fourier_prefix_fields(
            status="invalid_prefix_duration", complete=False, duration_s=duration
        )
    if duration == 0.0:
        return empty_fourier_prefix_fields(
            status="zero_length_R_far_to_R_switch", complete=True, duration_s=0.0
        )

    omega_modes = np.asarray(omega_vec, dtype=float).reshape(3)
    sample_times = np.linspace(0.0, duration, PREFIX_FORCE_SCALE_SAMPLES)
    sample_force_norms = []
    for sample_time in sample_times:
        sample_state = np.asarray(dense_solution(float(sample_time)), dtype=float)
        sample_force_norms.append(
            float(np.linalg.norm(fixed_origin_coulomb_force_on_ion(
                sample_state[:3], eps_value
            )))
        )
    force_scale = max(
        max(sample_force_norms, default=0.0),
        np.finfo(float).tiny,
    )

    amplitude = np.zeros(3, dtype=complex)
    absolute_error = np.zeros(3, dtype=float)
    for mode_index, omega in enumerate(omega_modes):
        def normalized_component(unit_time: float) -> float:
            local_time = float(unit_time) * duration
            state = np.asarray(dense_solution(local_time), dtype=float)
            force = fixed_origin_coulomb_force_on_ion(state[:3], eps_value)
            return float(force[mode_index] / force_scale)

        scaled_omega = float(omega) * duration
        if abs(scaled_omega) <= 1.0e-14:
            cosine_value, cosine_error = quad(
                normalized_component,
                0.0,
                1.0,
                epsabs=PREFIX_QUAD_EPSABS_DIMENSIONLESS,
                epsrel=PREFIX_QUAD_EPSREL,
                limit=PREFIX_QUAD_LIMIT,
            )
            sine_value, sine_error = 0.0, 0.0
        else:
            cosine_value, cosine_error = quad(
                normalized_component,
                0.0,
                1.0,
                weight="cos",
                wvar=scaled_omega,
                epsabs=PREFIX_QUAD_EPSABS_DIMENSIONLESS,
                epsrel=PREFIX_QUAD_EPSREL,
                limit=PREFIX_QUAD_LIMIT,
            )
            sine_value, sine_error = quad(
                normalized_component,
                0.0,
                1.0,
                weight="sin",
                wvar=scaled_omega,
                epsabs=PREFIX_QUAD_EPSABS_DIMENSIONLESS,
                epsrel=PREFIX_QUAD_EPSREL,
                limit=PREFIX_QUAD_LIMIT,
            )

        local_amplitude = (
            complex(cosine_value, sine_value) * force_scale * duration
        )
        amplitude[mode_index] = (
            local_amplitude * np.exp(-1j * float(omega) * duration)
        )
        absolute_error[mode_index] = (
            math.hypot(float(cosine_error), float(sine_error))
            * force_scale
            * duration
        )

    energy_modes = np.abs(amplitude) ** 2 / (2.0 * float(m_ion))
    return {
        "fourier_prefix_complete": True,
        "fourier_prefix_status": "integrated_dense_R_far_to_R_switch",
        "fourier_prefix_duration_s": duration,
        "fourier_prefix_time_origin": "incoming_R_switch_crossing",
        "fourier_prefix_cos_x_N_s": float(amplitude[0].real),
        "fourier_prefix_cos_y_N_s": float(amplitude[1].real),
        "fourier_prefix_cos_z_N_s": float(amplitude[2].real),
        "fourier_prefix_sin_x_N_s": float(amplitude[0].imag),
        "fourier_prefix_sin_y_N_s": float(amplitude[1].imag),
        "fourier_prefix_sin_z_N_s": float(amplitude[2].imag),
        "fourier_prefix_quad_abs_error_x_N_s": float(absolute_error[0]),
        "fourier_prefix_quad_abs_error_y_N_s": float(absolute_error[1]),
        "fourier_prefix_quad_abs_error_z_N_s": float(absolute_error[2]),
        "E_fourier_prefix_standalone_J": float(np.sum(energy_modes)),
        "E_fourier_prefix_x_J": float(energy_modes[0]),
        "E_fourier_prefix_y_J": float(energy_modes[1]),
        "E_fourier_prefix_z_J": float(energy_modes[2]),
    }



def empty_fourier_outgoing_tail_fields(
    *,
    status: str,
    complete: bool,
    duration_s: float = 0.0,
    end_radius_m: float = np.nan,
) -> dict[str, Any]:
    return {
        "fourier_outgoing_tail_complete": bool(complete),
        "fourier_outgoing_tail_status": str(status),
        "fourier_outgoing_tail_duration_s": float(duration_s),
        "fourier_outgoing_tail_end_radius_m": float(end_radius_m),
        "fourier_outgoing_tail_time_origin": "incoming_R_switch_crossing",
        "fourier_outgoing_tail_path_model": (
            "DM_trap_only_plus_free_harmonic_ion_no_Coulomb_backreaction"
        ),
        "fourier_outgoing_tail_cos_x_N_s": 0.0,
        "fourier_outgoing_tail_cos_y_N_s": 0.0,
        "fourier_outgoing_tail_cos_z_N_s": 0.0,
        "fourier_outgoing_tail_sin_x_N_s": 0.0,
        "fourier_outgoing_tail_sin_y_N_s": 0.0,
        "fourier_outgoing_tail_sin_z_N_s": 0.0,
        "fourier_outgoing_tail_quad_abs_error_x_N_s": np.nan,
        "fourier_outgoing_tail_quad_abs_error_y_N_s": np.nan,
        "fourier_outgoing_tail_quad_abs_error_z_N_s": np.nan,
        "E_fourier_outgoing_tail_standalone_J": 0.0,
        "E_fourier_outgoing_tail_x_J": 0.0,
        "E_fourier_outgoing_tail_y_J": 0.0,
        "E_fourier_outgoing_tail_z_J": 0.0,
    }


def free_harmonic_ion_position(
    local_time_s: float,
    x0_m: np.ndarray,
    v0_m_s: np.ndarray,
) -> np.ndarray:
    """Exact unforced harmonic-ion position after the coupled handoff."""
    local_time = float(local_time_s)
    x0 = np.asarray(x0_m, dtype=float).reshape(3)
    v0 = np.asarray(v0_m_s, dtype=float).reshape(3)
    omega = np.asarray(omega_vec, dtype=float).reshape(3)
    position = np.empty(3, dtype=float)
    oscillatory = np.abs(omega) > 0.0
    phase = omega[oscillatory] * local_time
    position[oscillatory] = (
        x0[oscillatory] * np.cos(phase)
        + v0[oscillatory] * np.sin(phase) / omega[oscillatory]
    )
    position[~oscillatory] = x0[~oscillatory] + v0[~oscillatory] * local_time
    return position


def coulomb_force_on_ion_from_positions(
    ion_position_m: np.ndarray,
    dm_position_m: np.ndarray,
    eps_value: float,
) -> np.ndarray:
    delta = np.asarray(ion_position_m, dtype=float).reshape(3) - np.asarray(
        dm_position_m, dtype=float
    ).reshape(3)
    radius2 = max(
        float(np.dot(delta, delta)) + float(COULOMB_SOFTENING_M) ** 2,
        np.finfo(float).tiny,
    )
    return (
        float(K)
        * float(Z_ion)
        * float(eps_value)
        * float(e) ** 2
        * delta
        * radius2 ** (-1.5)
    )


def integrate_fourier_outgoing_tail_from_dense_dm_path(
    dense_dm_solution: Callable[[float], np.ndarray],
    duration_s: float,
    coupled_end_time_s: float,
    x_ion_escape_m: np.ndarray,
    v_ion_escape_m_s: np.ndarray,
    eps_value: float,
    *,
    end_radius_m: float,
) -> dict[str, Any]:
    """Integrate the outgoing escape->R_far force on the global timeline."""
    duration = float(duration_s)
    coupled_end_time = float(coupled_end_time_s)
    if not np.isfinite(duration) or duration < 0.0:
        return empty_fourier_outgoing_tail_fields(
            status="invalid_outgoing_tail_duration",
            complete=False,
            duration_s=duration,
            end_radius_m=end_radius_m,
        )
    if duration == 0.0:
        return empty_fourier_outgoing_tail_fields(
            status="zero_length_escape_to_R_far",
            complete=True,
            duration_s=0.0,
            end_radius_m=end_radius_m,
        )

    omega_modes = np.asarray(omega_vec, dtype=float).reshape(3)

    def force_at(local_time_s: float) -> np.ndarray:
        dm_state = np.asarray(dense_dm_solution(float(local_time_s)), dtype=float)
        x_ion = free_harmonic_ion_position(
            local_time_s, x_ion_escape_m, v_ion_escape_m_s
        )
        return coulomb_force_on_ion_from_positions(
            x_ion, dm_state[:3], eps_value
        )

    sample_times = np.linspace(
        0.0, duration, OUTGOING_TAIL_FORCE_SCALE_SAMPLES
    )
    force_scale = max(
        max((float(np.linalg.norm(force_at(t))) for t in sample_times), default=0.0),
        np.finfo(float).tiny,
    )

    amplitude = np.zeros(3, dtype=complex)
    absolute_error = np.zeros(3, dtype=float)
    for mode_index, omega in enumerate(omega_modes):
        def normalized_component(unit_time: float) -> float:
            local_time = float(unit_time) * duration
            return float(force_at(local_time)[mode_index] / force_scale)

        scaled_omega = float(omega) * duration
        if abs(scaled_omega) <= 1.0e-14:
            cosine_value, cosine_error = quad(
                normalized_component,
                0.0,
                1.0,
                epsabs=OUTGOING_TAIL_QUAD_EPSABS_DIMENSIONLESS,
                epsrel=OUTGOING_TAIL_QUAD_EPSREL,
                limit=OUTGOING_TAIL_QUAD_LIMIT,
            )
            sine_value, sine_error = 0.0, 0.0
        else:
            cosine_value, cosine_error = quad(
                normalized_component,
                0.0,
                1.0,
                weight="cos",
                wvar=scaled_omega,
                epsabs=OUTGOING_TAIL_QUAD_EPSABS_DIMENSIONLESS,
                epsrel=OUTGOING_TAIL_QUAD_EPSREL,
                limit=OUTGOING_TAIL_QUAD_LIMIT,
            )
            sine_value, sine_error = quad(
                normalized_component,
                0.0,
                1.0,
                weight="sin",
                wvar=scaled_omega,
                epsabs=OUTGOING_TAIL_QUAD_EPSABS_DIMENSIONLESS,
                epsrel=OUTGOING_TAIL_QUAD_EPSREL,
                limit=OUTGOING_TAIL_QUAD_LIMIT,
            )

        local_amplitude = (
            complex(cosine_value, sine_value) * force_scale * duration
        )
        amplitude[mode_index] = local_amplitude * np.exp(
            1j * float(omega) * coupled_end_time
        )
        absolute_error[mode_index] = (
            math.hypot(float(cosine_error), float(sine_error))
            * force_scale
            * duration
        )

    energy_modes = np.abs(amplitude) ** 2 / (2.0 * float(m_ion))
    return {
        "fourier_outgoing_tail_complete": True,
        "fourier_outgoing_tail_status": "integrated_escape_to_outgoing_R_far",
        "fourier_outgoing_tail_duration_s": duration,
        "fourier_outgoing_tail_end_radius_m": float(end_radius_m),
        "fourier_outgoing_tail_time_origin": "incoming_R_switch_crossing",
        "fourier_outgoing_tail_path_model": (
            "DM_trap_only_plus_free_harmonic_ion_no_Coulomb_backreaction"
        ),
        "fourier_outgoing_tail_cos_x_N_s": float(amplitude[0].real),
        "fourier_outgoing_tail_cos_y_N_s": float(amplitude[1].real),
        "fourier_outgoing_tail_cos_z_N_s": float(amplitude[2].real),
        "fourier_outgoing_tail_sin_x_N_s": float(amplitude[0].imag),
        "fourier_outgoing_tail_sin_y_N_s": float(amplitude[1].imag),
        "fourier_outgoing_tail_sin_z_N_s": float(amplitude[2].imag),
        "fourier_outgoing_tail_quad_abs_error_x_N_s": float(absolute_error[0]),
        "fourier_outgoing_tail_quad_abs_error_y_N_s": float(absolute_error[1]),
        "fourier_outgoing_tail_quad_abs_error_z_N_s": float(absolute_error[2]),
        "E_fourier_outgoing_tail_standalone_J": float(np.sum(energy_modes)),
        "E_fourier_outgoing_tail_x_J": float(energy_modes[0]),
        "E_fourier_outgoing_tail_y_J": float(energy_modes[1]),
        "E_fourier_outgoing_tail_z_J": float(energy_modes[2]),
    }


def run_outgoing_dm_tail_to_r_far(
    x_ion_escape_m: np.ndarray,
    v_ion_escape_m_s: np.ndarray,
    x_dm_escape_m: np.ndarray,
    v_dm_escape_m_s: np.ndarray,
    *,
    coupled_end_time_s: float,
    r_far_m: float,
    m_dm_value: float,
    eps_value: float,
    invalid_z_floor_m: float | None,
) -> dict[str, Any]:
    """Propagate the weak outgoing tail without Coulomb backreaction."""
    if not INCLUDE_OUTGOING_R_FAR_TAIL:
        return empty_fourier_outgoing_tail_fields(
            status="outgoing_tail_disabled",
            complete=True,
            duration_s=0.0,
            end_radius_m=float(np.linalg.norm(x_dm_escape_m)),
        )

    x_dm0 = np.asarray(x_dm_escape_m, dtype=float).reshape(3)
    v_dm0 = np.asarray(v_dm_escape_m_s, dtype=float).reshape(3)
    initial_radius = float(np.linalg.norm(x_dm0))
    target_radius = float(r_far_m)
    if not np.isfinite(target_radius) or target_radius <= 0.0:
        return empty_fourier_outgoing_tail_fields(
            status="invalid_outgoing_R_far",
            complete=False,
            end_radius_m=initial_radius,
        )
    if initial_radius >= target_radius * (1.0 - 1.0e-10):
        return empty_fourier_outgoing_tail_fields(
            status="coupled_stage_already_at_or_beyond_R_far",
            complete=True,
            duration_s=0.0,
            end_radius_m=initial_radius,
        )

    def rhs(_time: float, state: np.ndarray) -> np.ndarray:
        x_dm = state[:3]
        v_dm = state[3:6]
        a_dm = trap_force(
            x_dm[None, :], m_dm_value, eps_value
        )[0] / m_dm_value
        return np.concatenate([v_dm, a_dm])

    def far_event(_time: float, state: np.ndarray) -> float:
        return float(np.linalg.norm(state[:3]) - target_radius)

    far_event.terminal = True
    far_event.direction = 1
    events: list[Callable] = [far_event]
    if invalid_z_floor_m is not None:
        def invalid_event(_time: float, state: np.ndarray) -> float:
            return float(state[2] - invalid_z_floor_m)
        invalid_event.terminal = True
        invalid_event.direction = -1
        events.append(invalid_event)

    speed = max(float(np.linalg.norm(v_dm0)), 1.0e-12)
    travel_estimate = max((target_radius - initial_radius) / speed, 0.0)
    t_max = max(
        OUTGOING_TAIL_TIME_FACTOR * travel_estimate,
        travel_estimate + OUTGOING_TAIL_MIN_TIME_S,
    )
    sol = solve_ivp(
        rhs,
        (0.0, t_max),
        np.concatenate([x_dm0, v_dm0]),
        method="DOP853",
        rtol=OUTGOING_TAIL_RTOL,
        atol=OUTGOING_TAIL_ATOL,
        max_step=OUTGOING_TAIL_MAX_STEP_S,
        events=events,
        dense_output=True,
    )
    reached_far = len(sol.t_events[0]) > 0
    invalid = len(events) > 1 and len(sol.t_events[1]) > 0
    final_radius = float(np.linalg.norm(sol.y[:3, -1]))
    if not reached_far or sol.sol is None:
        if invalid:
            status = "outgoing_tail_invalid_domain"
        elif not sol.success:
            status = "outgoing_tail_solver_failure"
        else:
            status = "outgoing_tail_did_not_reach_R_far"
        return empty_fourier_outgoing_tail_fields(
            status=status,
            complete=False,
            duration_s=float(sol.t[-1]) if len(sol.t) else 0.0,
            end_radius_m=final_radius,
        )

    duration = float(sol.t_events[0][0])
    fields = integrate_fourier_outgoing_tail_from_dense_dm_path(
        sol.sol,
        duration,
        coupled_end_time_s,
        x_ion_escape_m,
        v_ion_escape_m_s,
        eps_value,
        end_radius_m=target_radius,
    )
    fields["fourier_outgoing_tail_dm_initial_radius_m"] = initial_radius
    fields["fourier_outgoing_tail_dm_target_radius_m"] = target_radius
    fields["fourier_outgoing_tail_t_max_s"] = t_max
    fields.update(_test2_bowl_metrics_from_dense_solution(
        sol.sol, duration, prefix="outgoing_rfar", position_slice=slice(0, 3)
    ))
    return fields

def reconstruct_incoming_switch_state(
    x_inside: np.ndarray,
    v_inside: np.ndarray,
    *,
    m_dm_value: float,
    eps_value: float,
    r_switch_m: float,
    invalid_z_floor_m: float | None,
) -> dict[str, Any]:
    speed = max(float(np.linalg.norm(v_inside)), 1.0e-12)
    time_scale = max(
        DM_ONLY_TIME_FACTOR * r_switch_m / speed,
        DM_ONLY_MIN_AFTER_ARRIVAL_S,
    )

    def switch_event(_time: float, state: np.ndarray) -> float:
        return float(np.linalg.norm(state[:3]) - r_switch_m)

    switch_event.terminal = True
    switch_event.direction = 0
    events: list[Callable] = [switch_event]

    if invalid_z_floor_m is not None:
        def invalid_event(_time: float, state: np.ndarray) -> float:
            return float(state[2] - invalid_z_floor_m)

        invalid_event.terminal = True
        invalid_event.direction = 0
        events.append(invalid_event)

    sol = solve_ivp(
        dm_only_rhs(m_dm_value, eps_value),
        (0.0, -time_scale),
        np.concatenate([x_inside, v_inside]),
        method="DOP853",
        rtol=DM_ONLY_RTOL,
        atol=DM_ONLY_ATOL,
        max_step=DM_ONLY_MAX_STEP_S,
        events=events,
    )
    entered = len(sol.t_events[0]) > 0
    if not entered:
        return {
            "success": False,
            "status": "dm_only_inside_requires_rebuild",
            "message": str(sol.message),
            "x_switch_m": np.full(3, np.nan),
            "v_switch_m_s": np.full(3, np.nan),
            "t_switch_from_input_s": np.nan,
            "solution": sol,
        }
    state = np.asarray(sol.y_events[0][0], dtype=float)
    return {
        "success": True,
        "status": "dm_only_reconstructed_incoming_switch",
        "message": str(sol.message),
        "x_switch_m": state[:3],
        "v_switch_m_s": state[3:6],
        "t_switch_from_input_s": float(sol.t_events[0][0]),
        "solution": sol,
    }



def run_dm_trap_only_to_switch(
    x0: np.ndarray,
    v0: np.ndarray,
    *,
    m_dm_value: float,
    eps_value: float,
    r_switch_m: float,
    invalid_z_floor_m: float | None,
) -> dict[str, Any]:
    start_clock = time.perf_counter()
    x0 = np.asarray(x0, dtype=float).reshape(3)
    v0 = np.asarray(v0, dtype=float).reshape(3)
    r0 = float(np.linalg.norm(x0))
    speed0 = max(float(np.linalg.norm(v0)), 1.0e-12)

    if r0 < r_switch_m * (1.0 - 1.0e-10):
        rebuilt = reconstruct_incoming_switch_state(
            x0,
            v0,
            m_dm_value=m_dm_value,
            eps_value=eps_value,
            r_switch_m=r_switch_m,
            invalid_z_floor_m=invalid_z_floor_m,
        )
        sol = rebuilt.pop("solution")
        min_radius = sampled_min_radius(sol.y, slice(0, 3))
        runtime = time.perf_counter() - start_clock
        x_switch = np.asarray(rebuilt["x_switch_m"], dtype=float)
        v_switch = np.asarray(rebuilt["v_switch_m_s"], dtype=float)
        prefix = empty_fourier_prefix_fields(
            status="input_started_inside_R_switch_prefix_unavailable",
            complete=False,
            duration_s=0.0,
        )
        result = {
            "dm_only_success": bool(rebuilt["success"]),
            "dm_only_status": str(rebuilt["status"]),
            "dm_only_message": str(rebuilt["message"]),
            "entered_switch": bool(rebuilt["success"]),
            "dm_only_runtime_s": runtime,
            "dm_only_t_final_s": float(sol.t[-1]) if len(sol.t) else 0.0,
            "dm_only_t_arrival_estimate_s": r0 / speed0,
            "dm_only_t_max_s": abs(float(sol.t[-1])) if len(sol.t) else np.nan,
            "dm_only_initial_radius_m": r0,
            "dm_only_final_radius_m": (
                float(np.linalg.norm(sol.y[:3, -1])) if sol.y.size else r0
            ),
            "dm_only_min_radius_m": min_radius,
            "dm_only_final_radial_velocity_m_s": (
                radial_velocity(sol.y[:3, -1], sol.y[3:6, -1])
                if sol.y.size else np.nan
            ),
            "x_switch_x_m": x_switch[0],
            "x_switch_y_m": x_switch[1],
            "x_switch_z_m": x_switch[2],
            "vx_switch_m_s": v_switch[0],
            "vy_switch_m_s": v_switch[1],
            "vz_switch_m_s": v_switch[2],
        }
        result.update(prefix)
        result.update(_test2_bowl_metrics_from_samples(
            np.asarray(sol.t, dtype=float),
            np.asarray(sol.y[:3], dtype=float).T,
            prefix="dm_only",
        ))
        return result

    if np.isclose(r0, r_switch_m, rtol=1.0e-10, atol=1.0e-15):
        runtime = time.perf_counter() - start_clock
        result = {
            "dm_only_success": True,
            "dm_only_status": "dm_only_started_on_switch",
            "dm_only_message": "Initial state lies on R_switch",
            "entered_switch": True,
            "dm_only_runtime_s": runtime,
            "dm_only_t_final_s": 0.0,
            "dm_only_t_arrival_estimate_s": 0.0,
            "dm_only_t_max_s": 0.0,
            "dm_only_initial_radius_m": r0,
            "dm_only_final_radius_m": r0,
            "dm_only_min_radius_m": r0,
            "dm_only_final_radial_velocity_m_s": radial_velocity(x0, v0),
            "x_switch_x_m": x0[0],
            "x_switch_y_m": x0[1],
            "x_switch_z_m": x0[2],
            "vx_switch_m_s": v0[0],
            "vy_switch_m_s": v0[1],
            "vz_switch_m_s": v0[2],
        }
        result.update(empty_fourier_prefix_fields(
            status="zero_length_R_far_to_R_switch",
            complete=True,
            duration_s=0.0,
        ))
        result.update(_test2_bowl_metrics_from_samples(
            np.array([0.0]), x0[None, :], prefix="dm_only"
        ))
        return result

    arrival_estimate = r0 / speed0
    t_max = max(
        DM_ONLY_TIME_FACTOR * arrival_estimate,
        arrival_estimate + DM_ONLY_MIN_AFTER_ARRIVAL_S,
    )
    escape_radius = max(DM_ONLY_ESCAPE_FACTOR * r0, FULL_ESCAPE_FACTOR * r_switch_m)

    def switch_event(_time: float, state: np.ndarray) -> float:
        return float(np.linalg.norm(state[:3]) - r_switch_m)

    switch_event.terminal = True
    switch_event.direction = -1

    def escape_event(_time: float, state: np.ndarray) -> float:
        radius = float(np.linalg.norm(state[:3]))
        if radial_velocity(state[:3], state[3:6]) <= 0.0:
            return -abs(radius - escape_radius) - 1.0e-30
        return radius - escape_radius

    escape_event.terminal = True
    escape_event.direction = 1

    def closest_approach_event(_time: float, state: np.ndarray) -> float:
        return radial_velocity(state[:3], state[3:6])

    closest_approach_event.terminal = False
    closest_approach_event.direction = 1
    events: list[Callable] = [switch_event, escape_event, closest_approach_event]

    if invalid_z_floor_m is not None:
        def invalid_event(_time: float, state: np.ndarray) -> float:
            return float(state[2] - invalid_z_floor_m)

        invalid_event.terminal = True
        invalid_event.direction = -1
        events.append(invalid_event)

    sol = solve_ivp(
        dm_only_rhs(m_dm_value, eps_value),
        (0.0, t_max),
        np.concatenate([x0, v0]),
        method="DOP853",
        rtol=DM_ONLY_RTOL,
        atol=DM_ONLY_ATOL,
        max_step=DM_ONLY_MAX_STEP_S,
        events=events,
        dense_output=True,
    )
    entered = len(sol.t_events[0]) > 0
    escaped = len(sol.t_events[1]) > 0
    closest_states = (
        np.asarray(sol.y_events[2], dtype=float)
        if len(sol.y_events) > 2 and len(sol.y_events[2])
        else np.empty((0, 6), dtype=float)
    )
    closest_times = (
        np.asarray(sol.t_events[2], dtype=float)
        if len(sol.t_events) > 2 and len(sol.t_events[2])
        else np.empty(0, dtype=float)
    )
    invalid_index = 3
    invalid = len(events) > invalid_index and len(sol.t_events[invalid_index]) > 0
    reconstructed_message = ""
    prefix_duration = np.nan

    if entered:
        status = "dm_only_entered_switch"
        event_state = np.asarray(sol.y_events[0][0], dtype=float)[:6]
        prefix_duration = float(sol.t_events[0][0])
    elif len(closest_states):
        closest_radii = np.linalg.norm(closest_states[:, :3], axis=1)
        closest_index = int(np.argmin(closest_radii))
        closest_state = closest_states[closest_index]
        if closest_radii[closest_index] <= r_switch_m * (1.0 + 1.0e-10):
            rebuilt = reconstruct_incoming_switch_state(
                closest_state[:3],
                closest_state[3:6],
                m_dm_value=m_dm_value,
                eps_value=eps_value,
                r_switch_m=r_switch_m,
                invalid_z_floor_m=invalid_z_floor_m,
            )
            entered = bool(rebuilt["success"])
            reconstructed_message = str(rebuilt["message"])
            if entered:
                status = "dm_only_entered_switch_via_closest_approach"
                event_state = np.concatenate([
                    rebuilt["x_switch_m"], rebuilt["v_switch_m_s"]
                ])
                prefix_duration = (
                    float(closest_times[closest_index])
                    + float(rebuilt["t_switch_from_input_s"])
                )
            else:
                status = "dm_only_closest_approach_rebuild_failure"
                event_state = np.full(6, np.nan)
        elif escaped:
            status = "dm_only_escaped_without_switch"
            event_state = np.full(6, np.nan)
        elif invalid:
            status = "dm_only_invalid_domain"
            event_state = np.full(6, np.nan)
        elif not sol.success:
            status = "dm_only_solver_failure"
            event_state = np.full(6, np.nan)
        else:
            status = "dm_only_timeout"
            event_state = np.full(6, np.nan)
    elif escaped:
        status = "dm_only_escaped_without_switch"
        event_state = np.full(6, np.nan)
    elif invalid:
        status = "dm_only_invalid_domain"
        event_state = np.full(6, np.nan)
    elif not sol.success:
        status = "dm_only_solver_failure"
        event_state = np.full(6, np.nan)
    else:
        status = "dm_only_timeout"
        event_state = np.full(6, np.nan)

    if entered and np.isfinite(prefix_duration) and prefix_duration >= 0.0 and sol.sol is not None:
        prefix = integrate_fourier_prefix_from_dense_dm_path(
            sol.sol,
            prefix_duration,
            eps_value,
        )
    elif entered:
        prefix = empty_fourier_prefix_fields(
            status="switch_entered_but_prefix_unavailable",
            complete=False,
            duration_s=float(prefix_duration) if np.isfinite(prefix_duration) else 0.0,
        )
    else:
        prefix = empty_fourier_prefix_fields(
            status="not_applicable_no_switch_entry",
            complete=False,
            duration_s=0.0,
        )

    times, states = append_event_state(
        sol, 0 if len(sol.t_events[0]) else 1 if escaped else 0
    )
    final_state = states[:, -1]
    sampled_minimum = sampled_min_radius(states, slice(0, 3))
    if len(closest_states):
        sampled_minimum = min(
            sampled_minimum,
            float(np.min(np.linalg.norm(closest_states[:, :3], axis=1))),
        )
    runtime = time.perf_counter() - start_clock
    message = str(sol.message)
    if reconstructed_message:
        message = (
            f"{message}; switch reconstructed from closest approach: "
            f"{reconstructed_message}"
        )
    result = {
        "dm_only_success": bool(sol.success) and not status.endswith("failure"),
        "dm_only_status": status,
        "dm_only_message": message,
        "entered_switch": bool(entered),
        "dm_only_runtime_s": runtime,
        "dm_only_t_final_s": float(times[-1]),
        "dm_only_t_arrival_estimate_s": arrival_estimate,
        "dm_only_t_max_s": t_max,
        "dm_only_initial_radius_m": r0,
        "dm_only_final_radius_m": float(np.linalg.norm(final_state[:3])),
        "dm_only_min_radius_m": sampled_minimum,
        "dm_only_final_radial_velocity_m_s": radial_velocity(
            final_state[:3], final_state[3:6]
        ),
        "x_switch_x_m": float(event_state[0]),
        "x_switch_y_m": float(event_state[1]),
        "x_switch_z_m": float(event_state[2]),
        "vx_switch_m_s": float(event_state[3]),
        "vy_switch_m_s": float(event_state[4]),
        "vz_switch_m_s": float(event_state[5]),
    }
    result.update(prefix)
    result.update(_test2_bowl_metrics_from_dense_solution(
        sol.sol,
        float(times[-1]) if len(times) else 0.0,
        prefix="dm_only",
        position_slice=slice(0, 3),
    ))
    return result



def energy_threshold_diagnostics(
    mechanical_energy_j: float,
    fourier_detection_energy_j: float,
    *,
    fourier_switch_energy_j: float | None = None,
    trajectory_resolved: bool,
    fourier_detection_resolved: bool | None = None,
) -> dict[str, Any]:
    """Return threshold flags using E_FT(R_far) and consistency at R_switch."""
    mechanical = float(mechanical_energy_j)
    fourier_detection = float(fourier_detection_energy_j)
    fourier_switch = (
        float(fourier_switch_energy_j)
        if fourier_switch_energy_j is not None
        else fourier_detection
    )
    if fourier_detection_resolved is None:
        fourier_detection_resolved = bool(trajectory_resolved)

    mechanical_finite = bool(np.isfinite(mechanical))
    fourier_detection_finite = bool(np.isfinite(fourier_detection))
    fourier_switch_finite = bool(np.isfinite(fourier_switch))
    above_mechanical = bool(
        trajectory_resolved
        and mechanical_finite
        and mechanical >= TARGET_ION_ENERGY_J
    )
    above_fourier = bool(
        fourier_detection_resolved
        and fourier_detection_finite
        and fourier_detection >= TARGET_ION_ENERGY_J
    )

    if mechanical_finite and fourier_switch_finite:
        denominator = max(
            abs(mechanical),
            abs(fourier_switch),
            float(TARGET_ION_ENERGY_J) * 1.0e-12,
            np.finfo(float).tiny,
        )
        absolute_difference = abs(fourier_switch - mechanical)
        relative_difference = absolute_difference / denominator
        consistency_ok = bool(relative_difference <= FOURIER_MECHANICAL_REL_TOL)
    else:
        absolute_difference = np.nan
        relative_difference = np.nan
        consistency_ok = False

    if DETECTION_ENERGY_POLICY == "fourier":
        selected_energy = fourier_detection
        selected_above = above_fourier
    elif DETECTION_ENERGY_POLICY == "mechanical":
        selected_energy = mechanical
        selected_above = above_mechanical
    elif DETECTION_ENERGY_POLICY == "both":
        selected_energy = min(mechanical, fourier_detection)
        selected_above = above_mechanical and above_fourier
    elif DETECTION_ENERGY_POLICY == "either":
        selected_energy = max(mechanical, fourier_detection)
        selected_above = above_mechanical or above_fourier
    else:
        raise ValueError(
            "DETECTION_ENERGY_POLICY must be 'fourier', 'mechanical', "
            "'both', or 'either'"
        )

    if REQUIRE_FOURIER_MECHANICAL_AGREEMENT:
        selected_above = bool(selected_above and consistency_ok)

    return {
        "energy_threshold_policy": DETECTION_ENERGY_POLICY,
        "fourier_detection_start_radius": FOURIER_DETECTION_START_RADIUS,
        "fourier_detection_resolved": bool(fourier_detection_resolved),
        "E_detection_J": selected_energy,
        "above_energy_threshold": bool(selected_above),
        "above_energy_threshold_fourier": above_fourier,
        "above_energy_threshold_mechanical": above_mechanical,
        "above_energy_threshold_both": bool(above_fourier and above_mechanical),
        "above_energy_threshold_either": bool(above_fourier or above_mechanical),
        "fourier_mechanical_comparison_energy": "E_fourier_from_R_switch_J",
        "fourier_mechanical_abs_difference_J": absolute_difference,
        "fourier_mechanical_rel_difference": relative_difference,
        "fourier_mechanical_consistency_ok": consistency_ok,
        "fourier_mechanical_consistency_required": bool(
            REQUIRE_FOURIER_MECHANICAL_AGREEMENT
        ),
        "fourier_mechanical_rel_tolerance": float(FOURIER_MECHANICAL_REL_TOL),
    }



# =============================================================================
# LOCAL RUTHERFORD CONTINUATION
# =============================================================================


def reduced_mass(m_dm_value: float) -> float:
    return float(m_dm_value * float(m_ion) / (m_dm_value + float(m_ion)))


def run_local_rutherford_after_reach(
    x_switch: np.ndarray,
    v_switch: np.ndarray,
    *,
    m_dm_value: float,
    eps_value: float,
    r_full_m: float,
) -> dict[str, Any]:
    start = time.perf_counter()
    radius = float(np.linalg.norm(x_switch))
    speed = float(np.linalg.norm(v_switch))
    mu = reduced_mass(m_dm_value)
    coupling = abs(float(K) * float(Z_ion) * eps_value * float(e) ** 2)
    relative_energy = 0.5 * mu * speed**2 + coupling / max(radius, 1.0e-300)
    if relative_energy <= 0.0 or coupling <= 0.0:
        return {
            "full_success": False,
            "full_status": "rutherford_invalid_local_invariants",
            "full_message": "Invalid local Rutherford invariants",
            "rutherford_analytic_used": True,
            "reached_R_full": False,
            "above_energy_threshold": False,
            "d_min_m": np.nan,
            "E_final_J": np.nan,
            "delta_E_ion_J": np.nan,
            "E_fourier_total_J": np.nan,
            "E_fourier_from_R_switch_J": np.nan,
            "E_fourier_from_R_far_J": np.nan,
            "E_mechanical_final_J": np.nan,
            "full_runtime_s": time.perf_counter() - start,
        }

    v_inf = math.sqrt(2.0 * relative_energy / mu)
    angular_momentum = mu * float(np.linalg.norm(np.cross(x_switch, v_switch)))
    b_inf = angular_momentum / max(mu * v_inf, 1.0e-300)
    a = coupling / (mu * v_inf**2)
    r_min = a + math.sqrt(a * a + b_inf * b_inf)
    head_on_recoil = 2.0 * mu**2 * v_inf**2 / float(m_ion)
    recoil = head_on_recoil * a * a / max(a * a + b_inf * b_inf, 1.0e-300)
    reached = bool(r_min <= r_full_m)
    status = (
        "rutherford_local_reached_R_full"
        if reached else "rutherford_local_miss_R_full"
    )
    threshold = energy_threshold_diagnostics(
        recoil,
        recoil,
        fourier_switch_energy_j=recoil,
        trajectory_resolved=True,
        fourier_detection_resolved=True,
    )
    result = {
        "full_success": True,
        "full_status": status,
        "full_message": "Local Rutherford continuation from switch state",
        "rutherford_analytic_used": True,
        "rutherford_local_v_inf_m_s": v_inf,
        "rutherford_local_b_inf_m": b_inf,
        "rutherford_local_a_m": a,
        "rutherford_local_relative_energy_J": relative_energy,
        "reached_R_full": reached,
        "d_min_m": r_min,
        "d_min_um": r_min * 1.0e6,
        "E_initial_J": 0.0,
        "E_final_J": recoil,
        "delta_E_ion_J": recoil,
        "E_mechanical_final_J": recoil,
        "E_mechanical_average_J": recoil,
        "E_fourier_total_J": recoil,
        "E_fourier_from_R_switch_J": recoil,
        "E_fourier_from_R_far_J": recoil,
        "E_fourier_x_J": recoil / 3.0,
        "E_fourier_y_J": recoil / 3.0,
        "E_fourier_z_J": recoil / 3.0,
        "E_fourier_from_R_switch_x_J": recoil / 3.0,
        "E_fourier_from_R_switch_y_J": recoil / 3.0,
        "E_fourier_from_R_switch_z_J": recoil / 3.0,
        "E_fourier_from_R_far_x_J": recoil / 3.0,
        "E_fourier_from_R_far_y_J": recoil / 3.0,
        "E_fourier_from_R_far_z_J": recoil / 3.0,
        "fourier_prefix_complete": True,
        "fourier_detection_prefix_complete": True,
        "fourier_detection_outgoing_tail_complete": True,
        "fourier_detection_history": "analytic_Rutherford_from_infinity",
        "fourier_prefix_status": "analytic_Rutherford_from_infinity",
        "E_fourier_prefix_standalone_J": np.nan,
        "fourier_cos_x_N_s": np.nan,
        "fourier_cos_y_N_s": np.nan,
        "fourier_cos_z_N_s": np.nan,
        "fourier_sin_x_N_s": np.nan,
        "fourier_sin_y_N_s": np.nan,
        "fourier_sin_z_N_s": np.nan,
        "E_x_J": recoil / 3.0,
        "E_y_J": recoil / 3.0,
        "E_z_J": recoil / 3.0,
        "interaction_time_inside_R_switch_s": np.nan,
        "interaction_time_inside_R_switch_us": np.nan,
        "interaction_time_inside_R_full_s": np.nan,
        "interaction_time_inside_R_full_us": np.nan,
        "full_runtime_s": time.perf_counter() - start,
    }
    result.update(threshold)
    return result



# =============================================================================
# FULL COUPLED ION-DM PROPAGATION
# =============================================================================


# =============================================================================
# FULL COUPLED ION-DM PROPAGATION
# =============================================================================


def ion_energy_components(position: np.ndarray, velocity: np.ndarray) -> np.ndarray:
    position = np.asarray(position, dtype=float)
    velocity = np.asarray(velocity, dtype=float)
    if USE_HARMONIC_ION:
        omega = np.asarray(omega_vec, dtype=float).reshape(3, 1)
        return 0.5 * float(m_ion) * (velocity**2 + (omega * position) ** 2)
    kinetic = 0.5 * float(m_ion) * velocity**2
    return kinetic


def time_inside(times: np.ndarray, mask: np.ndarray) -> float:
    if len(times) < 2:
        return 0.0
    dt = np.diff(times)
    active = mask[:-1] | mask[1:]
    return float(np.sum(dt[active]))


def run_full_coupled_after_reach(
    x_switch: np.ndarray,
    v_switch: np.ndarray,
    *,
    m_dm_value: float,
    eps_value: float,
    r_full_m: float,
    r_switch_m: float,
    r_far_m: float,
    invalid_z_floor_m: float | None,
    prefix_cos_N_s: np.ndarray | None = None,
    prefix_sin_N_s: np.ndarray | None = None,
    prefix_complete: bool = True,
) -> dict[str, Any]:
    start_clock = time.perf_counter()
    if not USE_HARMONIC_ION and DETECTION_ENERGY_POLICY in {"fourier", "both"}:
        raise ValueError("Fourier secular-mode energy requires USE_HARMONIC_ION=True")

    prefix_cos = np.zeros(3, dtype=float) if prefix_cos_N_s is None else np.asarray(
        prefix_cos_N_s, dtype=float
    ).reshape(3)
    prefix_sin = np.zeros(3, dtype=float) if prefix_sin_N_s is None else np.asarray(
        prefix_sin_N_s, dtype=float
    ).reshape(3)
    prefix_cos = np.nan_to_num(prefix_cos, nan=0.0, posinf=0.0, neginf=0.0)
    prefix_sin = np.nan_to_num(prefix_sin, nan=0.0, posinf=0.0, neginf=0.0)

    x_ion0 = np.zeros(3, dtype=float)
    v_ion0 = np.zeros(3, dtype=float)
    force_cos0 = np.zeros(3, dtype=float)
    force_sin0 = np.zeros(3, dtype=float)
    state0 = np.concatenate([
        x_ion0, x_switch, v_ion0, v_switch, force_cos0, force_sin0,
    ])
    soft2 = float(COULOMB_SOFTENING_M) ** 2
    omega_modes = np.asarray(omega_vec, dtype=float).reshape(3)

    def rhs(_time: float, state: np.ndarray) -> np.ndarray:
        x_ion = state[0:3]
        x_dm = state[3:6]
        v_ion = state[6:9]
        v_dm = state[9:12]
        if USE_HARMONIC_ION:
            a_ion = -omega_modes**2 * x_ion
        else:
            a_ion = trap_force(
                x_ion[None, :], float(m_ion), float(Z_ion)
            )[0] / float(m_ion)
        a_dm = trap_force(x_dm[None, :], m_dm_value, eps_value)[0] / m_dm_value
        delta = x_ion - x_dm
        radius2 = max(float(np.dot(delta, delta)) + soft2, np.finfo(float).tiny)
        force_coulomb = (
            float(K) * float(Z_ion) * eps_value * float(e) ** 2
            * delta * radius2 ** (-1.5)
        )
        a_ion = a_ion + force_coulomb / float(m_ion)
        a_dm = a_dm - force_coulomb / m_dm_value
        phase = omega_modes * float(_time)
        return np.concatenate([
            v_ion,
            v_dm,
            a_ion,
            a_dm,
            force_coulomb * np.cos(phase),
            force_coulomb * np.sin(phase),
        ])

    escape_radius = FULL_ESCAPE_FACTOR * r_switch_m

    def escape_event(_time: float, state: np.ndarray) -> float:
        relative_position = state[3:6] - state[0:3]
        relative_velocity = state[9:12] - state[6:9]
        radius = float(np.linalg.norm(relative_position))
        if radial_velocity(relative_position, relative_velocity) <= 0.0:
            return -abs(radius - escape_radius) - 1.0e-30
        return radius - escape_radius

    escape_event.terminal = True
    escape_event.direction = 1
    events: list[Callable] = [escape_event]
    if invalid_z_floor_m is not None:
        def invalid_event(_time: float, state: np.ndarray) -> float:
            return float(min(state[2], state[5]) - invalid_z_floor_m)
        invalid_event.terminal = True
        invalid_event.direction = -1
        events.append(invalid_event)

    speed = max(float(np.linalg.norm(v_switch)), 1.0e-12)
    t_max = max(FULL_TIME_FACTOR * r_switch_m / speed, MIN_FULL_TIME_S)
    t_eval = np.arange(0.0, t_max, FULL_SAMPLE_DT_S, dtype=float)
    if t_eval.size == 0 or not np.isclose(t_eval[0], 0.0):
        t_eval = np.insert(t_eval, 0, 0.0)
    if not np.isclose(t_eval[-1], t_max):
        t_eval = np.append(t_eval, t_max)

    sol = solve_ivp(
        rhs,
        (0.0, t_max),
        state0,
        method="DOP853",
        t_eval=t_eval,
        rtol=FULL_RTOL,
        atol=FULL_ATOL,
        max_step=FULL_MAX_STEP_S,
        events=events,
    )
    escaped = len(sol.t_events[0]) > 0
    invalid = len(events) > 1 and len(sol.t_events[1]) > 0
    times, states = append_event_state(sol, 0 if escaped else 1 if invalid else 0)

    x_ion = states[0:3]
    x_dm = states[3:6]
    v_ion = states[6:9]
    v_dm = states[9:12]
    relative = x_dm - x_ion
    separations = np.linalg.norm(relative, axis=0)
    d_min = float(np.min(separations)) if separations.size else np.nan
    reached = bool(np.isfinite(d_min) and d_min <= r_full_m)

    energy_modes = ion_energy_components(x_ion, v_ion)
    total_energy = np.sum(energy_modes, axis=0)
    final_time = float(times[-1]) if len(times) else 0.0
    average_mask = times >= max(0.0, final_time - ENERGY_AVERAGE_WINDOW_S)
    if not np.any(average_mask):
        average_mask = np.ones_like(times, dtype=bool)
    averaged_mechanical_modes = np.mean(energy_modes[:, average_mask], axis=1)
    averaged_mechanical_energy = float(np.sum(averaged_mechanical_modes))
    instantaneous_mechanical_modes = energy_modes[:, -1]
    instantaneous_mechanical_energy = float(np.sum(instantaneous_mechanical_modes))
    initial_energy = float(total_energy[0]) if total_energy.size else 0.0
    delta_energy = averaged_mechanical_energy - initial_energy

    switch_cos = np.asarray(states[12:15, -1], dtype=float)
    switch_sin = np.asarray(states[15:18, -1], dtype=float)
    incoming_far_to_escape_cos = prefix_cos + switch_cos
    incoming_far_to_escape_sin = prefix_sin + switch_sin

    if bool(sol.success and escaped and not invalid):
        outgoing_tail = run_outgoing_dm_tail_to_r_far(
            x_ion[:, -1],
            v_ion[:, -1],
            x_dm[:, -1],
            v_dm[:, -1],
            coupled_end_time_s=final_time,
            r_far_m=r_far_m,
            m_dm_value=m_dm_value,
            eps_value=eps_value,
            invalid_z_floor_m=invalid_z_floor_m,
        )
    else:
        outgoing_tail = empty_fourier_outgoing_tail_fields(
            status="coupled_stage_unresolved_before_outgoing_tail",
            complete=False,
            end_radius_m=float(np.linalg.norm(x_dm[:, -1])),
        )

    tail_cos = np.array([
        outgoing_tail["fourier_outgoing_tail_cos_x_N_s"],
        outgoing_tail["fourier_outgoing_tail_cos_y_N_s"],
        outgoing_tail["fourier_outgoing_tail_cos_z_N_s"],
    ], dtype=float)
    tail_sin = np.array([
        outgoing_tail["fourier_outgoing_tail_sin_x_N_s"],
        outgoing_tail["fourier_outgoing_tail_sin_y_N_s"],
        outgoing_tail["fourier_outgoing_tail_sin_z_N_s"],
    ], dtype=float)

    roundtrip_far_cos = incoming_far_to_escape_cos + tail_cos
    roundtrip_far_sin = incoming_far_to_escape_sin + tail_sin

    switch_modes = (switch_cos**2 + switch_sin**2) / (2.0 * float(m_ion))
    incoming_far_to_escape_modes = (
        incoming_far_to_escape_cos**2 + incoming_far_to_escape_sin**2
    ) / (2.0 * float(m_ion))
    roundtrip_far_modes = (
        roundtrip_far_cos**2 + roundtrip_far_sin**2
    ) / (2.0 * float(m_ion))
    switch_energy = float(np.sum(switch_modes))
    incoming_far_to_escape_energy = float(np.sum(incoming_far_to_escape_modes))
    roundtrip_far_energy = float(np.sum(roundtrip_far_modes))

    inside_switch = separations <= r_switch_m
    inside_full = separations <= r_full_m
    interaction_switch = time_inside(times, inside_switch)
    rfull_metrics = _test2_interval_metrics(times, separations - r_full_m)
    interaction_full = float(rfull_metrics["time_inside_s"])
    full_bowl_metrics = _test2_bowl_metrics_from_samples(
        times, x_dm.T, prefix="full"
    )

    if invalid:
        full_status = "full_invalid_domain"
    elif not sol.success:
        full_status = "full_solver_failure"
    elif not escaped:
        full_status = "full_timeout"
    elif reached:
        full_status = "reached_R_full_and_escaped"
    else:
        full_status = "entered_switch_but_missed_R_full"

    trajectory_resolved = bool(sol.success and escaped and not invalid)
    prefix_resolved = bool(prefix_complete or not REQUIRE_R_FAR_PREFIX_COMPLETE)
    tail_complete = bool(outgoing_tail["fourier_outgoing_tail_complete"])
    tail_resolved = bool(
        tail_complete or not REQUIRE_OUTGOING_R_FAR_TAIL_COMPLETE
    )
    threshold = energy_threshold_diagnostics(
        instantaneous_mechanical_energy,
        roundtrip_far_energy,
        fourier_switch_energy_j=switch_energy,
        trajectory_resolved=trajectory_resolved,
        fourier_detection_resolved=bool(
            trajectory_resolved and prefix_resolved and tail_resolved
        ),
    )

    result = {
        "full_success": bool(sol.success),
        "full_status": full_status,
        "full_message": str(sol.message),
        "rutherford_analytic_used": False,
        "reached_R_full": reached,
        "d_min_m": d_min,
        "d_min_um": d_min * 1.0e6 if np.isfinite(d_min) else np.nan,
        "E_initial_J": initial_energy,
        "E_final_J": averaged_mechanical_energy,
        "delta_E_ion_J": delta_energy,
        "E_x_J": float(averaged_mechanical_modes[0]),
        "E_y_J": float(averaged_mechanical_modes[1]),
        "E_z_J": float(averaged_mechanical_modes[2]),
        "E_mechanical_final_J": instantaneous_mechanical_energy,
        "E_mechanical_average_J": averaged_mechanical_energy,
        "E_mechanical_x_final_J": float(instantaneous_mechanical_modes[0]),
        "E_mechanical_y_final_J": float(instantaneous_mechanical_modes[1]),
        "E_mechanical_z_final_J": float(instantaneous_mechanical_modes[2]),
        # Canonical production energy is now the complete incoming-R_far to
        # outgoing-R_far Fourier history.
        "E_fourier_total_J": roundtrip_far_energy,
        "E_fourier_x_J": float(roundtrip_far_modes[0]),
        "E_fourier_y_J": float(roundtrip_far_modes[1]),
        "E_fourier_z_J": float(roundtrip_far_modes[2]),
        "E_fourier_from_R_far_J": roundtrip_far_energy,
        "E_fourier_from_R_far_x_J": float(roundtrip_far_modes[0]),
        "E_fourier_from_R_far_y_J": float(roundtrip_far_modes[1]),
        "E_fourier_from_R_far_z_J": float(roundtrip_far_modes[2]),
        "E_fourier_R_far_in_to_escape_J": incoming_far_to_escape_energy,
        "E_fourier_R_far_in_to_escape_x_J": float(incoming_far_to_escape_modes[0]),
        "E_fourier_R_far_in_to_escape_y_J": float(incoming_far_to_escape_modes[1]),
        "E_fourier_R_far_in_to_escape_z_J": float(incoming_far_to_escape_modes[2]),
        "E_fourier_from_R_switch_J": switch_energy,
        "E_fourier_from_R_switch_x_J": float(switch_modes[0]),
        "E_fourier_from_R_switch_y_J": float(switch_modes[1]),
        "E_fourier_from_R_switch_z_J": float(switch_modes[2]),
        "fourier_detection_prefix_complete": bool(prefix_complete),
        "fourier_detection_outgoing_tail_complete": tail_complete,
        "fourier_detection_history": "incoming_R_far_to_outgoing_R_far",
        # Generic quadratures denote the complete round-trip R_far amplitude.
        "fourier_cos_x_N_s": float(roundtrip_far_cos[0]),
        "fourier_cos_y_N_s": float(roundtrip_far_cos[1]),
        "fourier_cos_z_N_s": float(roundtrip_far_cos[2]),
        "fourier_sin_x_N_s": float(roundtrip_far_sin[0]),
        "fourier_sin_y_N_s": float(roundtrip_far_sin[1]),
        "fourier_sin_z_N_s": float(roundtrip_far_sin[2]),
        "fourier_R_far_in_to_escape_cos_x_N_s": float(incoming_far_to_escape_cos[0]),
        "fourier_R_far_in_to_escape_cos_y_N_s": float(incoming_far_to_escape_cos[1]),
        "fourier_R_far_in_to_escape_cos_z_N_s": float(incoming_far_to_escape_cos[2]),
        "fourier_R_far_in_to_escape_sin_x_N_s": float(incoming_far_to_escape_sin[0]),
        "fourier_R_far_in_to_escape_sin_y_N_s": float(incoming_far_to_escape_sin[1]),
        "fourier_R_far_in_to_escape_sin_z_N_s": float(incoming_far_to_escape_sin[2]),
        "fourier_from_R_switch_cos_x_N_s": float(switch_cos[0]),
        "fourier_from_R_switch_cos_y_N_s": float(switch_cos[1]),
        "fourier_from_R_switch_cos_z_N_s": float(switch_cos[2]),
        "fourier_from_R_switch_sin_x_N_s": float(switch_sin[0]),
        "fourier_from_R_switch_sin_y_N_s": float(switch_sin[1]),
        "fourier_from_R_switch_sin_z_N_s": float(switch_sin[2]),
        "fastest_secular_period_s": FASTEST_SECULAR_PERIOD_S,
        "fourier_resolution_step_s": FOURIER_RESOLUTION_STEP_S,
        "interaction_time_inside_R_switch_s": interaction_switch,
        "interaction_time_inside_R_switch_us": interaction_switch * 1.0e6,
        "interaction_time_inside_R_full_s": interaction_full,
        "interaction_time_inside_R_full_us": interaction_full * 1.0e6,
        "R_full_entry_time_after_switch_s": float(rfull_metrics["entry_time_s"]),
        "R_full_exit_time_after_switch_s": float(rfull_metrics["exit_time_s"]),
        "R_full_entry_count": int(rfull_metrics["entry_count"]),
        "R_full_exit_count": int(rfull_metrics["exit_count"]),
        "ended_inside_R_full": bool(rfull_metrics["ended_inside"]),
        "full_t_final_s": final_time,
        "full_t_max_s": t_max,
        "fourier_history_t_final_s": final_time + float(
            outgoing_tail["fourier_outgoing_tail_duration_s"]
        ),
        "full_runtime_s": time.perf_counter() - start_clock,
    }
    result.update(outgoing_tail)
    result.update(full_bowl_metrics)
    result.update(threshold)
    if KEEP_SINGLE_TRAJECTORY_ARRAYS:
        result["debug_times_s"] = times
        result["debug_states"] = states
    return result



# =============================================================================
# ROW WORKERS AND OUTPUT DEFAULTS
# =============================================================================


# =============================================================================
# ROW WORKERS AND OUTPUT DEFAULTS
# =============================================================================

DM_ONLY_DEFAULTS: dict[str, Any] = {
    "dm_only_success": False,
    "dm_only_status": "not_run",
    "dm_only_message": "",
    "entered_switch": False,
    "dm_only_runtime_s": np.nan,
    "dm_only_t_final_s": np.nan,
    "dm_only_t_arrival_estimate_s": np.nan,
    "dm_only_t_max_s": np.nan,
    "dm_only_initial_radius_m": np.nan,
    "dm_only_final_radius_m": np.nan,
    "dm_only_min_radius_m": np.nan,
    "dm_only_min_radius_um": np.nan,
    "dm_only_min_radius_over_R_switch": np.nan,
    "dm_only_final_radial_velocity_m_s": np.nan,
    "b_over_R_switch": np.nan,
    "x_switch_x_m": np.nan,
    "x_switch_y_m": np.nan,
    "x_switch_z_m": np.nan,
    "vx_switch_m_s": np.nan,
    "vy_switch_m_s": np.nan,
    "vz_switch_m_s": np.nan,
    "fourier_prefix_complete": False,
    "fourier_prefix_status": "not_run",
    "fourier_prefix_duration_s": np.nan,
    "fourier_prefix_time_origin": "incoming_R_switch_crossing",
    "fourier_prefix_cos_x_N_s": 0.0,
    "fourier_prefix_cos_y_N_s": 0.0,
    "fourier_prefix_cos_z_N_s": 0.0,
    "fourier_prefix_sin_x_N_s": 0.0,
    "fourier_prefix_sin_y_N_s": 0.0,
    "fourier_prefix_sin_z_N_s": 0.0,
    "fourier_prefix_quad_abs_error_x_N_s": np.nan,
    "fourier_prefix_quad_abs_error_y_N_s": np.nan,
    "fourier_prefix_quad_abs_error_z_N_s": np.nan,
    "E_fourier_prefix_standalone_J": np.nan,
    "E_fourier_prefix_x_J": np.nan,
    "E_fourier_prefix_y_J": np.nan,
    "E_fourier_prefix_z_J": np.nan,
}

FULL_DEFAULTS: dict[str, Any] = {
    "full_success": False,
    "full_status": "not_run",
    "full_message": "",
    "rutherford_analytic_used": False,
    "reached_R_full": False,
    "above_energy_threshold": False,
    "above_energy_threshold_fourier": False,
    "above_energy_threshold_mechanical": False,
    "above_energy_threshold_both": False,
    "above_energy_threshold_either": False,
    "energy_threshold_policy": DETECTION_ENERGY_POLICY,
    "fourier_detection_start_radius": FOURIER_DETECTION_START_RADIUS,
    "fourier_detection_resolved": False,
    "E_detection_J": np.nan,
    "d_min_m": np.nan,
    "d_min_um": np.nan,
    "E_initial_J": np.nan,
    "E_final_J": np.nan,
    "delta_E_ion_J": np.nan,
    "E_x_J": np.nan,
    "E_y_J": np.nan,
    "E_z_J": np.nan,
    "E_mechanical_final_J": np.nan,
    "E_mechanical_average_J": np.nan,
    "E_mechanical_x_final_J": np.nan,
    "E_mechanical_y_final_J": np.nan,
    "E_mechanical_z_final_J": np.nan,
    "E_fourier_total_J": np.nan,
    "E_fourier_x_J": np.nan,
    "E_fourier_y_J": np.nan,
    "E_fourier_z_J": np.nan,
    "E_fourier_from_R_far_J": np.nan,
    "E_fourier_from_R_far_x_J": np.nan,
    "E_fourier_from_R_far_y_J": np.nan,
    "E_fourier_from_R_far_z_J": np.nan,
    "E_fourier_from_R_switch_J": np.nan,
    "E_fourier_from_R_switch_x_J": np.nan,
    "E_fourier_from_R_switch_y_J": np.nan,
    "E_fourier_from_R_switch_z_J": np.nan,
    "fourier_detection_prefix_complete": False,
    "fourier_detection_outgoing_tail_complete": False,
    "fourier_detection_history": "incoming_R_far_to_outgoing_R_far",
    "E_fourier_R_far_in_to_escape_J": np.nan,
    "E_fourier_R_far_in_to_escape_x_J": np.nan,
    "E_fourier_R_far_in_to_escape_y_J": np.nan,
    "E_fourier_R_far_in_to_escape_z_J": np.nan,
    "fourier_R_far_in_to_escape_cos_x_N_s": np.nan,
    "fourier_R_far_in_to_escape_cos_y_N_s": np.nan,
    "fourier_R_far_in_to_escape_cos_z_N_s": np.nan,
    "fourier_R_far_in_to_escape_sin_x_N_s": np.nan,
    "fourier_R_far_in_to_escape_sin_y_N_s": np.nan,
    "fourier_R_far_in_to_escape_sin_z_N_s": np.nan,
    "fourier_outgoing_tail_complete": False,
    "fourier_outgoing_tail_status": "not_run",
    "fourier_outgoing_tail_duration_s": np.nan,
    "fourier_outgoing_tail_end_radius_m": np.nan,
    "fourier_outgoing_tail_time_origin": "incoming_R_switch_crossing",
    "fourier_outgoing_tail_path_model": (
        "DM_trap_only_plus_free_harmonic_ion_no_Coulomb_backreaction"
    ),
    "fourier_outgoing_tail_cos_x_N_s": 0.0,
    "fourier_outgoing_tail_cos_y_N_s": 0.0,
    "fourier_outgoing_tail_cos_z_N_s": 0.0,
    "fourier_outgoing_tail_sin_x_N_s": 0.0,
    "fourier_outgoing_tail_sin_y_N_s": 0.0,
    "fourier_outgoing_tail_sin_z_N_s": 0.0,
    "fourier_outgoing_tail_quad_abs_error_x_N_s": np.nan,
    "fourier_outgoing_tail_quad_abs_error_y_N_s": np.nan,
    "fourier_outgoing_tail_quad_abs_error_z_N_s": np.nan,
    "E_fourier_outgoing_tail_standalone_J": np.nan,
    "E_fourier_outgoing_tail_x_J": np.nan,
    "E_fourier_outgoing_tail_y_J": np.nan,
    "E_fourier_outgoing_tail_z_J": np.nan,
    "fourier_history_t_final_s": np.nan,
    "fourier_cos_x_N_s": np.nan,
    "fourier_cos_y_N_s": np.nan,
    "fourier_cos_z_N_s": np.nan,
    "fourier_sin_x_N_s": np.nan,
    "fourier_sin_y_N_s": np.nan,
    "fourier_sin_z_N_s": np.nan,
    "fourier_from_R_switch_cos_x_N_s": np.nan,
    "fourier_from_R_switch_cos_y_N_s": np.nan,
    "fourier_from_R_switch_cos_z_N_s": np.nan,
    "fourier_from_R_switch_sin_x_N_s": np.nan,
    "fourier_from_R_switch_sin_y_N_s": np.nan,
    "fourier_from_R_switch_sin_z_N_s": np.nan,
    "fourier_mechanical_comparison_energy": "E_fourier_from_R_switch_J",
    "fourier_mechanical_abs_difference_J": np.nan,
    "fourier_mechanical_rel_difference": np.nan,
    "fourier_mechanical_consistency_ok": False,
    "fourier_mechanical_consistency_required": bool(
        REQUIRE_FOURIER_MECHANICAL_AGREEMENT
    ),
    "fourier_mechanical_rel_tolerance": FOURIER_MECHANICAL_REL_TOL,
    "fastest_secular_period_s": FASTEST_SECULAR_PERIOD_S,
    "fourier_resolution_step_s": FOURIER_RESOLUTION_STEP_S,
    "interaction_time_inside_R_switch_s": np.nan,
    "interaction_time_inside_R_switch_us": np.nan,
    "interaction_time_inside_R_full_s": np.nan,
    "interaction_time_inside_R_full_us": np.nan,
    "full_t_final_s": np.nan,
    "full_t_max_s": np.nan,
    "full_runtime_s": np.nan,
}

if "bowl_default_fields" in globals():
    DM_ONLY_DEFAULTS.update(bowl_default_fields("dm_only"))
    FULL_DEFAULTS.update(bowl_default_fields("full"))
    FULL_DEFAULTS.update(bowl_default_fields("outgoing_rfar"))
    FULL_DEFAULTS.update(bowl_default_fields("outgoing_outer"))
FULL_DEFAULTS.update({
    "bowl_model_available": False,
    "entered_bowl": False,
    "crossed_bowl_inward": False,
    "crossed_bowl_outward": False,
    "exited_bowl_after_entry": False,
    "ended_inside_bowl": False,
    "bowl_entry_stage": "none",
    "bowl_entry_channel": "none",
    "bowl_entry_x_m": np.nan,
    "bowl_entry_y_m": np.nan,
    "bowl_entry_z_m": np.nan,
    "bowl_entry_U_total_J": np.nan,
    "bowl_entry_U_dc_J": np.nan,
    "bowl_entry_U_rf_J": np.nan,
    "bowl_entry_F_dc_toward_center_N": np.nan,
    "bowl_entry_F_rf_toward_center_N": np.nan,
    "bowl_entry_F_total_toward_center_N": np.nan,
    "bowl_classification_level_J": np.nan,
    "bowl_escape_potential_dc_J": np.nan,
    "bowl_escape_potential_rf_J": np.nan,
    "interaction_time_inside_bowl_s": 0.0,
    "interaction_time_inside_bowl_us": 0.0,
    "minimum_bowl_signed_distance_m": np.nan,
    "minimum_bowl_signed_distance_um": np.nan,
    "bowl_trajectory_class": "not_classified",
    "bowl_hypothesis_counterexample": False,
    "R_full_entry_time_after_switch_s": np.nan,
    "R_full_exit_time_after_switch_s": np.nan,
    "R_full_entry_count": 0,
    "R_full_exit_count": 0,
    "ended_inside_R_full": False,
})

def dm_only_worker(row: dict[str, Any], invalid_z_floor_m: float | None) -> dict[str, Any]:
    row = recalculate_row_v_b_radii(row)
    output = dict(row)
    output.update(DM_ONLY_DEFAULTS)
    regime = infer_regime(row)
    output["trajectory_regime"] = regime
    r_switch_m = numeric_value(row, "R_switch_m")
    b_m = numeric_value(row, "b_m")
    output["b_over_R_switch"] = (
        b_m / r_switch_m
        if np.isfinite(b_m) and np.isfinite(r_switch_m) and r_switch_m > 0.0
        else np.nan
    )

    if regime == "adiabatic":
        output.update(
            {
                "dm_only_success": True,
                "dm_only_status": "test0p75_adiabatic_reject",
                "dm_only_message": "Analytical adiabatic rejection",
                "entered_switch": False,
            }
        )
        return output

    mass_name = "m_dm_kg" if "m_dm_kg" in row else "m_dm"
    x_far = vector_from_row(row, ("x_far_m", "y_far_m", "z_far_m"))
    v_far = vector_from_row(row, ("vx_far_m_s", "vy_far_m_s", "vz_far_m_s"))
    result = run_dm_trap_only_to_switch(
        x_far,
        v_far,
        m_dm_value=numeric_value(row, mass_name),
        eps_value=numeric_value(row, "eps"),
        r_switch_m=r_switch_m,
        invalid_z_floor_m=invalid_z_floor_m,
    )
    output.update(result)
    min_radius = numeric_value(output, "dm_only_min_radius_m")
    output["dm_only_min_radius_um"] = (
        min_radius * 1.0e6 if np.isfinite(min_radius) else np.nan
    )
    output["dm_only_min_radius_over_R_switch"] = (
        min_radius / r_switch_m
        if np.isfinite(min_radius) and r_switch_m > 0.0
        else np.nan
    )
    return output


def full_worker(row: dict[str, Any], invalid_z_floor_m: float | None) -> dict[str, Any]:
    output = dict(row)
    output.update(FULL_DEFAULTS)
    regime = infer_regime(row)
    output["trajectory_regime"] = regime

    if regime == "adiabatic":
        output.update({
            "full_success": True,
            "full_status": "test0p75_adiabatic_reject",
            "full_message": "Analytical adiabatic rejection",
            "above_energy_threshold": False,
        })
        return output
    if not bool_value(row, "entered_switch"):
        output["full_status"] = "not_entered_switch"
        return output

    x_switch = vector_from_row(row, ("x_switch_x_m", "x_switch_y_m", "x_switch_z_m"))
    v_switch = vector_from_row(row, ("vx_switch_m_s", "vy_switch_m_s", "vz_switch_m_s"))
    mass_name = "m_dm_kg" if "m_dm_kg" in row else "m_dm"
    common = {
        "m_dm_value": numeric_value(row, mass_name),
        "eps_value": numeric_value(row, "eps"),
        "r_full_m": numeric_value(row, "R_full_m"),
    }
    if regime == "rutherford":
        result = run_local_rutherford_after_reach(x_switch, v_switch, **common)
    else:
        prefix_cos = np.nan_to_num(np.array([
            numeric_value(row, "fourier_prefix_cos_x_N_s", 0.0),
            numeric_value(row, "fourier_prefix_cos_y_N_s", 0.0),
            numeric_value(row, "fourier_prefix_cos_z_N_s", 0.0),
        ], dtype=float), nan=0.0)
        prefix_sin = np.nan_to_num(np.array([
            numeric_value(row, "fourier_prefix_sin_x_N_s", 0.0),
            numeric_value(row, "fourier_prefix_sin_y_N_s", 0.0),
            numeric_value(row, "fourier_prefix_sin_z_N_s", 0.0),
        ], dtype=float), nan=0.0)
        x_far = vector_from_row(row, ("x_far_m", "y_far_m", "z_far_m"))
        result = run_full_coupled_after_reach(
            x_switch,
            v_switch,
            r_switch_m=numeric_value(row, "R_switch_m"),
            r_far_m=float(np.linalg.norm(x_far)),
            invalid_z_floor_m=invalid_z_floor_m,
            prefix_cos_N_s=prefix_cos,
            prefix_sin_N_s=prefix_sin,
            prefix_complete=bool_value(row, "fourier_prefix_complete"),
            **common,
        )
    output.update(result)
    return output



def finalize_status(row: pd.Series | dict[str, Any]) -> str:
    regime = infer_regime(row)
    if regime == "adiabatic":
        return "resolved_adiabatic_below_threshold"
    dm_status = str(row.get("dm_only_status", ""))
    full_status = str(row.get("full_status", ""))
    if dm_status in {
        "dm_only_escaped_without_switch",
        "trajectory_class_rejected_no_in_range_sample_reaches_switch",
        "stopping_distance_outward_reject",
        "existing_exact_analytic_reject",
    }:
        return "resolved_no_switch_entry"
    if any(token in dm_status for token in ("timeout", "failure", "invalid", "rebuild")):
        return "unresolved_dm_only"

    resolved_full_statuses = {
        "entered_switch_but_missed_R_full",
        "rutherford_local_miss_R_full",
        "reached_R_full_and_escaped",
        "rutherford_local_reached_R_full",
    }
    if full_status in resolved_full_statuses:
        if (
            DETECTION_ENERGY_POLICY in {"fourier", "both", "either"}
            and REQUIRE_R_FAR_PREFIX_COMPLETE
            and not bool_value(row, "fourier_detection_prefix_complete")
        ):
            return "unresolved_fourier_R_far_prefix"
        if (
            DETECTION_ENERGY_POLICY in {"fourier", "both", "either"}
            and REQUIRE_OUTGOING_R_FAR_TAIL_COMPLETE
            and not bool_value(
                row, "fourier_detection_outgoing_tail_complete"
            )
        ):
            return "unresolved_fourier_outgoing_R_far_tail"
        if (
            bool_value(row, "fourier_mechanical_consistency_required")
            and not bool_value(row, "fourier_mechanical_consistency_ok")
        ):
            return "unresolved_energy_inconsistent"
        if bool_value(row, "above_energy_threshold"):
            return "resolved_detectable"
        if full_status in {
            "entered_switch_but_missed_R_full",
            "rutherford_local_miss_R_full",
        }:
            return "resolved_entered_switch_but_missed_R_full"
        return "resolved_below_energy_threshold"
    if any(token in full_status for token in ("timeout", "failure", "invalid")):
        return "unresolved_full_stage"
    return "unresolved_other"


# =============================================================================
# BATCH EXECUTION
# =============================================================================


# =============================================================================
# BATCH EXECUTION
# =============================================================================


def compact_summary(prefix: str, index: int, total: int, row: dict[str, Any]) -> str:
    """Format one result line immediately after a trajectory finishes."""
    trajectory_id = str(row.get("trajectory_id", ""))
    status = str(
        row.get("dm_only_status") if prefix == "DM" else row.get("full_status")
    )
    v_inf = numeric_value(row, "v_inf_m_s")
    b_um = numeric_value(row, "b_m") * 1.0e6
    theta_deg = math.degrees(numeric_value(row, "theta_rad"))
    alpha_deg = math.degrees(numeric_value(row, "alpha_rad"))
    psi_deg = math.degrees(numeric_value(row, "psi_rad"))
    fields = [
        f"[{prefix} {index:5d}/{total:5d}]",
        f"id={trajectory_id}",
        f"v={v_inf:.3f}m/s",
        f"b={b_um:.3f}um",
        f"theta={theta_deg:.1f}deg",
        f"alpha={alpha_deg:.1f}deg",
        f"psi={psi_deg:.1f}deg",
        f"status={status}",
    ]
    if prefix == "DM":
        fields.extend(
            [
                f"switch={'Y' if bool_value(row, 'entered_switch') else 'N'}",
                f"b/Rs={numeric_value(row, 'b_over_R_switch'):.3f}",
                f"rmin={numeric_value(row, 'dm_only_min_radius_um'):.3f}um",
                f"tfinal={numeric_value(row, 'dm_only_t_final_s') * 1.0e6:.3f}us",
                f"runtime={numeric_value(row, 'dm_only_runtime_s'):.3f}s",
            ]
        )
    else:
        fields.extend(
            [
                f"reach={'Y' if bool_value(row, 'reached_R_full') else 'N'}",
                f"detect={'Y' if bool_value(row, 'above_energy_threshold') else 'N'}",
                f"dmin={numeric_value(row, 'd_min_um'):.3f}um",
                f"Emech={numeric_value(row, 'E_mechanical_final_J'):.3e}J",
                f"EftRs={numeric_value(row, 'E_fourier_from_R_switch_J'):.3e}J",
                f"EftRf={numeric_value(row, 'E_fourier_from_R_far_J'):.3e}J",
                f"Esel={numeric_value(row, 'E_detection_J'):.3e}J",
                f"dErel={numeric_value(row, 'fourier_mechanical_rel_difference'):.3e}",
                f"tfinal={numeric_value(row, 'full_t_final_s') * 1.0e6:.3f}us",
                f"runtime={numeric_value(row, 'full_runtime_s'):.3f}s",
            ]
        )
    return " ".join(fields)


def run_threaded_batch(
    rows: list[dict[str, Any]],
    worker: Callable[[dict[str, Any], float | None], dict[str, Any]],
    *,
    invalid_z_floor_m: float | None,
    n_threads: int,
    max_in_flight: int,
    checkpoint_path: Path,
    prefix: str,
) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame()
    completed: list[dict[str, Any]] = []
    total = len(rows)
    next_index = 0
    last_heartbeat = time.perf_counter()

    with ThreadPoolExecutor(max_workers=n_threads) as executor:
        futures: dict[Any, dict[str, Any]] = {}
        while next_index < total or futures:
            while next_index < total and len(futures) < max_in_flight:
                row = rows[next_index]
                future = executor.submit(worker, row, invalid_z_floor_m)
                futures[future] = row
                next_index += 1

            timeout = BATCH_HEARTBEAT_S if BATCH_HEARTBEAT_S is not None else None
            done, _ = wait(
                list(futures),
                timeout=timeout,
                return_when=FIRST_COMPLETED,
            )
            if not done:
                now = time.perf_counter()
                if BATCH_HEARTBEAT_S is not None and now - last_heartbeat >= BATCH_HEARTBEAT_S:
                    progress(
                        f"[{prefix}] heartbeat completed={len(completed)}/{total} "
                        f"in_flight={len(futures)}"
                    )
                    last_heartbeat = now
                continue

            for future in done:
                source = futures.pop(future)
                try:
                    result = future.result()
                except Exception as exc:
                    result = dict(source)
                    if prefix == "DM":
                        result.update(DM_ONLY_DEFAULTS)
                        result["dm_only_status"] = "dm_only_task_exception"
                        result["dm_only_message"] = repr(exc)
                    else:
                        result.update(FULL_DEFAULTS)
                        result["full_status"] = "full_task_exception"
                        result["full_message"] = repr(exc)
                completed.append(result)
                if PRINT_EACH_TRAJECTORY_SUMMARY:
                    progress(compact_summary(prefix, len(completed), total, result))
                if len(completed) % CHECKPOINT_EVERY == 0 or len(completed) == total:
                    pd.DataFrame(completed).to_csv(checkpoint_path, index=False)
                    progress(f"[{prefix}] checkpoint {len(completed)}/{total}: {checkpoint_path}")
    return pd.DataFrame(completed)


def load_existing(path: Path) -> pd.DataFrame:
    if RESUME_EXISTING_OUTPUT and path.exists() and path.stat().st_size > 0:
        data = pd.read_csv(path)
        if "trajectory_id" in data.columns:
            data["trajectory_id"] = data["trajectory_id"].astype(str)
        return data
    return pd.DataFrame()


def order_rows(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    if SORT_FASTEST_FIRST:
        r_far = np.sqrt(
            numeric_series(output, "x_far_m") ** 2
            + numeric_series(output, "y_far_m") ** 2
            + numeric_series(output, "z_far_m") ** 2
        )
        speed = np.sqrt(
            numeric_series(output, "vx_far_m_s") ** 2
            + numeric_series(output, "vy_far_m_s") ** 2
            + numeric_series(output, "vz_far_m_s") ** 2
        )
        output["_estimated_runtime_order"] = r_far / speed.replace(0.0, np.nan)
        output = output.sort_values("_estimated_runtime_order", kind="mergesort")
        output = output.drop(columns="_estimated_runtime_order")
    return output.reset_index(drop=True)


# =============================================================================
# WEIGHTED SUMMARIES
# =============================================================================


def analysis_weight(df: pd.DataFrame) -> pd.Series:
    for column in (
        "test2_analysis_weight_unconditional",
        "population_weight_unconditional",
        "test2_analysis_weight_conditional",
        "population_weight_conditional",
        "impact_weight",
    ):
        if column in df.columns:
            return numeric_series(df, column, 0.0).fillna(0.0).clip(lower=0.0)
    return pd.Series(1.0, index=df.index, dtype=float)


def build_weighted_summaries(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    output = df.copy()
    output["test2_status"] = [finalize_status(row) for _, row in output.iterrows()]
    weight = analysis_weight(output)
    output["analysis_weight_used"] = weight
    total = float(weight.sum())
    resolved = output["test2_status"].astype(str).str.startswith("resolved")
    detectable = output["test2_status"].eq("resolved_detectable")
    entered = bool_series(output, "entered_switch")
    reached = bool_series(output, "reached_R_full")
    global_summary = pd.DataFrame(
        [
            {
                "n_rows": len(output),
                "total_analysis_weight": total,
                "resolved_weight": float(weight[resolved].sum()),
                "unresolved_weight": float(weight[~resolved].sum()),
                "entered_switch_weight": float(weight[entered].sum()),
                "reached_R_full_weight": float(weight[reached].sum()),
                "detectable_weight": float(weight[detectable].sum()),
                "resolved_weight_fraction": (
                    float(weight[resolved].sum()) / total if total > 0.0 else np.nan
                ),
                "detectable_weight_fraction": (
                    float(weight[detectable].sum()) / total if total > 0.0 else np.nan
                ),
            }
        ]
    )
    classification = (
        output.assign(_weight=weight)
        .groupby("test2_status", dropna=False)
        .agg(n_rows=("trajectory_id", "size"), analysis_weight=("_weight", "sum"))
        .reset_index()
    )
    classification["weight_fraction"] = (
        classification["analysis_weight"] / total if total > 0.0 else np.nan
    )
    return output, global_summary, classification


# =============================================================================
# EMPTY / STUB-ONLY TEST 2 OUTPUTS
# =============================================================================


def _add_default_columns(
    frame: pd.DataFrame,
    defaults: dict[str, Any],
) -> pd.DataFrame:
    """Return a copy containing every default column, even with zero rows."""
    output = frame.copy()
    for name, value in defaults.items():
        if name not in output.columns:
            output[name] = pd.Series(index=output.index, dtype=object)
            if len(output):
                output[name] = value
    return output


def write_empty_or_stub_only_test2_outputs(selected: pd.DataFrame) -> None:
    """Write valid Test 2 products when Test 1.5 selected no dynamic rows.

    Analytical rejection stubs are still carried into the combined and weighted
    outputs, so an empty dynamic pilot remains an auditable, valid pipeline
    result rather than an exception.
    """
    progress(
        "[AUTO TEST 2] Test 1.5 supplied zero dynamic rows; writing "
        "header-safe empty/stub-only Test 2 outputs"
    )

    probes = pd.DataFrame(
        columns=["trajectory_id", "trajectory_class_id", "dm_only_status"]
    )
    class_summary = pd.DataFrame(
        columns=[
            "trajectory_class_id",
            "trajectory_class_reach_status",
            "class_reach_accepted",
            "class_reach_rejected",
            "class_reach_unresolved",
            "class_reach_n_planned",
            "class_reach_n_run",
            "class_reach_n_entered",
            "class_reach_n_resolved_miss",
        ]
    )
    dm_screen = _add_default_columns(selected, DM_ONLY_DEFAULTS)
    full_results = _add_default_columns(selected, FULL_DEFAULTS)

    probes.to_csv(CLASS_REACH_PROBES_CSV, index=False)
    class_summary.to_csv(CLASS_REACH_SUMMARY_CSV, index=False)
    dm_screen.to_csv(DM_ONLY_SCREEN_CSV, index=False)
    full_results.to_csv(FULL_STAGE_RESULTS_CSV, index=False)

    combined = _add_default_columns(selected, DM_ONLY_DEFAULTS)
    combined = _add_default_columns(combined, FULL_DEFAULTS)

    if (
        PREFILTER_REJECTION_STUB_CSV is not None
        and PREFILTER_REJECTION_STUB_CSV.exists()
        and PREFILTER_REJECTION_STUB_CSV.stat().st_size > 0
    ):
        try:
            stubs = pd.read_csv(PREFILTER_REJECTION_STUB_CSV)
        except pd.errors.EmptyDataError:
            stubs = pd.DataFrame()
        if len(stubs):
            stubs["trajectory_id"] = stubs["trajectory_id"].astype(str)
            stubs = _add_default_columns(stubs, DM_ONLY_DEFAULTS)
            stubs = _add_default_columns(stubs, FULL_DEFAULTS)
            reason = stubs.get(
                "test1p5_rejection_reason",
                pd.Series(
                    "existing_exact_analytic_reject", index=stubs.index
                ),
            ).astype(str)
            adiabatic = reason.eq("test0p75_adiabatic_reject")
            stubs["dm_only_success"] = True
            stubs["dm_only_status"] = reason
            stubs["entered_switch"] = False
            stubs["full_success"] = True
            stubs["full_status"] = reason
            stubs["full_simulation_run"] = False
            stubs["reached_R_full"] = False
            stubs["above_energy_threshold"] = False
            if "trajectory_regime" not in stubs.columns:
                stubs["trajectory_regime"] = np.where(
                    adiabatic, "adiabatic", "resonant"
                )
            else:
                stubs.loc[adiabatic, "trajectory_regime"] = "adiabatic"
            combined = pd.concat(
                [combined, stubs], ignore_index=True, sort=False
            )

    if "trajectory_id" not in combined.columns:
        combined["trajectory_id"] = pd.Series(dtype=str)
    combined = combined.drop_duplicates("trajectory_id", keep="last")
    combined["test2_status"] = [
        finalize_status(row) for _, row in combined.iterrows()
    ]
    combined.to_csv(COMBINED_RESULTS_CSV, index=False)

    weighted, global_summary, classification = build_weighted_summaries(combined)
    weighted.to_csv(WEIGHTED_ANALYSIS_CSV, index=False)
    global_summary.to_csv(WEIGHTED_GLOBAL_SUMMARY_CSV, index=False)
    classification.to_csv(WEIGHTED_CLASSIFICATION_SUMMARY_CSV, index=False)

    progress(f"Saved class reach probes: {CLASS_REACH_PROBES_CSV}")
    progress(f"Saved class reach summary: {CLASS_REACH_SUMMARY_CSV}")
    progress(f"Saved DM-only screen: {DM_ONLY_SCREEN_CSV}")
    progress(f"Saved full-stage results: {FULL_STAGE_RESULTS_CSV}")
    progress(f"Saved combined results: {COMBINED_RESULTS_CSV}")
    progress(f"Saved weighted analysis: {WEIGHTED_ANALYSIS_CSV}")
    progress(global_summary.to_string(index=False))


# =============================================================================
# MAIN
# =============================================================================


def main() -> None:
    validate_trap_environment()
    invalid_z_floor_m = resolve_invalid_z_floor()
    selected, production_input_path, diagnostic_input_path = load_test2_inputs()
    progress(f"Test 2 production input: {production_input_path}")
    progress(
        "Test 2 escape-channel diagnostics: "
        + (str(diagnostic_input_path) if diagnostic_input_path is not None else "[none]")
    )
    validate_input(selected)
    validate_parameter_dataframe(selected, "Test 1.5 input")
    selected = annotate_and_validate_b_support(selected)
    selected = add_trajectory_class_support(selected)
    if ROW_QUERY:
        selected = selected.query(ROW_QUERY).copy()
    selected = order_rows(selected)
    if MAX_TRAJECTORIES is not None:
        selected = selected.head(int(MAX_TRAJECTORIES)).copy()

    if len(selected) == 0:
        write_empty_or_stub_only_test2_outputs(selected)
        return

    diagnostic_mask = bool_series(selected, "test2_diagnostic_only", False)
    production_selected = selected.loc[~diagnostic_mask].copy()
    diagnostic_selected = selected.loc[diagnostic_mask].copy()

    probes, class_summary, anchors = run_class_reach_screen(
        production_selected, invalid_z_floor_m
    )
    if len(diagnostic_selected):
        diagnostic_summaries: list[dict[str, Any]] = []
        for class_id, group in diagnostic_selected.groupby(
            "trajectory_class_id", sort=False
        ):
            representative = group.iloc[0]
            diagnostic_summaries.append({
                "trajectory_class_id": str(class_id),
                "speed_label": str(representative.get("speed_label", "")),
                "trajectory_regime": infer_regime(representative),
                "theta_index": int(numeric_value(representative, "theta_index", -1)),
                "alpha_index": int(numeric_value(representative, "alpha_index", -1)),
                "theta_rad": numeric_value(representative, "theta_rad"),
                "alpha_rad": numeric_value(representative, "alpha_rad"),
                "psi_rad": numeric_value(representative, "psi_rad"),
                "R_switch_m": numeric_value(representative, "R_switch_m"),
                "trajectory_class_reach_status": "screen_disabled",
                "class_reach_accepted": True,
                "class_reach_rejected": False,
                "class_reach_unresolved": False,
                "class_reach_n_planned": 0,
                "class_reach_n_run": 0,
                "class_reach_n_entered": 0,
                "class_reach_n_resolved_miss": 0,
                "class_reach_message": (
                    "Focused escape-channel diagnostic; class probe screen skipped"
                ),
                "class_reach_support_lower_m": numeric_value(
                    representative, "class_b_support_lower_m"
                ),
                "class_reach_support_upper_m": numeric_value(
                    representative, "class_b_support_upper_m"
                ),
            })
        class_summary = pd.concat(
            [class_summary, pd.DataFrame(diagnostic_summaries)],
            ignore_index=True,
            sort=False,
        )

    probes.to_csv(CLASS_REACH_PROBES_CSV, index=False)
    class_summary.to_csv(CLASS_REACH_SUMMARY_CSV, index=False)
    selected = selected.merge(
        class_summary,
        on="trajectory_class_id",
        how="left",
        validate="many_to_one",
        suffixes=("", "_class"),
    )
    if len(anchors):
        anchors = anchors.merge(
            class_summary,
            on="trajectory_class_id",
            how="left",
            validate="many_to_one",
            suffixes=("", "_summary"),
        )

    rejected_mask = selected["trajectory_class_reach_status"].eq(
        "rejected_no_in_range_sample_reaches_switch"
    )
    class_rejected_stubs = make_class_rejection_stubs(selected.loc[rejected_mask])
    dynamic_selected = selected.loc[~rejected_mask].copy()

    existing_dm = load_existing(DM_ONLY_SCREEN_CSV)
    completed_ids = set(existing_dm.get("trajectory_id", pd.Series(dtype=str)).astype(str))
    pending_dm = dynamic_selected.loc[
        ~dynamic_selected["trajectory_id"].isin(completed_ids)
    ].copy()
    progress(
        f"[DM-ONLY] production={len(selected):,} class_rejected={int(rejected_mask.sum()):,} "
        f"dynamic={len(dynamic_selected):,} pending={len(pending_dm):,} "
        f"threads={N_THREADS_DM_ONLY}"
    )
    new_dm = run_threaded_batch(
        pending_dm.to_dict("records"),
        dm_only_worker,
        invalid_z_floor_m=invalid_z_floor_m,
        n_threads=N_THREADS_DM_ONLY,
        max_in_flight=MAX_IN_FLIGHT_DM_ONLY,
        checkpoint_path=DM_ONLY_SCREEN_CSV,
        prefix="DM",
    )
    dm_screen = pd.concat(
        [existing_dm, new_dm, class_rejected_stubs],
        ignore_index=True,
        sort=False,
    ).drop_duplicates("trajectory_id", keep="last")
    dm_screen.to_csv(DM_ONLY_SCREEN_CSV, index=False)

    existing_full = load_existing(FULL_STAGE_RESULTS_CSV)
    full_completed_ids = set(
        existing_full.get("trajectory_id", pd.Series(dtype=str)).astype(str)
    )
    production_needs_full = dm_screen.loc[
        bool_series(dm_screen, "entered_switch")
        | dm_screen.apply(lambda row: infer_regime(row) == "adiabatic", axis=1)
    ].copy()
    anchor_needs_full = anchors.copy() if len(anchors) else pd.DataFrame()
    needs_full = pd.concat(
        [production_needs_full, anchor_needs_full],
        ignore_index=True,
        sort=False,
    ).drop_duplicates("trajectory_id", keep="first")
    pending_full = needs_full.loc[
        ~needs_full["trajectory_id"].astype(str).isin(full_completed_ids)
    ].copy()
    progress(
        f"[FULL] production_or_anchor={len(needs_full):,} "
        f"already_completed={len(existing_full):,} pending={len(pending_full):,} "
        f"threads={N_THREADS_FULL}"
    )
    new_full = run_threaded_batch(
        pending_full.to_dict("records"),
        full_worker,
        invalid_z_floor_m=invalid_z_floor_m,
        n_threads=N_THREADS_FULL,
        max_in_flight=MAX_IN_FLIGHT_FULL,
        checkpoint_path=FULL_STAGE_RESULTS_CSV,
        prefix="FULL",
    )
    full_results = pd.concat([existing_full, new_full], ignore_index=True, sort=False)
    full_results = full_results.drop_duplicates("trajectory_id", keep="last")
    full_results.to_csv(FULL_STAGE_RESULTS_CSV, index=False)

    full_ids = set(full_results.get("trajectory_id", pd.Series(dtype=str)).astype(str))
    dm_only_terminal = dm_screen.loc[~dm_screen["trajectory_id"].isin(full_ids)].copy()
    for key, value in FULL_DEFAULTS.items():
        if key not in dm_only_terminal.columns:
            dm_only_terminal[key] = value
    combined = pd.concat([dm_only_terminal, full_results], ignore_index=True, sort=False)

    if (
        PREFILTER_REJECTION_STUB_CSV is not None
        and PREFILTER_REJECTION_STUB_CSV.exists()
        and PREFILTER_REJECTION_STUB_CSV.stat().st_size > 0
    ):
        stubs = pd.read_csv(PREFILTER_REJECTION_STUB_CSV)
        if len(stubs):
            stubs["trajectory_id"] = stubs["trajectory_id"].astype(str)
            for key, value in DM_ONLY_DEFAULTS.items():
                if key not in stubs.columns:
                    stubs[key] = value
            for key, value in FULL_DEFAULTS.items():
                if key not in stubs.columns:
                    stubs[key] = value
            reason = stubs.get(
                "test1p5_rejection_reason",
                pd.Series("existing_exact_analytic_reject", index=stubs.index),
            ).astype(str)
            adiabatic = reason.eq("test0p75_adiabatic_reject")
            stubs["dm_only_success"] = True
            stubs["dm_only_status"] = reason
            stubs["entered_switch"] = False
            stubs["full_success"] = True
            stubs["full_status"] = reason
            stubs["full_simulation_run"] = False
            stubs["reached_R_full"] = False
            stubs["above_energy_threshold"] = False
            if "trajectory_regime" not in stubs.columns:
                stubs["trajectory_regime"] = np.where(
                    adiabatic, "adiabatic", "resonant"
                )
            else:
                stubs.loc[adiabatic, "trajectory_regime"] = "adiabatic"
            combined = pd.concat([combined, stubs], ignore_index=True, sort=False)

    combined = combined.drop_duplicates("trajectory_id", keep="last")
    combined["test2_status"] = [finalize_status(row) for _, row in combined.iterrows()]
    combined.to_csv(COMBINED_RESULTS_CSV, index=False)

    weighted, global_summary, classification = build_weighted_summaries(combined)
    weighted.to_csv(WEIGHTED_ANALYSIS_CSV, index=False)
    global_summary.to_csv(WEIGHTED_GLOBAL_SUMMARY_CSV, index=False)
    classification.to_csv(WEIGHTED_CLASSIFICATION_SUMMARY_CSV, index=False)

    progress(f"Saved class reach probes: {CLASS_REACH_PROBES_CSV}")
    progress(f"Saved class reach summary: {CLASS_REACH_SUMMARY_CSV}")
    progress(f"Saved DM-only screen: {DM_ONLY_SCREEN_CSV}")
    progress(f"Saved full-stage results: {FULL_STAGE_RESULTS_CSV}")
    progress(f"Saved combined results: {COMBINED_RESULTS_CSV}")
    progress(f"Saved weighted analysis: {WEIGHTED_ANALYSIS_CSV}")
    if len(class_summary):
        progress(
            class_summary["trajectory_class_reach_status"]
            .value_counts(dropna=False)
            .to_string()
        )
    progress(global_summary.to_string(index=False))



# =============================================================================
# V8 FULL-PULSE OUTER-TAIL EXTENSION
# =============================================================================
# V7 includes incoming R_far -> core -> outgoing R_far. The Coulomb force is
# small but not identically zero at R_far. V8 adds the matching weak tails
# incoming R_outer -> R_far and outgoing R_far -> R_outer, preserving one
# switch-centered Fourier phase convention. Detection uses the coherent sum of
# all four segments. The original R_far round-trip energy is retained as a
# diagnostic.

INCLUDE_OUTER_R_TAILS = bool(globals().get("USER_INCLUDE_OUTER_R_TAILS", True))
REQUIRE_OUTER_R_TAILS_COMPLETE = bool(
    globals().get("USER_REQUIRE_OUTER_R_TAILS_COMPLETE", True)
)
FOURIER_OUTER_RADIUS_M = float(
    globals().get("USER_FOURIER_OUTER_RADIUS_M", 20.0e-3)
)
OUTER_TAIL_RTOL = float(globals().get("USER_OUTER_TAIL_RTOL", DM_ONLY_RTOL))
OUTER_TAIL_ATOL = float(globals().get("USER_OUTER_TAIL_ATOL", DM_ONLY_ATOL))
OUTER_TAIL_MAX_STEP_S = float(
    globals().get("USER_OUTER_TAIL_MAX_STEP_S", DM_ONLY_MAX_STEP_S)
)
OUTER_TAIL_TIME_FACTOR = float(globals().get("USER_OUTER_TAIL_TIME_FACTOR", 4.0))
OUTER_TAIL_MIN_TIME_S = float(
    globals().get("USER_OUTER_TAIL_MIN_TIME_S", 50.0e-6)
)
OUTER_TAIL_QUAD_EPSREL = float(
    globals().get("USER_OUTER_TAIL_QUAD_EPSREL", PREFIX_QUAD_EPSREL)
)
OUTER_TAIL_QUAD_EPSABS_DIMENSIONLESS = float(
    globals().get(
        "USER_OUTER_TAIL_QUAD_EPSABS_DIMENSIONLESS",
        PREFIX_QUAD_EPSABS_DIMENSIONLESS,
    )
)
OUTER_TAIL_QUAD_LIMIT = int(
    globals().get("USER_OUTER_TAIL_QUAD_LIMIT", PREFIX_QUAD_LIMIT)
)
OUTER_TAIL_FORCE_SCALE_SAMPLES = int(
    globals().get("USER_OUTER_TAIL_FORCE_SCALE_SAMPLES", 21)
)
if not np.isfinite(FOURIER_OUTER_RADIUS_M) or FOURIER_OUTER_RADIUS_M <= 0.0:
    raise ValueError("USER_FOURIER_OUTER_RADIUS_M must be finite and positive")
if OUTER_TAIL_FORCE_SCALE_SAMPLES < 3:
    raise ValueError("USER_OUTER_TAIL_FORCE_SCALE_SAMPLES must be at least 3")


def _empty_outer_fields(
    side: str,
    status: str,
    complete: bool,
    *,
    duration_s: float = 0.0,
    start_radius_m: float = np.nan,
    end_radius_m: float = np.nan,
) -> dict[str, Any]:
    side = str(side)
    p = f"fourier_{side}_outer_tail"
    return {
        f"{p}_complete": bool(complete),
        f"{p}_status": str(status),
        f"{p}_duration_s": float(duration_s),
        f"{p}_start_radius_m": float(start_radius_m),
        f"{p}_end_radius_m": float(end_radius_m),
        f"{p}_target_radius_m": float(FOURIER_OUTER_RADIUS_M),
        f"{p}_cos_x_N_s": 0.0,
        f"{p}_cos_y_N_s": 0.0,
        f"{p}_cos_z_N_s": 0.0,
        f"{p}_sin_x_N_s": 0.0,
        f"{p}_sin_y_N_s": 0.0,
        f"{p}_sin_z_N_s": 0.0,
        f"{p}_quad_abs_error_x_N_s": np.nan,
        f"{p}_quad_abs_error_y_N_s": np.nan,
        f"{p}_quad_abs_error_z_N_s": np.nan,
        f"{p}_force_at_outer_N": np.nan,
        f"{p}_force_at_R_far_N": np.nan,
        f"E_fourier_{side}_outer_tail_standalone_J": 0.0,
        f"E_fourier_{side}_outer_tail_x_J": 0.0,
        f"E_fourier_{side}_outer_tail_y_J": 0.0,
        f"E_fourier_{side}_outer_tail_z_J": 0.0,
    }


def _free_ion_velocity(t: float, x0: np.ndarray, v0: np.ndarray) -> np.ndarray:
    t = float(t)
    x0 = np.asarray(x0, dtype=float).reshape(3)
    v0 = np.asarray(v0, dtype=float).reshape(3)
    omega = np.asarray(omega_vec, dtype=float).reshape(3)
    result = np.empty(3, dtype=float)
    mask = np.abs(omega) > 0.0
    phase = omega[mask] * t
    result[mask] = -x0[mask] * omega[mask] * np.sin(phase) + v0[mask] * np.cos(phase)
    result[~mask] = v0[~mask]
    return result


def _integrate_outer_force(
    side: str,
    dm_position: Callable[[float], np.ndarray],
    ion_position: Callable[[float], np.ndarray],
    duration_s: float,
    global_start_time_s: float,
    eps_value: float,
    *,
    start_radius_m: float,
    end_radius_m: float,
) -> dict[str, Any]:
    duration = float(duration_s)
    if not np.isfinite(duration) or duration < 0.0:
        return _empty_outer_fields(
            side, "invalid_outer_tail_duration", False,
            duration_s=duration, start_radius_m=start_radius_m,
            end_radius_m=end_radius_m,
        )
    if duration == 0.0:
        return _empty_outer_fields(
            side, "zero_length_outer_tail", True,
            start_radius_m=start_radius_m, end_radius_m=end_radius_m,
        )
    p = f"fourier_{side}_outer_tail"
    omega_modes = np.asarray(omega_vec, dtype=float).reshape(3)

    def force_at(t: float) -> np.ndarray:
        return coulomb_force_on_ion_from_positions(
            ion_position(float(t)), dm_position(float(t)), eps_value
        )

    sample_times = np.linspace(0.0, duration, OUTER_TAIL_FORCE_SCALE_SAMPLES)
    scale = max(
        max(float(np.linalg.norm(force_at(t))) for t in sample_times),
        np.finfo(float).tiny,
    )
    amp = np.zeros(3, dtype=complex)
    err = np.zeros(3, dtype=float)
    for j, omega in enumerate(omega_modes):
        def component(u: float) -> float:
            return float(force_at(float(u) * duration)[j] / scale)
        w = float(omega) * duration
        if abs(w) <= 1.0e-14:
            cval, cerr = quad(
                component, 0.0, 1.0,
                epsabs=OUTER_TAIL_QUAD_EPSABS_DIMENSIONLESS,
                epsrel=OUTER_TAIL_QUAD_EPSREL,
                limit=OUTER_TAIL_QUAD_LIMIT,
            )
            sval, serr = 0.0, 0.0
        else:
            cval, cerr = quad(
                component, 0.0, 1.0, weight="cos", wvar=w,
                epsabs=OUTER_TAIL_QUAD_EPSABS_DIMENSIONLESS,
                epsrel=OUTER_TAIL_QUAD_EPSREL,
                limit=OUTER_TAIL_QUAD_LIMIT,
            )
            sval, serr = quad(
                component, 0.0, 1.0, weight="sin", wvar=w,
                epsabs=OUTER_TAIL_QUAD_EPSABS_DIMENSIONLESS,
                epsrel=OUTER_TAIL_QUAD_EPSREL,
                limit=OUTER_TAIL_QUAD_LIMIT,
            )
        local = complex(cval, sval) * scale * duration
        amp[j] = local * np.exp(1j * float(omega) * float(global_start_time_s))
        err[j] = math.hypot(float(cerr), float(serr)) * scale * duration
    modes = np.abs(amp) ** 2 / (2.0 * float(m_ion))
    force_start = float(np.linalg.norm(force_at(0.0)))
    force_end = float(np.linalg.norm(force_at(duration)))
    incoming = side == "incoming"
    return {
        f"{p}_complete": True,
        f"{p}_status": f"integrated_{side}_R_outer_R_far_tail",
        f"{p}_duration_s": duration,
        f"{p}_start_radius_m": float(start_radius_m),
        f"{p}_end_radius_m": float(end_radius_m),
        f"{p}_target_radius_m": float(FOURIER_OUTER_RADIUS_M),
        f"{p}_cos_x_N_s": float(amp[0].real),
        f"{p}_cos_y_N_s": float(amp[1].real),
        f"{p}_cos_z_N_s": float(amp[2].real),
        f"{p}_sin_x_N_s": float(amp[0].imag),
        f"{p}_sin_y_N_s": float(amp[1].imag),
        f"{p}_sin_z_N_s": float(amp[2].imag),
        f"{p}_quad_abs_error_x_N_s": float(err[0]),
        f"{p}_quad_abs_error_y_N_s": float(err[1]),
        f"{p}_quad_abs_error_z_N_s": float(err[2]),
        f"{p}_force_at_outer_N": force_start if incoming else force_end,
        f"{p}_force_at_R_far_N": force_end if incoming else force_start,
        f"E_fourier_{side}_outer_tail_standalone_J": float(np.sum(modes)),
        f"E_fourier_{side}_outer_tail_x_J": float(modes[0]),
        f"E_fourier_{side}_outer_tail_y_J": float(modes[1]),
        f"E_fourier_{side}_outer_tail_z_J": float(modes[2]),
    }


def _incoming_outer_tail(
    x_far: np.ndarray,
    v_far: np.ndarray,
    prefix_duration_s: float,
    m_dm_value: float,
    eps_value: float,
    invalid_z_floor_m: float | None,
) -> dict[str, Any]:
    x_far = np.asarray(x_far, dtype=float).reshape(3)
    v_far = np.asarray(v_far, dtype=float).reshape(3)
    r_far = float(np.linalg.norm(x_far))
    target = float(FOURIER_OUTER_RADIUS_M)
    if not INCLUDE_OUTER_R_TAILS:
        return _empty_outer_fields("incoming", "outer_tails_disabled", True,
                                   start_radius_m=r_far, end_radius_m=r_far)
    if target <= r_far * (1.0 + 1.0e-10):
        return _empty_outer_fields("incoming", "R_outer_not_beyond_R_far", True,
                                   start_radius_m=r_far, end_radius_m=r_far)
    base_rhs = dm_only_rhs(m_dm_value, eps_value)
    def rhs_back(t: float, y: np.ndarray) -> np.ndarray:
        return -np.asarray(base_rhs(-float(t), y), dtype=float)
    def event(_t: float, y: np.ndarray) -> float:
        return float(np.linalg.norm(y[:3]) - target)
    event.terminal = True
    event.direction = 1
    events: list[Callable] = [event]
    if invalid_z_floor_m is not None:
        def invalid_event(_t: float, y: np.ndarray) -> float:
            return float(y[2] - invalid_z_floor_m)
        invalid_event.terminal = True
        invalid_event.direction = 0
        events.append(invalid_event)
    speed = max(float(np.linalg.norm(v_far)), 1.0e-12)
    travel = max((target-r_far)/speed, 0.0)
    tmax = max(OUTER_TAIL_TIME_FACTOR*travel, travel+OUTER_TAIL_MIN_TIME_S)
    sol = solve_ivp(
        rhs_back, (0.0, tmax), np.concatenate([x_far, v_far]),
        method="DOP853", rtol=OUTER_TAIL_RTOL, atol=OUTER_TAIL_ATOL,
        max_step=OUTER_TAIL_MAX_STEP_S, events=events, dense_output=True,
    )
    reached = len(sol.t_events[0]) > 0
    invalid = len(events)>1 and len(sol.t_events[1])>0
    if not reached or sol.sol is None:
        status = "incoming_outer_invalid_domain" if invalid else (
            "incoming_outer_solver_failure" if not sol.success
            else "incoming_outer_did_not_reach_R_outer"
        )
        return _empty_outer_fields(
            "incoming", status, False,
            duration_s=float(sol.t[-1]) if len(sol.t) else 0.0,
            start_radius_m=float(np.linalg.norm(sol.y[:3,-1])),
            end_radius_m=r_far,
        )
    duration = float(sol.t_events[0][0])
    def dm_position(s: float) -> np.ndarray:
        return np.asarray(sol.sol(duration-float(s)), dtype=float)[:3]
    return _integrate_outer_force(
        "incoming", dm_position, lambda _s: np.zeros(3), duration,
        -(duration+float(prefix_duration_s)), eps_value,
        start_radius_m=target, end_radius_m=r_far,
    )


# Wrap the v7 DM-only worker so each entering trajectory also carries the
# incoming R_outer -> R_far Fourier amplitude.
_v7_dm_only_worker = dm_only_worker

def dm_only_worker(row: dict[str, Any], invalid_z_floor_m: float | None) -> dict[str, Any]:
    result = _v7_dm_only_worker(row, invalid_z_floor_m)
    if infer_regime(result) != "resonant" or not bool_value(result, "entered_switch"):
        result.update(_empty_outer_fields(
            "incoming", "not_applicable_no_resolved_switch_entry", False
        ))
        return result
    try:
        x_far = vector_from_row(result, ("x_far_m", "y_far_m", "z_far_m"))
        v_far = vector_from_row(result, ("vx_far_m_s", "vy_far_m_s", "vz_far_m_s"))
        mass_name = "m_dm_kg" if "m_dm_kg" in result else "m_dm"
        fields = _incoming_outer_tail(
            x_far, v_far,
            numeric_value(result, "fourier_prefix_duration_s", 0.0),
            numeric_value(result, mass_name), numeric_value(result, "eps"),
            invalid_z_floor_m,
        )
    except Exception as exc:
        fields = _empty_outer_fields(
            "incoming", f"incoming_outer_exception:{type(exc).__name__}", False
        )
    result.update(fields)
    return result


# Wrap the v7 outgoing escape->R_far routine and reconstruct its end states.
_v7_run_outgoing_dm_tail_to_r_far = run_outgoing_dm_tail_to_r_far

def run_outgoing_dm_tail_to_r_far(
    x_ion_escape_m: np.ndarray,
    v_ion_escape_m_s: np.ndarray,
    x_dm_escape_m: np.ndarray,
    v_dm_escape_m_s: np.ndarray,
    *,
    coupled_end_time_s: float,
    r_far_m: float,
    m_dm_value: float,
    eps_value: float,
    invalid_z_floor_m: float | None,
) -> dict[str, Any]:
    fields = _v7_run_outgoing_dm_tail_to_r_far(
        x_ion_escape_m, v_ion_escape_m_s, x_dm_escape_m, v_dm_escape_m_s,
        coupled_end_time_s=coupled_end_time_s, r_far_m=r_far_m,
        m_dm_value=m_dm_value, eps_value=eps_value,
        invalid_z_floor_m=invalid_z_floor_m,
    )
    if not bool(fields.get("fourier_outgoing_tail_complete", False)):
        return fields
    duration = float(fields.get("fourier_outgoing_tail_duration_s", 0.0))
    if duration == 0.0:
        dm_end = np.concatenate([
            np.asarray(x_dm_escape_m, dtype=float),
            np.asarray(v_dm_escape_m_s, dtype=float),
        ])
    else:
        def rhs(_t: float, y: np.ndarray) -> np.ndarray:
            a = trap_force(y[:3][None,:], m_dm_value, eps_value)[0]/m_dm_value
            return np.concatenate([y[3:6], a])
        sol = solve_ivp(
            rhs, (0.0, duration),
            np.concatenate([x_dm_escape_m, v_dm_escape_m_s]),
            method="DOP853", rtol=OUTGOING_TAIL_RTOL, atol=OUTGOING_TAIL_ATOL,
            max_step=OUTGOING_TAIL_MAX_STEP_S, dense_output=False,
        )
        if not sol.success:
            fields["fourier_outgoing_tail_complete"] = False
            fields["fourier_outgoing_tail_status"] = "end_state_reconstruction_failure"
            return fields
        dm_end = np.asarray(sol.y[:,-1], dtype=float)
    fields["_outgoing_r_far_dm_position_m"] = dm_end[:3]
    fields["_outgoing_r_far_dm_velocity_m_s"] = dm_end[3:6]
    fields["_outgoing_r_far_ion_position_m"] = free_harmonic_ion_position(
        duration, x_ion_escape_m, v_ion_escape_m_s
    )
    fields["_outgoing_r_far_ion_velocity_m_s"] = _free_ion_velocity(
        duration, x_ion_escape_m, v_ion_escape_m_s
    )
    return fields


def _outgoing_outer_tail(
    x_ion_far: np.ndarray,
    v_ion_far: np.ndarray,
    x_dm_far: np.ndarray,
    v_dm_far: np.ndarray,
    global_start_time_s: float,
    m_dm_value: float,
    eps_value: float,
    invalid_z_floor_m: float | None,
) -> dict[str, Any]:
    x_dm_far = np.asarray(x_dm_far, dtype=float).reshape(3)
    v_dm_far = np.asarray(v_dm_far, dtype=float).reshape(3)
    x_ion_far = np.asarray(x_ion_far, dtype=float).reshape(3)
    v_ion_far = np.asarray(v_ion_far, dtype=float).reshape(3)
    r_far = float(np.linalg.norm(x_dm_far))
    target = float(FOURIER_OUTER_RADIUS_M)
    if not INCLUDE_OUTER_R_TAILS:
        return _empty_outer_fields("outgoing", "outer_tails_disabled", True,
                                   start_radius_m=r_far, end_radius_m=r_far)
    if target <= r_far * (1.0+1.0e-10):
        return _empty_outer_fields("outgoing", "R_outer_not_beyond_R_far", True,
                                   start_radius_m=r_far, end_radius_m=r_far)
    def rhs(_t: float, y: np.ndarray) -> np.ndarray:
        a = trap_force(y[:3][None,:], m_dm_value, eps_value)[0]/m_dm_value
        return np.concatenate([y[3:6], a])
    def event(_t: float, y: np.ndarray) -> float:
        return float(np.linalg.norm(y[:3])-target)
    event.terminal=True
    event.direction=1
    events: list[Callable]=[event]
    if invalid_z_floor_m is not None:
        def invalid_event(_t: float, y: np.ndarray) -> float:
            return float(y[2]-invalid_z_floor_m)
        invalid_event.terminal=True
        invalid_event.direction=-1
        events.append(invalid_event)
    speed=max(float(np.linalg.norm(v_dm_far)),1.0e-12)
    travel=max((target-r_far)/speed,0.0)
    tmax=max(OUTER_TAIL_TIME_FACTOR*travel,travel+OUTER_TAIL_MIN_TIME_S)
    sol=solve_ivp(
        rhs,(0.0,tmax),np.concatenate([x_dm_far,v_dm_far]),method="DOP853",
        rtol=OUTER_TAIL_RTOL,atol=OUTER_TAIL_ATOL,max_step=OUTER_TAIL_MAX_STEP_S,
        events=events,dense_output=True,
    )
    reached=len(sol.t_events[0])>0
    invalid=len(events)>1 and len(sol.t_events[1])>0
    if not reached or sol.sol is None:
        status="outgoing_outer_invalid_domain" if invalid else (
            "outgoing_outer_solver_failure" if not sol.success
            else "outgoing_outer_did_not_reach_R_outer"
        )
        return _empty_outer_fields(
            "outgoing",status,False,duration_s=float(sol.t[-1]) if len(sol.t) else 0.0,
            start_radius_m=r_far,end_radius_m=float(np.linalg.norm(sol.y[:3,-1])),
        )
    duration=float(sol.t_events[0][0])
    fields = _integrate_outer_force(
        "outgoing",
        lambda t: np.asarray(sol.sol(float(t)),dtype=float)[:3],
        lambda t: free_harmonic_ion_position(t,x_ion_far,v_ion_far),
        duration,global_start_time_s,eps_value,
        start_radius_m=r_far,end_radius_m=target,
    )
    fields.update(_test2_bowl_metrics_from_dense_solution(
        sol.sol, duration, prefix="outgoing_outer", position_slice=slice(0, 3)
    ))
    return fields


_v7_full_worker = full_worker

def full_worker(row: dict[str, Any], invalid_z_floor_m: float | None) -> dict[str, Any]:
    result = _v7_full_worker(row, invalid_z_floor_m)
    if infer_regime(result) != "resonant":
        return _test2_combine_bowl_fields(result)
    if str(result.get("full_status", "")) not in {
        "reached_R_full_and_escaped", "entered_switch_but_missed_R_full"
    }:
        result.update(_empty_outer_fields(
            "outgoing", "not_applicable_unresolved_core", False
        ))
        return _test2_combine_bowl_fields(result)

    incoming_complete = bool_value(result, "fourier_incoming_outer_tail_complete")
    incoming_cos = np.array([
        numeric_value(result,"fourier_incoming_outer_tail_cos_x_N_s",0.0),
        numeric_value(result,"fourier_incoming_outer_tail_cos_y_N_s",0.0),
        numeric_value(result,"fourier_incoming_outer_tail_cos_z_N_s",0.0),
    ])
    incoming_sin = np.array([
        numeric_value(result,"fourier_incoming_outer_tail_sin_x_N_s",0.0),
        numeric_value(result,"fourier_incoming_outer_tail_sin_y_N_s",0.0),
        numeric_value(result,"fourier_incoming_outer_tail_sin_z_N_s",0.0),
    ])
    rfar_cos = np.array([
        numeric_value(result,"fourier_cos_x_N_s"),
        numeric_value(result,"fourier_cos_y_N_s"),
        numeric_value(result,"fourier_cos_z_N_s"),
    ])
    rfar_sin = np.array([
        numeric_value(result,"fourier_sin_x_N_s"),
        numeric_value(result,"fourier_sin_y_N_s"),
        numeric_value(result,"fourier_sin_z_N_s"),
    ])
    private_names=(
        "_outgoing_r_far_ion_position_m","_outgoing_r_far_ion_velocity_m_s",
        "_outgoing_r_far_dm_position_m","_outgoing_r_far_dm_velocity_m_s",
    )
    if all(name in result for name in private_names):
        mass_name="m_dm_kg" if "m_dm_kg" in result else "m_dm"
        outgoing = _outgoing_outer_tail(
            result[private_names[0]], result[private_names[1]],
            result[private_names[2]], result[private_names[3]],
            numeric_value(result,"fourier_history_t_final_s",0.0),
            numeric_value(result,mass_name),numeric_value(result,"eps"),
            invalid_z_floor_m,
        )
    else:
        outgoing=_empty_outer_fields(
            "outgoing","outgoing_R_far_end_state_unavailable",False
        )
    result.update(outgoing)
    outgoing_complete=bool_value(outgoing,"fourier_outgoing_outer_tail_complete")
    out_cos=np.array([
        numeric_value(outgoing,"fourier_outgoing_outer_tail_cos_x_N_s",0.0),
        numeric_value(outgoing,"fourier_outgoing_outer_tail_cos_y_N_s",0.0),
        numeric_value(outgoing,"fourier_outgoing_outer_tail_cos_z_N_s",0.0),
    ])
    out_sin=np.array([
        numeric_value(outgoing,"fourier_outgoing_outer_tail_sin_x_N_s",0.0),
        numeric_value(outgoing,"fourier_outgoing_outer_tail_sin_y_N_s",0.0),
        numeric_value(outgoing,"fourier_outgoing_outer_tail_sin_z_N_s",0.0),
    ])
    total_cos=incoming_cos+rfar_cos+out_cos
    total_sin=incoming_sin+rfar_sin+out_sin
    modes=(total_cos**2+total_sin**2)/(2.0*float(m_ion))
    energy=float(np.sum(modes))
    resolved=bool_value(result,"fourier_detection_resolved") and (
        incoming_complete or not REQUIRE_OUTER_R_TAILS_COMPLETE
    ) and (outgoing_complete or not REQUIRE_OUTER_R_TAILS_COMPLETE)
    threshold=energy_threshold_diagnostics(
        numeric_value(result,"E_mechanical_final_J"),energy,
        fourier_switch_energy_j=numeric_value(result,"E_fourier_from_R_switch_J"),
        trajectory_resolved=True,fourier_detection_resolved=resolved,
    )
    result.update({
        "E_fourier_from_R_outer_J":energy,
        "E_fourier_from_R_outer_x_J":float(modes[0]),
        "E_fourier_from_R_outer_y_J":float(modes[1]),
        "E_fourier_from_R_outer_z_J":float(modes[2]),
        "E_fourier_total_J":energy,
        "E_fourier_x_J":float(modes[0]),
        "E_fourier_y_J":float(modes[1]),
        "E_fourier_z_J":float(modes[2]),
        "fourier_R_far_roundtrip_cos_x_N_s":float(rfar_cos[0]),
        "fourier_R_far_roundtrip_cos_y_N_s":float(rfar_cos[1]),
        "fourier_R_far_roundtrip_cos_z_N_s":float(rfar_cos[2]),
        "fourier_R_far_roundtrip_sin_x_N_s":float(rfar_sin[0]),
        "fourier_R_far_roundtrip_sin_y_N_s":float(rfar_sin[1]),
        "fourier_R_far_roundtrip_sin_z_N_s":float(rfar_sin[2]),
        "fourier_cos_x_N_s":float(total_cos[0]),
        "fourier_cos_y_N_s":float(total_cos[1]),
        "fourier_cos_z_N_s":float(total_cos[2]),
        "fourier_sin_x_N_s":float(total_sin[0]),
        "fourier_sin_y_N_s":float(total_sin[1]),
        "fourier_sin_z_N_s":float(total_sin[2]),
        "fourier_detection_history":"incoming_R_outer_to_outgoing_R_outer",
        "fourier_detection_outer_radius_m":float(FOURIER_OUTER_RADIUS_M),
        "fourier_detection_incoming_outer_tail_complete":incoming_complete,
        "fourier_detection_outgoing_outer_tail_complete":outgoing_complete,
        "fourier_detection_resolved":resolved,
        "fourier_history_t_initial_s":-(
            numeric_value(result,"fourier_prefix_duration_s",0.0)
            +numeric_value(result,"fourier_incoming_outer_tail_duration_s",0.0)
        ),
        "fourier_history_t_final_s":numeric_value(result,"fourier_history_t_final_s",0.0)
            +numeric_value(outgoing,"fourier_outgoing_outer_tail_duration_s",0.0),
    })
    result.update(threshold)
    for name in private_names:
        result.pop(name,None)
    return _test2_combine_bowl_fields(result)


_v7_finalize_status = finalize_status

def finalize_status(row: pd.Series | dict[str, Any]) -> str:
    base = _v7_finalize_status(row)
    if DETECTION_ENERGY_POLICY in {"fourier","both","either"} and (
        str(row.get("full_status","")) in {
            "reached_R_full_and_escaped","entered_switch_but_missed_R_full"
        }
    ):
        if REQUIRE_OUTER_R_TAILS_COMPLETE and not bool_value(
            row,"fourier_detection_incoming_outer_tail_complete"
        ):
            return "unresolved_fourier_incoming_R_outer_tail"
        if REQUIRE_OUTER_R_TAILS_COMPLETE and not bool_value(
            row,"fourier_detection_outgoing_outer_tail_complete"
        ):
            return "unresolved_fourier_outgoing_R_outer_tail"
    return base


DM_ONLY_DEFAULTS.update(_empty_outer_fields("incoming","not_run",False))
FULL_DEFAULTS.update(_empty_outer_fields("outgoing","not_run",False))
FULL_DEFAULTS.update({
    "E_fourier_from_R_outer_J":np.nan,
    "E_fourier_from_R_outer_x_J":np.nan,
    "E_fourier_from_R_outer_y_J":np.nan,
    "E_fourier_from_R_outer_z_J":np.nan,
    "fourier_detection_history":"incoming_R_outer_to_outgoing_R_outer",
    "fourier_detection_outer_radius_m":float(FOURIER_OUTER_RADIUS_M),
    "fourier_detection_incoming_outer_tail_complete":False,
    "fourier_detection_outgoing_outer_tail_complete":False,
    "fourier_R_far_roundtrip_cos_x_N_s":np.nan,
    "fourier_R_far_roundtrip_cos_y_N_s":np.nan,
    "fourier_R_far_roundtrip_cos_z_N_s":np.nan,
    "fourier_R_far_roundtrip_sin_x_N_s":np.nan,
    "fourier_R_far_roundtrip_sin_y_N_s":np.nan,
    "fourier_R_far_roundtrip_sin_z_N_s":np.nan,
    "fourier_history_t_initial_s":np.nan,
})


# Preserve Test 2 in an isolated namespace before the Test 2.5 cell defines
# helpers with several of the same names. The cloned functions all share the
# isolated dictionary as their global namespace, so later Test 2.5 definitions
# cannot change the behavior of an automatic Test 2 refinement round.
import types as _test2_types

_TEST2_CELL_FILENAME = main.__code__.co_filename
TEST2_NAMESPACE = dict(globals())
for _test2_name, _test2_object in list(TEST2_NAMESPACE.items()):
    if (
        isinstance(_test2_object, _test2_types.FunctionType)
        and _test2_object.__code__.co_filename == _TEST2_CELL_FILENAME
    ):
        _test2_clone = _test2_types.FunctionType(
            _test2_object.__code__,
            TEST2_NAMESPACE,
            name=_test2_object.__name__,
            argdefs=_test2_object.__defaults__,
            closure=_test2_object.__closure__,
        )
        _test2_clone.__kwdefaults__ = _test2_object.__kwdefaults__
        _test2_clone.__annotations__ = dict(
            getattr(_test2_object, "__annotations__", {})
        )
        _test2_clone.__dict__.update(getattr(_test2_object, "__dict__", {}))
        _test2_clone.__doc__ = _test2_object.__doc__
        _test2_clone.__module__ = "embedded_test2"
        TEST2_NAMESPACE[_test2_name] = _test2_clone

TEST2_NAMESPACE["__name__"] = "embedded_test2"
TEST2_MAIN = TEST2_NAMESPACE["main"]


def _test2_file_sha256(path: Path) -> str:
    path = Path(path)
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _test2_pilot_fingerprint() -> dict[str, Any]:
    input_path = resolve_input_path(
        TEST1_INPUT_CSV, required=True, label="Test 2 pilot input"
    )
    assert input_path is not None
    diagnostic_path = None
    if EXTRA_DIAGNOSTIC_INPUT_CSV is not None:
        diagnostic_path = resolve_input_path(
            EXTRA_DIAGNOSTIC_INPUT_CSV,
            required=False,
            label="Test 2 diagnostic input",
        )
    return {
        "code_version": str(globals().get(
            "TRAJECTORY_REFINEMENT_CODE_VERSION", "unversioned"
        )),
        "input_path": str(input_path.resolve()),
        "input_sha256": _test2_file_sha256(input_path),
        "diagnostic_input_path": (
            str(diagnostic_path.resolve()) if diagnostic_path is not None else None
        ),
        "diagnostic_input_sha256": (
            _test2_file_sha256(diagnostic_path)
            if diagnostic_path is not None else None
        ),
        "m_dm_kg": float(USER_M_DM_KG),
        "eps": float(USER_EPS),
        "target_ion_energy_J": float(USER_TARGET_ION_ENERGY_J),
        "r_full_v_b_mode": str(USER_R_FULL_VB_MODE),
        "r_switch_factor": float(USER_R_SWITCH_FACTOR),
        "dm_only_rtol": float(DM_ONLY_RTOL),
        "dm_only_atol": float(DM_ONLY_ATOL),
        "full_rtol": float(FULL_RTOL),
        "full_atol": float(FULL_ATOL),
        "full_max_step_s": float(FULL_MAX_STEP_S),
        "detection_energy_policy": str(DETECTION_ENERGY_POLICY),
        "require_fourier_mechanical_agreement": bool(
            REQUIRE_FOURIER_MECHANICAL_AGREEMENT
        ),
        "fourier_mechanical_rel_tol": float(FOURIER_MECHANICAL_REL_TOL),
        "fourier_steps_per_secular_period": int(
            FOURIER_STEPS_PER_SECULAR_PERIOD
        ),
        "fourier_detection_start_radius": str(FOURIER_DETECTION_START_RADIUS),
        "require_R_far_prefix_complete": bool(REQUIRE_R_FAR_PREFIX_COMPLETE),
        "include_outgoing_R_far_tail": bool(INCLUDE_OUTGOING_R_FAR_TAIL),
        "require_outgoing_R_far_tail_complete": bool(
            REQUIRE_OUTGOING_R_FAR_TAIL_COMPLETE
        ),
        "outgoing_tail_rtol": float(OUTGOING_TAIL_RTOL),
        "outgoing_tail_atol": float(OUTGOING_TAIL_ATOL),
        "outgoing_tail_max_step_s": float(OUTGOING_TAIL_MAX_STEP_S),
        "outgoing_tail_quad_epsrel": float(OUTGOING_TAIL_QUAD_EPSREL),
        "outgoing_tail_quad_limit": int(OUTGOING_TAIL_QUAD_LIMIT),
        "prefix_quad_epsrel": float(PREFIX_QUAD_EPSREL),
        "prefix_quad_epsabs_dimensionless": float(
            PREFIX_QUAD_EPSABS_DIMENSIONLESS
        ),
        "prefix_quad_limit": int(PREFIX_QUAD_LIMIT),
    }


def _run_or_reuse_test2_pilot() -> None:
    manifest_path = Path(RUN_DIRECTORY) / "test2_pilot_manifest.json"
    expected_outputs = [
        Path(COMBINED_RESULTS_CSV),
        Path(WEIGHTED_ANALYSIS_CSV),
        Path(CLASS_REACH_SUMMARY_CSV),
    ]
    current = _test2_pilot_fingerprint()
    previous = None
    if manifest_path.exists():
        try:
            previous = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (OSError, ValueError, json.JSONDecodeError):
            previous = None

    can_reuse = (
        bool(globals().get("AUTO_REUSE_COMPLETED_TEST2_PILOT", True))
        and previous == current
        and all(path.exists() and path.stat().st_size > 0 for path in expected_outputs)
    )
    if can_reuse:
        progress(
            "[AUTO TEST 2] reusing completed pilot because the Test 1.5 "
            "input and refinement fingerprint are unchanged"
        )
        return

    progress("[AUTO TEST 2] running pilot Test 2")
    TEST2_MAIN()
    manifest_path.write_text(json.dumps(current, indent=2), encoding="utf-8")


_v7_test2_fingerprint = _test2_pilot_fingerprint

def _test2_pilot_fingerprint() -> dict[str, Any]:
    value = _v7_test2_fingerprint()
    value.update({
        "full_pulse_outer_tail_version":"v8_bowl_v1",
        "include_outer_R_tails":bool(INCLUDE_OUTER_R_TAILS),
        "require_outer_R_tails_complete":bool(REQUIRE_OUTER_R_TAILS_COMPLETE),
        "fourier_outer_radius_m":float(FOURIER_OUTER_RADIUS_M),
        "outer_tail_rtol":float(OUTER_TAIL_RTOL),
        "outer_tail_atol":float(OUTER_TAIL_ATOL),
        "outer_tail_max_step_s":float(OUTER_TAIL_MAX_STEP_S),
        "outer_tail_quad_epsrel":float(OUTER_TAIL_QUAD_EPSREL),
        "outer_tail_quad_limit":int(OUTER_TAIL_QUAD_LIMIT),
        "bowl_model_signature": str(getattr(
            globals().get("BOWL_MODEL", None), "model_signature", "unavailable"
        )),
        "bowl_time_sample_dt_s": float(globals().get(
            "USER_BOWL_TIME_SAMPLE_DT_S", 2.0e-8
        )),
    })
    return value

if __name__ == "__main__":
    # Pilot Test 2 run using the Test 1.5 input and class reach screen.
    if bool(globals().get("USER_RUN_TEST2", True)):
        _run_or_reuse_test2_pilot()
    else:
        progress("[SKIP] Test 2 pilot")


# Bowl Entry, Entry Channel, and Detectability — Pilot Report

This report tests

\[
\text{no bowl entry}\Longrightarrow E_{\rm FT}<E_{\rm th}
\]

using only resolved full-pulse trajectories. It also tabulates the selected
entry channel and the DC/RF force decomposition at the first inward crossing.
A resolved no-bowl above-threshold row remains a direct sampled
counterexample; absence of one is support only within the sampled domain.


In [ ]:
#!/usr/bin/env python3
"""Evaluate whether resolved no-bowl trajectories can exceed threshold.

Run after the bowl-aware Test 2 pilot (and again after Test 2.5 refinement).
The report never upgrades a sampled statement into a universal proof.  It
classifies the numerical evidence as:

* hypothesis_false_in_sample
* supported_in_sample_not_proven
* insufficient_no_bowl_coverage
* insufficient_resolved_coverage
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUN_DIRECTORY = Path(globals().get("RUN_DIRECTORY", Path.cwd()))
OUTPUT_PREFIX = RUN_DIRECTORY / "bowl_detectability_report_v2"
TRAJECTORY_AUDIT_CSV = Path(f"{OUTPUT_PREFIX}_trajectory_audit.csv")
CONTINGENCY_CSV = Path(f"{OUTPUT_PREFIX}_contingency.csv")
CONCLUSION_CSV = Path(f"{OUTPUT_PREFIX}_conclusion.csv")
COUNTEREXAMPLES_CSV = Path(f"{OUTPUT_PREFIX}_counterexamples.csv")
SHOW_PLOTS = bool(globals().get("USER_BOWL_REPORT_SHOW_PLOTS", True))
SAVE_PLOTS = bool(globals().get("USER_BOWL_REPORT_SAVE_PLOTS", False))
ENERGY_PLOT_PNG = Path(f"{OUTPUT_PREFIX}_energy_vs_bowl_time.png")
DISTANCE_PLOT_PNG = Path(f"{OUTPUT_PREFIX}_energy_vs_bowl_penetration.png")
CHANNEL_SUMMARY_CSV = Path(f"{OUTPUT_PREFIX}_entry_channel_summary.csv")


def _bool_series(frame: pd.DataFrame, column: str, default: bool = False) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(default, index=frame.index, dtype=bool)
    values = frame[column]
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(default).astype(bool)
    return values.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})


def _numeric(frame: pd.DataFrame, column: str, default=np.nan) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(default, index=frame.index, dtype=float)
    return pd.to_numeric(frame[column], errors="coerce")


def _resolve_results_path() -> Path:
    candidates: list[Path] = []
    explicit = globals().get("USER_BOWL_REPORT_INPUT_CSV", None)
    if explicit is not None:
        candidates.append(Path(explicit))
    for name in ("ALL_RESULTS_CSV", "WEIGHTED_ANALYSIS_CSV", "COMBINED_RESULTS_CSV"):
        if name in globals():
            candidates.append(Path(globals()[name]))
    if "TEST2_NAMESPACE" in globals():
        namespace = globals()["TEST2_NAMESPACE"]
        for name in ("WEIGHTED_ANALYSIS_CSV", "COMBINED_RESULTS_CSV"):
            if name in namespace:
                candidates.append(Path(namespace[name]))
    for name in ("WEIGHTED_ANALYSIS_CSV", "COMBINED_RESULTS_CSV"):
        if name in globals():
            candidates.append(Path(globals()[name]))
    for path in candidates:
        if path.exists() and path.stat().st_size > 0:
            return path
    discovered = sorted(
        list(RUN_DIRECTORY.glob("test2p5*_all_selected_ray_results.csv"))
        + list(RUN_DIRECTORY.glob("test2*_weighted_analysis_results.csv")),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if discovered:
        return discovered[0]
    raise FileNotFoundError("No bowl-aware Test 2 weighted result CSV was found")


def _weight_column(frame: pd.DataFrame) -> str | None:
    for name in (
        "test2_analysis_weight_conditional",
        "population_weight_conditional",
        "impact_weight",
    ):
        if name in frame.columns:
            return name
    return None


def build_report(results: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    data = results.copy()
    threshold = float(globals().get("USER_TARGET_ION_ENERGY_J", 1.0e-27))
    data["_resolved"] = _bool_series(data, "fourier_detection_resolved")
    data["_entered_bowl"] = _bool_series(data, "entered_bowl")
    data["_energy_J"] = _numeric(data, "E_fourier_from_R_outer_J")
    if not np.isfinite(data["_energy_J"]).any():
        data["_energy_J"] = _numeric(data, "E_fourier_total_J")
    data["_above_threshold"] = (
        data["_resolved"]
        & np.isfinite(data["_energy_J"])
        & (data["_energy_J"] >= threshold)
    )
    data["_no_bowl"] = ~data["_entered_bowl"]
    data["_counterexample"] = data["_resolved"] & data["_no_bowl"] & data["_above_threshold"]
    data["_energy_over_threshold"] = data["_energy_J"] / threshold

    weight_name = _weight_column(data)
    if weight_name is None:
        data["_weight"] = 1.0
        weight_name = "equal_row_weight"
    else:
        data["_weight"] = _numeric(data, weight_name, 0.0).fillna(0.0).clip(lower=0.0)

    contingency_rows: list[dict[str, Any]] = []
    for entered_label, entered_value in (("no_bowl_entry", False), ("entered_bowl", True)):
        for energy_label, above_value in (("below_threshold", False), ("above_threshold", True)):
            mask = (
                data["_resolved"]
                & data["_entered_bowl"].eq(entered_value)
                & data["_above_threshold"].eq(above_value)
            )
            subset = data.loc[mask]
            contingency_rows.append({
                "bowl_group": entered_label,
                "energy_group": energy_label,
                "resolved_count": int(mask.sum()),
                "weight_sum": float(subset["_weight"].sum()),
                "max_fourier_energy_J": float(subset["_energy_J"].max()) if len(subset) else np.nan,
                "max_energy_over_threshold": float(subset["_energy_over_threshold"].max()) if len(subset) else np.nan,
            })
    contingency = pd.DataFrame(contingency_rows)

    resolved = data.loc[data["_resolved"] & np.isfinite(data["_energy_J"])].copy()
    resolved_no_bowl = resolved.loc[~resolved["_entered_bowl"]].copy()
    counterexamples = resolved_no_bowl.loc[resolved_no_bowl["_above_threshold"]].copy()

    if len(resolved) == 0:
        evidence_status = "insufficient_resolved_coverage"
        conclusion = "No resolved full-pulse trajectories are available for the bowl necessity test."
    elif len(resolved_no_bowl) == 0:
        evidence_status = "insufficient_no_bowl_coverage"
        conclusion = (
            "Resolved trajectories exist, but none are classified as remaining outside the bowl; "
            "the necessity hypothesis has not been tested."
        )
    elif len(counterexamples):
        evidence_status = "hypothesis_false_in_sample"
        conclusion = (
            "At least one resolved trajectory remained outside the bowl and exceeded the ion-energy "
            "threshold. Bowl entry is therefore not necessary within the sampled data."
        )
    else:
        evidence_status = "supported_in_sample_not_proven"
        conclusion = (
            "No resolved no-bowl trajectory exceeded threshold. The data support bowl entry as a "
            "necessary condition within the sampled and converged domain, but do not prove a universal theorem."
        )

    max_no_bowl_energy = (
        float(resolved_no_bowl["_energy_J"].max()) if len(resolved_no_bowl) else np.nan
    )
    max_no_bowl_ratio = max_no_bowl_energy / threshold if np.isfinite(max_no_bowl_energy) else np.nan
    bowl_model = globals().get("BOWL_MODEL", None)
    escape_label = str(getattr(bowl_model, "escape_label", "unavailable")) if bowl_model is not None else "unavailable"
    rf_x_invariant = bool(getattr(bowl_model, "rf_x_invariant_within_tolerance", False)) if bowl_model is not None else False
    component_closed = bool(getattr(bowl_model, "component_closed", False)) if bowl_model is not None else False

    conclusion_table = pd.DataFrame([{
        "evidence_status": evidence_status,
        "conclusion": conclusion,
        "threshold_J": threshold,
        "n_total_rows": len(data),
        "n_resolved_rows": len(resolved),
        "n_resolved_no_bowl_rows": len(resolved_no_bowl),
        "n_no_bowl_above_threshold_counterexamples": len(counterexamples),
        "max_resolved_no_bowl_energy_J": max_no_bowl_energy,
        "max_resolved_no_bowl_energy_over_threshold": max_no_bowl_ratio,
        "weight_column": weight_name,
        "resolved_no_bowl_weight_sum": float(resolved_no_bowl["_weight"].sum()),
        "counterexample_weight_sum": float(counterexamples["_weight"].sum()),
        "selected_escape_label": escape_label,
        "rf_pseudopotential_x_invariant": rf_x_invariant,
        "bowl_component_closed": component_closed,
        "bowl_model_signature": str(getattr(bowl_model, "model_signature", "unavailable")),
    }])

    audit_columns = [
        column for column in (
            "trajectory_id", "ray_key", "trajectory_class_id", "speed_label",
            "v_inf_m_s", "theta_rad", "alpha_rad", "psi_rad", "b_m",
            "full_status", "fourier_detection_resolved", "entered_bowl",
            "crossed_bowl_inward", "crossed_bowl_outward", "bowl_trajectory_class",
            "bowl_entry_stage", "bowl_entry_channel",
            "bowl_entry_U_total_J", "bowl_entry_U_dc_J", "bowl_entry_U_rf_J",
            "bowl_entry_F_dc_toward_center_N", "bowl_entry_F_rf_toward_center_N",
            "bowl_entry_F_total_toward_center_N",
            "interaction_time_inside_bowl_s", "interaction_time_inside_R_full_s",
            "minimum_bowl_signed_distance_m", "d_min_m",
            "E_fourier_from_R_outer_J", "E_fourier_total_J", "above_energy_threshold",
            "bowl_hypothesis_counterexample",
        ) if column in data.columns
    ]
    audit = data[audit_columns].copy()
    audit["report_energy_J"] = data["_energy_J"]
    audit["report_energy_over_threshold"] = data["_energy_over_threshold"]
    audit["report_resolved"] = data["_resolved"]
    audit["report_entered_bowl"] = data["_entered_bowl"]
    audit["report_above_threshold"] = data["_above_threshold"]
    audit["report_counterexample"] = data["_counterexample"]
    audit["report_weight"] = data["_weight"]

    return audit, contingency, conclusion_table, counterexamples


def build_channel_summary(audit: pd.DataFrame) -> pd.DataFrame:
    if len(audit) == 0:
        return pd.DataFrame(columns=[
            "bowl_entry_channel", "resolved_count", "above_threshold_count",
            "weight_sum", "max_energy_J", "max_energy_over_threshold",
            "mean_time_inside_bowl_s", "mean_time_inside_R_full_s",
        ])
    data = audit.copy()
    if "bowl_entry_channel" not in data.columns:
        data["bowl_entry_channel"] = np.where(
            data.get("report_entered_bowl", False), "entered_unspecified", "no_bowl_entry"
        )
    data["bowl_entry_channel"] = data["bowl_entry_channel"].fillna("none").astype(str)
    data.loc[~data["report_entered_bowl"], "bowl_entry_channel"] = "no_bowl_entry"
    rows = []
    for channel, group in data.groupby("bowl_entry_channel", dropna=False):
        resolved = group.loc[group["report_resolved"]]
        rows.append({
            "bowl_entry_channel": str(channel),
            "resolved_count": int(len(resolved)),
            "above_threshold_count": int(resolved["report_above_threshold"].sum()),
            "weight_sum": float(resolved["report_weight"].sum()),
            "max_energy_J": float(pd.to_numeric(resolved["report_energy_J"], errors="coerce").max()) if len(resolved) else np.nan,
            "max_energy_over_threshold": float(pd.to_numeric(resolved["report_energy_over_threshold"], errors="coerce").max()) if len(resolved) else np.nan,
            "mean_time_inside_bowl_s": float(pd.to_numeric(resolved.get("interaction_time_inside_bowl_s", np.nan), errors="coerce").mean()) if len(resolved) else np.nan,
            "mean_time_inside_R_full_s": float(pd.to_numeric(resolved.get("interaction_time_inside_R_full_s", np.nan), errors="coerce").mean()) if len(resolved) else np.nan,
        })
    return pd.DataFrame(rows).sort_values(["above_threshold_count", "resolved_count"], ascending=False)


def show_report_plots(audit: pd.DataFrame, threshold: float) -> None:
    resolved = audit.loc[
        audit["report_resolved"] & np.isfinite(pd.to_numeric(audit["report_energy_J"], errors="coerce"))
    ].copy()
    if len(resolved) == 0:
        return

    energy = pd.to_numeric(resolved["report_energy_J"], errors="coerce").clip(lower=np.finfo(float).tiny)
    bowl_time = pd.to_numeric(
        resolved.get("interaction_time_inside_bowl_s", pd.Series(0.0, index=resolved.index)),
        errors="coerce",
    ).fillna(0.0)
    figure, axis = plt.subplots(figsize=(8, 5))
    for label, group_mask in (
        ("did not enter bowl", ~resolved["report_entered_bowl"]),
        ("entered bowl", resolved["report_entered_bowl"]),
    ):
        axis.scatter(bowl_time[group_mask] * 1.0e6, energy[group_mask], label=label)
    axis.axhline(threshold, linestyle="--", label="detection threshold")
    axis.set_yscale("log")
    axis.set_xlabel("time inside bowl [µs]")
    axis.set_ylabel("resolved full-pulse Fourier energy [J]")
    axis.set_title("Ion energy versus DM bowl residence time")
    axis.legend()
    figure.tight_layout()
    if SAVE_PLOTS:
        figure.savefig(ENERGY_PLOT_PNG, dpi=180)
    if SHOW_PLOTS:
        plt.show()
    plt.close(figure)

    if "minimum_bowl_signed_distance_m" in resolved.columns:
        signed = pd.to_numeric(resolved["minimum_bowl_signed_distance_m"], errors="coerce")
        finite = np.isfinite(signed)
        if finite.any():
            figure, axis = plt.subplots(figsize=(8, 5))
            axis.scatter(signed[finite] * 1.0e6, energy[finite])
            axis.axhline(threshold, linestyle="--", label="detection threshold")
            axis.axvline(0.0, linestyle=":", label="bowl boundary")
            axis.set_yscale("log")
            axis.set_xlabel("minimum signed distance to bowl boundary [µm]")
            axis.set_ylabel("resolved full-pulse Fourier energy [J]")
            axis.set_title("Detectability versus bowl penetration")
            axis.legend()
            figure.tight_layout()
            if SAVE_PLOTS:
                figure.savefig(DISTANCE_PLOT_PNG, dpi=180)
            if SHOW_PLOTS:
                plt.show()
            plt.close(figure)


def main() -> None:
    path = _resolve_results_path()
    results = pd.read_csv(path, low_memory=False)
    audit, contingency, conclusion, counterexamples = build_report(results)
    channel_summary = build_channel_summary(audit)
    audit.to_csv(TRAJECTORY_AUDIT_CSV, index=False)
    contingency.to_csv(CONTINGENCY_CSV, index=False)
    channel_summary.to_csv(CHANNEL_SUMMARY_CSV, index=False)
    conclusion.to_csv(CONCLUSION_CSV, index=False)
    counterexamples.to_csv(COUNTEREXAMPLES_CSV, index=False)

    print("\n-------- Bowl Necessity Test --------")
    print(f"Input: {path}")
    print(contingency.to_string(index=False))
    print("\n" + str(conclusion.iloc[0]["conclusion"]))
    print(
        "Maximum resolved no-bowl energy / threshold: "
        f"{conclusion.iloc[0]['max_resolved_no_bowl_energy_over_threshold']:.6e}"
    )
    if len(counterexamples):
        columns = [
            column for column in (
                "trajectory_id", "v_inf_m_s", "b_m", "d_min_m",
                "E_fourier_from_R_outer_J", "bowl_trajectory_class",
            ) if column in counterexamples.columns
        ]
        print("\nResolved no-bowl counterexamples:")
        print(counterexamples[columns].head(20).to_string(index=False))
    print(f"Saved: {TRAJECTORY_AUDIT_CSV}")
    print(f"Saved: {CONCLUSION_CSV}")

    threshold = float(conclusion.iloc[0]["threshold_J"])
    show_report_plots(audit, threshold)


if __name__ == "__main__":
    if bool(globals().get("USER_RUN_BOWL_NECESSITY_REPORT", True)):
        main()
    else:
        print("[SKIP] Bowl necessity report")


# Single-Trajectory Outer-Radius Convergence Audit

A resolved resonant trajectory is rerun at the configured values of $R_{\rm outer}$. `USER_SANITY_TRAJECTORY_ID=None` automatically selects a suitable trajectory. If no resolved resonant trajectory exists, the audit writes a clean skip record instead of failing.


In [ ]:
#!/usr/bin/env python3
"""Single-trajectory convergence audit for the v8 full Fourier pulse.

Run after the Test 2 cell has created TEST2_NAMESPACE. The script reruns one
resolved trajectory at a sequence of outer radii and checks whether the
coherent Fourier energy has converged as the endpoints move farther into the
weak-force tail.

This version treats USER_SANITY_TRAJECTORY_ID=None, "", "None", "nan", and
"null" as "auto-select a trajectory". It also exits cleanly with a header-safe
CSV when Test 2 contains no resolved resonant trajectory.
"""

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def _normalize_optional_text(value: Any) -> str:
    """Convert optional notebook values to a usable string."""
    if value is None:
        return ""
    text = str(value).strip()
    if text.lower() in {"", "none", "nan", "null"}:
        return ""
    return text


def _normalize_outer_radii(value: Any) -> tuple[float, ...]:
    """Accept a scalar or iterable of finite positive outer radii."""
    if value is None:
        raw = (20.0e-3, 40.0e-3)
    elif np.isscalar(value):
        raw = (value,)
    else:
        raw = tuple(value)

    radii: list[float] = []
    for item in raw:
        try:
            radius = float(item)
        except (TypeError, ValueError):
            continue
        if np.isfinite(radius) and radius > 0.0:
            radii.append(radius)

    radii = sorted(set(radii))
    if not radii:
        raise ValueError(
            "USER_SANITY_OUTER_RADII_M must contain at least one finite "
            "positive radius"
        )
    return tuple(radii)


OUTER_RADII_M = _normalize_outer_radii(
    globals().get("USER_SANITY_OUTER_RADII_M", (20.0e-3, 40.0e-3))
)
EXACT_TRAJECTORY_ID = _normalize_optional_text(
    globals().get("USER_SANITY_TRAJECTORY_ID", "")
)
OUTPUT_CSV = (
    Path(RUN_DIRECTORY)
    / "single_trajectory_full_pulse_outer_convergence_v9.csv"
)


OUTPUT_COLUMNS = [
    "sanity_status",
    "sanity_message",
    "trajectory_id",
    "R_outer_m",
    "R_outer_mm",
    "E_fourier_full_pulse_J",
    "E_fourier_R_far_roundtrip_J",
    "E_fourier_R_switch_J",
    "above_threshold",
    "fourier_detection_resolved",
    "incoming_outer_complete",
    "outgoing_outer_complete",
    "incoming_force_at_outer_N",
    "outgoing_force_at_outer_N",
    "full_status",
    "relative_change_from_previous",
]


def _load(path: Any) -> pd.DataFrame:
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(path)
    try:
        return pd.read_csv(path, low_memory=False)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def _series(
    frame: pd.DataFrame,
    column: str,
    default: Any,
    *,
    dtype: str | None = None,
) -> pd.Series:
    """Return a column-shaped default instead of a scalar default."""
    if column in frame.columns:
        return frame[column]
    result = pd.Series(default, index=frame.index)
    if dtype is not None:
        result = result.astype(dtype)
    return result


def _bools(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y"})
    )


def _choose_id(results: pd.DataFrame) -> str | None:
    """Choose an explicit or automatically selected resolved trajectory."""
    if len(results) == 0 or "trajectory_id" not in results.columns:
        return None

    trajectory_ids = results["trajectory_id"].astype(str)

    if EXACT_TRAJECTORY_ID:
        if not trajectory_ids.eq(EXACT_TRAJECTORY_ID).any():
            available = trajectory_ids.drop_duplicates().head(10).tolist()
            raise KeyError(
                f"Trajectory not found: {EXACT_TRAJECTORY_ID}. "
                f"First available IDs: {available}"
            )
        return EXACT_TRAJECTORY_ID

    full_success = _bools(_series(results, "full_success", False))
    full_status = _series(results, "full_status", "", dtype="object").astype(str)
    regime = (
        _series(results, "trajectory_regime", "resonant", dtype="object")
        .astype(str)
        .str.lower()
    )

    mask = full_success
    mask &= full_status.isin(
        {
            "reached_R_full_and_escaped",
            "entered_switch_but_missed_R_full",
        }
    )
    mask &= ~regime.str.contains("adiabatic|rutherford", regex=True, na=False)

    candidates = results.loc[mask].copy()
    if len(candidates) == 0:
        return None

    if "E_fourier_total_J" in candidates.columns:
        energy = pd.to_numeric(
            candidates["E_fourier_total_J"], errors="coerce"
        )
    elif "E_fourier_from_R_outer_J" in candidates.columns:
        energy = pd.to_numeric(
            candidates["E_fourier_from_R_outer_J"], errors="coerce"
        )
    else:
        energy = pd.Series(np.nan, index=candidates.index, dtype=float)

    threshold = float(
        globals().get("USER_TARGET_ION_ENERGY_J", 1.0e-27)
    )
    finite_positive = np.isfinite(energy) & (energy > 0.0)

    candidates["_distance"] = np.inf
    candidates.loc[finite_positive, "_distance"] = np.abs(
        np.log10(energy.loc[finite_positive] / threshold)
    )

    # Prefer a positive-weight production row. If none is available, allow a
    # zero-weight escape-channel diagnostic so the convergence test can still
    # audit the newly discovered channel.
    if "test2_diagnostic_only" in candidates.columns:
        candidates["_diagnostic_priority"] = _bools(
            candidates["test2_diagnostic_only"]
        ).astype(int)
    else:
        candidates["_diagnostic_priority"] = 0
    return str(
        candidates.sort_values(
            ["_diagnostic_priority", "_distance", "trajectory_id"],
            kind="mergesort",
        ).iloc[0]["trajectory_id"]
    )


def _write_skip_output(status: str, message: str) -> None:
    row = {column: np.nan for column in OUTPUT_COLUMNS}
    row.update(
        {
            "sanity_status": status,
            "sanity_message": message,
            "trajectory_id": EXACT_TRAJECTORY_ID or "",
            "above_threshold": False,
            "fourier_detection_resolved": False,
            "incoming_outer_complete": False,
            "outgoing_outer_complete": False,
            "full_status": "",
        }
    )
    table = pd.DataFrame([row], columns=OUTPUT_COLUMNS)
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(OUTPUT_CSV, index=False)
    print("Full-pulse outer-radius convergence test skipped")
    print(message)
    print(f"Saved audit status: {OUTPUT_CSV}")


def main() -> None:
    if "TEST2_NAMESPACE" not in globals():
        raise RuntimeError(
            "Run the Test 2 cell first so TEST2_NAMESPACE exists"
        )

    ns = TEST2_NAMESPACE
    results = _load(ns["WEIGHTED_ANALYSIS_CSV"])
    trajectory_id = _choose_id(results)

    if trajectory_id is None:
        _write_skip_output(
            "skipped_no_resolved_resonant_trajectory",
            (
                "Test 2 contains no resolved resonant trajectory suitable for "
                "the outer-radius convergence audit. This is expected when "
                "Test 1.5 selected no dynamic trajectories or when all Test 2 "
                "rows are analytical/rejection stubs."
            ),
        )
        return

    templates = _load(ns["TEST1_INPUT_CSV"])
    if "trajectory_id" in templates.columns:
        match = templates.loc[
            templates["trajectory_id"].astype(str).eq(trajectory_id)
        ]
    else:
        match = templates.iloc[0:0].copy()

    if len(match) == 0:
        # Some pilot outputs use a row copied through class screening; fall
        # back to the complete result row because it retains the far state.
        match = results.loc[
            results["trajectory_id"].astype(str).eq(trajectory_id)
        ]

    if len(match) == 0:
        raise RuntimeError(
            f"Could not reconstruct template for {trajectory_id}"
        )

    template = match.iloc[0].to_dict()
    invalid_z_floor = ns["resolve_invalid_z_floor"]()
    original_radius = float(ns["FOURIER_OUTER_RADIUS_M"])
    rows: list[dict[str, Any]] = []

    try:
        for radius in OUTER_RADII_M:
            ns["FOURIER_OUTER_RADIUS_M"] = float(radius)

            dm_row = ns["dm_only_worker"](template, invalid_z_floor)
            full_row = ns["full_worker"](dm_row, invalid_z_floor)

            rows.append(
                {
                    "sanity_status": "completed",
                    "sanity_message": "",
                    "trajectory_id": trajectory_id,
                    "R_outer_m": float(radius),
                    "R_outer_mm": float(radius) * 1.0e3,
                    "E_fourier_full_pulse_J": full_row.get(
                        "E_fourier_from_R_outer_J", np.nan
                    ),
                    "E_fourier_R_far_roundtrip_J": full_row.get(
                        "E_fourier_from_R_far_J", np.nan
                    ),
                    "E_fourier_R_switch_J": full_row.get(
                        "E_fourier_from_R_switch_J", np.nan
                    ),
                    "above_threshold": full_row.get(
                        "above_energy_threshold", False
                    ),
                    "fourier_detection_resolved": full_row.get(
                        "fourier_detection_resolved", False
                    ),
                    "incoming_outer_complete": full_row.get(
                        "fourier_detection_incoming_outer_tail_complete",
                        False,
                    ),
                    "outgoing_outer_complete": full_row.get(
                        "fourier_detection_outgoing_outer_tail_complete",
                        False,
                    ),
                    "incoming_force_at_outer_N": full_row.get(
                        "fourier_incoming_outer_tail_force_at_outer_N",
                        np.nan,
                    ),
                    "outgoing_force_at_outer_N": full_row.get(
                        "fourier_outgoing_outer_tail_force_at_outer_N",
                        np.nan,
                    ),
                    "full_status": full_row.get("full_status", ""),
                    "relative_change_from_previous": np.nan,
                }
            )
    finally:
        ns["FOURIER_OUTER_RADIUS_M"] = original_radius

    table = (
        pd.DataFrame(rows, columns=OUTPUT_COLUMNS)
        .sort_values("R_outer_m")
        .reset_index(drop=True)
    )

    energy = pd.to_numeric(
        table["E_fourier_full_pulse_J"], errors="coerce"
    )
    for index in range(1, len(table)):
        current = float(energy.iloc[index])
        previous = float(energy.iloc[index - 1])
        if np.isfinite(current) and np.isfinite(previous):
            denominator = max(abs(current), np.finfo(float).tiny)
            table.loc[index, "relative_change_from_previous"] = (
                abs(current - previous) / denominator
            )

    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(OUTPUT_CSV, index=False)

    print("Full-pulse outer-radius convergence test")
    print(f"Trajectory: {trajectory_id}")
    print(table.to_string(index=False))
    print(f"Saved: {OUTPUT_CSV}")


if __name__ == "__main__":
    if bool(globals().get("USER_RUN_OUTER_RADIUS_SANITY", True)):
        main()
    else:
        print("[SKIP] outer-radius convergence audit")


# Test 2.5 — Automatic $b$ and Speed Refinement

This stage alternates Test 2.5 and Test 2 until the geometric $b$ boundary and the selected speed/detectability boundary converge or the configured round cap is reached.


In [ ]:
#!/usr/bin/env python3
"""Standalone Test 2.5 v16: adaptive mapping with full-pulse R_outer Fourier detection.

Run after the Test 2 pilot:

    %run -i 04_test2p5_full_pulse_refinement_v16.py

This file contains the complete Test 2.5 logic. It does not import, patch,
execute, or read another Python script. It uses the standalone Test 1.5 class templates
and the standalone Test 2 results.

Trajectory classes rejected by the Test 2 class-level reach screen are excluded.
For accepted or unresolved classes, Test 2.5 generates exact new b values inside
the physical collision-energy support and refines brackets in b^2.
"""

from __future__ import annotations

import json
import hashlib
import math
import re
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd


# =============================================================================
# USER SETTINGS
# =============================================================================

CLASS_TEMPLATE_INPUT_CSV = Path(f"{TEST1P5_OUTPUT_PREFIX}_test2_input.csv")
PILOT_TEST2_RESULTS_CSV = Path(
    f"{TEST2_OUTPUT_PREFIX}_weighted_analysis_results.csv"
)
OUTPUT_PREFIX = TEST2P5_OUTPUT_PREFIX

ROUND_RESULTS_GLOB = f"{OUTPUT_PREFIX}_round*_trajectory_results.csv"

N_ANGULAR_CELLS_PER_SPEED = 6
N_PRIORITY_CELLS_PER_SPEED = 4
N_CONTROL_CELLS_PER_SPEED = 2
N_PSI_PER_ANGULAR_CELL = 2

# Initial production targets are equal-area fractions of the physical impact-
# parameter support from the previous collision-regime test. Five points are
# enough to seed the adaptive b^2 refinement without oversampling each ray.
INITIAL_PHYSICAL_B2_FRACTIONS = (
    0.00, 0.0025, 0.01, 0.015625, 0.04, 0.12, 0.30, 0.60, 1.00
)
INITIAL_MAX_B_PER_RAY = len(INITIAL_PHYSICAL_B2_FRACTIONS)
REQUIRE_POSITIVE_WEIGHT_PRODUCTION_ROWS = True
ACCEPTED_CLASS_STATUSES = {
    "accepted_has_switch_entry",
    "unresolved_class_reach_screen",
    "screen_disabled",
}
TRAJECTORY_CLASS_PSI_DECIMALS = 12

BMAX_BRACKET_TOLERANCE_M = 50.0e-6
TARGETED_BMAX_BRACKET_TOLERANCE_M = float(
    globals().get("USER_TARGETED_ESCAPE_B_TOLERANCE_M", 1.0e-6)
)
MAX_NEW_B_PER_RAY_PER_ROUND = 2
MAX_NEXT_ROUND_ROWS = 128
MAX_REFINEMENT_ROUNDS = 12
MAX_UNRESOLVED_NEIGHBORS_PER_RAY = 2

# Once the b scan is complete, refine the lower reachability speed at several
# confirmed-reaching b anchors for every retained angular geometry. This makes
# the final support conditional in both v and b instead of applying one b=0
# speed threshold to the entire impact-parameter interval.
ENABLE_SPEED_REACHABILITY_REFINEMENT = bool(
    globals().get("AUTO_REFINE_SPEED_REACHABILITY", True)
)
SPEED_REACHABILITY_TOLERANCE_M_S = float(
    globals().get("SPEED_REACHABILITY_TOLERANCE_M_S", 5.0)
)
TARGETED_SPEED_REACHABILITY_TOLERANCE_M_S = float(
    globals().get("USER_TARGETED_ESCAPE_SPEED_TOLERANCE_M_S", 1.0)
)
MAX_SPEED_REFINEMENT_ROWS_PER_ROUND = int(
    globals().get("MAX_SPEED_REFINEMENT_ROWS_PER_ROUND", 32)
)
SPEED_REACHABILITY_B_ANCHOR_COUNT = int(
    globals().get("SPEED_REACHABILITY_B_ANCHOR_COUNT", 5)
)
SPEED_REFINEMENT_TARGET = str(
    globals().get("USER_SPEED_REFINEMENT_TARGET", "detectability")
).strip().lower()
if SPEED_REFINEMENT_TARGET not in {"reachability", "detectability"}:
    raise ValueError(
        "SPEED_REFINEMENT_TARGET must be 'reachability' or 'detectability'"
    )
if SPEED_REACHABILITY_B_ANCHOR_COUNT < 1:
    raise ValueError("SPEED_REACHABILITY_B_ANCHOR_COUNT must be at least 1")
OVERWRITE_NEXT_ROUND_INPUT = True

PILOT_WEIGHT_COLUMNS = (
    "test2_analysis_weight_conditional",
    "population_weight_conditional",
    "impact_weight",
)

SELECTED_CELLS_CSV = Path(f"{OUTPUT_PREFIX}_selected_angular_cells.csv")
SELECTED_RAYS_CSV = Path(f"{OUTPUT_PREFIX}_selected_rays.csv")
ALL_RESULTS_CSV = Path(f"{OUTPUT_PREFIX}_all_selected_ray_results.csv")
RAY_SUMMARY_CSV = Path(f"{OUTPUT_PREFIX}_ray_bmax_summary.csv")
INTERVALS_CSV = Path(f"{OUTPUT_PREFIX}_sampled_reaching_intervals.csv")
THETA_ALPHA_SUMMARY_CSV = Path(f"{OUTPUT_PREFIX}_theta_alpha_bmax_summary.csv")
SPEED_REACHABILITY_SUMMARY_CSV = Path(
    f"{OUTPUT_PREFIX}_speed_reachability_summary.csv"
)
NEXT_SETTINGS_TXT = Path(f"{OUTPUT_PREFIX}_next_test2_settings.txt")


EMPTY_ALL_RESULTS_COLUMNS = [
    "trajectory_id", "trajectory_class_id", "speed_label", "v_inf_m_s",
    "theta_index", "alpha_index", "theta_rad", "alpha_rad", "psi_rad",
    "b_m", "ray_key", "geometry_key", "geometry_class",
    "above_energy_threshold", "entered_switch", "reached_R_full",
]
EMPTY_RAY_SUMMARY_COLUMNS = [
    "ray_key", "geometry_key", "speed_label", "v_inf_m_s",
    "theta_index", "alpha_index", "theta_rad", "alpha_rad", "psi_rad",
    "theta_deg", "alpha_deg", "psi_deg", "n_results", "n_reach",
    "n_miss", "n_unresolved", "single_cutoff_monotonic", "converged",
    "convergence_reason", "bmax_lower_m", "bmax_estimate_m",
    "bmax_upper_m",
]
EMPTY_INTERVAL_COLUMNS = [
    "ray_key", "geometry_key", "speed_label", "v_inf_m_s",
    "theta_index", "alpha_index", "theta_rad", "alpha_rad", "psi_rad",
    "interval_index", "b_start_m", "b_stop_m",
]
EMPTY_THETA_ALPHA_COLUMNS = [
    "speed_label", "v_inf_m_s", "theta_index", "alpha_index",
    "theta_rad", "alpha_rad", "theta_deg", "alpha_deg", "n_psi_rays",
]
EMPTY_SPEED_SUMMARY_COLUMNS = [
    "geometry_key", "speed_anchor_b_m", "speed_lower_confirmed_reach_m_s",
    "speed_lower_confirmed_target_m_s", "speed_lower_resolved_miss_m_s",
    "speed_transition_estimate_m_s", "speed_reachability_converged",
]


def write_header_safe_empty_refinement_outputs() -> None:
    pd.DataFrame(columns=EMPTY_ALL_RESULTS_COLUMNS).to_csv(
        ALL_RESULTS_CSV, index=False
    )
    pd.DataFrame(columns=EMPTY_RAY_SUMMARY_COLUMNS).to_csv(
        RAY_SUMMARY_CSV, index=False
    )
    pd.DataFrame(columns=EMPTY_INTERVAL_COLUMNS).to_csv(
        INTERVALS_CSV, index=False
    )
    pd.DataFrame(columns=EMPTY_THETA_ALPHA_COLUMNS).to_csv(
        THETA_ALPHA_SUMMARY_CSV, index=False
    )
    pd.DataFrame(columns=EMPTY_SPEED_SUMMARY_COLUMNS).to_csv(
        SPEED_REACHABILITY_SUMMARY_CSV, index=False
    )


# =============================================================================
# BASIC HELPERS
# =============================================================================


def progress(message: str = "") -> None:
    print(message, flush=True)


def load_csv(path: Path, *, required: bool = True) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        if required:
            raise FileNotFoundError(f"CSV not found or empty: {path}")
        return pd.DataFrame()
    data = pd.read_csv(path, low_memory=False)
    if "trajectory_id" in data.columns:
        data["trajectory_id"] = data["trajectory_id"].astype(str)
    return data


def numeric_series(df: pd.DataFrame, column: str, default=np.nan) -> pd.Series:
    if column not in df.columns:
        return pd.Series(default, index=df.index, dtype=float)
    return pd.to_numeric(df[column], errors="coerce")


def bool_series(df: pd.DataFrame, column: str, default=False) -> pd.Series:
    if column not in df.columns:
        return pd.Series(default, index=df.index, dtype=bool)
    value = df[column]
    if pd.api.types.is_bool_dtype(value):
        return value.fillna(default).astype(bool)
    return value.astype(str).str.strip().str.lower().isin(
        {"true", "1", "yes", "y"}
    )


def bool_value(row: pd.Series | dict[str, Any], column: str, default=False) -> bool:
    value = row.get(column, default)
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}



def merge_annuli(intervals: Iterable[tuple[float, float]]) -> list[tuple[float, float]]:
    clean = sorted(
        (float(lower), float(upper))
        for lower, upper in intervals
        if np.isfinite(lower) and np.isfinite(upper) and 0.0 <= lower < upper
    )
    if not clean:
        return []
    merged: list[list[float]] = [[clean[0][0], clean[0][1]]]
    for lower, upper in clean[1:]:
        previous = merged[-1]
        tolerance = max(1.0e-15, 1.0e-12 * max(previous[1], upper, 1.0e-12))
        if lower <= previous[1] + tolerance:
            previous[1] = max(previous[1], upper)
        else:
            merged.append([lower, upper])
    return [(lower, upper) for lower, upper in merged]


def parse_annuli_json(value: Any) -> list[tuple[float, float]]:
    try:
        raw = json.loads(str(value))
    except (TypeError, ValueError, json.JSONDecodeError):
        return []
    intervals: list[tuple[float, float]] = []
    if isinstance(raw, list):
        for item in raw:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                try:
                    intervals.append((float(item[0]), float(item[1])))
                except (TypeError, ValueError):
                    pass
    return merge_annuli(intervals)


def physical_b_annuli(ray_population: pd.DataFrame) -> list[tuple[float, float]]:
    production = physical_population_for_ray(ray_population)
    if len(production) == 0:
        return []
    for column in ("physical_b_annuli_json", "class_b_annuli_json", "class_reach_annuli_json"):
        if column in production.columns:
            for value in production[column].dropna():
                annuli = parse_annuli_json(value)
                if annuli:
                    return annuli
    return merge_annuli(
        zip(
            numeric_series(production, "b_lower_m").to_numpy(float),
            numeric_series(production, "b_upper_m").to_numpy(float),
        )
    )


def b_from_union_area_fraction(
    annuli: list[tuple[float, float]],
    fraction: float,
) -> tuple[float, int, float, float]:
    if not annuli:
        return np.nan, -1, np.nan, np.nan
    fraction = float(np.clip(fraction, 0.0, 1.0))
    widths = np.array([upper * upper - lower * lower for lower, upper in annuli])
    total = float(widths.sum())
    target = fraction * total
    cumulative = 0.0
    for index, ((lower, upper), width) in enumerate(zip(annuli, widths)):
        if target <= cumulative + width or index == len(annuli) - 1:
            local_area = float(np.clip(target - cumulative, 0.0, width))
            return math.sqrt(lower * lower + local_area), index, lower, upper
        cumulative += float(width)
    lower, upper = annuli[-1]
    return upper, len(annuli) - 1, lower, upper


def containing_annulus(
    b_value: float,
    annuli: list[tuple[float, float]],
) -> tuple[int, float, float] | None:
    for index, (lower, upper) in enumerate(annuli):
        tolerance = max(1.0e-15, 1.0e-12 * max(upper, 1.0e-12))
        if lower - tolerance <= b_value <= upper + tolerance:
            return index, lower, upper
    return None

def validate_test1_population(df: pd.DataFrame) -> None:
    required = {
        "trajectory_id",
        "speed_label",
        "v_inf_m_s",
        "R_full_m",
        "R_switch_m",
        "theta_index",
        "alpha_index",
        "theta_rad",
        "alpha_rad",
        "psi_rad",
        "b_m",
        "b_lower_m",
        "b_upper_m",
        "x_far_m",
        "y_far_m",
        "z_far_m",
        "vx_far_m_s",
        "vy_far_m_s",
        "vz_far_m_s",
        "eps",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError("Full Test 1 population is missing: " + ", ".join(missing))
    if "m_dm_kg" not in df.columns and "m_dm" not in df.columns:
        raise ValueError("Full Test 1 population must contain m_dm_kg or m_dm")


def validate_test2_results(df: pd.DataFrame, label: str) -> None:
    if len(df) == 0:
        return
    required = {
        "trajectory_id",
        "speed_label",
        "theta_index",
        "alpha_index",
        "psi_rad",
        "b_m",
        "entered_switch",
        "reached_R_full",
        "dm_only_status",
        "full_status",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"{label} is missing: " + ", ".join(missing))


def ensure_angle_columns(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    if "theta_deg" not in output.columns:
        output["theta_deg"] = np.degrees(numeric_series(output, "theta_rad"))
    if "alpha_deg" not in output.columns:
        output["alpha_deg"] = np.degrees(numeric_series(output, "alpha_rad"))
    if "psi_deg" not in output.columns:
        output["psi_deg"] = np.mod(
            np.degrees(numeric_series(output, "psi_rad")), 360.0
        )
    return output


def add_key_columns(df: pd.DataFrame) -> pd.DataFrame:
    output = ensure_angle_columns(df)
    output["speed_label"] = output["speed_label"].astype(str)
    output["theta_index"] = numeric_series(output, "theta_index").astype(int)
    output["alpha_index"] = numeric_series(output, "alpha_index").astype(int)
    output["psi_key"] = np.round(numeric_series(output, "psi_rad"), 12)
    output["cell_key"] = [
        f"{speed}|th{theta:03d}|al{alpha:03d}"
        for speed, theta, alpha in output[
            ["speed_label", "theta_index", "alpha_index"]
        ].itertuples(index=False, name=None)
    ]
    output["ray_key"] = [
        f"{speed}|th{theta:03d}|al{alpha:03d}|psi{psi:.12f}"
        for speed, theta, alpha, psi in output[
            ["speed_label", "theta_index", "alpha_index", "psi_key"]
        ].itertuples(index=False, name=None)
    ]
    output["geometry_key"] = [
        f"th{theta:03d}|al{alpha:03d}|psi{psi:.12f}"
        for theta, alpha, psi in output[
            ["theta_index", "alpha_index", "psi_key"]
        ].itertuples(index=False, name=None)
    ]
    output = add_trajectory_class_id(output)
    return output



def trajectory_class_id_from_row(row: pd.Series | dict[str, Any]) -> str:
    existing = str(row.get("trajectory_class_id", "")).strip()
    if existing and existing.lower() != "nan":
        return existing
    speed = str(row.get("speed_label", "unknown"))
    regime = str(row.get("trajectory_regime", row.get("collision_regime", "resonant")))
    theta = int(float(row.get("theta_index", -1)))
    alpha = int(float(row.get("alpha_index", -1)))
    psi = round(float(row.get("psi_rad", 0.0)), TRAJECTORY_CLASS_PSI_DECIMALS)
    return (
        f"{speed}|{regime}|th{theta:03d}|al{alpha:03d}|"
        f"psi{psi:.{TRAJECTORY_CLASS_PSI_DECIMALS}f}"
    )


def add_trajectory_class_id(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    output["trajectory_class_id"] = [
        trajectory_class_id_from_row(row) for _, row in output.iterrows()
    ]
    return output


def vector_from_template(
    row: pd.Series | dict[str, Any],
    names: tuple[str, str, str],
) -> np.ndarray:
    vector = np.array([float(row[name]) for name in names], dtype=float)
    if vector.shape != (3,) or not np.all(np.isfinite(vector)):
        raise ValueError(f"Invalid template vector columns: {names}")
    return vector


def fallback_impact_direction(velocity_hat: np.ndarray, psi_rad: float) -> np.ndarray:
    reference = np.array([0.0, 0.0, 1.0])
    if abs(float(np.dot(reference, velocity_hat))) > 0.9:
        reference = np.array([0.0, 1.0, 0.0])
    basis_1 = np.cross(velocity_hat, reference)
    basis_1 /= np.linalg.norm(basis_1)
    basis_2 = np.cross(velocity_hat, basis_1)
    direction = math.cos(psi_rad) * basis_1 + math.sin(psi_rad) * basis_2
    return direction / np.linalg.norm(direction)



def recalculate_row_v_b_radii(row: pd.Series | dict[str, Any]) -> pd.Series:
    output = pd.Series(dict(row), dtype=object)
    mass_name = "m_dm_kg" if "m_dm_kg" in output.index else "m_dm"
    radii = rutherford_v_b_radius_policy(
        float(output["v_inf_m_s"]),
        float(output["b_m"]),
        m_dm_kg=float(output[mass_name]),
        m_ion_kg=float(m_ion),
        eps_value=float(output["eps"]),
        ion_charge_number=float(Z_ion),
        coulomb_constant=float(K),
        elementary_charge_c=float(e),
        threshold_j=float(USER_TARGET_ION_ENERGY_J),
    )
    output["r_min_energy_threshold_m"] = radii["r_min_energy_threshold_m"]
    output["r_min_threshold_speed_only_m"] = radii["r_min_threshold_speed_only_m"]
    output["r_min_rutherford_v_b_m"] = radii["r_min_rutherford_v_b_m"]
    output["r_min_threshold_m"] = radii["R_full_m"]
    output["R_full_m"] = radii["R_full_m"]
    output["R_switch_m"] = radii["R_switch_m"]
    output["R_switch_factor"] = radii["R_switch_factor"]
    output["R_full_v_b_mode"] = radii["R_full_v_b_mode"]
    output["rutherford_recoil_v_b_J"] = radii["rutherford_recoil_v_b_J"]
    output["rutherford_above_threshold_v_b"] = radii["rutherford_above_threshold_v_b"]
    output["R_full_um"] = float(radii["R_full_m"]) * 1.0e6
    output["R_switch_um"] = float(radii["R_switch_m"]) * 1.0e6
    return output


def rebuild_template_at_b(
    template: pd.Series | dict[str, Any],
    b_target_m: float,
    *,
    reason: str,
    id_suffix: str,
) -> pd.Series:
    row = pd.Series(dict(template), dtype=object)
    x_far = vector_from_template(row, ("x_far_m", "y_far_m", "z_far_m"))
    v_far = vector_from_template(row, ("vx_far_m_s", "vy_far_m_s", "vz_far_m_s"))
    speed = float(np.linalg.norm(v_far))
    radius = float(np.linalg.norm(x_far))
    if speed <= 0.0 or not (0.0 <= b_target_m < radius):
        raise ValueError(
            f"Cannot rebuild b={b_target_m:.6e} m on far radius {radius:.6e} m"
        )
    annuli = []
    for column in ("physical_b_annuli_json", "class_b_annuli_json", "class_reach_annuli_json"):
        annuli = parse_annuli_json(row.get(column, ""))
        if annuli:
            break
    containing = containing_annulus(float(b_target_m), annuli) if annuli else None
    if annuli and containing is None:
        raise ValueError(f"Generated b={b_target_m:.6e} m lies in a forbidden support gap")

    velocity_hat = v_far / speed
    longitudinal = float(np.dot(x_far, velocity_hat))
    impact_vector = x_far - longitudinal * velocity_hat
    impact_norm = float(np.linalg.norm(impact_vector))
    if impact_norm > max(1.0e-15, 1.0e-12 * radius):
        impact_hat = impact_vector / impact_norm
    else:
        impact_hat = fallback_impact_direction(
            velocity_hat,
            float(row.get("psi_rad", 0.0)),
        )
    incoming_longitudinal = -math.sqrt(
        max(radius * radius - b_target_m * b_target_m, 0.0)
    )
    position = incoming_longitudinal * velocity_hat + b_target_m * impact_hat
    class_id = trajectory_class_id_from_row(row)
    row["x_far_m"], row["y_far_m"], row["z_far_m"] = position
    row["b_m"] = float(b_target_m)
    if containing is not None:
        annulus_index, lower, upper = containing
        row["b_lower_m"] = lower
        row["b_upper_m"] = upper
        row["test2p5_annulus_index"] = annulus_index
    row["trajectory_class_id"] = class_id
    row["trajectory_id"] = f"{class_id}|test2p5|{id_suffix}"
    row["test2p5_selection_reason"] = reason
    row["test2_sample_role"] = "test2p5_generated_physical_b"
    row = recalculate_row_v_b_radii(row)
    for column in (
        "impact_weight",
        "population_weight_conditional",
        "population_weight_unconditional",
        "test2_analysis_weight_conditional",
        "test2_analysis_weight_unconditional",
    ):
        if column in row.index:
            row[f"test2p5_original_{column}"] = row[column]
        row[column] = 0.0
    return row

def first_existing_weight(df: pd.DataFrame) -> np.ndarray:
    for column in PILOT_WEIGHT_COLUMNS:
        if column in df.columns:
            values = numeric_series(df, column, 0.0).to_numpy(float)
            return np.where(np.isfinite(values) & (values > 0.0), values, 0.0)
    return np.ones(len(df), dtype=float)


def unit_direction(theta_rad: float, alpha_rad: float) -> np.ndarray:
    return np.array(
        [
            math.cos(theta_rad),
            math.sin(theta_rad) * math.cos(alpha_rad),
            math.sin(theta_rad) * math.sin(alpha_rad),
        ],
        dtype=float,
    )


def angular_distance_deg(
    theta_a: float,
    alpha_a: float,
    theta_b: float,
    alpha_b: float,
) -> float:
    cosine = float(
        np.clip(
            np.dot(
                unit_direction(theta_a, alpha_a),
                unit_direction(theta_b, alpha_b),
            ),
            -1.0,
            1.0,
        )
    )
    return math.degrees(math.acos(cosine))


def parse_round_number(path: Path) -> int | None:
    match = re.search(r"_round(\d+)_trajectory_results\.csv$", path.name)
    return int(match.group(1)) if match else None


def find_round_result_files() -> list[tuple[int, Path]]:
    files: list[tuple[int, Path]] = []
    for path in Path(".").glob(ROUND_RESULTS_GLOB):
        number = parse_round_number(path)
        if number is not None:
            files.append((number, path))
    return sorted(files, key=lambda item: item[0])


def _speed_tag(value: float) -> str:
    return (
        f"{float(value):.9e}"
        .replace("+", "")
        .replace("-", "m")
        .replace(".", "p")
    )


def rebuild_template_at_speed_and_b(
    template: pd.Series | dict[str, Any],
    target_speed_m_s: float,
    target_b_m: float,
    *,
    geometry_key: str,
) -> pd.Series:
    """Rebuild a zero-weight Test 2 row at an exact midpoint speed and b.

    The template's far radius and trap-potential correction are retained. The
    far velocity is rescaled so that the requested asymptotic speed is used.
    A lower-speed template is preferred by the caller because its R_far is the
    conservative choice for a midpoint probe.
    """
    target_speed = float(target_speed_m_s)
    if not np.isfinite(target_speed) or target_speed <= 0.0:
        raise ValueError("target_speed_m_s must be finite and positive")
    suffix = f"speedref_v{_speed_tag(target_speed)}_b{_speed_tag(target_b_m)}"
    row = rebuild_template_at_b(
        template,
        float(target_b_m),
        reason="adaptive_speed_reachability_midpoint",
        id_suffix=suffix,
    )

    v_far = vector_from_template(row, ("vx_far_m_s", "vy_far_m_s", "vz_far_m_s"))
    old_far_speed = float(np.linalg.norm(v_far))
    old_v_inf = float(row["v_inf_m_s"])
    velocity_hat = v_far / max(old_far_speed, np.finfo(float).tiny)
    trap_speed_shift_sq = old_far_speed**2 - old_v_inf**2
    target_far_speed_sq = target_speed**2 + trap_speed_shift_sq
    if not np.isfinite(target_far_speed_sq) or target_far_speed_sq <= 0.0:
        raise ValueError(
            "The selected far state has nonpositive kinetic energy at the "
            f"target speed {target_speed:.12g} m/s"
        )
    target_far_speed = math.sqrt(target_far_speed_sq)
    target_velocity = target_far_speed * velocity_hat

    parent_label = str(row.get("speed_label", ""))
    row["speed_refinement_parent_label"] = parent_label
    row["speed_label"] = f"speed_refine_v{_speed_tag(target_speed)}"
    row["v_inf_m_s"] = target_speed
    row["speed_m_s"] = target_speed
    row["vx_far_m_s"], row["vy_far_m_s"], row["vz_far_m_s"] = target_velocity
    row["v_far_m_s"] = target_far_speed
    row["speed_refinement_probe"] = True
    row["speed_refinement_geometry_key"] = str(geometry_key)
    row["geometry_key"] = str(geometry_key)
    row["test2_sample_role"] = "test2p5_generated_speed_probe"
    row["trajectory_class_id"] = ""
    row = recalculate_row_v_b_radii(row)
    row["trajectory_class_id"] = trajectory_class_id_from_row(row)
    row["trajectory_id"] = (
        f"{row['trajectory_class_id']}|test2p5|{suffix}|{geometry_key}"
    )
    return row


def _kinematic_speed_floor_m_s() -> float:
    dm_mass = float(USER_M_DM_KG)
    ion_mass = float(m_ion)
    mu = dm_mass * ion_mass / (dm_mass + ion_mass)
    return math.sqrt(
        float(USER_TARGET_ION_ENERGY_J) * ion_mass / (2.0 * mu * mu)
    )


def _speed_reachability_anchor_table(
    classified: pd.DataFrame,
    candidate_geometries: set[str],
) -> pd.DataFrame:
    """Choose a small set of exact confirmed-reaching b anchors per geometry."""
    rows: list[dict[str, Any]] = []
    for geometry_key in sorted(candidate_geometries):
        data = classified.loc[
            classified["geometry_key"].astype(str).eq(geometry_key)
            & classified["geometry_class"].astype(str).eq("reach")
        ].copy()
        b_values = np.sort(np.unique(
            numeric_series(data, "b_m").to_numpy(float)
        ))
        b_values = b_values[np.isfinite(b_values) & (b_values >= 0.0)]
        if len(b_values) == 0:
            continue
        count = min(SPEED_REACHABILITY_B_ANCHOR_COUNT, len(b_values))
        if count == 1:
            indices = np.array([0], dtype=int)
        else:
            targets2 = np.linspace(b_values[0] ** 2, b_values[-1] ** 2, count)
            indices = np.array([
                int(np.argmin(np.abs(b_values * b_values - target2)))
                for target2 in targets2
            ], dtype=int)
            indices = np.unique(np.concatenate([
                indices,
                np.array([0, len(b_values) - 1], dtype=int),
            ]))
        selected_b = b_values[indices]
        denominator = max(selected_b[-1] ** 2 - selected_b[0] ** 2, 0.0)
        for rank, b_anchor in enumerate(selected_b):
            fraction = (
                (b_anchor ** 2 - selected_b[0] ** 2) / denominator
                if denominator > 0.0 else 0.0
            )
            rows.append({
                "geometry_key": geometry_key,
                "speed_anchor_rank": int(rank),
                "speed_anchor_b_m": float(b_anchor),
                "speed_anchor_b_um": float(b_anchor) * 1.0e6,
                "speed_anchor_b2_fraction": float(fraction),
                "speed_anchor_confirmed_at_existing_speed": True,
            })
    return pd.DataFrame(rows)


def analyze_speed_reachability(
    all_results: pd.DataFrame,
    selected_rays: pd.DataFrame,
    full_population: pd.DataFrame,
    ray_summary: pd.DataFrame,
) -> pd.DataFrame:
    """Refine the lower speed for reachability or actual full-pulse R_outer detectability.

    For detectability, the search first tests the parent speed-cell upper edge
    when no above-threshold sample exists. Once a success is found, ordinary
    resolved-miss/success bisection narrows the lower speed boundary.
    """
    if len(all_results) == 0 or len(selected_rays) == 0:
        return pd.DataFrame()
    classified = add_result_classifications(add_key_columns(all_results))
    classified["speed_target_class"] = [
        classify_speed_refinement_result(row)
        for _, row in classified.iterrows()
    ]
    selected = add_key_columns(selected_rays)

    # b anchors are chosen from geometrically reaching trajectories even when
    # the speed target is detectability. This lets the speed search discover a
    # first detectable point at a higher speed rather than requiring one in the
    # pilot calculation.
    geometric_geometries = set(
        classified.loc[
            bool_series(classified, "reached_R_full"),
            "geometry_key",
        ].astype(str)
    )
    selected_geometries = set(selected["geometry_key"].astype(str))
    candidate_geometries = geometric_geometries & selected_geometries
    if not candidate_geometries:
        return pd.DataFrame()

    anchors = _speed_reachability_anchor_table(
        classified,
        candidate_geometries,
    )
    if len(anchors) == 0:
        return pd.DataFrame()

    population = add_key_columns(full_population)
    v_kin = _kinematic_speed_floor_m_s()
    rows: list[dict[str, Any]] = []
    for anchor in anchors.itertuples(index=False):
        geometry_key = str(anchor.geometry_key)
        b_anchor = float(anchor.speed_anchor_b_m)
        b_tolerance = max(1.0e-15, 1.0e-9 * max(abs(b_anchor), 1.0e-6))
        data = classified.loc[
            classified["geometry_key"].astype(str).eq(geometry_key)
            & np.isclose(
                numeric_series(classified, "b_m").to_numpy(float),
                b_anchor,
                rtol=0.0,
                atol=b_tolerance,
            )
        ].copy()
        geometry_population = population.loc[
            population["geometry_key"].astype(str).eq(geometry_key)
        ].copy()
        if len(data) == 0 or len(geometry_population) == 0:
            continue

        speed_rows = []
        for speed_value, group in data.groupby("v_inf_m_s", sort=True):
            classes = group["speed_target_class"].astype(str)
            resolved = classes.isin(["reach", "miss"])
            speed_rows.append({
                "speed_m_s": float(speed_value),
                "any_reach": bool(classes.eq("reach").any()),
                "resolved": bool(resolved.any()),
                "all_resolved_miss": bool(
                    len(group) and resolved.all() and classes.eq("miss").all()
                ),
            })
        speed_table = pd.DataFrame(speed_rows).sort_values("speed_m_s")

        domain_lower_candidates: list[float] = []
        for column in (
            "speed_interval_lower_m_s",
            "minimum_speed_cutoff_m_s",
            "speed_lower_m_s",
        ):
            if column in geometry_population.columns:
                values = numeric_series(geometry_population, column).to_numpy(float)
                domain_lower_candidates.extend(values[np.isfinite(values)].tolist())
        domain_upper_candidates: list[float] = []
        for column in (
            "speed_interval_upper_m_s",
            "maximum_speed_cutoff_m_s",
            "speed_upper_m_s",
        ):
            if column in geometry_population.columns:
                values = numeric_series(geometry_population, column).to_numpy(float)
                domain_upper_candidates.extend(values[np.isfinite(values)].tolist())
        population_speeds = numeric_series(
            geometry_population, "v_inf_m_s"
        ).to_numpy(float)
        population_speeds = population_speeds[np.isfinite(population_speeds)]
        domain_lower = max(
            v_kin,
            min(domain_lower_candidates)
            if domain_lower_candidates
            else (float(np.min(population_speeds)) if population_speeds.size else v_kin),
        )
        domain_upper = (
            max(domain_upper_candidates)
            if domain_upper_candidates
            else (float(np.max(population_speeds)) if population_speeds.size else np.nan)
        )
        if not np.isfinite(domain_upper) or domain_upper < domain_lower:
            domain_upper = max(domain_lower, float(speed_table["speed_m_s"].max()))

        targeted_geometry = bool_series(
            geometry_population, "targeted_escape_refinement", False
        ).any() or geometry_population["speed_label"].astype(str).str.startswith(
            "escape_refine_"
        ).any()
        speed_tolerance_m_s = (
            TARGETED_SPEED_REACHABILITY_TOLERANCE_M_S
            if targeted_geometry
            else SPEED_REACHABILITY_TOLERANCE_M_S
        )

        reaching = speed_table.loc[speed_table["any_reach"]]
        lower_resolved_miss = np.nan
        lower_kind = ""
        confirmed_reach = np.nan
        transition_estimate = np.nan
        next_speed = np.nan
        width = np.nan

        if len(reaching):
            confirmed_reach = float(reaching["speed_m_s"].min())
            lower_misses = speed_table.loc[
                speed_table["all_resolved_miss"]
                & (speed_table["speed_m_s"] < confirmed_reach)
            ]
            if len(lower_misses):
                lower_bracket = float(lower_misses["speed_m_s"].max())
                lower_kind = "resolved_miss"
                lower_resolved_miss = lower_bracket
            else:
                lower_bracket = float(domain_lower)
                lower_kind = "domain_floor_without_resolved_miss"
            lower_bracket = min(lower_bracket, confirmed_reach)
            width = confirmed_reach - lower_bracket
            converged = bool(width <= speed_tolerance_m_s)
            next_speed = (
                0.5 * (lower_bracket + confirmed_reach)
                if not converged else np.nan
            )
            transition_estimate = 0.5 * (lower_bracket + confirmed_reach)
            search_status = "bracketed_target_transition"
        else:
            resolved_misses = speed_table.loc[speed_table["all_resolved_miss"]]
            if len(resolved_misses):
                lower_bracket = float(resolved_misses["speed_m_s"].max())
                lower_resolved_miss = lower_bracket
                lower_kind = "highest_resolved_miss_without_success"
            else:
                lower_bracket = float(domain_lower)
                lower_kind = "domain_floor_without_resolved_result"
            max_tested = float(speed_table["speed_m_s"].max())
            upper_already_tested = bool(np.any(np.isclose(
                speed_table["speed_m_s"].to_numpy(float),
                domain_upper,
                rtol=0.0,
                atol=1.0e-9,
            )))
            if domain_upper > max_tested + 1.0e-9:
                converged = False
                next_speed = float(domain_upper)
                width = float(domain_upper - lower_bracket)
                search_status = "searching_parent_upper_edge_for_first_target_success"
            elif upper_already_tested:
                upper_rows = speed_table.loc[np.isclose(
                    speed_table["speed_m_s"].to_numpy(float),
                    domain_upper,
                    rtol=0.0,
                    atol=1.0e-9,
                )]
                if len(upper_rows) and bool(upper_rows["all_resolved_miss"].all()):
                    converged = True
                    width = 0.0
                    search_status = "no_target_success_in_parent_speed_cell"
                else:
                    converged = False
                    width = np.nan
                    search_status = "parent_upper_edge_unresolved"
            else:
                converged = True
                width = 0.0
                search_status = "no_target_success_in_available_speed_domain"

        already_tested = speed_table["speed_m_s"].to_numpy(float)
        if np.isfinite(next_speed) and np.any(
            np.isclose(already_tested, next_speed, rtol=0.0, atol=1.0e-9)
        ):
            next_speed = np.nan
            if search_status == "bracketed_target_transition":
                converged = True
                search_status = "midpoint_already_tested"

        representative = selected.loc[
            selected["geometry_key"].astype(str).eq(geometry_key)
        ].sort_values("v_inf_m_s").iloc[-1]
        rows.append({
            "geometry_key": geometry_key,
            "theta_index": int(representative["theta_index"]),
            "alpha_index": int(representative["alpha_index"]),
            "psi_key": float(representative["psi_key"]),
            "speed_anchor_rank": int(anchor.speed_anchor_rank),
            "speed_anchor_b_m": b_anchor,
            "speed_anchor_b_um": b_anchor * 1.0e6,
            "speed_anchor_b2_fraction": float(anchor.speed_anchor_b2_fraction),
            "speed_lower_resolved_miss_m_s": lower_resolved_miss,
            "speed_lower_bracket_m_s": lower_bracket,
            "speed_lower_bracket_kind": lower_kind,
            "speed_lower_confirmed_reach_m_s": confirmed_reach,
            "speed_lower_confirmed_target_m_s": confirmed_reach,
            "speed_transition_estimate_m_s": transition_estimate,
            "speed_bracket_width_m_s": width,
            "speed_reachability_converged": converged,
            "speed_reachability_next_probe_m_s": next_speed,
            "speed_reachability_tolerance_m_s": float(speed_tolerance_m_s),
            "targeted_escape_refinement": bool(targeted_geometry),
            "speed_domain_lower_m_s": domain_lower,
            "speed_domain_upper_m_s": domain_upper,
            "n_speed_samples_at_anchor": int(len(speed_table)),
            "n_reaching_speeds_at_anchor": int(speed_table["any_reach"].sum()),
            "speed_refinement_target": SPEED_REFINEMENT_TARGET,
            "speed_search_status": search_status,
            "speed_reachability_basis": (
                "fixed_b_fourier_R_outer_full_pulse_detection_miss_success_bisection"
                if SPEED_REFINEMENT_TARGET == "detectability"
                else "fixed_b_reach_miss_bisection"
            ),
        })
    return pd.DataFrame(rows)


def prepare_speed_refinement_input(
    full_population: pd.DataFrame,
    all_results: pd.DataFrame,
    speed_summary: pd.DataFrame,
) -> pd.DataFrame:
    if not ENABLE_SPEED_REACHABILITY_REFINEMENT or len(speed_summary) == 0:
        return full_population.head(0).copy()
    population = add_key_columns(full_population)
    used_ids = set(all_results.get("trajectory_id", pd.Series(dtype=str)).astype(str))
    rows: list[pd.Series] = []
    pending_summary = speed_summary.loc[
        ~bool_series(speed_summary, "speed_reachability_converged")
    ].copy()
    pending_summary["_sample_count"] = numeric_series(
        pending_summary, "n_speed_samples_at_anchor", 0
    ).fillna(0.0)
    pending_summary["_bracket_width"] = numeric_series(
        pending_summary, "speed_bracket_width_m_s", np.inf
    ).fillna(np.inf)
    pending_summary = pending_summary.sort_values(
        ["_sample_count", "_bracket_width", "geometry_key", "speed_anchor_b_m"],
        ascending=[True, False, True, True],
        kind="mergesort",
    )
    for record in pending_summary.itertuples(index=False):
        if bool(record.speed_reachability_converged):
            continue
        target_speed = float(record.speed_reachability_next_probe_m_s)
        if not np.isfinite(target_speed):
            continue
        geometry_population = population.loc[
            population["geometry_key"].astype(str).eq(str(record.geometry_key))
        ].copy()
        if len(geometry_population) == 0:
            continue
        geometry_population["_far_radius"] = np.sqrt(
            numeric_series(geometry_population, "x_far_m")**2
            + numeric_series(geometry_population, "y_far_m")**2
            + numeric_series(geometry_population, "z_far_m")**2
        )
        lower_or_equal = geometry_population.loc[
            numeric_series(geometry_population, "v_inf_m_s") <= target_speed
        ]
        template_pool = lower_or_equal if len(lower_or_equal) else geometry_population
        template = template_pool.sort_values(
            ["_far_radius", "v_inf_m_s"], ascending=[False, True], kind="mergesort"
        ).iloc[0]
        try:
            row = rebuild_template_at_speed_and_b(
                template,
                target_speed,
                float(record.speed_anchor_b_m),
                geometry_key=str(record.geometry_key),
            )
        except ValueError as exc:
            progress(
                f"[SPEED-REFINE] skipped {record.geometry_key} at "
                f"{target_speed:.6f} m/s at "
                f"b={float(record.speed_anchor_b_m) * 1.0e6:.6f} um: {exc}"
            )
            continue
        if str(row["trajectory_id"]) in used_ids:
            continue
        rows.append(row)
        used_ids.add(str(row["trajectory_id"]))
        if len(rows) >= MAX_SPEED_REFINEMENT_ROWS_PER_ROUND:
            break
    if not rows:
        return full_population.head(0).copy()
    return pd.DataFrame(rows).drop(columns=["_far_radius"], errors="ignore")


# =============================================================================
# RESULT CLASSIFICATION
# =============================================================================

RESOLVED_DM_ONLY_MISS_STATUSES = {
    "dm_only_escaped_without_switch",
    "test0p75_adiabatic_reject",
    "stopping_distance_outward_reject",
    "existing_exact_analytic_reject",
}
RESOLVED_FULL_MISS_STATUSES = {
    "entered_switch_but_missed_R_full",
    "rutherford_analytic_miss",
    "rutherford_local_miss_R_full",
    "test0p75_adiabatic_reject",
    "stopping_distance_outward_reject",
    "existing_exact_analytic_reject",
}
UNRESOLVED_TOKENS = (
    "timeout",
    "invalid_domain",
    "solver_failure",
    "task_exception",
    "requires_rebuild",
    "unresolved",
)


def classify_geometry_result(row: pd.Series | dict[str, Any]) -> str:
    if bool_value(row, "reached_R_full"):
        return "reach"
    dm_status = str(row.get("dm_only_status", ""))
    full_status = str(row.get("full_status", ""))
    test2_status = str(row.get("test2_status", ""))
    if dm_status in RESOLVED_DM_ONLY_MISS_STATUSES or dm_status == (
        "trajectory_class_rejected_no_in_range_sample_reaches_switch"
    ):
        return "miss"
    if full_status in RESOLVED_FULL_MISS_STATUSES:
        return "miss"
    joined = "|".join([dm_status, full_status, test2_status]).lower()
    if any(token in joined for token in UNRESOLVED_TOKENS):
        return "unresolved"
    return "unresolved"


def add_result_classifications(df: pd.DataFrame) -> pd.DataFrame:
    output = add_key_columns(df)
    output["geometry_class"] = [
        classify_geometry_result(row) for _, row in output.iterrows()
    ]
    return output


def classify_speed_refinement_result(
    row: pd.Series | dict[str, Any],
) -> str:
    """Classify one row for the speed-refinement target.

    The b scan remains geometric so that it maps where trajectories reach the
    full-coupling region. The subsequent speed scan can instead use the actual
    Fourier-based detection flag, producing a final conditional (v,b) domain
    that is both reachable and above threshold.
    """
    if SPEED_REFINEMENT_TARGET == "reachability":
        return classify_geometry_result(row)
    if bool_value(row, "above_energy_threshold"):
        return "reach"
    dm_status = str(row.get("dm_only_status", ""))
    full_status = str(row.get("full_status", ""))
    test2_status = str(row.get("test2_status", ""))
    if dm_status in RESOLVED_DM_ONLY_MISS_STATUSES or dm_status == (
        "trajectory_class_rejected_no_in_range_sample_reaches_switch"
    ):
        return "miss"
    if full_status in {
        "entered_switch_but_missed_R_full",
        "rutherford_local_miss_R_full",
        "reached_R_full_and_escaped",
        "rutherford_local_reached_R_full",
        "test0p75_adiabatic_reject",
        "stopping_distance_outward_reject",
        "existing_exact_analytic_reject",
    }:
        return "miss"
    joined = "|".join([dm_status, full_status, test2_status]).lower()
    if any(token in joined for token in UNRESOLVED_TOKENS):
        return "unresolved"
    return "unresolved"


# =============================================================================
# PILOT CELL AND RAY SELECTION
# =============================================================================


PILOT_CELL_FEATURE_COLUMNS = [
    "speed_label",
    "theta_index",
    "alpha_index",
    "cell_key",
    "theta_rad",
    "alpha_rad",
    "theta_deg",
    "alpha_deg",
    "pilot_rows",
    "pilot_weight",
    "pilot_any_entered_switch",
    "pilot_any_reached_R_full",
    "pilot_entered_weight_fraction",
    "pilot_reached_weight_fraction",
    "pilot_min_radius_m",
    "pilot_min_b_m",
]


def build_pilot_cell_features(pilot: pd.DataFrame) -> pd.DataFrame:
    data = add_result_classifications(pilot)
    if "trajectory_class_reach_status" in data.columns:
        data = data.loc[
            data["trajectory_class_reach_status"].isin(ACCEPTED_CLASS_STATUSES)
        ].copy()

    # It is valid for the Test 2 pilot to contain no accepted or unresolved
    # trajectory classes. Return an empty table with a stable schema rather
    # than a columnless DataFrame, so downstream grouping and CSV output remain
    # well-defined.
    if len(data) == 0:
        return pd.DataFrame(columns=PILOT_CELL_FEATURE_COLUMNS)

    data["_weight"] = first_existing_weight(data)
    data["_entered"] = bool_series(data, "entered_switch")
    data["_reached"] = bool_series(data, "reached_R_full")
    full_dmin = numeric_series(data, "d_min_m")
    dm_dmin = numeric_series(data, "dm_only_min_radius_m")
    data["_minimum_radius"] = np.where(
        np.isfinite(full_dmin), full_dmin, dm_dmin
    )

    rows: list[dict[str, Any]] = []
    group_columns = ["speed_label", "theta_index", "alpha_index", "cell_key"]
    for keys, group in data.groupby(group_columns, sort=False, dropna=False):
        weight = group["_weight"].to_numpy(float)
        total_weight = float(np.sum(weight))
        minimum = group["_minimum_radius"].to_numpy(float)
        finite_minimum = minimum[np.isfinite(minimum)]
        representative = group.iloc[0]
        row = {column: value for column, value in zip(group_columns, keys)}
        row.update(
            {
                "theta_rad": float(representative["theta_rad"]),
                "alpha_rad": float(representative["alpha_rad"]),
                "theta_deg": float(representative["theta_deg"]),
                "alpha_deg": float(representative["alpha_deg"]),
                "pilot_rows": int(len(group)),
                "pilot_weight": total_weight,
                "pilot_any_entered_switch": bool(group["_entered"].any()),
                "pilot_any_reached_R_full": bool(group["_reached"].any()),
                "pilot_entered_weight_fraction": (
                    float(weight[group["_entered"].to_numpy(bool)].sum()) / total_weight
                    if total_weight > 0.0
                    else 0.0
                ),
                "pilot_reached_weight_fraction": (
                    float(weight[group["_reached"].to_numpy(bool)].sum()) / total_weight
                    if total_weight > 0.0
                    else 0.0
                ),
                "pilot_min_radius_m": (
                    float(np.min(finite_minimum)) if finite_minimum.size else np.nan
                ),
                "pilot_min_b_m": float(numeric_series(group, "b_m").min()),
            }
        )
        rows.append(row)
    return pd.DataFrame(rows, columns=PILOT_CELL_FEATURE_COLUMNS)


def choose_control_cells(
    candidates: pd.DataFrame,
    selected_priority: pd.DataFrame,
    n_controls: int,
) -> pd.DataFrame:
    if n_controls <= 0 or len(candidates) == 0:
        return candidates.head(0).copy()
    remaining = candidates.loc[
        ~candidates["cell_key"].isin(selected_priority["cell_key"])
    ].copy()
    if len(remaining) == 0:
        return remaining

    selected_rows: list[pd.Series] = []
    seed_rows = selected_priority if len(selected_priority) else candidates.head(1)
    for _ in range(min(n_controls, len(remaining))):
        best_index = None
        best_distance = -np.inf
        for index, row in remaining.iterrows():
            distances = [
                angular_distance_deg(
                    float(row["theta_rad"]),
                    float(row["alpha_rad"]),
                    float(seed["theta_rad"]),
                    float(seed["alpha_rad"]),
                )
                for _, seed in pd.concat(
                    [seed_rows, pd.DataFrame(selected_rows)],
                    ignore_index=True,
                ).iterrows()
            ]
            nearest = min(distances) if distances else 180.0
            if nearest > best_distance:
                best_distance = nearest
                best_index = index
        chosen = remaining.loc[best_index].copy()
        chosen["test2p5_distance_to_seed_deg"] = float(best_distance)
        selected_rows.append(chosen)
        remaining = remaining.drop(index=best_index)
    return pd.DataFrame(selected_rows)


def select_angular_cells(pilot: pd.DataFrame) -> pd.DataFrame:
    features = build_pilot_cell_features(pilot)
    if len(features) == 0:
        return features.copy()

    selected: list[pd.DataFrame] = []
    for speed_label, group in features.groupby("speed_label", sort=False):
        group = group.copy()
        group["_reach_rank"] = (~group["pilot_any_reached_R_full"]).astype(int)
        group["_entry_rank"] = (~group["pilot_any_entered_switch"]).astype(int)
        group["_radius_rank"] = numeric_series(group, "pilot_min_radius_m").fillna(np.inf)
        priority = group.sort_values(
            ["_reach_rank", "_entry_rank", "_radius_rank", "pilot_weight"],
            ascending=[True, True, True, False],
            kind="mergesort",
        ).head(min(N_PRIORITY_CELLS_PER_SPEED, len(group))).copy()
        priority["test2p5_cell_role"] = "priority"
        priority["test2p5_distance_to_seed_deg"] = 0.0

        controls = choose_control_cells(
            group,
            priority,
            min(N_CONTROL_CELLS_PER_SPEED, max(N_ANGULAR_CELLS_PER_SPEED - len(priority), 0)),
        )
        if len(controls):
            controls["test2p5_cell_role"] = "angular_control"
        combined = pd.concat([priority, controls], ignore_index=True, sort=False)
        combined = combined.head(N_ANGULAR_CELLS_PER_SPEED)
        selected.append(combined)
    if not selected:
        return features.head(0).copy()
    output = pd.concat(selected, ignore_index=True, sort=False)
    return output.drop(columns=["_reach_rank", "_entry_rank", "_radius_rank"], errors="ignore")


PILOT_RAY_FEATURE_COLUMNS = [
    "ray_key",
    "speed_label",
    "theta_index",
    "alpha_index",
    "psi_key",
    "psi_rad",
    "psi_deg",
    "pilot_ray_any_entered",
    "pilot_ray_any_reached",
    "pilot_ray_min_radius_m",
]


def build_pilot_ray_features(pilot: pd.DataFrame) -> pd.DataFrame:
    data = add_result_classifications(pilot)
    if "trajectory_class_reach_status" in data.columns:
        data = data.loc[
            data["trajectory_class_reach_status"].isin(ACCEPTED_CLASS_STATUSES)
        ].copy()

    if len(data) == 0:
        return pd.DataFrame(columns=PILOT_RAY_FEATURE_COLUMNS)

    data["_entered"] = bool_series(data, "entered_switch")
    data["_reached"] = bool_series(data, "reached_R_full")
    full_dmin = numeric_series(data, "d_min_m")
    dm_dmin = numeric_series(data, "dm_only_min_radius_m")
    data["_minimum_radius"] = np.where(np.isfinite(full_dmin), full_dmin, dm_dmin)
    rows: list[dict[str, Any]] = []
    for ray_key, group in data.groupby("ray_key", sort=False):
        representative = group.iloc[0]
        finite_min = group["_minimum_radius"].to_numpy(float)
        finite_min = finite_min[np.isfinite(finite_min)]
        rows.append(
            {
                "ray_key": ray_key,
                "speed_label": str(representative["speed_label"]),
                "theta_index": int(representative["theta_index"]),
                "alpha_index": int(representative["alpha_index"]),
                "psi_key": float(representative["psi_key"]),
                "psi_rad": float(representative["psi_rad"]),
                "psi_deg": float(representative["psi_deg"]),
                "pilot_ray_any_entered": bool(group["_entered"].any()),
                "pilot_ray_any_reached": bool(group["_reached"].any()),
                "pilot_ray_min_radius_m": (
                    float(np.min(finite_min)) if finite_min.size else np.nan
                ),
            }
        )
    return pd.DataFrame(rows, columns=PILOT_RAY_FEATURE_COLUMNS)


def choose_psi_values_for_cell(
    available_psi: np.ndarray,
    pilot_rays: pd.DataFrame,
    n_values: int,
) -> list[float]:
    available = np.sort(np.unique(np.asarray(available_psi, dtype=float)))
    if len(available) <= n_values:
        return available.tolist()
    chosen: list[float] = []
    if len(pilot_rays):
        ranked = pilot_rays.copy()
        ranked["_reach_rank"] = (~ranked["pilot_ray_any_reached"]).astype(int)
        ranked["_entry_rank"] = (~ranked["pilot_ray_any_entered"]).astype(int)
        ranked["_radius_rank"] = numeric_series(
            ranked, "pilot_ray_min_radius_m"
        ).fillna(np.inf)
        ranked = ranked.sort_values(
            ["_reach_rank", "_entry_rank", "_radius_rank"], kind="mergesort"
        )
        for value in ranked["psi_key"].to_numpy(float):
            nearest = float(available[np.argmin(np.abs(available - value))])
            if nearest not in chosen:
                chosen.append(nearest)
            if len(chosen) >= n_values:
                return chosen
    indices = np.unique(np.rint(np.linspace(0, len(available) - 1, n_values)).astype(int))
    for index in indices:
        value = float(available[index])
        if value not in chosen:
            chosen.append(value)
        if len(chosen) >= n_values:
            break
    return chosen



def physical_population_for_ray(ray_population: pd.DataFrame) -> pd.DataFrame:
    """Return usable templates for an accepted physical trajectory class."""
    data = ray_population.copy()
    if "trajectory_class_reach_status" in data.columns:
        data = data.loc[
            data["trajectory_class_reach_status"].isin(ACCEPTED_CLASS_STATUSES)
        ].copy()
    b = numeric_series(data, "b_m")
    finite_state = np.isfinite(b)
    for column in (
        "x_far_m", "y_far_m", "z_far_m",
        "vx_far_m_s", "vy_far_m_s", "vz_far_m_s",
    ):
        finite_state &= np.isfinite(numeric_series(data, column))
    if REQUIRE_POSITIVE_WEIGHT_PRODUCTION_ROWS and "impact_weight" in data.columns:
        finite_state &= numeric_series(data, "impact_weight", 0.0).fillna(0.0) > 0.0
    return data.loc[finite_state].copy().sort_values("b_m", kind="mergesort")



def physical_b_support(ray_population: pd.DataFrame) -> tuple[float, float]:
    annuli = physical_b_annuli(ray_population)
    if not annuli:
        return np.nan, np.nan
    return annuli[0][0], annuli[-1][1]

def select_candidate_rays(
    full_population: pd.DataFrame,
    pilot: pd.DataFrame,
    selected_cells: pd.DataFrame,
) -> pd.DataFrame:
    pilot_rays = build_pilot_ray_features(pilot)
    rows: list[dict[str, Any]] = []
    for cell in selected_cells.itertuples(index=False):
        cell_population = full_population.loc[
            (full_population["speed_label"] == str(cell.speed_label))
            & (full_population["theta_index"] == int(cell.theta_index))
            & (full_population["alpha_index"] == int(cell.alpha_index))
        ]
        available_psi = cell_population["psi_key"].dropna().unique()
        cell_pilot = pilot_rays.loc[
            (pilot_rays["speed_label"] == str(cell.speed_label))
            & (pilot_rays["theta_index"] == int(cell.theta_index))
            & (pilot_rays["alpha_index"] == int(cell.alpha_index))
        ]
        chosen_psi = choose_psi_values_for_cell(
            available_psi, cell_pilot, N_PSI_PER_ANGULAR_CELL
        )
        for psi_value in chosen_psi:
            ray_population = cell_population.loc[
                np.isclose(
                    cell_population["psi_key"].to_numpy(float),
                    psi_value,
                    rtol=0.0,
                    atol=1.0e-12,
                )
            ]
            if len(ray_population) == 0:
                continue
            production_population = physical_population_for_ray(ray_population)
            if len(production_population) == 0:
                continue
            support_lower, support_upper = physical_b_support(ray_population)
            representative = production_population.iloc[0]
            match = cell_pilot.loc[
                np.isclose(
                    cell_pilot["psi_key"].to_numpy(float),
                    psi_value,
                    rtol=0.0,
                    atol=1.0e-12,
                )
            ]
            rows.append(
                {
                    "speed_label": str(cell.speed_label),
                    "v_inf_m_s": float(representative["v_inf_m_s"]),
                    "R_full_m": float(representative["R_full_m"]),
                    "R_switch_m": float(representative["R_switch_m"]),
                    "theta_index": int(cell.theta_index),
                    "alpha_index": int(cell.alpha_index),
                    "theta_rad": float(representative["theta_rad"]),
                    "alpha_rad": float(representative["alpha_rad"]),
                    "theta_deg": float(representative["theta_deg"]),
                    "alpha_deg": float(representative["alpha_deg"]),
                    "psi_key": float(psi_value),
                    "psi_rad": float(representative["psi_rad"]),
                    "psi_deg": float(representative["psi_deg"]),
                    "cell_key": str(representative["cell_key"]),
                    "ray_key": str(representative["ray_key"]),
                    "geometry_key": str(representative["geometry_key"]),
                    "test2p5_cell_role": str(cell.test2p5_cell_role),
                    "test2p5_distance_to_seed_deg": float(
                        cell.test2p5_distance_to_seed_deg
                    ),
                    "pilot_ray_any_reached": (
                        bool(match.iloc[0]["pilot_ray_any_reached"])
                        if len(match)
                        else False
                    ),
                    "pilot_ray_any_entered": (
                        bool(match.iloc[0]["pilot_ray_any_entered"])
                        if len(match)
                        else False
                    ),
                    "pilot_ray_min_radius_m": (
                        float(match.iloc[0]["pilot_ray_min_radius_m"])
                        if len(match)
                        else np.nan
                    ),
                    "available_b_count": int(production_population["b_m"].nunique()),
                    "available_b_min_m": float(numeric_series(production_population, "b_m").min()),
                    "available_b_max_m": float(numeric_series(production_population, "b_m").max()),
                    "physical_b_support_lower_m": support_lower,
                    "physical_b_support_upper_m": support_upper,
                    "physical_b_support_lower_um": support_lower * 1.0e6,
                    "physical_b_support_upper_um": support_upper * 1.0e6,
                }
            )
    if not rows:
        return pd.DataFrame(
            columns=[
                "speed_label", "v_inf_m_s", "R_full_m", "R_switch_m",
                "theta_index", "alpha_index", "theta_rad", "alpha_rad",
                "psi_key", "psi_rad", "cell_key", "ray_key", "geometry_key",
            ]
        )
    return pd.DataFrame(rows).sort_values(
        ["speed_label", "theta_index", "alpha_index", "psi_key"],
        kind="mergesort",
    ).reset_index(drop=True)


# =============================================================================
# INITIAL PHYSICAL b^2-SUPPORT SELECTION
# =============================================================================



def nearest_unique_rows_in_physical_b2_support(
    ray_population: pd.DataFrame,
    target_fractions: Iterable[float],
    *,
    excluded_ids: set[str],
    max_rows: int | None = None,
) -> list[pd.Series]:
    data = physical_population_for_ray(ray_population)
    if len(data) == 0:
        return []
    annuli = physical_b_annuli(data)
    if not annuli:
        return []
    template = data.sort_values("b_m", kind="mergesort").iloc[0]
    chosen_rows: list[pd.Series] = []
    seen_b: list[float] = []
    for target_fraction in target_fractions:
        if max_rows is not None and len(chosen_rows) >= max_rows:
            break
        target_fraction = float(np.clip(target_fraction, 0.0, 1.0))
        target_b, annulus_index, annulus_lower, annulus_upper = b_from_union_area_fraction(
            annuli, target_fraction
        )
        if not np.isfinite(target_b):
            continue
        if any(np.isclose(target_b, value, rtol=0.0, atol=1.0e-15) for value in seen_b):
            continue
        suffix = f"initial_f{target_fraction:.8f}".replace(".", "p")
        row = rebuild_template_at_b(
            template,
            target_b,
            reason="initial_exact_annulus_union_b2_target",
            id_suffix=suffix,
        )
        if str(row["trajectory_id"]) in excluded_ids:
            continue
        row["test2p5_initial_b_role"] = "exact_annulus_union_b2_target"
        row["test2p5_target_physical_b2_fraction"] = target_fraction
        row["test2p5_actual_physical_b2_fraction"] = target_fraction
        row["test2p5_target_b_m"] = target_b
        row["test2p5_target_b_um"] = target_b * 1.0e6
        row["test2p5_target_annulus_index"] = annulus_index
        row["test2p5_target_annulus_lower_m"] = annulus_lower
        row["test2p5_target_annulus_upper_m"] = annulus_upper
        row["test2p5_physical_b_support_lower_m"] = annuli[0][0]
        row["test2p5_physical_b_support_upper_m"] = annuli[-1][1]
        row["test2p5_physical_b_support_lower_um"] = annuli[0][0] * 1.0e6
        row["test2p5_physical_b_support_upper_um"] = annuli[-1][1] * 1.0e6
        chosen_rows.append(row)
        seen_b.append(target_b)
    return chosen_rows

def prepare_initial_round_input(
    full_population: pd.DataFrame,
    pilot: pd.DataFrame,
    selected_rays: pd.DataFrame,
) -> pd.DataFrame:
    pilot_classified = add_result_classifications(pilot)
    resolved_pilot_ids = set(
        pilot_classified.loc[
            pilot_classified["geometry_class"].isin(["reach", "miss"]),
            "trajectory_id",
        ].astype(str)
    )
    rows: list[pd.Series] = []
    for ray in selected_rays.itertuples(index=False):
        ray_population = full_population.loc[
            full_population["ray_key"].astype(str).eq(str(ray.ray_key))
        ].copy()
        chosen = nearest_unique_rows_in_physical_b2_support(
            ray_population,
            INITIAL_PHYSICAL_B2_FRACTIONS,
            excluded_ids=resolved_pilot_ids,
            max_rows=INITIAL_MAX_B_PER_RAY,
        )
        for row in chosen:
            initial_role = str(
                row.get("test2p5_initial_b_role", "physical_b2_fraction_target")
            )
            row["test2p5_selection_reason"] = (
                f"initial_physical_b2_scan_{initial_role}"
            )
            row["test2p5_ray_key"] = str(ray.ray_key)
            rows.append(row)
    if not rows:
        return full_population.head(0).copy()
    return pd.DataFrame(rows).drop_duplicates("trajectory_id", keep="first")


# =============================================================================
# RAY ANALYSIS
# =============================================================================


def contiguous_true_intervals(values: np.ndarray) -> list[tuple[int, int]]:
    intervals: list[tuple[int, int]] = []
    start: int | None = None
    for index, value in enumerate(values):
        if value and start is None:
            start = index
        if start is not None and (not value or index == len(values) - 1):
            stop = index if value and index == len(values) - 1 else index - 1
            intervals.append((start, stop))
            start = None
    return intervals


def analyze_ray_results(results: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    summaries: list[dict[str, Any]] = []
    intervals: list[dict[str, Any]] = []
    for ray_key, group in results.groupby("ray_key", sort=False):
        data = group.copy().sort_values("b_m", kind="mergesort")
        data = data.drop_duplicates("trajectory_id", keep="last")
        b = numeric_series(data, "b_m").to_numpy(float)
        classes = data["geometry_class"].astype(str).to_numpy()
        resolved = np.isin(classes, ["reach", "miss"])
        reach = classes == "reach"
        miss = classes == "miss"
        unresolved = classes == "unresolved"

        resolved_data = data.loc[resolved].copy()
        resolved_b = numeric_series(resolved_data, "b_m").to_numpy(float)
        resolved_reach = resolved_data["geometry_class"].eq("reach").to_numpy(bool)
        resolved_intervals = contiguous_true_intervals(resolved_reach)
        nonmonotonic = False
        if len(resolved_reach):
            seen_miss = False
            for value in resolved_reach:
                if not value:
                    seen_miss = True
                elif seen_miss:
                    nonmonotonic = True
                    break

        reaching_b = b[reach]
        lower = float(np.max(reaching_b)) if reaching_b.size else np.nan
        upper_candidates = b[miss & (b > lower)] if np.isfinite(lower) else np.array([])
        upper = float(np.min(upper_candidates)) if upper_candidates.size else np.nan
        estimate = (
            math.sqrt(0.5 * (lower**2 + upper**2))
            if np.isfinite(lower) and np.isfinite(upper)
            else np.nan
        )
        bracket_width = upper - lower if np.isfinite(lower) and np.isfinite(upper) else np.nan

        representative = data.iloc[0]
        targeted_ray = bool_value(
            representative, "targeted_escape_refinement", False
        ) or str(representative.get("speed_label", "")).startswith("escape_refine_")
        b_tolerance_m = (
            TARGETED_BMAX_BRACKET_TOLERANCE_M
            if targeted_ray
            else BMAX_BRACKET_TOLERANCE_M
        )

        zero_tolerance = max(1.0e-15, 1.0e-9 * float(np.nanmax(b) if len(b) else 1.0))
        b0_miss = bool(np.any(miss & (np.abs(b) <= zero_tolerance)))
        b0_unresolved = bool(np.any(unresolved & (np.abs(b) <= zero_tolerance)))
        if b0_miss and not np.any(reach):
            lower = 0.0
            upper = 0.0
            estimate = 0.0
            bracket_width = 0.0
            converged = True
            convergence_reason = "resolved_b0_miss"
        elif nonmonotonic:
            converged = False
            convergence_reason = "nonmonotonic_reachability"
        elif np.isfinite(bracket_width) and bracket_width <= b_tolerance_m:
            converged = True
            convergence_reason = "bracket_tolerance"
        else:
            converged = False
            convergence_reason = "needs_refinement"

        support_lower_values = numeric_series(data, "test2p5_physical_b_support_lower_m")
        if not np.isfinite(support_lower_values).any():
            support_lower_values = numeric_series(data, "physical_b_support_lower_m")
        support_upper_values = numeric_series(data, "test2p5_physical_b_support_upper_m")
        if not np.isfinite(support_upper_values).any():
            support_upper_values = numeric_series(data, "physical_b_support_upper_m")
        support_lower = (
            float(support_lower_values.dropna().iloc[0])
            if np.isfinite(support_lower_values).any()
            else np.nan
        )
        support_upper = (
            float(support_upper_values.dropna().iloc[0])
            if np.isfinite(support_upper_values).any()
            else np.nan
        )
        summaries.append(
            {
                "ray_key": ray_key,
                "speed_label": str(representative["speed_label"]),
                "v_inf_m_s": float(representative["v_inf_m_s"]),
                "theta_index": int(representative["theta_index"]),
                "alpha_index": int(representative["alpha_index"]),
                "theta_rad": float(representative["theta_rad"]),
                "alpha_rad": float(representative["alpha_rad"]),
                "theta_deg": float(representative["theta_deg"]),
                "alpha_deg": float(representative["alpha_deg"]),
                "psi_key": float(representative["psi_key"]),
                "psi_rad": float(representative["psi_rad"]),
                "psi_deg": float(representative["psi_deg"]),
                "physical_b_support_lower_m": support_lower,
                "physical_b_support_upper_m": support_upper,
                "physical_b_support_lower_um": (
                    support_lower * 1.0e6 if np.isfinite(support_lower) else np.nan
                ),
                "physical_b_support_upper_um": (
                    support_upper * 1.0e6 if np.isfinite(support_upper) else np.nan
                ),
                "n_results": int(len(data)),
                "n_resolved": int(np.sum(resolved)),
                "n_reach": int(np.sum(reach)),
                "n_miss": int(np.sum(miss)),
                "n_unresolved": int(np.sum(unresolved)),
                "b0_resolved_miss": b0_miss,
                "b0_unresolved": b0_unresolved,
                "single_cutoff_monotonic": not nonmonotonic,
                "n_sampled_reaching_intervals": int(len(resolved_intervals)),
                "bmax_lower_m": lower,
                "bmax_estimate_m": estimate,
                "bmax_upper_m": upper,
                "bmax_lower_um": lower * 1.0e6 if np.isfinite(lower) else np.nan,
                "bmax_estimate_um": estimate * 1.0e6 if np.isfinite(estimate) else np.nan,
                "bmax_upper_um": upper * 1.0e6 if np.isfinite(upper) else np.nan,
                "bracket_width_m": bracket_width,
                "bracket_width_um": (
                    bracket_width * 1.0e6 if np.isfinite(bracket_width) else np.nan
                ),
                "largest_sampled_b_m": float(np.max(b)) if len(b) else np.nan,
                "smallest_sampled_b_m": float(np.min(b)) if len(b) else np.nan,
                "reaching_at_largest_sampled_b": bool(reach[-1]) if len(reach) else False,
                "converged": converged,
                "convergence_reason": convergence_reason,
                "targeted_escape_refinement": bool(targeted_ray),
                "bmax_bracket_tolerance_m": float(b_tolerance_m),
            }
        )

        for interval_index, (start, stop) in enumerate(resolved_intervals):
            intervals.append(
                {
                    "ray_key": ray_key,
                    "speed_label": str(representative["speed_label"]),
                    "theta_index": int(representative["theta_index"]),
                    "alpha_index": int(representative["alpha_index"]),
                    "psi_key": float(representative["psi_key"]),
                    "interval_index": interval_index,
                    "b_start_m": float(resolved_b[start]),
                    "b_stop_m": float(resolved_b[stop]),
                    "b_start_um": float(resolved_b[start] * 1.0e6),
                    "b_stop_um": float(resolved_b[stop] * 1.0e6),
                    "n_samples_in_interval": int(stop - start + 1),
                }
            )
    return pd.DataFrame(summaries), pd.DataFrame(intervals)


# =============================================================================
# ADAPTIVE NEXT-ROUND SELECTION
# =============================================================================


def nearest_unused_row(
    population: pd.DataFrame,
    target_b: float,
    used_ids: set[str],
    *,
    lower_bound: float | None = None,
    upper_bound: float | None = None,
) -> pd.Series | None:
    data = physical_population_for_ray(population)
    if len(data) == 0:
        return None
    support_lower, support_upper = physical_b_support(data)
    lower = max(support_lower, lower_bound) if lower_bound is not None else support_lower
    upper = min(support_upper, upper_bound) if upper_bound is not None else support_upper
    if not (np.isfinite(target_b) and np.isfinite(lower) and np.isfinite(upper)):
        return None
    if not (lower < target_b < upper):
        return None
    template = data.sort_values("b_m", kind="mergesort").iloc[0]
    suffix = (
        f"adaptive_b{target_b:.12e}"
        .replace("+", "")
        .replace("-", "m")
        .replace(".", "p")
    )
    row = rebuild_template_at_b(
        template,
        float(target_b),
        reason="adaptive_generated_b2_target",
        id_suffix=suffix,
    )
    if str(row["trajectory_id"]) in used_ids:
        return None
    return row


def unresolved_neighbor_rows(
    population: pd.DataFrame,
    ray_results: pd.DataFrame,
    used_ids: set[str],
) -> list[pd.Series]:
    unresolved = ray_results.loc[ray_results["geometry_class"].eq("unresolved")]
    if len(unresolved) == 0:
        return []
    support_lower, support_upper = physical_b_support(population)
    if not (np.isfinite(support_lower) and np.isfinite(support_upper)):
        return []
    width2 = support_upper**2 - support_lower**2
    rows: list[pd.Series] = []
    for index, b_unresolved in enumerate(numeric_series(unresolved, "b_m").to_numpy(float)):
        if len(rows) >= MAX_UNRESOLVED_NEIGHBORS_PER_RAY:
            break
        offset = 0.01 * width2
        direction = -1.0 if index % 2 == 0 else 1.0
        target2 = np.clip(
            b_unresolved**2 + direction * offset,
            support_lower**2,
            support_upper**2,
        )
        target = math.sqrt(float(target2))
        row = nearest_unused_row(
            population,
            target,
            used_ids,
            lower_bound=support_lower,
            upper_bound=support_upper,
        )
        if row is not None:
            row["test2p5_selection_reason"] = "generated_neighbor_of_unresolved_result"
            rows.append(row)
            used_ids.add(str(row["trajectory_id"]))
    return rows


def prepare_next_round_input(
    full_population: pd.DataFrame,
    all_results: pd.DataFrame,
    selected_rays: pd.DataFrame,
    ray_summary: pd.DataFrame,
) -> pd.DataFrame:
    used_ids = set(all_results["trajectory_id"].astype(str))
    selected_rows: list[pd.Series] = []

    for ray in selected_rays.itertuples(index=False):
        summary_match = ray_summary.loc[ray_summary["ray_key"].eq(str(ray.ray_key))]
        if len(summary_match) == 0:
            continue
        summary = summary_match.iloc[0]
        if bool(summary["converged"]):
            continue
        ray_population = physical_population_for_ray(
            full_population.loc[
                full_population["ray_key"].eq(str(ray.ray_key))
            ].copy()
        )
        ray_results = all_results.loc[
            all_results["ray_key"].eq(str(ray.ray_key))
        ].copy().sort_values("b_m", kind="mergesort")
        new_for_ray: list[pd.Series] = []

        lower = float(summary["bmax_lower_m"])
        upper = float(summary["bmax_upper_m"])
        if np.isfinite(lower) and np.isfinite(upper) and upper > lower:
            target = math.sqrt(0.5 * (lower**2 + upper**2))
            row = nearest_unused_row(
                ray_population,
                target,
                used_ids,
                lower_bound=lower,
                upper_bound=upper,
            )
            if row is not None:
                row["test2p5_selection_reason"] = "area_midpoint_reach_miss_bracket"
                row["test2p5_target_b_m"] = target
                new_for_ray.append(row)
        elif int(summary["n_reach"]) > 0:
            largest_reach = float(
                numeric_series(
                    ray_results.loc[ray_results["geometry_class"].eq("reach")],
                    "b_m",
                ).max()
            )
            support_lower, support_upper = physical_b_support(ray_population)
            if np.isfinite(support_upper) and support_upper > largest_reach:
                target = math.sqrt(0.5 * (largest_reach**2 + support_upper**2))
                row = nearest_unused_row(
                    ray_population,
                    target,
                    used_ids,
                    lower_bound=largest_reach,
                    upper_bound=support_upper,
                )
                if row is not None:
                    row["test2p5_selection_reason"] = "expand_above_largest_reach"
                    new_for_ray.append(row)
        else:
            resolved_misses = ray_results.loc[ray_results["geometry_class"].eq("miss")]
            if len(resolved_misses):
                smallest_miss = float(numeric_series(resolved_misses, "b_m").min())
                support_lower, support_upper = physical_b_support(ray_population)
                if np.isfinite(support_lower) and smallest_miss > support_lower:
                    target = math.sqrt(0.5 * (support_lower**2 + smallest_miss**2))
                    row = nearest_unused_row(
                        ray_population,
                        target,
                        used_ids,
                        lower_bound=support_lower,
                        upper_bound=smallest_miss,
                    )
                    if row is not None:
                        row["test2p5_selection_reason"] = "search_below_smallest_resolved_miss"
                        new_for_ray.append(row)

        if len(new_for_ray) < MAX_NEW_B_PER_RAY_PER_ROUND:
            neighbors = unresolved_neighbor_rows(ray_population, ray_results, used_ids)
            for row in neighbors:
                if len(new_for_ray) >= MAX_NEW_B_PER_RAY_PER_ROUND:
                    break
                if str(row["trajectory_id"]) not in {
                    str(item["trajectory_id"]) for item in new_for_ray
                }:
                    new_for_ray.append(row)

        for row in new_for_ray[:MAX_NEW_B_PER_RAY_PER_ROUND]:
            row["test2p5_ray_key"] = str(ray.ray_key)
            selected_rows.append(row)
            used_ids.add(str(row["trajectory_id"]))
            if len(selected_rows) >= MAX_NEXT_ROUND_ROWS:
                break
        if len(selected_rows) >= MAX_NEXT_ROUND_ROWS:
            break

    if not selected_rows:
        return full_population.head(0).copy()
    return pd.DataFrame(selected_rows).drop_duplicates("trajectory_id", keep="first")


# =============================================================================
# AGGREGATED ANGULAR SUMMARY
# =============================================================================


def build_theta_alpha_summary(ray_summary: pd.DataFrame) -> pd.DataFrame:
    if len(ray_summary) == 0:
        return pd.DataFrame()
    rows: list[dict[str, Any]] = []
    group_columns = [
        "speed_label",
        "v_inf_m_s",
        "theta_index",
        "alpha_index",
        "theta_rad",
        "alpha_rad",
        "theta_deg",
        "alpha_deg",
    ]
    for keys, group in ray_summary.groupby(group_columns, sort=False, dropna=False):
        estimates = numeric_series(group, "bmax_estimate_m").to_numpy(float)
        lower = numeric_series(group, "bmax_lower_m").to_numpy(float)
        upper = numeric_series(group, "bmax_upper_m").to_numpy(float)
        finite_est = np.isfinite(estimates)
        finite_low = np.isfinite(lower)
        finite_up = np.isfinite(upper)
        row = {column: value for column, value in zip(group_columns, keys)}
        row.update(
            {
                "n_psi_rays": int(len(group)),
                "n_psi_converged": int(bool_series(group, "converged").sum()),
                "converged_psi_fraction": float(bool_series(group, "converged").mean()),
                "any_nonmonotonic_psi": bool(
                    (~bool_series(group, "single_cutoff_monotonic")).any()
                ),
                "max_sampled_psi_bmax_lower_m": (
                    float(np.max(lower[finite_low])) if np.any(finite_low) else np.nan
                ),
                "max_sampled_psi_bmax_estimate_m": (
                    float(np.max(estimates[finite_est])) if np.any(finite_est) else np.nan
                ),
                "max_sampled_psi_bmax_upper_m": (
                    float(np.max(upper[finite_up])) if np.any(finite_up) else np.nan
                ),
                "median_sampled_psi_bmax_estimate_m": (
                    float(np.median(estimates[finite_est])) if np.any(finite_est) else np.nan
                ),
            }
        )
        rows.append(row)
    output = pd.DataFrame(rows)
    for column in (
        "max_sampled_psi_bmax_lower_m",
        "max_sampled_psi_bmax_estimate_m",
        "max_sampled_psi_bmax_upper_m",
        "median_sampled_psi_bmax_estimate_m",
    ):
        output[column.replace("_m", "_um")] = numeric_series(output, column) * 1.0e6
    return output


# =============================================================================
# ROUND OUTPUT
# =============================================================================


def write_next_round(
    next_input: pd.DataFrame,
    round_number: int,
) -> tuple[Path, Path]:
    input_path = Path(f"{OUTPUT_PREFIX}_round{round_number:02d}_input.csv")
    settings_path = NEXT_SETTINGS_TXT
    if input_path.exists() and not OVERWRITE_NEXT_ROUND_INPUT:
        raise FileExistsError(f"Round input already exists: {input_path}")
    next_input.to_csv(input_path, index=False)
    round_prefix = f"{OUTPUT_PREFIX}_round{round_number:02d}"
    settings = f'''# Apply these settings in test2_class_reach_outward_force_v11.py
TEST1_INPUT_CSV = Path({str(input_path)!r})
PREFILTER_REJECTION_STUB_CSV = None
OUTPUT_PREFIX = {str(round_prefix)!r}
ENABLE_TRAJECTORY_CLASS_REACH_SCREEN = False
CLASS_REACH_PROBES_CSV = Path(f"{{OUTPUT_PREFIX}}_class_reach_probes.csv")
CLASS_REACH_SUMMARY_CSV = Path(f"{{OUTPUT_PREFIX}}_class_reach_summary.csv")
DM_ONLY_SCREEN_CSV = Path(f"{{OUTPUT_PREFIX}}_dm_only_screen.csv")
FULL_STAGE_RESULTS_CSV = Path(f"{{OUTPUT_PREFIX}}_full_stage_results.csv")
COMBINED_RESULTS_CSV = Path(f"{{OUTPUT_PREFIX}}_trajectory_results.csv")
WEIGHTED_ANALYSIS_CSV = Path(f"{{OUTPUT_PREFIX}}_weighted_analysis_results.csv")
WEIGHTED_GLOBAL_SUMMARY_CSV = Path(f"{{OUTPUT_PREFIX}}_weighted_global_summary.csv")
WEIGHTED_CLASSIFICATION_SUMMARY_CSV = Path(f"{{OUTPUT_PREFIX}}_weighted_classification_summary.csv")
GENERATE_TEST2_PLOTS = False  # only if your local Test 2 file defines it
'''
    settings_path.write_text(settings, encoding="utf-8")
    return input_path, settings_path


# =============================================================================
# MAIN
# =============================================================================


def main() -> None:
    full_population = add_key_columns(load_csv(CLASS_TEMPLATE_INPUT_CSV))
    validate_test1_population(full_population)
    validate_parameter_dataframe(full_population, "Test 1.5 class template")
    pilot = add_key_columns(load_csv(PILOT_TEST2_RESULTS_CSV))
    validate_test2_results(pilot, "Pilot Test 2 results")
    validate_parameter_dataframe(pilot, "Pilot Test 2 results")

    if "trajectory_class_reach_status" in pilot.columns:
        status_table = (
            pilot[["trajectory_class_id", "trajectory_class_reach_status"]]
            .dropna(subset=["trajectory_class_reach_status"])
            .drop_duplicates("trajectory_class_id", keep="last")
        )
        full_population = full_population.drop(
            columns=["trajectory_class_reach_status"], errors="ignore"
        ).merge(
            status_table,
            on="trajectory_class_id",
            how="left",
            validate="many_to_one",
        )

    selected_cells = select_angular_cells(pilot)
    selected_cells.to_csv(SELECTED_CELLS_CSV, index=False)
    if len(selected_cells) == 0:
        progress(
            "Pilot Test 2 contains no accepted or unresolved trajectory "
            "classes. Test 2.5 has no angular cells to refine."
        )

    selected_rays = select_candidate_rays(full_population, pilot, selected_cells)
    selected_rays = add_key_columns(selected_rays)
    selected_rays.to_csv(SELECTED_RAYS_CSV, index=False)
    if len(selected_rays) == 0:
        write_header_safe_empty_refinement_outputs()
        progress("No accepted or unresolved trajectory classes remain for Test 2.5.")
        if NEXT_SETTINGS_TXT.exists():
            NEXT_SETTINGS_TXT.unlink()
        return

    round_files = find_round_result_files()
    completed_rounds: list[pd.DataFrame] = []
    for number, path in round_files:
        data = add_key_columns(load_csv(path))
        validate_test2_results(data, f"Round {number}")
        data["test2p5_round_number"] = number
        completed_rounds.append(data)

    pilot_selected = pilot.loc[
        pilot["geometry_key"].astype(str).isin(
            set(selected_rays["geometry_key"].astype(str))
        )
    ].copy()
    pilot_selected["test2p5_round_number"] = 0
    all_results = pd.concat(
        [pilot_selected, *completed_rounds],
        ignore_index=True,
        sort=False,
    )
    all_results = add_result_classifications(all_results)
    all_results = all_results.drop_duplicates("trajectory_id", keep="last")
    all_results.to_csv(ALL_RESULTS_CSV, index=False)

    ray_summary, intervals = analyze_ray_results(all_results)
    ray_summary.to_csv(RAY_SUMMARY_CSV, index=False)
    intervals.to_csv(INTERVALS_CSV, index=False)
    theta_alpha = build_theta_alpha_summary(ray_summary)
    theta_alpha.to_csv(THETA_ALPHA_SUMMARY_CSV, index=False)
    speed_summary = analyze_speed_reachability(
        all_results,
        selected_rays,
        full_population,
        ray_summary,
    )
    speed_summary.to_csv(SPEED_REACHABILITY_SUMMARY_CSV, index=False)

    if not round_files:
        next_input = prepare_initial_round_input(
            full_population,
            pilot,
            selected_rays,
        )
        next_round_number = 1
    else:
        next_round_number = max(number for number, _ in round_files) + 1
        if next_round_number > MAX_REFINEMENT_ROUNDS:
            next_input = full_population.head(0).copy()
        else:
            next_input = prepare_next_round_input(
                full_population,
                all_results,
                selected_rays,
                ray_summary,
            )

    refinement_phase = "b"
    if len(next_input) == 0 and ENABLE_SPEED_REACHABILITY_REFINEMENT:
        next_input = prepare_speed_refinement_input(
            full_population,
            all_results,
            speed_summary,
        )
        refinement_phase = "speed"

    progress(f"Selected angular cells: {len(selected_cells):,}")
    progress(f"Selected rays: {len(selected_rays):,}")
    progress(f"Available classified results: {len(all_results):,}")
    if len(ray_summary):
        progress(
            "Converged rays: "
            f"{int(bool_series(ray_summary, 'converged').sum()):,}/"
            f"{len(ray_summary):,}"
        )
    if len(speed_summary):
        progress(
            "Converged speed brackets: "
            f"{int(bool_series(speed_summary, 'speed_reachability_converged').sum()):,}/"
            f"{len(speed_summary):,}"
        )

    if len(next_input) == 0:
        progress("No new Test 2.5 rows remain. Adaptive mapping is complete or exhausted.")
        if NEXT_SETTINGS_TXT.exists():
            NEXT_SETTINGS_TXT.unlink()
    else:
        input_path, settings_path = write_next_round(next_input, next_round_number)
        progress(
            f"Saved next {refinement_phase} refinement input "
            f"({len(next_input):,} rows): {input_path}"
        )
        progress(f"Saved Test 2 settings: {settings_path}")
        progress(
            "Run test2_class_reach_outward_force_v11.py with those settings, "
            "then rerun this Test 2.5 file."
        )


# =============================================================================
# AUTOMATIC TEST 2 / TEST 2.5 ORCHESTRATION
# =============================================================================


def _test2p5_round_input_path(round_number: int) -> Path:
    return Path(
        f"{TEST2P5_OUTPUT_PREFIX}_round{int(round_number):02d}_input.csv"
    )


def _test2p5_round_result_path(round_number: int) -> Path:
    return Path(
        f"{TEST2P5_OUTPUT_PREFIX}_round{int(round_number):02d}"
        "_trajectory_results.csv"
    )


def _configure_embedded_test2_for_round(
    input_path: Path,
    round_number: int,
) -> Path:
    """Point the isolated Test 2 namespace at one Test 2.5 refinement round."""
    if "TEST2_NAMESPACE" not in globals():
        raise RuntimeError(
            "The Test 2 cell must run before Test 2.5 so TEST2_NAMESPACE exists."
        )

    input_path = Path(input_path)
    round_prefix = Path(
        f"{TEST2P5_OUTPUT_PREFIX}_round{int(round_number):02d}"
    )
    namespace = TEST2_NAMESPACE

    namespace["TEST1_INPUT_CSV"] = input_path
    namespace["PREFILTER_REJECTION_STUB_CSV"] = None
    # Escape-channel focused diagnostics belong only to the pilot. Do not
    # append them to every adaptive Test 2.5 refinement round.
    if "EXTRA_DIAGNOSTIC_INPUT_CSV" in namespace:
        namespace["EXTRA_DIAGNOSTIC_INPUT_CSV"] = None
    namespace["OUTPUT_PREFIX"] = str(round_prefix)
    namespace["ENABLE_TRAJECTORY_CLASS_REACH_SCREEN"] = False
    namespace["RESUME_EXISTING_OUTPUT"] = False

    namespace["CLASS_REACH_PROBES_CSV"] = Path(
        f"{round_prefix}_class_reach_probes.csv"
    )
    namespace["CLASS_REACH_SUMMARY_CSV"] = Path(
        f"{round_prefix}_class_reach_summary.csv"
    )
    namespace["DM_ONLY_SCREEN_CSV"] = Path(
        f"{round_prefix}_dm_only_screen.csv"
    )
    namespace["FULL_STAGE_RESULTS_CSV"] = Path(
        f"{round_prefix}_full_stage_results.csv"
    )
    namespace["COMBINED_RESULTS_CSV"] = Path(
        f"{round_prefix}_trajectory_results.csv"
    )
    namespace["WEIGHTED_ANALYSIS_CSV"] = Path(
        f"{round_prefix}_weighted_analysis_results.csv"
    )
    namespace["WEIGHTED_GLOBAL_SUMMARY_CSV"] = Path(
        f"{round_prefix}_weighted_global_summary.csv"
    )
    namespace["WEIGHTED_CLASSIFICATION_SUMMARY_CSV"] = Path(
        f"{round_prefix}_weighted_classification_summary.csv"
    )
    if "GENERATE_TEST2_PLOTS" in namespace:
        namespace["GENERATE_TEST2_PLOTS"] = False

    return namespace["COMBINED_RESULTS_CSV"]


def _remove_existing_test2p5_round_artifacts() -> None:
    """Delete only Test 2.5 round artifacts for the current parameter point."""
    prefix_path = Path(TEST2P5_OUTPUT_PREFIX)
    parent = prefix_path.parent
    stem = prefix_path.name
    removed = 0
    for path in sorted(parent.glob(f"{stem}_round*")):
        if path.is_file():
            path.unlink()
            removed += 1
    if NEXT_SETTINGS_TXT.exists():
        NEXT_SETTINGS_TXT.unlink()
    progress(
        f"[AUTO TEST 2/2.5] removed {removed} existing refinement-round files"
    )


def _test2p5_sha256(path: Path) -> str:
    path = Path(path)
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _test2p5_refinement_fingerprint() -> dict[str, Any]:
    dependencies = {
        "class_template": Path(CLASS_TEMPLATE_INPUT_CSV),
        "pilot_results": Path(PILOT_TEST2_RESULTS_CSV),
    }
    missing = [str(path) for path in dependencies.values() if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Cannot fingerprint Test 2.5 dependencies: " + ", ".join(missing)
        )
    return {
        "code_version": str(globals().get(
            "TRAJECTORY_REFINEMENT_CODE_VERSION", "unversioned"
        )),
        "dependency_hashes": {
            name: _test2p5_sha256(path) for name, path in dependencies.items()
        },
        "b_tolerance_m": float(BMAX_BRACKET_TOLERANCE_M),
        "speed_tolerance_m_s": float(SPEED_REACHABILITY_TOLERANCE_M_S),
        "speed_b_anchor_count": int(SPEED_REACHABILITY_B_ANCHOR_COUNT),
        "speed_refinement_target": str(SPEED_REFINEMENT_TARGET),
        "fourier_detection_history": "incoming_R_outer_to_outgoing_R_outer",
        "include_outer_R_tails": bool(globals().get("USER_INCLUDE_OUTER_R_TAILS", True)),
        "require_outer_R_tails_complete": bool(globals().get("USER_REQUIRE_OUTER_R_TAILS_COMPLETE", True)),
        "fourier_outer_radius_m": float(globals().get("USER_FOURIER_OUTER_RADIUS_M", 20.0e-3)),
        "max_rounds": int(MAX_REFINEMENT_ROUNDS),
        "r_full_v_b_mode": str(USER_R_FULL_VB_MODE),
        "r_switch_factor": float(USER_R_SWITCH_FACTOR),
    }


def _ensure_test2p5_refinement_fingerprint() -> None:
    manifest_path = Path(RUN_DIRECTORY) / "test2p5_refinement_manifest.json"
    current = _test2p5_refinement_fingerprint()
    previous = None
    if manifest_path.exists():
        try:
            previous = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (OSError, ValueError, json.JSONDecodeError):
            previous = None
    if previous != current:
        progress(
            "[AUTO TEST 2/2.5] upstream inputs or refinement settings changed; "
            "starting a clean adaptive refinement"
        )
        _remove_existing_test2p5_round_artifacts()
        manifest_path.write_text(json.dumps(current, indent=2), encoding="utf-8")


def run_automatic_test2p5_refinement() -> None:
    """Alternate Test 2.5 and Test 2 until b and conditional speed scans finish."""
    if not bool(globals().get("AUTO_RUN_TEST2P5_REFINEMENT", True)):
        main()
        return

    if bool(globals().get("RESTART_TEST2P5_REFINEMENT", False)):
        _remove_existing_test2p5_round_artifacts()
        manifest_path = Path(RUN_DIRECTORY) / "test2p5_refinement_manifest.json"
        if manifest_path.exists():
            manifest_path.unlink()

    _ensure_test2p5_refinement_fingerprint()

    user_round_cap = int(globals().get(
        "AUTO_TEST2P5_MAX_ROUNDS",
        MAX_REFINEMENT_ROUNDS,
    ))
    automatic_round_cap = min(user_round_cap, int(MAX_REFINEMENT_ROUNDS))
    if automatic_round_cap < 1:
        raise ValueError("AUTO_TEST2P5_MAX_ROUNDS must be at least 1")

    rounds_run_this_session = 0

    while True:
        # Analyze the pilot plus every completed refinement round. This either
        # declares completion or writes the next round input.
        main()

        completed_rounds = find_round_result_files()
        completed_numbers = [number for number, _ in completed_rounds]
        next_round_number = (
            max(completed_numbers) + 1 if completed_numbers else 1
        )
        next_input = _test2p5_round_input_path(next_round_number)
        next_result = _test2p5_round_result_path(next_round_number)

        # No new input means Test 2.5 is converged, exhausted, or has no rays.
        if not next_input.exists():
            progress(
                "[AUTO TEST 2/2.5] refinement complete: "
                f"{len(completed_numbers)} completed round(s)"
            )
            return

        if next_round_number > automatic_round_cap:
            raise RuntimeError(
                "Test 2.5 requested round "
                f"{next_round_number}, exceeding the automatic cap "
                f"{automatic_round_cap}. Increase AUTO_TEST2P5_MAX_ROUNDS "
                "or inspect the unconverged rays."
            )

        # A completed result should already have been included by main().
        # Reaching this branch with both files present indicates an unexpected
        # round-discovery problem rather than a reason to silently rerun.
        if next_result.exists():
            raise RuntimeError(
                "Test 2.5 found a pending input whose result already exists: "
                f"{next_result}. Inspect ROUND_RESULTS_GLOB and round filenames."
            )

        progress("")
        progress("=" * 100)
        progress(
            f"[AUTO TEST 2/2.5] running Test 2 refinement round "
            f"{next_round_number:02d} from {next_input}"
        )
        progress("=" * 100)

        expected_result = _configure_embedded_test2_for_round(
            next_input,
            next_round_number,
        )
        TEST2_NAMESPACE["main"]()
        rounds_run_this_session += 1

        if not Path(expected_result).exists():
            raise RuntimeError(
                "Automatic Test 2 finished without creating the expected "
                f"round result: {expected_result}"
            )

        progress(
            f"[AUTO TEST 2/2.5] completed round "
            f"{next_round_number:02d}; rerunning Test 2.5 analysis"
        )


if __name__ == "__main__":
    if bool(globals().get("USER_RUN_TEST2P5", True)):
        run_automatic_test2p5_refinement()
    else:
        progress("[SKIP] Test 2.5 refinement")


# Bowl Necessity Test After Test 2.5 Refinement

The same contingency test is repeated using the most recent Test 2.5 all-results table when available.  This is the more informative result because it includes the targeted $b$ and speed refinement rounds.


In [ ]:
# Prefer the accumulated Test 2.5 trajectory table for the final bowl test.
USER_BOWL_REPORT_INPUT_CSV = (
    Path(ALL_RESULTS_CSV)
    if "ALL_RESULTS_CSV" in globals() and Path(ALL_RESULTS_CSV).exists()
    else None
)
exec(compile(
    BOWL_DETECTABILITY_REPORT_SOURCE,
    "bowl_detectability_report_v1_post_refinement.py",
    "exec",
), globals(), globals())


# Strict Final Conditional Parameter Space

This is the production-quality support used for rates and probabilities. It intentionally requires confirmed finite-measure support and keeps unresolved candidates out of the production probability mass.


In [ ]:
#!/usr/bin/env python3
"""Build the final truncated six-parameter trajectory space.

Run after Test 0.75, Test 1.5, Test 2, and the final Test 2.5 round:

    %run -i build_final_truncated_parameter_space_v1.py

The six trajectory parameters are

    (v_inf, theta, alpha, psi, b, R_far).

The output is conditional rather than one rectangular box.  Each retained
(speed, theta, alpha, psi) cell carries its own exact union of allowed impact-
parameter intervals and its own validated far-launch radius.

The script uses, in order of authority:

1. Test 0.75 / Test 1 speed bounds, when available;
2. Test 1.5 exact physical b annuli after analytical/outward-force filtering;
3. Test 2 class reach-screen status;
4. Test 2.5 ray-level b_max brackets and sampled reaching intervals.

Plots are displayed with ``plt.show()`` and are never written to disk.
"""

from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import maxwell as _final_maxwell


# =============================================================================
# USER SETTINGS
# =============================================================================

BASE_DIR = RUN_DIRECTORY
OUTPUT_PREFIX = FINAL_OUTPUT_PREFIX

# Explicit current-run paths prevent accidental discovery of stale files from
# another (m_dm, eps) parameter point.
TEST0P75_ALLOWED_INTERVALS_CSV: Path | None = Path(
    f"{TEST0P75_OUTPUT_PREFIX}_test1_allowed_intervals.csv"
)
TEST0P75_SPEED_POLICY_CSV: Path | None = Path(
    f"{TEST0P75_OUTPUT_PREFIX}_test1_speed_policy.csv"
)
TEST1P5_TEMPLATE_CSV: Path | None = Path(f"{TEST1P5_OUTPUT_PREFIX}_test2_input.csv")
TEST1P5_CLASS_PLAN_CSV: Path | None = Path(
    f"{TEST1P5_OUTPUT_PREFIX}_trajectory_class_plan.csv"
)
TEST2_CLASS_SUMMARY_CSV: Path | None = Path(
    f"{TEST2_OUTPUT_PREFIX}_class_reach_summary.csv"
)
TEST2_WEIGHTED_RESULTS_CSV: Path | None = Path(
    f"{TEST2_OUTPUT_PREFIX}_weighted_analysis_results.csv"
)
TEST2P5_RAY_SUMMARY_CSV: Path | None = Path(
    f"{TEST2P5_OUTPUT_PREFIX}_ray_bmax_summary.csv"
)
TEST2P5_REACHING_INTERVALS_CSV: Path | None = Path(
    f"{TEST2P5_OUTPUT_PREFIX}_sampled_reaching_intervals.csv"
)
TEST2P5_ALL_RESULTS_CSV: Path | None = Path(
    f"{TEST2P5_OUTPUT_PREFIX}_all_selected_ray_results.csv"
)
TEST2P5_SPEED_REACHABILITY_CSV: Path | None = Path(
    f"{TEST2P5_OUTPUT_PREFIX}_speed_reachability_summary.csv"
)

# Tables are saved because they are the actual final parameter-space product.
# Diagnostic plots are displayed only and are never saved.
SAVE_TABLES = True
SHOW_PLOTS = True
PRINT_MAX_ROWS = 40

# Conservative inclusion policy.
REQUIRE_TEST2_ACCEPTED_CLASS = True
INCLUDE_UNRESOLVED_TEST2_CLASSES = False
REQUIRE_AT_LEAST_ONE_TEST2P5_REACH = True
INCLUDE_UNCONVERGED_MONOTONIC_RAYS = True
INCLUDE_NONMONOTONIC_SAMPLED_INTERVALS = True
INCLUDE_UNREFINED_ACCEPTED_RAYS = False

# For monotonic rays, the final sampler uses the largest confirmed reaching b.
# The estimate and first resolved miss are retained as uncertainty columns but
# are not used to enlarge the conservative sampling domain.
MONOTONIC_B_BOUND_MODE = "confirmed_reach"  # confirmed_reach only recommended

# If a nonmonotonic sampled reaching interval contains only one b point, it has
# zero cross-sectional area and is excluded from the sampler by default.
ALLOW_ZERO_WIDTH_B_INTERVALS = False

# A ray is a fixed (speed_label, theta_index, alpha_index, psi).
PSI_KEY_DECIMALS = 12
ANGLE_MATCH_ATOL_RAD = 1.0e-10

# Sampling demonstration. Set to a positive number to print example samples.
N_EXAMPLE_SAMPLES = 0
SAMPLER_RANDOM_SEED = 12345
SAMPLER_SPEED_MODE = "representative"  # representative or uniform_cell

# The b support comes from geometric reachability. When this target is
# detectability, the final conditional (v,b) table keeps only slabs with a
# confirmed above-threshold Fourier-energy speed anchor.
SPEED_REFINEMENT_TARGET = str(
    globals().get("USER_SPEED_REFINEMENT_TARGET", "detectability")
).strip().lower()


# =============================================================================
# OUTPUT FILES
# =============================================================================

FINAL_RAY_TABLE_CSV = Path(f"{OUTPUT_PREFIX}_ray_cells.csv")
FINAL_INTERVAL_TABLE_CSV = Path(f"{OUTPUT_PREFIX}_conditional_intervals.csv")
GLOBAL_BOUNDS_CSV = Path(f"{OUTPUT_PREFIX}_global_bounds.csv")
EXCLUDED_RAYS_CSV = Path(f"{OUTPUT_PREFIX}_excluded_rays.csv")
COVERAGE_SUMMARY_CSV = Path(f"{OUTPUT_PREFIX}_coverage_summary.csv")
SAMPLER_JSON = Path(f"{OUTPUT_PREFIX}_sampler_map.json")


# =============================================================================
# FILE DISCOVERY
# =============================================================================

FILE_PATTERNS: dict[str, tuple[str, ...]] = {
    "test0p75_allowed": (
        "test0p75*_test1_allowed_intervals.csv",
        "test0p75*_allowed_intervals.csv",
        "*test0p75*allowed*interval*.csv",
    ),
    "test0p75_speed": (
        "test0p75*_test1_speed_policy.csv",
        "test0p75*_speed_policy.csv",
        "*test0p75*speed*policy*.csv",
    ),
    "test1p5_template": (
        "test1p5_exact_annuli_lower_b_v12_test2_input.csv",
        "test1p5*_test2_input.csv",
    ),
    "test1p5_plan": (
        "test1p5_exact_annuli_lower_b_v12_trajectory_class_plan.csv",
        "test1p5*_trajectory_class_plan.csv",
    ),
    "test2_class": (
        "test2_exact_annuli_lower_b_reach_v12_class_reach_summary.csv",
        "test2*_class_reach_summary.csv",
    ),
    "test2_weighted": (
        "test2_exact_annuli_lower_b_reach_v12_weighted_analysis_results.csv",
        "test2*_weighted_analysis_results.csv",
    ),
    "test2p5_ray": (
        "test2p5_exact_annuli_refinement_v12_ray_bmax_summary.csv",
        "test2p5*_ray_bmax_summary.csv",
    ),
    "test2p5_intervals": (
        "test2p5_exact_annuli_refinement_v12_sampled_reaching_intervals.csv",
        "test2p5*_sampled_reaching_intervals.csv",
    ),
    "test2p5_all": (
        "test2p5_exact_annuli_refinement_v12_all_selected_ray_results.csv",
        "test2p5*_all_selected_ray_results.csv",
    ),
    "test2p5_speed": (
        "test2p5_exact_annuli_refinement_v12_speed_reachability_summary.csv",
        "test2p5*_speed_reachability_summary.csv",
    ),
}


def progress(message: str = "") -> None:
    print(message, flush=True)


def discover_file(
    explicit: Path | None,
    patterns: Sequence[str],
    *,
    required: bool,
    label: str,
) -> Path | None:
    """Resolve an input path without prepending BASE_DIR twice.

    Prefix globals may already include RUN_DIRECTORY. For a relative explicit
    path, try it exactly as provided first, then try BASE_DIR / path. If neither
    exists, use pattern discovery under BASE_DIR. Missing optional files return
    None rather than raising.
    """
    base_dir = Path(BASE_DIR)
    attempted: list[Path] = []

    if explicit is not None:
        explicit_path = Path(explicit)

        if explicit_path.is_absolute():
            explicit_candidates = [explicit_path]
        else:
            explicit_candidates = [
                explicit_path,
                base_dir / explicit_path,
            ]

        seen: set[str] = set()
        for candidate in explicit_candidates:
            key = str(candidate)
            if key in seen:
                continue
            seen.add(key)
            attempted.append(candidate)

            if candidate.is_file() and candidate.stat().st_size > 0:
                return candidate

    candidates: list[Path] = []
    for pattern in patterns:
        candidates.extend(base_dir.glob(pattern))

    # OUTPUT_PREFIX may itself include directories. Compare only against the
    # filename component when excluding this script's own outputs.
    output_prefix_name = Path(str(OUTPUT_PREFIX)).name

    candidates = sorted(
        {
            path.resolve()
            for path in candidates
            if path.is_file()
            and path.stat().st_size > 0
            and not path.name.startswith(output_prefix_name)
        },
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        return candidates[0]

    if required:
        attempted_text = ", ".join(str(path) for path in attempted)
        pattern_text = ", ".join(patterns)
        details: list[str] = []
        if attempted_text:
            details.append(f"explicit candidates: {attempted_text}")
        details.append(f"patterns under {base_dir}: {pattern_text}")
        raise FileNotFoundError(
            f"Could not find {label}. Tried " + "; ".join(details)
        )

    return None


@dataclass(frozen=True)
class InputFiles:
    test0p75_allowed: Path | None
    test0p75_speed: Path | None
    test1p5_template: Path
    test1p5_plan: Path | None
    test2_class: Path | None
    test2_weighted: Path | None
    test2p5_ray: Path
    test2p5_intervals: Path | None
    test2p5_all: Path | None
    test2p5_speed: Path | None


def resolve_inputs() -> InputFiles:
    return InputFiles(
        test0p75_allowed=discover_file(
            TEST0P75_ALLOWED_INTERVALS_CSV,
            FILE_PATTERNS["test0p75_allowed"],
            required=False,
            label="Test 0.75 allowed-interval",
        ),
        test0p75_speed=discover_file(
            TEST0P75_SPEED_POLICY_CSV,
            FILE_PATTERNS["test0p75_speed"],
            required=False,
            label="Test 0.75 speed-policy",
        ),
        test1p5_template=discover_file(
            TEST1P5_TEMPLATE_CSV,
            FILE_PATTERNS["test1p5_template"],
            required=True,
            label="Test 1.5 template",
        ),
        test1p5_plan=discover_file(
            TEST1P5_CLASS_PLAN_CSV,
            FILE_PATTERNS["test1p5_plan"],
            required=False,
            label="Test 1.5 class-plan",
        ),
        test2_class=discover_file(
            TEST2_CLASS_SUMMARY_CSV,
            FILE_PATTERNS["test2_class"],
            required=False,
            label="Test 2 class-summary",
        ),
        test2_weighted=discover_file(
            TEST2_WEIGHTED_RESULTS_CSV,
            FILE_PATTERNS["test2_weighted"],
            required=False,
            label="Test 2 weighted-results",
        ),
        test2p5_ray=discover_file(
            TEST2P5_RAY_SUMMARY_CSV,
            FILE_PATTERNS["test2p5_ray"],
            required=True,
            label="Test 2.5 ray-summary",
        ),
        test2p5_intervals=discover_file(
            TEST2P5_REACHING_INTERVALS_CSV,
            FILE_PATTERNS["test2p5_intervals"],
            required=False,
            label="Test 2.5 reaching-interval",
        ),
        test2p5_all=discover_file(
            TEST2P5_ALL_RESULTS_CSV,
            FILE_PATTERNS["test2p5_all"],
            required=False,
            label="Test 2.5 all-results",
        ),
        test2p5_speed=discover_file(
            TEST2P5_SPEED_REACHABILITY_CSV,
            FILE_PATTERNS["test2p5_speed"],
            required=False,
            label="Test 2.5 speed-reachability summary",
        ),
    )


def load_csv(path: Path | None) -> pd.DataFrame:
    if path is None:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, low_memory=False)
    except pd.errors.EmptyDataError:
        progress(f"Input CSV is completely empty and has no header: {path}. Treating it as an empty table.")
        return pd.DataFrame()


# =============================================================================
# GENERIC HELPERS
# =============================================================================


def numeric_series(df: pd.DataFrame, column: str, default=np.nan) -> pd.Series:
    if column not in df.columns:
        return pd.Series(default, index=df.index, dtype=float)
    return pd.to_numeric(df[column], errors="coerce")


def bool_series(df: pd.DataFrame, column: str, default=False) -> pd.Series:
    if column not in df.columns:
        return pd.Series(default, index=df.index, dtype=bool)
    values = df[column]
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(default).astype(bool)
    return values.astype(str).str.strip().str.lower().isin(
        {"true", "1", "yes", "y"}
    )


def first_existing_column(df: pd.DataFrame, names: Sequence[str]) -> str | None:
    for name in names:
        if name in df.columns:
            return name
    return None


def finite_or_nan(value: Any) -> float:
    try:
        result = float(value)
    except (TypeError, ValueError):
        return np.nan
    return result if np.isfinite(result) else np.nan


def safe_int(value: Any, default: int = 0) -> int:
    number = finite_or_nan(value)
    return int(number) if np.isfinite(number) else int(default)


def as_bool(value: Any, default: bool = False) -> bool:
    if value is None:
        return bool(default)
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        if not np.isfinite(value):
            return bool(default)
        return bool(value)
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n", "", "nan", "none"}:
        return False
    return bool(default)


def json_number(value: Any) -> float | None:
    number = finite_or_nan(value)
    return float(number) if np.isfinite(number) else None


def merge_intervals(
    intervals: Iterable[tuple[float, float]],
    *,
    allow_points: bool = False,
) -> list[tuple[float, float]]:
    clean: list[tuple[float, float]] = []
    for lower, upper in intervals:
        lower = finite_or_nan(lower)
        upper = finite_or_nan(upper)
        if not (np.isfinite(lower) and np.isfinite(upper) and lower >= 0.0):
            continue
        if upper < lower:
            lower, upper = upper, lower
        if upper == lower and not allow_points:
            continue
        clean.append((lower, upper))
    clean.sort()
    if not clean:
        return []
    merged: list[list[float]] = [[clean[0][0], clean[0][1]]]
    for lower, upper in clean[1:]:
        previous = merged[-1]
        tolerance = max(1.0e-15, 1.0e-12 * max(previous[1], upper, 1.0e-12))
        if lower <= previous[1] + tolerance:
            previous[1] = max(previous[1], upper)
        else:
            merged.append([lower, upper])
    return [(float(lower), float(upper)) for lower, upper in merged]


def parse_intervals_json(value: Any) -> list[tuple[float, float]]:
    if value is None:
        return []
    try:
        raw = json.loads(str(value))
    except (TypeError, ValueError, json.JSONDecodeError):
        return []
    if not isinstance(raw, list):
        return []
    intervals: list[tuple[float, float]] = []
    for item in raw:
        if isinstance(item, (list, tuple)) and len(item) == 2:
            intervals.append((finite_or_nan(item[0]), finite_or_nan(item[1])))
    return merge_intervals(intervals, allow_points=ALLOW_ZERO_WIDTH_B_INTERVALS)


def intervals_json(intervals: Iterable[tuple[float, float]]) -> str:
    return json.dumps(
        [[float(lower), float(upper)] for lower, upper in intervals],
        separators=(",", ":"),
    )


def intersect_interval_sets(
    left: Iterable[tuple[float, float]],
    right: Iterable[tuple[float, float]],
    *,
    allow_points: bool = False,
) -> list[tuple[float, float]]:
    a = merge_intervals(left, allow_points=allow_points)
    b = merge_intervals(right, allow_points=allow_points)
    intersections: list[tuple[float, float]] = []
    for lower_a, upper_a in a:
        for lower_b, upper_b in b:
            lower = max(lower_a, lower_b)
            upper = min(upper_a, upper_b)
            if upper > lower or (allow_points and upper == lower):
                intersections.append((lower, upper))
    return merge_intervals(intersections, allow_points=allow_points)


def clip_intervals_at_upper(
    intervals: Iterable[tuple[float, float]],
    upper_cap: float,
    *,
    allow_points: bool = False,
) -> list[tuple[float, float]]:
    if not np.isfinite(upper_cap):
        return []
    result: list[tuple[float, float]] = []
    for lower, upper in merge_intervals(intervals, allow_points=allow_points):
        clipped_upper = min(upper, upper_cap)
        if clipped_upper > lower or (allow_points and clipped_upper == lower):
            result.append((lower, clipped_upper))
    return merge_intervals(result, allow_points=allow_points)


def interval_area2(intervals: Iterable[tuple[float, float]]) -> float:
    return float(
        sum(max(upper * upper - lower * lower, 0.0) for lower, upper in intervals)
    )


def circular_distance(a: float, b: float) -> float:
    return abs((a - b + math.pi) % (2.0 * math.pi) - math.pi)


def nearest_key(value: float, keys: Sequence[float]) -> float:
    if not keys:
        return value
    return min(keys, key=lambda candidate: circular_distance(value, candidate))


def ray_key_frame(df: pd.DataFrame) -> pd.Series:
    speed = df["speed_label"].astype(str)
    theta = numeric_series(df, "theta_index", -1).fillna(-1).astype(int)
    alpha = numeric_series(df, "alpha_index", -1).fillna(-1).astype(int)
    psi = np.round(numeric_series(df, "psi_rad", 0.0), PSI_KEY_DECIMALS)
    return pd.Series(
        [
            f"{s}|th{t:03d}|al{a:03d}|psi{p:.{PSI_KEY_DECIMALS}f}"
            for s, t, a, p in zip(speed, theta, alpha, psi)
        ],
        index=df.index,
        dtype=str,
    )


def ensure_ray_key(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    if "ray_key" not in output.columns:
        required = {"speed_label", "theta_index", "alpha_index", "psi_rad"}
        if not required.issubset(output.columns):
            return output
        output["ray_key"] = ray_key_frame(output)
    output["ray_key"] = output["ray_key"].astype(str)
    return output


def actual_far_radius(df: pd.DataFrame) -> pd.Series:
    direct = first_existing_column(
        df,
        ("R_far_m", "r_far_m", "R_start_m", "r_start_m", "launch_radius_m"),
    )
    if direct is not None:
        values = numeric_series(df, direct)
    else:
        coordinates = ("x_far_m", "y_far_m", "z_far_m")
        if not all(column in df.columns for column in coordinates):
            return pd.Series(np.nan, index=df.index, dtype=float)
        xyz = df[list(coordinates)].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        values = pd.Series(np.linalg.norm(xyz, axis=1), index=df.index, dtype=float)
    return values


def physical_annuli_for_group(group: pd.DataFrame) -> list[tuple[float, float]]:
    for column in (
        "physical_b_annuli_json",
        "class_b_annuli_json",
        "class_reach_annuli_json",
    ):
        if column in group.columns:
            for value in group[column].dropna():
                parsed = parse_intervals_json(value)
                if parsed:
                    return parsed
    if {"b_lower_m", "b_upper_m"}.issubset(group.columns):
        return merge_intervals(
            zip(
                numeric_series(group, "b_lower_m").to_numpy(float),
                numeric_series(group, "b_upper_m").to_numpy(float),
            ),
            allow_points=False,
        )
    lower_col = first_existing_column(
        group,
        ("physical_b_support_lower_m", "class_b_support_lower_m"),
    )
    upper_col = first_existing_column(
        group,
        ("physical_b_support_upper_m", "class_b_support_upper_m"),
    )
    if lower_col and upper_col:
        lower = numeric_series(group, lower_col).dropna()
        upper = numeric_series(group, upper_col).dropna()
        if len(lower) and len(upper):
            return merge_intervals([(float(lower.iloc[0]), float(upper.iloc[0]))])
    return []


# =============================================================================
# CELL BOUNDS
# =============================================================================


def midpoint_bounds_linear(
    values: Sequence[float],
    *,
    domain_lower: float | None = None,
    domain_upper: float | None = None,
) -> dict[float, tuple[float, float]]:
    unique = sorted({float(value) for value in values if np.isfinite(value)})
    if not unique:
        return {}
    if len(unique) == 1:
        lower = unique[0] if domain_lower is None else domain_lower
        upper = unique[0] if domain_upper is None else domain_upper
        return {unique[0]: (float(lower), float(upper))}
    edges = [0.5 * (a + b) for a, b in zip(unique[:-1], unique[1:])]
    lower_edge = (
        float(domain_lower)
        if domain_lower is not None
        else max(0.0, unique[0] - 0.5 * (unique[1] - unique[0]))
    )
    upper_edge = (
        float(domain_upper)
        if domain_upper is not None
        else unique[-1] + 0.5 * (unique[-1] - unique[-2])
    )
    bounds: dict[float, tuple[float, float]] = {}
    for index, value in enumerate(unique):
        lower = lower_edge if index == 0 else edges[index - 1]
        upper = upper_edge if index == len(unique) - 1 else edges[index]
        bounds[value] = (float(lower), float(upper))
    return bounds


def circular_cell_bounds(values: Sequence[float]) -> dict[float, tuple[float, float, bool]]:
    unique = sorted(
        {float(value) % (2.0 * math.pi) for value in values if np.isfinite(value)}
    )
    if not unique:
        return {}
    if len(unique) == 1:
        return {unique[0]: (0.0, 2.0 * math.pi, False)}
    bounds: dict[float, tuple[float, float, bool]] = {}
    n = len(unique)
    for index, value in enumerate(unique):
        previous = unique[index - 1] if index > 0 else unique[-1] - 2.0 * math.pi
        following = unique[index + 1] if index < n - 1 else unique[0] + 2.0 * math.pi
        lower = 0.5 * (previous + value)
        upper = 0.5 * (value + following)
        lower_mod = lower % (2.0 * math.pi)
        upper_mod = upper % (2.0 * math.pi)
        wraps = lower_mod > upper_mod
        bounds[value] = (float(lower_mod), float(upper_mod), bool(wraps))
    return bounds


def speed_branch(label: str) -> str:
    label = str(label)
    match = re.match(r"(.+?)(?:_q[-+]?\d|_quantile|_vb_q)", label)
    return match.group(1) if match else label.split("|", 1)[0]


def candidate_speed_bound_columns(df: pd.DataFrame) -> tuple[str | None, str | None]:
    pairs = (
        ("speed_lower_m_s", "speed_upper_m_s"),
        ("v_lower_m_s", "v_upper_m_s"),
        ("v_inf_lower_m_s", "v_inf_upper_m_s"),
        ("speed_bin_lower_m_s", "speed_bin_upper_m_s"),
        ("v_bin_lower_m_s", "v_bin_upper_m_s"),
        ("interval_lower_m_s", "interval_upper_m_s"),
        ("v_min_m_s", "v_max_m_s"),
    )
    for lower, upper in pairs:
        if lower in df.columns and upper in df.columns:
            return lower, upper
    lower_candidates = [
        column
        for column in df.columns
        if "lower" in column.lower()
        and ("speed" in column.lower() or column.lower().startswith("v_"))
        and "b_" not in column.lower()
    ]
    upper_candidates = [
        column
        for column in df.columns
        if "upper" in column.lower()
        and ("speed" in column.lower() or column.lower().startswith("v_"))
        and "b_" not in column.lower()
    ]
    return (
        lower_candidates[0] if lower_candidates else None,
        upper_candidates[0] if upper_candidates else None,
    )


def infer_speed_cells(
    template: pd.DataFrame,
    speed_policy: pd.DataFrame,
    allowed_intervals: pd.DataFrame,
) -> pd.DataFrame:
    representatives = (
        template[["speed_label", "v_inf_m_s"]]
        .copy()
        .assign(v_inf_m_s=lambda frame: pd.to_numeric(frame["v_inf_m_s"], errors="coerce"))
        .dropna()
        .groupby("speed_label", as_index=False)["v_inf_m_s"]
        .median()
    )
    representatives["speed_branch"] = representatives["speed_label"].map(speed_branch)
    representatives["speed_lower_m_s"] = np.nan
    representatives["speed_upper_m_s"] = np.nan
    representatives["speed_bounds_source"] = "derived_midpoint"

    # Prefer explicit per-label bounds from the template or speed-policy table.
    for source_name, source in (("template", template), ("test0p75_speed_policy", speed_policy)):
        if len(source) == 0 or "speed_label" not in source.columns:
            continue
        lower_col, upper_col = candidate_speed_bound_columns(source)
        if lower_col is None or upper_col is None:
            continue
        mapping = (
            source[["speed_label", lower_col, upper_col]]
            .copy()
            .assign(
                **{
                    lower_col: lambda frame: pd.to_numeric(frame[lower_col], errors="coerce"),
                    upper_col: lambda frame: pd.to_numeric(frame[upper_col], errors="coerce"),
                }
            )
            .dropna()
            .drop_duplicates("speed_label", keep="last")
        )
        lower_map = mapping.set_index("speed_label")[lower_col]
        upper_map = mapping.set_index("speed_label")[upper_col]
        matched = representatives["speed_label"].isin(lower_map.index)
        representatives.loc[matched, "speed_lower_m_s"] = representatives.loc[
            matched, "speed_label"
        ].map(lower_map)
        representatives.loc[matched, "speed_upper_m_s"] = representatives.loc[
            matched, "speed_label"
        ].map(upper_map)
        representatives.loc[matched, "speed_bounds_source"] = source_name

    # Extract branch/global endpoints from Test 0.75 allowed intervals when possible.
    allowed_branch_bounds: dict[str, tuple[float, float]] = {}
    allowed_global: tuple[float, float] | None = None
    if len(allowed_intervals):
        lower_col, upper_col = candidate_speed_bound_columns(allowed_intervals)
        if lower_col and upper_col:
            lower_values = numeric_series(allowed_intervals, lower_col)
            upper_values = numeric_series(allowed_intervals, upper_col)
            valid = np.isfinite(lower_values) & np.isfinite(upper_values) & (upper_values > lower_values)
            if valid.any():
                allowed_global = (
                    float(lower_values[valid].min()),
                    float(upper_values[valid].max()),
                )
                branch_col = first_existing_column(
                    allowed_intervals,
                    ("speed_branch", "branch", "regime", "interval_label", "speed_label"),
                )
                if branch_col:
                    for branch_name, group in allowed_intervals.loc[valid].groupby(branch_col):
                        branch_key = speed_branch(str(branch_name))
                        allowed_branch_bounds[branch_key] = (
                            float(numeric_series(group, lower_col).min()),
                            float(numeric_series(group, upper_col).max()),
                        )

    # Fill missing bounds with midpoint Voronoi cells separately in each branch.
    for branch_name, group in representatives.groupby("speed_branch", sort=False):
        ordered = group.sort_values("v_inf_m_s")
        indices = ordered.index.to_list()
        values = ordered["v_inf_m_s"].to_numpy(float)
        if branch_name in allowed_branch_bounds:
            branch_lower, branch_upper = allowed_branch_bounds[branch_name]
        elif allowed_global is not None and len(representatives["speed_branch"].unique()) == 1:
            branch_lower, branch_upper = allowed_global
        else:
            branch_lower = None
            branch_upper = None
        bounds = midpoint_bounds_linear(
            values,
            domain_lower=branch_lower,
            domain_upper=branch_upper,
        )
        for index, value in zip(indices, values):
            if not np.isfinite(representatives.at[index, "speed_lower_m_s"]):
                representatives.at[index, "speed_lower_m_s"] = bounds[value][0]
            if not np.isfinite(representatives.at[index, "speed_upper_m_s"]):
                representatives.at[index, "speed_upper_m_s"] = bounds[value][1]

    representatives["speed_lower_m_s"] = representatives["speed_lower_m_s"].clip(lower=0.0)
    representatives["speed_upper_m_s"] = np.maximum(
        representatives["speed_upper_m_s"], representatives["speed_lower_m_s"]
    )
    return representatives


# =============================================================================
# STATUS AND WEIGHT TABLES
# =============================================================================


ACCEPTED_TEST2_STATUSES = {
    "accepted_has_switch_entry",
    "screen_disabled",
}
UNRESOLVED_TEST2_STATUSES = {
    "unresolved_class_reach_screen",
}
REJECTED_TEST2_TOKENS = (
    "reject",
    "no_in_range_sample_reaches_switch",
    "no_sampled_b_reaches_switch",
)


def class_status_by_ray(
    template: pd.DataFrame,
    class_summary: pd.DataFrame,
    weighted_results: pd.DataFrame,
) -> pd.DataFrame:
    records: list[pd.DataFrame] = []
    for source_name, source in (
        ("class_summary", class_summary),
        ("weighted_results", weighted_results),
        ("template", template),
    ):
        if len(source) == 0:
            continue
        data = ensure_ray_key(source)
        if "ray_key" not in data.columns:
            continue
        status_col = first_existing_column(
            data,
            (
                "trajectory_class_reach_status",
                "class_reach_status",
                "test2_class_status",
            ),
        )
        if status_col is None:
            continue
        part = data[["ray_key", status_col]].copy()
        part.columns = ["ray_key", "test2_class_status"]
        part["test2_class_status_source"] = source_name
        part = part.dropna(subset=["test2_class_status"]).drop_duplicates(
            "ray_key", keep="last"
        )
        records.append(part)
    if not records:
        return pd.DataFrame(columns=["ray_key", "test2_class_status", "test2_class_status_source"])
    combined = pd.concat(records, ignore_index=True)
    priority = {"class_summary": 0, "weighted_results": 1, "template": 2}
    combined["_priority"] = combined["test2_class_status_source"].map(priority).fillna(99)
    combined = combined.sort_values("_priority").drop_duplicates("ray_key", keep="first")
    return combined.drop(columns="_priority")


def classify_test2_status(status: Any) -> str:
    text = str(status).strip()
    lower = text.lower()
    if text in ACCEPTED_TEST2_STATUSES:
        return "accepted"
    if text in UNRESOLVED_TEST2_STATUSES or "unresolved" in lower:
        return "unresolved"
    if any(token in lower for token in REJECTED_TEST2_TOKENS):
        return "rejected"
    if not text or lower == "nan":
        return "missing"
    return "other"


def ray_weights(template: pd.DataFrame, weighted_results: pd.DataFrame) -> pd.DataFrame:
    candidates = (
        "test2_analysis_weight_conditional",
        "population_weight_conditional",
        "impact_weight",
    )
    for source_name, source in (("test2_weighted", weighted_results), ("test1p5_template", template)):
        if len(source) == 0:
            continue
        data = ensure_ray_key(source)
        if "ray_key" not in data.columns:
            continue
        column = first_existing_column(data, candidates)
        if column is None:
            continue
        values = numeric_series(data, column, 0.0).fillna(0.0).clip(lower=0.0)
        data = data.assign(_ray_weight=values)
        grouped = data.groupby("ray_key", as_index=False)["_ray_weight"].sum()
        if (grouped["_ray_weight"] > 0.0).any():
            grouped = grouped.rename(columns={"_ray_weight": "ray_weight_raw"})
            grouped["ray_weight_source"] = f"{source_name}:{column}"
            return grouped
    keys = ensure_ray_key(template)["ray_key"].drop_duplicates()
    output = pd.DataFrame({"ray_key": keys, "ray_weight_raw": 1.0})
    output["ray_weight_source"] = "equal_ray_fallback"
    return output


# =============================================================================
# FINAL SUPPORT CONSTRUCTION
# =============================================================================


def reaching_intervals_for_ray(
    ray_key: str,
    interval_table: pd.DataFrame,
) -> list[tuple[float, float]]:
    if len(interval_table) == 0 or "ray_key" not in interval_table.columns:
        return []
    group = interval_table.loc[interval_table["ray_key"].astype(str).eq(str(ray_key))]
    if len(group) == 0:
        return []
    lower_col = first_existing_column(group, ("b_start_m", "interval_lower_m", "b_lower_m"))
    upper_col = first_existing_column(group, ("b_stop_m", "interval_upper_m", "b_upper_m"))
    if lower_col is None or upper_col is None:
        return []
    return merge_intervals(
        zip(
            numeric_series(group, lower_col).to_numpy(float),
            numeric_series(group, upper_col).to_numpy(float),
        ),
        allow_points=ALLOW_ZERO_WIDTH_B_INTERVALS,
    )


def final_b_support_for_ray(
    ray: pd.Series,
    physical_annuli: list[tuple[float, float]],
    sampled_intervals: list[tuple[float, float]],
) -> tuple[list[tuple[float, float]], str, str]:
    n_reach = int(finite_or_nan(ray.get("n_reach", 0.0)) or 0)
    monotonic = as_bool(ray.get("single_cutoff_monotonic", True), True)
    converged = as_bool(ray.get("converged", False), False)
    b_lower = finite_or_nan(ray.get("bmax_lower_m"))

    if REQUIRE_AT_LEAST_ONE_TEST2P5_REACH and n_reach <= 0:
        return [], "excluded", "test2p5_no_reaching_sample"

    if monotonic:
        if not np.isfinite(b_lower):
            return [], "excluded", "monotonic_missing_confirmed_reach_bound"
        if not converged and not INCLUDE_UNCONVERGED_MONOTONIC_RAYS:
            return [], "excluded", "monotonic_not_converged"
        support = clip_intervals_at_upper(
            physical_annuli,
            b_lower,
            allow_points=ALLOW_ZERO_WIDTH_B_INTERVALS,
        )
        if not support:
            return [], "excluded", "confirmed_reach_bound_below_physical_support"
        source = "monotonic_confirmed_reach_cap"
        quality = "converged" if converged else "confirmed_reach_unconverged_bracket"
        return support, source, quality

    if not INCLUDE_NONMONOTONIC_SAMPLED_INTERVALS:
        return [], "excluded", "nonmonotonic_disabled"
    support = intersect_interval_sets(
        physical_annuli,
        sampled_intervals,
        allow_points=ALLOW_ZERO_WIDTH_B_INTERVALS,
    )
    if not support:
        return [], "excluded", "nonmonotonic_no_positive_area_sampled_interval"
    return support, "nonmonotonic_sampled_reaching_intervals", "sampled_contiguous_only"


def make_angular_bound_maps(template: pd.DataFrame) -> tuple[
    dict[float, tuple[float, float]],
    dict[float, tuple[float, float, bool]],
    dict[float, tuple[float, float, bool]],
]:
    theta_values = numeric_series(template, "theta_rad").dropna().to_numpy(float)
    alpha_values = np.mod(
        numeric_series(template, "alpha_rad").dropna().to_numpy(float),
        2.0 * math.pi,
    )
    psi_values = np.mod(
        numeric_series(template, "psi_rad").dropna().to_numpy(float),
        2.0 * math.pi,
    )
    return (
        midpoint_bounds_linear(theta_values, domain_lower=0.0, domain_upper=math.pi),
        circular_cell_bounds(alpha_values),
        circular_cell_bounds(psi_values),
    )


def match_linear_bound(
    value: float,
    mapping: dict[float, tuple[float, float]],
) -> tuple[float, float]:
    if not mapping:
        return value, value
    key = min(mapping, key=lambda candidate: abs(candidate - value))
    return mapping[key]


def match_circular_bound(
    value: float,
    mapping: dict[float, tuple[float, float, bool]],
) -> tuple[float, float, bool]:
    if not mapping:
        return value, value, False
    key = nearest_key(value % (2.0 * math.pi), list(mapping))
    return mapping[key]


def _maxwell_speed_probability(lower_m_s: float, upper_m_s: float) -> float:
    lower = max(float(lower_m_s), 0.0)
    upper = max(float(upper_m_s), lower)
    if upper <= lower:
        return 0.0
    scale = math.sqrt(1.380649e-23 * float(USER_T_DM_K) / float(USER_M_DM_KG))
    return float(
        _final_maxwell.cdf(upper, scale=scale)
        - _final_maxwell.cdf(lower, scale=scale)
    )


def _conditional_v_b_intervals(
    b_intervals: list[tuple[float, float]],
    speed_records: pd.DataFrame,
    *,
    parent_v_lower: float,
    parent_v_upper: float,
    representative_v: float,
) -> list[dict[str, Any]]:
    """Build conditional speed-b support from confirmed target anchors.

    Geometric reachability retains the previous inclusive interpolation. For
    detectability, no support is extrapolated beyond the largest confirmed
    detectable b anchor. Consecutive successful anchors define finite-area b
    slabs, and each slab uses the higher-b anchor's confirmed speed threshold.
    """
    records = speed_records.copy()
    if len(records):
        records["_b"] = numeric_series(records, "speed_anchor_b_m")
        records["_v"] = numeric_series(
            records, "speed_lower_confirmed_target_m_s"
            if "speed_lower_confirmed_target_m_s" in records.columns
            else "speed_lower_confirmed_reach_m_s",
        )
        records = records.loc[np.isfinite(records["_b"])]
        records = records.sort_values("_b", kind="mergesort")
        records["_success"] = np.isfinite(records["_v"])
        if records["_success"].any():
            successful = records.loc[records["_success"]].copy()
            successful["_v_conservative"] = np.maximum.accumulate(
                successful["_v"].to_numpy(float)
            )
        else:
            successful = records.head(0).copy()
    else:
        successful = records

    output: list[dict[str, Any]] = []
    for base_lower, base_upper in b_intervals:
        if not base_upper > base_lower:
            continue

        if len(records) == 0:
            if SPEED_REFINEMENT_TARGET == "detectability":
                continue
            output.append({
                "b_lower_m": float(base_lower),
                "b_upper_m": float(base_upper),
                "v_lower_m_s": max(float(parent_v_lower), float(representative_v)),
                "v_upper_m_s": max(float(parent_v_upper), float(representative_v)),
                "speed_source": "confirmed_reaching_representative_fallback",
                "speed_anchor_upper_b_m": np.nan,
                "speed_anchor_count_used": 0,
            })
            continue

        if SPEED_REFINEMENT_TARGET == "detectability":
            successful_inside = successful.loc[
                (successful["_b"] >= base_lower - 1.0e-15)
                & (successful["_b"] <= base_upper + 1.0e-15)
            ].copy()
            if len(successful_inside) < 2:
                # A single successful point has zero confirmed b^2 area.
                continue
            successful_inside = successful_inside.sort_values("_b", kind="mergesort")
            anchor_b = successful_inside["_b"].to_numpy(float)
            anchor_v = successful_inside["_v"].to_numpy(float)
            # Only consecutive successful anchors establish an interval. A gap
            # containing a failed anchor must not be bridged.
            all_b = records["_b"].to_numpy(float)
            all_success = records["_success"].to_numpy(bool)
            for i in range(1, len(anchor_b)):
                lower = max(float(base_lower), float(anchor_b[i - 1]))
                upper = min(float(base_upper), float(anchor_b[i]))
                if not upper > lower:
                    continue
                between = (
                    (all_b > lower + 1.0e-15)
                    & (all_b < upper - 1.0e-15)
                )
                if np.any(between & ~all_success):
                    continue
                speed_lower = max(
                    float(parent_v_lower),
                    float(anchor_v[i]),
                )
                speed_upper = max(float(parent_v_upper), speed_lower)
                output.append({
                    "b_lower_m": lower,
                    "b_upper_m": upper,
                    "v_lower_m_s": speed_lower,
                    "v_upper_m_s": speed_upper,
                    "speed_source": "test2p5_confirmed_fourier_detectability_anchors",
                    "speed_anchor_upper_b_m": float(anchor_b[i]),
                    "speed_anchor_count_used": i + 1,
                })
            continue

        anchors_inside = records.loc[
            (records["_b"] > base_lower + 1.0e-15)
            & (records["_b"] < base_upper - 1.0e-15)
        ]["_b"].to_numpy(float)
        boundaries = np.unique(np.concatenate([
            np.array([base_lower], dtype=float),
            anchors_inside,
            np.array([base_upper], dtype=float),
        ]))
        for lower, upper in zip(boundaries[:-1], boundaries[1:]):
            tolerance = max(1.0e-15, 1.0e-9 * max(lower, 1.0e-6))
            eligible = successful.loc[
                successful["_b"] <= lower + tolerance
            ]
            if len(eligible) == 0:
                eligible = successful.head(1)
            if len(eligible) == 0:
                continue
            speed_lower = max(
                float(parent_v_lower),
                float(eligible["_v_conservative"].iloc[-1]),
            )
            speed_upper = max(float(parent_v_upper), speed_lower)
            output.append({
                "b_lower_m": float(lower),
                "b_upper_m": float(upper),
                "v_lower_m_s": speed_lower,
                "v_upper_m_s": speed_upper,
                "speed_source": "test2p5_conditional_lower_b_anchor_reach",
                "speed_anchor_upper_b_m": float(eligible["_b"].iloc[-1]),
                "speed_anchor_count_used": int(len(eligible)),
            })

    merged: list[dict[str, Any]] = []
    for record in output:
        if (
            merged
            and math.isclose(
                float(merged[-1]["b_upper_m"]),
                float(record["b_lower_m"]),
                rel_tol=0.0,
                abs_tol=1.0e-15,
            )
            and math.isclose(
                float(merged[-1]["v_lower_m_s"]),
                float(record["v_lower_m_s"]),
                rel_tol=0.0,
                abs_tol=1.0e-9,
            )
        ):
            merged[-1]["b_upper_m"] = float(record["b_upper_m"])
            merged[-1]["speed_anchor_upper_b_m"] = record[
                "speed_anchor_upper_b_m"
            ]
            merged[-1]["speed_anchor_count_used"] = record[
                "speed_anchor_count_used"
            ]
        else:
            merged.append(dict(record))
    return merged


def build_final_tables(
    template: pd.DataFrame,
    class_plan: pd.DataFrame,
    class_summary: pd.DataFrame,
    weighted_results: pd.DataFrame,
    ray_summary: pd.DataFrame,
    reaching_intervals: pd.DataFrame,
    speed_policy: pd.DataFrame,
    allowed_intervals: pd.DataFrame,
    speed_reachability: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    template = ensure_ray_key(template)
    ray_summary = ensure_ray_key(ray_summary)
    reaching_intervals = ensure_ray_key(reaching_intervals)
    class_plan = ensure_ray_key(class_plan)

    if "ray_key" not in ray_summary.columns:
        raise ValueError("Test 2.5 ray summary must contain ray_key or enough columns to build it")

    speed_cells = infer_speed_cells(template, speed_policy, allowed_intervals)
    speed_cells = speed_cells.set_index("speed_label", drop=False)
    speed_reachability_map: dict[str, pd.DataFrame] = {}
    if len(speed_reachability) and "geometry_key" in speed_reachability.columns:
        speed_reachability_map = {
            str(key): group.copy()
            for key, group in speed_reachability.groupby("geometry_key", sort=False)
        }
    theta_map, alpha_map, psi_map = make_angular_bound_maps(template)
    status_table = class_status_by_ray(template, class_summary, weighted_results)
    weight_table = ray_weights(template, weighted_results)

    template_groups = {str(key): group for key, group in template.groupby("ray_key", sort=False)}
    plan_groups = {str(key): group for key, group in class_plan.groupby("ray_key", sort=False)} if len(class_plan) and "ray_key" in class_plan.columns else {}
    status_map = status_table.set_index("ray_key").to_dict("index") if len(status_table) else {}
    weight_map = weight_table.set_index("ray_key").to_dict("index") if len(weight_table) else {}

    ray_rows: list[dict[str, Any]] = []
    interval_rows: list[dict[str, Any]] = []
    excluded_rows: list[dict[str, Any]] = []

    for _, ray in ray_summary.iterrows():
        key = str(ray["ray_key"])
        template_group = template_groups.get(key)
        if template_group is None or len(template_group) == 0:
            excluded_rows.append({
                "ray_key": key,
                "final_exclusion_reason": "missing_test1p5_template_ray",
            })
            continue

        representative = template_group.iloc[0]
        plan_group = plan_groups.get(key, pd.DataFrame())
        physical_annuli = physical_annuli_for_group(
            plan_group if len(plan_group) else template_group
        )
        if not physical_annuli:
            excluded_rows.append({
                "ray_key": key,
                "final_exclusion_reason": "missing_physical_b_annuli",
            })
            continue

        status_record = status_map.get(key, {})
        status = status_record.get("test2_class_status", np.nan)
        status_class = classify_test2_status(status)
        if REQUIRE_TEST2_ACCEPTED_CLASS:
            accepted = status_class == "accepted"
            unresolved_allowed = (
                status_class == "unresolved" and INCLUDE_UNRESOLVED_TEST2_CLASSES
            )
            # When no status file is available, a Test 2.5 reach is adequate
            # evidence that the ray reached the switch/full region.
            inferred_accepted = status_class == "missing" and finite_or_nan(ray.get("n_reach", 0)) > 0
            if not (accepted or unresolved_allowed or inferred_accepted):
                excluded_rows.append({
                    "ray_key": key,
                    "speed_label": str(ray.get("speed_label", representative.get("speed_label", ""))),
                    "test2_class_status": status,
                    "test2_class_status_class": status_class,
                    "final_exclusion_reason": f"test2_class_{status_class}",
                })
                continue

        sampled = reaching_intervals_for_ray(key, reaching_intervals)
        final_b_intervals, support_source, quality = final_b_support_for_ray(
            ray, physical_annuli, sampled
        )
        if not final_b_intervals:
            if INCLUDE_UNREFINED_ACCEPTED_RAYS and physical_annuli:
                final_b_intervals = physical_annuli
                support_source = "unrefined_full_physical_support"
                quality = "unrefined_not_recommended"
            else:
                excluded_rows.append({
                    "ray_key": key,
                    "speed_label": str(ray.get("speed_label", representative.get("speed_label", ""))),
                    "test2_class_status": status,
                    "n_reach": ray.get("n_reach", np.nan),
                    "single_cutoff_monotonic": ray.get("single_cutoff_monotonic", np.nan),
                    "converged": ray.get("converged", np.nan),
                    "final_exclusion_reason": quality,
                })
                continue

        speed_label = str(ray.get("speed_label", representative["speed_label"]))
        v_representative = finite_or_nan(ray.get("v_inf_m_s", representative.get("v_inf_m_s")))
        if speed_label in speed_cells.index:
            speed_row = speed_cells.loc[speed_label]
            if isinstance(speed_row, pd.DataFrame):
                speed_row = speed_row.iloc[0]
            v_lower = float(speed_row["speed_lower_m_s"])
            v_upper = float(speed_row["speed_upper_m_s"])
            v_source = str(speed_row["speed_bounds_source"])
        else:
            v_lower = v_representative
            v_upper = v_representative
            v_source = "representative_only"

        geometry_key = (
            f"th{int(finite_or_nan(ray.get('theta_index', representative.get('theta_index', -1)))):03d}|"
            f"al{int(finite_or_nan(ray.get('alpha_index', representative.get('alpha_index', -1)))):03d}|"
            f"psi{round(finite_or_nan(ray.get('psi_key', representative.get('psi_rad', 0.0))), PSI_KEY_DECIMALS):.{PSI_KEY_DECIMALS}f}"
        )
        parent_v_lower = float(v_lower)
        parent_v_upper = float(v_upper)
        speed_refinement = speed_reachability_map.get(
            geometry_key, pd.DataFrame()
        )
        confirmed_reaches = numeric_series(
            speed_refinement, "speed_lower_confirmed_reach_m_s"
        ).to_numpy(float)
        confirmed_reaches = confirmed_reaches[np.isfinite(confirmed_reaches)]
        resolved_misses = numeric_series(
            speed_refinement, "speed_lower_resolved_miss_m_s"
        ).to_numpy(float)
        resolved_misses = resolved_misses[np.isfinite(resolved_misses)]
        transition_estimates = numeric_series(
            speed_refinement, "speed_transition_estimate_m_s"
        ).to_numpy(float)
        transition_estimates = transition_estimates[np.isfinite(transition_estimates)]
        speed_lower_miss = (
            float(np.min(resolved_misses)) if resolved_misses.size else np.nan
        )
        speed_lower_confirmed_reach = (
            float(np.min(confirmed_reaches)) if confirmed_reaches.size else np.nan
        )
        speed_transition_estimate = (
            float(np.min(transition_estimates)) if transition_estimates.size else np.nan
        )
        conditional_v_b = _conditional_v_b_intervals(
            final_b_intervals,
            speed_refinement,
            parent_v_lower=parent_v_lower,
            parent_v_upper=parent_v_upper,
            representative_v=v_representative,
        )
        if conditional_v_b:
            v_lower = min(float(item["v_lower_m_s"]) for item in conditional_v_b)
            v_upper = max(float(item["v_upper_m_s"]) for item in conditional_v_b)
            v_source = (
                "test2p5_conditional_v_b_full_pulse_rout_fourier_detectability"
                if SPEED_REFINEMENT_TARGET == "detectability"
                else "test2p5_conditional_v_b_reachability"
            )
        elif SPEED_REFINEMENT_TARGET == "detectability":
            excluded_rows.append({
                "ray_key": key,
                "speed_label": speed_label,
                "geometry_key": geometry_key,
                "final_exclusion_reason": (
                    "no_confirmed_detectable_speed_b_support"
                ),
            })
            continue
        else:
            v_lower = max(parent_v_lower, v_representative)
            v_upper = max(parent_v_upper, v_lower)
            v_source = "confirmed_reaching_representative_fallback"

        theta = finite_or_nan(ray.get("theta_rad", representative.get("theta_rad")))
        alpha = finite_or_nan(ray.get("alpha_rad", representative.get("alpha_rad"))) % (2.0 * math.pi)
        psi = finite_or_nan(ray.get("psi_rad", representative.get("psi_rad"))) % (2.0 * math.pi)
        theta_lower, theta_upper = match_linear_bound(theta, theta_map)
        alpha_lower, alpha_upper, alpha_wrap = match_circular_bound(alpha, alpha_map)
        psi_lower, psi_upper, psi_wrap = match_circular_bound(psi, psi_map)
        at_pole = math.isclose(theta, 0.0, abs_tol=ANGLE_MATCH_ATOL_RAD) or math.isclose(
            theta, math.pi, abs_tol=ANGLE_MATCH_ATOL_RAD
        )
        if at_pole:
            alpha_lower, alpha_upper, alpha_wrap = 0.0, 2.0 * math.pi, False

        far_values = actual_far_radius(template_group)
        far_values = far_values[np.isfinite(far_values) & (far_values > 0.0)]
        if len(far_values):
            r_far_lower = float(far_values.min())
            r_far_upper = float(far_values.max())
            r_far_recommended = r_far_upper
        else:
            r_far_lower = r_far_upper = r_far_recommended = np.nan

        physical_area2 = interval_area2(physical_annuli)
        final_area2 = interval_area2(final_b_intervals)
        retained_area_fraction = final_area2 / physical_area2 if physical_area2 > 0 else np.nan
        weight_record = weight_map.get(key, {})
        ray_weight_raw = finite_or_nan(weight_record.get("ray_weight_raw", 1.0))
        if not np.isfinite(ray_weight_raw) or ray_weight_raw < 0.0:
            ray_weight_raw = 0.0

        ray_record = {
            "ray_key": key,
            "trajectory_class_id": str(representative.get("trajectory_class_id", "")),
            "speed_label": speed_label,
            "trajectory_regime": str(representative.get("trajectory_regime", representative.get("collision_regime", ""))),
            "v_inf_representative_m_s": v_representative,
            "v_inf_lower_m_s": v_lower,
            "v_inf_upper_m_s": v_upper,
            "speed_bounds_source": v_source,
            "speed_refinement_target": SPEED_REFINEMENT_TARGET,
            "geometry_key": geometry_key,
            "speed_lower_resolved_miss_m_s": speed_lower_miss,
            "speed_lower_confirmed_reach_m_s": speed_lower_confirmed_reach,
            "speed_transition_estimate_m_s": speed_transition_estimate,
            "speed_reachability_converged": bool(
                len(speed_refinement)
                and bool_series(speed_refinement, "speed_reachability_converged").all()
            ),
            "n_speed_b_anchors": int(len(speed_refinement)),
            "v_inf_all_b_conservative_lower_m_s": (
                max(float(item["v_lower_m_s"]) for item in conditional_v_b)
                if conditional_v_b else v_lower
            ),
            "theta_index": int(finite_or_nan(ray.get("theta_index", representative.get("theta_index", -1)))),
            "alpha_index": int(finite_or_nan(ray.get("alpha_index", representative.get("alpha_index", -1)))),
            "theta_representative_rad": theta,
            "theta_lower_rad": theta_lower,
            "theta_upper_rad": theta_upper,
            "theta_representative_deg": math.degrees(theta),
            "theta_lower_deg": math.degrees(theta_lower),
            "theta_upper_deg": math.degrees(theta_upper),
            "alpha_representative_rad": alpha,
            "alpha_lower_rad": alpha_lower,
            "alpha_upper_rad": alpha_upper,
            "alpha_wraps_zero": alpha_wrap,
            "alpha_degenerate_at_pole": at_pole,
            "alpha_representative_deg": math.degrees(alpha),
            "alpha_lower_deg": math.degrees(alpha_lower),
            "alpha_upper_deg": math.degrees(alpha_upper),
            "psi_representative_rad": psi,
            "psi_lower_rad": psi_lower,
            "psi_upper_rad": psi_upper,
            "psi_wraps_zero": psi_wrap,
            "psi_representative_deg": math.degrees(psi),
            "psi_lower_deg": math.degrees(psi_lower),
            "psi_upper_deg": math.degrees(psi_upper),
            "physical_b_annuli_json": intervals_json(physical_annuli),
            "final_b_intervals_json": intervals_json(final_b_intervals),
            "n_final_b_intervals": len(final_b_intervals),
            "physical_b_lower_m": physical_annuli[0][0],
            "physical_b_upper_m": physical_annuli[-1][1],
            "final_b_lower_m": final_b_intervals[0][0],
            "final_b_upper_m": final_b_intervals[-1][1],
            "physical_b_lower_um": physical_annuli[0][0] * 1.0e6,
            "physical_b_upper_um": physical_annuli[-1][1] * 1.0e6,
            "final_b_lower_um": final_b_intervals[0][0] * 1.0e6,
            "final_b_upper_um": final_b_intervals[-1][1] * 1.0e6,
            "physical_b_area2_m2": physical_area2,
            "final_b_area2_m2": final_area2,
            "retained_b_area_fraction": retained_area_fraction,
            "b_support_source": support_source,
            "support_quality": quality,
            "bmax_confirmed_reach_m": finite_or_nan(ray.get("bmax_lower_m")),
            "bmax_estimate_m": finite_or_nan(ray.get("bmax_estimate_m")),
            "bmax_first_miss_m": finite_or_nan(ray.get("bmax_upper_m")),
            "bmax_confirmed_reach_um": finite_or_nan(ray.get("bmax_lower_m")) * 1.0e6,
            "bmax_estimate_um": finite_or_nan(ray.get("bmax_estimate_m")) * 1.0e6,
            "bmax_first_miss_um": finite_or_nan(ray.get("bmax_upper_m")) * 1.0e6,
            "single_cutoff_monotonic": as_bool(ray.get("single_cutoff_monotonic", True), True),
            "test2p5_converged": as_bool(ray.get("converged", False), False),
            "test2p5_convergence_reason": str(ray.get("convergence_reason", "")),
            "test2p5_n_results": safe_int(ray.get("n_results", 0)),
            "test2p5_n_reach": safe_int(ray.get("n_reach", 0)),
            "test2p5_n_miss": safe_int(ray.get("n_miss", 0)),
            "test2p5_n_unresolved": safe_int(ray.get("n_unresolved", 0)),
            "test2_class_status": status,
            "test2_class_status_class": status_class,
            "R_far_lower_m": r_far_lower,
            "R_far_upper_m": r_far_upper,
            "R_far_recommended_m": r_far_recommended,
            "R_far_lower_mm": r_far_lower * 1.0e3,
            "R_far_upper_mm": r_far_upper * 1.0e3,
            "R_far_recommended_mm": r_far_recommended * 1.0e3,
            "R_switch_m": finite_or_nan(representative.get("R_switch_m")),
            "R_full_m": finite_or_nan(representative.get("R_full_m")),
            "ray_weight_raw": ray_weight_raw,
            "ray_weight_source": str(weight_record.get("ray_weight_source", "equal_ray_fallback")),
            "final_include": True,
        }
        ray_rows.append(ray_record)

        for interval_index, conditional_record in enumerate(conditional_v_b):
            b_lower = float(conditional_record["b_lower_m"])
            b_upper = float(conditional_record["b_upper_m"])
            interval_v_lower = float(conditional_record["v_lower_m_s"])
            interval_v_upper = float(conditional_record["v_upper_m_s"])
            area2 = max(b_upper * b_upper - b_lower * b_lower, 0.0)
            interval_fraction_of_physical = area2 / physical_area2 if physical_area2 > 0 else 0.0
            parent_speed_probability = _maxwell_speed_probability(
                parent_v_lower, parent_v_upper
            )
            retained_speed_probability = _maxwell_speed_probability(
                interval_v_lower, interval_v_upper
            )
            retained_speed_fraction = (
                retained_speed_probability / parent_speed_probability
                if parent_speed_probability > 0.0 else 1.0
            )
            interval_rows.append(
                {
                    **ray_record,
                    "v_inf_lower_m_s": interval_v_lower,
                    "v_inf_upper_m_s": interval_v_upper,
                    "speed_bounds_source": str(conditional_record["speed_source"]),
                    "speed_anchor_upper_b_m": conditional_record["speed_anchor_upper_b_m"],
                    "speed_anchor_count_used": conditional_record["speed_anchor_count_used"],
                    "parent_speed_probability": parent_speed_probability,
                    "retained_speed_probability": retained_speed_probability,
                    "retained_speed_fraction_of_parent": retained_speed_fraction,
                    "b_interval_index": interval_index,
                    "b_interval_lower_m": b_lower,
                    "b_interval_upper_m": b_upper,
                    "b_interval_lower_um": b_lower * 1.0e6,
                    "b_interval_upper_um": b_upper * 1.0e6,
                    "b_interval_area2_m2": area2,
                    "b_interval_fraction_of_physical_support": interval_fraction_of_physical,
                    "R_full_at_representative_v_b_lower_m": rutherford_v_b_radius_policy(
                        v_representative,
                        b_lower,
                        m_dm_kg=float(USER_M_DM_KG),
                        m_ion_kg=float(m_ion),
                        eps_value=float(USER_EPS),
                        ion_charge_number=float(Z_ion),
                        coulomb_constant=float(K),
                        elementary_charge_c=float(e),
                        threshold_j=float(USER_TARGET_ION_ENERGY_J),
                    )["R_full_m"],
                    "R_full_at_representative_v_b_upper_m": rutherford_v_b_radius_policy(
                        v_representative,
                        b_upper,
                        m_dm_kg=float(USER_M_DM_KG),
                        m_ion_kg=float(m_ion),
                        eps_value=float(USER_EPS),
                        ion_charge_number=float(Z_ion),
                        coulomb_constant=float(K),
                        elementary_charge_c=float(e),
                        threshold_j=float(USER_TARGET_ION_ENERGY_J),
                    )["R_full_m"],
                    "R_full_v_b_mode": str(USER_R_FULL_VB_MODE),
                    "sampling_weight_raw": (
                        ray_weight_raw
                        * interval_fraction_of_physical
                        * retained_speed_fraction
                    ),
                }
            )

    ray_table = pd.DataFrame(ray_rows)
    interval_table = pd.DataFrame(interval_rows)
    excluded = pd.DataFrame(excluded_rows)

    if len(interval_table):
        total = float(numeric_series(interval_table, "sampling_weight_raw", 0.0).sum())
        if total > 0.0:
            interval_table["sampling_probability"] = (
                numeric_series(interval_table, "sampling_weight_raw", 0.0) / total
            )
        else:
            interval_table["sampling_probability"] = 1.0 / len(interval_table)
        ray_prob = interval_table.groupby("ray_key")["sampling_probability"].sum()
        ray_table["sampling_probability"] = ray_table["ray_key"].map(ray_prob).fillna(0.0)
    else:
        ray_table["sampling_probability"] = np.nan

    return ray_table, interval_table, excluded


# =============================================================================
# GLOBAL SUMMARY AND SAMPLER
# =============================================================================


def global_bounds_table(ray_table: pd.DataFrame, interval_table: pd.DataFrame) -> pd.DataFrame:
    if len(ray_table) == 0:
        return pd.DataFrame(columns=["parameter", "lower", "upper", "units", "note"])
    rows = [
        {
            "parameter": "v_inf",
            "lower": numeric_series(interval_table, "v_inf_lower_m_s").min(),
            "upper": numeric_series(interval_table, "v_inf_upper_m_s").max(),
            "units": "m/s",
            "note": (
                "Union of Test-2.5 confirmed conditional speed-b cells; "
                "inspect the interval table for b-dependent lower bounds."
            ),
        },
        {
            "parameter": "theta",
            "lower": numeric_series(ray_table, "theta_lower_rad").min(),
            "upper": numeric_series(ray_table, "theta_upper_rad").max(),
            "units": "rad",
            "note": "Conditional angular-cell union.",
        },
        {
            "parameter": "alpha",
            "lower": 0.0,
            "upper": 2.0 * math.pi,
            "units": "rad",
            "note": "Circular cells; not every alpha is retained at every theta/speed.",
        },
        {
            "parameter": "psi",
            "lower": 0.0,
            "upper": 2.0 * math.pi,
            "units": "rad",
            "note": "Circular cells; support is ray dependent.",
        },
        {
            "parameter": "b",
            "lower": numeric_series(interval_table, "b_interval_lower_m").min(),
            "upper": numeric_series(interval_table, "b_interval_upper_m").max(),
            "units": "m",
            "note": "Strongly conditional; sample only from interval table.",
        },
        {
            "parameter": "R_far",
            "lower": numeric_series(ray_table, "R_far_lower_m").min(),
            "upper": numeric_series(ray_table, "R_far_upper_m").max(),
            "units": "m",
            "note": "Use the per-ray recommended launch radius, not an independent draw.",
        },
    ]
    return pd.DataFrame(rows)


def coverage_summary(
    template: pd.DataFrame,
    ray_summary: pd.DataFrame,
    ray_table: pd.DataFrame,
    excluded: pd.DataFrame,
) -> pd.DataFrame:
    template_keys = set(ensure_ray_key(template).get("ray_key", pd.Series(dtype=str)).astype(str))
    refined_keys = set(ensure_ray_key(ray_summary).get("ray_key", pd.Series(dtype=str)).astype(str))
    included_keys = set(ray_table.get("ray_key", pd.Series(dtype=str)).astype(str))
    rows = [
        {"stage": "Test 1.5 physical rays", "count": len(template_keys)},
        {"stage": "Test 2.5 refined rays", "count": len(refined_keys)},
        {"stage": "Final included rays", "count": len(included_keys)},
        {"stage": "Final excluded refined rays", "count": len(excluded)},
        {"stage": "Physical rays not refined by Test 2.5", "count": len(template_keys - refined_keys)},
    ]
    return pd.DataFrame(rows)


def sampler_map(ray_table: pd.DataFrame, interval_table: pd.DataFrame) -> dict[str, Any]:
    cells: list[dict[str, Any]] = []
    for _, ray in ray_table.iterrows():
        intervals = interval_table.loc[
            interval_table["ray_key"].astype(str).eq(str(ray["ray_key"]))
        ].sort_values("b_interval_index")
        cells.append(
            {
                "ray_key": str(ray["ray_key"]),
                "speed_label": str(ray["speed_label"]),
                "v_inf": {
                    "representative_m_s": json_number(ray["v_inf_representative_m_s"]),
                    "lower_m_s": json_number(ray["v_inf_lower_m_s"]),
                    "upper_m_s": json_number(ray["v_inf_upper_m_s"]),
                },
                "theta": {
                    "representative_rad": json_number(ray["theta_representative_rad"]),
                    "lower_rad": json_number(ray["theta_lower_rad"]),
                    "upper_rad": json_number(ray["theta_upper_rad"]),
                },
                "alpha": {
                    "representative_rad": json_number(ray["alpha_representative_rad"]),
                    "lower_rad": json_number(ray["alpha_lower_rad"]),
                    "upper_rad": json_number(ray["alpha_upper_rad"]),
                    "wraps_zero": as_bool(ray["alpha_wraps_zero"]),
                },
                "psi": {
                    "representative_rad": json_number(ray["psi_representative_rad"]),
                    "lower_rad": json_number(ray["psi_lower_rad"]),
                    "upper_rad": json_number(ray["psi_upper_rad"]),
                    "wraps_zero": as_bool(ray["psi_wraps_zero"]),
                },
                "b_intervals_m": [
                    [json_number(record.b_interval_lower_m), json_number(record.b_interval_upper_m)]
                    for record in intervals.itertuples(index=False)
                ],
                "conditional_v_b_intervals": [
                    {
                        "v_lower_m_s": json_number(record.v_inf_lower_m_s),
                        "v_upper_m_s": json_number(record.v_inf_upper_m_s),
                        "b_lower_m": json_number(record.b_interval_lower_m),
                        "b_upper_m": json_number(record.b_interval_upper_m),
                        "sampling_probability": json_number(record.sampling_probability),
                    }
                    for record in intervals.itertuples(index=False)
                ],
                "R_far_recommended_m": json_number(ray["R_far_recommended_m"]),
                "sampling_probability": json_number(ray["sampling_probability"]),
                "support_quality": str(ray["support_quality"]),
            }
        )
    return {
        "schema": "conditional_trajectory_parameter_space_v6_escape_channel_full_pulse",
        "parameters": ["v_inf", "theta", "alpha", "psi", "b", "R_far"],
        "b_sampling": "uniform_in_b_squared_within_selected_interval",
        "theta_sampling": "uniform_in_cos_theta_within_selected_cell",
        "R_far_sampling": "fixed_to_per_ray_recommended_value",
        "energy_selection": {
            "primary_policy": str(globals().get(
                "USER_DETECTION_ENERGY_POLICY", "fourier"
            )),
            "speed_refinement_target": SPEED_REFINEMENT_TARGET,
            "threshold_J": float(globals().get(
                "USER_TARGET_ION_ENERGY_J", np.nan
            )),
            "fourier_and_mechanical_values_saved": True,
            "fourier_detection_history": (
                "incoming_R_outer_to_outgoing_R_outer"
            ),
            "incoming_prefix_required": bool(globals().get(
                "USER_REQUIRE_R_FAR_PREFIX_COMPLETE", True
            )),
            "outgoing_tail_required": bool(globals().get(
                "USER_REQUIRE_OUTGOING_R_FAR_TAIL_COMPLETE", True
            )),
            "outer_tails_required": bool(globals().get(
                "USER_REQUIRE_OUTER_R_TAILS_COMPLETE", True
            )),
            "fourier_outer_radius_m": float(globals().get(
                "USER_FOURIER_OUTER_RADIUS_M", 20.0e-3
            )),
        },
        "cells": cells,
    }


def sample_circular_interval(
    rng: np.random.Generator,
    lower: float,
    upper: float,
    wraps: bool,
    size: int,
) -> np.ndarray:
    if not wraps:
        return rng.uniform(lower, upper, size=size)
    width = (upper - lower) % (2.0 * math.pi)
    return (lower + rng.uniform(0.0, width, size=size)) % (2.0 * math.pi)


def sample_final_parameter_space(
    interval_table: pd.DataFrame,
    n_samples: int,
    *,
    seed: int = SAMPLER_RANDOM_SEED,
    speed_mode: str = SAMPLER_SPEED_MODE,
) -> pd.DataFrame:
    """Draw samples from the final conditional support.

    This is a geometry-aware proposal sampler.  Its row-selection probability
    uses the retained Test 1.5/Test 2 weight when available and the retained
    annular-area fraction.  ``b`` is always sampled uniformly in b^2.
    """
    if n_samples <= 0:
        return pd.DataFrame()
    if len(interval_table) == 0:
        raise ValueError("Final interval table is empty")
    rng = np.random.default_rng(seed)
    probabilities = numeric_series(interval_table, "sampling_probability", 0.0).to_numpy(float)
    probabilities = np.where(np.isfinite(probabilities) & (probabilities > 0.0), probabilities, 0.0)
    if probabilities.sum() <= 0.0:
        probabilities = np.full(len(interval_table), 1.0 / len(interval_table))
    else:
        probabilities /= probabilities.sum()
    selected_indices = rng.choice(len(interval_table), size=n_samples, p=probabilities)
    selected = interval_table.iloc[selected_indices].reset_index(drop=True).copy()

    if speed_mode == "representative":
        v_inf = numeric_series(selected, "v_inf_representative_m_s").to_numpy(float)
    elif speed_mode == "uniform_cell":
        v_inf = rng.uniform(
            numeric_series(selected, "v_inf_lower_m_s").to_numpy(float),
            numeric_series(selected, "v_inf_upper_m_s").to_numpy(float),
        )
    else:
        raise ValueError("speed_mode must be 'representative' or 'uniform_cell'")

    theta_lower = numeric_series(selected, "theta_lower_rad").to_numpy(float)
    theta_upper = numeric_series(selected, "theta_upper_rad").to_numpy(float)
    cos_lower = np.cos(theta_lower)
    cos_upper = np.cos(theta_upper)
    cos_theta = rng.uniform(np.minimum(cos_lower, cos_upper), np.maximum(cos_lower, cos_upper))
    theta = np.arccos(np.clip(cos_theta, -1.0, 1.0))

    alpha = np.empty(n_samples)
    psi = np.empty(n_samples)
    for index, row in selected.iterrows():
        alpha[index] = sample_circular_interval(
            rng,
            float(row["alpha_lower_rad"]),
            float(row["alpha_upper_rad"]),
            as_bool(row["alpha_wraps_zero"]),
            1,
        )[0]
        psi[index] = sample_circular_interval(
            rng,
            float(row["psi_lower_rad"]),
            float(row["psi_upper_rad"]),
            as_bool(row["psi_wraps_zero"]),
            1,
        )[0]

    b_lower = numeric_series(selected, "b_interval_lower_m").to_numpy(float)
    b_upper = numeric_series(selected, "b_interval_upper_m").to_numpy(float)
    b = np.sqrt(rng.uniform(b_lower * b_lower, b_upper * b_upper))
    sampled_radii = [
        rutherford_v_b_radius_policy(
            speed,
            impact,
            m_dm_kg=float(USER_M_DM_KG),
            m_ion_kg=float(m_ion),
            eps_value=float(USER_EPS),
            ion_charge_number=float(Z_ion),
            coulomb_constant=float(K),
            elementary_charge_c=float(e),
            threshold_j=float(USER_TARGET_ION_ENERGY_J),
        )
        for speed, impact in zip(v_inf, b)
    ]

    return pd.DataFrame(
        {
            "ray_key": selected["ray_key"].astype(str),
            "speed_label": selected["speed_label"].astype(str),
            "v_inf_m_s": v_inf,
            "theta_rad": theta,
            "alpha_rad": alpha,
            "psi_rad": psi,
            "b_m": b,
            "r_min_energy_threshold_m": [
                value["r_min_energy_threshold_m"] for value in sampled_radii
            ],
            "r_min_rutherford_v_b_m": [
                value["r_min_rutherford_v_b_m"] for value in sampled_radii
            ],
            "R_full_m": [value["R_full_m"] for value in sampled_radii],
            "R_switch_m": [value["R_switch_m"] for value in sampled_radii],
            "R_full_v_b_mode": [value["R_full_v_b_mode"] for value in sampled_radii],
            "R_far_m": numeric_series(selected, "R_far_recommended_m").to_numpy(float),
        }
    )


# =============================================================================
# PLOTS: DISPLAY ONLY, NEVER SAVE
# =============================================================================


def show_plots(
    ray_table: pd.DataFrame,
    interval_table: pd.DataFrame,
    excluded: pd.DataFrame,
    coverage: pd.DataFrame,
) -> None:
    if not SHOW_PLOTS:
        return

    # 1. Pipeline coverage.
    plt.figure(figsize=(9, 4.8))
    plt.bar(coverage["stage"], coverage["count"])
    plt.ylabel("Number of rays")
    plt.title("Trajectory-space coverage through the truncation pipeline")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()
    plt.close()

    if len(ray_table) == 0:
        return

    # 2. Confirmed b reach bound versus speed.
    plt.figure(figsize=(8, 5))
    plt.scatter(
        numeric_series(ray_table, "v_inf_representative_m_s"),
        numeric_series(ray_table, "final_b_upper_um"),
    )
    plt.xlabel(r"$v_\infty$ (m/s)")
    plt.ylabel(r"Final conservative $b$ upper bound ($\mu$m)")
    plt.title("Conservative impact-parameter support versus speed")
    plt.tight_layout()
    plt.show()
    plt.close()

    # 3. Angular coverage, color-coded by final b upper bound.
    plt.figure(figsize=(8, 5.5))
    scatter = plt.scatter(
        numeric_series(ray_table, "alpha_representative_deg"),
        numeric_series(ray_table, "theta_representative_deg"),
        c=numeric_series(ray_table, "final_b_upper_um"),
        s=45,
    )
    plt.xlabel(r"$\alpha$ (deg)")
    plt.ylabel(r"$\theta$ (deg)")
    plt.title("Retained angular rays colored by conservative b support")
    plt.colorbar(scatter, label=r"Final $b_{\max}$ ($\mu$m)")
    plt.tight_layout()
    plt.show()
    plt.close()

    # 4. Retained cross-sectional-area fraction versus speed.
    plt.figure(figsize=(8, 5))
    plt.scatter(
        numeric_series(ray_table, "v_inf_representative_m_s"),
        numeric_series(ray_table, "retained_b_area_fraction"),
    )
    plt.xlabel(r"$v_\infty$ (m/s)")
    plt.ylabel("Retained physical b-area fraction")
    plt.ylim(bottom=0.0)
    plt.title("Fraction of the Test 0.75 impact area retained")
    plt.tight_layout()
    plt.show()
    plt.close()

    # 5. Recommended launch radius versus speed.
    finite_far = np.isfinite(numeric_series(ray_table, "R_far_recommended_mm"))
    if finite_far.any():
        plt.figure(figsize=(8, 5))
        plt.scatter(
            numeric_series(ray_table.loc[finite_far], "v_inf_representative_m_s"),
            numeric_series(ray_table.loc[finite_far], "R_far_recommended_mm"),
        )
        plt.xlabel(r"$v_\infty$ (m/s)")
        plt.ylabel(r"Recommended $R_{far}$ (mm)")
        plt.title("Validated launch radius versus speed")
        plt.tight_layout()
        plt.show()
        plt.close()

    # 6. Exclusion reasons.
    if len(excluded) and "final_exclusion_reason" in excluded.columns:
        counts = excluded["final_exclusion_reason"].astype(str).value_counts().sort_values()
        plt.figure(figsize=(9, max(4.0, 0.35 * len(counts) + 1.5)))
        plt.barh(counts.index, counts.values)
        plt.xlabel("Number of rays")
        plt.title("Reasons refined rays were excluded from the final sampler")
        plt.tight_layout()
        plt.show()
        plt.close()

    # 7. Interval widths, useful for identifying tiny or degenerate supports.
    if len(interval_table):
        widths_um = (
            numeric_series(interval_table, "b_interval_upper_um")
            - numeric_series(interval_table, "b_interval_lower_um")
        )
        finite_widths = widths_um[np.isfinite(widths_um)]
        if len(finite_widths):
            plt.figure(figsize=(8, 5))
            plt.hist(finite_widths, bins=min(30, max(5, int(math.sqrt(len(finite_widths))))))
            plt.xlabel(r"Retained b-interval width ($\mu$m)")
            plt.ylabel("Count")
            plt.title("Distribution of final impact-parameter interval widths")
            plt.tight_layout()
            plt.show()
            plt.close()


# Columns used when Test 2.5 legitimately produced no rays.  Keeping a
# header-only schema prevents pandas EmptyDataError in later post-processing.
EMPTY_FINAL_RAY_COLUMNS = [
    "ray_key",
    "trajectory_class_id",
    "speed_label",
    "trajectory_regime",
    "v_inf_representative_m_s",
    "v_inf_lower_m_s",
    "v_inf_upper_m_s",
    "theta_representative_rad",
    "theta_lower_rad",
    "theta_upper_rad",
    "alpha_representative_rad",
    "alpha_lower_rad",
    "alpha_upper_rad",
    "alpha_wraps_zero",
    "psi_representative_rad",
    "psi_lower_rad",
    "psi_upper_rad",
    "psi_wraps_zero",
    "physical_b_annuli_json",
    "final_b_intervals_json",
    "final_b_lower_m",
    "final_b_upper_m",
    "R_far_recommended_m",
    "R_switch_m",
    "R_full_m",
    "sampling_probability",
    "support_quality",
    "final_include",
]

EMPTY_FINAL_INTERVAL_COLUMNS = EMPTY_FINAL_RAY_COLUMNS + [
    "b_interval_index",
    "b_interval_lower_m",
    "b_interval_upper_m",
    "b_interval_lower_um",
    "b_interval_upper_um",
    "b_interval_area2_m2",
    "b_interval_fraction_of_physical_support",
    "sampling_weight_raw",
]

EMPTY_EXCLUDED_COLUMNS = [
    "ray_key",
    "trajectory_class_id",
    "speed_label",
    "test2_class_status",
    "test2_class_status_class",
    "final_exclusion_reason",
]


def no_refinement_outputs(
    template: pd.DataFrame,
    class_summary: pd.DataFrame,
    weighted_results: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build header-safe empty final tables when Test 2.5 has no rays.

    This is the expected outcome when Test 2 rejected every class.  We still
    preserve one audit row per Test 1.5 ray so the reason for the empty final
    parameter space is explicit.
    """
    ray_table = pd.DataFrame(columns=EMPTY_FINAL_RAY_COLUMNS)
    interval_table = pd.DataFrame(columns=EMPTY_FINAL_INTERVAL_COLUMNS)

    template_with_key = ensure_ray_key(template)
    if "ray_key" not in template_with_key.columns or len(template_with_key) == 0:
        return ray_table, interval_table, pd.DataFrame(columns=EMPTY_EXCLUDED_COLUMNS)

    keep = [
        column
        for column in ("ray_key", "trajectory_class_id", "speed_label")
        if column in template_with_key.columns
    ]
    excluded = template_with_key[keep].drop_duplicates("ray_key", keep="first").copy()

    statuses = class_status_by_ray(template, class_summary, weighted_results)
    if len(statuses):
        excluded = excluded.merge(statuses, on="ray_key", how="left", validate="one_to_one")
    else:
        excluded["test2_class_status"] = np.nan

    excluded["test2_class_status_class"] = excluded["test2_class_status"].map(
        classify_test2_status
    )
    excluded["final_exclusion_reason"] = np.where(
        excluded["test2_class_status_class"].eq("rejected"),
        "test2_rejected_no_switch_entry_no_test2p5_refinement",
        "no_test2p5_refined_ray_available",
    )

    for column in EMPTY_EXCLUDED_COLUMNS:
        if column not in excluded.columns:
            excluded[column] = np.nan
    return ray_table, interval_table, excluded[EMPTY_EXCLUDED_COLUMNS]


# =============================================================================
# REPORTING AND MAIN
# =============================================================================


def print_input_report(files: InputFiles) -> None:
    progress("Resolved input files:")
    for name, path in files.__dict__.items():
        progress(f"  {name:22s}: {path if path is not None else '[not found / optional]'}")


def print_results(
    ray_table: pd.DataFrame,
    interval_table: pd.DataFrame,
    excluded: pd.DataFrame,
    bounds: pd.DataFrame,
    coverage: pd.DataFrame,
) -> None:
    progress("\nCoverage summary:")
    progress(coverage.to_string(index=False))

    progress("\nGlobal six-parameter envelope:")
    progress(bounds.to_string(index=False))

    progress("\nImportant: this envelope is descriptive only. Sample from the conditional interval table, not from one six-dimensional rectangle.")

    if len(ray_table):
        display_columns = [
            "speed_label",
            "v_inf_representative_m_s",
            "theta_representative_deg",
            "alpha_representative_deg",
            "psi_representative_deg",
            "final_b_lower_um",
            "final_b_upper_um",
            "R_far_recommended_mm",
            "retained_b_area_fraction",
            "support_quality",
            "test2p5_converged",
        ]
        display_columns = [column for column in display_columns if column in ray_table.columns]
        progress(f"\nFinal included ray cells: {len(ray_table):,}")
        progress(ray_table[display_columns].head(PRINT_MAX_ROWS).to_string(index=False))
        if len(ray_table) > PRINT_MAX_ROWS:
            progress(f"... {len(ray_table) - PRINT_MAX_ROWS:,} additional included rays are in {FINAL_RAY_TABLE_CSV}")
    else:
        progress("\nNo rays met the conservative final inclusion policy.")

    if len(excluded):
        progress("\nExcluded-ray reasons:")
        progress(
            excluded["final_exclusion_reason"]
            .astype(str)
            .value_counts(dropna=False)
            .to_string()
        )

    progress(f"\nFinal conditional b intervals: {len(interval_table):,}")
    if len(interval_table):
        probability_sum = numeric_series(interval_table, "sampling_probability", 0.0).sum()
        progress(f"Sampling probabilities sum to {probability_sum:.12f}")


def save_outputs(
    ray_table: pd.DataFrame,
    interval_table: pd.DataFrame,
    excluded: pd.DataFrame,
    bounds: pd.DataFrame,
    coverage: pd.DataFrame,
) -> None:
    if not SAVE_TABLES:
        return
    ray_table.to_csv(FINAL_RAY_TABLE_CSV, index=False)
    interval_table.to_csv(FINAL_INTERVAL_TABLE_CSV, index=False)
    excluded.to_csv(EXCLUDED_RAYS_CSV, index=False)
    bounds.to_csv(GLOBAL_BOUNDS_CSV, index=False)
    coverage.to_csv(COVERAGE_SUMMARY_CSV, index=False)
    SAMPLER_JSON.write_text(
        json.dumps(sampler_map(ray_table, interval_table), indent=2, allow_nan=False),
        encoding="utf-8",
    )
    progress("\nSaved final parameter-space products:")
    for path in (
        FINAL_RAY_TABLE_CSV,
        FINAL_INTERVAL_TABLE_CSV,
        GLOBAL_BOUNDS_CSV,
        EXCLUDED_RAYS_CSV,
        COVERAGE_SUMMARY_CSV,
        SAMPLER_JSON,
    ):
        progress(f"  {path}")
    progress("No plot files were saved.")


def main() -> None:
    files = resolve_inputs()
    print_input_report(files)

    allowed = load_csv(files.test0p75_allowed)
    speed_policy = load_csv(files.test0p75_speed)
    template = load_csv(files.test1p5_template)
    class_plan = load_csv(files.test1p5_plan)
    class_summary = load_csv(files.test2_class)
    weighted = load_csv(files.test2_weighted)
    ray_summary = load_csv(files.test2p5_ray)
    intervals = load_csv(files.test2p5_intervals)
    all_results = load_csv(files.test2p5_all)
    speed_reachability = load_csv(files.test2p5_speed)

    if len(template):
        validate_parameter_dataframe(template, "Final-builder Test 1.5 template")
    if len(weighted):
        validate_parameter_dataframe(weighted, "Final-builder Test 2 results")
    if len(ray_summary):
        validate_parameter_dataframe(ray_summary, "Final-builder Test 2.5 summary")

    if len(template) == 0 and len(ray_summary) == 0:
        progress(
            "\nTest 1.5 and Test 2.5 contain no dynamic rays. Writing a valid "
            "empty final parameter space."
        )
        ray_table = pd.DataFrame(columns=EMPTY_FINAL_RAY_COLUMNS)
        interval_table = pd.DataFrame(columns=EMPTY_FINAL_INTERVAL_COLUMNS)
        excluded = pd.DataFrame(columns=EMPTY_EXCLUDED_COLUMNS)
        bounds = global_bounds_table(ray_table, interval_table)
        coverage = coverage_summary(template, ray_summary, ray_table, excluded)
        print_results(ray_table, interval_table, excluded, bounds, coverage)
        save_outputs(ray_table, interval_table, excluded, bounds, coverage)
        show_plots(ray_table, interval_table, excluded, coverage)
        return

    required_template = {
        "speed_label",
        "v_inf_m_s",
        "theta_index",
        "alpha_index",
        "theta_rad",
        "alpha_rad",
        "psi_rad",
    }
    missing = sorted(required_template - set(template.columns))
    if missing:
        raise ValueError("Test 1.5 template is missing: " + ", ".join(missing))
    if len(ray_summary) == 0:
        progress(
            "\nTest 2.5 produced no refined rays. This is expected when Test 2 "
            "rejected every trajectory class. Building an empty final parameter "
            "space with explicit audit rows instead of raising an exception."
        )
        ray_table, interval_table, excluded = no_refinement_outputs(
            template=template,
            class_summary=class_summary,
            weighted_results=weighted,
        )
        bounds = global_bounds_table(ray_table, interval_table)
        coverage = coverage_summary(template, ray_summary, ray_table, excluded)
        print_results(ray_table, interval_table, excluded, bounds, coverage)
        save_outputs(ray_table, interval_table, excluded, bounds, coverage)
        show_plots(ray_table, interval_table, excluded, coverage)
        return

    # all_results is optional, but useful as a fallback source for class status.
    if len(weighted) == 0 and len(all_results):
        weighted = all_results.copy()

    ray_table, interval_table, excluded = build_final_tables(
        template=template,
        class_plan=class_plan,
        class_summary=class_summary,
        weighted_results=weighted,
        ray_summary=ray_summary,
        reaching_intervals=intervals,
        speed_policy=speed_policy,
        allowed_intervals=allowed,
        speed_reachability=speed_reachability,
    )
    bounds = global_bounds_table(ray_table, interval_table)
    coverage = coverage_summary(template, ray_summary, ray_table, excluded)

    print_results(ray_table, interval_table, excluded, bounds, coverage)
    save_outputs(ray_table, interval_table, excluded, bounds, coverage)
    show_plots(ray_table, interval_table, excluded, coverage)

    if N_EXAMPLE_SAMPLES > 0 and len(interval_table):
        examples = sample_final_parameter_space(
            interval_table,
            N_EXAMPLE_SAMPLES,
            seed=SAMPLER_RANDOM_SEED,
            speed_mode=SAMPLER_SPEED_MODE,
        )
        progress("\nExample samples from the final conditional support:")
        progress(examples.to_string(index=False))


if __name__ == "__main__":
    if bool(globals().get("USER_RUN_FINAL_BUILDER", True)):
        main()
    else:
        progress("[SKIP] strict final parameter-space builder")


# Exploratory Candidate Report

This separate table preserves unresolved larger-$R_{\max}$ directions, trajectories that enter the bowl but miss $R_{\rm switch}$, focused escape-channel diagnostics, unresolved Test 2 rows, insufficient Test 2.5 anchors, and strict-policy exclusions. These candidates are targets for more simulation, not production probability mass.


In [ ]:
#!/usr/bin/env python3
"""Build an exploratory candidate report alongside the strict final sampler.

This file does not relax the production probability policy. Instead it gathers
unresolved, diagnostic, bowl-entering, and insufficiently refined rays into one
prioritized table for additional simulation.

Run after Test 2.5 and the strict final builder:

    %run -i 06_exploratory_candidate_report_v1.py
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


RUN_DIR = Path(globals().get("RUN_DIRECTORY", "."))
OUTPUT_CSV = RUN_DIR / "exploratory_trajectory_candidates_v1.csv"
SUMMARY_CSV = RUN_DIR / "exploratory_trajectory_candidates_v1_summary.csv"

ADAPTIVE_STILL_UNRESOLVED = RUN_DIR / (
    "escape_channel_adaptive_rmax_bowl_v2_still_unresolved.csv"
)
BOWL_AUDIT = RUN_DIR / "escape_channel_adaptive_rmax_bowl_v2_bowl_entry_audit.csv"
FOCUS_INPUT = RUN_DIR / (
    "escape_channel_adaptive_rmax_bowl_v2_escape_channel_focus_test2_input.csv"
)
TEST2_RESULTS = Path(f"{TEST2_OUTPUT_PREFIX}_weighted_analysis_results.csv")
TEST2P5_RAYS = Path(f"{TEST2P5_OUTPUT_PREFIX}_ray_bmax_summary.csv")
FINAL_EXCLUDED = Path(f"{FINAL_OUTPUT_PREFIX}_excluded_rays.csv")


def progress(message: str = "") -> None:
    print(message, flush=True)


def resolve_optional(path: Path) -> Path | None:
    path = Path(path)
    candidates = [path] if path.is_absolute() else [path, RUN_DIR / path]
    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.is_file() and candidate.stat().st_size > 0:
            return candidate
    return None


def load_optional(path: Path) -> pd.DataFrame:
    resolved = resolve_optional(path)
    if resolved is None:
        return pd.DataFrame()
    try:
        frame = pd.read_csv(resolved, low_memory=False)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()
    frame["_source_file"] = str(resolved)
    return frame


def numeric(frame: pd.DataFrame, column: str, default=np.nan) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(default, index=frame.index, dtype=float)
    return pd.to_numeric(frame[column], errors="coerce")


def bools(frame: pd.DataFrame, column: str, default=False) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(default, index=frame.index, dtype=bool)
    values = frame[column]
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(default).astype(bool)
    return values.astype(str).str.strip().str.lower().isin(
        {"true", "1", "yes", "y"}
    )


def common_record(row: pd.Series, source: str) -> dict[str, Any]:
    return {
        "source": source,
        "trajectory_id": str(row.get("trajectory_id", "")),
        "ray_key": str(row.get("ray_key", "")),
        "geometry_key": str(row.get("geometry_key", "")),
        "speed_label": str(row.get("speed_label", "")),
        "v_inf_m_s": float(row.get("v_inf_m_s", np.nan)),
        "theta_rad": float(row.get("theta_rad", np.nan)),
        "alpha_rad": float(row.get("alpha_rad", np.nan)),
        "psi_rad": float(row.get("psi_rad", np.nan)),
        "b_m": float(row.get("b_m", np.nan)),
        "R_full_m": float(row.get("R_full_m", np.nan)),
        "R_switch_m": float(row.get("R_switch_m", np.nan)),
        "R_far_m": float(row.get("R_far_m", row.get("R_far_recommended_m", np.nan))),
        "test2_diagnostic_only": bool(
            str(row.get("test2_diagnostic_only", False)).lower()
            in {"true", "1", "yes", "y"}
        ),
        "source_file": str(row.get("_source_file", "")),
    }


def add_adaptive_unresolved(rows: list[dict[str, Any]]) -> None:
    frame = load_optional(ADAPTIVE_STILL_UNRESOLVED)
    for _, row in frame.iterrows():
        record = common_record(row, "adaptive_Rmax_unresolved")
        record.update({
            "status": str(row.get("outer_status", row.get("test2_readiness_reason", ""))),
            "candidate_category": "needs_larger_Rmax_or_outer_path_resolution",
            "priority": 1,
            "recommended_action": (
                "Extend the adaptive R_max ladder for this exact direction and b; "
                "do not treat it as a physical miss yet."
            ),
            "adaptive_R_max_used_m": float(row.get("adaptive_R_max_used_m", np.nan)),
            "bowl_status": str(row.get("bowl_status", "")),
            "dynamic_bowl_status": str(row.get("dynamic_bowl_status", "")),
        })
        rows.append(record)


def add_bowl_candidates(rows: list[dict[str, Any]]) -> None:
    frame = load_optional(BOWL_AUDIT)
    if len(frame) == 0:
        return
    status = frame.get("dynamic_bowl_status", pd.Series("", index=frame.index)).astype(str)
    mask = status.isin({
        "entered_bowl_but_escaped_before_R_switch",
        "turned_before_bowl",
        "timeout_unresolved",
        "invalid_domain_before_resolution",
    })
    for _, row in frame.loc[mask].iterrows():
        record = common_record(row, "bowl_entry_audit")
        entered = bool(row.get("dynamic_entered_bowl", False))
        record.update({
            "status": str(row.get("dynamic_bowl_status", "")),
            "candidate_category": (
                "entered_bowl_but_missed_switch"
                if entered else "barrier_or_bowl_boundary_candidate"
            ),
            "priority": 2 if entered else 3,
            "recommended_action": (
                "Refine b and psi locally after the bowl lip."
                if entered
                else "Refine angle and speed near the minimum barrier direction."
            ),
            "bowl_status": str(row.get("bowl_status", "")),
            "dynamic_bowl_status": str(row.get("dynamic_bowl_status", "")),
            "bowl_barrier_J": float(row.get("bowl_barrier_J", np.nan)),
            "bowl_v_min_m_s": float(row.get("bowl_v_min_m_s", np.nan)),
            "bowl_lip_radius_m": float(row.get("bowl_lip_radius_m", np.nan)),
        })
        rows.append(record)


def add_test2_candidates(rows: list[dict[str, Any]]) -> None:
    frame = load_optional(TEST2_RESULTS)
    if len(frame) == 0:
        return
    status = frame.get("test2_status", pd.Series("", index=frame.index)).astype(str)
    unresolved = status.str.startswith("unresolved")
    diagnostic = bools(frame, "test2_diagnostic_only", False)
    for _, row in frame.loc[unresolved | diagnostic].iterrows():
        is_diag = str(row.get("test2_diagnostic_only", False)).lower() in {
            "true", "1", "yes", "y"
        }
        test2_status = str(row.get("test2_status", ""))
        record = common_record(row, "test2")
        record.update({
            "status": test2_status,
            "candidate_category": (
                "escape_channel_diagnostic"
                if is_diag else "unresolved_test2_trajectory"
            ),
            "priority": 1 if is_diag and bool(row.get("entered_switch", False)) else 2,
            "recommended_action": (
                "Promote the surrounding finite angular cell to a weighted refinement."
                if is_diag and bool(row.get("entered_switch", False))
                else "Rerun with the resolution implied by the unresolved status."
            ),
            "entered_switch": bool(row.get("entered_switch", False)),
            "reached_R_full": bool(row.get("reached_R_full", False)),
            "above_energy_threshold": bool(row.get("above_energy_threshold", False)),
            "E_fourier_total_J": float(row.get("E_fourier_total_J", np.nan)),
            "full_status": str(row.get("full_status", "")),
            "dm_only_status": str(row.get("dm_only_status", "")),
        })
        rows.append(record)


def add_test2p5_candidates(rows: list[dict[str, Any]]) -> None:
    frame = load_optional(TEST2P5_RAYS)
    if len(frame) == 0:
        return
    converged = bools(frame, "converged", False)
    n_reach = numeric(frame, "n_reach", 0.0).fillna(0.0)
    mask = (~converged) | (n_reach <= 0.0)
    for _, row in frame.loc[mask].iterrows():
        record = common_record(row, "test2p5")
        record.update({
            "status": str(row.get("convergence_reason", "")),
            "candidate_category": (
                "test2p5_no_confirmed_reach"
                if float(row.get("n_reach", 0.0)) <= 0.0
                else "test2p5_unconverged_bracket"
            ),
            "priority": 2,
            "recommended_action": (
                "Add lower-b and speed anchors; a single success is not enough "
                "to certify finite measure."
            ),
            "bmax_lower_m": float(row.get("bmax_lower_m", np.nan)),
            "bmax_estimate_m": float(row.get("bmax_estimate_m", np.nan)),
            "bmax_upper_m": float(row.get("bmax_upper_m", np.nan)),
            "n_reach": float(row.get("n_reach", np.nan)),
            "n_unresolved": float(row.get("n_unresolved", np.nan)),
        })
        rows.append(record)


def add_final_exclusions(rows: list[dict[str, Any]]) -> None:
    frame = load_optional(FINAL_EXCLUDED)
    for _, row in frame.iterrows():
        record = common_record(row, "strict_final_builder")
        reason = str(row.get("final_exclusion_reason", ""))
        record.update({
            "status": reason,
            "candidate_category": "strict_policy_exclusion",
            "priority": 3,
            "recommended_action": (
                "Use this row only as an exploratory target. Do not assign "
                "production probability until the exclusion reason is resolved."
            ),
        })
        rows.append(record)


def main() -> None:
    rows: list[dict[str, Any]] = []
    add_adaptive_unresolved(rows)
    add_bowl_candidates(rows)
    add_test2_candidates(rows)
    add_test2p5_candidates(rows)
    add_final_exclusions(rows)

    if rows:
        table = pd.DataFrame(rows)
        identity = table["trajectory_id"].astype(str)
        fallback = table["ray_key"].astype(str)
        table["candidate_identity"] = np.where(
            identity.str.len() > 0,
            identity,
            np.where(fallback.str.len() > 0, fallback, table.index.astype(str)),
        )
        table = table.sort_values(
            ["priority", "candidate_category", "candidate_identity"],
            kind="mergesort",
        )
        table = table.drop_duplicates(
            ["source", "candidate_identity", "status"], keep="first"
        ).reset_index(drop=True)
    else:
        table = pd.DataFrame(columns=[
            "source", "candidate_identity", "status", "candidate_category",
            "priority", "recommended_action",
        ])

    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(OUTPUT_CSV, index=False)
    if len(table):
        summary = (
            table.groupby(
                ["priority", "candidate_category", "status"],
                dropna=False,
            )
            .size()
            .rename("count")
            .reset_index()
            .sort_values(["priority", "count"], ascending=[True, False])
        )
    else:
        summary = pd.DataFrame(
            columns=["priority", "candidate_category", "status", "count"]
        )
    summary.to_csv(SUMMARY_CSV, index=False)

    progress(f"Exploratory candidates: {len(table):,}")
    if len(summary):
        progress(summary.to_string(index=False))
    progress(f"Saved: {OUTPUT_CSV}")
    progress(f"Saved: {SUMMARY_CSV}")


if __name__ == "__main__":
    if bool(globals().get("USER_RUN_EXPLORATORY_REPORT", True)):
        main()
    else:
        progress("[SKIP] exploratory candidate report")


# Ground-Plane-Aware Single-Point Event Rate and Uncertainty

This stage uses the **same v16 phase-space convention**

$$
\hat{\mathbf u}=(\cos\theta,\,\sin\theta\cos\alpha,\,\sin\theta\sin\alpha),
\qquad
\mathbf r_{\rm in}=b\hat{\mathbf b}-\sqrt{R^2-b^2}\,\hat{\mathbf u}.
$$

It consumes the weighted Test-1.5 representatives rather than replacing the original truncation and refinement tests. Each Test-1.5 representative is treated as a **deterministic phase-space stratum** with physical flux weight

$$
W_i=n_\chi v_i\pi b_{\max,i}^2 w_i.
$$

For branch replica $r$ of stratum $i$, let $Y_{ir}$ denote one of the event observables

$$
Y_{ir}^{(\rm ph)}=\bar n_{k,ir},\qquad
Y_{ir}^{(\ge1)}=1-e^{-\bar n_{k,ir}},\qquad
Y_{ir}^{(M)}=e^{-\bar n_{k,ir}}\frac{\bar n_{k,ir}^{M}}{M!}.
$$

With $R_i$ finite ground-branch replicas, the single-point estimator is the stratified weighted sum

$$
\widehat{\mathcal R}=\sum_i W_i\,\overline{Y}_i,\qquad
\overline{Y}_i=\frac{1}{R_i}\sum_rY_{ir}.
$$

The notebook now reports the **branch Monte Carlo standard error**

$$
\operatorname{SE}_{\rm branch}(\widehat{\mathcal R})
=\sqrt{\sum_i W_i^2\frac{s_i^2}{R_i}},
$$

where $s_i^2$ is the sample variance of the branch replicas within stratum $i$. It also constructs one total rate per branch-replica index,

$$
\widehat{\mathcal R}^{(r)}=\sum_iW_iY_{ir},
$$

and reports $\operatorname{std}(\widehat{\mathcal R}^{(r)})/\sqrt{R}$ as an independent cross-check of the same branch-sampling uncertainty.

**Interpretation.** This is a true statistical standard error for the finite stochastic reflection/transmission/delayed-reemission sampling. It does **not** include the deterministic phase-space support/discretization uncertainty addressed by Tests 0--2.5, ODE-resolution uncertainty, uncertainty in $n_\chi$, or systematic uncertainty of the phenomenological copper model. Those must be quoted separately rather than folded into this Monte Carlo SE.

For each representative the trajectory is propagated from the outer radius using the full v16 DC plus RF pseudopotential force. Outside $R_{\rm switch}$ the ion evolves freely in its harmonic trap; inside $R_{\rm switch}$ the ion and MCP are propagated as a coupled system. Ground-plane encounters may produce specular reflection, prompt transmission through copper, or thermalization followed by delayed re-emission. A reflected trajectory can return to the interaction region and undergo another coupled core pass.

The selected-mode phonon-production rate is

$$
\dot N_{{\rm ph},k}=\sum_i W_i\,\langle\bar n_{k,i}\rangle_{\rm branches},\qquad
\bar n_{k,i}=\frac{|I_{k,i}|^2}{2m_{\rm ion}\hbar\omega_k}.
$$

The code also reports the rate of passages producing at least one phonon and the rate of exactly `USER_GROUND_EVENT_RATE_EXACT_PHONON_NUMBER` phonons, each with its branch-Monte-Carlo SE, relative SE, and $z\times\mathrm{SE}$ interval half-width. The possibility of MCPs being emitted spontaneously from material below the ion is not included.


In [ ]:
from v16_ground_plane_extension import (
    MODULE_VERSION as GROUND_EXTENSION_VERSION,
    MetalPrism,
    GroundRateConfig,
    run_event_rate_from_test1p5,
)

UPPER_COPPER_GROUND = MetalPrism(
    name="upper_copper_ground",
    x_min_m=float(USER_UPPER_GROUND_X_BOUNDS_M[0]),
    x_max_m=float(USER_UPPER_GROUND_X_BOUNDS_M[1]),
    y_min_m=float(USER_UPPER_GROUND_Y_BOUNDS_M[0]),
    y_max_m=float(USER_UPPER_GROUND_Y_BOUNDS_M[1]),
    z_min_m=float(USER_UPPER_GROUND_Z_BOUNDS_M[0]),
    z_max_m=float(USER_UPPER_GROUND_Z_BOUNDS_M[1]),
    material="copper",
    double_layer_eV=float(USER_UPPER_GROUND_DOUBLE_LAYER_EV),
    density_kg_m3=8960.0,
    atomic_mass_u=63.546,
    atomic_number=29,
    temperature_K=float(USER_UPPER_GROUND_TEMPERATURE_K),
    enable_delayed_reemission=bool(USER_UPPER_GROUND_ENABLE_DELAYED_REEMISSION),
)

GROUND_EVENT_RATE_CONFIG = GroundRateConfig(
    density_m3=float(USER_DM_NUMBER_DENSITY_M3),
    target_mode=int(USER_GROUND_EVENT_RATE_TARGET_MODE),
    exact_phonon_number=int(USER_GROUND_EVENT_RATE_EXACT_PHONON_NUMBER),
    branch_replicas=int(USER_GROUND_BRANCH_REPLICAS),
    max_rows=(None if USER_GROUND_EVENT_RATE_MAX_ROWS is None else int(USER_GROUND_EVENT_RATE_MAX_ROWS)),
    random_seed=int(USER_GROUND_EVENT_RATE_RANDOM_SEED),
    outer_radius_m=float(USER_GROUND_EVENT_RATE_OUTER_RADIUS_M),
    max_ground_interactions=int(USER_GROUND_MAX_INTERACTIONS),
    max_core_passes=int(USER_GROUND_MAX_CORE_PASSES),
    outside_rtol=float(USER_GROUND_OUTSIDE_RTOL),
    outside_atol=float(USER_GROUND_OUTSIDE_ATOL),
    outside_max_step_s=float(USER_GROUND_OUTSIDE_MAX_STEP_S),
    core_rtol=float(USER_GROUND_CORE_RTOL),
    core_atol=float(USER_GROUND_CORE_ATOL),
    core_max_step_s=float(USER_GROUND_CORE_MAX_STEP_S),
    coulomb_softening_m=float(USER_GROUND_COULOMB_SOFTENING_M),
    metal_transport_steps=int(USER_GROUND_METAL_TRANSPORT_STEPS),
    unresolved_policy=str(USER_GROUND_UNRESOLVED_POLICY),
    uncertainty_z=float(USER_GROUND_EVENT_RATE_UNCERTAINTY_Z),
)

print(f"Ground extension module version: {GROUND_EXTENSION_VERSION}")

if USER_RUN_GROUND_EVENT_RATE:
    ground_event_histories, ground_event_rate_summary = run_event_rate_from_test1p5(
        globals(),
        [UPPER_COPPER_GROUND],
        GROUND_EVENT_RATE_CONFIG,
        output_dir=RUN_DIRECTORY,
    )
    display(ground_event_rate_summary)

    uncertainty_columns = [
        "selected_mode_phonon_rate_s_inv",
        "selected_mode_phonon_rate_branch_mc_se_s_inv",
        "selected_mode_phonon_rate_branch_mc_rel_se",
        "selected_mode_phonon_rate_branch_mc_z_halfwidth_s_inv",
        "event_rate_ge1_s_inv",
        "event_rate_ge1_branch_mc_se_s_inv",
        "event_rate_ge1_branch_mc_rel_se",
        "event_rate_ge1_branch_mc_z_halfwidth_s_inv",
        "event_rate_exact_M_s_inv",
        "event_rate_exact_M_branch_mc_se_s_inv",
        "event_rate_exact_M_branch_mc_rel_se",
        "event_rate_exact_M_branch_mc_z_halfwidth_s_inv",
        "unresolved_flux_weight_fraction",
    ]
    uncertainty_columns = [c for c in uncertainty_columns if c in ground_event_rate_summary.columns]
    print("\nSingle-point branch Monte Carlo uncertainty summary")
    display(ground_event_rate_summary[uncertainty_columns])
    print(
        "The reported branch-MC SE covers only finite stochastic ground-branch replicas. "
        "Tests 0-2.5 support/discretization, ODE-resolution, density-normalization, and copper-model "
        "systematics remain separate uncertainties."
    )

    if len(ground_event_histories):
        status_summary = (
            ground_event_histories.groupby("status", dropna=False)
            .agg(
                histories=("status", "size"),
                resolved_fraction=("resolved", "mean"),
                phonon_rate_s_inv=("phonon_rate_contribution_s_inv", "sum"),
                ge1_rate_s_inv=("ge1_rate_contribution_s_inv", "sum"),
            )
            .sort_values("phonon_rate_s_inv", ascending=False)
        )
        display(status_summary)
else:
    print("Ground-plane event-rate stage disabled by USER_RUN_GROUND_EVENT_RATE=False")


# Current-Run Output Summary


In [ ]:
# Compact end-of-run summary. Individual stages print detailed reports above.
print("\nCompleted monolithic dark-matter trajectory pipeline")
print(f"  m_dm = {m_dm:.12e} kg")
print(f"  eps  = {eps:.12e}")
print(f"  T_dm = {T_dm:.6g} K")
print(f"  results directory = {RUN_DIRECTORY.resolve()}")
print("  parameter metadata =", (RUN_DIRECTORY / "run_parameters.json").resolve())
print("  strict final prefix =", FINAL_OUTPUT_PREFIX)
print(
    "  exploratory report =",
    (Path(RUN_DIRECTORY) / "exploratory_trajectory_candidates_v1.csv").resolve(),
)


print("\nBowl products")
for name in (
    "BOWL_BOUNDARY_CSV",
    "BOWL_SUMMARY_CSV",
    "BOWL_AXIS_PROFILES_CSV",
    "TRAJECTORY_AUDIT_CSV",
    "CONTINGENCY_CSV",
    "CONCLUSION_CSV",
):
    if name in globals():
        print(f"  {name}: {globals()[name]}")

if "ground_event_rate_summary" in globals():
    print("\nGround-plane-aware event-rate summary")
    if "GROUND_EXTENSION_VERSION" in globals():
        print(f"  ground extension module version = {GROUND_EXTENSION_VERSION}")
    print(ground_event_rate_summary.to_string(index=False))
